In [ ]:
# -*- coding: utf-8 -*-
"""R59 MLP ML 因子 (ml_mlp_wf35) 平台推理管线 — 单文件版。
35 特征平台侧现算 (口径同 scripts/mine_r57b_mltrain.py FEATURES 表;
bar1m volume/deal_number/amount 当日累计->差分, close/盘口快照不差分)。
模型: MLP 35->96->32->1 GELU walk-forward 年度 fold, 权重 JSON 内嵌,
  预测年 Y 用 fold Y (Y<=2020->2020 ... >=2024->2024);
  纯 numpy 手写前向 (GELU=erf 有理逼近, |err|<1.5e-7), 零第三方依赖。
输出: 逐日截面 z 特征面板 fillna(0) -> MLP -> SIGN=+1 ->
  bigalpha_2026_instruments inner merge。
"""
import json
import os

import numpy as np
import pandas as pd

SIGN = 1

FEATURE_COLS = [
    "deals_absret_corr", "deals_ac1", "bigdeal_ratio", "updn_asym_15m",
    "updn_asym_ma5", "ret_tail3", "resid_rv5_20", "z_retrange30_volac1",
    "retmax15_cashq", "retmax15_cffps", "vwapdevchg5_cffps", "gk5", "park5",
    "vol_ts_5_20", "rv5_tsz20", "vwap_disp", "imb_x_ret_std", "exec_int_mean",
    "exec_int_std", "vollead_corr_ma5", "turn", "rv_skew_ma5", "rv_skew_ma5_15m",
    "upvol_asym_ma5", "kurt_ma5", "n_reversals_30m", "ret_max_15m",
    "absret_ac1_ma5", "vwap_dev_chg5", "gap_freq_20", "b_pvsign_resid_rank_top1",
    "down_vol_share_15", "nm_vol_ret_corr", "nm_n_reversals", "upvol_asym_ma5_15m",
]

MORNING = (575, 630)  # 09:35-10:30

# ---------------------------------------------------------------- 数据访问

def _iter_month_ranges(q_start, q_end):
    q_start = pd.Timestamp(q_start)
    q_end = pd.Timestamp(q_end)
    cur = pd.Timestamp(year=q_start.year, month=q_start.month, day=1)
    while cur <= q_end:
        nxt = cur + pd.offsets.MonthBegin(1)
        cs = max(q_start, cur)
        ce = min(q_end, nxt - pd.Timedelta(seconds=1))
        if cs <= ce:
            yield cs, ce
        cur = nxt


def _dai_query_monthly(sql, q_start, q_end):
    """按月分片查询, 避免一次性加载全量 bar1m 导致 OOM。"""
    import dai
    frames = []
    for cs, ce in _iter_month_ranges(q_start, q_end):
        part = dai.query(
            sql,
            filters={"date": [cs.strftime("%Y-%m-%d %H:%M:%S"),
                              ce.strftime("%Y-%m-%d %H:%M:%S")]},
            compression=True,
        ).df()
        if len(part):
            frames.append(part)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _load_bar1m(datasources, start_date, end_date, buf_days=100):
    """平台 bar1m: volume/deal_number/amount 当日累计 -> 组内差分 (R45 结论);
    close/open/盘口字段为快照, 不差分 (R47 结论)。"""
    bar1m = datasources["bar1m"]
    q_start = pd.to_datetime(start_date) - pd.Timedelta(days=buf_days)
    q_end = pd.to_datetime(end_date)
    sql = (f"SELECT date, instrument::STRING AS instrument, open, high, low, close, "
           f"pre_close, volume, amount, deal_number, bid_price1, ask_price1, "
           f"bid_volume1, bid_volume2, bid_volume3, "
           f"ask_volume1, ask_volume2, ask_volume3, "
           f"bid_num_orders1, bid_num_orders2, bid_num_orders3, "
           f"ask_num_orders1, ask_num_orders2, ask_num_orders3 "
           f"FROM {bar1m} WHERE close > 0 ORDER BY instrument, date")
    df = _dai_query_monthly(sql, q_start, q_end)
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df["td"] = df["date"].dt.normalize()
    df["mins"] = df["date"].dt.hour * 60 + df["date"].dt.minute
    gk = ["instrument", "td"]
    for col, out in (("volume", "vol_min"), ("amount", "amt_min"),
                     ("deal_number", "dn_min")):
        df[out] = df[col] - df.groupby(gk, sort=False)[col].shift(1)
        df[out] = df[out].fillna(df[col]).clip(lower=0)
    return df


def _load_factorlib(start_date, end_date, buf_days=100):
    q_start = pd.to_datetime(start_date) - pd.Timedelta(days=buf_days)
    q_end = pd.to_datetime(end_date)
    sql = "SELECT date, instrument, daily_return, turn FROM bigalpha_2026_factorlib"
    df = _dai_query_monthly(sql, q_start, q_end)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    return df.sort_values(["instrument", "date"]).reset_index(drop=True)


def _fin_grid_end(end_date):
    """财务 ffill 的 canonical 窗口右端 (R58 移植发现, 见 _load_fin 注释):
    缓存构建用 grid 2019-01-01..2024-12-31 + fin 2018-07..2024-12-31;
    verify_f8 证明右端取 2023-12-31 即可逐值复现缓存 (≤2023-12-31),
    生产期 (>2023-12-31) 用实际 end_date 向右自然延展。"""
    return max(pd.to_datetime(end_date), pd.Timestamp("2023-12-31"))


def _load_fin(datasources, start_date, end_date):
    """财务 PIT, 固定 canonical 窗口 2018-07-01 .. _fin_grid_end(end_date)。
    注意 (R58 移植发现): mine_r26_tilt2.fin_components 的 grid.merge+ffill 结果
    依赖合并后行序 (当前 pandas 下左合并不保序, 乱序 ffill 成为缓存构建的一部分),
    只有用与缓存构建相同的 canonical 输入窗口才能复现缓存值。"""
    fin = datasources["financial"]
    q_start = pd.Timestamp("2018-07-01")
    q_end = _fin_grid_end(end_date)
    sql = (f"SELECT date, instrument::STRING AS instrument, category, "
           f"net_cffoa, net_profit, net_cfffa, latest_shares "
           f"FROM {fin} WHERE shift=0")
    df = _dai_query_monthly(sql, q_start, q_end)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    return df


def _stk(start_date, end_date):
    import dai
    stk = dai.query("SELECT date, instrument FROM bigalpha_2026_instruments",
                    filters={"date": [str(pd.to_datetime(start_date)),
                                      str(pd.to_datetime(end_date))]}).df()
    stk["date"] = pd.to_datetime(stk["date"]).dt.normalize()
    stk["instrument"] = stk["instrument"].astype(str)
    return stk


# ---------------------------------------------------------------- 通用算子

def _grp_corr_nan(m, x, y, min_n=10):
    """组内 pearson; 样本<min_n 或零方差 -> NaN (R45 系约定)。"""
    t = m[["instrument", "td", x, y]].dropna(subset=[x, y])
    t = t.assign(xx=t[x] * t[x], yy=t[y] * t[y], xy=t[x] * t[y])
    gb = t.groupby(["instrument", "td"], sort=False)
    s = gb[[x, y, "xx", "yy", "xy"]].sum()
    n = gb.size()
    num = n * s["xy"] - s[x] * s[y]
    den = np.sqrt((n * s["xx"] - s[x] ** 2) * (n * s["yy"] - s[y] ** 2))
    return (num / den.replace(0, np.nan)).where(n >= min_n)


def _grp_corr_zero(m, x, y, min_n=30):
    """组内 pearson; n<min_n 或零方差 -> 0.0 (R17/R20 系约定)。"""
    t = m[["instrument", "td", x, y]].dropna(subset=[x, y]).copy()
    t["xy"] = t[x] * t[y]
    t["xx"] = t[x] ** 2
    t["yy"] = t[y] ** 2
    a = t.groupby(["instrument", "td"], sort=False).agg(
        n=(x, "size"), sx=(x, "sum"), sy=(y, "sum"),
        sxy=("xy", "sum"), sxx=("xx", "sum"), syy=("yy", "sum"))
    cov = a["sxy"] - a["sx"] * a["sy"] / a["n"]
    vx = a["sxx"] - a["sx"] ** 2 / a["n"]
    vy = a["syy"] - a["sy"] ** 2 / a["n"]
    out = cov / np.sqrt(vx.clip(lower=0) * vy.clip(lower=0)).replace(0, np.nan)
    out[(a["n"] < min_n) | (vx <= 0) | (vy <= 0)] = 0.0
    return out.fillna(0.0)


def _winsorize_mad(s, n=3.0):
    med = s.median()
    mad = (s - med).abs().median()
    if mad == 0 or mad != mad:
        return s
    return s.clip(med - n * 1.4826 * mad, med + n * 1.4826 * mad)


def _xs_z(df, col):
    """逐日 MAD-winsorize + z (mine_r26_tilt2 口径)。"""
    def _proc(g):
        g = g.copy()
        g[col] = _winsorize_mad(g[col])
        std = g[col].std()
        if std and std > 0:
            g[col] = (g[col] - g[col].mean()) / std
        return g
    out = df.dropna(subset=[col]).replace([np.inf, -np.inf], np.nan).dropna(subset=[col])
    return out.groupby("date", group_keys=False).apply(_proc)


def _to_nm(df, n):
    """1m -> %n 采样, 加 ret/absret/dn/deal_size/volume/amount (R45/R46 口径)。"""
    m = df[df["date"].dt.minute % n == 0].copy()
    gk = ["instrument", "td"]
    prev_c = m.groupby(gk, sort=False)["close"].shift(1)
    m["ret"] = (m["close"] / prev_c - 1.0).fillna(0.0)
    m["absret"] = m["ret"].abs()
    m["volume"] = m["vol_min"]
    m["amount"] = m["amt_min"]
    dn = m["dn_min"].astype(float)
    m["dn"] = dn.where(dn > 0)
    m["deal_size"] = (m["volume"] / dn.replace(0, np.nan)).where(dn > 0)
    return m


def _updn_asym(m):
    """涨/跌 bar 均笔(或均量) 不对称 (ret=0 剔除), 列名 dn 或 volume 自适应。"""
    gk = ["instrument", "td"]
    up = m["ret"] > 0
    dwn = m["ret"] < 0
    vu = m.assign(vu_up=m["_x"] * up, cnt_up=up.astype(float),
                  vu_dn=m["_x"] * dwn, cnt_dn=dwn.astype(float))
    va = vu.groupby(gk, sort=False).agg(
        vu_up=("vu_up", "sum"), cnt_up=("cnt_up", "sum"),
        vu_dn=("vu_dn", "sum"), cnt_dn=("cnt_dn", "sum"))
    mu = va["vu_up"] / va["cnt_up"].replace(0, np.nan)
    md = va["vu_dn"] / va["cnt_dn"].replace(0, np.nan)
    return ((mu - md) / (mu + md).replace(0, np.nan))


def _roll5(s, mp=3):
    return s.rolling(5, min_periods=mp).mean()


# ---------------------------------------------------------------- 特征计算

def _features(df1m, fl, fin, stk_grid):
    """返回 dict[name] -> [date, instrument, factor] (与缓存一致的带符号值)。"""
    F = {}
    m5 = _to_nm(df1m, 5)
    m15 = _to_nm(df1m, 15)
    m30 = _to_nm(df1m, 30)

    def day(s, name):
        d = s.rename(name).reset_index()
        d.columns = ["instrument", "date", name] if d.shape[1] == 3 else d.columns
        return d

    # ---- L1 deal_number 系 ----
    F["deals_absret_corr"] = -_grp_corr_nan(m5, "dn", "absret", 10)
    m5["dn_lag"] = m5.groupby(["instrument", "td"], sort=False)["dn"].shift(1)
    F["deals_ac1"] = -_grp_corr_nan(m5.dropna(subset=["dn", "dn_lag"]),
                                    "dn", "dn_lag", 10)
    q = m5.groupby(["instrument", "td"], sort=False)["deal_size"].agg(["max", "median"])
    F["bigdeal_ratio"] = q["max"] / q["median"].replace(0, np.nan)

    m15["_x"] = m15["dn"]
    F["updn_asym_15m"] = -_updn_asym(m15.dropna(subset=["dn"]))
    m5["_x"] = m5["dn"]
    a5 = _updn_asym(m5.dropna(subset=["dn"])).rename("v").reset_index()
    a5 = a5.sort_values(["instrument", "td"])
    a5["v"] = a5.groupby("instrument", group_keys=False)["v"].transform(_roll5)
    F["updn_asym_ma5"] = -a5.set_index(["instrument", "td"])["v"]

    # ret_tail3 (1m)
    g1 = ["instrument", "td"]
    df1m["seq"] = df1m.groupby(g1, sort=False).cumcount()
    df1m["nbar"] = df1m.groupby(g1, sort=False)["seq"].transform("max") + 1
    df1m["close_l3"] = df1m.groupby(g1, sort=False)["close"].shift(3)
    last = df1m[df1m["seq"] == df1m["nbar"] - 1]
    F["ret_tail3"] = -(last["close"] / last["close_l3"] - 1.0) \
        .set_axis(pd.MultiIndex.from_frame(last[["instrument", "td"]]))

    # ---- L2 在榜基底 ----
    # z_retrange30_volac1: (z_mad(retrange30)+z_mad(vol_ac1_30m))/2, sign -1
    a = m30.groupby(["instrument", "td"], sort=False).agg(
        rmax=("ret", "max"), rmin=("ret", "min"))
    rr = (a["rmax"] - a["rmin"]).rename("v").reset_index() \
        .rename(columns={"td": "date"})
    m30["vol_l1"] = m30.groupby(g1, sort=False)["volume"].shift(1)
    t = m30.dropna(subset=["vol_l1"]).copy()
    t["xy"] = t["volume"] * t["vol_l1"]
    t["xx"] = t["volume"] ** 2
    t["yy"] = t["vol_l1"] ** 2
    a2 = t.groupby(g1, sort=False).agg(
        n=("volume", "size"), sx=("volume", "sum"), sy=("vol_l1", "sum"),
        sxy=("xy", "sum"), sxx=("xx", "sum"), syy=("yy", "sum"))
    cov = a2["sxy"] - a2["sx"] * a2["sy"] / a2["n"]
    vx = a2["sxx"] - a2["sx"] ** 2 / a2["n"]
    vy = a2["syy"] - a2["sy"] ** 2 / a2["n"]
    va = (cov / np.sqrt(vx.clip(lower=0) * vy.clip(lower=0)).replace(0, np.nan))
    va[(a2["n"] < 5) | (vx <= 0) | (vy <= 0)] = np.nan
    va = va.rename("v").reset_index().rename(columns={"td": "date"})
    za = _xs_z(rr, "v").rename(columns={"v": "za"})
    zb = _xs_z(va.dropna(subset=["v"]), "v").rename(columns={"v": "zb"})
    mm = za.merge(zb, on=["date", "instrument"], how="inner")
    F["z_retrange30_volac1"] = -((mm["za"] + mm["zb"]) / 2.0) \
        .set_axis(pd.MultiIndex.from_frame(mm[["date", "instrument"]]))

    # ret_max_15m (raw, 也供 tilt 用)
    rmax15 = m15.groupby(["instrument", "td"], sort=False)["ret"].max()
    F["ret_max_15m"] = rmax15

    # fin cashq / cffps (r13 PIT 口径, ffill 到池网格)
    fin_z = {}
    if fin is not None and len(fin):
        grid = stk_grid[["date", "instrument"]].drop_duplicates()

        def _fin_ratio(num, den):
            a = fin[fin["category"] == "ttm"][["date", "instrument", num]] \
                .rename(columns={num: "vn"})
            b = fin[fin["category"] == "ttm"][["date", "instrument", den]] \
                .rename(columns={den: "vd"})
            m = a.merge(b, on=["date", "instrument"])
            m["fv"] = m["vn"] / (m["vd"] + 1e-8)
            m = m[["date", "instrument", "fv"]].dropna()
            g = grid.merge(m, on=["date", "instrument"], how="left")
            g["fv"] = g.groupby("instrument", group_keys=False)["fv"].ffill()
            return g.dropna(subset=["fv"])

        def _fin_ps(num):
            a = fin[fin["category"] == "ttm"][["date", "instrument", num]] \
                .rename(columns={num: "vn"})
            b = fin[fin["category"] == "lf"][["date", "instrument", "latest_shares"]] \
                .rename(columns={"latest_shares": "sh"})
            m = a.merge(b, on=["date", "instrument"])
            m["fv"] = m["vn"] / (m["sh"] + 1e-8)
            m = m[["date", "instrument", "fv"]].dropna()
            g = grid.merge(m, on=["date", "instrument"], how="left")
            g["fv"] = g.groupby("instrument", group_keys=False)["fv"].ffill()
            return g.dropna(subset=["fv"])

        fin_z["cashq"] = _fin_ratio("net_cffoa", "net_profit")
        fin_z["cffps"] = _fin_ps("net_cfffa")

    rm = rmax15.rename("v").reset_index().rename(columns={"td": "date"})
    zrm = _xs_z(rm, "v").rename(columns={"v": "zb"})
    for fid, fin_name, w, fsign in (("retmax15_cashq", "cashq", 1.0, 1),
                                    ("retmax15_cffps", "cffps", 1.0, -1)):
        zf = _xs_z(fin_z[fin_name], "fv").rename(columns={"fv": "zf"})
        mm2 = zrm.merge(zf, on=["date", "instrument"], how="inner")
        # cache = -[zb * (1+w*(fsign*zf).clip(-2,2))] (cffps FIN_SIGN=-1, 见 r37 #05)
        F[fid] = -(mm2["zb"] * (1.0 + w * (fsign * mm2["zf"]).clip(-2.0, 2.0))) \
            .set_axis(pd.MultiIndex.from_frame(mm2[["date", "instrument"]]))

    # vwapdevchg5_cffps: -z(vwap_dev_chg5) * (1+0.5*clip(-z(cffps),-2,2)), sign +1
    am5 = m5.groupby(["instrument", "td"], sort=False).agg(
        amt=("amount", "sum"), vol=("volume", "sum"), close_d=("close", "last"))
    dev = (am5["close_d"] / (am5["amt"] / am5["vol"].replace(0, np.nan)) - 1.0) \
        .rename("v").reset_index().rename(columns={"td": "date"})
    dev = dev.sort_values(["instrument", "date"])
    dev["v"] = dev["v"] - dev.groupby("instrument", group_keys=False)["v"].shift(5)
    F["vwap_dev_chg5"] = -dev.set_index(["date", "instrument"])["v"]  # sign -1
    zdev = _xs_z(dev.dropna(subset=["v"]), "v").rename(columns={"v": "zb"})
    zcf = _xs_z(fin_z["cffps"], "fv").rename(columns={"fv": "zf"})
    mm3 = zdev.merge(zcf, on=["date", "instrument"], how="inner")
    F["vwapdevchg5_cffps"] = ((-mm3["zb"])
                              * (1.0 + 0.5 * (-mm3["zf"]).clip(-2.0, 2.0))) \
        .set_axis(pd.MultiIndex.from_frame(mm3[["date", "instrument"]]))

    # ---- L3 波动族 ----
    # vwap_disp (raw, R54 口径)
    hl = m5.groupby(["instrument", "td"], sort=False).agg(
        vol=("volume", "sum"), amt=("amount", "sum"))
    vwap_d = (hl["amt"] / hl["vol"].replace(0, np.nan)).rename("vwap")
    m5j = m5.join(vwap_d, on=["instrument", "td"])
    dv = m5j["close"] - m5j["vwap"]
    m5j["wv2"] = m5j["volume"] * dv ** 2
    w2 = m5j.groupby(["instrument", "td"], sort=False).agg(
        wv2=("wv2", "sum"), vol=("volume", "sum"))
    w2 = w2.join(vwap_d, on=["instrument", "td"])
    F["vwap_disp"] = (np.sqrt(w2["wv2"] / w2["vol"].replace(0, np.nan))
                      / w2["vwap"].replace(0, np.nan))

    # imb_x_ret_std (sign -1, R47 口径)
    bv = m5["bid_volume1"] + m5["bid_volume2"] + m5["bid_volume3"]
    av = m5["ask_volume1"] + m5["ask_volume2"] + m5["ask_volume3"]
    ok_vol = (bv + av) > 0
    m5["imb3"] = ((bv - av) / (bv + av)).where(ok_vol)
    m5["imb_x_ret"] = m5["imb3"] * m5["ret"]
    F["imb_x_ret_std"] = -m5.groupby(["instrument", "td"], sort=False)[
        "imb_x_ret"].std()

    # exec_int (raw, R52 口径)
    bno = m5["bid_num_orders1"] + m5["bid_num_orders2"] + m5["bid_num_orders3"]
    ano = m5["ask_num_orders1"] + m5["ask_num_orders2"] + m5["ask_num_orders3"]
    no_total = (bno + ano).where((bno + ano) > 0)
    m5["exec_int"] = (m5["dn_min"].astype(float) / no_total).where(no_total > 0)
    F["exec_int_mean"] = m5.groupby(["instrument", "td"], sort=False)["exec_int"].mean()
    F["exec_int_std"] = m5.groupby(["instrument", "td"], sort=False)["exec_int"].std()

    # vollead_corr_ma5 (sign -1): corr(vol_t, |ret_t+1|) 5m, roll5
    m5["absret_lead"] = m5.groupby(["instrument", "td"], sort=False)["absret"].shift(-1)
    vl = _grp_corr_nan(m5, "volume", "absret_lead", 10).rename("v").reset_index()
    vl = vl.sort_values(["instrument", "td"])
    vl["v"] = vl.groupby("instrument", group_keys=False)["v"].transform(_roll5)
    F["vollead_corr_ma5"] = -vl.set_index(["instrument", "td"])["v"]

    # turn (raw factorlib)
    F["turn"] = fl.set_index(["date", "instrument"])["turn"]

    # resid_rv5_20 (sign -1, R52 口径; factorlib daily_return)
    P = fl.sort_values(["instrument", "date"]).reset_index(drop=True)
    mkt = P.groupby("date")["daily_return"].mean().rename("mkt")
    P = P.merge(mkt, on="date", how="left")
    g = P.groupby("instrument", group_keys=False)
    xy = P["daily_return"] * P["mkt"]
    y2 = P["mkt"] ** 2
    ex = g["daily_return"].transform(lambda s: s.rolling(60, min_periods=45).mean())
    ey = g["mkt"].transform(lambda s: s.rolling(60, min_periods=45).mean())
    exy = xy.groupby(P["instrument"], group_keys=False).transform(
        lambda s: s.rolling(60, min_periods=45).mean())
    ey2 = y2.groupby(P["instrument"], group_keys=False).transform(
        lambda s: s.rolling(60, min_periods=45).mean())
    beta60 = (exy - ex * ey) / (ey2 - ey ** 2).replace(0, np.nan)
    resid = P["daily_return"] - beta60 * P["mkt"]
    gr = resid.groupby(P["instrument"], group_keys=False)
    rv5 = gr.transform(lambda s: s.rolling(5, min_periods=4).std())
    rv20 = gr.transform(lambda s: s.rolling(20, min_periods=15).std())
    P["resid_rv5_20"] = -(rv5 / rv20.replace(0, np.nan))
    F["resid_rv5_20"] = P.set_index(["date", "instrument"])["resid_rv5_20"]

    # vol_ts_5_20 / rv5_tsz20 (raw, factorlib daily_return)
    rv5r = g["daily_return"].transform(lambda s: s.rolling(5, min_periods=4).std())
    rv20r = g["daily_return"].transform(lambda s: s.rolling(20, min_periods=15).std())
    P["vol_ts_5_20"] = rv5r / rv20r.replace(0, np.nan)
    g5 = rv5r.groupby(P["instrument"], group_keys=False)
    ma20 = g5.transform(lambda s: s.rolling(20, min_periods=15).mean())
    sd20 = g5.transform(lambda s: s.rolling(20, min_periods=15).std())
    P["rv5_tsz20"] = (rv5r - ma20) / sd20.replace(0, np.nan)
    F["vol_ts_5_20"] = P.set_index(["date", "instrument"])["vol_ts_5_20"]
    F["rv5_tsz20"] = P.set_index(["date", "instrument"])["rv5_tsz20"]

    # ---- L4 历史强因子 ----
    # rv_skew (r2 矩偏度, r19b 口径) ma5, 5m/15m
    for m, name in ((m5, "rv_skew_ma5"), (m15, "rv_skew_ma5_15m")):
        m["r2"] = m["ret"] ** 2
        m["r4"] = m["r2"] ** 2
        m["r6"] = m["r2"] ** 3
        a = m.groupby(["instrument", "td"], sort=False).agg(
            n=("r2", "size"), s2=("r2", "sum"), s4=("r4", "sum"), s6=("r6", "sum"))
        m1, m2_, m3 = a["s2"] / a["n"], a["s4"] / a["n"], a["s6"] / a["n"]
        var = (m2_ - m1 ** 2).clip(lower=0)
        sk = ((m3 - 3 * m1 * m2_ + 2 * m1 ** 3) / var.pow(1.5).replace(0, np.nan)) \
            .rename("v").reset_index()
        sk = sk.sort_values(["instrument", "td"])
        sk["v"] = sk.groupby("instrument", group_keys=False)["v"].transform(_roll5)
        F[name] = -sk.set_index(["instrument", "td"])["v"]  # sign -1

    # upvol_asym_ma5 / upvol_asym_ma5_15m (volume 版, sign -1 缓存为 -raw)
    for m, name in ((m5, "upvol_asym_ma5"), (m15, "upvol_asym_ma5_15m")):
        m["_x"] = m["volume"]
        ua = _updn_asym(m).rename("v").reset_index()
        ua = ua.sort_values(["instrument", "td"])
        ua["v"] = ua.groupby("instrument", group_keys=False)["v"].transform(_roll5)
        F[name] = -ua.set_index(["instrument", "td"])["v"]

    # kurt_ma5 (r19c 口径: 5m ret 超额峰度, 全 bar; roll5; sign -1)
    m5["r1"] = m5["ret"]
    m5["s2_"] = m5["r1"] ** 2
    m5["s3_"] = m5["r1"] ** 3
    m5["s4_"] = m5["r1"] ** 4
    a = m5.groupby(["instrument", "td"], sort=False).agg(
        n=("r1", "size"), s1=("r1", "sum"), s2=("s2_", "sum"),
        s3=("s3_", "sum"), s4=("s4_", "sum"))
    n = a["n"]
    m1 = a["s1"] / n
    m2v = a["s2"] / n - m1 ** 2
    m4 = (a["s4"] / n - 4 * m1 * a["s3"] / n
          + 6 * m1 ** 2 * a["s2"] / n - 3 * m1 ** 4)
    kd = (m4 / m2v.clip(lower=0) ** 2 - 3.0)
    kd[m2v <= 0] = np.nan
    kd = kd.rename("v").reset_index()
    kd = kd.sort_values(["instrument", "td"])
    kd["v"] = kd.groupby("instrument", group_keys=False)["v"].transform(_roll5)
    F["kurt_ma5"] = -kd.set_index(["instrument", "td"])["v"]  # sign -1

    # n_reversals_30m (B_N_REV_30M, sign -1)
    s = np.sign(m30["ret"]).replace(0, np.nan)
    sff = s.groupby([m30["instrument"], m30["td"]], sort=False).ffill()
    prev_s = sff.groupby([m30["instrument"], m30["td"]], sort=False).shift(1)
    rev = ((sff * prev_s) < 0).astype(float)
    F["n_reversals_30m"] = -rev.groupby([m30["instrument"], m30["td"]],
                                        sort=False).sum()

    # absret_ac1_ma5 (R17 系 fill-0 corr min30, roll5, sign -1)
    m5["absret_l1"] = m5.groupby(["instrument", "td"], sort=False)["absret"].shift(1)
    ac = _grp_corr_zero(m5.dropna(subset=["absret_l1"]), "absret", "absret_l1", 30) \
        .rename("v").reset_index()
    ac = ac.sort_values(["instrument", "td"])
    ac["v"] = ac.groupby("instrument", group_keys=False)["v"].transform(_roll5)
    F["absret_ac1_ma5"] = -ac.set_index(["instrument", "td"])["v"]  # sign -1

    # gap_freq_20 (sign -1): |open/pre_close-1|>0.01 的 20 日均 (min10)
    # down_vol_share_15 (sign +1)
    # park5 / gk5 (raw)
    dy = df1m.groupby(["instrument", "td"], sort=False).agg(
        open=("open", "first"), high=("high", "max"), low=("low", "min"),
        close=("close", "last"))
    dy = dy.reset_index().rename(columns={"td": "date"})
    dy = dy.sort_values(["instrument", "date"]).reset_index(drop=True)
    dy["pre_close"] = dy.groupby("instrument", group_keys=False)["close"].shift(1)
    # 注: 平台 pre_close 官方昨收与 shift(close) 在非除权日一致;
    # r17 系缓存经实测与 shift(close) 口径最接近 (gap 0.99995/down_vol 0.99998,
    # 优于 csv pre_close 比例修正的 0.993/0.977), 故用 shift 口径。
    dy["gap"] = dy["open"] / dy["pre_close"] - 1.0
    is_gap = (dy["gap"].abs() > 0.01).astype(float).where(dy["gap"].notna())
    dy["ret"] = dy["close"] / dy["pre_close"] - 1.0
    dy["ret_dn2"] = np.minimum(dy["ret"], 0.0) ** 2
    dy["ret2"] = dy["ret"] ** 2
    gdy = dy.groupby("instrument", group_keys=False)
    F["gap_freq_20"] = -is_gap.groupby(dy["instrument"], group_keys=False).transform(
        lambda s: s.rolling(20, min_periods=10).mean()) \
        .set_axis(pd.MultiIndex.from_frame(dy[["date", "instrument"]]))
    dn15 = gdy["ret_dn2"].transform(lambda s: s.rolling(15, min_periods=7).sum())
    tt15 = gdy["ret2"].transform(lambda s: s.rolling(15, min_periods=7).sum())
    F["down_vol_share_15"] = (dn15 / tt15.replace(0, np.nan)) \
        .set_axis(pd.MultiIndex.from_frame(dy[["date", "instrument"]]))
    ln_hl2 = np.log(dy["high"] / dy["low"]) ** 2
    ln_co2 = np.log(dy["close"] / dy["open"]) ** 2
    m5_hl = ln_hl2.groupby(dy["instrument"], group_keys=False).transform(
        lambda s: s.rolling(5, min_periods=4).mean())
    m5_co = ln_co2.groupby(dy["instrument"], group_keys=False).transform(
        lambda s: s.rolling(5, min_periods=4).mean())
    F["park5"] = np.sqrt(m5_hl / (4.0 * np.log(2))) \
        .set_axis(pd.MultiIndex.from_frame(dy[["date", "instrument"]]))
    F["gk5"] = np.sqrt((m5_hl * 0.5 - m5_co * (2 * np.log(2) - 1)).clip(lower=0)) \
        .set_axis(pd.MultiIndex.from_frame(dy[["date", "instrument"]]))

    # b_pvsign_resid_rank_top1 (sign -1; 池内残差, R20 notebook 口径)
    prev_c1 = df1m.groupby(g1, sort=False)["close"].shift(1)
    df1m["ret1"] = df1m["close"] / prev_c1 - 1.0
    ok = df1m["ret1"].notna() & (df1m["vol_min"] >= 0)
    t1 = df1m[ok].copy()
    t1["sv"] = np.sign(t1["ret1"]) * t1["vol_min"]
    pr = t1.groupby(g1, sort=False).agg(sv=("sv", "sum"), vv=("vol_min", "sum"))
    pr["pressure"] = -(pr["sv"] / (pr["vv"] + 1e-8))
    prev_v5 = m5.groupby(g1, sort=False)["volume"].shift(1)
    m5["dv"] = (m5["volume"] / prev_v5.replace(0, np.nan) - 1.0).fillna(0.0)
    pvs = _grp_corr_zero(m5, "dv", "ret", 30).rename("pvsign").reset_index()
    day = pvs.merge(pr[["pressure"]].reset_index(), on=g1, how="inner")
    day = day.rename(columns={"td": "date"})
    day = day.merge(stk_grid[["date", "instrument"]].drop_duplicates(),
                    on=["date", "instrument"], how="inner")

    def _resid(g):
        y = g["pvsign"].to_numpy(dtype=float)
        x = g["pressure"].rank().to_numpy(dtype=float)
        okk = ~np.isnan(y) & ~np.isnan(x)
        res = np.full_like(y, np.nan)
        if okk.sum() >= 31:
            A = np.column_stack([np.ones(okk.sum()), x[okk]])
            beta, *_ = np.linalg.lstsq(A, y[okk], rcond=None)
            res[okk] = y[okk] - A @ beta
        return pd.Series(res, index=g.index)

    day["pvsign_resid"] = day.groupby("date", group_keys=False).apply(_resid)
    F["b_pvsign_resid_rank_top1"] = -day.set_index(["date", "instrument"])[
        "pvsign_resid"]

    # nm_vol_ret_corr / nm_n_reversals (【5m】口径, recompute_nm3.factors_from_5m:
    # corr(5m volume, |5m ret|) 与 5m close.diff 非零符号变号次数, n<30->0.0, sign -1)
    F["nm_vol_ret_corr"] = -_grp_corr_zero(m5, "volume", "absret", 30)
    s1 = np.sign((m5["close"] - m5.groupby(g1, sort=False)["close"].shift(1))
                 .fillna(0.0))
    regime = s1.replace(0, np.nan).groupby([m5["instrument"], m5["td"]],
                                           sort=False).ffill()
    prev_regime = regime.groupby([m5["instrument"], m5["td"]],
                                 sort=False).shift(1)
    nrev = (regime.notna() & prev_regime.notna()
            & (regime != prev_regime)).astype(float)
    nrev_d = nrev.groupby([m5["instrument"], m5["td"]], sort=False).sum()
    nbar5 = m5.groupby(g1, sort=False)["close"].size()
    F["nm_n_reversals"] = -nrev_d.where(nbar5 >= 30, 0.0)

    out = {}
    for name, s in F.items():
        d = s.rename("factor").reset_index()
        d = d.rename(columns={"td": "date"})
        d["date"] = pd.to_datetime(d["date"]).dt.normalize()
        out[name] = d[["date", "instrument", "factor"]] \
            .replace([np.inf, -np.inf], np.nan)
    return out


# ---------------------------------------------------------------- 模型推理

def _model_year(y):
    if y <= 2020:
        return 2020
    if y == 2021:
        return 2021
    if y == 2022:
        return 2022
    if y == 2023:
        return 2023
    return 2024



MLP_WEIGHTS = json.loads('''{"2020": {"W1": [[-0.09370448440313339, -0.13325367867946625, 0.05772978812456131, -0.06518124043941498, 0.05181168392300606, -0.11528649181127548, 0.048574887216091156, -0.05555601418018341, 0.029079578816890717, 0.09967511147260666, 0.06671739369630814, -0.03348876163363457, -0.001258949632756412, 0.15211018919944763, 0.06662917137145996, -0.11523151397705078, -0.10971338301897049, -0.13364914059638977, -0.04064455255866051, 0.06687871366739273, 0.000498511188197881, 0.13683660328388214, -0.08487903326749802, -0.01619880087673664, 0.11071330308914185, -0.03498974069952965, -0.12282812595367432, 0.11386535316705704, -0.17184525728225708, 0.07079808413982391, 0.1306973695755005, -0.11149171739816666, -0.11227255314588547, -0.13381043076515198, -0.03440980240702629], [-0.09746458381414413, -0.17372842133045197, -0.1278255581855774, -0.12523281574249268, 0.058316804468631744, -0.04596050828695297, -0.10099655389785767, 0.08970405906438828, -0.17329072952270508, -0.0006403785082511604, 0.066193126142025, 0.17153391242027283, -0.06876541674137115, -0.12418363988399506, 0.008868619799613953, 0.1267862766981125, 0.05560853332281113, 0.1275903731584549, 0.04352414235472679, 0.06695384532213211, -0.05923863872885704, -0.04901155084371567, -0.12882234156131744, -0.015174406580626965, -0.11499585211277008, -0.07449175417423248, -0.12205514311790466, 0.017617709934711456, -0.13893814384937286, -0.04109836742281914, -0.13068410754203796, 0.1279483288526535, 0.05732835456728935, -0.06176450848579407, -0.07727856189012527], [-0.10819011181592941, 0.041217558085918427, 0.05811252444982529, -0.08059318363666534, -0.1357775330543518, -0.13919804990291595, -0.13887694478034973, 0.14196406304836273, 0.0981229916214943, -0.03636760264635086, 0.15362408757209778, 0.06619139760732651, 0.053526055067777634, -0.07727458328008652, 0.11598756909370422, -0.10467536747455597, -0.026021914556622505, 0.17281973361968994, 0.12490511685609818, 0.08370456099510193, -0.11364845186471939, 0.0690845176577568, -0.051496293395757675, -0.11194508522748947, 0.025377444922924042, 0.11817379295825958, 0.09781285375356674, 0.16322389245033264, -0.03625978156924248, 0.12472306191921234, 0.03426803648471832, 0.0785408541560173, 0.00405337568372488, 0.002014423254877329, 0.09117349982261658], [0.11404258012771606, 0.06935706734657288, -0.03949674963951111, -0.07794693112373352, 0.10921957343816757, 0.11200462281703949, 0.12760169804096222, 0.04733360558748245, -0.10471203923225403, 0.003096106695011258, 0.08540130406618118, 0.1616469919681549, 0.07094994932413101, 0.12706024944782257, 0.0002663847990334034, -0.11478685587644577, -0.07439549267292023, 0.01585358940064907, -0.019764212891459465, -0.11764488369226456, -0.057317253202199936, -0.12689751386642456, 0.11821956187486649, -0.12033619731664658, 0.039896197617053986, -0.05834311619400978, -0.10493828356266022, 0.031089548021554947, -0.17819133400917053, -0.07994380593299866, 0.08931492269039154, -0.12211507558822632, -0.1320122629404068, 0.24061399698257446, 0.02795718051493168], [0.043868523091077805, -0.014336780644953251, 0.04114271327853203, -0.03844410926103592, 0.1102500632405281, -0.09532410651445389, 0.09627609699964523, 0.09887298196554184, -0.051138751208782196, 0.05327612906694412, 0.08759427815675735, -0.020238230004906654, -0.05574684590101242, -0.04646250233054161, -0.007677124347537756, -0.10124500840902328, 0.10223135352134705, -0.04664250835776329, -0.0019913427531719208, 0.04899927228689194, -0.04221867024898529, 0.02396688424050808, -0.030475031584501266, -0.08577544242143631, 0.055359337478876114, 0.10282406210899353, -0.09027513116598129, 0.04853710159659386, 0.08777830004692078, 0.05600276216864586, -0.09310698509216309, -0.09681999683380127, -0.08668338507413864, -0.1357681006193161, 0.017552392557263374], [0.025544660165905952, 0.07205893099308014, -0.12442637234926224, -0.16373971104621887, 0.10012859106063843, 0.029893314465880394, -0.07362689822912216, -0.03227092698216438, 0.015562857501208782, -0.041522588580846786, -0.1320527344942093, -0.026312001049518585, -0.12038344889879227, -0.1316337287425995, 0.1225554570555687, 0.040530432015657425, -0.08077828586101532, 0.057105470448732376, 0.14667470753192902, -0.09833574295043945, -0.04811429977416992, 0.00798296183347702, 0.05947467312216759, -0.07286519557237625, -0.07178744673728943, -0.00011520348925841972, -0.005419936031103134, 0.04039522632956505, 0.01107192225754261, 0.13049812614917755, -0.11861162632703781, 0.09722576290369034, 0.12850627303123474, -0.04905809462070465, 0.1503954529762268], [-0.10809554159641266, 0.09479892253875732, -0.0953151136636734, -0.11651577055454254, -0.12735499441623688, -0.16520459949970245, -0.025168683379888535, 0.06045643612742424, 0.02028808183968067, -0.14173999428749084, -0.021640343591570854, 0.0346415638923645, -0.017259545624256134, 0.026879074051976204, 0.05133058875799179, 0.11282972246408463, 0.037041522562503815, -0.04646823927760124, -0.06190478056669235, -0.08606237918138504, 0.06165016070008278, 0.04832277074456215, 0.13520395755767822, -0.03219090774655342, -0.07119835913181305, -0.036483511328697205, -0.04985117167234421, -0.09174217283725739, -0.019855845719575882, 0.10401377826929092, 0.05818615481257439, -0.12530416250228882, 0.06319673359394073, 0.030351776629686356, 0.04818400740623474], [0.06820815801620483, -0.11680220812559128, -0.08954489976167679, -0.1536369025707245, -0.015621124766767025, 0.130060076713562, 0.0963825061917305, -0.12264801561832428, 0.1364452987909317, -0.016336502507328987, -0.14072827994823456, 0.005366435740143061, -0.06182289496064186, -0.07811293005943298, -0.06775231659412384, 0.13644415140151978, 0.006419344339519739, -0.07867493480443954, -0.06711574643850327, -0.06816482543945312, 0.05371711030602455, -0.0832955464720726, 0.06846588104963303, -0.054969001561403275, -0.08709149807691574, -0.0653558075428009, 0.11999259144067764, 0.03783547505736351, -0.036492541432380676, -0.09561260789632797, -0.08485139161348343, -0.12116971611976624, -0.11277810484170914, -0.0012791770277544856, -0.09448137134313583], [0.09124491363763809, 0.017579609528183937, 0.10795162618160248, -0.02056010439991951, -0.14253674447536469, -0.1471419632434845, -0.09847413003444672, -0.16517935693264008, -0.0956624299287796, -0.10468518733978271, 0.11928320676088333, 0.080610491335392, -0.13452361524105072, -0.108700692653656, -0.08612371981143951, 0.014748115092515945, 0.05407014116644859, 0.020400214940309525, 0.1236959844827652, 0.09027111530303955, -0.022050200030207634, -0.06632529199123383, 0.006989579647779465, 0.06932497769594193, -7.82797797000967e-05, -0.06493650376796722, 0.11556293070316315, 0.15127556025981903, 0.003562947968021035, -0.016679417341947556, -0.009388573467731476, -0.05762043595314026, -0.05851675197482109, -0.08862292021512985, -0.13022835552692413], [0.029236193746328354, -0.013688762672245502, -0.07941966503858566, 0.09261509776115417, -0.05774657800793648, 0.021566445007920265, -0.09557164460420609, 0.005873895715922117, 0.1075592041015625, -0.037523575127124786, -0.019532063975930214, 0.10879508405923843, 0.10141865909099579, -0.04871102049946785, -0.14188240468502045, -0.06463672965765, -0.1141720786690712, -0.08826564252376556, 0.11563601344823837, -0.07562614977359772, 0.17953917384147644, -0.05208926275372505, -0.1298125982284546, -0.12991224229335785, -0.11095574498176575, -0.0498824417591095, 0.018299438059329987, -0.04203318431973457, -0.00023487747239414603, -0.15048840641975403, -0.052582748234272, -0.051449548453092575, -0.007358077447861433, -0.08802806586027145, -0.1375514566898346], [-0.0783466324210167, 0.008543510921299458, -0.05524912104010582, -0.08736951649188995, -0.10670554637908936, 0.0846976786851883, 0.098293237388134, -0.052461717277765274, 0.07173480838537216, 0.13554590940475464, -0.15041309595108032, 0.0835781991481781, 0.11339686810970306, -0.11392804980278015, 0.01426964532583952, 0.11055067926645279, -0.10411307960748672, -0.024715496227145195, -0.08458906412124634, 0.08273006975650787, 0.03871158882975578, -0.07576168328523636, 0.08428432047367096, 0.11598766595125198, -0.10300899296998978, 0.0035872720181941986, 0.039324771612882614, -0.013750104233622551, -0.10534866899251938, -0.06373678147792816, -0.0034056734293699265, 0.027179203927516937, 0.033162955194711685, -0.09318649023771286, 0.0017185549950227141], [-0.11927317082881927, 0.014889664947986603, 0.009657506830990314, -0.12792378664016724, -0.03583807498216629, -0.11785200238227844, -0.10489610582590103, 0.04273322969675064, 0.08663125336170197, 0.021115316078066826, 0.09098101407289505, 0.1659025400876999, -0.09366369247436523, -0.10452108085155487, -0.02390819601714611, -0.10440216958522797, 0.08944381773471832, -0.027976715937256813, -0.02332952246069908, -0.08558334410190582, -0.02131039835512638, 0.08658112585544586, -0.05336226522922516, -0.018503060564398766, 0.06835316121578217, 0.17099536955356598, -0.10633157193660736, -0.004289217293262482, -0.11909565329551697, -0.08915214985609055, -0.05974249169230461, -0.0004852263955399394, -0.08508069813251495, -0.03820475563406944, -0.0828690379858017], [-0.10703834891319275, -0.1504686325788498, 0.0693015530705452, -0.06679610908031464, 0.07293709367513657, 0.12173926830291748, 0.02452174760401249, 0.028451139107346535, -0.09998670220375061, -0.07414334267377853, 0.06461717933416367, -0.10512973368167877, -0.11404518038034439, 0.09900139272212982, 0.17478534579277039, 0.02014799602329731, -0.12314440310001373, -0.01894812099635601, -0.1553487777709961, 0.09788288176059723, -0.12007927894592285, 0.04457928612828255, 0.0596977137029171, -0.10189148038625717, -0.11928613483905792, 0.12384036928415298, 0.07709895819425583, -0.07524462789297104, 0.10482540726661682, -0.02646268531680107, 0.06859314441680908, -0.16994020342826843, 0.02637620083987713, 0.1170116513967514, -0.08046446740627289], [0.10305920243263245, -0.013538241386413574, 0.09790320694446564, 0.02789905294775963, 0.011750989593565464, 0.03668101504445076, -0.12579858303070068, 0.04432930424809456, -0.024527985602617264, -0.13137602806091309, 0.07239443063735962, 0.11924256384372711, 0.14042367041110992, 0.020020540803670883, 0.03555818274617195, 0.005909263622015715, -0.09317299723625183, 0.07565145194530487, 0.030417850241065025, 0.10212604701519012, -0.13538698852062225, 0.05575833097100258, -0.097063809633255, -0.07702530920505524, 0.07383152842521667, -0.028062669560313225, -0.1193646639585495, 0.08909142762422562, 0.1465306282043457, 0.05548875406384468, 0.025670483708381653, -0.0932970941066742, 0.06605243682861328, 0.0917452797293663, -0.13277162611484528], [-0.12079163640737534, 0.09483346343040466, -0.09323866665363312, 0.08940917253494263, -0.1090271845459938, -0.10262088477611542, -0.015133662149310112, 0.12972351908683777, 0.05596165731549263, 0.04036177694797516, -0.13699689507484436, -0.01813419908285141, -0.07648476958274841, 0.02294193208217621, -0.08337298035621643, 0.07362903654575348, 0.06214416027069092, 0.03253518044948578, 0.07645212858915329, -0.13439372181892395, 0.13829617202281952, 0.1027226373553276, -0.12373583018779755, -0.1419522762298584, -0.03316428139805794, -0.002002925844863057, -0.08834128081798553, -0.06587395071983337, 0.10893487185239792, 0.051615700125694275, 0.017368996515870094, -0.148375004529953, 0.11013972014188766, -0.04474128410220146, 0.004240780603140593], [-0.02661384455859661, 0.10299722105264664, 0.03408512473106384, 0.16058556735515594, -0.01045731920748949, -0.09882214665412903, 0.006081902887672186, 0.0372270792722702, -0.007255835458636284, 0.058342792093753815, -0.04926910623908043, -0.026441963389515877, -0.183674618601799, -0.0013574439799413085, 0.02758624777197838, -0.08612813800573349, 0.042355991899967194, 0.05400758236646652, -0.030480975285172462, -0.11012329161167145, -0.00912928394973278, 0.027588699012994766, -0.1060061976313591, -0.05101088061928749, -0.07962814718484879, -0.06355664879083633, -0.07497277855873108, -0.10416856408119202, 0.02805987559258938, -0.04434392973780632, 0.05063629522919655, 0.10590865463018417, 0.011137844994664192, 0.056344352662563324, 0.03067363053560257], [0.06238337606191635, 0.07090485095977783, -0.04251495748758316, 0.02245255373418331, -0.003827831707894802, -0.1484660804271698, 0.09871986508369446, -0.10900609195232391, -0.0912666767835617, 0.03658922761678696, 0.052365098148584366, 0.07027595490217209, -0.06455537676811218, -0.02751750871539116, 0.07429221272468567, 0.17046192288398743, 0.0022502178326249123, 0.08305566012859344, 0.0797577053308487, 0.05717965587973595, 0.12433933466672897, 0.09838074445724487, -0.006899356842041016, -0.039901599287986755, -0.09493213146924973, 0.00468404870480299, -0.05230109021067619, 0.016803141683340073, -0.10398443043231964, 0.00408339174464345, 0.08342761546373367, 0.0636829063296318, 0.028800496831536293, 0.1252267211675644, 0.10371766984462738], [-0.004195988178253174, 0.07242230325937271, 0.03538721427321434, 0.11693720519542694, 0.06264986097812653, -0.06712404638528824, -0.02693759836256504, -0.010591966100037098, 0.1388220489025116, -0.044900063425302505, 0.12020829319953918, 0.09642118215560913, -0.04536188021302223, 0.14910729229450226, 0.1494278460741043, 0.09124709665775299, -0.10407380014657974, 0.018093951046466827, 0.07368942350149155, 0.09164067357778549, 0.06026553362607956, -0.02115788497030735, 0.09282746911048889, 0.08066736161708832, -0.12612490355968475, 0.04288148507475853, -0.09442659467458725, 0.024420443922281265, 0.07347657531499863, 0.039557021111249924, 0.003445810405537486, 0.05618097260594368, 0.0034430224914103746, -0.1604149043560028, -0.04576745256781578], [0.08954378217458725, 0.08634237200021744, -0.044385526329278946, -0.011491579934954643, -0.00482928054407239, -0.13377216458320618, 0.07460935413837433, 0.005588681437075138, -0.015761643648147583, 0.03846602886915207, -0.04722744598984718, -0.026882776990532875, -0.07979561388492584, -0.06379532068967819, -0.06979629397392273, 0.025631599128246307, 0.02451302669942379, 0.060180313885211945, 0.013113920576870441, -0.07964877039194107, 0.013494216836988926, -0.007054629735648632, 0.064995676279068, 0.03190628066658974, -0.07438834011554718, 0.030619094148278236, -0.001371040241792798, 0.0758112221956253, -0.00947888195514679, 0.04645252227783203, -0.09613009542226791, 0.053567469120025635, -0.05602939799427986, -0.08701609075069427, -0.11478409916162491], [0.0036602262407541275, 0.09046534448862076, -0.056624285876750946, -0.09682504832744598, -0.09319796413183212, -0.05581367015838623, 0.005610884167253971, -0.014940189197659492, -0.06920120865106583, 0.08048264682292938, 0.004664496053010225, -0.11699356883764267, -0.09366580098867416, -0.1012178286910057, 0.05109420791268349, 0.06933782994747162, -0.07195141166448593, 0.10448193550109863, -0.14589519798755646, 0.10774223506450653, 0.0929475799202919, -0.06810597330331802, 0.12342578172683716, 0.12029147893190384, 0.025379076600074768, -0.10697803646326065, 0.1617700606584549, -0.05393114313483238, -0.1250208169221878, 0.0780918151140213, 0.13821011781692505, 0.06839269399642944, 0.08516711741685867, 0.03513799235224724, 0.01750580407679081], [0.02828085795044899, -0.07784366607666016, -0.07739336043596268, -0.12686710059642792, -0.09720072895288467, -0.009409412741661072, -0.09049451351165771, -0.08264030516147614, -0.015588468872010708, -0.11041009426116943, -0.06732076406478882, 0.10964595526456833, -0.03820379450917244, 0.08202804625034332, 0.005594132002443075, 0.12329065054655075, 0.12681333720684052, -0.05191561579704285, -0.03873756155371666, 0.11106105893850327, -0.0703086256980896, 0.09834981709718704, -0.007777105085551739, 0.009443637914955616, -0.1389947235584259, 0.07976635545492172, -0.005105073098093271, -0.08732185512781143, 0.09004580974578857, -0.13644349575042725, -0.07422817498445511, -0.10195766389369965, 0.10085500031709671, -0.1265438348054886, 0.022672098129987717], [0.11018658429384232, 0.11399409919977188, 0.03207395598292351, 0.14264751970767975, 0.08522391319274902, -0.03231396526098251, -0.10341569036245346, -0.03576832264661789, 0.009000678546726704, 0.0653233528137207, -0.12880182266235352, 0.06616894155740738, -0.05426209419965744, -0.07215604931116104, -0.10086966305971146, 0.09560301154851913, 0.08111736178398132, -0.07930479943752289, -0.11126653850078583, 0.1156310960650444, 0.10949275642633438, 0.030394410714507103, 0.024691348895430565, -0.0010285857133567333, 0.03901286423206329, 0.07778451591730118, 0.18699294328689575, -0.06222797930240631, -0.1293565183877945, 0.08960144966840744, -0.0674777626991272, -0.009088756516575813, -0.14635993540287018, -0.11713318526744843, -0.020445527508854866], [-0.06345884501934052, 0.11335877329111099, 0.05465025082230568, -0.028374958783388138, 0.06525582075119019, 0.01266823522746563, 0.043060559779405594, -0.03924964368343353, -0.012653307057917118, -0.10584207624197006, 0.06899850070476532, 0.08360464125871658, -0.0015284037217497826, 0.09865336865186691, 0.06072486937046051, -0.08202587068080902, 0.10901331156492233, 0.03823496773838997, 0.07910403609275818, 0.049301594495773315, 0.03172849491238594, -0.07590172439813614, 0.007187349256128073, 0.042347252368927, 0.06432066112756729, -0.12226484715938568, -0.12978558242321014, 0.00673668272793293, -0.0631520226597786, -0.17906475067138672, -0.0675237625837326, 0.1453610360622406, -0.14050167798995972, 0.1469912976026535, 0.007068936247378588], [0.0023202812299132347, -0.056000418961048126, 0.14981651306152344, 0.10497558116912842, -0.058172088116407394, -0.08976635336875916, 0.0667896419763565, -0.044055771082639694, 0.04045524820685387, 0.06408289819955826, 0.08261839300394058, -0.11531968414783478, -0.04852684214711189, -0.009382803924381733, 0.003109482815489173, 0.1766863316297531, -0.12605494260787964, -0.04496127367019653, 0.11677577346563339, 0.031157998368144035, 0.04552086815237999, 0.00842655636370182, 0.032664332538843155, 0.03286600485444069, -0.050035782158374786, 0.14799845218658447, 0.01811770349740982, 0.02753610722720623, -0.18985652923583984, -0.1158079206943512, 0.06706391274929047, -0.1104976013302803, 0.03456971049308777, 0.1083899512887001, -0.04465660825371742], [-0.13406197726726532, -0.12460459768772125, -0.1626872718334198, -0.11156665533781052, 0.0962940976023674, -0.06893378496170044, 0.1266278177499771, -0.08258725702762604, 0.10085725784301758, -0.07995185256004333, 0.10249931365251541, -0.05945669487118721, 0.005567918531596661, -0.03797818347811699, -0.05865248292684555, 0.09467292577028275, -0.07606329023838043, -0.07392710447311401, -0.042538322508335114, 0.11301134526729584, -0.07805758714675903, -0.09773016721010208, 0.12470458447933197, 0.0037402180023491383, -0.02447308786213398, -0.09566082805395126, 0.16892650723457336, 0.067092664539814, -0.16303013265132904, -0.06286357343196869, -0.1410365253686905, -0.07270550727844238, 0.05699577182531357, -0.09038124978542328, 0.09351072460412979], [0.019058305770158768, -0.07724253088235855, 0.10815484076738358, -0.13999196887016296, 0.10815968364477158, -0.055702853947877884, 0.10720565170049667, -0.019639844074845314, -0.056552592664957047, 0.04895457625389099, 0.04777878150343895, 0.08652922511100769, -0.01132133137434721, -0.04179639369249344, -0.01216106303036213, -0.01140646729618311, 0.1780424416065216, -0.07201472669839859, -0.052839942276477814, -0.1245054081082344, 0.10857118666172028, -0.07037513703107834, 0.0064475322142243385, -0.09396708011627197, -0.019373470917344093, 0.10289715975522995, -0.11112495511770248, -0.015209311619400978, 0.024113014340400696, -0.092928946018219, -0.15884581208229065, 0.007097555324435234, 0.008997110649943352, 0.19054685533046722, -0.07102659344673157], [0.05167289450764656, -0.029411625117063522, -0.06243342161178589, 0.03654031082987785, 0.07834161072969437, 0.15398651361465454, -0.04217889532446861, -0.149118110537529, -0.06161918118596077, -0.0021791968028992414, -0.09896942228078842, -0.023300359025597572, -0.02691134437918663, -0.014427630230784416, 0.07706756889820099, -0.1067831814289093, -0.029727820307016373, 0.12609975039958954, -0.11691217869520187, -0.05280734598636627, 0.008731652051210403, 0.04538885876536369, -0.015595728531479836, 0.10098423063755035, 0.07586502283811569, -0.05041493847966194, 0.001091893413104117, 0.01097668707370758, 0.10203798860311508, 0.013133100233972073, -0.12979567050933838, -0.07853848487138748, 0.08656120300292969, 0.0338544026017189, 0.11954737454652786], [0.0009568493696860969, -0.05472476780414581, -0.11901705712080002, 0.05459548905491829, -0.11554508656263351, -0.06079050153493881, -0.0960751548409462, -0.022648820653557777, 0.04595975950360298, -0.009928802028298378, -0.14760805666446686, 0.15135006606578827, 0.06192704290151596, 0.1332494467496872, 0.12294454872608185, 0.10191294550895691, -0.012881693430244923, 0.09323883056640625, -0.029152950271964073, 0.1531091034412384, 0.12266504019498825, -0.11914731562137604, -0.06457007676362991, 0.08723779022693634, 0.10583759099245071, 0.02506345883011818, 0.1276749074459076, -0.09820292145013809, -0.09650906175374985, -0.10016702115535736, -0.062365688383579254, -0.13808299601078033, 0.1316860318183899, -0.05195952206850052, 0.009781141765415668], [0.08919117599725723, 0.10729856789112091, 0.11886709928512573, 0.017169171944260597, -0.1456386297941208, 0.09861483424901962, -0.004239101428538561, 0.04795686900615692, -0.06707151979207993, -0.045685164630413055, 0.09226708859205246, -0.08761519193649292, 0.09551103413105011, -0.1068040058016777, -0.09913303703069687, -0.07127707451581955, 0.057834476232528687, -0.0014605981996282935, -0.10255084931850433, -0.07395458966493607, 0.02789285033941269, -0.026635654270648956, -0.041771430522203445, -0.13753224909305573, -0.027527714148163795, -0.04071887210011482, 0.01919865608215332, -0.056719958782196045, -0.002615058096125722, 0.09212290495634079, 0.11000586301088333, -0.16347897052764893, -0.0527259036898613, 0.1056065782904625, -0.10958823561668396], [-0.04771645739674568, 0.07183610647916794, 0.05785878747701645, -0.10621726512908936, -0.10275514423847198, 0.05851682275533676, 0.059304118156433105, 0.05117865651845932, 0.040293365716934204, -0.03558768704533577, 0.013158943504095078, 0.04779589921236038, 0.10543894022703171, -0.10919653624296188, -0.1405974179506302, 0.07895238697528839, 0.09735583513975143, 0.05570825934410095, -0.04212618246674538, 0.07220757752656937, 0.09258691966533661, -0.1077272817492485, 0.1072365865111351, -0.1188158392906189, -0.04661605507135391, -0.05084143206477165, 0.05139632523059845, -0.02254457212984562, -0.003264679340645671, 0.047712936997413635, -0.10455497354269028, 0.038209930062294006, 0.12075634300708771, 0.04610327258706093, -0.08820990473031998], [0.07061618566513062, -0.05391398072242737, 0.13963790237903595, -0.06412581354379654, 0.08467524498701096, 0.11223194748163223, 0.038398146629333496, -0.16595305502414703, -0.06121038645505905, 0.12338919937610626, -0.013845611363649368, 0.11966601759195328, 0.030658861622214317, -0.0763307735323906, 0.02414221689105034, -0.05123305320739746, -0.09239162504673004, -0.018956992775201797, -0.07056868821382523, 0.10171826928853989, 0.19053411483764648, 0.022530559450387955, 0.0886959359049797, -0.1308601051568985, -0.07745181769132614, 0.10408920794725418, 0.11922947317361832, 0.039065901190042496, 0.04863986000418663, 0.07641874253749847, 0.042966462671756744, 0.017355365678668022, 0.01606614515185356, -0.03351375460624695, -0.1535635143518448], [-0.08909577131271362, 0.0934368148446083, 0.12160220742225647, 0.04084055870771408, 0.13334304094314575, -0.02984490431845188, -0.12448639422655106, -0.01045467983931303, 0.0959344357252121, 0.08879746496677399, -0.04191574826836586, 0.04742885380983353, -0.062271714210510254, 0.009315875358879566, 0.10597304999828339, 0.04220720753073692, 0.06864501535892487, 0.024610038846731186, -0.10431239753961563, -0.03451981395483017, -0.0674491599202156, 0.05176926404237747, 0.16558825969696045, -0.03558354452252388, 0.06441380828619003, -0.15088453888893127, -0.061274778097867966, -0.06507251411676407, -0.044006478041410446, 0.030033133924007416, -0.11009353399276733, -0.03889502212405205, 0.0736953467130661, 0.08298735320568085, -0.07490001618862152], [-0.09507621079683304, -0.08248530328273773, 0.1048094630241394, 0.11371974647045135, -0.06049936264753342, 0.02257978357374668, -0.05761019140481949, -0.1392623782157898, 0.0317491851747036, -0.13312086462974548, -0.0795261561870575, 0.003252884838730097, 0.029395444318652153, -0.11161928623914719, -0.04023706912994385, -0.0650009885430336, -0.06288393586874008, 0.01887139119207859, -0.00023817387409508228, -0.08818981796503067, 0.020384838804602623, 0.05960703641176224, -0.006833091843873262, -0.051064614206552505, -0.08441629260778427, 0.038972657173871994, 0.006277950014919043, -0.03997814282774925, -0.04686950892210007, -0.07906961441040039, -0.09990252554416656, 0.12480717152357101, 0.0024415927473455667, 0.04807247221469879, -0.08417585492134094], [0.04761015623807907, -0.017265912145376205, 0.03458798676729202, 0.034100353717803955, -0.03853631764650345, -0.08477018028497696, -0.04454085975885391, 0.03766587749123573, 0.03954262286424637, 0.11658460646867752, 0.029600145295262337, -0.028147704899311066, -0.04698984697461128, -0.039348505437374115, -0.10863702744245529, 0.06359706073999405, -0.023882035166025162, 0.04933982342481613, 0.06787150353193283, -0.11659073084592819, 0.11188630014657974, 0.07984098792076111, -0.11173959821462631, 0.12024042010307312, 0.11372368037700653, -0.11255911737680435, 0.04093395918607712, 0.10626406222581863, 0.08214637637138367, 0.027959350496530533, 0.031446848064661026, 0.1541702300310135, -0.13619858026504517, -0.04471061751246452, 0.031087862327694893], [0.009890259243547916, -0.0908748060464859, -0.07522386312484741, -0.083433598279953, 0.08371134847402573, 0.0856923907995224, 0.023274466395378113, -0.009035068564116955, -0.11721553653478622, -0.050867509096860886, -0.1414387971162796, 0.02351311780512333, -0.04243510216474533, 0.08673430979251862, 0.1323445737361908, -0.10192497819662094, 0.05560524761676788, 0.08177416771650314, -0.0839487612247467, -0.11670877039432526, 0.13118478655815125, -0.12252184748649597, 0.09444166719913483, 0.027392225340008736, 0.0314638689160347, -0.1684691607952118, 0.07620099931955338, -0.1071128398180008, -0.12125136703252792, 0.14518272876739502, -0.07623559236526489, -0.06395123153924942, -0.16784918308258057, 0.03935008496046066, -0.08169865608215332], [-0.020967356860637665, 0.040190570056438446, 0.05815330520272255, -0.08157731592655182, -0.04477892443537712, 0.032497674226760864, 0.014917724765837193, -0.02758410945534706, 0.01607379876077175, 0.028629548847675323, -0.020096158608794212, -0.0694974884390831, -0.006929348688572645, -0.04188074544072151, -0.08558625727891922, 0.135199174284935, 0.09467114508152008, 0.02952280081808567, -0.004803206771612167, -0.0684557855129242, 0.04333607852458954, -0.03524889051914215, -0.004143405240029097, -0.06907659024000168, -0.017834700644016266, -0.08464385569095612, 0.049661606550216675, 0.07418730109930038, 0.08723171055316925, -0.02521040476858616, 0.08514988422393799, 0.049522433429956436, 0.03282856196165085, -0.093620665371418, -0.014541557990014553], [-0.10416486859321594, -0.07483135908842087, -0.0034756704699248075, 0.09999778121709824, -0.07996463775634766, -0.051626574248075485, -0.00508483313024044, -0.06153005734086037, -0.15004876255989075, 0.12585961818695068, -0.11922375857830048, -0.11011765897274017, -0.10070867091417313, -0.08018772304058075, 0.0673028901219368, -0.0199268851429224, -0.06975317001342773, -0.03429167717695236, -0.12200557440519333, -0.08498380333185196, 0.03358852118253708, -0.050044503062963486, -0.1134275570511818, -0.0805155485868454, 0.06128152087330818, 0.07675116509199142, -0.1310984045267105, 0.13845865428447723, 0.06441354006528854, -0.028361504897475243, 0.010957328602671623, 0.00884008314460516, -0.12145091593265533, -0.008352730423212051, 0.04867996275424957], [0.0519903339445591, -0.04190255329012871, -0.06059110164642334, 0.12223948538303375, 0.10910067707300186, -0.06069730594754219, -0.1151982992887497, 0.11239105463027954, -0.05301891639828682, -0.0400787815451622, -0.043634042143821716, 0.03873462602496147, 0.013411186635494232, 0.14055243134498596, 0.06363823264837265, -0.10307390987873077, -0.11835859715938568, -0.03186799958348274, 0.073545902967453, 0.08918339014053345, 0.1242586150765419, 0.047527119517326355, -0.03715815767645836, 0.03220686689019203, 0.12986622750759125, 0.10519307106733322, -0.1689550280570984, -0.03142240643501282, 0.11333382874727249, -0.13780541718006134, -0.1176638975739479, 0.17932827770709991, -0.04454904794692993, 0.10984876751899719, 0.006848536431789398], [0.06517817080020905, -0.0812663584947586, -0.07372686266899109, -0.06542402505874634, 0.09205834567546844, 0.050418999046087265, -0.08918707072734833, 0.06203857436776161, 0.0682038813829422, -0.05568293109536171, 0.03387874364852905, -0.024839013814926147, 0.009875801391899586, 0.040787968784570694, 0.11350956559181213, 0.11742593348026276, -0.0716320276260376, -0.013988692313432693, 0.076556496322155, 0.04267240688204765, 0.0006302068941295147, -0.06989669054746628, 0.07079103589057922, 0.12717384099960327, 0.047172799706459045, 0.11119235306978226, -0.016562925651669502, -0.006425143219530582, -0.012401056475937366, -0.048348065465688705, 0.05668821558356285, -0.0259590744972229, -0.03664441034197807, -0.0032872920855879784, 0.15802441537380219], [-0.03042433224618435, -0.023512255400419235, 0.10340841114521027, 0.06635946780443192, -0.061878178268671036, 0.09657785296440125, 0.08325788378715515, 0.15812985599040985, -0.03986763581633568, 0.04467020183801651, -0.0993233397603035, 0.04344383627176285, -0.03726794198155403, 0.06627430021762848, 0.09088586270809174, -0.12405690550804138, -0.12251685559749603, -0.079477958381176, 0.05498442053794861, -0.008314097300171852, 0.062467217445373535, -0.029961297288537025, -0.046927254647016525, 0.0764516144990921, -0.1311885267496109, 0.14997778832912445, -0.07673656940460205, -0.011041512712836266, -0.10194948315620422, 0.07119537889957428, -0.013469931669533253, -0.062285393476486206, 0.09457535296678543, 0.04378984868526459, 0.037618882954120636], [0.039088789373636246, 0.128116175532341, 0.06904511898756027, 0.10414186865091324, 0.11828028410673141, 0.0024953139945864677, 0.08923812210559845, -0.0033014649525284767, 0.0703316256403923, 0.09497600048780441, -0.004265851341187954, -0.11125745624303818, 0.002165529178455472, -0.11231060326099396, -0.1280643343925476, -0.04383556544780731, 0.07774001359939575, -0.06172945350408554, 0.022517375648021698, -0.034884728491306305, -0.0271537397056818, -0.0008632497629150748, -0.11834196001291275, 0.05704042688012123, 0.09097351133823395, 0.12294051051139832, -0.0013711856445297599, -0.06430920958518982, -0.006347768474370241, -0.16886554658412933, 0.03377016633749008, 0.0771777331829071, 0.019816599786281586, 0.00896564219146967, 0.06162815913558006], [-0.06534949690103531, -0.022337283939123154, -0.05688508227467537, -0.05994963273406029, -0.05351117625832558, 0.015256810933351517, 0.04747505486011505, 0.08281107246875763, 0.09510424733161926, -0.004252559971064329, -0.03237205743789673, -0.09521541744470596, -0.08951164782047272, -0.08492147922515869, -0.033433303236961365, -0.0049622259102761745, -0.07404099404811859, 0.027044029906392097, 0.04609385505318642, 0.00013339488941710442, -0.08513647317886353, 0.05575854331254959, -0.07058773189783096, -0.13606280088424683, -0.16625487804412842, 0.002813013968989253, -0.12471432238817215, -0.10662403702735901, -0.06357363611459732, 0.1246090903878212, 0.1142284944653511, -0.0033442473504692316, 0.12171129137277603, -0.08747420459985733, -0.11893696337938309], [-0.025406567379832268, 0.11619774997234344, -0.08935762196779251, 0.0010793555993586779, 0.041806116700172424, -0.07923019677400589, -0.05802711844444275, 0.008417666889727116, 0.09915440529584885, 0.10877226293087006, -0.001533322618342936, -0.12874680757522583, 0.1497313380241394, -0.009056991897523403, 0.060754239559173584, -0.014666649512946606, -0.07328429818153381, 0.002307936316356063, 0.09972980618476868, 0.10395118594169617, 0.010292603634297848, -0.07332908362150192, -0.07698631286621094, 0.06763090193271637, 0.016684943810105324, 0.12066175788640976, 0.1285867840051651, 0.12075623124837875, 0.010780131444334984, -0.11768901348114014, 0.03170967474579811, 0.026272984221577644, -0.08239497989416122, -0.050108663737773895, -0.1071195900440216], [-0.04963507130742073, 0.14215776324272156, -0.05828303471207619, 0.11903867870569229, 0.10221105813980103, 0.024093128740787506, 0.012056950479745865, 0.033933594822883606, 0.05417804792523384, -0.03160732984542847, 0.10575704276561737, -0.11212985217571259, -0.0906575620174408, -0.03100532293319702, 0.07243730127811432, -0.1916881948709488, 0.043584033846855164, -0.08177419006824493, 0.011965477839112282, -0.03012089617550373, 0.12588724493980408, -0.06974807381629944, -0.09326760470867157, -0.0853039026260376, -0.0020379137713462114, -0.08352364599704742, -0.1267051249742508, 0.0723479762673378, 0.08089780807495117, -0.014052055776119232, 0.06245410069823265, 0.003280563745647669, -0.024133600294589996, 0.11130916327238083, -0.0766533687710762], [-0.09316614270210266, 0.08681207150220871, -0.028684768825769424, 0.044097352772951126, 0.11926578730344772, 0.03456195071339607, 0.07188447564840317, -0.13049976527690887, 0.07688528299331665, 0.10353845357894897, -0.02846645936369896, -0.06717424839735031, -0.02522306516766548, 0.08884511142969131, -0.07238157838582993, 0.08795355260372162, 0.03805339336395264, 0.07353710383176804, -0.09703174233436584, 0.10824564099311829, 0.1006651297211647, 0.033100441098213196, -0.08350008726119995, 0.14047513902187347, 0.13255998492240906, -0.10536585003137589, 0.11486770212650299, 0.08460374921560287, 0.043377432972192764, -0.1145072877407074, 0.10197293758392334, 0.17058072984218597, -0.13711906969547272, -0.021075015887618065, -0.0023452506866306067], [0.10238132625818253, -0.04705841839313507, -0.07066508382558823, 0.10622646659612656, 0.005012304987758398, -0.09875233471393585, -0.02806275710463524, -0.024073563516139984, 0.07728877663612366, -0.0049814824014902115, -0.11288897693157196, 0.031096050515770912, 0.07640769332647324, 0.03909136354923248, -0.024012621492147446, -0.0361899770796299, -0.03268991410732269, -0.07898466289043427, 0.022615952417254448, 0.08251933753490448, -0.11093844473361969, 0.011799684725701809, 0.06254507601261139, -0.13388250768184662, 0.0628170594573021, 0.04734399914741516, -0.07304982841014862, -0.052301689982414246, 0.07202456891536713, 0.0509217269718647, -0.0767846554517746, -0.13291674852371216, 0.053585559129714966, 0.1365666687488556, -0.11295884102582932], [-0.00393944326788187, 0.05184880271553993, -0.08159701526165009, 0.046344511210918427, 0.020502163097262383, 0.12927190959453583, 0.015403570607304573, -0.034657564014196396, 0.08562944829463959, -0.06846833229064941, -0.08570684492588043, -0.08252547681331635, -0.14445073902606964, -0.10069838166236877, -0.009132509119808674, -0.06138617917895317, 0.01045672595500946, 0.07841870188713074, 0.09791433066129684, -0.03376638889312744, 0.02736906334757805, -0.07071536034345627, 0.06049292907118797, -0.02242867276072502, -0.03664105758070946, -0.11636082082986832, -0.009186241775751114, 0.09266677498817444, -0.05371828004717827, 0.0851263701915741, -0.10752512514591217, -0.04058389365673065, -0.03652425855398178, 0.028090042993426323, 0.07831010967493057], [-0.0003962400951422751, 0.1351626068353653, -0.10412893444299698, -0.0007559608784504235, -0.062092650681734085, -0.028191961348056793, -0.05664458125829697, 0.07912217825651169, -0.06114208325743675, -0.06540670990943909, 0.07777519524097443, -0.0540410578250885, 0.0706566795706749, 0.04829643666744232, -0.048430223017930984, 0.05511019006371498, 0.13464610278606415, 0.15537229180335999, -0.1656832993030548, -0.05294400453567505, -0.10615599155426025, -0.1808060258626938, -0.061931680887937546, 0.07478442788124084, 0.060080546885728836, -0.08953186124563217, -0.10609163343906403, -0.014142570085823536, 0.01747596263885498, 0.008557092398405075, -0.1918802559375763, 0.11579477041959763, -0.09038510173559189, -0.1028435081243515, -0.06156018003821373], [0.014147666282951832, 0.1477641463279724, -0.09688917547464371, 0.008674842305481434, -0.03200255706906319, 0.11212149262428284, 0.10523933172225952, 0.02063486911356449, 0.05429363623261452, -0.008601821959018707, -0.08496423065662384, 0.17698538303375244, 0.006169498432427645, 0.09096316993236542, -0.1404515504837036, -0.11144930869340897, -0.13132275640964508, -0.08157116919755936, -0.04598249867558479, -0.028294168412685394, 0.1328679621219635, -0.037767842411994934, 0.07786258310079575, 0.020999344065785408, -0.02154194749891758, -0.01152876764535904, 0.12367799133062363, -0.11803345382213593, -0.024756887927651405, 0.10214956849813461, 0.01640610583126545, 0.055050622671842575, -0.0361051969230175, 0.1278657168149948, 0.11493200808763504], [-0.1371108591556549, 0.013769607059657574, -0.010667367838323116, -0.034032244235277176, 0.13747604191303253, 0.033086396753787994, -0.11929716914892197, 0.004323347005993128, -0.022078057751059532, 0.038016416132450104, -0.03095022216439247, -0.048261161893606186, 0.12948362529277802, 0.03875911980867386, -0.08298927545547485, -0.08725234866142273, 0.020806914195418358, -0.10962806642055511, 0.05482998862862587, -0.05873303487896919, 0.01899554207921028, 0.06525596976280212, 0.08161360770463943, 0.007330138236284256, -0.03156902268528938, 0.06328283995389938, 0.010306053794920444, -0.0886925607919693, -0.034060101956129074, 0.04115999862551689, -0.04115539789199829, -0.01922866888344288, 0.05959024652838707, 0.009384767152369022, -0.023276347666978836], [0.030729763209819794, -0.03988706320524216, 0.1266234815120697, 0.0258168987929821, -0.10779473930597305, 0.15413248538970947, 0.051023777574300766, 0.015904195606708527, -0.0359041653573513, 0.01541761215776205, 0.035877764225006104, 0.06226851046085358, 0.0008557637338526547, 0.035192009061574936, 0.13752871751785278, 0.004812161438167095, -0.0044878097251057625, -0.03335423767566681, 0.035495858639478683, -0.10366875678300858, -0.1520034372806549, -0.0009327346342615783, 0.077835313975811, -0.07558820396661758, -0.002277981722727418, 0.12434711307287216, 0.06715928763151169, -0.12454412132501602, 0.07569606602191925, -0.04032351076602936, 0.1802617907524109, -0.20490242540836334, 0.02775328978896141, 0.17065636813640594, 0.12831279635429382], [0.15248173475265503, -0.036745019257068634, -0.14767305552959442, 0.09969311952590942, 0.12689116597175598, -0.02630811743438244, -0.08765845000743866, 0.0316573791205883, -0.030641332268714905, -0.11201562732458115, 0.10910580307245255, 0.09928231686353683, 0.156971275806427, 0.027546264231204987, 0.08061233907938004, -0.09193551540374756, 0.020586440339684486, 0.10107770562171936, -0.13202857971191406, -0.07444771379232407, -0.0014996333047747612, 0.040118083357810974, 0.04608028382062912, -0.1164245530962944, 0.09666895866394043, 0.09296920895576477, 0.07843823730945587, -0.05102141946554184, 0.16383276879787445, 0.04633992165327072, 0.0009225590038113296, 0.04118577763438225, -0.11606867611408234, 0.013062287122011185, 0.026152458041906357], [0.0011151392245665193, -0.15204013884067535, 0.152141273021698, -0.09494487196207047, 0.12806116044521332, -0.09932604432106018, 0.011770794168114662, 0.05329292640089989, -0.08477097004652023, 0.009488645009696484, 0.014159904792904854, 0.11291928589344025, -0.08623300492763519, -0.07667111605405807, 0.04921181499958038, -0.04318374767899513, 0.13308295607566833, -0.07759334146976471, -0.01482432708144188, 0.032572854310274124, 0.06336133182048798, 0.060149453580379486, 0.06166549399495125, -0.005513744428753853, -0.025182973593473434, 0.07666856795549393, 0.08611347526311874, -0.026492517441511154, 0.10666938871145248, 0.09627319872379303, -0.09170185774564743, -0.12163429707288742, 0.13916939496994019, 0.1512754261493683, 0.09445510059595108], [-0.037473514676094055, 0.07000239938497543, 0.08115901798009872, -0.02431919239461422, 0.08873797953128815, -0.10981682687997818, -0.032869286835193634, 0.03406345099210739, -0.11847569793462753, -0.023283913731575012, 0.0913318395614624, -0.019418178126215935, -0.06132461875677109, -0.09589267522096634, -0.0032068348955363035, 0.0939878299832344, 0.08612551540136337, -0.054173544049263, -0.08809036761522293, 0.042239513248205185, -0.06548457592725754, -0.05593159794807434, -0.0978488028049469, -0.12446477264165878, -0.0880768746137619, -0.001956499647349119, 0.13204534351825714, 0.09570998698472977, -0.008673960343003273, 0.03988925367593765, 0.04877452924847603, 0.05780057609081268, -0.10850875824689865, 0.0785483717918396, -0.006671446841210127], [-0.01933486945927143, -0.10096687823534012, 0.030198976397514343, -0.11090020835399628, -0.06505805253982544, -0.015261264517903328, 0.12815535068511963, -0.1055690124630928, -0.09043054282665253, 0.010054003447294235, -0.04058685526251793, -0.12272647023200989, -0.049313873052597046, 0.09164030104875565, 0.047090787440538406, -0.01248233113437891, 0.10355750471353531, 0.06490844488143921, -0.07705167680978775, -0.08208008110523224, 0.04910667985677719, 0.02550431713461876, 0.13406753540039062, 0.024767324328422546, 0.002022923668846488, -0.03351795673370361, -0.09531253576278687, -0.03417668119072914, -0.030711401253938675, -0.09152323752641678, -0.009151493199169636, -0.12450184673070908, 0.08748462796211243, 0.021610843017697334, -0.10374448448419571], [-0.052486781030893326, 0.03367225080728531, 0.1843681037425995, 0.061831481754779816, 0.12797479331493378, -0.0863649845123291, 0.004401871003210545, -0.1280895322561264, -0.014629547484219074, 0.03481481596827507, 0.1814662516117096, -0.037338219583034515, -0.03496306389570236, 0.16892677545547485, -0.06279289722442627, 0.022344930097460747, -0.1383601874113083, 0.007251248694956303, 0.05040609464049339, -0.12952686846256256, 0.10251037031412125, -0.051960933953523636, -0.01717318966984749, 0.10216245800256729, -0.045611824840307236, -0.004152611363679171, -0.020439820364117622, -0.054433148354291916, 0.1031799241900444, 0.10022400319576263, -0.006518647540360689, 0.13289806246757507, 0.061148419976234436, -0.058767952024936676, 0.014173709787428379], [0.14197836816310883, -0.0022232832852751017, 0.05780061334371567, 0.02369261533021927, -0.1269623339176178, -0.082658551633358, -0.05064953491091728, 0.14278103411197662, -0.08751367032527924, -0.11929178982973099, 0.071520134806633, 0.031076839193701744, 0.06584932655096054, 0.055502839386463165, 0.04655954986810684, 0.07935710996389389, -0.07937946915626526, 0.04426326975226402, 0.04611262306571007, -0.023231465369462967, 0.14791060984134674, 0.1366010159254074, 0.0881960541009903, -0.033345043659210205, 0.041589610278606415, -0.011273683980107307, 0.10670069605112076, -0.06827930361032486, 0.10954930633306503, -0.08956263214349747, -0.114748515188694, 0.1051146537065506, 0.12432144582271576, 0.0031931032426655293, -0.13387857377529144], [-0.07872556895017624, -0.08553009480237961, 0.09183090925216675, -0.030017312616109848, 0.06956560909748077, -0.022641798481345177, 0.11390822380781174, -0.003488878021016717, -0.0743476077914238, -0.027566978707909584, -0.03913775086402893, 0.004889993462711573, 0.15148401260375977, -0.025864647701382637, 0.05074264854192734, 0.0646820217370987, 0.08658447116613388, -0.09184593707323074, 0.04270291328430176, 0.03789729252457619, -0.09155823290348053, -0.16903425753116608, 0.08551096171140671, 0.13342472910881042, -0.12651163339614868, 0.21780842542648315, 0.13942454755306244, -0.07599571347236633, -0.10239601880311966, 0.08767080307006836, 0.06650730222463608, -0.13370399177074432, -0.0067727756686508656, 0.11853048205375671, -0.037148818373680115], [-0.11400891095399857, 0.028430556878447533, -0.00016313297965098172, 0.11780636757612228, -0.028671499341726303, 0.09519101679325104, -0.077886201441288, 0.12570618093013763, 0.18906934559345245, 0.01920166239142418, 0.020213549956679344, -0.08215771615505219, 0.048124998807907104, -0.01658494956791401, -0.09703251719474792, -0.06545930355787277, -0.060439299792051315, -0.05403461679816246, 0.1094062477350235, 0.040907345712184906, 0.17761336266994476, 0.06593502312898636, 0.07651831954717636, -0.025846637785434723, -0.06815949082374573, 0.12819711863994598, -0.14489038288593292, -0.0706264078617096, -0.09412412345409393, -0.03200225159525871, -0.06975498050451279, 0.07233662903308868, -0.05999220162630081, -0.004723129794001579, -0.08670935034751892], [0.03131522983312607, 0.039513714611530304, -0.0011911536566913128, -0.0738789439201355, 0.054686304181814194, -0.007099957671016455, -0.04589872062206268, -0.02649569883942604, -0.00872746855020523, 0.03964566811919212, -0.019443051889538765, -0.06863700598478317, 0.035251323133707047, 0.08143341541290283, 0.10785267502069473, 0.003184122499078512, -0.07978653162717819, -0.08444475382566452, 0.06994854658842087, -0.05020340159535408, -0.08082642406225204, -0.019860712811350822, -0.025233276188373566, -0.02023514360189438, -0.11203613132238388, -0.12721344828605652, 0.09460020065307617, 0.09461213648319244, 0.05394073948264122, 0.028210612013936043, -0.056530024856328964, -0.008203581906855106, 0.10067436099052429, -0.038458507508039474, 0.006241695024073124], [0.051795653998851776, 0.01890512928366661, 0.09098036587238312, -0.11440057307481766, 0.03988157957792282, 0.04988322779536247, 0.04255906119942665, -0.06526091694831848, -0.053881850093603134, -0.1381414234638214, 0.07496242970228195, 0.14210332930088043, 0.09133967012166977, 0.022824842482805252, 0.025524768978357315, -0.0451975092291832, -0.09194720536470413, -0.06692609190940857, -0.05711859464645386, -0.04191932454705238, 0.1022171676158905, 0.08370080590248108, -0.12620173394680023, 0.07278952747583389, -0.13966181874275208, 0.11896001547574997, 0.10830286145210266, 0.03389587253332138, -0.067820243537426, -0.141580730676651, 0.10320676863193512, -0.04420243576169014, 0.07727693021297455, 0.004518402740359306, 0.06990132480859756], [0.07425674051046371, -0.0492398664355278, 0.10619647055864334, 0.069125235080719, 0.09986128658056259, -0.05916791781783104, 0.09194552898406982, -0.04799265041947365, 0.05350394546985626, -0.03817687928676605, -0.022934067994356155, 0.08210854977369308, 0.019780870527029037, 0.002443271689116955, 0.0904911682009697, 0.01678999699652195, -0.14879444241523743, 0.02821563556790352, 0.10448628664016724, -0.12692606449127197, 0.008288131095468998, -0.0005191219388507307, 0.12124191969633102, -0.1219923198223114, 0.0006837052060291171, 0.09750448912382126, 0.09862148016691208, 0.03388508781790733, 0.06300117075443268, 0.08833420276641846, -0.09280484914779663, -0.11748894304037094, 0.01594875380396843, 0.00857805646955967, 0.04927443712949753], [0.05771106854081154, -0.05551713705062866, -0.09134315699338913, -0.11716862767934799, 0.0017751952400431037, -0.06802207231521606, 0.016257325187325478, -0.13660679757595062, -0.03888791799545288, -0.013461070135235786, 0.028983591124415398, 0.010328739881515503, 0.06533432006835938, 0.02423545904457569, -0.0785527378320694, -0.054514944553375244, -0.05772585794329643, 0.08087291568517685, 0.025039786472916603, 0.04320405051112175, -0.10062163323163986, -0.001243359292857349, 0.0010251738131046295, 0.11564937978982925, -0.04571327194571495, -0.032587308436632156, 0.13513906300067902, 0.0569821372628212, -0.017842203378677368, -0.034713875502347946, 0.014055450446903706, -0.01601998880505562, 0.021394936367869377, 0.06716880947351456, -0.02556353062391281], [-0.08932805806398392, 0.07893840968608856, -0.0036057503893971443, 0.09788840264081955, -0.14826233685016632, 0.14712996780872345, -0.016987457871437073, -0.05812430381774902, 0.009373319335281849, 0.01741805113852024, -0.10312535613775253, -0.07009903341531754, -0.04261051490902901, 0.03048483282327652, -0.022613100707530975, -0.1292191445827484, 0.01134625542908907, -0.049954649060964584, 0.034118663519620895, 0.08333440870046616, 0.0024659307673573494, -0.0596524178981781, -0.05565127357840538, 0.030640466138720512, 0.14573337137699127, 0.0716736912727356, -0.12472749501466751, 0.008819064125418663, -0.15002720057964325, -0.14873452484607697, 0.14464536309242249, 0.10498310625553131, -0.08889592438936234, -0.030582968145608902, 0.04726194962859154], [0.07309520989656448, 0.092231884598732, -0.022914400324225426, -0.055223941802978516, -0.1325686126947403, -0.06381868571043015, -0.10471072047948837, 0.05021180585026741, 0.08757221698760986, 0.11354093253612518, -0.09450070559978485, -0.09184610843658447, 0.02416391484439373, 0.002623143373057246, -0.1474304050207138, 0.11556605249643326, -0.01571483723819256, -0.019220860674977303, -0.10402004420757294, 0.02745148539543152, 0.07016290724277496, -0.023933999240398407, -0.035631321370601654, 0.05695866793394089, 0.03019530698657036, -0.11867790669202805, -0.040201276540756226, -0.008537886664271355, 0.08581370115280151, -0.1316458284854889, -0.10978153347969055, -0.08626686781644821, 0.13769717514514923, 0.11846526712179184, -0.04976924508810043], [-0.1176576018333435, 0.08562256395816803, 0.10395398736000061, 0.10027161985635757, -0.09929970651865005, 0.17330531775951385, 0.015451104380190372, -0.008285967633128166, -0.04525038227438927, 0.009565913118422031, 0.06756624579429626, 0.08330229669809341, -0.008577731437981129, -0.014955587685108185, -0.028227727860212326, 0.07737433165311813, 0.10706790536642075, 0.17895981669425964, 0.013161109760403633, 0.07112628221511841, -0.14588309824466705, 0.1482062041759491, 0.007287115789949894, 0.06484174728393555, 0.0886080265045166, 0.04961281269788742, -0.1586226373910904, 0.05149523541331291, 0.05500970408320427, 0.06833770871162415, 0.021311761811375618, -0.057819072157144547, -0.012269911356270313, 0.00010093842138303444, -0.04729881137609482], [-0.10421502590179443, 0.04320688545703888, 0.024580230936408043, -0.025171712040901184, 0.01639021560549736, 0.1474812626838684, 0.12645339965820312, -0.13121870160102844, 0.06558864563703537, 0.017817271873354912, 0.12350937724113464, -0.08553727716207504, -0.05612224340438843, -0.03876698017120361, -0.051793791353702545, -0.12569670379161835, 0.05208154022693634, 0.02387183904647827, 0.06643905490636826, 0.11050395667552948, 0.01080357190221548, 0.03566788136959076, -0.07946691662073135, -0.04559427127242088, -0.10439012199640274, -0.15703226625919342, -0.13295531272888184, 0.05437165126204491, -0.04201529547572136, 0.11630558967590332, -0.052931372076272964, 0.12486754357814789, 0.10847432911396027, -0.17281100153923035, -0.008309544995427132], [-0.04257538169622421, -0.11888731271028519, -0.059719011187553406, -0.1284441202878952, 0.011538972146809101, 0.07482185959815979, -0.03437008708715439, 0.04155413433909416, -0.10634226351976395, -0.1418982297182083, 0.11382797360420227, -0.004358275327831507, 0.022579705342650414, -0.08032569289207458, 0.1324162632226944, -0.031359367072582245, 0.08268850296735764, 0.018002407625317574, -0.04598233848810196, -0.006594105623662472, 0.007376502268016338, -0.025501444935798645, 0.007167375646531582, 0.060862258076667786, -0.11778085678815842, -0.058367252349853516, -0.010628774762153625, 0.028923863545060158, -0.053446214646101, 0.14110209047794342, -0.05249673128128052, 0.03226104751229286, -0.1303180456161499, -0.03446397930383682, -0.03246161341667175], [0.12420806288719177, -0.01657291315495968, 0.016493404284119606, -0.12310967594385147, 0.034926846623420715, -0.08986373990774155, 0.03464715927839279, 0.10324761271476746, 0.02226617932319641, 0.10875411331653595, 0.029364952817559242, -0.07935313135385513, -0.0380668081343174, 0.06146802753210068, 0.06647134572267532, 0.14627039432525635, 0.08040131628513336, 0.12124457955360413, 0.06852944195270538, 0.13338284194469452, 0.11200590431690216, 0.0808028131723404, 0.0434468537569046, 0.03993818163871765, 0.03600608557462692, 0.04951779171824455, 0.0767236202955246, 0.03968161717057228, 0.03060888685286045, 0.08736497163772583, 0.039605822414159775, -0.03485911339521408, -0.13180378079414368, -0.006285690702497959, -0.08108870685100555], [0.13442464172840118, -0.045194365084171295, 0.017640626057982445, 0.1409090906381607, 0.012658551335334778, 0.001906012766994536, 0.1333790123462677, -0.07899262011051178, 0.0308306273072958, -0.0586756207048893, 0.13797569274902344, -0.0847979262471199, 0.1425914168357849, -0.03651270642876625, -0.07117237895727158, -0.1310044825077057, -0.11432678252458572, 0.11939027905464172, -0.1595737636089325, 0.014232478104531765, -0.06697528064250946, -0.11836367100477219, 0.08942761272192001, -0.12067347764968872, -0.09471176564693451, -0.10227298736572266, 0.08680044114589691, 0.12169896811246872, 0.0842486247420311, -0.1274164468050003, -0.1147492304444313, 0.0026700214948505163, -0.10966074466705322, -0.04092967510223389, 0.059655942022800446], [-0.10302110016345978, -0.031638190150260925, 0.034999072551727295, 0.013430921360850334, 0.0692092701792717, 0.1291598528623581, 0.10354842990636826, -0.0043959664180874825, 0.11927320808172226, 0.03243599459528923, -0.09765747934579849, 0.02540777064859867, 0.05476953089237213, -0.058995671570301056, -0.06710641086101532, 0.025944238528609276, 0.16774442791938782, 0.014949298463761806, -0.09383080154657364, 0.0640580877661705, -0.08853890746831894, -0.06815347075462341, 0.015024019405245781, -0.041228413581848145, -0.1571376919746399, 0.05678950250148773, -0.023501453921198845, 0.015790749341249466, -0.10324344784021378, -0.05638594180345535, -0.06728741526603699, -0.08992435038089752, -0.01929561421275139, -0.05047272890806198, -0.10181498527526855], [-0.05846480652689934, 0.04657837748527527, -0.02765478752553463, 0.004795888438820839, -0.0035341489128768444, 0.06629527360200882, -0.12990638613700867, -0.0315684974193573, 0.10556598752737045, -0.06980126351118088, -0.027516840025782585, 0.0020280673634260893, -0.0233329888433218, 0.11140870302915573, -0.06154792010784149, 0.12686507403850555, -0.15833933651447296, 0.001257088384591043, 0.07070024311542511, -0.13797861337661743, -0.00870216079056263, -0.10495373606681824, 0.061607301235198975, -0.07132211327552795, -0.11059128493070602, 0.14938239753246307, 0.12963512539863586, 0.10613951086997986, -0.04043802618980408, 0.0010948502458631992, 0.09794960916042328, -0.05978225916624069, -0.05237952247262001, 0.2522614598274231, 0.06199666112661362], [0.00966273806989193, -0.13917303085327148, 0.10177087783813477, -0.10464246571063995, -0.08545532077550888, 0.11804050952196121, -0.015547288581728935, -0.04381055384874344, -0.1080184206366539, 0.04336879029870033, -0.16251057386398315, 0.11889921873807907, 0.07129271328449249, 0.10425474494695663, -0.0886596292257309, 0.07681775093078613, -0.01681022346019745, 0.1187858134508133, -0.006332367658615112, 0.0006984536303207278, -0.10019885748624802, 0.024143638089299202, 0.13061189651489258, -0.08388539403676987, -0.11629687994718552, -0.053091295063495636, -0.13864658772945404, 0.16904030740261078, -0.10398267209529877, 0.050106629729270935, -0.13528910279273987, 0.03369128331542015, 0.051128048449754715, 0.02879314124584198, -0.017821457237005234], [0.07498501241207123, 0.07636480778455734, -0.10466033965349197, -0.06221912056207657, -0.05185727775096893, -0.14502359926700592, -0.08675842732191086, -0.038983650505542755, -0.10304421186447144, 0.029507065191864967, 0.00899280235171318, -0.08433935791254044, 0.027532221749424934, -0.07232818752527237, -0.005565139930695295, 0.025552794337272644, -0.12791989743709564, -0.15332527458667755, 0.14069236814975739, 0.1190590187907219, 0.06625624746084213, 0.08521818369626999, 0.04647191986441612, 0.03403361514210701, -0.08382247388362885, -0.0007501739892177284, -0.10650394856929779, 0.11910046637058258, 0.0019923679064959288, 0.10195595771074295, -0.060643453150987625, -0.0885407030582428, 0.06985146552324295, -0.012877702713012695, 0.009731973521411419], [0.006593475118279457, -0.038359418511390686, 0.12891463935375214, 0.020208360627293587, -0.00517421355471015, -0.002396960277110338, -0.05505569279193878, 0.033530306071043015, 0.02394464798271656, 0.05422760918736458, -0.04793180152773857, -0.13124054670333862, -0.08621642738580704, -0.03092097118496895, -0.0924893170595169, 0.14978614449501038, -0.09038879722356796, 0.14218464493751526, 0.05594877153635025, -0.021956101059913635, -0.009583399631083012, -0.07624100893735886, -0.027415232732892036, -0.16481564939022064, 0.10572519153356552, 0.11128727346658707, 0.147792786359787, 0.019125333055853844, 0.06268564611673355, -0.01497847680002451, 0.07547402381896973, -0.13715185225009918, -0.05195247381925583, -0.02029670774936676, -0.12069424241781235], [0.04031083732843399, -0.08780483901500702, -0.019324293360114098, -0.031203627586364746, -0.049743812531232834, -0.03411995247006416, 0.09230347722768784, -0.0765816941857338, 0.12029874324798584, 0.08822634071111679, -0.026392510160803795, -0.0507960319519043, -0.08390642702579498, 0.12806451320648193, 0.05550789833068848, -0.004167219158262014, 0.14892621338367462, 0.023562923073768616, 0.07818493992090225, -0.11881809681653976, -0.15308961272239685, 0.002432393841445446, -0.03802637755870819, -0.10503619909286499, 0.03481071814894676, 0.110944002866745, -0.12478001415729523, 0.09650829434394836, -0.02477915585041046, 0.12069728970527649, -0.09673745930194855, 0.04590132087469101, 0.05517946183681488, -0.09360863268375397, -0.02909580059349537], [-0.011028946377336979, -0.06838639080524445, -0.10710003972053528, -0.04238217696547508, -0.009742466732859612, -0.02587534673511982, 0.009037117473781109, -0.0967152938246727, -0.12495189160108566, 0.1261197328567505, 0.03437364473938942, 0.07211914658546448, 0.09014351665973663, 0.09080631285905838, 0.07032392919063568, 0.1450873762369156, 0.038475051522254944, -0.09313192963600159, 0.0920347347855568, 0.11056419461965561, 0.15431544184684753, -0.11647751182317734, 0.10296247154474258, -0.007249312475323677, 0.05169529467821121, 0.08347474038600922, 0.018903011456131935, 0.05041130259633064, -0.0036742067895829678, 0.01855333149433136, -0.11804763227701187, 0.05312833562493324, 0.04956327751278877, -0.07370655238628387, -0.006735402625054121], [0.11416005343198776, -0.08763401955366135, -0.16070285439491272, 0.11214554309844971, -0.12332449853420258, -0.07168125361204147, 0.12839275598526, 0.012734797783195972, 0.07778791338205338, -0.040725767612457275, 0.04700532928109169, 0.0818147286772728, 0.18690094351768494, 0.007822159677743912, 0.057259224355220795, 0.052128516137599945, -0.10050079226493835, -0.11327619105577469, 0.04361730441451073, 0.10501861572265625, -0.037560343742370605, 0.06525349617004395, -0.09748061746358871, 0.013944950886070728, -0.07928347587585449, -0.0005488510360009968, -0.03796738013625145, -0.11343885958194733, 0.06600024551153183, 0.10425983369350433, -0.15250350534915924, -0.14627912640571594, -0.049587320536375046, -0.04470134899020195, -0.06326548755168915], [-0.003493937198072672, 0.0681198462843895, 0.025301963090896606, 0.03205195814371109, -0.149911567568779, 0.1872449815273285, -0.06044290214776993, 0.03892688453197479, -0.08774562925100327, -0.017990317195653915, -0.06504984200000763, -0.0692717581987381, 0.14147622883319855, -0.09181256592273712, 0.13366197049617767, 0.02425472065806389, -0.01981266587972641, 0.18696771562099457, 0.08579958975315094, -0.15879739820957184, 0.18124422430992126, 0.013059237040579319, -0.059550732374191284, 0.03390742093324661, 0.07390034198760986, 0.05379393324255943, 0.0316728875041008, 0.03692442923784256, -0.06095241755247116, 0.08617894351482391, -0.11892812699079514, -0.06559541821479797, -0.06137348338961601, -0.0724816769361496, -0.045404594391584396], [0.08258097618818283, -0.08375320583581924, 0.13028673827648163, 0.09915103018283844, -0.05752348527312279, -0.08306816965341568, -0.1778220534324646, -0.1372905671596527, -0.14729325473308563, -0.05379056930541992, -0.041188083589076996, -0.07444017380475998, 0.09869574010372162, 0.12511686980724335, 0.08178208768367767, 0.13799762725830078, 0.032544419169425964, 0.14301206171512604, -0.01732254959642887, -0.03289145231246948, -0.03505336865782738, -0.1300395131111145, 0.17027413845062256, -0.0898788720369339, -0.04842217266559601, -0.054840195924043655, -0.1426054835319519, -0.13733002543449402, 0.09972670674324036, -0.14752750098705292, -0.1334473341703415, 0.1250666379928589, -0.10018564760684967, 0.02908734790980816, 0.12316208332777023], [0.09347288310527802, 0.1041218489408493, -0.09660057723522186, 0.14299213886260986, -0.07627101242542267, 0.08465699851512909, -0.13037392497062683, -0.1724138855934143, -0.14132896065711975, -0.11825971305370331, -0.057769063860177994, -0.01797596551477909, -0.07717244327068329, 0.12106592208147049, -0.01533969771116972, 0.06945505738258362, 0.06319599598646164, -0.020262951031327248, 0.02497146651148796, 0.09282632172107697, -0.04692533239722252, 0.11215507984161377, -0.03033752366900444, 0.04301128163933754, 0.004020350053906441, 0.1291918158531189, -0.05210117623209953, -0.009757659398019314, 0.02737746573984623, -0.14459243416786194, -0.12729708850383759, 0.06482212990522385, 0.15712738037109375, -0.12757745385169983, 0.05080869793891907], [0.028639664873480797, 0.07703183591365814, 0.06909529119729996, -0.04611533135175705, -0.15698759257793427, 0.13831061124801636, 0.13522258400917053, -0.09796183556318283, -0.020255351439118385, 0.08652331680059433, -0.1886248141527176, 0.13429363071918488, 0.01770964451134205, 0.07703257352113724, -0.048725709319114685, -0.08753371983766556, -0.16789621114730835, -0.07835844904184341, 0.06905897706747055, -0.08621207624673843, -0.005266175139695406, 0.13687075674533844, -0.019844958558678627, -0.012231195345520973, 0.1105881854891777, -0.1150343045592308, -0.04738856852054596, -0.026332175359129906, -0.1040002703666687, 0.058591969311237335, 0.0011765214148908854, -0.10734433680772781, 0.057837992906570435, -0.04621262103319168, -0.08435400575399399], [-0.08605021238327026, -0.1244698017835617, 0.06743084639310837, 0.0028990020509809256, -0.13022860884666443, 0.06120920181274414, -0.03526824340224266, 0.016577433794736862, -0.10831812769174576, 0.11848365515470505, 0.08492082357406616, 0.06632836163043976, -0.060557764023542404, 0.12400344014167786, 0.0011294264113530517, -0.07855329662561417, 0.14049240946769714, -0.06106669083237648, -0.08270879089832306, -0.10552219301462173, -0.07424195110797882, 0.11627987027168274, 0.07853436470031738, -0.15186874568462372, -0.10240776091814041, 0.0547977089881897, 0.16365694999694824, -0.06894812732934952, -0.1454562544822693, 0.11248631030321121, 0.08566799759864807, 0.10666970908641815, 0.02538481540977955, -0.16497954726219177, 0.0800006240606308], [0.06286293268203735, 0.03297407552599907, 0.15931253135204315, 0.0294954814016819, 0.148509219288826, 0.05780824273824692, 0.025467127561569214, -0.05263666436076164, 0.0763176828622818, 0.10921090841293335, -0.03387874737381935, 0.08297806978225708, -0.025604380294680595, 0.08089930564165115, -0.038596756756305695, 0.09124191850423813, -0.09101983159780502, 0.01990148425102234, -0.04551025107502937, -0.08422693610191345, -0.13725988566875458, -0.13794218003749847, -0.07603012025356293, 0.06059977412223816, 0.040385350584983826, 0.04018737003207207, 0.12965768575668335, 0.06369554251432419, -0.016187814995646477, 0.09070931375026703, 0.07382889837026596, 0.04475923627614975, -0.11211778223514557, 0.06446975469589233, 0.04899563267827034], [0.059003882110118866, -0.09550217539072037, 0.0952073484659195, -0.028899241238832474, -0.0145041448995471, -0.011130795814096928, -0.0875282883644104, 0.030135691165924072, 0.021679634228348732, 0.020614661276340485, -0.03857555612921715, 0.12327657639980316, 0.055355723947286606, -0.1303170919418335, 0.011847782880067825, 0.10728087276220322, 0.12385384738445282, 0.018536485731601715, 0.04093731939792633, -0.053998321294784546, -0.09942246228456497, -0.05790745094418526, 0.05530847981572151, -0.15953436493873596, -0.1255747377872467, -0.18505170941352844, 0.0030909867491573095, 0.0501493364572525, -0.11981711536645889, 0.07436524331569672, -0.13586847484111786, -0.1235981434583664, -0.00012060024891979992, 0.021977804601192474, 0.011911699548363686], [-0.08322899788618088, 0.07581435143947601, 0.039969418197870255, -0.10212251543998718, 0.0343550443649292, 0.05427206680178642, 0.018165679648518562, -0.08213590085506439, 0.006766620557755232, 0.011750913225114346, 0.025701969861984253, 0.014339586719870567, -0.06889644265174866, 0.02388896606862545, 0.004718356765806675, -0.10866223275661469, -0.0200498104095459, 0.07649645209312439, 0.11739937961101532, 0.016406642273068428, -0.006156779360026121, 0.0027071682270616293, -0.03694889321923256, 0.01835951954126358, 0.09497079253196716, -0.07977568358182907, -0.1370697319507599, -0.04602485150098801, -0.047189339995384216, 0.06327173858880997, -0.08463688939809799, -0.09847281128168106, -0.07345022261142731, -0.0475340336561203, -0.03656187653541565], [0.0597211979329586, 0.09922939538955688, 0.08947447687387466, 0.1447327733039856, 0.005415880586951971, -0.12414959073066711, 0.12678661942481995, -0.12843188643455505, -0.08123868703842163, -0.06640107184648514, -0.12067148834466934, -0.10839012265205383, -0.09436863660812378, 0.053450897336006165, 0.06399822980165482, -0.01280012633651495, 0.04100785776972771, -0.05759354308247566, 0.07197842001914978, -0.11515939235687256, -0.006233661901205778, -0.06771738082170486, -0.052360404282808304, 0.1373344510793686, 0.013240455649793148, 0.02766619436442852, -0.07785137742757797, 0.1020285114645958, 0.1226126179099083, -0.14305725693702698, -0.039669301360845566, -0.05737197399139404, 0.039822500199079514, -0.026407599449157715, -0.08448415994644165], [0.05573808774352074, 0.04306962341070175, 0.07222704589366913, -0.09385788440704346, -0.08264204114675522, 0.10989166796207428, -0.01713409274816513, -0.08308416604995728, 0.10812122374773026, 0.09374100714921951, -0.0925496369600296, 0.13755522668361664, 0.022120261564850807, 0.04710659384727478, -0.014755352400243282, 0.04943222925066948, -0.09383665025234222, -0.030103156343102455, -0.10188303142786026, 0.008689098991453648, 0.09920970350503922, -0.019466331228613853, -0.10847576707601547, -0.07557618618011475, -0.1252383589744568, -0.104520283639431, -0.09589719027280807, 0.01997784711420536, -0.061391204595565796, 0.0941944345831871, -0.0024746847338974476, -0.08118531107902527, -0.02811536192893982, -0.09024566411972046, 0.08003799617290497], [-0.08669488877058029, 0.09931925684213638, 0.025450943037867546, 0.023595182225108147, -0.10441028326749802, 0.022507406771183014, -0.06584737449884415, -0.09447839856147766, -0.06511621177196503, 0.09067320078611374, -0.01765378750860691, -0.04545534774661064, -0.05188067629933357, -0.05600447952747345, 0.04177239164710045, 0.11036241799592972, 0.07028276473283768, 0.004855419043451548, 0.10488086193799973, 0.0916723981499672, 0.0763375386595726, -0.06367221474647522, -0.029946964234113693, 0.01629459671676159, 0.06916867196559906, 0.04479234665632248, 0.062381401658058167, 0.08123283088207245, 0.03670915588736534, 0.08543313294649124, -0.056076984852552414, -0.053540170192718506, 0.11592939496040344, -0.0017161611467599869, -0.019754460081458092], [-0.03743815794587135, -0.03315313160419464, 0.09879329055547714, 0.021247586235404015, 0.007489362265914679, -0.017267530784010887, 0.10301341116428375, -0.015223253518342972, 0.019670803099870682, -0.12044595927000046, -0.11809759587049484, 0.0004115899500902742, -0.10848792642354965, 0.0877627283334732, 0.006993223913013935, -0.10309021919965744, -0.12719474732875824, 0.003214128315448761, 0.10113111138343811, -0.11686857044696808, -0.04121656343340874, 0.04543740674853325, -0.121089406311512, -0.024491257965564728, -0.11973696947097778, -0.005267379805445671, 0.12138774245977402, -0.02295886166393757, 0.07132124900817871, -0.0022050226107239723, 0.06119895353913307, 0.02980910800397396, 0.0420612134039402, 0.18376272916793823, 0.09558314830064774], [-0.04048031568527222, 0.06467496603727341, -0.0899268090724945, -0.11366607248783112, -0.012109599076211452, 0.045029595494270325, 0.039725691080093384, -0.13624687492847443, 0.04008966684341431, -0.1282009333372116, 0.13714410364627838, 0.008387773297727108, 0.11560041457414627, -0.09873265773057938, -0.1089516207575798, -0.10430469363927841, 0.11295650154352188, 0.004725311882793903, 0.022284189239144325, 0.009002440609037876, -0.06056157127022743, 0.048515092581510544, 0.11334257572889328, -0.020270604640245438, -0.07346517592668533, 0.020884407684206963, -0.09470081329345703, -0.03696780651807785, 0.07846095412969589, 0.0021647915709763765, -0.14329883456230164, 0.13451087474822998, 0.057148527354002, -0.013612507842481136, 0.024098878726363182], [-0.10459157824516296, -0.10648930817842484, 0.0033675634767860174, -0.14881163835525513, -0.09357799589633942, -0.11194705963134766, -0.05094815418124199, 0.05367480590939522, -0.06125365197658539, 0.033435698598623276, -0.09669887274503708, 0.02444559894502163, 0.1095079705119133, 0.023593643680214882, -0.14640022814273834, -0.06432291865348816, -0.13492752611637115, 0.02456943318247795, 0.0703560933470726, 0.00046830440987832844, -0.11154769361019135, -0.006672828458249569, -0.02219318225979805, 0.0622924268245697, 0.012143274769186974, -0.0902397409081459, -0.0016005657380446792, 0.012713558971881866, 0.07651153206825256, -0.05514354258775711, -0.030199626460671425, -0.06989166140556335, -0.09781131148338318, -0.08843483775854111, -0.0095539391040802], [0.02102336846292019, 0.03445463255047798, -0.013502405025064945, 0.06895174086093903, -0.02885323017835617, 0.09392949193716049, -0.025126157328486443, -0.05500110611319542, -0.04547826573252678, -0.11845140159130096, -0.08998336642980576, 0.0619828924536705, -0.01671016216278076, 0.12044241279363632, -0.07292746007442474, 0.004305557813495398, 0.09311246871948242, 0.0857425183057785, -0.09265739470720291, -0.10476274788379669, 0.11673568934202194, 0.035635899752378464, 0.062459900975227356, 0.13040749728679657, -0.07473262399435043, 0.028633883222937584, 0.08594975620508194, 0.07074286788702011, -0.11587069183588028, 0.03616315498948097, 0.11148052662611008, 0.07929807901382446, 0.05470801144838333, 0.21227283775806427, 0.003257117001339793], [0.1189705953001976, -0.08701545745134354, -0.03197873756289482, -0.12104099243879318, -0.10781361907720566, -0.10890250653028488, -0.02720825746655464, 0.08530374616384506, -0.050161540508270264, 0.039420727640390396, -0.06564807891845703, -0.05553170293569565, 0.07205478847026825, -0.002431289292871952, 0.06074511259794235, 0.0948750227689743, 0.0766477957367897, -0.16935035586357117, 0.1742384135723114, 0.03307589888572693, 0.13060782849788666, 0.12295085936784744, -0.08369392901659012, -0.1397145688533783, -0.12077109515666962, 0.05182037875056267, -0.09853745996952057, -0.07857109606266022, 0.046416956931352615, -0.04002373665571213, 0.05702243745326996, 0.07506725192070007, 0.0893407016992569, 0.03647064045071602, -0.07063443958759308], [0.0511496402323246, 0.03361697494983673, 0.06413339078426361, -0.156528040766716, -0.03304094076156616, 0.04022398591041565, -0.017187250778079033, 0.03483712300658226, -0.036273274570703506, 0.09712319076061249, -0.13119961321353912, 0.11126051843166351, -0.11377701163291931, -0.057425856590270996, 0.01844327338039875, -0.0584723986685276, 0.06296002864837646, 0.009251764044165611, 0.044507939368486404, 0.023444078862667084, -0.022417830303311348, 0.14326663315296173, 0.012683951295912266, -0.015801189467310905, 0.03807534649968147, 0.06887049227952957, -0.07607610523700714, 0.0552891306579113, -0.06062009558081627, 0.10112351179122925, 0.06892696022987366, -0.16192975640296936, 0.05065559223294258, -0.1274711638689041, -0.04504476115107536], [-0.047535233199596405, -0.06089154630899429, 0.005034840200096369, 0.13294260203838348, 0.06795521825551987, 0.1332857757806778, 0.10324089229106903, 0.06392127275466919, -0.07850386947393417, 0.04647265374660492, 0.004647241439670324, 0.05366923287510872, 0.060219258069992065, -0.05390489846467972, -0.02207515761256218, -0.03698665648698807, -0.09111393988132477, -0.06875894218683243, 0.04959213361144066, -0.0849660262465477, -0.12131156772375107, 0.059988196939229965, 0.089704729616642, 0.11083138734102249, 0.0669938176870346, 0.06585165858268738, 0.1276126503944397, -0.01814805157482624, 0.06106182932853699, 0.07881426811218262, 0.013841252774000168, 0.053160447627305984, -0.03140641003847122, 0.010120933875441551, 0.060364510864019394]], "b1": [-0.06658570468425751, 0.03574546054005623, 0.1215343028306961, 0.11196182668209076, -0.013960177078843117, 0.09833270311355591, -0.13751186430454254, 0.09950956702232361, 0.12774281203746796, 0.03301151096820831, -0.14288856089115143, 0.12425132840871811, 0.007011590991169214, 0.020997755229473114, 0.06803371012210846, -0.014554060064256191, 0.10999401658773422, -0.13565754890441895, 0.036207329481840134, -0.029323291033506393, 0.07483571767807007, -0.05766412615776062, -0.10987676680088043, -0.03288533166050911, -0.014479178935289383, 0.08405107259750366, -0.12379473447799683, 0.13097745180130005, 0.0010660486295819283, 0.04632388427853584, -0.1440703272819519, -0.06607288867235184, -0.007559044286608696, 0.10156825929880142, 0.14662620425224304, 0.03451994061470032, -0.13064485788345337, -0.10649211704730988, -0.03274709731340408, 0.12061088532209396, 0.035899870097637177, 0.06158662214875221, 0.026690388098359108, 0.11455897241830826, 0.07555603981018066, -0.02627433091402054, -0.06784717738628387, -0.09145757555961609, -0.1176232248544693, 0.007648832630366087, -0.11098643392324448, -0.11833611130714417, -0.11264769732952118, -0.11441695690155029, 0.059486065059900284, -0.14163227379322052, 0.09379828721284866, -0.07511148601770401, 0.01165144145488739, 0.1075829565525055, -0.07153329998254776, -0.082099050283432, 0.022834639996290207, 0.04174547642469406, -0.09456926584243774, 0.1533970832824707, 0.18300162255764008, -0.01125798188149929, 0.1348680555820465, 0.01127241924405098, 0.032473381608724594, -0.10082519054412842, -0.0713687315583229, 0.09164834767580032, -0.005025914404541254, -0.06332658231258392, 0.09918607771396637, -0.1466018408536911, 0.0004572382604237646, -0.0859198197722435, -0.10924916714429855, -0.022738687694072723, -0.10030606389045715, 0.12210241705179214, -0.11528447270393372, 0.016774415969848633, 0.05599707365036011, 0.10632792860269547, 0.11921926587820053, -0.09134314209222794, 0.006192397326231003, -0.09431996196508408, -0.09117365628480911, -0.06577004492282867, 0.06757757812738419, -0.03270667418837547], "W2": [[0.007549210451543331, 0.05142971873283386, -0.05626098811626434, -0.013598774559795856, 0.03247443959116936, -0.04967840388417244, -0.007780734449625015, 0.06757982820272446, -0.05154969170689583, -0.032321151345968246, 0.07674036175012589, 0.00482987891882658, 0.00392055232077837, 0.07447835803031921, 0.04886918142437935, 0.08628060668706894, 0.04023304954171181, 0.036278363317251205, -0.0715675801038742, 0.05419924110174179, 0.013089881278574467, 0.021710412576794624, 0.1270877569913864, 0.042156901210546494, 0.06918346136808395, 0.01582189090549946, 0.002536061918362975, 0.06833776086568832, 0.031083887442946434, 0.08929647505283356, 0.06632360070943832, 0.03279953822493553, 0.010858629830181599, 0.06309483200311661, 0.07566407322883606, 0.04724283888936043, 0.041185732930898666, -0.029666217043995857, -0.06451255083084106, 0.033214882016181946, 0.07062512636184692, -0.06582899391651154, 0.07504061609506607, 0.10309293121099472, -0.06339555233716965, 0.03622400760650635, 0.08335776627063751, -0.09227104485034943, 0.043483585119247437, 0.046376291662454605, -0.05426923930644989, 0.05969007685780525, 0.06136944517493248, 0.05741799622774124, -0.03370758891105652, 0.0681450366973877, 0.010490015149116516, -0.017133736982941628, 0.1064736396074295, 0.0009750411845743656, -0.05985004082322121, 0.07880700379610062, -0.014171462506055832, 0.020264778286218643, 0.022502843290567398, -0.02069329097867012, -0.035627707839012146, 0.07323174178600311, -0.015086290426552296, 0.013323131948709488, -0.028629112988710403, 0.004272060934454203, 0.06492049247026443, 0.02361433394253254, 0.01259381789714098, 0.0011208041105419397, -0.06082313880324364, 0.051359329372644424, 0.011484766378998756, 0.06221625208854675, 0.009756425395607948, 0.054194461554288864, 0.04860489070415497, -0.029064280912280083, 0.07363960146903992, 0.010924926958978176, 0.04871548339724541, 0.09908755123615265, -0.027949286624789238, 0.005239947699010372, 0.07092584669589996, -0.02893034555017948, 0.041468165814876556, 0.10521344840526581, 0.027886098250746727, -0.039396319538354874], [0.040357839316129684, 0.043797362595796585, 0.028581300750374794, 0.08118372410535812, 0.04791685566306114, -0.02280663698911667, -0.0751599371433258, -0.018462972715497017, 0.08990422636270523, 0.0789603739976883, -0.038104742765426636, -0.05438641086220741, 0.023244911804795265, -0.030591359362006187, 0.048336055129766464, -0.06509992480278015, 0.054931819438934326, 0.0282280296087265, -0.03339603543281555, -0.06994369626045227, 0.017346380278468132, -0.015870986506342888, -0.01412593200802803, 0.0175243578851223, 0.04082996025681496, -0.05969633162021637, 0.04351715371012688, 0.046734247356653214, -0.0006502483738586307, -0.026366429403424263, 0.0571996234357357, 0.02360018528997898, 0.04314615949988365, -0.04933219403028488, -0.00644657202064991, -0.0704231783747673, -0.05841123312711716, -0.03316526487469673, -0.019029488787055016, -0.06356632709503174, 0.0170539952814579, 0.051078878343105316, -0.00717746838927269, 0.08959198743104935, -0.052260566502809525, 0.011133408173918724, 0.07376540452241898, -0.029832864180207253, 0.08916947990655899, -0.04859503358602524, -0.05491999536752701, 0.06403391808271408, 0.023465441539883614, 0.006565073970705271, -0.05345403403043747, -0.028092702850699425, -0.08737513422966003, 0.07111246138811111, 0.042399197816848755, -0.059362322092056274, 0.0079000573605299, -0.03497157245874405, 0.03885045275092125, -0.025289015844464302, -0.049924422055482864, 0.0028129261918365955, 0.03830226883292198, -0.022135965526103973, 0.03951706737279892, -0.0393320769071579, -0.05687052756547928, 0.014670314267277718, 0.020447863265872, 0.06888687610626221, 0.01775396429002285, -0.03416784480214119, 0.06362991034984589, 0.07638068497180939, 0.024825697764754295, -0.08535492420196533, -0.05283109471201897, 0.07255860418081284, 0.03975753113627434, -0.07644525915384293, -0.05064684897661209, 0.06884770095348358, 0.06400234252214432, 0.11500591039657593, 0.050273340195417404, -0.04771485552191734, -0.037384994328022, 0.0036427476443350315, -0.046528615057468414, 0.0465364009141922, -0.0474451519548893, 0.042777322232723236], [0.06388097256422043, -0.00765661895275116, 0.09319532662630081, 0.060683589428663254, -0.0932212695479393, 0.030527282506227493, -0.07804779708385468, -0.04948153719305992, -0.08584857732057571, 0.004527294542640448, 0.01577041484415531, 0.012950441800057888, 0.0999099537730217, -0.027390627190470695, 0.0708003044128418, -0.018507014960050583, -0.13389772176742554, -0.06989266723394394, -0.02428467944264412, -0.0018888527993112803, -0.008015994913876057, -0.023223306983709335, -0.0281220730394125, -0.03685560077428818, 0.06414920091629028, 0.02904673106968403, 0.016189632937312126, -0.016258247196674347, 0.09571486711502075, 0.09799826145172119, -0.06774599105119705, 0.06333771347999573, -0.03750881925225258, 0.02598932571709156, 0.005703700240701437, 0.0414276160299778, -0.010130210779607296, -0.07270806282758713, -0.03388633206486702, 0.10117679089307785, -0.11049573123455048, 0.048571377992630005, -0.04593433812260628, -0.01331307739019394, -0.038805361837148666, 0.10684140771627426, 0.09572106599807739, -0.0026285923086106777, -0.04613921046257019, -0.01612175442278385, 0.1486465185880661, -0.10878047347068787, 0.10576155036687851, -0.07220776379108429, 0.02072923630475998, 0.01568792387843132, -0.07953772693872452, 0.007794062606990337, -0.06271795183420181, 0.05565623566508293, -0.07007564604282379, -0.04952830821275711, -0.0650930255651474, 0.041722897440195084, 0.04205918312072754, 0.10120481252670288, 0.022179007530212402, -0.046242207288742065, 0.05907959118485451, -0.017136849462985992, 0.03423111513257027, -0.03585745021700859, -0.010223674587905407, -0.08049280941486359, -0.0009734802879393101, 0.0550951287150383, -0.1081688329577446, 0.04996071383357048, -0.0780167430639267, 0.01117605622857809, 0.08277557045221329, 0.07664333283901215, 0.062414638698101044, -0.023765763267874718, 0.11409606039524078, 0.08903073519468307, -0.09721527248620987, -0.05625583976507187, 0.038080137223005295, -0.00639784149825573, 0.04406726360321045, 0.04261092096567154, -0.0746169462800026, 0.013276834040880203, -0.03323108330368996, -0.020256981253623962], [0.020400380715727806, -0.07424420118331909, -0.037181973457336426, -0.04429503530263901, 0.05808865278959274, 0.013889930211007595, 0.0034801990259438753, -0.0902535691857338, 0.042699992656707764, 0.09543881565332413, -0.09350358694791794, -0.08380944281816483, 0.05577657371759415, -0.09477081149816513, 0.02450110763311386, -0.08580252528190613, 0.029658855870366096, -0.11501497775316238, -0.016240715980529785, -0.08227475732564926, 0.04762536287307739, -0.07895690202713013, -0.022169610485434532, -0.05282540246844292, -0.08205457031726837, -0.06750447303056717, 0.040903959423303604, 0.003463981905952096, -0.07800368219614029, 0.07414701581001282, -0.05603046715259552, 0.08395813405513763, 0.008381959050893784, -0.011756490916013718, 0.0035965442657470703, -0.04577307030558586, -0.10014569759368896, 0.07245608419179916, -0.0002902372507378459, -0.024447429925203323, -0.012918748892843723, -0.012344678863883018, 0.08461770415306091, -0.04270489141345024, 0.06491788476705551, -0.035070549696683884, 0.016429448500275612, -0.016028771176934242, -0.006610304117202759, 0.07510566711425781, -0.07565823197364807, 0.005174567922949791, 0.0009784832363948226, 0.0786885917186737, -0.0717744380235672, 0.03531545028090477, -0.03604636341333389, 0.054270707070827484, -0.0029697641730308533, -0.01668485626578331, -0.033346425741910934, 0.05113562196493149, -0.04983774200081825, 0.014918340370059013, 0.04284645617008209, 0.06498536467552185, -0.03785685449838638, 0.031202230602502823, 0.04158208519220352, 0.04533198103308678, -0.03310311213135719, -0.1044427752494812, -0.034570809453725815, -0.022104064002633095, -0.015306288376450539, 0.03732576593756676, 0.08723025023937225, 0.006126156076788902, -0.01871619187295437, 0.030950360000133514, -0.03945823758840561, -0.006699147168546915, 0.005446117836982012, 0.04319993779063225, 0.038109082728624344, 0.07672836631536484, -0.016971493139863014, 0.09368282556533813, -0.04988110065460205, 0.04581131786108017, -0.00848904624581337, -0.052634645253419876, -0.049069371074438095, -0.060666099190711975, -0.07908926904201508, 0.0231220293790102], [0.024340922012925148, -0.016544660553336143, 0.004630777053534985, 0.08573709428310394, 0.07733900099992752, 0.00022070137492846698, -0.04467388987541199, 0.040336236357688904, -0.007708684075623751, -0.07943557947874069, 0.043245669454336166, 0.02677440270781517, 0.07118178904056549, 0.035434868186712265, 0.07182298600673676, -0.023627210408449173, 0.011822217144072056, -0.02047967165708542, 0.0033953587990254164, 0.07853470742702484, -0.08295691013336182, 0.05210806429386139, -0.07787757366895676, 0.10684706270694733, -0.1028401181101799, 0.033870115876197815, -0.012751915492117405, 0.06827278435230255, -0.07234326004981995, 0.00024321681121364236, -0.007847361266613007, 0.045694056898355484, 0.003327252110466361, 0.09252893924713135, -0.04574979841709137, 0.008027715608477592, 0.012331384234130383, -0.008420970290899277, 0.004243854433298111, -0.05619394779205322, 0.05935557559132576, -0.020507199689745903, 0.022778121754527092, 0.03261621668934822, 0.05724291875958443, 0.0645909532904625, -0.0862564742565155, 0.03112826682627201, 0.045224905014038086, 0.07817010581493378, 0.03631925582885742, -0.0560653954744339, 0.05801869556307793, 0.07006686925888062, -0.0031793066300451756, -0.09973198175430298, -0.05073453113436699, -0.04165643826127052, 0.0025818352587521076, -0.007521853316575289, -0.021641768515110016, 0.08857265114784241, 0.01728856936097145, -0.10447665303945541, 0.025979049503803253, -0.0712084248661995, 0.03872932121157646, 0.0006682674284093082, 0.09351928532123566, -0.05122137442231178, -0.0096322912722826, 0.10142144560813904, -0.013396228663623333, 0.07783089578151703, 0.012170965783298016, 0.00888057705014944, -0.03835468366742134, -0.04059143364429474, 0.030381087213754654, 0.01329011470079422, 0.06164691597223282, 0.004114871844649315, -0.0800916999578476, 0.029604461044073105, 0.014236252754926682, -0.013554736040532589, -0.09587156027555466, -0.0279903095215559, -0.08116251230239868, -0.025153085589408875, -0.035178378224372864, -0.05376113951206207, 0.08857084810733795, 0.07232927531003952, 0.07746703922748566, -0.02802092395722866], [0.04423382133245468, 0.06099992245435715, -0.0332418829202652, -0.005148094613105059, -0.015801824629306793, 0.0485389269888401, 0.0705113410949707, -0.009285695850849152, 0.042760543525218964, 0.009643477387726307, 0.05701624974608421, 0.022496560588479042, -0.005271182395517826, -0.014197105541825294, 0.012313070707023144, 0.04870278015732765, 0.005606136284768581, 0.06612008064985275, -0.04429541155695915, -0.007777184713631868, 0.07981929183006287, -0.017441583797335625, 0.013235656544566154, -0.04572603106498718, 0.029364705085754395, 0.09681861847639084, -0.031070342287421227, 0.03746700659394264, -0.03440607711672783, -0.00929652713239193, 0.038225408643484116, -0.03391699120402336, 0.04571930691599846, -0.04176398366689682, 0.0026735705323517323, 0.01304102223366499, 0.04691072180867195, -0.017447644844651222, -0.025430377572774887, 0.02457573264837265, 0.0386933907866478, -0.027867281809449196, 0.029142070561647415, 0.058859605342149734, -0.01737380400300026, -0.00205432903021574, 0.041778720915317535, 0.0038714816328138113, -0.010916405357420444, 0.009103343822062016, -0.02255687117576599, -0.021506700664758682, 0.04561038315296173, -0.02278890088200569, -0.005046735517680645, 0.03674711659550667, -0.06304050981998444, 0.03391794115304947, 0.0247883889824152, 0.00045982145820744336, 0.0068900263868272305, 0.06297781318426132, -0.002796998480334878, 0.06801890581846237, -0.011002577841281891, -0.02211740054190159, -0.03354710713028908, 0.06401684880256653, 0.039249397814273834, -0.042114708572626114, 0.01617238111793995, 0.05459680035710335, -0.01933673769235611, 0.037463024258613586, -0.02583065629005432, -0.022230098024010658, 0.056108664721250534, 0.05131751671433449, -0.05121419206261635, -0.04779847711324692, -0.015366580337285995, -0.0015972061082720757, 0.03129378706216812, -0.01524416171014309, 0.02248370088636875, -0.024516159668564796, -0.01131110917776823, -0.02761254273355007, -0.03314879909157753, -0.007858672179281712, 0.021173512563109398, -0.03776552900671959, -0.024699486792087555, 0.043667372316122055, 0.04593678191304207, -0.0038182542193681], [-0.0853818953037262, 0.0798182338476181, -0.022452302277088165, -0.016531188040971756, -0.10980642586946487, 0.0831058993935585, -0.09763603657484055, -0.039507485926151276, -0.04698088765144348, 0.07745812088251114, -0.08425319939851761, -0.09221082925796509, 0.06428291648626328, -0.04010595381259918, 0.0027122478932142258, 0.03188995644450188, 0.0683506652712822, 0.041950829327106476, -0.03723886236548424, 0.06626375764608383, 0.06199508532881737, -0.048478446900844574, -0.015941251069307327, -0.0020827529951930046, -0.00793479010462761, -0.06382886320352554, 0.012206725776195526, 0.014058238826692104, -0.0844489112496376, -0.03691646084189415, 0.025678757578134537, -0.04256469011306763, 0.025697315111756325, 0.061884284019470215, 0.026692377403378487, -0.013240084983408451, 0.060803983360528946, -0.050168685615062714, 0.024711530655622482, 0.07514675706624985, 0.09420517832040787, -0.006895383354276419, -0.07255084812641144, -0.03956981748342514, 0.08516958355903625, -0.13323435187339783, -0.060759223997592926, 0.0515427365899086, -0.03740972653031349, 0.02773108333349228, -0.02065795473754406, 0.06610575318336487, -0.07112637162208557, -0.08078277111053467, -0.06981569528579712, -0.07857736200094223, 0.0005109812482260168, 0.0029220341239124537, 0.02792949043214321, -0.08373785018920898, -0.02909211441874504, -0.034133438020944595, -0.00226434669457376, -0.02275269478559494, -0.06310562789440155, 0.09108870476484299, -0.003884553210809827, -0.03704025223851204, 0.051294587552547455, 0.09212532639503479, 0.09132842719554901, -0.05752423405647278, -0.0716259703040123, 0.0507492832839489, -0.03539326786994934, -0.06046971306204796, 0.009486495517194271, -0.07101032137870789, 0.03695309907197952, 0.006445029750466347, 0.08298756927251816, -0.0023597199469804764, -0.07011320441961288, 0.06909533590078354, -0.010534767061471939, -0.007507801055908203, -0.1191982850432396, 0.0061081768944859505, 0.08463674038648605, -0.08525768667459488, -0.02053149603307247, -0.02620331011712551, -0.0823645368218422, -0.05096287652850151, -0.10101764649152756, 0.011607818305492401], [0.036705132573843, -0.057586509734392166, -0.06748481839895248, 0.0707571730017662, 0.036845266819000244, -0.023708265274763107, 0.011443842202425003, -0.0736764520406723, -0.024917151778936386, -0.05514221638441086, 0.027462856844067574, -0.014866701327264309, 0.09714511781930923, -0.07743363082408905, -0.06813275068998337, -0.017382537946105003, -0.08735176175832748, 0.11170411109924316, 0.007234711665660143, 0.08138467371463776, -0.0496864952147007, 0.047530464828014374, -0.029275212436914444, -0.012200147844851017, 0.024712197482585907, 0.00355714769102633, 0.04188081622123718, 0.022419705986976624, -0.03989754989743233, -0.011074650101363659, -0.06415843963623047, 0.04309964179992676, 0.01775858737528324, -0.04350567236542702, 0.0789591372013092, 0.03772985562682152, 0.045910872519016266, 0.0900326743721962, 0.07968769967556, -0.055365290492773056, 0.011326314881443977, -0.03892011567950249, -0.07371874898672104, -0.018595315515995026, 0.026324527338147163, 0.0037247901782393456, 0.05094298720359802, -0.053833525627851486, -0.06210503354668617, -0.032432179898023605, -0.030382957309484482, 0.08903486281633377, 0.035545285791158676, 0.03597794845700264, 0.05736333131790161, 0.051244158297777176, 0.07598409801721573, 0.08166300505399704, -0.05828385055065155, -0.008906693197786808, -0.05459952726960182, 0.008558301255106926, 0.03176034986972809, -0.047114405781030655, 0.04582936316728592, -0.09814116358757019, -0.06275083869695663, -0.01786317676305771, 0.02435973472893238, -0.08784806728363037, -0.04617694020271301, 0.03448078781366348, -0.06988900899887085, -0.0039055023808032274, 0.05316536873579025, -0.06770151853561401, 0.005988490767776966, 0.02251567505300045, -0.049916550517082214, 0.05992516875267029, -0.06517240405082703, 0.08573806285858154, 0.006520252674818039, -0.07977872341871262, -0.038900624960660934, -0.023453600704669952, 0.032496072351932526, 0.0006661101360805333, -0.03444408252835274, 0.04826003685593605, -0.06591565161943436, -0.03324003890156746, 0.008305137977004051, 0.04167104884982109, -0.02559819631278515, -0.05946364626288414], [0.06457004696130753, -0.08931877464056015, -0.03607207536697388, 0.0414070226252079, -0.013220197521150112, 0.0525350347161293, -0.0735999345779419, 0.006781501695513725, -0.04836476966738701, -0.0813790112733841, 0.03134393319487572, -0.08716011047363281, 0.07804670929908752, 0.06400007754564285, 0.06455331295728683, -0.06235627457499504, -0.0017910331953316927, -0.11166521906852722, 0.005298280622810125, -0.024609943851828575, -0.022064505144953728, -0.022428138181567192, -0.02182062715291977, 0.08265939354896545, 0.06842990219593048, -0.06415746361017227, 0.049997881054878235, -0.020758364349603653, -0.05878358334302902, -0.055406395345926285, -0.027289096266031265, -0.040399082005023956, 0.06964800506830215, 0.03883996605873108, -0.027399638667702675, 0.059514474123716354, -0.008048344403505325, -0.04779428988695145, 0.05962479114532471, 0.11954116076231003, 0.076607346534729, -0.04437447339296341, 0.003989890683442354, -0.03301621600985527, -0.029159067198634148, 0.050632134079933167, -0.030917080119252205, -0.011795736849308014, -0.05753811076283455, -0.01470456924289465, 0.08544585108757019, -0.038177940994501114, -0.024895623326301575, -0.016839809715747833, -0.015541397966444492, 0.02958790585398674, -0.07745438814163208, 0.07399798929691315, 0.06616838276386261, 0.004011081997305155, 0.10951776802539825, -0.1072845309972763, 0.03910842910408974, 0.026451153680682182, -0.04719677194952965, -0.026174195110797882, 0.08717376738786697, -0.0683821439743042, -0.05645342171192169, 0.08410938829183578, -0.006658356636762619, -0.011651312932372093, 0.09001033753156662, -0.06455393135547638, 0.09292212873697281, 0.04427468776702881, -0.07094781845808029, 0.023962894454598427, 0.07483740150928497, 0.03539086505770683, -0.03867640346288681, -0.07141316682100296, -0.08901353925466537, 0.03763575851917267, 0.028447626158595085, 0.009979094378650188, -0.06143666431307793, -0.03753972053527832, -0.04363589361310005, 0.0071632531471550465, -0.04839746281504631, -0.051763866096735, 0.08969922363758087, -0.08573964238166809, 0.08414559066295624, 0.040370456874370575], [0.10226315259933472, 0.09876307845115662, 0.06668192893266678, 0.07768414914608002, -0.025630785152316093, 0.07358187437057495, 0.04487443342804909, 0.0809929221868515, 0.06142837926745415, -0.011355418711900711, 0.035927146673202515, 0.003071460872888565, 0.03465273603796959, 0.020285706967115402, 0.07906416803598404, 0.03927726671099663, 0.11820954829454422, 0.03639593720436096, -0.0343206450343132, -0.02905149944126606, 0.11931746453046799, 0.013796037063002586, 0.14878898859024048, -0.030345795676112175, -0.0458008348941803, 0.10242195427417755, -0.07991455495357513, 0.06439629197120667, -0.06429965794086456, 0.06103413552045822, 0.0075396285392344, -0.07193177938461304, 0.09894906729459763, -0.05959552899003029, 0.004652810748666525, 0.042972005903720856, -0.04387883096933365, -0.007232386153191328, 0.051217708736658096, -0.03204747289419174, -0.01806589774787426, -0.10194817930459976, 0.06839131563901901, -0.05970754474401474, 0.0007367291254922748, 0.0688137337565422, -0.04015948995947838, -0.002721158554777503, -0.0027991586830466986, -0.04959644749760628, 0.009485851041972637, 0.05142183601856232, 0.000445022335043177, -0.04390590637922287, -0.04070669412612915, 0.05153239145874977, -0.08431761711835861, 0.03337347134947777, 0.003251411020755768, 0.06845337152481079, 0.08411327004432678, -0.07082415372133255, 0.08215207606554031, 0.0803307518362999, -0.0028300811536610126, -0.006155739072710276, -0.11505866795778275, 0.02667883411049843, 0.04819173738360405, -0.06772037595510483, -0.04965969920158386, 0.04275304079055786, -0.025095602497458458, 0.035117797553539276, -0.016821876168251038, -0.06252804398536682, -0.0709918960928917, 0.0855075940489769, -0.009035365656018257, -0.00103976100217551, -0.0032331212423741817, 0.07963881641626358, 0.00194105738773942, -0.005786533001810312, 0.07290306687355042, 0.005394754931330681, 0.05000535771250725, 0.09848836809396744, -0.0852658599615097, -0.06843727082014084, 0.0004360380698926747, 0.10989020764827728, -0.033685848116874695, 0.006924779620021582, 0.023934630677103996, -0.025409020483493805], [-0.0030822879634797573, -0.028505530208349228, -0.06190074607729912, -0.03815549239516258, 0.04331039637327194, -0.010538171976804733, 0.0024118178989738226, -0.02241373062133789, -0.031077228486537933, -0.07617831230163574, -0.011494425125420094, -0.054051510989665985, -0.0315651074051857, -0.08461984246969223, -0.03019232489168644, -0.026135753840208054, 0.06528571248054504, 0.02186502143740654, -0.00128132663667202, 0.07468400150537491, -0.006432702299207449, 0.08040519803762436, -0.022096963599324226, 0.027057575061917305, -0.021046819165349007, -0.05313801392912865, -0.015263160690665245, -0.022211093455553055, -0.02253778465092182, -0.04490402340888977, 0.02036866545677185, -0.04016188904643059, -0.01881004311144352, -0.054088469594717026, 0.005836633034050465, 0.031573377549648285, -0.013660678640007973, -0.07200054824352264, -0.06120285391807556, 0.008011024445295334, -0.026297321543097496, -0.05333920568227768, -0.08768802881240845, -0.02471548318862915, -0.059953171759843826, 0.03356418013572693, -0.01863499917089939, -0.03890761360526085, -0.01131579838693142, 0.006512664724141359, 0.03035527653992176, -0.027131913229823112, -0.021164335310459137, 0.058656658977270126, -0.04916713759303093, -0.03492191806435585, -0.047406960278749466, 0.10893943905830383, -0.08781441301107407, -0.02907881885766983, 0.018000658601522446, 0.02845604158937931, 0.06624901294708252, -0.06413602083921432, -0.015057234093546867, -0.05127756670117378, -0.08669096231460571, 0.026680542156100273, 0.06237826123833656, -0.00437132129445672, -0.022133734077215195, -0.06424589455127716, 0.0046747769229114056, -0.026056908071041107, -0.019064243882894516, 0.04322812706232071, -0.0007399346795864403, -0.04123365506529808, -0.05667532607913017, -0.0429152175784111, 0.04745496064424515, 0.005997870117425919, 0.06731921434402466, -0.0052933646366000175, -0.0007995229680091143, -0.028112098574638367, 0.08551090210676193, -0.06729305535554886, 0.05089448764920235, -0.056405648589134216, -0.07165993750095367, -0.06217857450246811, -0.06262490153312683, -0.05922219529747963, 0.03903025761246681, 0.07304676622152328], [-0.09246478229761124, 0.05333107337355614, 0.04961590841412544, 0.04240814223885536, 0.03136026859283447, -0.02426433563232422, 0.07626143097877502, 0.05889155715703964, 0.03483632951974869, 0.06624901294708252, -0.044814109802246094, 0.030726328492164612, 0.06308558583259583, 0.07540161162614822, 0.007686810102313757, 0.057283591479063034, -0.08958901464939117, -0.006687391083687544, -0.002284522633999586, -0.06775989383459091, 0.06939077377319336, -0.03603053838014603, -0.08739994466304779, 0.010876709595322609, -0.04354602098464966, -0.07255251705646515, 0.00265093008056283, -0.008844250813126564, -0.010895677842199802, -0.06755580753087997, -0.08742698282003403, 0.05151636153459549, 0.0027255811728537083, -0.06612278521060944, 0.06268342584371567, 0.007619780022650957, -0.08357934653759003, 0.03600481525063515, 0.037135276943445206, -0.03131159767508507, -0.006848006974905729, 0.03747003898024559, 0.07085210829973221, -0.0027415372896939516, 0.00956722628325224, 0.09563145041465759, 0.002728348597884178, 0.08318596333265305, -0.025699643418192863, -0.0550210177898407, 0.12411826103925705, -0.017100989818572998, 0.07041439414024353, 0.05340991169214249, 0.020286401733756065, -0.09974130243062973, 0.05087558552622795, 0.08191625028848648, 0.025772858411073685, 0.06185038387775421, -0.07443966716527939, -0.07099270820617676, -0.04121037945151329, 0.004265272058546543, -0.02361631579697132, -0.018652023747563362, 0.05109371244907379, -0.04977627843618393, 0.05578264221549034, 0.037978239357471466, 0.018074313178658485, 0.03169003501534462, 0.04389231279492378, -0.08624562621116638, 0.04043981060385704, -0.054516155272722244, -0.03733500465750694, -0.035926274955272675, -0.0399576872587204, -0.03095608949661255, 0.036624353379011154, -0.07687489688396454, -0.05575161054730415, -0.014023971743881702, -0.01728951558470726, 0.05658480525016785, -0.0027874645311385393, 0.01341228373348713, -0.00429861806333065, 0.08254421502351761, 0.008549383841454983, 0.06567543745040894, -0.02308918908238411, -0.028390947729349136, 0.06755246967077255, -0.004061006475239992], [-0.005084595177322626, 0.050525471568107605, -0.09785956144332886, -0.04230484738945961, -0.03919515013694763, -0.011129911057651043, -0.012056617066264153, 0.05047113820910454, 0.041445739567279816, 0.023327626287937164, 0.006523113697767258, -0.09242574125528336, 0.061726875603199005, -0.05990377441048622, -0.028424447402358055, 0.047695841640233994, -0.0212513729929924, -0.08706722408533096, 0.0761500895023346, 0.021333858370780945, -0.029922161251306534, 0.0029383504297584295, -0.05843283608555794, -0.03308534249663353, -0.00977315939962864, -0.0750221461057663, -0.012193777598440647, 0.06244144216179848, -0.08378397673368454, 0.07708606123924255, -0.10887591540813446, 0.04278085380792618, -0.08286137133836746, 0.007559990044683218, -0.06476432085037231, -0.02926897443830967, 0.03370184451341629, 0.07094185054302216, -0.017150353640317917, 0.0553724505007267, 0.08414289355278015, 0.06468050926923752, 0.0050305332988500595, -0.0597585067152977, 0.002064336324110627, -0.07344970852136612, -0.03050931729376316, 0.08314939588308334, -0.08061057329177856, -0.011427221819758415, -0.05352199450135231, -0.024558618664741516, -0.015026407316327095, -0.03284531459212303, 0.07139807194471359, -0.040859464555978775, 0.0359560027718544, -0.0698930025100708, -0.023226769641041756, 0.07828173041343689, -0.05489018186926842, -0.02058345638215542, 0.04072466492652893, 0.05549566075205803, -0.04058767855167389, 0.06707349419593811, 0.0417662188410759, -0.043061498552560806, -0.061172984540462494, 0.06475307047367096, 0.060488417744636536, 0.035988397896289825, 0.00425745639950037, -0.03807811811566353, 0.008413650095462799, -0.05577217414975166, -0.00896114856004715, -0.01793993078172207, 0.07562720775604248, -0.034763265401124954, -0.003305159043520689, -0.07079968601465225, -0.0826638862490654, 0.016815967857837677, 0.06456975638866425, -0.0497162826359272, -0.041497983038425446, 0.02451779693365097, 0.004848263692110777, -0.031706783920526505, -0.026263220235705376, -0.029472265392541885, 0.05151946470141411, -0.09410569816827774, -0.05509016662836075, 0.07115248590707779], [-0.029791047796607018, -0.045163217931985855, -0.034463074058294296, 0.021404780447483063, -0.017536044120788574, -0.027875414118170738, -0.016943126916885376, 0.0648585632443428, 0.051634594798088074, 0.040168844163417816, 0.0118479048833251, -0.045097433030605316, -0.053571611642837524, 0.029686057940125465, -0.030850937590003014, -0.026834815740585327, -0.019347095862030983, -0.019707517698407173, -0.03408333659172058, -0.02959432452917099, -0.0018568329978734255, -0.03464905172586441, -0.04470807686448097, -0.025530407205224037, -0.06344287842512131, 0.018370987847447395, 0.04120892658829689, 0.03101225197315216, 0.032080356031656265, -0.02153855562210083, -0.03256665915250778, 0.027589095756411552, -0.031901538372039795, 0.05333142727613449, 0.001031801919452846, -0.02184821106493473, 0.030200285837054253, -0.017743373289704323, -0.010652629658579826, 0.011079755611717701, 0.007701965514570475, 0.013600345700979233, 0.04064234718680382, -0.024521104991436005, -0.029970915988087654, 0.019244302064180374, 0.014159198850393295, 0.017909925431013107, -0.04694221913814545, -0.013649629428982735, 0.017801813781261444, -0.05496807396411896, 0.020538605749607086, -0.031036531552672386, -0.030793780460953712, -0.000741418160032481, 0.037990719079971313, -0.025463616475462914, -0.02015373297035694, 0.009569074027240276, -0.050428032875061035, 0.023849718272686005, -0.0699358731508255, 0.01785106211900711, -0.047870784997940063, -0.022778430953621864, -0.029733387753367424, -0.024327052757143974, -0.017929567024111748, -0.003350885584950447, -0.015527279116213322, 0.055467963218688965, 0.041754383593797684, 0.007673506159335375, 0.05399060994386673, -0.008047868497669697, -0.012249130755662918, -0.07793597877025604, -0.010811750777065754, 0.031573306769132614, 0.007082295138388872, -0.009166870266199112, 0.03379214555025101, 0.01103741955012083, -0.01401328295469284, -0.042263954877853394, 0.029977494850754738, 0.06927409023046494, 0.007810805458575487, 0.0035613488871604204, -0.010371760465204716, 0.058888256549835205, 0.006686106324195862, -0.029966911301016808, 0.004256273154169321, 0.017646584659814835], [0.007186796050518751, -0.06440403312444687, 0.03345697373151779, 0.0055924030020833015, -0.0083171920850873, 0.02221676893532276, 0.025103483349084854, -0.018524490296840668, -0.0003479295701254159, -0.06588716059923172, 0.014285922981798649, -0.06955703347921371, -0.048055753111839294, -0.019975921139121056, 0.02490522898733616, -0.00972620490938425, 0.043074432760477066, -0.06509923189878464, 0.010418090038001537, -0.015339137986302376, 0.050262026488780975, -0.0026239268481731415, -0.008208471350371838, -0.015673812478780746, -0.03777820244431496, -0.04387815669178963, -0.0017869072034955025, 0.06294906884431839, -0.04143742099404335, 0.01304980181157589, 0.057006336748600006, -0.020697230473160744, -0.043084029108285904, -0.020335044711828232, 0.03421426936984062, 0.03913213312625885, 0.0017781511414796114, -0.007152148988097906, 0.019543439149856567, -0.024967435747385025, 0.017883367836475372, 0.035585563629865646, -0.058410726487636566, 0.012989518232643604, -0.014433187432587147, -0.0031087796669453382, -0.009087077341973782, 0.0083918496966362, 0.005177342798560858, -0.02445259504020214, -0.04905161261558533, -0.015468393452465534, -0.06407351046800613, -0.025979872792959213, -0.03901343047618866, -0.011725722812116146, 0.06262508034706116, 0.06545636057853699, 0.04644588753581047, 0.024068960919976234, 0.05346544086933136, -0.03943224251270294, 0.025475220754742622, -0.021627793088555336, -0.019781209528446198, -0.007317249197512865, 0.009238500148057938, 0.048779334872961044, -0.009224661625921726, 0.01937466859817505, -0.02015356905758381, 0.020990150049328804, -0.04366838559508324, 0.042045749723911285, -0.06469213962554932, 0.0211962778121233, -0.021828677505254745, -0.03722677007317543, -0.04691429063677788, 0.01537542324513197, -0.04159490019083023, -0.035309527069330215, -0.049625564366579056, 0.06697911024093628, -0.06052882596850395, 0.058105871081352234, 0.01735020987689495, 0.07439851760864258, 0.007389404345303774, 0.07053008675575256, -0.04718303680419922, -0.00112908985465765, -0.00445992685854435, -0.03691645711660385, -0.04731834679841995, -0.0055219558998942375], [0.07102253288030624, 0.05250069871544838, -0.03664005920290947, 0.03800917789340019, 0.03708748146891594, -0.06352752447128296, 0.044188711792230606, 0.08222924172878265, -0.04710211977362633, -0.0724242776632309, -0.0022590027656406164, 0.0985042005777359, -0.006649927236139774, -0.02602268010377884, -0.0022396701388061047, 0.08283562958240509, -0.013698226772248745, 0.09059275686740875, -0.0194876529276371, 0.00324163562618196, 0.022186148911714554, 0.048723265528678894, 0.029867151752114296, -0.033783599734306335, 0.026550211012363434, 0.11850488930940628, 0.007250800728797913, -0.012026888318359852, 0.019511505961418152, -0.05365510284900665, 0.007855422794818878, -0.008941755630075932, -0.014857647940516472, -0.07609853148460388, -0.010978229343891144, -0.030817124992609024, 0.06644072383642197, 0.06064532324671745, -0.01446053758263588, -0.024743562564253807, 0.0684182271361351, -0.0028003149200230837, 0.03394107148051262, -0.023113861680030823, 0.008234389126300812, -0.056965067982673645, -0.022679779678583145, -0.09355676919221878, -0.0485076867043972, 0.024348240345716476, 0.05076028034090996, 0.08450798690319061, -0.04846790060400963, -0.05953621864318848, -0.022032294422388077, 0.043666280806064606, -0.02824699692428112, 0.019621392711997032, 0.06893158704042435, -0.000667969579808414, -0.006941980682313442, -0.031515587121248245, 0.013799023814499378, 0.005307300481945276, -0.003778609447181225, -0.043896786868572235, 0.016088131815195084, 0.07878953963518143, -0.03927064687013626, -0.06644664704799652, -0.009074890986084938, -0.02310887910425663, 0.022299764677882195, 0.021751511842012405, 0.04186832904815674, 0.05286917835474014, 0.04386797919869423, 0.06928922981023788, -0.014125097543001175, -0.0018847635947167873, -0.05800582095980644, 0.012101978994905949, 0.05081213265657425, 0.06577557325363159, 0.058911919593811035, 0.03714045137166977, -0.03595219925045967, 0.01015479862689972, 0.059505291283130646, 0.06019838899374008, 0.045789021998643875, 0.01569676212966442, 0.03823840618133545, 0.05304301157593727, -0.009442921727895737, -0.03534950688481331], [-0.024169528856873512, -0.03151979297399521, -0.05312325805425644, 0.09396714717149734, -0.055216461420059204, 0.024448543787002563, -0.06588631868362427, -0.004680524114519358, -0.06768518686294556, -0.01940898783504963, 0.0069565786980092525, 0.026399262249469757, 0.049256499856710434, -0.05823276937007904, 0.046485986560583115, -0.0058172233402729034, 0.014733717776834965, -0.049651727080345154, 0.03004211187362671, 0.019471894949674606, 0.009719493798911572, -0.02796269953250885, 0.015697741881012917, 0.09061772376298904, -0.02718260884284973, -0.06354415416717529, 0.04997941106557846, -0.06755626201629639, -0.02425030618906021, -0.04685353487730026, -0.030177289620041847, -0.008136569522321224, -0.027291536331176758, 0.037829503417015076, -0.004844533745199442, 0.03991382196545601, -0.023431867361068726, 0.014343257993459702, -0.0472988560795784, 0.04000762104988098, -0.0809495821595192, -0.0025901678018271923, -0.028684159740805626, 0.0040526073426008224, 0.025133177638053894, 0.006932276301085949, -0.020435478538274765, 0.041279882192611694, -0.024521613493561745, 0.04633035510778427, 0.06025361642241478, -0.023369213566184044, 0.015228689648211002, 0.02439090423285961, -0.06205357238650322, -0.0021681066136807203, -0.05255963280797005, 0.08374609798192978, -0.022232363000512123, -0.021963797509670258, 0.04973025992512703, 0.052072905004024506, -0.011922329664230347, -0.03843959420919418, -0.031007517129182816, 0.009342697449028492, -0.030455606058239937, -0.05479337275028229, 0.06448283046483994, -0.0394381619989872, -0.03794441372156143, 0.026948455721139908, 0.03241261467337608, -0.04041234031319618, 0.06465224921703339, 0.04330971837043762, 0.0013601495884358883, -0.018030153587460518, 0.06985224038362503, 0.025570722296833992, 0.004422492813318968, -0.030387291684746742, -0.024834491312503815, 0.04982032626867294, -0.03438650071620941, 0.011814342811703682, -0.003885174635797739, 0.01810072921216488, 0.05883565917611122, -0.04468654841184616, 0.025167647749185562, -0.0028213050682097673, 0.05422704294323921, 0.01829809881746769, -0.007973325438797474, 0.005904488731175661], [-0.03149033337831497, 0.089756079018116, 0.043901219964027405, -0.09462862461805344, -0.07538806647062302, 0.045308880507946014, -0.052170056849718094, 0.03045196831226349, -0.04869982227683067, 0.07164695858955383, -0.04203666374087334, -0.019966846331954002, 0.08991753309965134, -0.05350792407989502, 0.020069384947419167, 0.06357333064079285, 0.011312585324048996, -0.04682745039463043, 0.02434525080025196, 0.07910651713609695, 0.010932362638413906, 0.06742042303085327, 0.07391244918107986, -0.05443659797310829, -0.013781671412289143, 0.013811642304062843, 0.048433490097522736, 0.061718814074993134, 0.01123382244259119, 0.09695327281951904, 0.0345475859940052, -0.05713469162583351, -0.002731908345595002, 0.07555100321769714, 0.056180670857429504, 0.0827716514468193, 0.028567446395754814, 0.10233062505722046, 0.076875239610672, -0.003615681082010269, -0.051832232624292374, -0.005818743724375963, -0.07521459460258484, -0.04218097776174545, 0.09494027495384216, 0.050817083567380905, -0.07014480978250504, -0.12057705968618393, 0.07645885646343231, 0.016350066289305687, 0.007941638119518757, -0.07631760835647583, 0.08265165984630585, 0.0452057346701622, -0.07740391045808792, 0.07842834293842316, -0.031874243170022964, -0.036209896206855774, 0.010284538380801678, 0.03235721215605736, -0.03594694286584854, -0.018382418900728226, -0.028779949992895126, 0.11694107204675674, -0.05817973241209984, -0.04009493067860603, -0.08539208769798279, -0.008663175627589226, 0.009849801659584045, -0.02378588356077671, -0.08295121043920517, 0.047007966786623, -0.013943190686404705, 0.08051219582557678, -0.08492626249790192, -0.0805709958076477, 0.08846595883369446, 0.016200968995690346, 0.00748123973608017, -0.020537609234452248, -0.05226854979991913, 0.056793034076690674, 0.10169970244169235, -0.015019385144114494, 0.08041200786828995, 0.09027117490768433, 0.030917566269636154, -0.021159950643777847, 0.09505876898765564, -0.027202976867556572, -0.08041851222515106, -0.03755548968911171, -0.06263753026723862, 0.09519913047552109, 0.026929669082164764, -0.057012930512428284], [-0.03517678752541542, -0.07584984600543976, -0.017134588211774826, 0.03394206240773201, -0.014305170625448227, -0.07761599123477936, -0.03373638540506363, -0.04666683077812195, -0.0048042964190244675, -0.008162559010088444, -0.02413770742714405, -0.042263224720954895, -0.030321141704916954, -0.04544756934046745, 0.061661794781684875, -0.06556069105863571, 0.05085757002234459, 0.03565269708633423, -0.05515989288687706, 0.06391084939241409, 0.07174420356750488, 0.04899616539478302, -0.02836228348314762, 0.017029404640197754, 0.04295701906085014, -0.002937791869044304, -0.0002643016050569713, 0.09448002278804779, 0.004647327121347189, -0.06095920875668526, -0.04952135682106018, -0.002710771979764104, -0.044629331678152084, 0.005328760482370853, -0.07059898227453232, -0.014337511733174324, -0.0875244140625, 0.006201909389346838, 0.05864598974585533, 0.024951467290520668, -0.06539375334978104, -0.042683668434619904, -0.02125362679362297, 0.02823045291006565, 0.020696774125099182, -0.08942495286464691, -0.019861219450831413, 0.02113090641796589, 0.007351091131567955, 0.007220656145364046, 0.027418101206421852, -0.0346708819270134, 0.06192975491285324, 0.002631180686876178, 0.06614100933074951, -0.005040670279413462, 0.0024278583005070686, 0.09399206936359406, -0.05654756352305412, -0.049399442970752716, 0.04555898904800415, 0.017040252685546875, 0.029312819242477417, 0.020784225314855576, 0.06570404022932053, -0.00036599484155885875, 0.06135668233036995, 0.04538435861468315, -0.04656944051384926, 0.033763572573661804, -0.08614635467529297, -0.01803620159626007, -0.008882502093911171, -0.03663262724876404, 0.05720677226781845, -0.055978771299123764, -0.07371437549591064, 0.011506875976920128, 0.04712244123220444, -0.0018108991207554936, -0.014363391324877739, 0.02035747468471527, 0.029513243585824966, -0.038706328719854355, -0.02201511152088642, 0.043662507086992264, 0.06154294312000275, 0.07633449882268906, -0.061516571789979935, -0.04453765228390694, 0.03255853056907654, 0.03545256704092026, 0.011560125276446342, -0.02168280817568302, -0.08370792120695114, -0.016036314889788628], [-0.04282274469733238, -0.011576918885111809, 0.030010540038347244, -0.07546840608119965, -0.10900001972913742, 0.0661243200302124, -0.02724621258676052, 0.005037101451307535, -0.052880410104990005, 0.019973300397396088, 0.027527056634426117, -0.0342189259827137, -0.0843028798699379, 0.016816075891256332, -0.08280793577432632, -0.04139019921422005, -0.06847298890352249, -0.034070465713739395, 0.05530611053109169, 0.05124083161354065, 0.021354515105485916, -0.05397949367761612, 0.0364302359521389, -0.11870362609624863, -0.07744382321834564, 0.059387050569057465, 0.11814234405755997, -0.038174647837877274, 0.015403726138174534, -0.012989792041480541, -0.07600802928209305, -0.03804603964090347, -0.06024186685681343, -0.06126023083925247, -0.04758705571293831, 0.01828150823712349, -0.06498342007398605, -0.05333379656076431, -0.06352110952138901, -0.05818812549114227, -0.06354521960020065, -0.02503906935453415, 0.08264057338237762, 0.04280198737978935, -0.0769481360912323, 0.047607842832803726, -0.026836136355996132, 0.05557925999164581, -0.07437728345394135, 0.07880672067403793, -0.07624486833810806, 0.07318468391895294, -0.06256569921970367, -0.006109628826379776, 0.040075480937957764, -0.040918055921792984, -0.06591136008501053, -0.15066410601139069, -0.028154145926237106, 0.09832701086997986, -0.08144942671060562, -0.027533244341611862, -0.023860037326812744, -0.020660657435655594, 0.07806970924139023, 0.05249953642487526, 0.0471508726477623, 0.02617284469306469, -0.04607797786593437, 0.016670875251293182, 0.03957512229681015, -0.11951935291290283, 0.06140937656164169, -0.03826601803302765, -0.08436323702335358, -0.03654804453253746, 0.05556045100092888, -0.015447940677404404, 0.07608738541603088, 0.075410395860672, -0.056867003440856934, 0.09923212230205536, 0.08718517422676086, 0.05431529879570007, 0.05094393342733383, -0.026688899844884872, 0.006118087098002434, -0.011917640455067158, 0.07701843976974487, 0.022926034405827522, -0.003846358275040984, 0.0007789700757712126, 0.02740120142698288, -0.08535412698984146, -0.004980676341801882, -0.024790430441498756], [0.025515498593449593, 0.03644798696041107, -0.050352782011032104, -0.030845938250422478, -0.011965091340243816, -0.032767049968242645, 0.02948562428355217, -0.008415128104388714, -0.07407380640506744, 0.054691098630428314, -0.08380661904811859, -0.0826166495680809, -0.042514413595199585, -0.0017604584572836757, -0.02070755325257778, -0.00748715503141284, 0.08018878102302551, -0.06990223377943039, -0.053869642317295074, 0.011857589706778526, 0.05583968386054039, 0.03488287702202797, -0.04463239759206772, 0.016956772655248642, -0.07064303755760193, 0.014790451154112816, 0.08425997197628021, 0.006202371791005135, -0.02836616337299347, -0.024215005338191986, -0.03151595965027809, -0.04957791417837143, 0.07602798193693161, 0.026984620839357376, -0.08125902712345123, 0.07437727600336075, -0.05945901200175285, -0.0380474328994751, -0.001334934844635427, 0.005748546216636896, 0.06268356740474701, 0.01891319453716278, 0.048923250287771225, -0.08155365288257599, -0.0017611195798963308, 0.0301725622266531, -0.04090963304042816, -0.022384410724043846, -0.07204730808734894, -0.05385497584939003, -0.007429054472595453, -0.06376788020133972, -0.10568535327911377, 0.024882057681679726, -0.06700995564460754, -0.025751765817403793, -0.07618850469589233, 0.07153943926095963, -0.023327084258198738, 0.002527154516428709, 0.0250116977840662, 0.07691913843154907, -0.06605567783117294, -0.02867574244737625, 0.05587967485189438, -0.04856255650520325, 0.015604172833263874, 0.03958005830645561, -0.05947783589363098, 0.006367252208292484, 0.05577543005347252, 0.009178982116281986, -0.08770746737718582, -0.003828436601907015, 0.10003798454999924, 0.07491892576217651, -0.02211863361299038, -0.01945442147552967, 0.08418883383274078, 0.07340969145298004, 0.004742463119328022, 0.03219742327928543, -0.09450917690992355, 0.08254525065422058, -0.0030498031992465258, 0.02472686767578125, 0.005457315128296614, -0.057889118790626526, 0.022988473996520042, 0.05150005221366882, -0.07855010032653809, 0.07631305605173111, -0.05438701808452606, -0.05919469892978668, -0.06263253837823868, 0.056770939379930496], [-0.025477631017565727, 0.0024115436244755983, 0.03541902080178261, -0.08391618728637695, -0.018240133300423622, 0.04027938470244408, -0.06877979636192322, 0.04627365246415138, -0.08413387835025787, -0.002072491217404604, -0.107805535197258, 0.014473652467131615, -0.0934610590338707, -0.03226618468761444, -0.020757248625159264, -0.08107266575098038, 0.062054138630628586, 0.05513874068856239, 0.01388065330684185, -0.055590949952602386, 0.08851765841245651, -0.10052657872438431, 0.00255404831841588, -0.04354799538850784, -0.05875561013817787, -0.003004633355885744, 0.054465435445308685, -0.024600200355052948, 0.0814957469701767, 0.09789776057004929, 0.05652548745274544, 0.08044980466365814, -0.036522816866636276, 0.0690872073173523, -0.07228673249483109, 0.002784425625577569, -0.012225205078721046, 0.046349771320819855, 0.06692782789468765, -0.13293282687664032, 0.07837957888841629, -0.031597014516592026, 0.08280057460069656, -0.10992328077554703, 0.10674453526735306, 0.05065300315618515, 0.05955171212553978, 0.09555455297231674, 0.058874767273664474, -0.048060182482004166, -0.1329568773508072, 0.01999894715845585, -0.056810542941093445, 0.009398045018315315, -0.014633355662226677, 0.014663982205092907, 0.04096513241529465, -0.133054718375206, -0.019095472991466522, -0.06207595393061638, -0.06704738736152649, -0.0598442479968071, 0.03944685310125351, -0.046416543424129486, 0.018550025299191475, 0.026229385286569595, -0.051894739270210266, 0.001702622277662158, 0.06043209880590439, 0.01766810193657875, 0.09060605615377426, -0.12961165606975555, 0.10395961254835129, 0.03534803166985512, -0.0010285730240866542, 0.0771699845790863, -0.006145637482404709, 0.0001939158100867644, 0.08162947744131088, 0.0036409066524356604, -0.03318861871957779, 0.03544937074184418, -0.002872196026146412, -0.09077132493257523, 0.10100062936544418, -0.08806399255990982, 0.07620665431022644, -0.017978131771087646, 0.008604718372225761, -0.10483335703611374, -0.02718980237841606, -0.09814196079969406, -0.05036170780658722, 0.04924728348851204, 0.011195328086614609, 0.02391844242811203], [-0.10319754481315613, 0.04912202060222626, 0.052674029022455215, -0.08305297046899796, -0.05569159984588623, -0.06261608749628067, 0.015021709725260735, -0.004994044080376625, 0.07959585636854172, -0.03453850746154785, 0.03603233024477959, -0.08932125568389893, -0.0916271060705185, -0.039792969822883606, 0.0619022399187088, 0.04247160628437996, 0.01047375239431858, 0.056859180331230164, 0.05551699176430702, 0.04253784567117691, -0.0025667177978903055, 0.0026681972667574883, -0.012332146055996418, -0.009376990608870983, -0.04965229332447052, -0.04353639483451843, -0.024045370519161224, 0.027092425152659416, 0.02221936546266079, 0.007961410097777843, 0.08066430687904358, -0.0792803093791008, 0.08373270183801651, 0.014820953831076622, -0.07562021166086197, -0.02080307900905609, 0.06220580264925957, 0.0057512386702001095, 0.058982472866773605, -0.020710334181785583, -0.07076683640480042, 0.05060657858848572, -0.011834335513412952, -0.058172740042209625, -0.037551455199718475, 0.019988905638456345, -0.0027490288484841585, -0.04237427935004234, -0.07680004090070724, -0.0826481357216835, -0.10227877646684647, 0.0458298921585083, -0.014253126457333565, 0.039272136986255646, -0.0034186129923909903, -0.10637684166431427, 0.10129449516534805, -0.0735601857304573, -0.0647520199418068, 0.06064068526029587, 0.07877356559038162, -0.09707791358232498, 0.062418125569820404, -0.10507212579250336, 0.016837747767567635, -0.002610533032566309, -0.03818012773990631, -0.09528160095214844, 0.09315821528434753, 0.01688394695520401, 0.00398583710193634, -0.011338600888848305, -0.02676481381058693, -0.0006164612132124603, 0.028310544788837433, -0.002046052599325776, 0.05781061574816704, -0.03860724717378616, 0.08571012318134308, 0.04368323087692261, 0.08769618719816208, -0.0478963702917099, 0.018158428370952606, -0.0516989603638649, 0.020543979480862617, -0.05551941692829132, 0.012097799219191074, -0.05153067782521248, 0.0633254274725914, -0.09120102971792221, 0.0665898323059082, 0.002455265959724784, 0.03869497776031494, 0.05513686314225197, -0.0031829699873924255, -0.007585242856293917], [0.0607469379901886, 0.0024604348000139, -0.047768350690603256, 0.03600578382611275, -0.02123216912150383, 0.022490888833999634, -0.07828798145055771, 0.024810589849948883, 0.09688471257686615, -0.034497592598199844, 0.0005472935154102743, 0.007365980185568333, 0.10618249326944351, -0.02911180444061756, 0.014898216351866722, 0.10749328881502151, 0.0736616775393486, 0.02858586423099041, -0.12032779306173325, 0.01800120249390602, 0.11044628173112869, 0.08019149303436279, 0.036306675523519516, 0.09689003974199295, 0.007408127188682556, 0.08163934201002121, -0.0028473760467022657, -0.08724556863307953, -0.03412576764822006, -0.03343323990702629, 0.07475093752145767, -0.05964057520031929, 0.11973729729652405, -0.05199355259537697, -0.08253639191389084, -0.02037769928574562, 0.1109347864985466, 0.11941326409578323, -0.03515097498893738, 0.05033845826983452, 0.05002722516655922, 0.04706723988056183, -0.023650750517845154, 0.015394050627946854, -0.09827113896608353, -0.05368442460894585, 0.0361676849424839, 0.03373066335916519, 0.0015234800521284342, 0.005504338536411524, -0.08496725559234619, -0.045556653290987015, -0.0024411745835095644, 0.03903958201408386, -0.06675383448600769, -0.05749768391251564, -0.01172746904194355, -0.04790666699409485, 0.12976714968681335, -0.037460435181856155, -0.053202711045742035, -0.06800166517496109, -0.009849660098552704, 0.046626344323158264, 0.061753761023283005, 0.004008142277598381, 0.06396770477294922, 0.024767383933067322, 0.021267205476760864, 0.02228916995227337, 0.07434511929750443, -0.014622364193201065, 0.07514307647943497, 0.10439028590917587, -0.04746120050549507, -0.02923913300037384, 0.053031112998723984, 0.10888505727052689, 0.003481753636151552, 0.02190028503537178, -0.024575987830758095, 0.10529685020446777, 0.044653814285993576, -0.054974015802145004, -0.04035826772451401, -0.059385549277067184, 0.003326697973534465, 0.03638038411736488, 0.055545881390571594, 0.04103432223200798, -0.09441787749528885, -0.060976505279541016, 0.05029255524277687, -0.01441966276615858, 0.08990584313869476, -0.051532089710235596], [-0.0796508640050888, 0.05016374960541725, -0.05001075193285942, 0.06820164620876312, -0.037875931710004807, -0.10571493953466415, -0.024360325187444687, -0.033292584121227264, 0.047643378376960754, -0.020399682223796844, 0.052207376807928085, -0.0996641218662262, 0.08482721447944641, -0.010294557549059391, -0.04709184542298317, -0.031368356198072433, -0.047973036766052246, -0.10417670756578445, -0.019564678892493248, -0.04206762835383415, -0.0636005848646164, 0.04369783028960228, -0.06864635646343231, 0.07275164127349854, 0.032510314136743546, -0.01762589067220688, 0.03121359646320343, -0.06584674119949341, 0.08412685990333557, -0.06352147459983826, -0.009961364790797234, -0.04631862789392471, -0.008391201496124268, -0.05574355274438858, -0.025425050407648087, 0.026656556874513626, -0.01727919466793537, -0.017683090642094612, -0.009766132570803165, 0.1050516813993454, -0.09758719801902771, 0.08985025435686111, 0.08239264041185379, -0.013235309161245823, 0.04573686048388481, 0.05395316332578659, -0.06515993922948837, 0.023495743051171303, -0.09164537489414215, 0.01160187553614378, 0.0158834271132946, 0.0041918386705219746, 0.049842335283756256, -0.04871336743235588, 0.02732275240123272, 0.00946740061044693, 0.006091769319027662, 0.11092357337474823, -0.04558784142136574, -0.1055067703127861, 0.08224093168973923, 0.058270446956157684, -0.018579808995127678, -0.03289826214313507, 0.01930633932352066, 0.005508081521838903, -0.04453044757246971, -0.07723017036914825, -0.0453500971198082, -0.043791819363832474, 0.04718147963285446, 0.10191610455513, -0.05206240713596344, -0.008582820184528828, 0.08453315496444702, 0.05610016733407974, -0.09516098350286484, -0.026608074083924294, 0.03286440297961235, 0.06220882385969162, -0.02880924567580223, 0.013806127943098545, -0.05589105561375618, 0.016455035656690598, -0.06542223691940308, 0.02494235336780548, -0.05246906727552414, 0.05325046181678772, -0.034093279391527176, 0.03999030217528343, 0.048679448664188385, -0.07299861311912537, 0.036542050540447235, 0.02112129144370556, 0.0983206033706665, -0.03662034869194031], [-0.08531616628170013, 0.0642385482788086, 0.04937811195850372, -0.053195759654045105, 0.05660734325647354, -0.0429251566529274, 0.002387783955782652, 0.0389612540602684, 0.030157018452882767, 0.007444486487656832, -0.042672909796237946, -0.014718902297317982, 0.0007934450986795127, -0.0908486619591713, 0.015249659307301044, -0.04715090990066528, -0.07010477036237717, 0.005473503842949867, -0.07274248450994492, -0.061624377965927124, -0.02510201744735241, 0.034487344324588776, -0.09542553126811981, 0.03662926331162453, 0.022074317559599876, -0.09637701511383057, -0.010293686762452126, -0.027663826942443848, -0.032540321350097656, 0.02002687193453312, -0.01896623522043228, 0.03606677055358887, -0.008082299493253231, 0.05934436619281769, -0.029581474140286446, 0.03491930291056633, 0.011756418272852898, 0.0118961026892066, -0.058051034808158875, -0.0014509375905618072, 0.06798163056373596, -0.02408769354224205, 0.02624674327671528, -0.02251739799976349, 0.07755934447050095, -0.012264871969819069, -0.06612524390220642, 0.008901332505047321, 0.027242757380008698, 0.06603237986564636, -0.0856529101729393, 0.03152969852089882, 0.01979159377515316, -0.017288831993937492, -0.04498622566461563, -0.0791524350643158, 0.08148100972175598, -0.0925728902220726, -0.03426474705338478, 0.03235148265957832, 0.10017183423042297, -0.07286153733730316, -0.0018884125165641308, -0.07959576696157455, 0.043194692581892014, -0.06401717662811279, 0.09360621869564056, -0.003941124305129051, 0.026605311781167984, 0.09306362271308899, -0.04481639713048935, 0.024480322375893593, 0.03825918957591057, -0.04354420304298401, 0.013869411312043667, 0.0590829961001873, 0.0988188162446022, -0.10451909154653549, 0.03195098415017128, -0.03915387764573097, -0.043051425367593765, 0.026974910870194435, -0.08034028857946396, -0.03692805767059326, -0.05766935646533966, -0.02895493619143963, -0.052461739629507065, -0.07446954399347305, 0.06526544690132141, 0.021404491737484932, 0.06605815142393112, -0.010533844120800495, 0.006960473023355007, -0.09794485569000244, -0.039788469672203064, -0.03605553135275841], [0.035227399319410324, 0.009808245114982128, -0.06495583802461624, 0.0885641798377037, 0.0867658257484436, 0.07495010644197464, -0.0046721030957996845, 0.06050480529665947, 0.0091140391305089, 0.09909993410110474, 0.08405698835849762, 0.050136860460042953, -0.059575580060482025, 0.0752854272723198, -0.07332827150821686, 0.07252093404531479, -0.008613815531134605, 0.05542388930916786, 0.03528967872262001, -0.0077515593729913235, -0.06739534437656403, 0.024158669635653496, 0.004694022703915834, -0.08513622730970383, 0.002267597010359168, 0.09954103827476501, 0.01659892126917839, 0.0876556858420372, -0.07165416330099106, -0.030181337147951126, -0.04391305148601532, 0.004565018229186535, 0.03808989003300667, 0.06753645837306976, 0.10105616599321365, -0.02326304465532303, 0.08752630650997162, 0.10688241571187973, -0.07918938994407654, -0.06326398998498917, 0.037460807710886, -0.07167915254831314, -0.04943546652793884, 0.0934150367975235, -0.03770231083035469, -0.00031251300242729485, -0.06949872523546219, 0.03967766836285591, 0.1118965744972229, -0.054543931037187576, -0.0711790919303894, 0.06962384283542633, -0.036704134196043015, 0.0033269342966377735, 0.08001334965229034, 0.10446853190660477, 0.08285930007696152, -0.013079957105219364, 0.09147614985704422, 0.02556396648287773, 0.0682024359703064, 0.02953600324690342, -0.013149051927030087, 0.0642780289053917, -0.02653316594660282, -0.060420695692300797, -0.04968876764178276, -0.06898324191570282, -0.06998363137245178, 0.02019011601805687, -0.022314656525850296, 0.018584517762064934, 0.12381427735090256, 0.030709831044077873, -0.07241161167621613, 0.004707596730440855, -0.08161679655313492, 0.12651598453521729, 0.01813843660056591, -0.03716108947992325, -0.09290638566017151, -0.012630254961550236, -0.07629082351922989, -0.028115946799516678, 0.10110089927911758, -0.051431261003017426, -0.07386504858732224, -0.039462048560380936, -0.0947687178850174, 0.04672883078455925, -0.051786985248327255, 0.015768617391586304, 0.06075216829776764, 0.05626760423183441, -0.014798616990447044, 0.043009039014577866], [0.07713741064071655, 0.058095984160900116, -0.06824638694524765, -0.06683418899774551, -0.06581462919712067, 0.08411484956741333, 0.0856180489063263, -0.0025701457634568214, 0.06703175604343414, 0.024171477183699608, 0.07015644758939743, 0.033368002623319626, 0.11216791719198227, 0.03650307282805443, 0.0989067479968071, -0.02695535309612751, -0.0159381665289402, 0.009553228504955769, 0.027262549847364426, 0.09432370960712433, -0.006622459273785353, 0.05659287050366402, 0.03120935708284378, 0.04364369437098503, -0.0003576355811674148, -0.05464225634932518, -0.0905342549085617, 0.021495303139090538, 0.09641478210687637, -0.05395671725273132, -0.021873056888580322, -0.02144039049744606, -0.06443046778440475, 0.03531331941485405, 0.000981753459200263, 0.025021756067872047, 0.025081172585487366, -0.08119861036539078, 0.028767747804522514, -0.07010243088006973, -0.009817458689212799, 0.03847571089863777, -0.05508517473936081, -0.06402138620615005, 0.078606516122818, -0.05795731022953987, 0.0997077226638794, -0.05403389781713486, 0.01121284905821085, -0.05680190399289131, -0.08823709934949875, 0.08682388067245483, 0.024818895384669304, -0.014865267090499401, 0.08938322961330414, 0.025107691064476967, 0.08529104292392731, -0.0006044694455340505, 0.00906112790107727, -0.04728594422340393, 0.06852234899997711, -0.006845811381936073, 0.03577796742320061, 0.08757683634757996, 0.08309153467416763, -0.061785902827978134, -0.03352431207895279, 0.03288515657186508, 0.03518233448266983, -0.0376933254301548, 0.09632142633199692, 0.08350169658660889, 0.044085849076509476, -0.060427241027355194, 0.08579555153846741, 0.07728191465139389, -0.024243976920843124, 0.04675536975264549, 0.09805862605571747, -0.017940465360879898, 0.006122676655650139, -0.005123034585267305, -0.0252449419349432, -0.023088905960321426, -0.050891563296318054, -0.0173086766153574, -0.029116468504071236, 0.06913009285926819, -0.05715476721525192, -0.02758742868900299, 0.022908074781298637, 0.02572573348879814, 0.04776567965745926, -0.004661437589675188, 0.020106473937630653, -0.02144456095993519], [-0.07907844334840775, -0.1045738086104393, 0.005275246687233448, -0.008432227186858654, 0.05838333070278168, 0.0391707569360733, -0.07981237024068832, -0.054226551204919815, -0.027533672749996185, -0.054143961519002914, 0.06800934672355652, -0.04127452149987221, -0.011862401850521564, 0.04228277876973152, 0.06798316538333893, -0.06859386712312698, 0.0683121308684349, -0.08744175732135773, 0.015529111959040165, 0.039074648171663284, -0.05645662173628807, 0.023679301142692566, 0.014689200557768345, 0.10141917318105698, -0.08498070389032364, 0.01214092317968607, -0.06575816869735718, 0.08461061120033264, 0.05433015897870064, 0.012205720879137516, -0.03222079202532768, -0.018096867948770523, -0.0898013785481453, -0.0072104367427527905, 0.06617938727140427, 0.06314021348953247, -0.0006001180154271424, -0.04778201878070831, 0.08474891632795334, 0.09967322647571564, -0.08591528236865997, 0.013860896229743958, -0.03433116152882576, -0.10957587510347366, 0.05449560284614563, -0.08415671437978745, -0.06018320098519325, 0.09196673333644867, 0.06640416383743286, 0.062498051673173904, 0.0402698777616024, -0.08192555606365204, -0.08174668252468109, 0.028893588110804558, 0.047896500676870346, -0.019612979143857956, 0.02792184054851532, 0.08266715705394745, 0.004817810375243425, -0.004523514304310083, 0.03081263042986393, -0.028752276673913002, 0.025346314534544945, 0.01841181516647339, -0.009111330844461918, 0.0940253809094429, 0.04493667557835579, 0.001098050270229578, 0.08570338785648346, -0.062192484736442566, -0.032396864145994186, 0.07528261095285416, 0.03712032362818718, -0.006014011334627867, -0.038888610899448395, 0.05473018437623978, -0.07767315208911896, -0.002026345580816269, 0.007048311643302441, 0.07574707269668579, -0.07555010914802551, -0.054984383285045624, -0.08518871665000916, 0.011843540705740452, -0.027996502816677094, 0.07369124889373779, 0.019612660631537437, 0.055263154208660126, 0.07510822266340256, -0.03065185435116291, -0.03450398147106171, -0.08132999390363693, -0.014010435901582241, -0.081290602684021, -0.09775254875421524, -0.05907271057367325], [-0.11046633869409561, -0.09636537730693817, 0.072784923017025, 0.019174911081790924, 0.038643524050712585, 0.002678664866834879, 0.08365367352962494, -0.019717618823051453, 0.004833770915865898, -0.01944599486887455, -0.03861447423696518, 0.013945035636425018, -0.014091933146119118, -0.01343224011361599, 0.0789092630147934, 0.049283217638731, 0.0604596845805645, -0.01552852988243103, -0.020155951380729675, -0.015315907076001167, 0.06359855830669403, -0.02996174804866314, 0.019120950251817703, 0.08702217042446136, -0.060848668217659, 0.015509759075939655, 0.028886549174785614, -0.07889025658369064, 0.021380014717578888, 0.09056011587381363, -0.08611118048429489, 0.04886464402079582, 0.08248336613178253, -0.06406558305025101, 0.05565900355577469, 0.023980339989066124, 0.020258305594325066, -0.07935553789138794, -0.026509730145335197, 0.034341420978307724, -0.027499530464410782, -0.0007079239585436881, -0.014348205178976059, -0.08284630626440048, 0.07210671156644821, 0.03631476312875748, 0.03138544782996178, 0.07662200927734375, 0.05143922567367554, 0.032828979194164276, 0.02928290143609047, -0.06327531486749649, -0.03926593437790871, 0.059709448367357254, -0.09222157299518585, -0.10687416791915894, 0.056931253522634506, 0.08106784522533417, -0.001985829556360841, -0.004574900958687067, 0.09770350158214569, 0.04469259828329086, -0.03045930154621601, -0.05328671634197235, -0.09800407290458679, 0.04554912820458412, -0.015428684651851654, -0.03268659859895706, 0.07593050599098206, -0.07236343622207642, -0.055404432117938995, 0.010618804953992367, -0.06524718552827835, -0.08557015657424927, -0.010038619861006737, -0.026425331830978394, 0.07697055488824844, -0.003048695158213377, 0.03541797772049904, 0.00404119910672307, 0.058235734701156616, -0.04254157841205597, 0.06212761253118515, -0.012814655900001526, 0.08367744833230972, -0.07879813015460968, 0.020078249275684357, -0.0439101979136467, 0.019162720069289207, 0.028046902269124985, 0.02791360206902027, 0.03664048761129379, 0.07098434120416641, 0.07870560139417648, 0.012586348690092564, 0.013478446751832962], [0.032148510217666626, -0.029778294265270233, -0.08328678458929062, 0.023074902594089508, 0.03714422509074211, 0.089629627764225, 0.02429410256445408, 0.10196533799171448, 0.03512873500585556, 0.0801866427063942, -0.04494260251522064, 0.06214179843664169, -0.0162498876452446, 0.022203311324119568, 0.02736479975283146, 0.013024485670030117, -0.0658479779958725, -0.026991326361894608, -0.017684146761894226, 0.08130143582820892, 0.09900938719511032, -0.030259082093834877, 0.05250238627195358, -0.007758490741252899, 0.057975031435489655, 0.08309478312730789, 0.06789139658212662, 0.03162142261862755, -0.06617353856563568, 0.052138928323984146, 0.002983183367177844, -0.03213868290185928, 0.06743594259023666, -0.05744834244251251, -0.04677008464932442, 0.045280907303094864, 0.078631192445755, 0.03390885889530182, 0.05204380676150322, 0.002795552369207144, 0.07591373473405838, -0.03583763539791107, 0.09714197367429733, 0.02376004494726658, -0.05938670039176941, -0.06872005015611649, -0.02841201052069664, -0.10193689912557602, 0.09377899765968323, 0.05789374187588692, -0.045868806540966034, -0.06149233505129814, 0.05945022776722908, 0.0710970088839531, 0.026399459689855576, 0.1088649183511734, 0.09194432199001312, -0.014981096610426903, 0.016084568575024605, 0.003058506641536951, 0.09215101599693298, 0.007067004684358835, 0.051045116037130356, 0.054349858313798904, 0.01202072761952877, -0.09621765464544296, -0.0530390627682209, -0.015665533021092415, 0.054369088262319565, -0.07742137461900711, 0.06948304921388626, -0.019324539229273796, 0.0907754898071289, 0.01725422777235508, 0.03717704117298126, 0.03600204363465309, 0.0073949359357357025, 0.07182376086711884, 0.08189139515161514, -0.02362067438662052, -0.05547128617763519, 0.046287547796964645, -0.006508866790682077, -0.005828846246004105, 0.05321783572435379, 0.08269914984703064, 0.09573464840650558, -0.030950644984841347, 0.029239995405077934, -0.01530440803617239, -0.017575964331626892, -0.03250458091497421, -0.012916367501020432, 0.10578569769859314, -0.007736155763268471, -0.004236743785440922], [-0.05855865776538849, -0.025230586528778076, -0.05741056799888611, 0.017146168276667595, -0.05593357980251312, -0.03677806630730629, -0.03637804836034775, 0.009484976530075073, -0.04043400660157204, 0.046287622302770615, 0.046008847653865814, -0.00236512697301805, 0.04355315491557121, 0.009446716867387295, 0.01907537318766117, -0.0417889729142189, -0.013010763563215733, 0.03558962792158127, -0.03747183457016945, 0.0427747406065464, 0.009355971589684486, 0.014223255217075348, -0.0038466532714664936, 0.06572161614894867, 0.020137624815106392, -0.05853806063532829, 0.10388527810573578, -0.03231954574584961, 0.009927098639309406, -0.034897785633802414, -0.024431264027953148, -0.046936046332120895, 0.058840665966272354, -0.017691969871520996, 0.07875365018844604, -0.05250183865427971, 0.016170281916856766, -0.025579944252967834, 0.04680493101477623, 0.043122634291648865, 0.02049579657614231, -0.0315961018204689, -0.06844928115606308, -0.038327038288116455, -0.05523495376110077, -0.03986463323235512, 0.05262620747089386, 0.09373623132705688, 0.015097169205546379, 0.01727023534476757, 0.051110826432704926, -0.04061023145914078, -0.036390271037817, -0.007712990045547485, -0.004950206261128187, -0.033612944185733795, -0.06555522978305817, -0.08218949288129807, 0.016181711107492447, 0.03343365341424942, 0.04462862014770508, -0.03294792026281357, -0.04912292957305908, 0.01811056397855282, -0.021767940372228622, -0.0019947574473917484, 0.0723758116364479, 0.005023317877203226, -0.013188560493290424, 0.010350515134632587, -0.07571296393871307, -0.08253980427980423, -0.05487329140305519, -0.06468410044908524, -0.018237000331282616, 0.036252252757549286, 0.08061372488737106, 0.012879190035164356, -0.02456364966928959, 0.04285861924290657, -0.02099849469959736, -0.04962177947163582, -0.01530615333467722, 0.06815416365861893, -0.04221002012491226, 0.07088768482208252, -0.06961212307214737, -0.048477642238140106, -0.005636231508105993, 0.020242074504494667, 0.0025581137742847204, -0.011272881180047989, 0.006612684112042189, -0.08066494762897491, -0.08199556171894073, 0.03608431667089462]], "b2": [-0.04043254256248474, 0.021326325833797455, -0.058381613343954086, -0.006596135441213846, 0.0877443253993988, -0.06480713188648224, 0.06759996712207794, -0.008789286948740482, 0.06446509808301926, 0.032997261732816696, -0.06983084976673126, 0.08145324885845184, 0.03587093949317932, -0.032492272555828094, -0.04680389538407326, -0.08326862007379532, 0.043611206114292145, 0.09067035466432571, -0.02156726084649563, -0.06219864636659622, -0.08553557842969894, 0.020013049244880676, -0.039751969277858734, 0.08460517227649689, -0.016051413491368294, 0.025470787659287453, 0.01380359847098589, -0.09890116751194, 0.024206653237342834, 0.09773258119821548, 0.049202851951122284, -0.0124391233548522], "W3": [[-0.05870530381798744, -0.0710253044962883, 0.18410706520080566, 0.12332304567098618, 0.13161061704158783, -0.03885452821850777, 0.20304787158966064, -0.08982416242361069, 0.12834036350250244, -0.11565419286489487, -0.07595904171466827, 0.08731135725975037, 0.07682515680789948, 0.01836925931274891, 0.021731536835432053, -0.06744074821472168, 0.049046117812395096, -0.17394113540649414, 0.07185909897089005, 0.16596713662147522, 0.16875724494457245, 0.2061210721731186, 0.12807394564151764, -0.2044391632080078, 0.18830536305904388, 0.10003244876861572, -0.16512548923492432, -0.1344468593597412, 0.17202521860599518, 0.16572368144989014, -0.11088322103023529, 0.06980542838573456]], "b3": [-0.04262178763747215]}, "2021": {"W1": [[-0.04950256645679474, -0.10849064588546753, 0.038988515734672546, -0.07263103127479553, 0.047503381967544556, -0.13294485211372375, 0.054639097303152084, -0.017859039828181267, 0.02308390662074089, 0.06947880238294601, 0.03784432262182236, -0.03761757165193558, -0.03211662545800209, 0.12988661229610443, 0.05162513256072998, -0.09954908490180969, -0.10278550535440445, -0.13557536900043488, -0.04652291163802147, 0.07696150243282318, -0.024199407547712326, 0.11783609539270401, -0.09911699593067169, 0.0014991149073466659, 0.06791190803050995, -0.04218490421772003, -0.04926881566643715, 0.08291173726320267, -0.18742693960666656, 0.08896224945783615, 0.1418457329273224, -0.08847799152135849, -0.08375592529773712, -0.11533362418413162, -0.03512217849493027], [-0.08282367140054703, -0.14161814749240875, -0.14343665540218353, -0.09836726635694504, 0.04303041473031044, -0.04004015773534775, -0.06844552606344223, 0.06021416559815407, -0.14619891345500946, 0.009019927121698856, 0.045067012310028076, 0.16612142324447632, -0.05087808892130852, -0.11531040072441101, -0.005390966776758432, 0.10903730243444443, 0.06152905151247978, 0.116023488342762, 0.014112688601016998, 0.0738426148891449, -0.0471404530107975, -0.03238304704427719, -0.09437568485736847, -0.0038600864354521036, -0.08794303983449936, -0.0586051270365715, -0.11451765149831772, 0.04984625428915024, -0.12925024330615997, -0.018856091424822807, -0.11877678334712982, 0.08594653755426407, 0.034293707460165024, -0.08575844019651413, -0.04751945286989212], [-0.12980350852012634, 0.015868594869971275, 0.054516006261110306, -0.07449972629547119, -0.13595573604106903, -0.11560869961977005, -0.1351761668920517, 0.1054733544588089, 0.05522310733795166, -0.061747848987579346, 0.16985075175762177, 0.06532397121191025, 0.05186137184500694, -0.05443127080798149, 0.10320853441953659, -0.09077177196741104, -0.021815011277794838, 0.14704188704490662, 0.09982933849096298, 0.06897874921560287, -0.10734214633703232, 0.04253639653325081, -0.056874725967645645, -0.11026609688997269, 0.0050848741084337234, 0.0940064862370491, 0.11240760236978531, 0.10299757868051529, 0.0014852732419967651, 0.11751723289489746, 0.028007376939058304, 0.06221470609307289, -0.022036775946617126, -0.000490700826048851, 0.06644408404827118], [0.04548826813697815, -0.005154334008693695, 0.002819522749632597, -0.04707366228103638, 0.07789001613855362, 0.06814973056316376, 0.12991537153720856, 0.06772670894861221, -0.04493606463074684, 0.028496649116277695, 0.05831862613558769, 0.11663596332073212, 0.05016428977251053, 0.0716790035367012, 0.00246136705391109, -0.10318412631750107, -0.04177199676632881, 0.005298614501953125, -0.01675555109977722, -0.08772137761116028, -0.1085185557603836, -0.08917231112718582, 0.06347514688968658, -0.08567706495523453, 0.02385672926902771, 0.020694829523563385, -0.10563021898269653, 0.057298995554447174, -0.1363367885351181, -0.06679419428110123, 0.1469058096408844, -0.22251589596271515, -0.05341941863298416, 0.2571257948875427, 0.03299269452691078], [0.05179990082979202, -0.07686229050159454, 0.07728652656078339, -0.007821180857717991, 0.05201765149831772, -0.11548834294080734, 0.031220348551869392, 0.08313307911157608, 0.06799276918172836, 0.07843342423439026, 0.09004341810941696, 0.0021311554592102766, 0.01218117494136095, 0.0151504036039114, 0.04097399860620499, -0.1567966490983963, 0.14986953139305115, -0.0900253877043724, -0.07896475493907928, 0.035992275923490524, -0.10265113413333893, 0.006666578818112612, -0.07319115847349167, -0.07249844074249268, 0.001945024123415351, 0.06552164256572723, -0.11106716841459274, 0.02621988207101822, 0.1214095950126648, 0.10602934658527374, 0.03150274604558945, -0.2352629154920578, 0.004726495128124952, -0.06845711916685104, -0.023104218766093254], [0.05111871659755707, 0.029095986858010292, -0.07953532040119171, -0.13635005056858063, 0.08121243864297867, 0.012506961822509766, -0.05636917054653168, -0.029839325696229935, 0.05094592645764351, -0.03799736127257347, -0.12726359069347382, -0.004728962667286396, -0.09522334486246109, -0.1253480166196823, 0.114864282310009, 0.001806358341127634, -0.07317522168159485, 0.05633923038840294, 0.14093735814094543, -0.06952414661645889, -0.0172711331397295, 0.0303107351064682, 0.052715178579092026, -0.057818979024887085, -0.07252775877714157, 0.043607234954833984, -0.0022733008954674006, 0.06150735169649124, 0.00943168718367815, 0.10595519095659256, -0.08499302715063095, 0.10379676520824432, 0.14760459959506989, -0.009359326213598251, 0.13913655281066895], [-0.06315101683139801, 0.06295409053564072, -0.06868808716535568, -0.0911903828382492, -0.0933077335357666, -0.09468793123960495, -0.008741641417145729, 0.05659809336066246, 0.005117000080645084, -0.11008384078741074, -0.003968181554228067, 0.03195146098732948, -0.021407008171081543, 0.009694419801235199, 0.027744140475988388, 0.08149904757738113, 0.051182232797145844, -0.06508752703666687, -0.07487928867340088, -0.08541427552700043, 0.04145410656929016, 0.053769271820783615, 0.09277504682540894, -0.009077196009457111, -0.04560737684369087, -0.039232656359672546, -0.056377675384283066, -0.08131986856460571, 0.008463848382234573, 0.07624886929988861, 0.06176977604627609, -0.10489873588085175, 0.05593122914433479, -0.016118066385388374, 0.024385131895542145], [0.07120992988348007, -0.13706578314304352, -0.07969265431165695, -0.13432081043720245, -0.03414621204137802, 0.15066508948802948, 0.08224000781774521, -0.1131732240319252, 0.13122881948947906, -0.004510586149990559, -0.11729654669761658, 0.0132900420576334, -0.06533218920230865, -0.07026706635951996, -0.06618984788656235, 0.11573780328035355, 0.018615487962961197, -0.10308363288640976, -0.07870838046073914, -0.06159406155347824, 0.05733656883239746, -0.08549699932336807, 0.04099346697330475, -0.04717637225985527, -0.08629108965396881, -0.0864839106798172, 0.12290625274181366, 0.02524266019463539, -0.009716312400996685, -0.04373982921242714, -0.08425071835517883, -0.1372656524181366, -0.09947235882282257, -0.04619692265987396, -0.08039721846580505], [0.08110924810171127, 0.0066289291717112064, 0.09238573908805847, 0.02543816901743412, -0.11259420961141586, -0.12586212158203125, -0.09527870267629623, -0.13370807468891144, -0.06121407449245453, -0.08587218075990677, 0.11247861385345459, 0.09499628841876984, -0.10795947164297104, -0.08705911785364151, -0.08226829022169113, 0.023213066160678864, 0.04833227023482323, 0.026403024792671204, 0.11798279732465744, 0.07061279565095901, -0.014555594883859158, -0.06230194866657257, 0.0010329082142561674, 0.07932530343532562, -0.012540382333099842, -0.07349035888910294, 0.08619625866413116, 0.12628835439682007, 0.008899983018636703, -0.01213224045932293, 0.0033417867962270975, -0.0841532051563263, -0.06984846293926239, -0.09618648886680603, -0.10450627654790878], [0.031023988500237465, -0.04578689485788345, -0.09577001631259918, 0.06600257009267807, -0.0720735713839531, 0.006315579637885094, -0.11118493974208832, -0.006252591498196125, 0.1124461442232132, -0.032670117914676666, -0.019981488585472107, 0.10730444639921188, 0.09450749307870865, -0.015036600641906261, -0.07967384159564972, -0.062458813190460205, -0.10925577580928802, -0.08181674778461456, 0.11672775447368622, -0.06315718591213226, 0.18841078877449036, -0.05977216362953186, -0.12061451375484467, -0.13010118901729584, -0.09204280376434326, -0.04558033123612404, 0.02724399045109749, -0.036345139145851135, 0.005606488790363073, -0.10208100080490112, -0.052209604531526566, -0.06576883792877197, -0.012283333577215672, -0.09559325873851776, -0.13105075061321259], [-0.06067882850766182, 0.013837839476764202, -0.065308578312397, -0.08473344892263412, -0.090721994638443, 0.10187557339668274, 0.09353592246770859, -0.0526895746588707, 0.06742739677429199, 0.12881138920783997, -0.14367076754570007, 0.08891206979751587, 0.09616228938102722, -0.12357170879840851, -0.03855952247977257, 0.09482324123382568, -0.0938650518655777, -0.018807295709848404, -0.06722784787416458, 0.1022416427731514, 0.03901856392621994, -0.04801173880696297, 0.06132112443447113, 0.10859009623527527, -0.08324934542179108, -0.012544055469334126, 0.033955689519643784, 0.014351824298501015, -0.0855521634221077, -0.021643899381160736, -0.021896744146943092, 0.029824381694197655, 0.02461199089884758, -0.1113034039735794, 0.028379933908581734], [-0.08887220174074173, 0.020563369616866112, -0.0019218162633478642, -0.1712034046649933, -0.059964776039123535, -0.11256144940853119, -0.08182300627231598, 0.06151482090353966, 0.07239628583192825, 0.020486842840909958, 0.06350550055503845, 0.16057491302490234, -0.10127080976963043, -0.09788872301578522, -0.048883937299251556, -0.09603755921125412, 0.09617248177528381, -0.028126481920480728, -0.03209444880485535, -0.0763336569070816, 0.0003205078828614205, 0.05297726020216942, -0.057482171803712845, -0.014086762443184853, 0.06550224125385284, 0.14992451667785645, -0.10896143317222595, -0.003073715837672353, -0.12165375053882599, -0.06162386015057564, -0.04339054971933365, -0.015337346121668816, -0.06210257112979889, -0.05409661680459976, -0.08388443291187286], [-0.14995479583740234, -0.13666631281375885, 0.05991869792342186, -0.035189248621463776, 0.025953732430934906, 0.12202624976634979, 0.02349824644625187, 0.024511195719242096, -0.06721288710832596, -0.05558789521455765, 0.04120129719376564, -0.08885543793439865, -0.06947789341211319, 0.07357025891542435, 0.14594076573848724, -0.010848031379282475, -0.10484178364276886, 0.0574938990175724, -0.07445068657398224, 0.052529726177453995, -0.09433139860630035, 0.04601401090621948, 0.016407815739512444, -0.07995268702507019, -0.06081349030137062, 0.14110919833183289, 0.06758835166692734, -0.02468089945614338, 0.06134793162345886, -0.09224908798933029, 0.027010628953576088, -0.18926726281642914, 0.04470162093639374, 0.14463675022125244, -0.036200981587171555], [0.049074701964855194, -0.09116105735301971, 0.08022083342075348, 0.014046692289412022, 0.017008429393172264, 0.021236006170511246, -0.02865998074412346, 0.05424525588750839, 0.02494760975241661, -0.0686383992433548, 0.025839589536190033, 0.06547816842794418, 0.08393236994743347, -0.013974413275718689, 0.004331381991505623, -0.0545697845518589, -0.003893671790137887, 0.06997412443161011, 0.04041465371847153, 0.0463392436504364, -0.12399231642484665, 0.03343072533607483, -0.08752874284982681, -0.024099895730614662, 0.010563581250607967, -0.02046201378107071, -0.1342291682958603, 0.06605035066604614, 0.11886602640151978, 0.059346430003643036, 0.05742062255740166, -0.1707528978586197, 0.10809680074453354, 0.1126067116856575, -0.047242309898138046], [-0.11152101308107376, 0.04170220345258713, -0.015204625204205513, 0.07097537815570831, -0.0664871484041214, -0.0871572494506836, -0.011948026716709137, 0.10260433703660965, 0.04287120699882507, 0.04424063116312027, -0.08381664752960205, -0.006113693118095398, -0.03614823892712593, 0.029670920222997665, -0.0452277697622776, 0.026768527925014496, 0.06996282935142517, 0.025806965306401253, 0.060114845633506775, -0.11064700782299042, 0.10629914700984955, 0.0678570419549942, -0.10255672037601471, -0.09160498529672623, -0.057512398809194565, 0.016009729355573654, -0.10019762068986893, -0.02671174705028534, 0.10087631642818451, 0.0321313813328743, 0.02042817696928978, -0.14536410570144653, 0.09545992314815521, -0.00035428162664175034, 0.03884056583046913], [-0.022966554388403893, 0.08675741404294968, -0.003024416510015726, 0.1281484067440033, -0.04179568588733673, -0.079469695687294, 0.013239003717899323, 0.03308083489537239, 0.003371333470568061, 0.07534422725439072, -0.03326718881726265, -0.03952751308679581, -0.1727162003517151, -0.008640879765152931, 0.03910999372601509, -0.06719958037137985, 0.0009158168686553836, 0.029572894796729088, -0.05785389989614487, -0.06471247971057892, -0.007155804894864559, 0.005373282358050346, -0.053271081298589706, -0.06006551906466484, -0.05063312128186226, -0.04535016417503357, -0.04098200425505638, -0.02843780815601349, 0.00777923408895731, -0.05785604566335678, 0.03281780332326889, 0.11938336491584778, 0.05135146528482437, 0.07054900377988815, -0.005043884739279747], [0.03627000004053116, 0.049850452691316605, 0.018525399267673492, -0.011629372835159302, -0.0038589686155319214, -0.1304389089345932, 0.06374981999397278, -0.09164541959762573, -0.07644997537136078, 0.01586695946753025, 0.02027260698378086, 0.07543469965457916, -0.0427468903362751, -0.012773102149367332, 0.06021527200937271, 0.134024515748024, 0.0038043493404984474, 0.0686953067779541, 0.06778407096862793, 0.048874691128730774, 0.10413864254951477, 0.09627289324998856, 0.023470884189009666, -0.025541409850120544, -0.07276342809200287, 0.012753129005432129, -0.04046524688601494, -0.001475977129302919, -0.0886097401380539, 0.011949810199439526, 0.040363870561122894, 0.029524384066462517, 0.016265777871012688, 0.09896697849035263, 0.11134730279445648], [-0.03112940862774849, 0.05531587451696396, 0.04434023052453995, 0.12226594984531403, 0.05934252217411995, -0.06473229080438614, -0.009331093169748783, 0.017627660185098648, 0.10322976857423782, -0.01621326617896557, 0.12000687420368195, 0.05364963412284851, -0.07133335620164871, 0.09814716130495071, 0.11517898738384247, 0.07626761496067047, -0.11014064401388168, -0.012801929377019405, 0.06414686143398285, 0.09105309098958969, 0.001930137281306088, -0.03133702278137207, 0.11818640679121017, 0.04209962114691734, -0.1119774580001831, 0.04323401674628258, -0.05865573510527611, 0.013327352702617645, 0.09267120808362961, -0.005046328064054251, 0.011746148578822613, 0.08008629083633423, -0.020149266347289085, -0.08709095418453217, -0.052867595106363297], [0.047188933938741684, 0.048707425594329834, 0.05573034659028053, 0.00033503593294881284, 0.03740723803639412, -0.09516368061304092, 0.06716746091842651, 0.023332050070166588, -0.006446403916925192, 0.010247199796140194, -0.02133137173950672, -0.04102472960948944, -0.09851709753274918, -0.0836765319108963, -0.09827443957328796, -0.0060098981484770775, 0.058890730142593384, 0.0816347673535347, 0.037882715463638306, -0.06053641065955162, 0.011738156899809837, -0.02526991255581379, 0.03687051683664322, 0.046201057732105255, -0.06042531877756119, 0.0726483017206192, 0.0003518042794894427, 0.05835738033056259, -0.04147474467754364, -0.02201937884092331, -0.05825422331690788, 0.07483116537332535, -0.025530066341161728, 0.0014786093961447477, -0.08621660619974136], [0.022263463586568832, 0.043176181614398956, 0.02220023050904274, -0.03707748278975487, -0.07827083766460419, -0.0625327005982399, -0.005448872689157724, 0.04664602503180504, -0.01987219974398613, 0.08615202456712723, -0.01841735653579235, -0.09430231153964996, -0.0939832478761673, -0.05848465859889984, 0.05669807270169258, -0.0008575720712542534, -0.03458278998732567, 0.06706780195236206, -0.06356387585401535, 0.04755285009741783, 0.07049227505922318, -0.04828277975320816, 0.08502838015556335, 0.05633753910660744, -0.008878717198967934, -0.023699181154370308, 0.12789571285247803, -0.010596205480396748, -0.09856904298067093, -0.006991451606154442, 0.12153692543506622, 0.09597887098789215, 0.10984519869089127, 0.11591523885726929, 0.002426179824396968], [0.041643980890512466, -0.10037940740585327, -0.0748150572180748, -0.11013772338628769, -0.06604249775409698, -0.01865517348051071, -0.052528802305459976, -0.08001924306154251, -0.010282186791300774, -0.09439937025308609, -0.04465958848595619, 0.11599497497081757, -0.01484249159693718, 0.04831073433160782, -0.00713820056989789, 0.11289647966623306, 0.09698820114135742, -0.060257162898778915, -0.05056064948439598, 0.0687454342842102, -0.06104866415262222, 0.09772918373346329, -0.003765175584703684, 0.0061016264371573925, -0.11249077320098877, 0.044138696044683456, 0.002290086355060339, -0.07527966052293777, 0.08308658003807068, -0.050977904349565506, -0.01813417486846447, -0.11123432219028473, 0.057694222778081894, -0.14358441531658173, 0.04809959605336189], [0.02858937904238701, 0.10413631796836853, -0.06796211004257202, 0.08246254175901413, 0.06441468000411987, -0.025802476331591606, -0.09364256262779236, -0.0654674619436264, -0.02565418742597103, 0.017157116904854774, -0.11124563217163086, 0.06506321579217911, -0.025514259934425354, -0.03633077070116997, -0.08519632369279861, 0.1189810186624527, 0.010868662968277931, -0.04922764003276825, -0.0685005933046341, 0.09225761145353317, 0.10889916121959686, 0.048918675631284714, 0.006565771996974945, 0.022218335419893265, 0.05322788283228874, 0.02716878242790699, 0.15829959511756897, -0.025973455980420113, -0.12634608149528503, 0.051166463643312454, -0.07298135757446289, 0.06225470080971718, -0.13927029073238373, -0.14445525407791138, 0.0064221532084047794], [-0.08912248909473419, 0.12256825715303421, 0.05590302497148514, -0.07345230877399445, 0.029658952727913857, -0.00933004915714264, 0.03985069692134857, -0.025878388434648514, -0.02653554081916809, -0.041668884456157684, 0.07323618978261948, 0.08141866326332092, -0.01479364838451147, 0.088639035820961, 0.06200645491480827, -0.1395881175994873, 0.11935825645923615, 0.07965883612632751, 0.09116987138986588, 0.05124221369624138, 0.07735419273376465, -0.08414391428232193, -0.007655184715986252, 0.0049834661185741425, 0.05473460257053375, -0.10999877750873566, -0.1746673882007599, -0.0060064103454351425, -0.06850089877843857, -0.2163132131099701, -0.11952011287212372, 0.2591513991355896, -0.12995555996894836, 0.17847758531570435, 0.011541838757693768], [-0.026989957317709923, -0.06821737438440323, 0.14119179546833038, 0.07720410823822021, -0.03865901380777359, -0.1005912646651268, 0.04911806061863899, -0.03448890522122383, 0.028766442090272903, 0.04937973991036415, 0.07844472676515579, -0.10117016732692719, -0.01886732690036297, 0.005329064559191465, 0.009065239690244198, 0.1827455312013626, -0.10528525710105896, -0.047395914793014526, 0.11070790886878967, -0.0024989561643451452, 0.0470150001347065, 0.024074586108326912, 0.020349951460957527, 0.04641124606132507, -0.056608930230140686, 0.17173102498054504, 0.0006370782502926886, 0.026142627000808716, -0.1725006401538849, -0.13311654329299927, 0.04093848913908005, -0.16637447476387024, 0.04596786946058273, 0.1453138142824173, -0.031419966369867325], [-0.0877716913819313, -0.10748884081840515, -0.11501969397068024, -0.07980091869831085, 0.10434680432081223, -0.06662648916244507, 0.11409630626440048, -0.0644235759973526, 0.0930345430970192, -0.058780331164598465, 0.08405305445194244, -0.0405646413564682, 0.0009985200595110655, -0.05116552859544754, -0.09775953739881516, 0.06526263058185577, -0.0595758892595768, -0.06125473231077194, -0.02861551195383072, 0.10878310352563858, -0.08209153264760971, -0.067491814494133, 0.08022437989711761, 0.030702000483870506, -0.0356568805873394, -0.09271516650915146, 0.15588480234146118, 0.035656657069921494, -0.1629728376865387, -0.01790367253124714, -0.11030009388923645, -0.04477645456790924, 0.05006919801235199, -0.12313499301671982, 0.12191958725452423], [0.007391558960080147, -0.09079542011022568, 0.11643356829881668, -0.16444970667362213, 0.07550351321697235, -0.02245582826435566, 0.0813901275396347, 0.014009064063429832, -0.04824630916118622, 0.05912415310740471, 0.0501614511013031, 0.09404608607292175, -0.010840237140655518, -0.020705007016658783, 0.016855932772159576, -0.019811855629086494, 0.15504935383796692, -0.08740384131669998, -0.06331798434257507, -0.1490343064069748, 0.08425023406744003, -0.09605248272418976, 0.01722450740635395, -0.09864288568496704, -0.005373041611164808, 0.09844370931386948, -0.11227905005216599, -0.033553075045347214, 0.0719112828373909, -0.07543974369764328, -0.12017709761857986, -0.004158887546509504, 0.01708667352795601, 0.1761804223060608, -0.06737593561410904], [0.050042781978845596, -0.03010985068976879, -0.03728487715125084, 0.035427726805210114, 0.012411186471581459, 0.17558278143405914, -0.04788242653012276, -0.10883486270904541, -0.06180957704782486, -0.0014804488746449351, -0.030022388324141502, -0.01028304360806942, -0.00879305973649025, 0.018789606168866158, 0.082924023270607, -0.10181843489408493, -0.036848220974206924, 0.08464379608631134, -0.08283576369285583, -0.06892320513725281, -0.00871730875223875, 0.011024679988622665, -0.010463329963386059, 0.051214974373579025, 0.026527397334575653, -0.03598099946975708, 0.002019785810261965, -0.001960389083251357, 0.0691458210349083, -0.004780726507306099, -0.08340515941381454, -0.0729215070605278, 0.06507653743028641, 0.0681181326508522, 0.07465894520282745], [0.008603275753557682, -0.04265741631388664, -0.09322785586118698, 0.028966519981622696, -0.09294918924570084, -0.05052763968706131, -0.09445354342460632, -0.030551590025424957, 0.04350090026855469, -0.013754368759691715, -0.16204315423965454, 0.14825911819934845, 0.059159040451049805, 0.12975172698497772, 0.12615826725959778, 0.09741254150867462, -0.005486373323947191, 0.07442784309387207, -0.02744103968143463, 0.13947153091430664, 0.13119129836559296, -0.12129613757133484, -0.07597678154706955, 0.08837441354990005, 0.09078076481819153, 0.03580364212393761, 0.1254255622625351, -0.08492317795753479, -0.09671644121408463, -0.06302948296070099, -0.07002913951873779, -0.1319637894630432, 0.12060122191905975, -0.058190200477838516, 0.002662639133632183], [0.051192715764045715, 0.054193831980228424, 0.10915635526180267, 0.026874518021941185, -0.087788887321949, 0.07334873825311661, 0.03480501100420952, 0.032525207847356796, -0.05250335484743118, -0.041979070752859116, 0.07835959643125534, -0.07108686864376068, 0.09565041214227676, -0.09720652550458908, -0.0845237746834755, -0.07482793927192688, 0.06591018289327621, 0.026802409440279007, -0.06066002696752548, -0.06563940644264221, -0.00544886477291584, 0.00823769811540842, -0.06261686980724335, -0.08523137122392654, -0.049902379512786865, -0.021723762154579163, 0.008698761463165283, -0.035982970148324966, -0.0019646717701107264, 0.08038945496082306, 0.10612113773822784, -0.17983534932136536, -0.042047034949064255, 0.1231895238161087, -0.05775820463895798], [-0.03756967931985855, 0.042941391468048096, 0.08828306943178177, -0.07943771779537201, -0.08952601999044418, 0.06280849128961563, 0.08839230984449387, 0.04399167001247406, 0.030058715492486954, -0.007270364556461573, -0.0124604981392622, 0.022711629047989845, 0.05887318029999733, -0.12748365104198456, -0.12641680240631104, 0.034390877932310104, 0.10034194588661194, 0.04029005393385887, -0.016380751505494118, 0.05401572957634926, 0.12141627818346024, -0.0883074700832367, 0.03028509020805359, -0.10226717591285706, -0.055767737329006195, -0.03881151229143143, 0.018880901858210564, -0.005944698583334684, -0.02852366305887699, 0.010680525563657284, -0.09030776470899582, 0.08154149353504181, 0.08080536127090454, 0.05783943459391594, -0.07166583836078644], [0.0683625265955925, -0.024484192952513695, 0.14866071939468384, -0.03256508708000183, 0.056197889149188995, 0.09812744706869125, 0.034348055720329285, -0.09365446120500565, -0.014629230834543705, 0.10640501230955124, -0.03078131191432476, 0.0721726194024086, -0.0062017254531383514, -0.06353144347667694, 0.020072486251592636, -0.06513474136590958, -0.08841460943222046, -0.016320226714015007, -0.03619008511304855, 0.09407153725624084, 0.14843517541885376, 0.010321796871721745, 0.07165541499853134, -0.11050735414028168, -0.06315372884273529, 0.07732918858528137, 0.10734090954065323, 0.026053454726934433, 0.03966252878308296, 0.05576646327972412, 0.0222660880535841, 0.05647171288728714, 0.028290066868066788, -0.00801150407642126, -0.1005319207906723], [-0.04732147231698036, 0.04254373535513878, 0.13414603471755981, 0.025358840823173523, 0.09117510169744492, -0.05677896738052368, -0.06905700266361237, 0.007127712480723858, 0.07485313713550568, 0.04609205573797226, -0.003300461685284972, 0.02104363404214382, -0.03836417570710182, 0.015499833039939404, 0.05854479596018791, 0.004244579467922449, 0.04464471712708473, 0.0012046323390677571, -0.05836540833115578, -0.02097884565591812, -0.05020603537559509, 0.05690144747495651, 0.06169673800468445, -0.012812317349016666, 0.053920846432447433, -0.09872733056545258, -0.04400753229856491, -0.045518431812524796, 0.012208161875605583, 0.05662521719932556, -0.016378698870539665, -0.056641366332769394, 0.015628093853592873, 0.03133361041545868, -0.042953815311193466], [-0.0998663529753685, -0.07462060451507568, 0.07887180894613266, 0.07417097687721252, -0.08513263612985611, 0.02825385145843029, -0.07056529074907303, -0.11535881459712982, 0.009194517508149147, -0.07870268821716309, -0.07691525667905807, -0.0018533536931499839, 0.011451387777924538, -0.0568021722137928, 0.011818050406873226, -0.03273530676960945, -0.05001200735569, 0.01578245684504509, 0.029233833774924278, -0.05401553958654404, 0.07054152339696884, 0.027906950563192368, -0.017740847542881966, -0.06027916446328163, -0.0660277009010315, 0.10090097784996033, 0.004247479606419802, -0.0016584937693551183, -0.034432798624038696, -0.08084691315889359, -0.07149588316679001, 0.13675178587436676, -0.001831851084716618, 0.10696714371442795, -0.04386661574244499], [0.015053139999508858, -0.0066581303253769875, 0.06955605000257492, 0.010712879709899426, -0.027748316526412964, -0.08233918249607086, -0.04748041555285454, 0.033184051513671875, 0.07394856214523315, 0.09812059998512268, -0.003075254149734974, -0.06010450795292854, -0.08088499307632446, -0.01266535185277462, -0.04353420436382294, 0.025742623955011368, -0.007479020860046148, 0.02354852296411991, 0.07363726198673248, -0.10186179727315903, 0.0961463525891304, 0.03965301066637039, -0.07778648287057877, 0.06702231615781784, 0.08337683230638504, -0.013955165632069111, 0.018620731309056282, 0.06607475131750107, 0.05053550377488136, 0.015606286004185677, -0.026041973382234573, 0.12447065114974976, -0.09267578274011612, 0.058174017816782, -0.030165327712893486], [0.014542713761329651, -0.09867483377456665, -0.061944711953401566, -0.07066372036933899, 0.06369812041521072, 0.0655938982963562, 0.02423091046512127, -0.024231556802988052, -0.09974531084299088, -0.051651231944561005, -0.13852624595165253, 0.03549898788332939, -0.03462417051196098, 0.07047194242477417, 0.11830629408359528, -0.09280075877904892, 0.062484804540872574, 0.05954117327928543, -0.07792183011770248, -0.10440772771835327, 0.15437261760234833, -0.09942638128995895, 0.055779095739126205, 0.03406631201505661, 0.03233586251735687, -0.1424597203731537, 0.07941962033510208, -0.0794513002038002, -0.11814291775226593, 0.13370680809020996, -0.04649358615279198, -0.0721290335059166, -0.13669274747371674, 0.011175153777003288, -0.07192013412714005], [-0.07122533768415451, -0.047786254435777664, 0.049313079565763474, -0.06627092510461807, -0.04241981729865074, 0.1270986646413803, -0.03780617192387581, -0.021579554304480553, 0.04070718586444855, 0.05743153765797615, -0.0171975065022707, -0.06105050444602966, -0.02091824822127819, 0.00987801980227232, 0.013461061753332615, 0.1243768185377121, 0.06285610049962997, -0.005943804048001766, 0.022675776854157448, -0.10532879829406738, 0.08218485116958618, -0.043557994067668915, -0.03211403638124466, -0.05616169422864914, -0.08354216814041138, 0.0458771176636219, 0.019857659935951233, 0.05718229338526726, 0.10610254108905792, -0.07463417202234268, 0.04670020565390587, 0.010479084216058254, 0.03864743188023567, 0.1476231813430786, -0.024802112951874733], [-0.08040757477283478, -0.06889065355062485, 0.020678043365478516, 0.08169308304786682, -0.09291139990091324, -0.03121902607381344, -0.028593339025974274, -0.02871282398700714, -0.10988065600395203, 0.1269635260105133, -0.0893820971250534, -0.07166951149702072, -0.07874244451522827, -0.03015744686126709, 0.08193133026361465, -0.006430299486964941, -0.05532817915081978, -0.026361502707004547, -0.057605575770139694, -0.08115073293447495, 0.05077854171395302, -0.07180415093898773, -0.08214645832777023, -0.07671239972114563, 0.04351896792650223, 0.07553458958864212, -0.10254641622304916, 0.12181591242551804, 0.06578962504863739, -0.03896629065275192, -0.005446729715913534, 0.06474966555833817, -0.07155727595090866, 0.036790184676647186, 0.007522857282310724], [0.0377848744392395, 0.006195111200213432, -0.0031893900595605373, 0.06763520836830139, 0.09885205328464508, -0.0883273333311081, -0.08200561255216599, 0.12092793732881546, -0.03222157806158066, 0.013743000105023384, -0.032738834619522095, 0.018075736239552498, -0.014357284642755985, 0.10158072412014008, 0.06342340260744095, -0.14038996398448944, -0.10209616273641586, -0.022866323590278625, 0.07051926851272583, 0.07285541296005249, 0.11906557530164719, 0.004316410049796104, -0.022188358008861542, 0.003401451976969838, 0.10424073785543442, 0.1206081360578537, -0.1679716408252716, -0.036660242825746536, 0.14801740646362305, -0.15810257196426392, -0.12859182059764862, 0.26807132363319397, -0.010411441326141357, 0.1435350775718689, -0.011144626885652542], [0.014738120138645172, -0.03703732788562775, -0.0011002777609974146, -0.0592944398522377, 0.011689538136124611, 0.04204680398106575, -0.08684750646352768, 0.014171499758958817, 0.028508294373750687, -0.0008686965447850525, 0.0316760390996933, -0.01726127229630947, 0.011581117287278175, 0.06767631322145462, 0.09735483676195145, 0.06095209717750549, -0.042973846197128296, -0.054887618869543076, 0.006087604910135269, -0.009405942633748055, 0.011556695215404034, -0.04381534084677696, 0.008795174770057201, 0.02534567192196846, -0.018544403836131096, 0.07750175893306732, -0.01201679278165102, -0.02207903377711773, 0.01304573193192482, -0.01733427122235298, -0.00763741135597229, 0.05284413322806358, -0.04602629691362381, 0.08059567958116531, 0.030284786596894264], [-0.038963381201028824, -0.030324626713991165, 0.08314592391252518, 0.06328526884317398, -0.028338581323623657, 0.09517363458871841, 0.07404737174510956, 0.12359743565320969, -0.02533828280866146, 0.0673665925860405, -0.08558662235736847, 0.023139946162700653, -0.03200441226363182, 0.06208835169672966, 0.0702250525355339, -0.11817134916782379, -0.0834275484085083, -0.08806587010622025, 0.041504278779029846, -0.030475204810500145, 0.04988637566566467, 0.008995605632662773, -0.04632159322500229, 0.11259587854146957, -0.12459979206323624, 0.13123644888401031, -0.1146828755736351, 0.016298500820994377, -0.08962182700634003, 0.05429857224225998, -0.01339638326317072, -0.07886746525764465, 0.08311795443296432, 0.04777587577700615, 0.057742442935705185], [0.04576818272471428, 0.1147356852889061, 0.046568527817726135, 0.07283610850572586, 0.12003973126411438, -0.027899768203496933, 0.09879139065742493, 0.023504721000790596, 0.020296035334467888, 0.06784269958734512, 0.023648567497730255, -0.08891115337610245, -0.04017619788646698, -0.13545173406600952, -0.1574370563030243, -0.01048930361866951, 0.04982532560825348, 0.0008575398242101073, -2.8425042728486005e-06, -0.033288389444351196, -0.05347590520977974, -0.01066694688051939, -0.04415634647011757, 0.064170703291893, 0.10016327351331711, 0.12102064490318298, -0.015786318108439445, -0.07481618970632553, -0.014461533166468143, -0.14838866889476776, 0.04442864656448364, 0.1714218556880951, 0.018660835921764374, 0.01467046421021223, 0.04263670742511749], [-0.041426029056310654, -0.03403296694159508, -0.028685947880148888, -0.04455995932221413, -0.027910521253943443, -0.007583003491163254, 0.058105118572711945, 0.0863177478313446, 0.02638651244342327, -0.0029884285759180784, -0.04308187589049339, -0.055159468203783035, -0.03442868962883949, -0.07466459274291992, -0.046423956751823425, -0.033825431019067764, -0.025157468393445015, 0.0462653748691082, 0.055955611169338226, -0.020115027204155922, -0.11534923315048218, 0.038001157343387604, -0.08816129714250565, -0.1133226752281189, -0.1732635647058487, -0.04922933503985405, -0.1217736154794693, -0.11274439841508865, -0.021589377894997597, 0.11866689473390579, 0.10310047119855881, 0.014741532504558563, 0.09594830870628357, -0.07624805718660355, -0.08127595484256744], [-0.023800186812877655, 0.08287381380796432, -0.09851595014333725, -0.023938409984111786, 0.013266450725495815, -0.05851221829652786, -0.03305814042687416, 0.00010547139390837401, 0.06109585613012314, 0.07638978958129883, -0.0005790612194687128, -0.11810073256492615, 0.12191584706306458, -0.01815255731344223, 0.0396263487637043, -0.0021848552860319614, -0.05297888070344925, -0.021670207381248474, 0.062136221677064896, 0.07588479667901993, 0.004567460622638464, -0.051216740161180496, -0.06483040004968643, 0.03534313291311264, 0.02515818364918232, 0.11659205704927444, 0.11047356575727463, 0.08710440993309021, -0.007947840727865696, -0.11666373908519745, 0.0034856698475778103, 0.038883667439222336, -0.07925484329462051, -0.014731091447174549, -0.07914786785840988], [-0.03768608719110489, 0.11181576550006866, -0.03348826617002487, 0.1287776231765747, 0.06791053712368011, 0.032352130860090256, 0.007974429987370968, 0.04167450591921806, 0.05086357146501541, -0.007669271435588598, 0.09385066479444504, -0.1023520901799202, -0.10540705919265747, -0.022024327889084816, 0.07222148030996323, -0.09997755289077759, -0.04956505820155144, -0.08459010720252991, 0.02108113281428814, 0.00013054146256763488, 0.08001139014959335, -0.06927134841680527, -0.06678774207830429, -0.09990695863962173, 0.005066519603133202, -0.11160248517990112, -0.05679537355899811, 0.05454622581601143, 0.08869041502475739, -0.0021647210232913494, 0.04968760535120964, -0.01241875346750021, 0.0011896261712536216, 0.0541539341211319, -0.11298329383134842], [-0.07448412477970123, 0.07840625941753387, 0.006672442425042391, 0.032975438982248306, 0.08824347704648972, 0.03948002681136131, 0.05172478035092354, -0.08883331716060638, 0.06447858363389969, 0.06929128617048264, -0.012838131748139858, -0.07178187370300293, -0.0019355332478880882, 0.07860542088747025, -0.04717370495200157, 0.08242281526327133, 0.027275871485471725, 0.01955052651464939, -0.10989230871200562, 0.05577271059155464, 0.10716421157121658, -0.033162496984004974, -0.06258323043584824, 0.11977435648441315, 0.0898752510547638, -0.05375620722770691, 0.09528349339962006, 0.08690011501312256, 0.029697801917791367, -0.12125785648822784, 0.0800667554140091, 0.12878145277500153, -0.14232832193374634, 0.013011240400373936, 0.004364189226180315], [0.028668908402323723, -0.09241346269845963, -0.04906216636300087, 0.08502338081598282, 0.015880871564149857, -0.06586320698261261, 0.0012017623521387577, -0.020064005628228188, 0.062259867787361145, -0.00472836522385478, -0.10455842316150665, 0.032456036657094955, 0.08881037682294846, 0.027868367731571198, -0.009726257063448429, -0.026580745354294777, -0.026068944483995438, -0.06839405745267868, 0.03967008367180824, 0.05931207537651062, -0.1291007697582245, 0.013899010606110096, 0.005683096591383219, -0.10055652260780334, 0.017748048529028893, 0.09028249979019165, -0.08080065995454788, -0.030296312645077705, 0.043772853910923004, 0.009474494494497776, -0.018282674252986908, -0.18444380164146423, 0.05718589946627617, 0.19554860889911652, -0.07475457340478897], [0.015575360506772995, 0.0330500528216362, -0.027519620954990387, 0.05375324934720993, 0.03551184758543968, 0.00915027316659689, 0.020396307110786438, 0.004794744309037924, 0.055804263800382614, -0.032790206372737885, -0.07546979188919067, -0.018130620941519737, -0.05366694554686546, -0.05696506053209305, -0.040197670459747314, -0.055153198540210724, 0.012698632664978504, 0.06791628152132034, 0.07169712334871292, -0.00900382362306118, -0.013555031269788742, -0.03226805105805397, 0.027818582952022552, -0.0012372860219329596, -0.027696892619132996, -0.06505551934242249, 0.008005030453205109, 0.008025319315493107, -0.07031844556331635, 0.05885423719882965, -0.04217700660228729, -0.03912634029984474, -0.01436656340956688, 0.005295358598232269, 0.03726869449019432], [0.030836639925837517, 0.11799325793981552, -0.09538624435663223, -0.01633625291287899, -0.06692365556955338, -0.01729438826441765, -0.05251990258693695, 0.06975630670785904, -0.058375079184770584, -0.060318101197481155, 0.06362511962652206, -0.04286978766322136, 0.07123230397701263, 0.04301666468381882, -0.02511722780764103, 0.0356145054101944, 0.06452172994613647, 0.15826618671417236, -0.15385876595973969, -0.06228683516383171, -0.08529572933912277, -0.17637264728546143, -0.06299244612455368, 0.041160330176353455, 0.05097649246454239, -0.08775609731674194, -0.044161904603242874, -0.04564881697297096, -0.014428867027163506, 0.009143705479800701, -0.17434613406658173, 0.14776788651943207, -0.06529561430215836, -0.08171506226062775, -0.06479965895414352], [0.026280555874109268, 0.08993885666131973, -0.04544948786497116, 0.0014999017585068941, -0.01841302402317524, 0.09683112055063248, 0.09778327494859695, 0.0124390609562397, 0.05860035866498947, -0.0038998376112431288, -0.10008572041988373, 0.17339234054088593, 0.0037034135311841965, 0.062092747539281845, -0.11955177783966064, -0.12114416807889938, -0.1253954917192459, -0.12403993308544159, -0.032046206295490265, -0.004353454802185297, 0.1449907422065735, -0.03229615464806557, 0.05748558044433594, 0.011278452351689339, -0.031763628125190735, -0.009566436521708965, 0.11226700246334076, -0.10080752521753311, -0.016819484531879425, 0.11104240268468857, 0.020630186423659325, 0.056119516491889954, -0.0041392650455236435, 0.11149158328771591, 0.11888106912374496], [-0.11139024794101715, 0.007420864887535572, 0.016484972089529037, 0.00540215102955699, 0.07541023194789886, 0.023732151836156845, -0.1094915121793747, 0.016539350152015686, -0.0032783686183393, 0.04266246408224106, -0.035836804658174515, -0.05334019660949707, 0.10740194469690323, 0.04432388022542, -0.05857126787304878, -0.07717277854681015, 0.028316834941506386, -0.07342726737260818, 0.059379443526268005, -0.05552752688527107, 0.0069901542738080025, 0.05371886119246483, 0.06800097227096558, -0.018236909061670303, -0.0126187764108181, 0.05551675707101822, -0.015181170776486397, -0.08345912396907806, -0.050883594900369644, 0.025518305599689484, -0.06993003934621811, -0.014314930886030197, 0.05220773071050644, 0.012342872098088264, -0.027727119624614716], [-0.018449503928422928, -0.06525740772485733, 0.09324397891759872, 0.018562335520982742, -0.08853821456432343, 0.1324848234653473, 0.06200949475169182, -0.0002303954679518938, -0.03859873116016388, -0.009095700457692146, 0.042013175785541534, 0.06542512029409409, 0.017825327813625336, 0.017166683450341225, 0.11813697963953018, 0.019972683861851692, -0.00538388267159462, -0.009209446609020233, 0.057735249400138855, -0.09202615916728973, -0.1297433078289032, 0.052270445972681046, 0.04584190249443054, -0.048155348747968674, 0.0022753130178898573, 0.13914570212364197, 0.06291491538286209, -0.06445936858654022, 0.08253493160009384, -0.09315399080514908, 0.1871969848871231, -0.2510574460029602, 0.02070888876914978, 0.20001406967639923, 0.11449450254440308], [0.1538291722536087, -0.040908075869083405, -0.19691817462444305, 0.09813472628593445, 0.14661139249801636, -0.02094786986708641, -0.09284297376871109, 0.025044525042176247, -0.04087027162313461, -0.08406611531972885, 0.08214472234249115, 0.1154710128903389, 0.16419653594493866, 0.043823227286338806, 0.0920424610376358, -0.04836243391036987, 0.003640139475464821, 0.04266105219721794, -0.149908185005188, -0.10267633199691772, -0.02873547188937664, 0.012312784790992737, 0.04859844595193863, -0.07863196730613708, 0.08559581637382507, 0.10435303300619125, 0.07810713350772858, -0.0484171137213707, 0.1581496298313141, 0.04501863196492195, 0.011676750145852566, 0.006894888821989298, -0.07633111625909805, 0.0016201426042243838, 0.062307219952344894], [-0.054891929030418396, -0.19224008917808533, 0.13629236817359924, -0.035532113164663315, 0.06926020979881287, -0.05746597424149513, 0.03013039380311966, 0.06357607245445251, -0.05255439132452011, 0.01731160469353199, -0.014068607240915298, 0.09471649676561356, -0.021227581426501274, -0.04671833664178848, 0.0854731872677803, -0.05427183955907822, 0.08688047528266907, 0.01119948085397482, 0.044006939977407455, 0.019173434004187584, -0.02401108294725418, 0.0562279112637043, -0.04719851166009903, -0.02261023409664631, -0.03610983490943909, 0.0936710312962532, 0.028348639607429504, 0.02027440443634987, 0.08177899569272995, 0.04911777004599571, 0.022130219265818596, -0.2629369795322418, 0.15700313448905945, 0.21479900181293488, 0.049495771527290344], [-0.03394200652837753, 0.03571275249123573, 0.07937407493591309, 0.023275911808013916, 0.09223949164152145, -0.059753358364105225, -0.04181571677327156, 0.032521381974220276, -0.09083392471075058, -0.002863137051463127, 0.07824640721082687, -0.006054839119315147, -0.05592451989650726, -0.052628323435783386, 0.01512724719941616, 0.0733528733253479, 0.06936608999967575, -0.07378248870372772, -0.07850294560194016, 0.014106393791735172, -0.06878697872161865, -0.04432646930217743, -0.06549542397260666, -0.07400887459516525, -0.07824405282735825, -0.0028396446723490953, 0.0985293760895729, 0.0728401392698288, 0.018940519541502, 0.024763518944382668, 0.045008476823568344, 0.06035624071955681, -0.07196559756994247, 0.057078782469034195, -0.015291841700673103], [-0.00035371477133594453, -0.09806131571531296, 0.03275475278496742, -0.10532386600971222, -0.06394822895526886, 3.637521876953542e-05, 0.08242779225111008, -0.056566134095191956, -0.05950632691383362, 0.012424396350979805, -0.020608248189091682, -0.08053707331418991, -0.04271605238318443, 0.07218648493289948, 0.03949848935008049, -0.02453095279633999, 0.10789080709218979, 0.04834921658039093, -0.0708218663930893, -0.07220660895109177, 0.030549833551049232, 0.018923623487353325, 0.07233714312314987, 0.01580001786351204, 0.019925151020288467, -0.015077030286192894, -0.07680688798427582, -0.012874091044068336, 0.00141701300162822, -0.07290027290582657, 0.01539532095193863, -0.12416931241750717, 0.09207726269960403, 0.030731255188584328, -0.08750131726264954], [-0.062465086579322815, 0.036845602095127106, 0.2026698738336563, 0.08279211819171906, 0.1599469780921936, -0.08532721549272537, -0.01014264952391386, -0.12465420365333557, 0.011387617327272892, 0.028297755867242813, 0.17854043841362, -0.05205351486802101, -0.05126539617776871, 0.16769862174987793, -0.036139655858278275, 0.024043995887041092, -0.14848005771636963, -0.022239765152335167, 0.04360608384013176, -0.08069472014904022, 0.06587974727153778, -0.04253549873828888, -0.011011464521288872, 0.09290991723537445, -0.03604460880160332, 0.021991392597556114, 0.0023103186395019293, -0.05155928060412407, 0.12619100511074066, 0.06920599192380905, -0.004543775226920843, 0.18566186726093292, 0.05657048150897026, 0.006497072521597147, -0.003542795544490218], [0.10029140114784241, 0.01233451534062624, 0.0650724321603775, 0.031988032162189484, -0.10098026692867279, -0.05133572965860367, -0.04234284907579422, 0.13730372488498688, -0.042408186942338943, -0.045875806361436844, 0.07121164351701736, -0.001975052524358034, 0.027002649381756783, 0.04508071392774582, 0.05738650634884834, 0.03961636498570442, -0.06201741099357605, 0.01613697223365307, 0.03603942319750786, -0.02514614351093769, 0.13504716753959656, 0.08028566092252731, 0.06477206200361252, -0.03903389349579811, 0.028039956465363503, 0.025500766932964325, 0.07747583091259003, -0.042443081736564636, 0.11362158507108688, -0.06738854944705963, -0.0746670737862587, 0.1083739623427391, 0.08164896816015244, 0.060866329818964005, -0.082486592233181], [-0.08496459573507309, -0.09995559602975845, 0.079125314950943, -0.02134723775088787, 0.027025479823350906, -0.02971000410616398, 0.09394433349370956, 0.004776558373123407, -0.04646367207169533, -0.011319511570036411, -0.019517499953508377, -0.008459159173071384, 0.1404576450586319, -0.012122169137001038, 0.05779951065778732, 0.07042670249938965, 0.09452977776527405, -0.07295654714107513, 0.053544506430625916, 0.023489056155085564, -0.10719120502471924, -0.10274701565504074, 0.0626119002699852, 0.09262505173683167, -0.11371432989835739, 0.23978762328624725, 0.1062634065747261, -0.01445501483976841, -0.0774562805891037, 0.03589378297328949, 0.08210429549217224, -0.20523975789546967, 0.02659931220114231, 0.1695450246334076, -0.033640630543231964], [-0.09600693732500076, 0.031666625291109085, 0.015459557995200157, 0.06754627823829651, -0.0621492899954319, 0.11809351295232773, -0.06723253428936005, 0.10461565852165222, 0.1405518352985382, 0.04644658416509628, 0.02313242107629776, -0.0874256119132042, 0.005701528396457434, -0.015597591176629066, -0.05385344848036766, -0.0729324072599411, -0.05685193091630936, -0.034220997244119644, 0.10109752416610718, 0.06801243871450424, 0.16647256910800934, 0.027671724557876587, 0.03948931396007538, -0.04296122491359711, -0.058253213763237, 0.1147383600473404, -0.08909375220537186, -0.04022025689482689, -0.05949130654335022, -0.044571105390787125, -0.0871119275689125, 0.1098518818616867, -0.03378002345561981, 0.04599319025874138, -0.0859755128622055], [0.02811439521610737, -0.008938675746321678, -0.00634282361716032, -0.03646815940737724, 0.04702724516391754, 0.032564762979745865, -0.03985719755291939, -0.03032633475959301, -0.004950321279466152, 0.03328658267855644, 0.0007277227123267949, -0.04418161138892174, 0.03221941739320755, 0.06194710358977318, 0.10309631377458572, 0.013826052658259869, -0.0959651842713356, -0.11265946179628372, 0.05593584105372429, -0.024679064750671387, -0.0663585513830185, 0.0001523967512184754, -0.008937826380133629, -0.03747774660587311, -0.10443342477083206, -0.11697618663311005, 0.10820470750331879, 0.04829767718911171, 0.07142453640699387, 0.06630422174930573, -0.020495319738984108, -0.014492763206362724, 0.06759700179100037, -0.04178996384143829, -0.008962396532297134], [0.008417118340730667, -0.011176228523254395, 0.04436952993273735, -0.10328571498394012, 0.02274324744939804, 0.026147153228521347, 0.04259592294692993, -0.08094228059053421, -0.052328839898109436, -0.10926889628171921, 0.05241294950246811, 0.1295662671327591, 0.07577260583639145, 0.0010555294575169683, -0.011511790566146374, -0.022081883624196053, -0.07241901010274887, -0.03837496042251587, -0.02851487137377262, -0.02537284791469574, 0.10416760295629501, 0.07139085978269577, -0.09646792709827423, 0.048635248094797134, -0.11593238264322281, 0.12898008525371552, 0.07512259483337402, 0.054469034075737, -0.05235129967331886, -0.12453599274158478, 0.07403666526079178, -0.04383931681513786, 0.06387452781200409, 0.04469531401991844, 0.07356198132038116], [0.03646107017993927, -0.03398346155881882, 0.05878005549311638, 0.03628493472933769, 0.06960709393024445, -0.0608154833316803, 0.030703634023666382, -0.03149433061480522, 0.03291643410921097, -0.020542677491903305, -0.015471434220671654, 0.08717416226863861, 0.03955521807074547, 0.018917318433523178, 0.05792204663157463, 0.054680295288562775, -0.12181893736124039, 0.044720038771629333, 0.11767254769802094, -0.11590418964624405, 0.03134895861148834, -0.00900212675333023, 0.08517739176750183, -0.07531402260065079, -0.006297353655099869, 0.10173062235116959, 0.0541284941136837, 0.03789161145687103, 0.05859580263495445, 0.013917164877057076, -0.03792014345526695, -0.10862039774656296, 0.018938902765512466, 0.025559645146131516, 0.022073883563280106], [0.06846540421247482, -0.042632367461919785, -0.024820806458592415, -0.06691841036081314, 0.06871135532855988, -0.06551532447338104, 0.01416957750916481, -0.106479212641716, -0.010117676109075546, -0.011002645827829838, -0.0012767115840688348, 0.013158408924937248, 0.04782357066869736, 0.005373634397983551, -0.07727710902690887, -0.06292238086462021, -0.032739389687776566, 0.04115624725818634, -0.0029844623059034348, 0.03873182088136673, -0.09203702956438065, 0.0076309083960950375, 0.021328169852495193, 0.12855203449726105, -0.034987673163414, -0.02857951447367668, 0.10593420267105103, 0.04052573814988136, -0.021205365657806396, 0.01733102835714817, 0.043210774660110474, -0.04847919940948486, 0.020292825996875763, 0.0073415194638073444, 0.013761065900325775], [-0.09333325177431107, 0.05166586861014366, 0.017629554495215416, 0.06479305028915405, -0.14798840880393982, 0.12638619542121887, -0.015663381665945053, -0.026632392778992653, 0.05314916744828224, 0.06704233586788177, -0.07756553590297699, -0.04792584478855133, -0.06396393477916718, 0.029663678258657455, 0.012984316796064377, -0.10235150903463364, 0.0030330524314194918, -0.04953548312187195, 0.015635643154382706, 0.04578593373298645, 0.03197887912392616, -0.06318971514701843, -0.04161832481622696, 0.0014411179581657052, 0.07959011942148209, 0.07384423166513443, -0.08460631221532822, 0.015560882166028023, -0.11953607201576233, -0.10542923212051392, 0.08384785801172256, 0.11394057422876358, -0.0383305549621582, 0.025095688179135323, 0.019092265516519547], [0.048402704298496246, 0.09357158839702606, -0.025352906435728073, -0.03212545067071915, -0.05375850573182106, -0.06720320135354996, -0.09724457561969757, 0.0256195068359375, 0.06555402278900146, 0.04863559454679489, -0.11437961459159851, -0.0804099589586258, 0.021755922585725784, 0.03150608018040657, -0.08487407118082047, 0.11573679000139236, -0.024951649829745293, -0.03934600576758385, -0.1052374392747879, 0.046435870230197906, 0.028487620875239372, -0.036530397832393646, -8.03585680841934e-06, 0.07504604756832123, 0.005526276770979166, -0.06129719689488411, -0.01178773958235979, 0.030099766328930855, 0.006302185356616974, -0.08851572871208191, -0.10514216125011444, -0.06465058028697968, 0.09332122653722763, 0.08820200711488724, -0.025325698778033257], [-0.07792887091636658, 0.059836748987436295, 0.11559849232435226, 0.08153903484344482, -0.08354833722114563, 0.17275424301624298, -0.00829311553388834, -0.01062805950641632, -0.03000638447701931, 0.006295489147305489, 0.08497592061758041, 0.07528579235076904, 0.001892476575449109, 0.009382491931319237, -0.0029175919480621815, 0.004675861913710833, 0.08829367905855179, 0.12941096723079681, -0.009080319665372372, 0.04536723345518112, -0.09784539043903351, 0.12807178497314453, 0.007658881600946188, 0.058410294353961945, 0.08280076831579208, 0.039186280220746994, -0.1243237555027008, 0.041042301803827286, 0.04738662391901016, 0.07059665769338608, 0.033682193607091904, -0.06073226407170296, -0.014851079322397709, -0.001417152234353125, -0.02604018896818161], [-0.059783030301332474, 0.03139619901776314, 0.04052059352397919, -0.011086535640060902, 0.023441273719072342, 0.16493189334869385, 0.0973508283495903, -0.11684851348400116, 0.02451469376683235, 0.005586277227848768, 0.10586914420127869, -0.03777522221207619, -0.014696641825139523, -0.020089197903871536, -0.03722847253084183, -0.12056025862693787, 0.06407123804092407, 0.014816168695688248, 0.04330300912261009, 0.05124275013804436, -0.005511659197509289, 0.02365046739578247, -0.057523638010025024, -0.029039692133665085, -0.10131434351205826, -0.1560811996459961, -0.12768642604351044, 0.034663982689380646, -0.0259250421077013, 0.10075637698173523, -0.0009859256679192185, 0.11699461936950684, 0.10705235600471497, -0.1689799726009369, 0.0007106952252797782], [-0.0379188247025013, -0.12658584117889404, -0.018401814624667168, -0.1161244735121727, 0.015319357626140118, 0.05056433752179146, -0.02927115000784397, 0.04396834596991539, -0.10287702083587646, -0.12974712252616882, 0.06190155819058418, 0.025119634345173836, 0.03828468173742294, -0.06265084445476532, 0.10615481436252594, -0.017663540318608284, 0.08656901866197586, -0.050134532153606415, -0.08695106953382492, -0.019939547404646873, 0.018700676038861275, -0.004196797497570515, -0.02031411975622177, 0.05838954076170921, -0.09276507794857025, -0.06593398749828339, -0.014994677156209946, 0.0074863312765955925, -0.0684497281908989, 0.12846849858760834, -0.05274881049990654, -0.005372813902795315, -0.11200297623872757, -0.06987500935792923, -0.03640652447938919], [0.09686339646577835, -0.005981158930808306, 0.05371669679880142, -0.1029418334364891, 0.02739506959915161, -0.08595199882984161, 0.003778457175940275, 0.07921279966831207, 0.0073737455531954765, 0.08594942092895508, 0.025230079889297485, -0.08815896511077881, -0.04483422264456749, 0.07158537954092026, 0.06010489538311958, 0.10597510635852814, 0.0931992307305336, 0.07079609483480453, 0.03501949459314346, 0.10812284797430038, 0.08661231398582458, 0.039074789732694626, 0.034945178776979446, 0.026577293872833252, 0.019478389993309975, 0.07450030744075775, 0.038571182638406754, 0.03392647206783295, 0.019722476601600647, 0.10257209837436676, 0.03190568462014198, -0.018328247591853142, -0.11806264519691467, 0.03961559385061264, -0.06904685497283936], [0.08632822334766388, -0.03483547270298004, -0.00013154829503037035, 0.08302134275436401, -0.016412340104579926, 0.005001116078346968, 0.11455031484365463, -0.07940341532230377, 0.00906087551265955, -0.06256754696369171, 0.14395135641098022, -0.07291713356971741, 0.13889732956886292, -0.02356668747961521, -0.017956454306840897, -0.12736162543296814, -0.10805756598711014, 0.11524694412946701, -0.17131754755973816, -0.0026590158231556416, -0.056625381112098694, -0.09714123606681824, 0.0934993326663971, -0.12634077668190002, -0.07256641238927841, -0.08774132281541824, 0.10144732147455215, 0.09867870062589645, 0.06582039594650269, -0.13531669974327087, -0.09403309971094131, -0.004528045654296875, -0.10710147768259048, -0.024097932502627373, 0.019666574895381927], [-0.07223683595657349, -0.03578858822584152, 0.03724623844027519, 0.0376351960003376, 0.08249393105506897, 0.13202103972434998, 0.08078144490718842, 0.007418951019644737, 0.11178913712501526, 0.031065937131643295, -0.08113601058721542, 0.003630748949944973, 0.06808538734912872, -0.02467239275574684, -0.03713390231132507, -0.012121566571295261, 0.14498752355575562, 0.016961941495537758, -0.05332815647125244, 0.040102776139974594, -0.07299262285232544, -0.04810483008623123, 0.011058586649596691, -0.01765933819115162, -0.14326585829257965, 0.03271673992276192, -0.0710422620177269, 0.015534995123744011, -0.06876574456691742, -0.03704492002725601, -0.03143487498164177, -0.08273861557245255, -0.010045554488897324, -0.018908511847257614, -0.08169233798980713], [-0.08544634282588959, -0.03149505332112312, 0.032770510762929916, 0.031520698219537735, -0.009386619552969933, 0.04613139107823372, -0.07231371849775314, -0.0014743632636964321, 0.08920035511255264, -0.030883049592375755, -0.03761909529566765, -0.01136582437902689, -0.0108396140858531, 0.06845040619373322, -0.04029717296361923, 0.11808609217405319, -0.11805947124958038, 0.020676955580711365, 0.084358349442482, -0.12263815850019455, -0.026063190773129463, -0.04221699759364128, 0.015629658475518227, -0.06374944746494293, -0.09569244831800461, 0.20806322991847992, 0.07706958800554276, 0.12395909428596497, -0.024289119988679886, -0.06735555082559586, 0.12982149422168732, -0.18393875658512115, 0.0444859080016613, 0.3293381333351135, 0.0011716799344867468], [0.026861874386668205, -0.11891374737024307, 0.10458317399024963, -0.07641054689884186, -0.10342390090227127, 0.12176141887903214, 0.008598840795457363, -0.030364081263542175, -0.09811670333147049, 0.05041195824742317, -0.14915917813777924, 0.10445019602775574, 0.052183765918016434, 0.057082489132881165, -0.08540936559438705, 0.045216117054224014, -0.0030562381725758314, 0.11169896274805069, -0.02093368209898472, 0.033447638154029846, -0.07844925671815872, 0.011556623503565788, 0.09616469591856003, -0.08118492364883423, -0.0948249101638794, -0.053179074078798294, -0.12211357802152634, 0.16747429966926575, -0.09854739159345627, 0.044933922588825226, -0.12303494662046432, 0.05612652748823166, 0.05841460078954697, 0.028811322525143623, 0.012386428192257881], [0.061017170548439026, 0.05882303789258003, -0.015737753361463547, -0.0357944630086422, -0.05919855833053589, -0.10769572108983994, -0.06312674283981323, 0.006577693857252598, -0.05605979636311531, 0.03477327525615692, 0.010122910141944885, -0.06566965579986572, 0.010095314122736454, -0.05819335952401161, 0.0035047216806560755, -0.003237831871956587, -0.11322886496782303, -0.09583824127912521, 0.11843950301408768, 0.09679576009511948, 0.04617655649781227, 0.05815007537603378, 0.038727886974811554, 0.010120335966348648, -0.06829186528921127, -0.0025284914299845695, -0.06721799820661545, 0.07561996579170227, 0.02081473357975483, 0.05603323131799698, -0.039612915366888046, -0.0352550745010376, 0.057855311781167984, -0.002713957568630576, -0.01093391701579094], [-0.022789079695940018, -0.037024445831775665, 0.10404036939144135, -0.028904318809509277, 0.004104624968022108, 0.012155108153820038, -0.05904649198055267, 0.019454052671790123, -0.01635909453034401, 0.03190354257822037, -0.054814305156469345, -0.12243331223726273, -0.06685616075992584, -0.011080192402005196, -0.06574003398418427, 0.1454557627439499, -0.08448679000139236, 0.14172445237636566, 0.059553518891334534, -0.019378693774342537, -0.02038681134581566, -0.046533990651369095, -0.015196249820291996, -0.1430862843990326, 0.10382940620183945, 0.13073693215847015, 0.1427580565214157, 0.012032859027385712, 0.050245948135852814, -0.03462016582489014, 0.04455878585577011, -0.11236800998449326, -0.049038685858249664, 0.020625822246074677, -0.08620395511388779], [0.02466697059571743, -0.06244349852204323, 0.023624679073691368, -0.013217166066169739, -0.007272692862898111, -0.03682924434542656, 0.03682246059179306, -0.05501372739672661, 0.09669031202793121, 0.07333876192569733, 0.005239297170192003, -0.047537911683321, -0.0612013079226017, 0.11091352254152298, 0.048135846853256226, -0.023029258474707603, 0.10864990949630737, 0.01622120477259159, 0.08314414322376251, -0.06186254695057869, -0.10497144609689713, 0.01796060986816883, -0.026789195835590363, -0.05777042359113693, 0.032984938472509384, 0.08853339403867722, -0.10215382277965546, 0.04346415400505066, 0.010743805207312107, 0.12174446135759354, -0.0358305461704731, 0.0043393648229539394, 0.029159631580114365, -0.08408787101507187, -0.009714925661683083], [0.010648478753864765, -0.06284242123365402, -0.06498180329799652, 0.008604883216321468, -0.019762985408306122, -0.022146593779325485, -0.0027249581180512905, -0.03284851089119911, -0.06294947117567062, 0.1468850076198578, 0.09636998176574707, 0.0871431827545166, 0.09159963577985764, 0.07428660988807678, 0.08244694769382477, 0.1030259057879448, 0.042241550981998444, -0.09145224094390869, 0.06565839797258377, 0.05783841013908386, 0.09004257619380951, -0.11780831217765808, 0.09467652440071106, -0.030703820288181305, 0.018094545230269432, 0.07959648221731186, -0.00577146653085947, 0.0002410432934993878, 0.06614378839731216, 0.009973711334168911, -0.08173475414514542, 0.036267559975385666, 0.042118411511182785, -0.0383567251265049, -0.00026221261941827834], [0.102225162088871, -0.09158486127853394, -0.1352335512638092, 0.13318192958831787, -0.11888806521892548, -0.061128564178943634, 0.13029499351978302, 0.02037867158651352, 0.06089748069643974, -0.03884902223944664, 0.02310366742312908, 0.058250486850738525, 0.15193241834640503, -0.01173906959593296, 0.032313261181116104, 0.06440005451440811, -0.09423065185546875, -0.138844832777977, 0.03678341209888458, 0.12481312453746796, -0.03567451983690262, 0.044723931699991226, -0.11905798316001892, 0.008715378120541573, -0.07371769100427628, 0.0003598978219088167, -0.027800269424915314, -0.07822475582361221, 0.07817330956459045, 0.10674266517162323, -0.1531727910041809, -0.15690305829048157, -0.06041773036122322, -0.04077289626002312, -0.0729748085141182], [0.014854722656309605, 0.045217305421829224, 0.039184145629405975, 0.012367953546345234, -0.1480645388364792, 0.18794791400432587, -0.059079889208078384, 0.047847289592027664, -0.10167520493268967, -0.025748275220394135, -0.048843931406736374, -0.0719749853014946, 0.13301432132720947, -0.07770437747240067, 0.12701210379600525, -0.001117286505177617, -0.049498312175273895, 0.16896489262580872, 0.06707517802715302, -0.15676963329315186, 0.15931783616542816, 0.01162814348936081, -0.025383101776242256, 0.005890762433409691, 0.05524788424372673, 0.04770559445023537, 0.05279941111803055, 0.029774829745292664, -0.07302244752645493, 0.06522704660892487, -0.08973860740661621, -0.04178979620337486, -0.041074495762586594, -0.04914143308997154, -0.03479490801692009], [0.06564074754714966, -0.08163668215274811, 0.12065620720386505, 0.07979866117238998, -0.04864487424492836, -0.07607223093509674, -0.17163284122943878, -0.1135900542140007, -0.137537881731987, -0.049128126353025436, -0.039707209914922714, -0.0786963701248169, 0.08939655125141144, 0.12524208426475525, 0.08353205025196075, 0.13143213093280792, 0.0250584464520216, 0.12701040506362915, -0.016917837783694267, -0.03620586916804314, -0.028738753870129585, -0.10656631737947464, 0.14218242466449738, -0.07510195672512054, -0.046459902077913284, -0.021302884444594383, -0.12430454790592194, -0.13226266205310822, 0.06797455996274948, -0.13751433789730072, -0.11734437197446823, 0.12474893033504486, -0.09458713233470917, 0.05227135494351387, 0.10487157106399536], [0.020738350227475166, 0.08803829550743103, -0.057377491146326065, 0.06042507290840149, -0.03817342221736908, 0.03635663911700249, -0.07044582068920135, -0.09593068808317184, -0.10238148272037506, -0.08978217840194702, -0.05384911596775055, -0.008022582158446312, -0.047710567712783813, 0.07473074644804001, -0.012591217644512653, 0.052534911781549454, 0.025719260796904564, -0.0294222179800272, 0.02581116184592247, 0.07134409993886948, -0.030768048018217087, 0.0846785381436348, -0.01616837829351425, 0.032228123396635056, 0.026991143822669983, 0.08592493832111359, -0.03470587357878685, -0.002896778518334031, 0.002340159844607115, -0.09066904336214066, -0.07706215232610703, 0.06208919733762741, 0.03452830761671066, -0.09844554960727692, 0.02273050881922245], [0.05531557649374008, 0.017790814861655235, 0.047980327159166336, -0.0017975487280637026, -0.15420441329479218, 0.13875168561935425, 0.0670551061630249, -0.062138091772794724, -0.027986541390419006, 0.047389883548021317, -0.1442824751138687, 0.10426053404808044, 0.007905657403171062, 0.06613970547914505, -0.00396168464794755, -0.08475581556558609, -0.15304486453533173, -0.05464180186390877, 0.041863031685352325, -0.05040643364191055, 0.0023803520016372204, 0.08063055574893951, -0.0372513122856617, -0.03625001758337021, 0.06015899404883385, -0.11662700027227402, -0.008743423037230968, -0.030831919983029366, -0.0730864405632019, 0.08594413101673126, 0.041851550340652466, -0.13406163454055786, 0.052796870470047, -0.07957020401954651, -0.04972197860479355], [-0.08429089188575745, -0.09971819818019867, 0.05832599103450775, 0.0004066465771757066, -0.1362272948026657, 0.07326950877904892, -0.022251935675740242, 0.02280382439494133, -0.09545700252056122, 0.10149624943733215, 0.013292694464325905, 0.020458007231354713, -0.1059904396533966, 0.1078239232301712, 0.009603806771337986, -0.08068054914474487, 0.13623036444187164, -0.14847774803638458, -0.1631058007478714, -0.0895673930644989, -0.06317021697759628, 0.08226696401834488, 0.08554921299219131, -0.14421561360359192, -0.10104821622371674, 0.04405417665839195, 0.1463477462530136, -0.032137274742126465, -0.16039371490478516, 0.10058153420686722, 0.03974725306034088, 0.11703123897314072, 0.025819171220064163, -0.1158781424164772, 0.059418994933366776], [0.05597635358572006, 0.04680952802300453, 0.1146525964140892, -0.0020840067882090807, 0.09045512974262238, 0.06445585191249847, -0.0010097110643982887, -0.04590626806020737, 0.054819438606500626, 0.07690822333097458, -0.030470075085759163, 0.06010252609848976, -0.025797082111239433, 0.07817372679710388, -0.01794304884970188, 0.07921703904867172, -0.0773007944226265, 0.023802468553185463, -0.04218951240181923, -0.06333591043949127, -0.10164182633161545, -0.1001841202378273, -0.04380710422992706, 0.03686710074543953, 0.04671061038970947, 0.018450886011123657, 0.11430200189352036, 0.024663547053933144, -0.026301609352231026, 0.07879840582609177, 0.026964042335748672, 0.03919143229722977, -0.10390489548444748, 0.03186837211251259, 0.024880478158593178], [0.039969153702259064, -0.1119416207075119, 0.04649018123745918, -0.022812234237790108, -0.026994828134775162, 0.044083595275878906, -0.016095684841275215, 0.005060912575572729, 0.011936030350625515, 0.01909545063972473, -0.051740072667598724, 0.11226792633533478, 0.05116872116923332, -0.15050092339515686, -0.03957637399435043, 0.07626761496067047, 0.11645957827568054, 0.01375303603708744, 0.03828839585185051, -0.026059789583086967, -0.06892898678779602, -0.049154508858919144, 0.0069024888798594475, -0.12762565910816193, -0.10807931423187256, -0.1367824822664261, 0.0038671353831887245, 0.061278168112039566, -0.12678860127925873, 0.06347466260194778, -0.11407303810119629, -0.09806390851736069, -0.007313610054552555, 0.01619900017976761, 0.025774473324418068], [-0.07391625642776489, 0.06974195688962936, -0.02125346101820469, -0.05934073030948639, 0.0023402401711791754, 0.055408135056495667, -0.03295791521668434, -0.08696221560239792, -0.028850197792053223, -0.008390742354094982, -0.007253340445458889, 0.008795330300927162, -0.028029389679431915, 0.04569392278790474, 0.03134908899664879, -0.03596840426325798, -0.02088400162756443, 0.10150720179080963, 0.10474417358636856, 0.021527642384171486, 0.04462401941418648, 0.008526491932570934, 0.009548457339406013, 0.0044007934629917145, 0.08379831165075302, -0.09321383386850357, -0.06324504315853119, -0.024552663788199425, -0.01315294485539198, 0.06421942263841629, -0.09108446538448334, -0.03659684956073761, -0.049223437905311584, -0.10892631113529205, -0.009915667586028576], [0.06356515735387802, 0.08273351937532425, 0.09167976677417755, 0.1880173534154892, 0.005342144053429365, -0.1042061373591423, 0.10858247429132462, -0.07984165847301483, -0.08319196850061417, -0.0687066987156868, -0.0866980105638504, -0.09920993447303772, -0.11517102271318436, 0.05030065029859543, 0.04436662793159485, 0.013780727051198483, 0.017458373680710793, -0.07318572700023651, 0.01773638091981411, -0.09899226576089859, -0.06856530904769897, -0.08727087825536728, -0.039095111191272736, 0.10984497517347336, 0.03115621767938137, 0.019126681610941887, -0.028712322935461998, 0.06247973442077637, 0.1499246209859848, -0.09488844126462936, 0.00027420432888902724, -0.033549610525369644, 0.04043849557638168, -0.04524937644600868, -0.10345613956451416], [0.046508029103279114, 0.013881641440093517, 0.05614043027162552, -0.06625551730394363, -0.08126558363437653, 0.1115163043141365, -0.012848589569330215, -0.060109931975603104, 0.08039844036102295, 0.06951890885829926, -0.07491316646337509, 0.1090761199593544, 0.013002126477658749, 0.024241935461759567, -0.014747627079486847, 0.04392490163445473, -0.08325609564781189, -0.03239379823207855, -0.06763754785060883, 0.03314176946878433, 0.09346325695514679, 0.0014727682573720813, -0.08412700146436691, -0.06271803379058838, -0.08812396228313446, -0.07719556987285614, -0.07449077069759369, 0.037194423377513885, -0.05269017070531845, 0.08917771279811859, -0.02620578557252884, -0.07070115208625793, -0.023441238328814507, -0.07251544296741486, 0.05719432607293129], [-0.040339395403862, 0.06827694177627563, 0.04487575963139534, 0.003945806063711643, -0.06349656730890274, 0.0038947160355746746, -0.03140327334403992, -0.05795475095510483, -0.052507173269987106, 0.06541910767555237, 0.01615019515156746, -0.0421762615442276, -0.041487231850624084, -0.04478641226887703, 0.023917367681860924, 0.07069659978151321, 0.046357445418834686, -0.014465531334280968, 0.05660441890358925, 0.058461807668209076, 0.05178533121943474, -0.06056523323059082, 0.009316048584878445, 0.011549470014870167, 0.044751930981874466, 0.03996855765581131, 0.05531783401966095, 0.05828268826007843, 0.037957604974508286, 0.06345114856958389, -0.017662102356553078, -0.020450299605727196, 0.07145728915929794, 0.025742435827851295, -0.00656322343274951], [-0.045062799006700516, -0.06288287788629532, 0.05469881370663643, 0.02388780005276203, -0.012559995986521244, -0.01596103049814701, 0.07251482456922531, 0.0047588045708835125, 0.018280066549777985, -0.0598306879401207, -0.05720139294862747, -0.0037857964634895325, -0.0728360265493393, 0.04742918908596039, 0.01841391809284687, -0.08095350116491318, -0.09251846373081207, 0.02894110605120659, 0.09226635843515396, -0.06812554597854614, -0.045978158712387085, 0.03730209171772003, -0.09218507260084152, -0.033106159418821335, -0.09249527752399445, 0.06986352801322937, 0.07743743807077408, 0.03673355653882027, 0.07474740594625473, -0.06678633391857147, 0.07505331188440323, -0.05088770389556885, 0.08259047567844391, 0.2323381006717682, 0.057405874133110046], [-0.03920883312821388, 0.07208679616451263, -0.06874622404575348, -0.0843740776181221, -0.034116972237825394, -0.0031875111162662506, 0.02227475680410862, -0.11194011569023132, -0.0012826168676838279, -0.12080617249011993, 0.06417764723300934, 0.018094491213560104, 0.0852547213435173, -0.054130472242832184, -0.0810006707906723, -0.05552554130554199, 0.05114256218075752, 0.05737366899847984, 0.060611966997385025, 0.0029602916911244392, -0.03079381212592125, 0.03325063735246658, 0.07361060380935669, -0.03244227543473244, -0.04872860014438629, -0.03531905636191368, -0.02737048827111721, -0.007408135570585728, 0.02642844244837761, 0.019999481737613678, -0.08019262552261353, 0.06321993470191956, 0.018020587041974068, -0.06972169876098633, 0.0040587871335446835], [-0.06191541627049446, -0.08961531519889832, -0.011650443077087402, -0.11376704275608063, -0.043459802865982056, -0.09193729609251022, -0.04223254323005676, 0.031418416649103165, -0.060103680938482285, 0.021053854376077652, -0.05645080283284187, 0.04809634014964104, 0.10227961093187332, 0.012395565398037434, -0.1303418129682541, -0.03133220225572586, -0.1096830740571022, 0.04660496860742569, 0.08998463302850723, 0.012118588201701641, -0.06034707650542259, 0.007522501051425934, -0.016780326142907143, 0.07590825110673904, 0.006871876306831837, -0.04718806967139244, -0.010538262315094471, 0.018934885039925575, 0.061953380703926086, -0.0374222956597805, -0.021538076922297478, -0.06176185607910156, -0.08265433460474014, -0.09628558903932571, 0.03654589131474495], [0.018844494596123695, 0.01492488943040371, 0.014489551074802876, 0.05660875886678696, -0.045320797711610794, 0.12853099405765533, -0.030255768448114395, -0.020126333460211754, -0.023889711126685143, -0.08025538921356201, -0.01062047854065895, 0.03854293003678322, -0.007226928137242794, 0.11491068452596664, -0.0536004863679409, 0.008254850283265114, 0.0915592759847641, 0.0721546933054924, -0.08273646980524063, -0.08307937532663345, 0.09143087267875671, 0.028180191293358803, 0.03850560635328293, 0.11127322912216187, -0.09287887066602707, 0.04502132534980774, 0.052761420607566833, 0.07810313999652863, -0.04452791064977646, 0.017898688092827797, 0.0965638980269432, 0.014716226607561111, 0.06104479357600212, 0.2441830337047577, 0.01726401410996914], [0.0836707204580307, -0.08144786953926086, -0.006552486680448055, -0.1036762222647667, -0.12097490578889847, -0.056789446622133255, -0.02189747989177704, 0.07232487946748734, -0.017060602083802223, 0.05356259271502495, -0.06939415633678436, -0.05558299645781517, 0.055240895599126816, 0.004695565439760685, 0.08438973128795624, 0.07581453770399094, 0.03705013543367386, -0.15893879532814026, 0.17798979580402374, 0.0238228477537632, 0.14989201724529266, 0.08587343990802765, -0.0693955346941948, -0.135899156332016, -0.11712997406721115, 0.044727884232997894, -0.08447524160146713, -0.028852269053459167, 0.060422688722610474, -0.03952386602759361, 0.04929623007774353, 0.0480533242225647, 0.08266106247901917, 0.053444284945726395, -0.09888218343257904], [0.039257388561964035, -0.006296114530414343, 0.031607478857040405, -0.15381814539432526, -0.03023507446050644, 0.027446182444691658, 0.005665219388902187, 0.06044677644968033, 0.014308205805718899, 0.1085139587521553, -0.09750369936227798, 0.0965772271156311, -0.10092852264642715, -0.05681278184056282, 0.026793859899044037, -0.08482854068279266, 0.0917137935757637, 0.003732458921149373, -0.0016443009953945875, 0.02068565972149372, -0.023220697417855263, 0.11073349416255951, 0.0023350976407527924, -0.002191602485254407, 0.024296460673213005, 0.09898002445697784, -0.09366381168365479, 0.05133288726210594, -0.05620696395635605, 0.05529326945543289, 0.0678228884935379, -0.16065292060375214, 0.07835367321968079, -0.06509469449520111, -0.01941544935107231], [-0.004145387094467878, -0.02507937327027321, 0.032112982124090195, 0.11974364519119263, 0.07010168582201004, 0.10543429106473923, 0.04188266396522522, 0.04752736911177635, -0.053292542695999146, 0.03791448101401329, 0.00010565228149062023, 0.03927640989422798, 0.0420122928917408, -0.015277660451829433, -0.00374105223454535, -0.03675612062215805, -0.06372103095054626, -0.04615332558751106, 0.03611263632774353, -0.06589256227016449, -0.10408314317464828, 0.04119432717561722, 0.07415131479501724, 0.08562271296977997, 0.056389156728982925, 0.05160621181130409, 0.0997067391872406, -0.034539371728897095, 0.029757415875792503, 0.07719291001558304, 0.011605857871472836, 0.026047484949231148, -0.017739204689860344, -0.01743452623486519, 0.042386509478092194]], "b1": [-0.046651244163513184, 0.025570236146450043, 0.1156868040561676, -0.003617025213316083, -0.021059321239590645, 0.08051817119121552, -0.09468496590852737, 0.08711414039134979, 0.12030936032533646, 0.03084862045943737, -0.12617981433868408, 0.11870239675045013, -0.06978721171617508, -0.04554314166307449, 0.03141578286886215, -0.03255954384803772, 0.08890579640865326, -0.10986535251140594, 0.08760013431310654, -0.0066085332073271275, 0.0690428763628006, -0.05290384963154793, -0.1370331346988678, -0.08209322392940521, -0.003878580406308174, 0.058860111981630325, -0.07600225508213043, 0.11745686084032059, -0.034310877323150635, 0.03346458077430725, -0.09506832808256149, -0.04186414182186127, -0.01691918633878231, 0.06760648638010025, 0.11913459002971649, 0.004076629411429167, -0.09430570900440216, -0.12786045670509338, 0.005404740571975708, 0.07142897695302963, 0.06535294651985168, 0.040431685745716095, 0.023371975868940353, 0.0929008424282074, 0.05919002741575241, -0.07801251858472824, 0.018233971670269966, -0.07662184536457062, -0.10129060596227646, 0.007722195703536272, -0.17703686654567719, -0.05187821388244629, -0.2361915558576584, -0.0755300223827362, 0.03491649404168129, -0.13159582018852234, 0.07321842014789581, -0.14879357814788818, 0.0021205879747867584, 0.0911543071269989, -0.07620882987976074, -0.06919842213392258, 0.0342407189309597, 0.011792238801717758, -0.07453393191099167, 0.12726014852523804, 0.16663314402103424, -0.005062949378043413, 0.11092822253704071, 0.02355247549712658, 0.022969096899032593, -0.18273957073688507, -0.05773758143186569, 0.07356133311986923, -0.030664287507534027, -0.05043237283825874, 0.09372531622648239, -0.13342010974884033, 0.012655992060899734, -0.07323088496923447, -0.07023248076438904, -0.004612341057509184, -0.08570221811532974, 0.09815825521945953, -0.09931701421737671, -0.003775321412831545, 0.0759386345744133, 0.07867930829524994, 0.08136110007762909, -0.12159225344657898, -0.005002230405807495, -0.05845300853252411, -0.08709171414375305, -0.06073785573244095, 0.06355566531419754, 0.003310056636109948], "W2": [[-0.003592744702473283, 0.04191450774669647, -0.04224878177046776, -0.008777330629527569, -0.041055526584386826, -0.03444654121994972, -0.007358689326792955, 0.06723051518201828, -0.03808993473649025, -0.024651149287819862, 0.06990844756364822, 0.018394703045487404, -0.0023856579791754484, 0.0332157164812088, 0.02218100242316723, 0.06877755373716354, 0.041751883924007416, 0.02912777289748192, -0.02328396402299404, 0.05034257471561432, 0.01904943957924843, 0.0017415547044947743, 0.18068787455558777, 0.037393830716609955, 0.04895338416099548, 0.03128431364893913, 0.00959879532456398, 0.07289150357246399, 0.00744065223261714, 0.07684178650379181, 0.057603199034929276, 0.01131277997046709, 0.024470165371894836, 0.06333787739276886, 0.074868343770504, 0.06953752040863037, 0.05417853593826294, 0.037933606654405594, -0.03085719607770443, 0.02897229976952076, 0.05678864195942879, -0.09038695693016052, 0.06235602870583534, 0.08453798294067383, -0.04864662513136864, 0.023371152579784393, 0.03839190676808357, -0.06828856468200684, 0.044408682733774185, 0.017059100791811943, -0.04178660735487938, 0.05373659357428551, 0.03414350003004074, 0.045505329966545105, -0.016150128096342087, 0.06830281764268875, 0.015119712799787521, -0.00723855197429657, 0.10223352164030075, 0.015610568225383759, -0.0538337342441082, 0.07181847095489502, -0.0015008813934400678, 0.04133105278015137, -0.00802395585924387, -0.02468891814351082, -0.04418334364891052, 0.06292222440242767, -0.013220827095210552, -0.0007119813235476613, -0.04465346410870552, 0.018334468826651573, 0.06791070103645325, 0.00673598051071167, 0.013432788662612438, -0.0029039334040135145, -0.04626286402344704, 0.05876287817955017, 0.022246379405260086, 0.06234465911984444, 0.01922844536602497, 0.05093924328684807, 0.05812123790383339, -0.018625451251864433, 0.06140413135290146, -0.002816329477354884, 0.039073262363672256, 0.0927724689245224, -0.00994815118610859, 0.02560628391802311, 0.05177689343690872, -0.020733926445245743, 0.03630741313099861, 0.09603362530469894, 0.013095080852508545, -0.012991131283342838], [0.03521232679486275, 0.04693743586540222, 0.01921788416802883, 0.06878422945737839, 0.04806533083319664, -0.00904599018394947, -0.0599357932806015, -0.011067090556025505, 0.08670687675476074, 0.07566452026367188, -0.024778882041573524, -0.03470584377646446, 7.311160516110249e-06, -0.027028288692235947, 0.028707120567560196, -0.07169362157583237, 0.052444200962781906, 0.017503416165709496, -0.03562598302960396, -0.06451260298490524, 0.014504866674542427, -0.01780584454536438, -0.03585619851946831, 0.007424924522638321, 0.03414677083492279, -0.07647543400526047, 0.018198536708950996, 0.048744622617959976, 0.006467146798968315, -0.017988186329603195, 0.03704629838466644, 0.032206278294324875, 0.026338819414377213, -0.03891833499073982, -0.006544740870594978, -0.06058904156088829, -0.057037316262722015, -0.06952530145645142, -0.024927619844675064, -0.0641394704580307, -0.011119293048977852, 0.01605299301445484, -0.003435875289142132, 0.07120827585458755, -0.04448673874139786, 0.00725922454148531, 0.040393225848674774, -0.009163870476186275, 0.07013605535030365, -0.03903131186962128, -0.04623512923717499, 0.037990130484104156, 0.013441536575555801, 0.002416081726551056, -0.039661649614572525, -0.01935473643243313, -0.08526409417390823, 0.07725612819194794, 0.0032645417377352715, -0.04471251741051674, 0.013822417706251144, -0.028292426839470863, 0.04189615696668625, -0.05509885773062706, -0.05696775019168854, 0.014490417204797268, 0.03878622129559517, -0.017368298023939133, 0.03315066546201706, -0.03137264400720596, -0.050388939678668976, 0.011217597872018814, 0.011725385673344135, 0.04013214260339737, 0.015778232365846634, -0.02412290684878826, 0.039670731872320175, 0.07071678340435028, 0.020349377766251564, -0.08261783421039581, -0.019954165443778038, 0.03815225884318352, 0.02728898823261261, -0.046902675181627274, -0.014939714223146439, 0.042845021933317184, 0.0484997034072876, 0.10318291187286377, 0.02805619314312935, -0.041103482246398926, -0.008669333532452583, 0.00806382019072771, -0.05825819447636604, 0.020533978939056396, -0.02997414767742157, 0.0392843522131443], [0.060677189379930496, 0.011590338312089443, 0.08452987670898438, 0.0792812928557396, -0.03043317422270775, 0.02318713627755642, -0.07623493671417236, -0.04927785322070122, -0.09041358530521393, 0.005978947971016169, 0.04523966833949089, 0.0007171992328949273, 0.10702292621135712, 0.0029101248364895582, 0.07454236596822739, -0.022727886214852333, -0.11085659265518188, -0.07066022604703903, -0.05422819405794144, -0.024446722120046616, -0.04543141648173332, -0.0017986975144594908, -0.09738481789827347, -0.0061286925338208675, 0.08480344712734222, 0.035574186593294144, 0.054248400032520294, -0.021349681541323662, 0.09964651614427567, 0.08023486286401749, -0.05605701357126236, 0.059021614491939545, -0.03972804173827171, 0.018308954313397408, -0.009932754561305046, 0.021590879186987877, 0.0031994166783988476, -0.07491326332092285, -0.032340385019779205, 0.10610808432102203, -0.13296519219875336, 0.0515337735414505, -0.034874167293310165, -0.01649547554552555, -0.01902606524527073, 0.12840332090854645, 0.09471765160560608, 0.017679188400506973, -0.030171601101756096, -0.01150985062122345, 0.15308381617069244, -0.13969220221042633, 0.15469065308570862, -0.05939391627907753, 0.030995257198810577, 0.005505353212356567, -0.10146215558052063, 0.04585803672671318, -0.07735547423362732, 0.022572478279471397, -0.05670427531003952, -0.06675353646278381, -0.04861839860677719, 0.021264418959617615, 0.0291748084127903, 0.08938606828451157, 0.029835209250450134, -0.056690461933612823, 0.04251352697610855, -0.024091871455311775, 0.06411770731210709, 4.062796051584883e-06, -0.013979715295135975, -0.06100548058748245, -0.01663617417216301, 0.05592167377471924, -0.11112277954816818, 0.01249525137245655, -0.07938935607671738, -0.03331562876701355, 0.07052342593669891, 0.0680157020688057, 0.026444291695952415, -0.04931025952100754, 0.11857983469963074, 0.10011645406484604, -0.09753472357988358, -0.056217990815639496, 0.02152971364557743, 0.039011966437101364, 0.05245763808488846, 0.03840864077210426, -0.009772822260856628, 0.019932527095079422, -0.01932597905397415, -0.01244398020207882], [0.02211170829832554, -0.0672040656208992, -0.007447910960763693, -0.08706798404455185, 0.016399256885051727, 0.012096446007490158, 0.015176158398389816, -0.08346087485551834, 0.03830414265394211, 0.09049134701490402, -0.0924881100654602, -0.05379437282681465, 0.03358258306980133, -0.10064122080802917, 0.02476273477077484, -0.02345249243080616, 0.019498316571116447, -0.08852791786193848, 0.021284116432070732, -0.031713470816612244, 0.04126254841685295, -0.08461549878120422, 0.03270433843135834, -0.07823263108730316, -0.08268429338932037, -0.0395335927605629, 0.03030724823474884, 0.007792116608470678, -0.08478785306215286, 0.061947304755449295, -0.026609212160110474, 0.06369674205780029, 0.008271556347608566, 0.000904761953279376, 0.014879504218697548, -0.057536445558071136, -0.05221737548708916, 0.113021619617939, 0.014068448916077614, -0.016485940665006638, 0.022525645792484283, -0.006594195496290922, 0.0744163915514946, -0.04309028014540672, 0.044824179261922836, -0.056178946048021317, 0.0039396993815898895, -0.040871310979127884, -0.018398767337203026, 0.07666546106338501, -0.12292099744081497, 0.018007410690188408, -0.07278131693601608, 0.07213958352804184, -0.06532704085111618, 0.04571279510855675, -0.020075583830475807, -0.010144335217773914, 0.0429118350148201, -0.015629759058356285, -0.051509395241737366, 0.04412014037370682, -0.052970271557569504, 0.058864861726760864, 0.03806431218981743, 0.04491066932678223, -0.06432943791151047, 0.022893264889717102, 0.06272178888320923, 0.04124574735760689, -0.04478466883301735, -0.13972654938697815, -0.01199000421911478, 0.004159434232860804, -0.01296126376837492, 0.03388272225856781, 0.09423989802598953, 0.016940278932452202, -0.009306111373007298, 0.03543337434530258, -0.06791097670793533, 0.025456970557570457, 0.02690299227833748, 0.038867998868227005, 0.008172238245606422, 0.06819378584623337, -0.012736589647829533, 0.08310491591691971, -0.026812385767698288, 0.012648142874240875, -0.03887747973203659, -0.04751758277416229, -0.034006018191576004, -0.038468942046165466, -0.08001352101564407, 0.006001459900289774], [0.015064047649502754, -0.010700933635234833, -0.0038444327656179667, 0.08063412457704544, 0.1150311604142189, -0.002600929234176874, -0.04898262396454811, 0.03368891403079033, -0.019603801891207695, -0.07915591448545456, 0.04694370925426483, 0.009516545571386814, 0.09132282435894012, 0.056898027658462524, 0.07158573716878891, -0.03104313462972641, -0.0009175525628961623, -0.028219209983944893, -0.031756822019815445, 0.05244860425591469, -0.08860326558351517, 0.061883389949798584, -0.14241580665111542, 0.11690152436494827, -0.09254568815231323, 0.03411105275154114, -0.005747419316321611, 0.05883539095520973, -0.05328305810689926, -0.007745586801320314, -0.00618748227134347, 0.04403568059206009, 0.00588267995044589, 0.06149854138493538, -0.05371447280049324, 0.00038351930561475456, 0.019283683970570564, -0.03969490900635719, 0.00034055690048262477, -0.045263197273015976, 0.030244195833802223, -0.009328252635896206, 0.02293047122657299, 0.016350191086530685, 0.05472586303949356, 0.08715660870075226, -0.07578157633543015, 0.0390644446015358, 0.031663212925195694, 0.06206876412034035, 0.046638552099466324, -0.08282626420259476, 0.10324949026107788, 0.05397465080022812, 0.004278369713574648, -0.10862156748771667, -0.052177101373672485, -0.02730477601289749, -0.016527090221643448, -0.025935065001249313, -0.018904944881796837, 0.07877268642187119, -0.005841968581080437, -0.08608533442020416, 0.04830707609653473, -0.07899019122123718, 0.02546539530158043, -0.013921370729804039, 0.0781778022646904, -0.047834549099206924, 0.011847245506942272, 0.10618843138217926, -0.01381017453968525, 0.07369516044855118, 0.011007591150701046, 0.017899328842759132, -0.03443778678774834, -0.055378418415784836, 0.02675006538629532, 0.005736296530812979, 0.05588800087571144, -0.007493956945836544, -0.08800424635410309, 0.008121488615870476, 0.016523847356438637, 0.006332674063742161, -0.09413036704063416, -0.029667437076568604, -0.06476648896932602, -0.019865823909640312, -0.03123093768954277, -0.05101904273033142, 0.09205692261457443, 0.07294850051403046, 0.07103452831506729, -0.05228356644511223], [0.016648437827825546, 0.04108245670795441, -0.020698171108961105, 0.0119936503469944, -0.030260620638728142, 0.029328467324376106, 0.02932271733880043, 0.0006358984974212945, 0.031039346009492874, 0.01765151508152485, 0.045720815658569336, 0.02160004712641239, -0.01764935813844204, -0.023003607988357544, -0.0023924352135509253, 0.021200448274612427, 0.009415085427463055, 0.024761617183685303, -0.0037266607396304607, 0.004465165548026562, 0.049257613718509674, -0.01091847661882639, 0.044244080781936646, -0.03530220314860344, 0.018288735300302505, 0.051303453743457794, -0.0072914185002446175, 0.041461702436208725, -0.022919949144124985, 0.004829305224120617, 0.02079729735851288, -0.013642397709190845, 0.03455830737948418, -0.008692952804267406, 0.009754308499395847, 0.020381441339850426, 0.02126236818730831, 0.01580958440899849, -0.009070813655853271, 0.012571832165122032, 0.01829538494348526, -0.032737549394369125, 0.0167686864733696, 0.02686840295791626, -0.01087441761046648, -0.002172045409679413, 0.0064566973596811295, -0.0012131375260651112, 0.005535387434065342, 0.002078287536278367, -0.012424372136592865, -0.0016155906487256289, 0.01581803895533085, -0.012739068828523159, -0.0024076763074845076, 0.020124835893511772, -0.04082786664366722, 0.03198473900556564, 0.02050037309527397, 0.00512046879157424, 0.00968790240585804, 0.04744170978665352, 0.010812794789671898, 0.028017420321702957, -0.021925296634435654, -0.01041561271995306, -0.014780688099563122, 0.031453073024749756, 0.015687519684433937, -0.025990670546889305, -0.0037208236753940582, 0.04631311446428299, -0.0003116526349913329, 0.005709544755518436, -0.0163364689797163, -0.01409961562603712, 0.033437423408031464, 0.042140327394008636, -0.027732890099287033, -0.03583475202322006, -0.0016801974270492792, 0.005269163753837347, 0.021708328276872635, -0.0025367389898747206, 0.014743639156222343, -0.015167994424700737, 0.0005462910630740225, -0.011456165462732315, -0.011654351837933064, 0.0016993411118164659, 0.010174599476158619, -0.012804633937776089, -0.009638572111725807, 0.020253490656614304, 0.020822793245315552, 0.01197584718465805], [-0.08644253760576248, 0.06679078191518784, 0.008985458873212337, -0.03973957896232605, -0.1423400193452835, 0.0703321322798729, -0.07291806489229202, -0.031963273882865906, -0.04014195129275322, 0.0798322781920433, -0.09580690413713455, -0.08900891244411469, 0.05472173914313316, -0.051500141620635986, -0.0022400417365133762, 0.03934726491570473, 0.05364052578806877, 0.03669623285531998, -0.013903223909437656, 0.07987013459205627, 0.04855189472436905, -0.07130421698093414, 0.011696252971887589, -0.021420728415250778, -0.016096096485853195, -0.04166879504919052, 0.020059503614902496, 0.017543980851769447, -0.08048601448535919, -0.0337173230946064, 0.03688182309269905, -0.06098499894142151, 0.043015021830797195, 0.056460700929164886, 0.03798520565032959, -0.009949398227036, 0.07273642718791962, -0.007935510948300362, 0.03167975693941116, 0.06553799659013748, 0.0998229905962944, -0.008385756053030491, -0.06396710127592087, -0.0402018204331398, 0.07227620482444763, -0.14247138798236847, -0.052020881325006485, 0.02275140769779682, -0.0443146787583828, 0.04669694975018501, -0.047668322920799255, 0.07261787354946136, -0.12708736956119537, -0.07129726558923721, -0.07491928339004517, -0.07309295982122421, 0.017264021560549736, -0.04325227066874504, 0.06035345420241356, -0.07601191848516464, -0.04391633719205856, -0.026439085602760315, -0.014085892587900162, 0.019506094977259636, -0.04265729337930679, 0.0668955147266388, -0.020612971857190132, -0.038957446813583374, 0.06207175925374031, 0.09412264823913574, 0.07109972834587097, -0.07919716089963913, -0.051732759922742844, 0.06005585193634033, -0.028871508315205574, -0.06482148170471191, 0.02526053600013256, -0.06373552232980728, 0.05109018459916115, 0.021090928465127945, 0.03656475991010666, 0.03332696482539177, -0.058185067027807236, 0.050551146268844604, -0.02448757365345955, -0.005312292370945215, -0.11303004622459412, 0.0064116837456822395, 0.07813336700201035, -0.09222029894590378, -0.056901510804891586, -0.025074979290366173, -0.05759888514876366, -0.029973359778523445, -0.12988415360450745, 0.0001317046844633296], [0.03808552771806717, -0.08174454420804977, -0.06423656642436981, 0.07560843229293823, -0.0361643023788929, -0.036744873970746994, 0.019556431099772453, -0.08096785098314285, -0.043179724365472794, -0.061320710927248, -0.017962587997317314, 0.012591254897415638, 0.08912865072488785, -0.0903497189283371, -0.038449373096227646, 0.028529133647680283, -0.08534800261259079, 0.11910173296928406, 0.035869769752025604, 0.0890514999628067, -0.024494117125868797, 0.024792218580842018, 0.07745730131864548, -0.010257214307785034, 0.0023048443254083395, 0.04930618405342102, 0.03047410026192665, 0.017013901844620705, -0.03220490738749504, -0.030974622815847397, -0.053965337574481964, 0.028442751616239548, 0.01624501869082451, -0.005112322513014078, 0.06969500333070755, 0.08087839186191559, 0.052595168352127075, 0.14268076419830322, 0.09049084782600403, -0.03087647818028927, 0.028636347502470016, -0.021423717960715294, -0.06969096511602402, 0.018562795594334602, 0.01138954609632492, -0.008077318780124187, 0.02636163868010044, -0.05863349139690399, -0.047814786434173584, -0.011343993246555328, -0.012731684371829033, 0.11389374732971191, 0.016870243474841118, 0.03739847615361214, 0.04487360268831253, 0.07753990590572357, 0.07514560967683792, 0.07723405957221985, 0.011129685677587986, 0.020980294793844223, -0.05209784209728241, 0.0008840252412483096, 0.027379650622606277, -0.008692541159689426, 0.01255414541810751, -0.09092885255813599, -0.04584818333387375, 8.241328032454476e-05, 0.013558217324316502, -0.10397237539291382, -0.03675828501582146, 0.047140493988990784, -0.076817087829113, -0.013553737662732601, 0.03859861195087433, -0.06055815890431404, 0.009035946801304817, 0.024608951061964035, -0.06942295283079147, 0.07049423456192017, -0.03750358149409294, 0.060027822852134705, 0.030714547261595726, -0.051414936780929565, -0.05670676752924919, -0.04913981631398201, 0.05325697362422943, -0.01168271154165268, -0.03381520137190819, 0.05120342597365379, -0.06382708996534348, -0.043569035828113556, 0.007539886981248856, 0.06375251710414886, -0.020364854484796524, -0.016537697985768318], [0.047691043466329575, -0.08340437710285187, -0.01642589457333088, 0.05329063907265663, 0.030209261924028397, 0.0388106144964695, -0.07079935818910599, 0.0046555702574551105, -0.056097667664289474, -0.07798385620117188, 0.03426177054643631, -0.09530185163021088, 0.10176734626293182, 0.0926361009478569, 0.05590028315782547, -0.06609492003917694, -0.011219179257750511, -0.09439406543970108, -0.03586995601654053, -0.025268856436014175, -0.04373957961797714, -0.006962926592677832, -0.09792152047157288, 0.09757370501756668, 0.07051296532154083, -0.0537431463599205, 0.06517242640256882, -0.023635609075427055, -0.03806751221418381, -0.06017657369375229, -0.018346283584833145, -0.023326100781559944, 0.07371795177459717, 0.01713285967707634, -0.03789881244301796, 0.06149539351463318, 0.004903710912913084, -0.06486594676971436, 0.0525808185338974, 0.11470123380422592, 0.044228922575712204, -0.037264734506607056, 0.01907309889793396, -0.04209207370877266, -0.006854670122265816, 0.09171108156442642, -0.02635672502219677, 0.0021432156208902597, -0.05390347167849541, -0.005641710013151169, 0.1039610430598259, -0.06738734990358353, 0.02748495526611805, -0.02124033495783806, 0.001233531627804041, 0.021211285144090652, -0.0646744966506958, 0.07895725220441818, 0.03702826425433159, -0.011054431088268757, 0.1032031700015068, -0.10675999522209167, 0.01926322840154171, 0.0232780110090971, 0.00042617766303010285, -0.024860583245754242, 0.08270887285470963, -0.07676046341657639, -0.05347936972975731, 0.08114068955183029, 0.01647307723760605, 0.009991376660764217, 0.07910210639238358, -0.04768040403723717, 0.09121393412351608, 0.03624676540493965, -0.06345662474632263, -0.005542062222957611, 0.0780632346868515, 0.02308328077197075, -0.032216351479291916, -0.07201912254095078, -0.09389284253120422, 0.013248708099126816, 0.025127360597252846, 0.023311257362365723, -0.06051655858755112, -0.040011756122112274, -0.03651817515492439, 0.018463989719748497, -0.03866453096270561, -0.0532212033867836, 0.10922811925411224, -0.05708418786525726, 0.06708890199661255, 0.017419368028640747], [0.08193129301071167, 0.09756012260913849, 0.06183787062764168, 0.08376967161893845, -0.09234374761581421, 0.07052811235189438, 0.034738704562187195, 0.08654572069644928, 0.05916287377476692, -0.007429391145706177, 0.04312765225768089, 0.022458219900727272, 0.021644581109285355, -0.009159965440630913, 0.05990372970700264, 0.03076799027621746, 0.12242873013019562, 0.030604014173150063, -0.012375318445265293, -0.018628539517521858, 0.11758314818143845, -0.0003333109780214727, 0.20820331573486328, -0.02931850589811802, -0.0447847954928875, 0.11558569967746735, -0.0546070896089077, 0.07035685330629349, -0.07229649275541306, 0.06641124933958054, 0.014469223096966743, -0.061424218118190765, 0.10201005637645721, -0.03905440866947174, 0.010973318479955196, 0.0677369013428688, -0.02173672430217266, 0.04222657531499863, 0.051927704364061356, -0.022150812670588493, -0.0024818286765366793, -0.1350751668214798, 0.06392759084701538, -0.048125408589839935, -0.0013048226246610284, 0.05752941593527794, -0.051206525415182114, -0.013355854898691177, 0.013391081243753433, -0.02968701347708702, 0.017771441489458084, 0.05315018072724342, -0.009262129664421082, -0.027977442368865013, -0.032468076795339584, 0.07399477809667587, -0.07689138501882553, 0.048064280301332474, 0.021845724433660507, 0.06876581162214279, 0.08755350857973099, -0.06164666265249252, 0.07982584089040756, 0.07241078466176987, -0.034530144184827805, -0.001602341653779149, -0.1226101964712143, 0.032447513192892075, 0.04203414544463158, -0.08258441835641861, -0.063609279692173, 0.0512034073472023, -0.010525388643145561, 0.014915283769369125, -0.014070970937609673, -0.057900696992874146, -0.06171594560146332, 0.08812832832336426, -0.003230223897844553, 0.00635520089417696, 0.020253615453839302, 0.06630252301692963, 0.020774055272340775, 0.01472764927893877, 0.0681385025382042, -0.0068910554982721806, 0.049632202833890915, 0.09810278564691544, -0.06262534111738205, -0.04809843376278877, 0.01615334302186966, 0.10933081060647964, -0.018093548715114594, 0.009797447361052036, 0.01626047119498253, 0.004068343434482813], [0.011885551735758781, -0.034139182418584824, -0.0749979317188263, 0.00012225640239194036, 0.06294332444667816, -0.0019607581198215485, 0.013435207307338715, -0.02483719401061535, -0.037457630038261414, -0.08168169111013412, -0.01912611909210682, -0.04399710148572922, -0.027391312643885612, -0.018436837941408157, -0.009587998501956463, -0.0516192689538002, 0.04776407778263092, 0.00534114008769393, -0.02428632602095604, 0.011782164685428143, 0.03193875029683113, 0.06311025470495224, -0.12757977843284607, 0.004673272371292114, -0.01918550208210945, -0.09906528145074844, -0.020335081964731216, -0.015313363634049892, 0.013492253609001637, -0.05608734115958214, -0.011996232904493809, -0.007318469230085611, -0.05571094527840614, -0.06017335504293442, 0.0003236503398511559, -0.02804415114223957, -0.07211342453956604, -0.16262325644493103, -0.04276229068636894, -0.010004648007452488, -0.03734616935253143, 0.00017765851225703955, -0.07558248937129974, -0.02382553741335869, -0.03456583619117737, 0.02073719911277294, -0.013051637448370457, -0.007201176602393389, -0.008577232249081135, -0.013918315060436726, 0.05840040370821953, 0.027989884838461876, -0.0022090068086981773, 0.029945196583867073, -0.03977767378091812, -0.053847454488277435, -0.05714354291558266, 0.12123873829841614, -0.12495411187410355, -0.017695479094982147, 0.047309428453445435, 0.041610199958086014, 0.05902538076043129, -0.09152204543352127, -0.040142737329006195, -0.005594716407358646, -0.04072193801403046, 0.023059211671352386, 0.02180194854736328, -0.004602421075105667, 0.012753425166010857, -0.048702970147132874, -0.03571236878633499, -0.03612661361694336, -0.023380808532238007, 0.006232146173715591, -0.026135670021176338, -0.02376057766377926, -0.08759769052267075, -0.024266812950372696, 0.04575253278017044, -0.037238508462905884, 0.0367639884352684, 0.04577123746275902, -0.007640657015144825, -0.05192578211426735, 0.05664416030049324, -0.0547960102558136, -0.0024315188638865948, -0.07931789755821228, -0.031248372048139572, -0.03022085689008236, -0.06136709451675415, -0.10071536898612976, 0.05483445152640343, 0.06899190694093704], [-0.08210008591413498, 0.05014262720942497, 0.053604286164045334, 0.053502365946769714, 0.0624023899435997, -0.025277774780988693, 0.06447922438383102, 0.05009745433926582, 0.02728177420794964, 0.06313973665237427, -0.036630693823099136, 0.010643037036061287, 0.08015944808721542, 0.09244440495967865, 0.010165425948798656, 0.04282992333173752, -0.07458098232746124, -0.012067842297255993, -0.03560402989387512, -0.06394480168819427, 0.052840325981378555, -0.012514792382717133, -0.12545204162597656, 0.028490249067544937, -0.03258703649044037, -0.05577956885099411, 0.016079675406217575, -0.011284778825938702, 0.0009281523525714874, -0.06356427818536758, -0.07110618054866791, 0.03142978623509407, 0.008594144135713577, -0.06633824855089188, 0.05279835686087608, 0.003618901828303933, -0.0588395856320858, 0.01920733042061329, 0.028314245864748955, -0.02567901462316513, -0.030564002692699432, 0.04189406707882881, 0.06741391122341156, -0.017502907663583755, 0.0191176850348711, 0.10473094135522842, -0.00014354317681863904, 0.07748537510633469, -0.024818625301122665, -0.04134956747293472, 0.1342134326696396, -0.020898832008242607, 0.1175071969628334, 0.04211300611495972, 0.0227336585521698, -0.10601133108139038, 0.048500414937734604, 0.08610396087169647, 0.006347301881760359, 0.04485355317592621, -0.05596819147467613, -0.0653834268450737, -0.03886917978525162, 0.005461743101477623, -0.0020866035483777523, -0.029740935191512108, 0.03926105797290802, -0.04503236711025238, 0.03670787811279297, 0.04237264022231102, 0.035096824169158936, 0.04042235389351845, 0.039322298020124435, -0.06862973421812057, 0.0385248176753521, -0.03838440775871277, -0.033343300223350525, -0.04796425625681877, -0.03750688210129738, -0.032569434493780136, 0.016377223655581474, -0.06253117322921753, -0.06460844725370407, -0.03366727754473686, -0.010896364226937294, 0.05676768720149994, -0.028688887134194374, 0.005186627618968487, -0.013996337540447712, 0.08836935460567474, 0.008947455324232578, 0.05951054394245148, 0.011309468187391758, -0.01683095656335354, 0.05365399271249771, -0.018282495439052582], [0.0025472166016697884, 0.03532709926366806, -0.04748360812664032, -0.06488107889890671, -0.0753234401345253, -0.013687432743608952, 0.0005581550649367273, 0.051028165966272354, 0.03855719044804573, 0.012667041271924973, 0.0016524770762771368, -0.06775548309087753, 0.04450324922800064, -0.06246393918991089, -0.03842717781662941, 0.029168715700507164, -0.03449686989188194, -0.04662824794650078, 0.0682835504412651, 0.03158596530556679, -0.02782556228339672, -0.0053711202926933765, -0.010426662862300873, -0.06250521540641785, 0.006643896922469139, -0.048395514488220215, 0.011048720218241215, 0.0555461086332798, -0.07728905975818634, 0.06197672709822655, -0.06008067727088928, 0.014431541785597801, -0.06394722312688828, 0.010768959298729897, -0.04683200269937515, -0.030546551570296288, 0.019129151478409767, 0.09541890025138855, 0.012420335784554482, 0.026341566815972328, 0.08073796331882477, 0.043098084628582, 0.012238324619829655, -0.05772754177451134, 0.010186818428337574, -0.07826138287782669, -0.01245788112282753, 0.03590591251850128, -0.05576343089342117, 0.015474858693778515, -0.08882950991392136, -0.009940452873706818, -0.08707604557275772, -0.014527012594044209, 0.03489987552165985, 0.001545844366773963, 0.044181011617183685, -0.1077713742852211, 0.01533769816160202, 0.07001761347055435, -0.061860937625169754, -0.02643684670329094, 0.028639275580644608, 0.054188549518585205, -0.01174985896795988, 0.04752252623438835, 0.02141624689102173, -0.0303022563457489, -0.02755398489534855, 0.05989827215671539, 0.036176301538944244, -0.014284437522292137, 0.013368218205869198, -0.015460602007806301, 0.011126125231385231, -0.05340488627552986, -0.0020284561906009912, -0.01932208426296711, 0.07065272331237793, -0.021556057035923004, -0.023135794326663017, -0.023336663842201233, -0.047600895166397095, 0.022784093394875526, 0.033824991434812546, -0.0343594066798687, -0.03554120287299156, 0.021026132628321648, 0.006826549768447876, -0.056756533682346344, -0.037183452397584915, -0.03350214287638664, 0.030649594962596893, -0.07160820066928864, -0.07068973779678345, 0.04956961050629616], [-0.0036741301883012056, -0.016316590830683708, -0.007305795792490244, 0.006789801176637411, 0.005479350686073303, -0.007575713563710451, -0.006546996533870697, 0.030672941356897354, 0.020694095641374588, 0.02330528199672699, 0.0032981769181787968, -0.010436061769723892, -0.014270051382482052, 0.01690702699124813, -0.0036631631664931774, -0.0018364384304732084, -0.00644649239256978, -0.004277436062693596, -0.003174778539687395, -0.0027959300205111504, -0.00580098619684577, -0.005090681836009026, -0.00944169145077467, -0.007288902997970581, -0.01662435382604599, 0.009302817285060883, 0.00836582388728857, 0.015943406149744987, 0.009608854539692402, -0.006523503456264734, -0.006347802933305502, 0.007347780279815197, -0.012380348518490791, 0.012091422453522682, -0.0011758935870602727, -0.006031672935932875, 0.0036473136860877275, -0.0023790718987584114, 0.0013379061128944159, 0.0061206575483083725, 0.0033940395805984735, 0.007842988707125187, 0.017011065036058426, -0.008731765672564507, -0.001528001157566905, 0.008156961761415005, 0.0010386169888079166, -0.001340872491709888, -0.011438592337071896, 0.001627378980629146, 0.0075989593751728535, -0.01717652939260006, 0.012525069527328014, -0.0024077799171209335, -0.002395740943029523, 0.002060095313936472, 0.021818390116095543, -0.011291010305285454, -0.004930148366838694, 0.0005287660751491785, -0.01851777359843254, 0.010080091655254364, -0.02300858683884144, 0.0024422104470431805, 0.0012733895564451814, -0.002842860296368599, -0.009166078642010689, -0.006263205781579018, -0.0009444037568755448, -0.0003732241748366505, 0.0022181328386068344, 0.021338779479265213, 0.00902609620243311, 0.001239353558048606, 0.022122874855995178, 0.0031540989875793457, -0.0030289620626717806, -0.028647828847169876, -0.0077527244575321674, 0.014596636407077312, 0.0006993585266172886, -0.00281774764880538, 0.0029715579003095627, 0.001175159472040832, -0.005476274993270636, -0.006617083214223385, 0.004650248680263758, 0.030648402869701385, 0.0009212403674609959, 0.0020139494445174932, -0.003661746857687831, 0.022728143259882927, 0.007380974944680929, -0.006107178051024675, 0.0019768772181123495, -0.0012757752556353807], [-0.0030368107836693525, -0.028196480125188828, 0.024190764874219894, -0.02991354651749134, -0.029614515602588654, 0.006156910210847855, 0.012620161287486553, -0.012342393398284912, 0.0018093291437253356, -0.04994938150048256, 0.005326053127646446, -0.03039822354912758, -0.03468373045325279, -0.02324019931256771, 0.003938907757401466, 0.0019959760829806328, 0.013227047398686409, -0.022134535014629364, 0.006294913589954376, 9.554240386933088e-05, 0.02508881501853466, -0.010682378895580769, 0.011778158135712147, -0.026249829679727554, -0.018187088891863823, -0.028323043137788773, 0.0031623784452676773, 0.04821086302399635, -0.03261347487568855, 0.0034184642136096954, 0.01808854378759861, -0.014362800866365433, -0.018652310594916344, -0.005231210961937904, 0.02698487415909767, 0.004611591342836618, 0.006182420998811722, 0.015246225520968437, 0.0104312589392066, -0.021425342187285423, 0.01284483540803194, 0.0143710533156991, -0.02428795024752617, -0.0018096811836585402, -0.006236766465008259, -0.019212139770388603, 8.512294880347326e-05, -0.004223098512738943, -0.008189534768462181, 0.00017188808124046773, -0.05995284020900726, -0.008842439390718937, -0.06628958135843277, -0.011477821506559849, -0.021296082064509392, -0.005671777296811342, 0.04367835074663162, 0.002214775886386633, 0.02672872692346573, 0.011791139841079712, 0.015798844397068024, -0.02901439182460308, 0.006424338091164827, 0.007072707172483206, 0.004862090572714806, -0.009825024753808975, -0.008570379577577114, 0.02067880518734455, 0.006314409896731377, 0.01781674660742283, -0.01820862665772438, -0.016724061220884323, -0.008397932164371014, 0.01581110991537571, -0.02777443826198578, 0.0009276348864659667, -0.003646243130788207, -0.013877945952117443, -0.01949739083647728, 0.011653564870357513, -0.016784369945526123, -0.0021262820810079575, -0.016304470598697662, 0.020627494901418686, -0.025442400947213173, 0.024102678522467613, -0.003443137276917696, 0.04456645995378494, 0.008392772637307644, 0.015344531275331974, -0.018832219764590263, -0.0055085294879972935, -0.00423249788582325, -0.01646859385073185, -0.03320417180657387, -0.007085749879479408], [0.02648014761507511, 0.023800496011972427, -0.036593418568372726, 0.0516517199575901, -0.0235657449811697, -0.03624425083398819, 0.03210167586803436, 0.06471125036478043, -0.04832932725548744, -0.06581615656614304, -0.006602045614272356, 0.07147309184074402, -0.002727701561525464, -0.04295426607131958, -0.008950990624725819, 0.06148340180516243, -0.006321590859442949, 0.05096559599041939, 0.008652819320559502, 0.025037242099642754, 0.03356211632490158, 0.024344339966773987, 0.103244349360466, -0.023684438318014145, 0.0035919267684221268, 0.10557551681995392, -0.005783494561910629, -0.008010352030396461, -0.004462387412786484, -0.03258638456463814, -0.0037731765769422054, -0.021733397617936134, -0.0029727278742939234, -0.023870037868618965, -0.0010357704013586044, 0.020353345200419426, 0.0530497170984745, 0.09939169883728027, 0.013035242445766926, -0.010396738536655903, 0.05595481023192406, -0.025357743725180626, 0.0020195443648844957, -0.009483428671956062, -0.0044197277165949345, -0.030304836109280586, -0.023517735302448273, -0.05286116153001785, -0.015658631920814514, 0.018407171592116356, 0.06115201115608215, 0.08253844827413559, -0.03055991418659687, -0.030473140999674797, -0.007467308547347784, 0.03435889258980751, -0.01738699898123741, 0.03408579155802727, 0.06566231697797775, 0.007834939286112785, -0.0019775761757045984, -0.022486088797450066, 0.008217430673539639, 0.02673151157796383, -0.019314616918563843, -0.042498718947172165, 0.0026949390303343534, 0.05685485526919365, -0.030832329764962196, -0.06354296952486038, -0.01851654425263405, 0.007048111874610186, 0.01707800291478634, -0.006866543088108301, 0.02286987565457821, 0.0031407633796334267, 0.0419597364962101, 0.05156392231583595, -0.015426050871610641, 0.016338778659701347, -0.024899797514081, 0.0012546138605102897, 0.03981881961226463, 0.04682627692818642, 0.02058982290327549, 0.003543865866959095, -0.01247122697532177, 0.0026207284536212683, 0.02062809281051159, 0.045936085283756256, 0.005707154516130686, 0.016192417591810226, 0.027841825038194656, 0.04488794505596161, -0.01299214642494917, -0.008672299794852734], [-0.028297223150730133, -0.025822434574365616, -0.030858773738145828, 0.07770001143217087, 0.01873871311545372, 0.012687847018241882, -0.0440349206328392, -0.010373992845416069, -0.05456531420350075, -0.013968936167657375, 0.00790491048246622, -0.002469198079779744, 0.059281058609485626, -0.0008378713391721249, 0.04103471711277962, -0.005937539041042328, 0.01166451908648014, -0.03484634682536125, -0.010375816375017166, 0.005857094656676054, -0.017456430941820145, -0.00428749481216073, -0.04615842178463936, 0.08914362639188766, -0.01859557256102562, -0.032436031848192215, 0.03380993753671646, -0.06397490948438644, 0.0036691268905997276, -0.033384282141923904, -0.016379591077566147, 0.0029925054404884577, -0.009403183124959469, 0.011402947828173637, -0.01703047566115856, 0.03388985991477966, -0.0012542366748675704, -0.014202136546373367, -0.020020529627799988, 0.038881514221429825, -0.06506218016147614, 0.00957728736102581, -0.011093363165855408, -0.008783005177974701, 0.021729176864027977, 0.04683013632893562, -0.013387727551162243, 0.012828134000301361, -0.016437336802482605, 0.016035864129662514, 0.06569715589284897, -0.04613853991031647, 0.0607086718082428, 0.012387197464704514, -0.01730899140238762, -0.014036374166607857, -0.03592556715011597, 0.07829016447067261, -0.030710754916071892, -0.029190927743911743, 0.04037116840481758, 0.039168212562799454, -0.02573373168706894, -0.013678887858986855, 0.014986743219196796, 0.0007120729424059391, -0.014418442733585835, -0.046737972646951675, 0.031155187636613846, -0.021599004045128822, 0.002893885364755988, 0.03879719227552414, 0.010442471131682396, -0.010016444139182568, 0.047224897891283035, 0.030415724962949753, 0.002986243460327387, -0.028438404202461243, 0.05686786025762558, 0.01103009283542633, 0.0038976031355559826, -0.029217028990387917, -0.02678564563393593, 0.01161254197359085, -0.017212972044944763, 0.013913234695792198, -0.013047570362687111, 0.005819100886583328, 0.026058724150061607, -0.011563976295292377, 0.005364961456507444, -0.004826256074011326, 0.05838402360677719, 0.03159457817673683, -0.005253052804619074, -0.01613769307732582], [-0.03292370215058327, 0.08496233820915222, 0.04464113339781761, -0.07595045864582062, -0.13329486548900604, 0.04859751835465431, -0.04403636232018471, 0.04048990085721016, -0.04681152105331421, 0.08076666295528412, -0.027332235127687454, 0.00752746406942606, 0.08390649408102036, -0.07175391912460327, 0.01005593966692686, 0.07473932206630707, 0.01661542057991028, -0.031387053430080414, 0.03670414537191391, 0.07597891986370087, 0.022757964208722115, 0.055400628596544266, 0.15506963431835175, -0.042135875672101974, -0.014634069055318832, 0.04639828950166702, 0.04468631371855736, 0.06895968317985535, -0.0009280100930482149, 0.1025579422712326, 0.04108059033751488, -0.050428830087184906, 0.015541343949735165, 0.08181276172399521, 0.0622636154294014, 0.11916153132915497, 0.04892165958881378, 0.14653772115707397, 0.07868689298629761, 0.0036973783280700445, -0.03126966953277588, -0.04009585455060005, -0.06598546355962753, -0.01842145062983036, 0.0822877287864685, 0.04718632251024246, -0.06959626078605652, -0.11542388796806335, 0.09088857471942902, 0.009578557685017586, 0.031039100140333176, -0.05406523123383522, 0.07811962068080902, 0.04357932507991791, -0.05816153809428215, 0.10219153761863708, -0.020579418167471886, -0.025296347215771675, 0.04357844218611717, 0.04847513884305954, -0.0242443960160017, -0.011667473241686821, -0.017602166160941124, 0.12477340549230576, -0.06629777699708939, -0.048154156655073166, -0.09295447170734406, 0.007823877036571503, 0.0015897861449047923, -0.029877563938498497, -0.09140565246343613, 0.066899374127388, 0.0038466888945549726, 0.05652201175689697, -0.07592704147100449, -0.0791015475988388, 0.0864250436425209, 0.03818802535533905, 0.014330725185573101, -0.00793387833982706, -0.030842440202832222, 0.05911107361316681, 0.11635711789131165, -0.008997688069939613, 0.08388608694076538, 0.07470247149467468, 0.04523709788918495, -0.01242414303123951, 0.08260723948478699, -0.006825512740761042, -0.04655107110738754, -0.02898424305021763, -0.05225814878940582, 0.10925985872745514, 0.0193864144384861, -0.037412650883197784], [-0.04776398465037346, -0.07042255997657776, -0.003613014705479145, 0.05016418546438217, 0.03398514166474342, -0.06175156682729721, -0.028501195833086967, -0.04567870870232582, -0.010588327422738075, -0.006687929388135672, -0.021156400442123413, -0.04397739842534065, 0.004353415686637163, -0.001296056667342782, 0.057712946087121964, -0.03578375279903412, 0.03607805073261261, 0.028261804953217506, -0.05009687319397926, 0.0371723547577858, 0.05205639451742172, 0.04380384832620621, -0.06197214126586914, 0.03770674765110016, 0.03216751664876938, 0.02367601916193962, 0.009918578900396824, 0.08486655354499817, 0.023223232477903366, -0.05268531292676926, -0.036123309284448624, -0.0019208032172173262, -0.030752692371606827, -0.008049186319112778, -0.06603621691465378, 0.008009024895727634, -0.04622542858123779, 0.0024704106617718935, 0.04394065961241722, 0.044655781239271164, -0.05278046429157257, -0.016491465270519257, -0.01418130099773407, 0.011973855085670948, 0.024141600355505943, -0.02331378497183323, -0.017618291079998016, 0.010081660002470016, 0.000990923959761858, 0.01136737409979105, 0.05898230895400047, -0.033408306539058685, 0.11085847020149231, -0.0011544119333848357, 0.06360584497451782, -0.013153666630387306, 0.0058885239996016026, 0.09625005722045898, -0.04797797277569771, -0.044875942170619965, 0.04625195264816284, 0.018747545778751373, 0.01121192891150713, 0.028043905273079872, 0.06272534281015396, -0.0010532712331041694, 0.04245886206626892, 0.02822004072368145, -0.04109378904104233, 0.030810024589300156, -0.02881511114537716, 0.007446366362273693, -0.005283400882035494, -0.030790535733103752, 0.055544547736644745, -0.021328341215848923, -0.05528807267546654, 0.00029465201077982783, 0.048576902598142624, -0.001068395795300603, -0.019336018711328506, 0.019551852717995644, 0.007929735817015171, -0.04095716029405594, -0.01393101830035448, 0.044040169566869736, 0.034670837223529816, 0.06212790682911873, -0.04227415472269058, -0.018337992951273918, 0.010754719376564026, 0.024828888475894928, 0.04240650311112404, -0.0006692107417620718, -0.0557725727558136, -0.02842080593109131], [-0.04143626242876053, -0.01120322197675705, 0.05468011274933815, -0.061468131840229034, -0.15276867151260376, 0.04215174913406372, -0.03573490306735039, 0.022144127637147903, -0.04390323534607887, 0.02972261793911457, 0.010066713206470013, -0.016158604994416237, -0.07815075665712357, 0.013472014106810093, -0.09385427832603455, -0.017058279365301132, -0.08665354549884796, -0.03626779839396477, 0.07990339398384094, 0.07489386200904846, 0.013301410712301731, -0.08811989426612854, 0.09685181826353073, -0.15539562702178955, -0.05055014416575432, 0.09736379235982895, 0.13387861847877502, -0.03275750204920769, 0.004701799713075161, -0.007114465348422527, -0.015149371698498726, -0.055601395666599274, -0.037401288747787476, -0.035944122821092606, -0.024921096861362457, 0.024697190150618553, -0.04742487892508507, 0.03301705792546272, -0.027302011847496033, -0.026182230561971664, -0.014654068276286125, -0.009710466489195824, 0.06134498119354248, 0.035058166831731796, -0.05996722728013992, -0.001543760416097939, -0.020834924653172493, 0.016042117029428482, -0.03520062193274498, 0.08151192218065262, -0.11441272497177124, 0.09292801469564438, -0.13725119829177856, 0.012085399590432644, 0.02855641208589077, -0.0005249633104540408, -0.04038907214999199, -0.18430787324905396, 0.03772956505417824, 0.10736403614282608, -0.05580110847949982, -0.03759381175041199, -7.915419701021165e-05, 0.01547173224389553, 0.047290023416280746, 0.057422805577516556, 0.01995820365846157, 0.022866729646921158, -0.018917640671133995, 0.025364907458424568, 0.03252020105719566, -0.17384716868400574, 0.07279200851917267, -0.03210429102182388, -0.07994867861270905, -0.04334248602390289, 0.055965621024370193, 0.006263860035687685, 0.06990677863359451, 0.06317794322967529, -0.07843857258558273, 0.14007212221622467, 0.09238123148679733, 0.07075770199298859, 0.03385365009307861, -0.04536595940589905, -0.0021956285927444696, 0.003927385434508324, 0.051666297018527985, 0.010388510301709175, -0.028551962226629257, -0.0200259517878294, 0.053800538182258606, -0.059797462075948715, -0.022451071068644524, 0.013084032572805882], [-0.02860528789460659, 0.032379016280174255, -0.039783775806427, -0.03777956962585449, -0.015228727832436562, -0.01826312020421028, 0.024026378989219666, -0.015608541667461395, -0.08380180597305298, 0.05896051973104477, -0.08405467867851257, -0.10643996298313141, -0.02249293588101864, -0.012184827588498592, -0.010328513570129871, 0.0053702229633927345, 0.07926899939775467, -0.06350013613700867, -0.0648089125752449, 0.021629566326737404, 0.043992478400468826, 0.04530360549688339, -0.06808833032846451, 0.03129453957080841, -0.07508521527051926, 0.004931244999170303, 0.05987301096320152, 0.010426163673400879, -0.04289719834923744, -0.05383947864174843, -0.041245318949222565, -0.07396566867828369, 0.09078218042850494, 0.009008402936160564, -0.07337260246276855, 0.07186564803123474, -0.031581003218889236, -0.02678520604968071, 0.012836371548473835, 0.00010504125384613872, 0.04179208353161812, 0.031769681721925735, 0.054323531687259674, -0.08888255804777145, -0.004183049313724041, 0.031333476305007935, -0.04520604759454727, -0.017518160864710808, -0.061604615300893784, -0.02081119269132614, 0.004720757715404034, -0.06529577076435089, -0.09047481417655945, 0.006329426076263189, -0.06196364760398865, -0.058690402656793594, -0.060083795338869095, 0.06949403136968613, -0.012718770653009415, -0.002191987819969654, 0.030004948377609253, 0.07615694403648376, -0.07590560615062714, 0.00038437891635112464, 0.06471140682697296, -0.07604318857192993, 0.00606915820389986, 0.027767574414610863, -0.060129716992378235, 0.004023930989205837, 0.03744225949048996, 0.02321719005703926, -0.08575529605150223, 0.0028709007892757654, 0.10292057693004608, 0.051649149507284164, 0.00018903690215665847, -0.03027634508907795, 0.09614162147045135, 0.08274175226688385, -0.014138567261397839, 0.025068826973438263, -0.0880938395857811, 0.045941028743982315, -0.028751570731401443, 0.03550043702125549, -0.03388098254799843, -0.06800928711891174, 0.01243569329380989, 0.054929327219724655, -0.08080904930830002, 0.07309241592884064, -0.015453943982720375, -0.021630439907312393, -0.08480841666460037, 0.02081921510398388], [0.011145689524710178, 0.0068788123317062855, 0.0449986457824707, -0.09929420053958893, -0.04445610195398331, 0.026125816628336906, -0.07005832344293594, 0.06221926212310791, -0.07537629455327988, -0.014022666029632092, -0.10194951295852661, 0.04935529828071594, -0.10731843113899231, -0.06389922648668289, -0.04171071946620941, -0.028752097859978676, 0.02806689776480198, 0.03538890928030014, 0.06732092797756195, -0.006344506982713938, 0.06432874500751495, -0.1364111304283142, 0.07433684170246124, -0.10468930751085281, -0.042444080114364624, 0.04460098594427109, 0.06174953654408455, -0.04001060873270035, 0.07022156566381454, 0.10821350663900375, 0.09017432481050491, 0.07396388798952103, -0.016666783019900322, 0.06531666219234467, -0.05652180314064026, -0.029099196195602417, 0.012340147979557514, 0.09063373506069183, 0.07160020619630814, -0.09399069100618362, 0.11745242774486542, -0.01128676813095808, 0.06908467411994934, -0.08534692972898483, 0.1036629006266594, 0.014641031622886658, 0.06182010844349861, 0.04213022068142891, 0.06785977631807327, -0.00016310536011587828, -0.1896776705980301, 0.019042963162064552, -0.12951688468456268, 0.057729098945856094, -0.01701854169368744, 0.06290319561958313, 0.03802889585494995, -0.15748633444309235, 0.026118380948901176, -0.05304872244596481, -0.05433011054992676, -0.09436311572790146, 0.04301368072628975, 0.007882682606577873, 0.019051648676395416, 0.020654279738664627, -0.06777366995811462, -0.0048087844625115395, 0.07690442353487015, 0.011520045809447765, 0.07983770966529846, -0.19977815449237823, 0.12287645787000656, 0.04649899899959564, -0.028718410059809685, 0.06435568630695343, -0.015071140602231026, 0.002081482671201229, 0.0472150593996048, -0.008625333197414875, -0.05436824634671211, 0.06710051000118256, 0.0376235656440258, -0.03325475752353668, 0.08032196760177612, -0.09494151920080185, 0.10775338113307953, -0.016609644517302513, 0.013489801436662674, -0.102585569024086, -0.05487992987036705, -0.11428309977054596, -0.01096514891833067, 0.05782580003142357, 0.003038628725335002, 0.04133099690079689], [-0.10182356834411621, 0.04370077699422836, 0.05473589897155762, -0.10138852149248123, -0.08053624629974365, -0.058445315808057785, 0.010211212560534477, -0.009633532725274563, 0.06738539040088654, -0.03169041872024536, 0.01171303540468216, -0.07455725967884064, -0.08419317752122879, -0.052467405796051025, 0.04164845868945122, 0.05990615487098694, -0.0029074556659907103, 0.04265754297375679, 0.06864801049232483, 0.04274642467498779, -0.0037355462554842234, -0.0318920761346817, 0.021768338978290558, -0.03724720701575279, -0.07063949853181839, -0.02276165783405304, -0.02532256208360195, 0.018198570236563683, 0.0003602183423936367, 0.013453781604766846, 0.07461527734994888, -0.09864121675491333, 0.07995742559432983, 0.007404508534818888, -0.062042366713285446, -0.01505630649626255, 0.06175296753644943, 0.03293214738368988, 0.053801123052835464, -0.016751307994127274, -0.03590910881757736, 0.0580822229385376, -0.01697855070233345, -0.04755474627017975, -0.04276057332754135, -0.022569766268134117, -0.0028516596648842096, -0.0558716356754303, -0.06710877269506454, -0.03227413073182106, -0.11155793815851212, 0.05739827826619148, -0.07882530242204666, 0.029616493731737137, -0.008534951135516167, -0.10230764001607895, 0.10053674876689911, -0.10119599103927612, -0.012471619993448257, 0.05085612088441849, 0.07490640133619308, -0.08798865973949432, 0.04006754979491234, -0.028442157432436943, 0.020401906222105026, -0.038704272359609604, -0.04949116334319115, -0.07118415087461472, 0.08461993932723999, 0.027587516233325005, 0.0021962272003293037, -0.03841327130794525, -0.017169255763292313, 0.0030796010978519917, 0.02476983331143856, -0.004212359432131052, 0.06267577409744263, -0.034177280962467194, 0.07783441990613937, 0.04819699749350548, 0.03710581362247467, -0.025751739740371704, 0.02390720508992672, -0.051420848816633224, -0.011404504999518394, -0.053395211696624756, -0.007804723922163248, -0.04863719642162323, 0.0514242947101593, -0.08504350483417511, 0.023399556055665016, -0.0012078508734703064, 0.03350447490811348, 0.05806683003902435, -0.02500075288116932, -0.01776610128581524], [0.04231641814112663, -0.01913277432322502, -0.04493153095245361, 0.041433218866586685, -0.049240484833717346, 0.013068092986941338, -0.06749044358730316, 0.010408789850771427, 0.09190889447927475, -0.041011203080415726, -0.012265482917428017, 0.0058170706033706665, 0.08636967092752457, -0.04276580736041069, 0.010443218983709812, 0.11604789644479752, 0.07361863553524017, 0.024399152025580406, -0.07638872414827347, 0.04306439310312271, 0.10770882666110992, 0.08113855868577957, 0.11139584332704544, 0.10092414170503616, -0.0007367413491010666, 0.07508803904056549, -0.02384258434176445, -0.0845274031162262, -0.036392588168382645, -0.030520642176270485, 0.04674509912729263, -0.05415469780564308, 0.11335015296936035, -0.030622173100709915, -0.08286400884389877, 0.01819867454469204, 0.10296490043401718, 0.139572873711586, -0.03938119485974312, 0.04873203858733177, 0.06292286515235901, 0.028042079880833626, -0.028493959456682205, 0.025535840541124344, -0.093125119805336, -0.05055000260472298, 0.013613337650895119, 0.026672935113310814, -0.006662170402705669, -0.01053554005920887, -0.06154194101691246, -0.022671090438961983, -0.008875680156052113, 0.035596609115600586, -0.07171440869569778, -0.0536189079284668, -0.01045451033860445, -0.028494134545326233, 0.11982209980487823, -0.030447835102677345, -0.0610869862139225, -0.06102829799056053, -0.007111832033842802, 0.05448141694068909, 0.032844506204128265, 0.004204721190035343, 0.06416038423776627, 0.027544399723410606, 0.008697985671460629, 0.023168347775936127, 0.055650290101766586, -0.008019798435270786, 0.05617537721991539, 0.07512305676937103, -0.03744450584053993, -0.035256609320640564, 0.04353035241365433, 0.11793670803308487, -0.005658465437591076, 0.028400816023349762, -0.020359685644507408, 0.08012926578521729, 0.05010761320590973, -0.04070063307881355, -0.04502719268202782, -0.07198779284954071, 0.004896099679172039, 0.022850222885608673, 0.035987719893455505, 0.0515415258705616, -0.08120959252119064, -0.0430276058614254, 0.03611146658658981, -0.0200213473290205, 0.08107474446296692, -0.03909243643283844], [-0.09300126880407333, 0.045147377997636795, -0.044000305235385895, 0.07700160890817642, 0.017577985301613808, -0.09594540297985077, -0.03153504431247711, -0.04470887407660484, 0.03279665485024452, -0.02215958945453167, 0.051314577460289, -0.12187347561120987, 0.1106228306889534, 0.02589806541800499, -0.037929046899080276, -0.033672817051410675, -0.03642643988132477, -0.09313623607158661, -0.06887541711330414, -0.044732291251420975, -0.07639238238334656, 0.0653175637125969, -0.1564219444990158, 0.0976080372929573, 0.030139878392219543, -0.021133622154593468, 0.049228910356760025, -0.06947294622659683, 0.08643367141485214, -0.07284944504499435, -0.011087126098573208, -0.04459182173013687, -0.00423338683322072, -0.06717152893543243, -0.04429112374782562, 0.030029505491256714, 0.005057936068624258, -0.045717231929302216, -0.012731270864605904, 0.09450935572385788, -0.12849636375904083, 0.08573057502508163, 0.08845890313386917, -0.032052576541900635, 0.0585436187684536, 0.09543544799089432, -0.05889977887272835, 0.04859009385108948, -0.094722218811512, -0.0035837655887007713, 0.04337092116475105, -0.02457507513463497, 0.11215531826019287, -0.05473998188972473, 0.0372794009745121, -0.018499335274100304, 0.009463039226830006, 0.11557524651288986, -0.06357493996620178, -0.11431020498275757, 0.08275322616100311, 0.055847082287073135, -0.03326188400387764, -0.025419561192393303, 0.0541720911860466, -0.011173468083143234, -0.0187065526843071, -0.08853963762521744, -0.051993176341056824, -0.03147537633776665, 0.060122162103652954, 0.12210977077484131, -0.055578816682100296, 0.0049607460387051105, 0.0875411331653595, 0.05740557610988617, -0.08259653300046921, -0.04685966670513153, 0.039433740079402924, 0.05342404171824455, -0.018406834453344345, 0.0008885302231647074, -0.0796276107430458, -0.01735083758831024, -0.06486072391271591, 0.043799109756946564, -0.07184922695159912, 0.03874184563755989, -0.027965068817138672, 0.05384628474712372, 0.04869044944643974, -0.06742916256189346, 0.06395953893661499, 0.03745922073721886, 0.08335040509700775, -0.06397520750761032], [-0.07560043036937714, 0.052999429404735565, 0.07346977293491364, -0.07120562344789505, -1.3766288248007186e-05, -0.04711182788014412, 0.008023831062018871, 0.03574962913990021, 0.031204506754875183, 0.004690024070441723, -0.04220922663807869, -0.015857988968491554, 0.009204396978020668, -0.09056894481182098, 0.0009975252905860543, -0.01640145666897297, -0.05690399929881096, 0.004959029145538807, -0.032497573643922806, -0.019931450486183167, -0.029940057545900345, 0.018312234431505203, -0.04898499324917793, 0.028299331665039062, 0.020103922113776207, -0.0772172287106514, 0.003731893142685294, -0.020738812163472176, -0.0529971681535244, 0.01253970991820097, -0.010155179537832737, -0.0008873142069205642, 0.010678086429834366, 0.044324472546577454, -0.018416447564959526, 0.027723755687475204, 0.03140479698777199, 0.04543045535683632, -0.026215987280011177, -0.012427713721990585, 0.07318270206451416, -0.019561557099223137, 0.03647937253117561, -0.02770964987576008, 0.06609143316745758, -0.032524608075618744, -0.050143543630838394, -0.02230065129697323, 0.0061919051222503185, 0.04522702470421791, -0.08042532950639725, 0.04185173287987709, -0.049522701650857925, 0.0036101066507399082, -0.03941779211163521, -0.05891932174563408, 0.08575288951396942, -0.10423240065574646, -0.001438413979485631, 0.01832101307809353, 0.08585105836391449, -0.06276721507310867, -0.006013812497258186, -0.017243828624486923, 0.0457928404211998, -0.06774093955755234, 0.05521184578537941, -0.004298181738704443, 0.0247690137475729, 0.09120184183120728, -0.040540192276239395, 0.003117406740784645, 0.04030488803982735, -0.030249761417508125, 0.02607227861881256, 0.02936757542192936, 0.09273882955312729, -0.0967305600643158, 0.03628270700573921, -0.021326493471860886, -0.05372039973735809, 0.03211498633027077, -0.04550063982605934, -0.03104223683476448, -0.06276494264602661, -0.033809032291173935, -0.03982387110590935, -0.07661230862140656, 0.05655120685696602, 0.013641361147165298, 0.02261824533343315, -0.008556196466088295, 0.02640305459499359, -0.07895071804523468, -0.06223202124238014, -0.03359357640147209], [0.028845595195889473, 0.010317707434296608, -0.07163151353597641, 0.09105601906776428, 0.036189835518598557, 0.08017073571681976, 0.0005454412894323468, 0.07060960680246353, 0.007152350153774023, 0.10380581021308899, 0.08843330293893814, 0.06355919688940048, -0.06646057963371277, 0.03341133892536163, -0.07606153935194016, 0.06961391866207123, 0.002597818849608302, 0.05243457108736038, 0.042032405734062195, 0.005368000362068415, -0.053456563502550125, 0.013571585528552532, 0.08835167437791824, -0.07749862968921661, -0.00019318489648867399, 0.1268816888332367, 0.020161008462309837, 0.09666851162910461, -0.07780330628156662, -0.015870122238993645, -0.028904683887958527, 0.0062098721973598, 0.046208083629608154, 0.07907408475875854, 0.10753964632749557, 0.00753834517672658, 0.09996158629655838, 0.1473502367734909, -0.056563302874565125, -0.05273643508553505, 0.036847326904535294, -0.10164307802915573, -0.04406822845339775, 0.09664744883775711, -0.04282978177070618, -0.0020568205509334803, -0.07078942656517029, 0.019777128472924232, 0.12068681418895721, -0.0420340821146965, -0.07194834202528, 0.06753888726234436, -0.026942284777760506, 0.00652251485735178, 0.08033981174230576, 0.11438757926225662, 0.07682644575834274, 0.009172569029033184, 0.1072867289185524, 0.033305902034044266, 0.06953729689121246, 0.029075337573885918, -0.006769883446395397, 0.07186029106378555, -0.04865799844264984, -0.07299356907606125, -0.04964511841535568, -0.05213528871536255, -0.05790230259299278, 0.006508717313408852, -0.03260878473520279, 0.03884512186050415, 0.12182632088661194, 0.018446864560246468, -0.07275629788637161, 0.0033679974731057882, -0.0691414326429367, 0.1330646574497223, 0.02242988348007202, -0.03680653125047684, -0.07219463586807251, -0.013397729955613613, -0.05401328578591347, -0.020720046013593674, 0.10188572108745575, -0.05683586001396179, -0.050002600997686386, -0.026456551626324654, -0.06594660133123398, 0.06042240560054779, -0.03948540613055229, 0.019204502925276756, 0.06670855730772018, 0.06767456978559494, -0.008689481765031815, 0.05068332701921463], [0.06362714618444443, 0.05011626332998276, -0.07007632404565811, -0.06587447971105576, -0.08340965956449509, 0.07590050995349884, 0.07343067228794098, 0.003231396432965994, 0.059382904320955276, 0.024152487516403198, 0.06738141179084778, 0.042935945093631744, 0.09542623162269592, 0.013960939832031727, 0.0805385410785675, -0.009698034264147282, -0.012533597648143768, -0.0024010511115193367, 0.03465019538998604, 0.07909712940454483, -0.004473361186683178, 0.037359945476055145, 0.05443660914897919, 0.04562479630112648, -0.006065403576940298, -0.043146584182977676, -0.06161634624004364, 0.02257382683455944, 0.08558094501495361, -0.041723720729351044, -0.018395207822322845, -0.009178413078188896, -0.0541095994412899, 0.03607610985636711, 0.005905609577894211, 0.028987031430006027, 0.02354024350643158, -0.06436238437891006, 0.016714731231331825, -0.04770766198635101, 0.0002631827665027231, 0.00909285619854927, -0.053449466824531555, -0.029286248609423637, 0.05771050974726677, -0.049417540431022644, 0.07126467674970627, -0.044273823499679565, 0.016759201884269714, -0.04819454997777939, -0.08984687179327011, 0.06992801278829575, 0.019060218706727028, -0.012161488644778728, 0.08459361642599106, 0.025497402995824814, 0.07369085401296616, 0.009176553227007389, 0.016692547127604485, -0.03604516386985779, 0.06733178347349167, -0.009890320710837841, 0.03704959154129028, 0.06424743682146072, 0.039664894342422485, -0.04098872095346451, -0.04037989303469658, 0.03333471715450287, 0.034805670380592346, -0.05195555090904236, 0.07101842761039734, 0.07784018665552139, 0.04385486990213394, -0.059970978647470474, 0.07708264887332916, 0.0597439669072628, -0.03597044199705124, 0.04849592223763466, 0.09299785643815994, -0.02760794572532177, 0.016572657972574234, -0.008137894794344902, -0.008485469035804272, -0.011769585311412811, -0.031024852767586708, -0.024152571335434914, -0.0010748904896900058, 0.07216671109199524, -0.04736977815628052, -0.01376320794224739, 0.02954746037721634, 0.019842438399791718, 0.03485465049743652, -0.008111286908388138, 0.024922864511609077, -0.003190847346559167], [-0.07832693308591843, -0.09713703393936157, 0.0006985390209592879, 0.006941632833331823, 0.09878580272197723, 0.03752972558140755, -0.07258091121912003, -0.05084339901804924, -0.03554225340485573, -0.05490695312619209, 0.06814467906951904, -0.051096390932798386, 0.01698574796319008, 0.06510447710752487, 0.06639890372753143, -0.060949381440877914, 0.055053647607564926, -0.0767795741558075, -0.008519800379872322, 0.038034263998270035, -0.06473881006240845, 0.04598601907491684, -0.04157925024628639, 0.11410526186227798, -0.06728736311197281, 0.02692253142595291, -0.03713516518473625, 0.07791027426719666, 0.06194697692990303, 0.004821400158107281, -0.02542226016521454, -0.007941709831357002, -0.073801189661026, -0.01876235380768776, 0.054460376501083374, 0.06999474763870239, 0.01524669211357832, -0.05619248002767563, 0.07697895914316177, 0.10203352570533752, -0.09457952529191971, 0.016651900485157967, -0.02531961351633072, -0.10528314113616943, 0.05240369960665703, -0.018488895148038864, -0.05408124998211861, 0.07640334218740463, 0.057234540581703186, 0.05801669508218765, 0.046852339059114456, -0.10016140341758728, -0.010207452811300755, 0.026891035959124565, 0.04452119395136833, -0.024484820663928986, 0.022588923573493958, 0.09089063107967377, -0.010191845707595348, -0.015776176005601883, 0.027575012296438217, -0.027807360514998436, 0.0021675170864909887, 0.02654910460114479, 0.025621801614761353, 0.07056768238544464, 0.03460923209786415, -0.011807874776422977, 0.07495532184839249, -0.05487916246056557, -0.005994546227157116, 0.08829715102910995, 0.025073271244764328, 0.007061276584863663, -0.03762494772672653, 0.04781225323677063, -0.06645748764276505, -0.013279182836413383, 0.00381644768640399, 0.06701584905385971, -0.0658104196190834, -0.057295821607112885, -0.0918613150715828, -0.0015545310452580452, -0.022012855857610703, 0.06861698627471924, 0.00787784531712532, 0.050056058913469315, 0.0647612139582634, -0.01911146007478237, -0.04389788582921028, -0.08001955598592758, 0.017588285729289055, -0.05990087613463402, -0.09413767606019974, -0.06969770789146423], [-0.11653237789869308, -0.08812074363231659, 0.07089176774024963, 0.023480193689465523, 0.08171375095844269, -0.00836261361837387, 0.06899982690811157, -0.03463650122284889, -0.006516204681247473, -0.01798844523727894, -0.048368148505687714, 0.0001585993159096688, 0.0020042583346366882, 0.009653476998209953, 0.08478426933288574, 0.053916241973638535, 0.05449361354112625, -0.013993319123983383, -0.04125424101948738, -0.02910374291241169, 0.040366604924201965, -0.028723301365971565, -0.045018259435892105, 0.09799103438854218, -0.07596852630376816, 0.021767213940620422, 0.028161989524960518, -0.09037790447473526, 0.03339001536369324, 0.07401441037654877, -0.08156298100948334, 0.03499313071370125, 0.07879776507616043, -0.06491971015930176, 0.035772714763879776, 0.025664061307907104, 0.034956250339746475, -0.09457121044397354, -0.03135182335972786, 0.03371582180261612, -0.03866414725780487, 0.012084337882697582, -0.014054655097424984, -0.07337179780006409, 0.06697484850883484, 0.056541115045547485, 0.018923891708254814, 0.07914041727781296, 0.030770260840654373, 0.024845514446496964, 0.04498535767197609, -0.07094331085681915, 0.01362207718193531, 0.05465393885970116, -0.08068132400512695, -0.11413037776947021, 0.05376401171088219, 0.08696241676807404, -0.01598099060356617, -0.026384834200143814, 0.09979043155908585, 0.038032166659832, -0.045850928872823715, -0.043227940797805786, -0.06617678701877594, 0.021200548857450485, -0.016856983304023743, -0.04427485540509224, 0.06088504567742348, -0.05911339819431305, -0.026419339701533318, 0.016584936529397964, -0.07730385661125183, -0.07413127273321152, -0.012869092635810375, -0.01313055120408535, 0.074217289686203, -0.02022361382842064, 0.029561743140220642, -0.0010778396390378475, 0.033572759479284286, -0.057587794959545135, 0.03937589377164841, -0.046370506286621094, 0.07601914554834366, -0.06954684108495712, 0.004244327545166016, -0.053826507180929184, 0.013028974644839764, 0.04052039608359337, 0.018811339512467384, 0.026341846212744713, 0.08584123104810715, 0.08596857637166977, 0.0006657537305727601, -0.009573651477694511], [0.02012784034013748, -0.029152054339647293, -0.07722096145153046, 0.02674570120871067, -0.02014443650841713, 0.08573571592569351, 0.021464349702000618, 0.10689157992601395, 0.03224007785320282, 0.08381986618041992, -0.0348145067691803, 0.06687499582767487, -0.01936577446758747, -0.00015828877803869545, 0.015272407792508602, 0.012857494875788689, -0.057586535811424255, -0.023678766563534737, -0.0058668553829193115, 0.07601595669984818, 0.0979759693145752, -0.038817428052425385, 0.1237735003232956, -0.007958137430250645, 0.04861748218536377, 0.08937471359968185, 0.05926986411213875, 0.039139993488788605, -0.07242151349782944, 0.05303184688091278, 0.008525311015546322, -0.03298761695623398, 0.07475588470697403, -0.039302073419094086, -0.034342601895332336, 0.07193736732006073, 0.0800684317946434, 0.07704867422580719, 0.053827881813049316, 0.006360579747706652, 0.06267722696065903, -0.0650489330291748, 0.08849884569644928, 0.027416566386818886, -0.06169235333800316, -0.057420533150434494, -0.0391521081328392, -0.09421851485967636, 0.09523462504148483, 0.04253915697336197, -0.0404648631811142, -0.04629553109407425, 0.05002914369106293, 0.06133768707513809, 0.02462332509458065, 0.11196497827768326, 0.08998018503189087, -0.0006050284137018025, 0.03138637915253639, 0.01626761071383953, 0.08876314759254456, 0.010690021328628063, 0.04967709258198738, 0.059259843081235886, -0.01497581321746111, -0.09236577153205872, -0.06183081865310669, -0.0021413664799183607, 0.0438448004424572, -0.08465401083230972, 0.040537066757678986, -0.0025297885295003653, 0.08900006860494614, -0.002480225171893835, 0.03663986548781395, 0.02228793315589428, 0.008187167346477509, 0.0822295993566513, 0.08554621785879135, -0.01871231384575367, -0.035997893661260605, 0.04413041099905968, 0.010769199579954147, 0.003177229780703783, 0.05221687629818916, 0.06378604471683502, 0.08016151934862137, -0.01915726810693741, 0.02562425099313259, 0.004288535565137863, -0.009739956818521023, -0.02535197325050831, -0.0051549575291574, 0.10268634557723999, -0.008392736315727234, 0.003827536478638649], [-0.04265047609806061, -0.021535087376832962, -0.011745586059987545, -0.017970697954297066, -0.08199241012334824, -0.021480785682797432, -0.01771041750907898, 0.01181931421160698, -0.02045208401978016, 0.03870019316673279, 0.029346855357289314, -0.011354804039001465, 0.024886146187782288, -0.014000947587192059, 0.005742141045629978, -0.014693493954837322, -0.006586352363228798, 0.023358382284641266, -0.0031908047385513783, 0.048357415944337845, -0.0040435041300952435, 0.0003041500458493829, 0.02007465995848179, 0.038495492190122604, 0.028034141287207603, -0.04319675266742706, 0.07383008301258087, -0.01957584358751774, -0.02065892145037651, -0.02301548793911934, -0.012515625916421413, -0.04923447594046593, 0.05539314076304436, -0.01077969092875719, 0.07710038870573044, -0.04308444634079933, 0.015818098559975624, 0.020243385806679726, 0.03377460688352585, 0.010715511627495289, 0.030553527176380157, -0.023386353626847267, -0.031472716480493546, -0.036979157477617264, -0.03907092660665512, -0.04788811504840851, 0.03699303790926933, 0.041966404765844345, -0.0024644252844154835, 0.01969260349869728, -0.01472761482000351, -0.029446717351675034, -0.09700754284858704, -0.0027234526351094246, -0.0158926360309124, -0.029866211116313934, -0.03737976402044296, -0.10740162432193756, 0.036649059504270554, 0.02545945532619953, 0.012848577462136745, -0.02265022136271, -0.029895002022385597, 0.03488510474562645, 0.012016921304166317, -0.006975638680160046, 0.036134023219347, -0.003193500218912959, 0.009667778387665749, 0.018576476722955704, -0.05171051248908043, -0.09523127228021622, -0.02269606478512287, -0.020864926278591156, 0.0034851140808314085, 0.005047615151852369, 0.07107321918010712, 0.006491870619356632, -0.0057210554368793964, 0.04331887140870094, -0.03802325576543808, -0.006414621137082577, -0.016691362485289574, 0.03852592408657074, -0.03392777219414711, 0.049099069088697433, -0.05041320249438286, -0.042182184755802155, 0.009203131310641766, -0.010605091229081154, -0.016726095229387283, -0.011773140169680119, 0.009459826163947582, -0.05849636718630791, -0.08222752809524536, 0.01271939929574728]], "b2": [-0.04019850492477417, 0.02359979785978794, -0.08125912398099899, 0.0006296572391875088, 0.07003600895404816, -0.05169861391186714, 0.07026894390583038, -0.006603572051972151, 0.04578814655542374, 0.03402193635702133, -0.03412601351737976, 0.0643167495727539, 0.037037093192338943, -0.02512255124747753, -0.03914179280400276, -0.07160785049200058, 0.021442003548145294, 0.08205713331699371, -0.028124259784817696, -0.03281126916408539, -0.0921267420053482, 0.039221204817295074, -0.028888940811157227, 0.07550963759422302, -0.04305793717503548, 0.02603418566286564, 0.009548653848469257, -0.0926133543252945, 0.01050391886383295, 0.08100464940071106, 0.042250242084264755, -0.010947160422801971], "W3": [[-0.07647992670536041, -0.07883844524621964, 0.23220594227313995, 0.17363116145133972, 0.1576569676399231, -0.03763984143733978, 0.23671644926071167, -0.13407060503959656, 0.156308114528656, -0.134093776345253, -0.11989147216081619, 0.10533309727907181, 0.10680994391441345, 0.009642151184380054, 0.032722316682338715, -0.08343129605054855, 0.05612136423587799, -0.19940906763076782, 0.07753536105155945, 0.24037514626979828, 0.1822875440120697, 0.2841562330722809, 0.16038550436496735, -0.20518146455287933, 0.2464977651834488, 0.12441133707761765, -0.1839192807674408, -0.12578733265399933, 0.17619751393795013, 0.17773795127868652, -0.12253939360380173, 0.09374094754457474]], "b3": [-0.046405091881752014]}, "2022": {"W1": [[-0.004733396228402853, -0.09959355741739273, 0.02873837761580944, -0.08755774796009064, 0.03581078723073006, -0.13314326107501984, 0.05272383987903595, -0.025898659601807594, 0.044085316359996796, 0.03196817263960838, 0.03162185102701187, -0.0009875918040052056, -0.015011167153716087, 0.0992833599448204, 0.036884013563394547, -0.09374813735485077, -0.08615460246801376, -0.09318441897630692, -0.03555679693818092, 0.07659460604190826, -0.010398558340966702, 0.08869069814682007, -0.094706229865551, -0.0071763573214411736, 0.08346611261367798, -0.0224484633654356, -0.026378458365797997, 0.08145425468683243, -0.16450925171375275, 0.061203356832265854, 0.1352677196264267, -0.08007046580314636, -0.0532974936068058, -0.1129290834069252, 0.006015946622937918], [-0.06742092221975327, -0.13431979715824127, -0.10049419850111008, -0.07958630472421646, 0.023561177775263786, -0.03267242759466171, -0.060334645211696625, 0.048326507210731506, -0.1158250942826271, 0.018115991726517677, 0.06894352287054062, 0.1544400006532669, -0.053213100880384445, -0.08644843846559525, 0.0018579954048618674, 0.0925724133849144, 0.05405448004603386, 0.09499344974756241, -0.001961172791197896, 0.06041910871863365, -0.0548710860311985, -0.03769543394446373, -0.06968159973621368, 0.0003782453713938594, -0.08402951806783676, -0.04575405642390251, -0.10328543931245804, 0.034869104623794556, -0.08747933059930801, -0.011462398804724216, -0.12383545935153961, 0.07107701152563095, 0.028549131006002426, -0.06065681204199791, -0.04593376815319061], [-0.09163287281990051, 0.005752690602093935, 0.04129564017057419, -0.06419961899518967, -0.12145574390888214, -0.09298238158226013, -0.09272582828998566, 0.07014485448598862, 0.03736618161201477, -0.06047326326370239, 0.12474563717842102, 0.05094452202320099, 0.03445245325565338, -0.04580584168434143, 0.06256908923387527, -0.0693434402346611, -0.02060302160680294, 0.11783789098262787, 0.08690665662288666, 0.04430019482970238, -0.08072806149721146, 0.04251912236213684, -0.03830200806260109, -0.10147900134325027, 0.009887179359793663, 0.071358323097229, 0.10344091802835464, 0.06854865700006485, 0.006328083109110594, 0.09547848999500275, 0.03184748440980911, 0.045900460332632065, -0.0235198512673378, 0.0014485131250694394, 0.022481193765997887], [0.022098645567893982, -0.005362531635910273, 0.020866304636001587, -0.04628366976976395, 0.06728234887123108, 0.03532743453979492, 0.10431721061468124, 0.040977660566568375, -0.02690545655786991, 0.0418335385620594, 0.05418507382273674, 0.1159588098526001, 0.05673396587371826, 0.060358718037605286, -0.00852732639759779, -0.13019587099552155, -0.0262787863612175, 0.022455891594290733, -0.01572389528155327, -0.04497383534908295, -0.13623853027820587, -0.051674943417310715, 0.02291947416961193, -0.06058516725897789, 0.01899145543575287, 0.04318646714091301, -0.09354482591152191, 0.04630599915981293, -0.1182810366153717, -0.10556045919656754, 0.13149967789649963, -0.2306024432182312, 0.0028512768913060427, 0.25136569142341614, 0.028941692784428596], [0.02888449840247631, -0.05731281638145447, 0.08718495815992355, -0.008272036910057068, 0.03910580277442932, -0.1021040678024292, -0.014311868697404861, 0.04529854655265808, 0.06448791921138763, 0.06201761215925217, 0.10229936987161636, 0.0021876113023608923, 0.010845501907169819, 0.0669064149260521, 0.058125656098127365, -0.1483997255563736, 0.17512241005897522, -0.09665337204933167, -0.1254139095544815, 0.0436360165476799, -0.08516282588243484, 0.03727671876549721, -0.042964447289705276, -0.04834256321191788, 0.02047654055058956, 0.07621407508850098, -0.0876075029373169, 0.045970361679792404, 0.14531880617141724, 0.08932885527610779, 0.047581229358911514, -0.20585615932941437, -0.014980918727815151, -0.04452425241470337, -0.0376288928091526], [0.03411353379487991, 0.012784261256456375, 0.002146054757758975, -0.10315550118684769, 0.08430342376232147, 0.010475787334144115, -0.03895173966884613, -0.012277974747121334, 0.04485369846224785, -0.01683579757809639, -0.08237050473690033, -0.0033851265907287598, -0.09720242023468018, -0.10005757957696915, 0.09245973825454712, -0.0186399444937706, -0.04646557942032814, 0.050686679780483246, 0.13695010542869568, -0.06674928218126297, -0.006685241591185331, 0.016694923862814903, 0.053122226148843765, -0.03478798642754555, -0.07178466022014618, 0.048170652240514755, -0.005630606785416603, 0.048432059586048126, 0.024960586801171303, 0.0768798366189003, -0.07632733136415482, 0.10359760373830795, 0.12109813094139099, 0.027133150026202202, 0.10028789192438126], [-0.055817022919654846, 0.05046481639146805, -0.02843533828854561, -0.10018398612737656, -0.08431430160999298, -0.08199912309646606, -0.011604673229157925, 0.038949303328990936, 0.005066806450486183, -0.09378014504909515, -0.010472497902810574, 0.029368754476308823, -0.03004516288638115, 0.01587342657148838, 0.04548100382089615, 0.07707736641168594, 0.05843736603856087, -0.11288901418447495, -0.08603905141353607, -0.08633307367563248, 0.036703746765851974, 0.055019836872816086, 0.05036803334951401, -0.007964774034917355, -0.03932889923453331, -0.05402509495615959, -0.05108568072319031, -0.09517891705036163, 0.008538641035556793, 0.07835064828395844, 0.06586931645870209, -0.09523849934339523, 0.03190302476286888, -0.02054629474878311, 0.014080137014389038], [0.07275538891553879, -0.14361362159252167, -0.09208301454782486, -0.14043229818344116, -0.03652701526880264, 0.15740077197551727, 0.0651029497385025, -0.10473384708166122, 0.11309342086315155, 0.006279362365603447, -0.09799318015575409, 0.014990312047302723, -0.08374478667974472, -0.0489790178835392, -0.05990137904882431, 0.10488671809434891, 0.026437466964125633, -0.1217465028166771, -0.08361557871103287, -0.06726331263780594, 0.06771975010633469, -0.07626540958881378, 0.033770136535167694, -0.02573353238403797, -0.08208057284355164, -0.09242311865091324, 0.11086352169513702, 0.026755154132843018, -0.00127465242985636, -0.02759777195751667, -0.0790591835975647, -0.1433316022157669, -0.0903129130601883, -0.06252756714820862, -0.08391845226287842], [0.07508858293294907, 0.005302001256495714, 0.07711952924728394, 0.04857274144887924, -0.09740743041038513, -0.11578284204006195, -0.08347843587398529, -0.10515253245830536, -0.05139267072081566, -0.07900791615247726, 0.10839782655239105, 0.10464152693748474, -0.10057315230369568, -0.06902731209993362, -0.08431380987167358, 0.00701260007917881, 0.03357532247900963, 0.0259128175675869, 0.10628535598516464, 0.06170802190899849, -0.017984047532081604, -0.05959292873740196, 0.005713683553040028, 0.06965315341949463, -0.023810114711523056, -0.07361295819282532, 0.09344842284917831, 0.12604005634784698, -0.007864783518016338, -0.013027491979300976, 0.018430495634675026, -0.07842057198286057, -0.06144123524427414, -0.08437313884496689, -0.10610540211200714], [0.04036974906921387, -0.041033267974853516, -0.09485774487257004, 0.05732259526848793, -0.06872188299894333, 0.013418590649962425, -0.093410424888134, 0.001267624320462346, 0.1048501506447792, -0.02197268046438694, -0.009924056939780712, 0.09232616424560547, 0.06164495274424553, -0.006267042830586433, -0.07310613989830017, -0.08266455680131912, -0.10052964091300964, -0.08082561194896698, 0.1008368507027626, -0.04690394550561905, 0.16889573633670807, -0.048830289393663406, -0.1042293831706047, -0.10482805967330933, -0.07185559719800949, -0.06846825778484344, 0.01771129108965397, -0.019255608320236206, 0.013852708041667938, -0.08131109923124313, -0.06507725268602371, -0.065477155148983, -0.002054952783510089, -0.10141907632350922, -0.13175497949123383], [-0.06925136595964432, 0.01923977956175804, -0.052644964307546616, -0.0653315857052803, -0.07352685928344727, 0.10029039531946182, 0.04464970529079437, -0.051627036184072495, 0.02451607957482338, 0.08797216415405273, -0.10028605908155441, 0.08174724131822586, 0.07757731527090073, -0.06842935085296631, -0.027663296088576317, 0.09247970581054688, -0.08580298721790314, -0.01943577639758587, -0.05344322323799133, 0.0695587620139122, 0.04672987759113312, -0.038063205778598785, 0.04918092116713524, 0.08461377769708633, -0.06543643772602081, -0.028667885810136795, 0.048087771981954575, 0.011835234239697456, -0.07383662462234497, -0.01365568581968546, -0.02978377230465412, 0.036387816071510315, -0.0005860856035724282, -0.11901815980672836, 0.01289499644190073], [-0.06550417840480804, 0.00325019471347332, -0.022044075652956963, -0.2003312110900879, -0.06835929304361343, -0.10849764198064804, -0.07163996994495392, 0.04910115897655487, 0.08164236694574356, 0.032507482916116714, 0.05993083119392395, 0.14555774629116058, -0.09287451207637787, -0.09411214292049408, -0.048096492886543274, -0.10946603864431381, 0.07144824415445328, -0.014977412298321724, -0.0338214747607708, -0.0760946273803711, 0.01585198938846588, 0.03462597727775574, -0.04324571043252945, -0.010148863308131695, 0.05414137989282608, 0.11294837296009064, -0.09803016483783722, -0.018982669338583946, -0.11948932707309723, -0.06227516010403633, -0.03098672442138195, -0.0006404240266419947, -0.04136189445853233, -0.049415431916713715, -0.07647660374641418], [-0.13273245096206665, -0.11352547258138657, 0.049342069774866104, -0.024814924225211143, 0.008928082883358002, 0.09681661427021027, 0.027833333238959312, -0.0018460239516571164, -0.05808676406741142, -0.04759326949715614, 0.04389583319425583, -0.07097907364368439, -0.035136960446834564, 0.05365705490112305, 0.11792538315057755, -0.03645099326968193, -0.08494363725185394, 0.049921125173568726, -0.03820359334349632, 0.03573272377252579, -0.06925702095031738, 0.04296546429395676, 0.008808784186840057, -0.054930999875068665, -0.058237504214048386, 0.10784647613763809, 0.05386970192193985, -0.02280156873166561, 0.05378775671124458, -0.10813732445240021, 0.025908365845680237, -0.16665637493133545, 0.05815526098012924, 0.15314726531505585, -0.027087636291980743], [0.04293302446603775, -0.0677567571401596, 0.04636214300990105, 0.0347641296684742, 0.02590598352253437, 0.01180028822273016, 0.015426326543092728, 0.06660431623458862, 0.06651686131954193, -0.0064133512787520885, 0.04963649809360504, 0.051184602081775665, 0.06051846221089363, -0.021923096850514412, -0.001520698657259345, -0.08947259187698364, 0.01037655584514141, 0.038361165672540665, 0.008415228687226772, 0.018380479887127876, -0.12170557677745819, 0.006541364826261997, -0.05079910159111023, -0.009571436792612076, -0.0160810686647892, -0.018656592816114426, -0.10511713474988937, 0.035271599888801575, 0.12342865765094757, 0.01953626237809658, 0.06588159501552582, -0.1795797199010849, 0.09770350903272629, 0.07571037113666534, -0.035589661449193954], [-0.07807054370641708, -0.001040913281030953, 0.005338698625564575, 0.05414367839694023, -0.053759556263685226, -0.07089880108833313, 0.009310661815106869, 0.06736458092927933, 0.029288101941347122, 0.031869154423475266, -0.06190769746899605, 0.01204894669353962, -0.006888503208756447, 0.006292919162660837, -0.04292745515704155, 0.007450851611793041, 0.08291836082935333, 0.03094073012471199, 0.06298816204071045, -0.08720006048679352, 0.1006944552063942, 0.05362707003951073, -0.061075448989868164, -0.062227144837379456, -0.06259473413228989, 0.017148157581686974, -0.10240188241004944, -0.004453985020518303, 0.0981435477733612, 0.01281551830470562, 0.04909629002213478, -0.13105738162994385, 0.09120497852563858, 0.0036138733848929405, 0.04254639521241188], [-0.012550966814160347, 0.11341475695371628, 0.00331861968152225, 0.11323230713605881, -0.033679768443107605, -0.09204097837209702, 0.0015240514185279608, 0.04594545438885689, 0.00021766923600807786, 0.06606661528348923, -0.020983492955565453, -0.03017512708902359, -0.12286434322595596, -0.008053896948695183, 0.0474386066198349, -0.06555896997451782, -0.0021481227595359087, 0.04953177645802498, -0.014246569946408272, -0.048566121608018875, -0.005200916435569525, 0.01983431726694107, -0.03924884274601936, -0.04820089787244797, -0.02731378935277462, -0.03938895836472511, -0.01285168994218111, -0.02554844506084919, -0.007435241248458624, -0.05966022238135338, 0.04226700961589813, 0.11156772077083588, 0.044963546097278595, 0.07527622580528259, -0.0036713951267302036], [0.0010478789918124676, 0.01869765855371952, 0.01773729920387268, -0.03806565701961517, -0.0181858092546463, -0.11891517788171768, 0.012822787277400494, -0.08605573326349258, -0.08527252078056335, 0.011886141262948513, 0.006758939474821091, 0.049020200967788696, -0.0048042465932667255, 0.016096217557787895, 0.04070475324988365, 0.13795305788516998, -0.026018161326646805, 0.059277936816215515, 0.07399644702672958, 0.014209048822522163, 0.10726549476385117, 0.050453074276447296, 0.022864125669002533, -0.018427981063723564, -0.05485427379608154, 0.08907398581504822, -0.02441408298909664, 0.003761051222681999, -0.09470202773809433, -0.016344115138053894, 0.013225484639406204, 0.04375464469194412, 0.003854414913803339, 0.11547606438398361, 0.08560678362846375], [-0.03673884645104408, 0.04195278882980347, 0.04990018159151077, 0.11347141861915588, 0.05635831877589226, -0.059592608362436295, -0.00040420974255539477, 0.006980941165238619, 0.08438202738761902, -0.027521487325429916, 0.09540075063705444, 0.04892244189977646, -0.07064095139503479, 0.07305207848548889, 0.11542706191539764, 0.07490585744380951, -0.0817624107003212, -0.008248033002018929, 0.06555674970149994, 0.0681401938199997, -0.0017644846811890602, -0.034938566386699677, 0.07900107651948929, 0.031439658254384995, -0.09964002668857574, 0.03511352837085724, -0.06783828884363174, 0.005755264777690172, 0.08708693832159042, -0.0014432868920266628, 0.01907437853515148, 0.1011122390627861, 0.0020449114963412285, -0.05615512654185295, -0.07147400826215744], [0.04283417761325836, 0.05907084047794342, 0.06362900137901306, -0.015023241750895977, 0.03754973039031029, -0.06581953912973404, 0.05245673656463623, 0.054405879229307175, -0.023138295859098434, 0.011877347715198994, -0.022570915520191193, -0.03012182004749775, -0.07261428982019424, -0.06527082622051239, -0.0822015255689621, -0.01778959482908249, 0.038090117275714874, 0.08008718490600586, 0.04614310339093208, -0.03556437790393829, 0.008631022647023201, -0.01118785236030817, 0.04246475175023079, 0.02514803782105446, -0.0415942445397377, 0.029792917892336845, 0.007245379034429789, 0.014743570238351822, -0.044496089220047, -0.04633839800953865, -0.062474776059389114, 0.09394180774688721, 0.00894592609256506, 0.02569546550512314, -0.07505013793706894], [0.014537314884364605, 0.001180144026875496, 0.009225865826010704, -0.013120027258992195, -0.04466263949871063, -0.021231167018413544, -0.019476115703582764, 0.023702355101704597, -0.022937722504138947, 0.040187492966651917, -0.00812850147485733, -0.03888848051428795, -0.0451364703476429, -0.014986270107328892, 0.051782842725515366, 0.016098419204354286, -0.023767556995153427, 0.04108814895153046, -0.02664259262382984, 0.016246095299720764, 0.05785577744245529, -0.028966786339879036, 0.03523845225572586, 0.014239582233130932, -0.010478717274963856, -0.026937391608953476, 0.10598060488700867, -0.002863441128283739, -0.049001503735780716, 0.020414792001247406, 0.07077563554048538, 0.0047506955452263355, 0.05100550130009651, 0.051485735923051834, -0.00593431806191802], [0.04424259439110756, -0.11081768572330475, -0.08015784621238708, -0.07930104434490204, -0.030838260427117348, -0.022754579782485962, -0.03792966529726982, -0.07410651445388794, -0.008970271795988083, -0.07547564059495926, -0.03730230778455734, 0.10784464329481125, -0.019527841359376907, 0.04240822419524193, -0.01346905529499054, 0.11239171028137207, 0.09552265703678131, -0.06281080096960068, -0.056647274643182755, 0.06605998426675797, -0.06154134124517441, 0.0856989249587059, 0.007276564836502075, 0.013778776861727238, -0.08621198683977127, 0.028314197435975075, -0.006120442412793636, -0.05116520822048187, 0.056866925209760666, -0.024816201999783516, -0.0024131364189088345, -0.09974859654903412, 0.038120415061712265, -0.11720924079418182, 0.05941582843661308], [0.041585858911275864, 0.11991193890571594, -0.06458038836717606, 0.05519822984933853, 0.06112445145845413, -0.04621545225381851, -0.0795041099190712, -0.05826636031270027, -0.023484744131565094, -0.0010458468459546566, -0.09925708174705505, 0.07289311289787292, -0.012379982508718967, -0.028152963146567345, -0.07045654207468033, 0.08069305866956711, 0.016365978866815567, -0.028360050171613693, -0.04577210173010826, 0.08618854731321335, 0.0914149135351181, 0.06210282817482948, 0.0173734650015831, 0.02383113093674183, 0.04035641625523567, 0.003263057442381978, 0.16623827815055847, -0.03561216965317726, -0.12909886240959167, 0.054171472787857056, -0.06280013173818588, 0.06901337206363678, -0.13518409430980682, -0.1431487798690796, 0.017938662320375443], [-0.0899001732468605, 0.13105320930480957, 0.03830773010849953, -0.09670106321573257, 0.015918567776679993, -0.016181522980332375, 0.050002604722976685, -0.016337960958480835, -0.014097662642598152, -0.01340373046696186, 0.06994205713272095, 0.06082766875624657, -0.030497493222355843, 0.06598382443189621, 0.061477676033973694, -0.16930466890335083, 0.09210547804832458, 0.0722653865814209, 0.07325968891382217, 0.02569778822362423, 0.09296360611915588, -0.0883193388581276, 0.022046338766813278, 0.015048281289637089, 0.032170072197914124, -0.10073871165513992, -0.16047434508800507, -0.010982399806380272, -0.029630472883582115, -0.19306810200214386, -0.15654340386390686, 0.31409236788749695, -0.09702397137880325, 0.17403246462345123, -0.009103100746870041], [-0.03431188315153122, -0.06650431454181671, 0.12831979990005493, 0.05591876804828644, -0.03753724694252014, -0.08724143356084824, 0.045327696949243546, -0.052722398191690445, 0.02269183099269867, 0.049037400633096695, 0.06846538186073303, -0.1133153960108757, -0.007427433039993048, 0.002657611621543765, 0.002120456425473094, 0.16068895161151886, -0.09322429448366165, -0.045962173491716385, 0.10009472072124481, -0.0003340588300488889, 0.04306541383266449, 0.022084232419729233, 0.012967047281563282, 0.034194424748420715, -0.0630347803235054, 0.1786833256483078, -0.007441455032676458, 0.013278322294354439, -0.17825187742710114, -0.13736291229724884, 0.04994400590658188, -0.1682409644126892, 0.04936216399073601, 0.15423761308193207, -0.01042945310473442], [-0.06854607164859772, -0.08681471645832062, -0.08157434314489365, -0.082526795566082, 0.0733800157904625, -0.05558253824710846, 0.08632846176624298, -0.06244474649429321, 0.07215612381696701, -0.06041761487722397, 0.07383327186107635, -0.01131961215287447, 0.00670196395367384, -0.02446526475250721, -0.07811466604471207, 0.057703256607055664, -0.05174372345209122, -0.02576465532183647, -0.00732856709510088, 0.0798950344324112, -0.06936637312173843, -0.05793323740363121, 0.07265432924032211, 0.006408653222024441, -0.021452587097883224, -0.09891188144683838, 0.15965525805950165, 0.056006088852882385, -0.15134775638580322, -0.014569549821317196, -0.098190076649189, -0.03030461072921753, 0.036998260766267776, -0.13904479146003723, 0.08376190066337585], [-0.0021473716478794813, -0.09642684459686279, 0.09188809990882874, -0.1735570728778839, 0.058642126619815826, -0.00672631012275815, 0.05749639496207237, 0.0058541810140013695, -0.04200959578156471, 0.05356236919760704, 0.039393484592437744, 0.07728441059589386, -0.018945207819342613, -0.011127827689051628, 0.016713907942175865, -0.02876427210867405, 0.11084475368261337, -0.08076681941747665, -0.05209975317120552, -0.13511386513710022, 0.07397154718637466, -0.08525232970714569, 0.012797192670404911, -0.07683972269296646, -0.012299662455916405, 0.07266402244567871, -0.12413445115089417, -0.02643871307373047, 0.05471210181713104, -0.06325621902942657, -0.11914197355508804, 0.01978806033730507, 0.023999283090233803, 0.17979197204113007, -0.06306690722703934], [0.041368529200553894, -0.034580428153276443, -0.01850258745253086, 0.02632267028093338, -0.008897422812879086, 0.1408689022064209, -0.03373383358120918, -0.09072693437337875, -0.041338834911584854, -0.003171598305925727, -0.014157857745885849, -0.011235366575419903, -0.013605273328721523, 0.011615744791924953, 0.06274675577878952, -0.08201442658901215, -0.01592213846743107, 0.06165734678506851, -0.0648505836725235, -0.039090316742658615, -0.01858340948820114, 0.006469863001257181, 0.00017737165035214275, 0.04740694910287857, 0.025983335450291634, -0.047555480152368546, 0.0063719367608428, -0.009129802696406841, 0.05288374423980713, 0.00846362579613924, -0.05460744723677635, -0.06071757525205612, 0.05457111448049545, 0.0428328663110733, 0.0509086437523365], [0.0037923110648989677, -0.026951344683766365, -0.08609052002429962, 0.019392425194382668, -0.08297538757324219, -0.041007936000823975, -0.08718868345022202, -0.02062457799911499, 0.04112597182393074, -0.00302307796664536, -0.14395977556705475, 0.14036040008068085, 0.044797103852033615, 0.13296453654766083, 0.1235962063074112, 0.09215778112411499, -0.015187007375061512, 0.06258407235145569, -0.029575277119874954, 0.11835821717977524, 0.12362156063318253, -0.10497752577066422, -0.07227804511785507, 0.08137799054384232, 0.08480867743492126, 0.03280538320541382, 0.10944350808858871, -0.08471623063087463, -0.09328459203243256, -0.04709380492568016, -0.07249914854764938, -0.12719759345054626, 0.09749459475278854, -0.06637785583734512, -0.008436060510575771], [0.05022392049431801, 0.0227216724306345, 0.10122118145227432, 0.016780942678451538, -0.07759741693735123, 0.07356936484575272, 0.04098564013838768, -0.00211139302700758, -0.04354700446128845, -0.042566634714603424, 0.10289470851421356, -0.047075334936380386, 0.0922837182879448, -0.084654301404953, -0.06649064272642136, -0.0739775151014328, 0.08178635686635971, 0.04775840789079666, -0.04953460395336151, -0.05035138130187988, 0.004241220187395811, 0.014985799789428711, -0.047891292721033096, -0.07490503787994385, -0.033227115869522095, -0.017973506823182106, 0.01784091256558895, -0.04114771634340286, 0.02370646223425865, 0.06985118240118027, 0.10854906588792801, -0.1468697041273117, -0.027937697246670723, 0.09596484899520874, -0.02891472540795803], [-0.012199458666145802, 0.045667488127946854, 0.0908391997218132, -0.04162966087460518, -0.02639283612370491, 0.002644506748765707, 0.05622747167944908, 0.06238633021712303, -0.004691610112786293, -0.009918679483234882, -0.029029112309217453, -0.004783141892403364, 0.011935118585824966, -0.07325693219900131, -0.07880116999149323, 0.013142556883394718, 0.08314745873212814, 0.05752437934279442, 0.021003354340791702, 0.03627505525946617, 0.06087769940495491, -0.07141091674566269, 0.06013759598135948, -0.05438356101512909, -0.05297496169805527, -0.005442263558506966, 0.0055778068490326405, -0.014282280579209328, -0.03821360319852829, -0.035677719861269, -0.056514374911785126, 0.12928228080272675, 0.0646936297416687, 0.09714356809854507, -0.06841283291578293], [0.06160564348101616, 0.009005766361951828, 0.145926371216774, -0.029084715992212296, 0.05188139155507088, 0.07020566612482071, 0.03150717541575432, -0.04190414771437645, -0.020404569804668427, 0.06307247281074524, -0.029864409938454628, 0.02638065256178379, -0.04659896343946457, -0.06348412483930588, -0.010240334086120129, -0.07286360114812851, -0.062114834785461426, 0.02903280034661293, 0.00911327451467514, 0.0764090046286583, 0.09680355340242386, 0.0037570965941995382, 0.06404543668031693, -0.06717462837696075, -0.046413011848926544, 0.041232410818338394, 0.07980848848819733, 0.016104815527796745, 0.017620744183659554, 0.0096984151750803, 1.710804644972086e-05, 0.08413305133581161, 0.03955845534801483, 0.002410746179521084, -0.07268066704273224], [-0.018852580338716507, 0.03198734298348427, 0.13497669994831085, 0.009589544497430325, 0.06272688508033752, -0.04743483290076256, -0.03170304745435715, 0.010107126086950302, 0.018971621990203857, 0.00394394900649786, 0.004520331509411335, 0.00576493376865983, -0.026127314195036888, 0.020446963608264923, 0.010201038792729378, 0.033310361206531525, 0.02471114508807659, -0.026547562330961227, -0.021658265963196754, -0.006381149869412184, -0.04735371470451355, 0.0390726663172245, 0.021793168038129807, 0.00740874744951725, 0.025539491325616837, -0.031830236315727234, -0.017867667600512505, -0.022889310494065285, 0.019251983612775803, 0.05155384913086891, 0.035021066665649414, -0.002577270148321986, -0.020115798339247704, 0.007492778357118368, -0.02033478394150734], [-0.09637923538684845, -0.07239363342523575, 0.05540555343031883, 0.0586482472717762, -0.07630991190671921, 0.02402130514383316, -0.06317301094532013, -0.1101021021604538, -0.004275788087397814, -0.07824625819921494, -0.06204213947057724, -0.003481845138594508, 0.003689088858664036, -0.04609006643295288, 0.013868806883692741, -0.03732352331280708, -0.04404739663004875, 0.024736914783716202, 0.03485182672739029, -0.05274776369333267, 0.07619409263134003, 0.028675386682152748, 0.005766208749264479, -0.03204029053449631, -0.06216248497366905, 0.08803708106279373, -0.0010811376851052046, 0.019236788153648376, -0.02043033577501774, -0.07302242517471313, -0.06941407918930054, 0.12179167568683624, 0.006755438167601824, 0.1368110030889511, -0.051743824034929276], [0.019414346665143967, -0.007402354851365089, 0.046000394970178604, 0.017751779407262802, -0.03599494323134422, -0.05287393182516098, -0.03731772303581238, 0.02707519195973873, 0.06663293391466141, 0.07584452629089355, -0.00526773277670145, -0.03914706036448479, -0.07833290845155716, -0.013169348239898682, -0.042988371104002, 0.03418562561273575, -0.031855080276727676, 0.030206015333533287, 0.08750387281179428, -0.08889075368642807, 0.10048684477806091, 0.022608358412981033, -0.08671780675649643, 0.04489576816558838, 0.06474728137254715, -0.04597073793411255, 0.025842424482107162, 0.06849878281354904, 0.05756001174449921, 0.0187620110809803, -0.033480044454336166, 0.0801389291882515, -0.07321338355541229, 0.026863953098654747, -0.03702317178249359], [0.02443382516503334, -0.09044650942087173, -0.04953226447105408, -0.07670358568429947, 0.0602097325026989, 0.06391091644763947, 0.0330265648663044, -0.019135896116495132, -0.0853976234793663, -0.03718781843781471, -0.12590831518173218, 0.03605515509843826, -0.04933549091219902, 0.05737047269940376, 0.09614188969135284, -0.08799071609973907, 0.059088125824928284, 0.05652166157960892, -0.07035268843173981, -0.09205400943756104, 0.15229858458042145, -0.06597583740949631, 0.04191868007183075, 0.04804357513785362, 0.03297493979334831, -0.12112757563591003, 0.0730370432138443, -0.0686914324760437, -0.10066548734903336, 0.11130283772945404, -0.04199434071779251, -0.06507008522748947, -0.11961391568183899, -0.0021724230609834194, -0.06290727108716965], [-0.07683585584163666, -0.042741063982248306, 0.03531231731176376, -0.021812915802001953, -0.033585067838430405, 0.06136689707636833, -0.013583448715507984, -0.020126385614275932, -0.008571465499699116, 0.043918076902627945, -0.007645336911082268, -0.038572829216718674, 0.008385476656258106, 0.019581584259867668, 0.03397321701049805, 0.06652000546455383, 0.038853175938129425, -0.014057320542633533, 0.019323555752635002, -0.03711911663413048, 0.08388543874025345, -0.024574952200055122, -0.02652410790324211, -0.019174637272953987, -0.06494662165641785, 0.09011173248291016, -0.011120656505227089, 0.039622124284505844, 0.03630991652607918, -0.08266733586788177, 0.013074955902993679, 0.016017688438296318, 0.029009506106376648, 0.15855202078819275, 0.014447374269366264], [-0.058940380811691284, -0.07181759178638458, 0.010475792922079563, 0.0713571086525917, -0.08609946817159653, -0.00013957939518149942, -0.03597218543291092, -0.010505784302949905, -0.09336709976196289, 0.09034605324268341, -0.06078163906931877, -0.06032538414001465, -0.07313568890094757, -0.014080757275223732, 0.08920909464359283, 0.013221930712461472, -0.0579463392496109, -0.040945056825876236, -0.045478060841560364, -0.07367388904094696, 0.05298970639705658, -0.06749946624040604, -0.06680051237344742, -0.05206463113427162, 0.047959182411432266, 0.03880254179239273, -0.09185262769460678, 0.10547851771116257, 0.07482326030731201, -0.019062479957938194, -0.004082394763827324, 0.049071572721004486, -0.04926345497369766, 0.03683818504214287, -0.010479643940925598], [0.025549935176968575, 0.023644085973501205, -0.0054391613230109215, 0.06349056959152222, 0.10572901368141174, -0.05397666618227959, -0.06067441776394844, 0.11890948563814163, -0.027697322890162468, 0.026806075125932693, -0.025517255067825317, 0.017728352919220924, -0.013081716373562813, 0.0774950161576271, 0.058158546686172485, -0.13778606057167053, -0.09655539691448212, -0.005414905492216349, 0.06436752527952194, 0.06228827312588692, 0.10197937488555908, -0.007141143549233675, -0.026474473997950554, 0.003602920100092888, 0.10338287800550461, 0.11586783826351166, -0.18391898274421692, -0.04300398752093315, 0.14944292604923248, -0.1268058568239212, -0.14863356947898865, 0.2962701916694641, 0.006980159319937229, 0.14242836833000183, -0.018688147887587547], [0.00600680336356163, -0.028597813099622726, 0.003920039627701044, -0.024421604350209236, 0.004900669679045677, 0.03349212184548378, -0.08420180529356003, 0.02467334270477295, 0.03362284600734711, -0.009888808242976665, 0.02264968305826187, -0.02660905383527279, 0.01822805032134056, 0.06881149113178253, 0.072385273873806, 0.07594360411167145, -0.03480396419763565, -0.05270465835928917, 0.025172356516122818, -0.018221788108348846, 0.03772484138607979, -0.04944657161831856, 0.01795155368745327, 0.053612079471349716, -0.02891453728079796, 0.10343357175588608, -0.03108387067914009, -0.006571351550519466, -0.025642653927206993, -0.038653720170259476, -0.016152333468198776, 0.0431034080684185, -0.04562871530652046, 0.11947522312402725, 0.05840158089995384], [-0.0011184245813637972, -0.027696073055267334, 0.0715799629688263, 0.07586533576250076, -0.005791161209344864, 0.09582154452800751, 0.08517424762248993, 0.09258381277322769, 0.00464394548907876, 0.06842683255672455, -0.07912088930606842, 0.035480011254549026, -0.006948532536625862, 0.046194277703762054, 0.04329816251993179, -0.1283678114414215, -0.05116170272231102, -0.07221607118844986, 0.03081192448735237, -0.027998127043247223, 0.04588599503040314, 0.025398828089237213, -0.04121130704879761, 0.12002543359994888, -0.12579375505447388, 0.09322415292263031, -0.13797594606876373, 0.012072344310581684, -0.07583335787057877, 0.04037477821111679, 0.01554564107209444, -0.08028983324766159, 0.10252155363559723, 0.006685296073555946, 0.07981888949871063], [0.029961738735437393, 0.09090365469455719, 0.03458978608250618, 0.03748482093214989, 0.116185761988163, -0.04534997045993805, 0.07247047871351242, 0.04038773849606514, -0.01426883228123188, 0.03993409126996994, 0.06410331279039383, -0.04922568425536156, -0.033606838434934616, -0.11761023849248886, -0.15465791523456573, -0.005635390989482403, 0.027903201058506966, 0.03174645081162453, 0.005926945246756077, -0.030946100130677223, -0.08605466037988663, -0.010617595165967941, -0.03116050362586975, 0.037499066442251205, 0.061517875641584396, 0.1047663614153862, -0.03885584697127342, -0.0639943778514862, 0.03369826078414917, -0.12829509377479553, 0.03971860557794571, 0.1528477519750595, 0.011897369287908077, -0.00749591737985611, 0.03786678612232208], [-0.030833931639790535, -0.040596310049295425, 0.0047067319974303246, -0.023495355620980263, -0.02011636272072792, -0.01946847513318062, 0.04247896000742912, 0.0793200358748436, 0.03717045858502388, -0.008050997741520405, -0.05382988974452019, -0.04215528070926666, -0.031126875430345535, -0.03898710384964943, -0.027195444330573082, -0.03511175140738487, -0.019966362044215202, 0.03899218887090683, 0.05031872168183327, -0.010233999229967594, -0.1338692009449005, 0.014786246232688427, -0.10070788115262985, -0.08677888661623001, -0.1487809717655182, -0.06064055114984512, -0.08186686784029007, -0.07157360017299652, -0.008492193184792995, 0.10094375163316727, 0.08021578192710876, -0.021931109949946404, 0.0678555890917778, -0.046106401830911636, -0.05012502893805504], [-0.013875055126845837, 0.04381813853979111, -0.06922701001167297, -0.03001413494348526, 0.005681945942342281, -0.05711464211344719, -0.020654795691370964, 2.3113430870580487e-05, 0.05018184706568718, 0.06424544006586075, -0.018352745100855827, -0.09371598809957504, 0.08859331160783768, -0.011705975979566574, 0.020979996770620346, 0.0013415986904874444, -0.04612326994538307, -0.039198748767375946, 0.027371680364012718, 0.03459039330482483, -0.0071236020885407925, -0.027878666296601295, -0.043782737106084824, 0.0126594677567482, 0.009554116055369377, 0.08726777136325836, 0.07994833588600159, 0.056206174194812775, -0.019172342494130135, -0.08745331317186356, -0.028106173500418663, 0.03235148638486862, -0.05387931317090988, 0.0035312692634761333, -0.06676778942346573], [0.036475904285907745, 0.05648024007678032, 0.01821104623377323, 0.08206108957529068, 0.0462893545627594, -0.009243191219866276, 0.025836724787950516, 0.07023310661315918, 0.07376673072576523, 0.021605011075735092, 0.07193587720394135, -0.020321613177657127, -0.03846567124128342, -0.03747078776359558, -0.016210056841373444, -0.12250012904405594, 0.016873640939593315, -0.04423875734210014, -0.04589412361383438, 0.0044554597698152065, -0.047786638140678406, -0.047402381896972656, -0.02176298201084137, -0.05216674134135246, 0.0038444006349891424, -0.09494968503713608, -0.046041492372751236, 0.017416058108210564, 0.07389730960130692, 0.046922340989112854, 0.03670459985733032, -0.08644578605890274, 0.021264275535941124, -0.07614947110414505, -0.05128324031829834], [-0.06186158210039139, 0.05648137256503105, 0.023782575502991676, 0.03665947541594505, 0.08497269451618195, 0.020791957154870033, 0.06263068318367004, -0.07994676381349564, 0.05423008278012276, 0.08019520342350006, 0.00039297534385696054, -0.0675627812743187, 0.0016404414782300591, 0.04770538955926895, -0.053447410464286804, 0.08851383626461029, 0.011350609362125397, -0.0031820752192288637, -0.08496715128421783, 0.04725902900099754, 0.10204771906137466, -0.013326063752174377, -0.032510653138160706, 0.1186448410153389, 0.06953147053718567, -0.03864217922091484, 0.04682254046201706, 0.04485074430704117, 0.014616571366786957, -0.1115044504404068, 0.05069689452648163, 0.11592384427785873, -0.12487325072288513, 0.00512703089043498, 0.01741850934922695], [0.004291050601750612, -0.07646297663450241, -0.0480712465941906, 0.0342758372426033, 0.0029861479997634888, -0.06064974144101143, 0.014329499565064907, -0.021352611482143402, 0.06511449068784714, -0.011135263368487358, -0.11001244187355042, 0.021166475489735603, 0.08418868482112885, 0.015114348381757736, -0.012327185831964016, -0.034622922539711, -0.02748752199113369, -0.06265175342559814, 0.026034748181700706, 0.053697921335697174, -0.1214829683303833, 0.008749640546739101, 0.009328480809926987, -0.0907699316740036, 0.008032643236219883, 0.08712657541036606, -0.061380479484796524, -0.01145312748849392, 0.020078929141163826, -0.02194633148610592, -0.03386411443352699, -0.1604326367378235, 0.04417666047811508, 0.19176213443279266, -0.06582257151603699], [0.02007230557501316, 0.05221133679151535, 0.02759578451514244, 0.0516449511051178, 0.008199602365493774, 0.03487372398376465, 0.028175506740808487, -0.0012064111651852727, 0.06249517947435379, -0.03973302245140076, -0.06896822154521942, -0.03226606547832489, -0.09427590668201447, -0.07924758642911911, -0.05065436661243439, -0.09343486279249191, -0.0014481558464467525, 0.09523458033800125, 0.08221331983804703, 0.03492944687604904, 0.03992462530732155, -0.02250528149306774, -0.0021154333371669054, -0.02391287311911583, -0.03860178217291832, -0.04648533836007118, 0.049232061952352524, 0.06489135324954987, -0.09183033555746078, 0.03423216566443443, -0.0632232278585434, -0.04543858766555786, -0.02001119777560234, 0.01932990737259388, 0.033391278237104416], [0.028226366266608238, 0.11246821284294128, -0.08466867357492447, -0.033235225826501846, -0.048567548394203186, -0.012004136107861996, -0.05376593396067619, 0.05553596466779709, -0.05282632261514664, -0.059381503611803055, 0.06672658771276474, -0.062436848878860474, 0.04355956241488457, 0.04743386432528496, -0.019786007702350616, 0.05868932232260704, 0.048749733716249466, 0.12991540133953094, -0.14702552556991577, -0.03896491229534149, -0.06303227692842484, -0.1620338261127472, -0.06530652195215225, 0.03406903147697449, 0.05541666969656944, -0.045770056545734406, -0.02561200223863125, -0.021876078099012375, -0.016677672043442726, 0.015946904197335243, -0.16932764649391174, 0.15541288256645203, -0.07474400103092194, -0.05125577747821808, -0.06850838661193848], [0.027124933898448944, 0.07979798316955566, -0.04930118843913078, -0.021528175100684166, -0.010742858983576298, 0.09110858291387558, 0.07613712549209595, 0.00328921340405941, 0.0507911778986454, -0.004134667105972767, -0.08405950665473938, 0.1555309146642685, -0.0033878276590257883, 0.04333765059709549, -0.1105944812297821, -0.11590244621038437, -0.11218401789665222, -0.0892825648188591, -0.012219514697790146, -0.016286879777908325, 0.14112137258052826, -0.03278736770153046, 0.03301054984331131, 0.01847200281918049, -0.02899642288684845, -0.0149748669937253, 0.0973326563835144, -0.0712859183549881, -0.016171736642718315, 0.08333317190408707, 0.016932422295212746, 0.02968578413128853, -0.004267948213964701, 0.08817479014396667, 0.06822127103805542], [-0.09042049944400787, -0.011626588180661201, 0.024829013273119926, 0.010545330122113228, 0.041907038539648056, 0.008860049769282341, -0.07960814982652664, 0.011870834976434708, -0.0020280741155147552, 0.039249930530786514, -0.015371467918157578, -0.036777835339307785, 0.09320850670337677, 0.03776799142360687, -0.03211278095841408, -0.03931617736816406, 0.007247579749673605, -0.06388870626688004, 0.045770395547151566, -0.04251899570226669, 0.025324692949652672, 0.02394615299999714, 0.036507897078990936, -0.005964148789644241, -0.014985015615820885, 0.06816420704126358, -0.01089860126376152, -0.03863763436675072, -0.047523897141218185, 0.0054512606002390385, -0.033147238194942474, -0.011489424854516983, 0.02621367760002613, 0.04088355973362923, -0.006594945676624775], [-0.0381590761244297, -0.06779088079929352, 0.0856333002448082, -0.0005125464522279799, -0.08750249445438385, 0.1032312884926796, 0.05846797302365303, -0.02657989226281643, -0.029174786061048508, -0.00033753764000721276, 0.0439307726919651, 0.04296543449163437, 0.023768050596117973, 0.02035771869122982, 0.11228231340646744, 0.006873074918985367, -0.0024951647501438856, -0.0026035436894744635, 0.0669909119606018, -0.0755353569984436, -0.10408986359834671, 0.05327324569225311, 0.04532669857144356, -0.057441093027591705, -0.007403409108519554, 0.12685681879520416, 0.05667218938469887, -0.05135595425963402, 0.0862850695848465, -0.10937488824129105, 0.16465428471565247, -0.2468748241662979, 0.025171231478452682, 0.20318850874900818, 0.09053710848093033], [0.11601749062538147, -0.053831007331609726, -0.19937491416931152, 0.06357792019844055, 0.0890129879117012, 0.003974179737269878, -0.08497462421655655, 0.021787352859973907, -0.031614359468221664, -0.04895734786987305, 0.06466919183731079, 0.11512283980846405, 0.14222954213619232, 0.04919521510601044, 0.09311912208795547, -0.0332060344517231, 0.0050308601930737495, 0.01657022163271904, -0.12241614609956741, -0.08801925927400589, -0.04057104513049126, 0.0030263797380030155, -0.00864848680794239, -0.072199247777462, 0.0682293176651001, 0.09314362704753876, 0.06121455878019333, -0.04723586142063141, 0.11706507951021194, 0.058362144976854324, 0.003005680162459612, -0.00464724050834775, -0.07490570843219757, 0.002470517996698618, 0.06903905421495438], [-0.05767462030053139, -0.14625979959964752, 0.11527373641729355, -0.039132434874773026, 0.03999621793627739, -0.019793977960944176, 0.046279679983854294, 0.02077365852892399, -0.03933711722493172, 0.04577365145087242, -0.0030464730225503445, 0.061573248356580734, -0.015101180411875248, -0.04855291545391083, 0.0547722727060318, -0.09671482443809509, 0.09797808527946472, 0.019010361284017563, 0.030804462730884552, 0.031405918300151825, -0.009478666819632053, 0.04809241369366646, -0.06397385895252228, -0.020854054018855095, -0.030039189383387566, 0.07317984849214554, -0.00024037797993514687, 0.0192011259496212, 0.07052192836999893, -0.009514864534139633, 0.009043360128998756, -0.225126713514328, 0.1399570256471634, 0.1904357224702835, 0.03084024041891098], [-0.005854272283613682, 0.035501882433891296, 0.06810438632965088, 0.050414543598890305, 0.0954573005437851, -0.08455166220664978, -0.02869098260998726, 0.026248149573802948, -0.08482202887535095, -0.0075738993473351, 0.05219398811459541, 0.004148515872657299, -0.04745154827833176, -0.05647240951657295, 0.007282621692866087, 0.0760723277926445, 0.07859338819980621, -0.06338758766651154, -0.0423184409737587, 0.011308923363685608, -0.09688443690538406, -0.029523136094212532, -0.028459183871746063, -0.04295625165104866, -0.04918024316430092, -0.016999192535877228, 0.0707763135433197, 0.08191778510808945, 0.01117610838264227, 0.03918952867388725, 0.08638603240251541, 0.038062963634729385, -0.04804442077875137, 0.0307269599288702, 0.018272506073117256], [-2.066745400952641e-05, -0.06896715611219406, 0.033197276294231415, -0.08432585746049881, -0.037469860166311264, 0.014027140103280544, 0.0504891499876976, -0.0323651097714901, -0.049163125455379486, 0.012144295498728752, -0.022049909457564354, -0.050094809383153915, -0.03721978887915611, 0.04945753887295723, 0.02295813336968422, -0.03983589634299278, 0.06645246595144272, 0.027872074395418167, -0.05801234021782875, -0.059904634952545166, 0.009794394485652447, 0.01837877742946148, 0.038982782512903214, 0.0030814174097031355, 0.023107776418328285, -0.016848228871822357, -0.04977980628609657, -0.016574695706367493, 0.007898068055510521, -0.0639645978808403, -0.004146779887378216, -0.10557322949171066, 0.07192158699035645, 0.0033133188262581825, -0.05540244281291962], [-0.06287363171577454, 0.03626208007335663, 0.23693259060382843, 0.09721656888723373, 0.16823624074459076, -0.0841829925775528, -0.005491463001817465, -0.10745199769735336, 0.02644987218081951, 0.010930903255939484, 0.1610485166311264, -0.04855690151453018, -0.057756949216127396, 0.15761572122573853, -0.010811498388648033, 0.0233808271586895, -0.1406535655260086, -0.02499445155262947, 0.05489766597747803, -0.07948228716850281, 0.06015189364552498, -0.05138377472758293, 0.002167433500289917, 0.07386791706085205, -0.03323553130030632, 0.028077514842152596, -0.011721362359821796, -0.0392998605966568, 0.13277162611484528, 0.07906487584114075, -0.0018063319148495793, 0.20221304893493652, 0.0596403107047081, 0.03453563153743744, -0.019300473853945732], [0.09185999631881714, 0.008637240156531334, 0.04859341308474541, 0.031784649938344955, -0.0903005301952362, -0.037902768701314926, -0.030575312674045563, 0.13304434716701508, -0.049424074590206146, -0.03128449618816376, 0.07434701174497604, 0.023548245429992676, 0.029355837032198906, 0.037511736154556274, 0.044793158769607544, 0.04437805712223053, -0.06460912525653839, 0.038597509264945984, 0.060461290180683136, -0.025372760370373726, 0.13316050171852112, 0.0745847076177597, 0.05938982591032982, -0.049796897917985916, 0.034070685505867004, 0.020062247291207314, 0.06017296016216278, -0.03614085167646408, 0.11927860230207443, -0.0752900019288063, -0.06914283335208893, 0.09835866093635559, 0.07299921661615372, 0.04697442427277565, -0.09367556124925613], [-0.0808437317609787, -0.10303269326686859, 0.07745512574911118, -0.01978965848684311, 0.0077234371565282345, -0.020373737439513206, 0.08963070809841156, -0.013491744175553322, -0.030093718320131302, 0.00799520779401064, -0.017715562134981155, -0.03134306147694588, 0.13300156593322754, -0.01526452787220478, 0.04017362371087074, 0.050104644149541855, 0.08236489444971085, -0.08524103462696075, 0.03887590020895004, 0.040488921105861664, -0.10137419402599335, -0.06949455291032791, 0.03414217755198479, 0.053518444299697876, -0.1058584451675415, 0.23666445910930634, 0.08530497550964355, -0.008579934015870094, -0.09397725015878677, -0.0012950521195307374, 0.0936877429485321, -0.20795753598213196, 0.04703294113278389, 0.18369975686073303, -0.012763934209942818], [-0.07903803884983063, 0.018878037109971046, 0.011136699467897415, 0.05066118761897087, -0.05927366018295288, 0.11994321644306183, -0.05296190455555916, 0.06765343993902206, 0.09546035528182983, 0.041467051953077316, 0.013765928335487843, -0.07731406390666962, -0.0017429149011150002, -0.02058205008506775, -0.04418431594967842, -0.06934022903442383, -0.058909282088279724, -0.006725179497152567, 0.09745774418115616, 0.06611235439777374, 0.15341170132160187, 0.01640985906124115, 0.025794830173254013, -0.025130201131105423, -0.05117560923099518, 0.07714027166366577, -0.06843538582324982, -0.035354647785425186, -0.05602530762553215, -0.025941038504242897, -0.08916717767715454, 0.08031190186738968, -0.0030988522339612246, 0.04865160584449768, -0.09557384997606277], [0.023948252201080322, -0.016363784670829773, -0.00588830653578043, -0.017081674188375473, 0.028000470250844955, 0.03201039135456085, -0.042681463062763214, -0.03235403075814247, -0.007486061193048954, 0.005244823172688484, -0.011435517109930515, -0.011164804920554161, 0.025320878252387047, 0.05954397842288017, 0.10026463866233826, 0.018879951909184456, -0.08230297267436981, -0.09048241376876831, 0.049584560096263885, -0.016618968918919563, -0.04325440153479576, -0.00881211832165718, -0.005507203750312328, -0.03780616819858551, -0.08109602332115173, -0.11153043806552887, 0.10338801890611649, 0.04922371357679367, 0.05931350588798523, 0.06789149343967438, 0.004999896977096796, -0.041725292801856995, 0.04395987465977669, -0.042861487716436386, -0.024504775181412697], [-0.012724906206130981, -0.02245410531759262, 0.02730928175151348, -0.09166154265403748, 0.010978865437209606, 0.03134728968143463, 0.03629660978913307, -0.08602260053157806, -0.04437738656997681, -0.0945919007062912, 0.05513285472989082, 0.10988844186067581, 0.06250959634780884, 0.0014962275745347142, -0.030825858935713768, -0.030906476080417633, -0.06461790949106216, -0.051038503646850586, -0.039010971784591675, -0.009173475205898285, 0.09787530452013016, 0.056692935526371, -0.07349531352519989, 0.02493223547935486, -0.09162908792495728, 0.12339389324188232, 0.04899849742650986, 0.049555595964193344, -0.04430641233921051, -0.10202008485794067, 0.07386939227581024, -0.045510176569223404, 0.04980575665831566, 0.07870184630155563, 0.04250451549887657], [0.0020587616600096226, -0.0309409461915493, 0.054331663995981216, -0.00044074890320189297, 0.03652028366923332, -0.0379013866186142, 0.01844615861773491, -0.0321648009121418, 0.027018539607524872, -0.012180075980722904, -0.008827892132103443, 0.035612232983112335, 0.027060307562351227, 0.020318880677223206, 0.038972169160842896, 0.02801579236984253, -0.10424847900867462, 0.021298600360751152, 0.0882834866642952, -0.07536042481660843, 0.0251305028796196, -0.0036926728207618, 0.04700223356485367, -0.05971192196011543, -0.008269581012427807, 0.09541501104831696, 0.032349843531847, 0.030582785606384277, 0.03604843094944954, 0.001716841128654778, -0.023015158250927925, -0.08058270066976547, 0.02448519878089428, 0.03250505402684212, 0.018527802079916], [0.06791738420724869, -0.022529030218720436, -0.020021796226501465, -0.048017144203186035, 0.06421792507171631, -0.061871111392974854, 0.017302559688687325, -0.07617557048797607, -0.00836497638374567, -0.009191934950649738, -0.004350197967141867, 0.03723727911710739, 0.04353487491607666, 0.0003817645483650267, -0.08108482509851456, -0.06396173685789108, -0.014935832470655441, 0.043151676654815674, -0.003629214596003294, 0.02005518414080143, -0.07512810081243515, 0.0004937151679769158, 0.03132057189941406, 0.0924643948674202, -0.026527447625994682, -0.008591053076088428, 0.0875529944896698, 0.028830017894506454, -0.031081389635801315, 0.010732521302998066, 0.03693937510251999, -0.024496309459209442, 0.012907966040074825, -0.007958155125379562, 0.02872290089726448], [-0.07709701359272003, 0.018606793135404587, 0.0036356328055262566, 0.02255672588944435, -0.15415741503238678, 0.11699589341878891, -0.02769562043249607, -0.02616923488676548, 0.04487154632806778, 0.04697811231017113, -0.05012549087405205, -0.0527983196079731, -0.07184851169586182, 0.03025149554014206, 0.04330458864569664, -0.10146462172269821, -0.007639430463314056, -0.07247476279735565, -0.010361630469560623, 0.0513722188770771, 0.03954760730266571, -0.04054993391036987, -0.04582713916897774, -0.004459647927433252, 0.06663021445274353, 0.04679014906287193, -0.04966190457344055, 0.015738410875201225, -0.08472724258899689, -0.08062025904655457, 0.08092140406370163, 0.07695189863443375, -0.01663826033473015, 0.03553971275687218, -0.015955546870827675], [0.03220623731613159, 0.052173443138599396, -0.0012092696269974113, -0.017893072217702866, -0.030630474910140038, -0.05663912743330002, -0.06408832222223282, 0.01337197795510292, 0.05461610108613968, 0.017458591610193253, -0.09208545088768005, -0.05611850693821907, 0.01915302313864231, 0.02250341884791851, -0.05363689363002777, 0.08896872401237488, -0.02204366773366928, -0.03165436536073685, -0.07717707008123398, 0.03980623558163643, 0.005031444597989321, -0.026078635826706886, 0.003271131543442607, 0.05141741409897804, -0.001369720557704568, -0.025789247825741768, 2.793084604491014e-05, 0.017929522320628166, 0.009009587578475475, -0.06838605552911758, -0.07896572351455688, -0.03414727374911308, 0.06741176545619965, 0.0628432035446167, -0.016125839203596115], [-0.04777420312166214, 0.04871115833520889, 0.12691037356853485, 0.04447381570935249, -0.07201026380062103, 0.14124491810798645, 0.0040936460718512535, -0.022311978042125702, -0.02781054936349392, 0.0038078108336776495, 0.07687082141637802, 0.060941655188798904, -0.007209894713014364, 0.0026752816047519445, -0.012446981854736805, 0.012580979615449905, 0.0687335878610611, 0.11144307255744934, 0.0045525734312832355, 0.02792554348707199, -0.07070853561162949, 0.11655734479427338, 0.015244841575622559, 0.048893239349126816, 0.09703691303730011, 0.018699886277318, -0.08777355402708054, 0.04031447693705559, 0.0404740571975708, 0.08419609069824219, 0.04598192125558853, -0.04518987610936165, -0.025325892493128777, -0.011187005788087845, -0.0180868748575449], [-0.03548075631260872, 0.04974443092942238, 0.04665110632777214, -0.0016127033159136772, 0.018704712390899658, 0.13800518214702606, 0.10222989320755005, -0.09341508150100708, 0.016628962010145187, -0.009371830150485039, 0.06843472272157669, -0.030483964830636978, -0.01887187547981739, -0.02979615144431591, -0.04239043965935707, -0.08845625817775726, 0.059575438499450684, 0.01855630986392498, 0.0405673012137413, 0.045780621469020844, -0.02520653046667576, 0.024291297420859337, -0.05732715129852295, -0.015954146161675453, -0.07478567212820053, -0.1505683958530426, -0.09987650066614151, 0.013246438466012478, -0.031247427687048912, 0.09354326128959656, 0.0003718145890161395, 0.09560936689376831, 0.09050484001636505, -0.15939128398895264, -0.005711516831070185], [-0.034658242017030716, -0.11605914682149887, -0.006185571197420359, -0.10873226076364517, 0.01937675289809704, 0.04378759115934372, -0.020412512123584747, 0.021283593028783798, -0.06754224747419357, -0.09065646678209305, 0.049007367342710495, 0.025741582736372948, 0.02199864201247692, -0.058176856487989426, 0.08149363845586777, -0.0004270999925211072, 0.07024937123060226, -0.06938280165195465, -0.0764780268073082, -0.026614118367433548, 0.00831393338739872, -0.03274152800440788, -0.0071989623829722404, 0.059570420533418655, -0.09906814247369766, -0.03643106669187546, -0.03904615342617035, 0.01188705489039421, -0.036837074905633926, 0.10766681283712387, -0.051662154495716095, 0.016700800508260727, -0.09334135055541992, -0.04774883762001991, -0.019396185874938965], [0.08375909924507141, -0.00988113321363926, 0.03861379995942116, -0.07864128798246384, 0.008570232428610325, -0.0710291936993599, 0.008383152075111866, 0.06813003122806549, -0.00411128718405962, 0.06767510622739792, 0.04057495668530464, -0.05897527560591698, -0.01540297083556652, 0.051448240876197815, 0.03443191200494766, 0.1000247448682785, 0.08856192976236343, 0.06936292350292206, 0.035565149039030075, 0.07643765211105347, 0.09142524749040604, 0.03241753205657005, 0.051186300814151764, 0.024778304621577263, 0.00921162310987711, 0.06391647458076477, 0.025802383199334145, 0.055531445890665054, 0.030764775350689888, 0.07049863040447235, 0.04501784220337868, -0.01760191284120083, -0.1023944839835167, 0.039757709950208664, -0.05996619164943695], [0.06296959519386292, -0.031742554157972336, -0.0017770944396033883, 0.045840371400117874, -0.03697044029831886, 0.003408630145713687, 0.09662536531686783, -0.06993596255779266, 0.01168740913271904, -0.04948920011520386, 0.13479116559028625, -0.0805894285440445, 0.11515220254659653, -0.026051506400108337, 0.002562823472544551, -0.11335666477680206, -0.10112926363945007, 0.10316424816846848, -0.15493597090244293, 0.015127592720091343, -0.05790381506085396, -0.0661485567688942, 0.08493491262197495, -0.11527428030967712, -0.04899527132511139, -0.07326117902994156, 0.10559503734111786, 0.04572702944278717, 0.08092425018548965, -0.10386563092470169, -0.08146370202302933, -0.009353079833090305, -0.11057280004024506, -0.0002703503123484552, 0.009317178279161453], [-0.03622101992368698, -0.015071261674165726, 0.04792896658182144, 0.03137117251753807, 0.07986807823181152, 0.0846865177154541, 0.06610845774412155, 0.012145642191171646, 0.07983266562223434, 0.021558020263910294, -0.08427012711763382, 0.012010292150080204, 0.05623511224985123, -0.01836974546313286, -0.028332941234111786, -0.010642297565937042, 0.12280158698558807, 0.012496231123805046, -0.01800525188446045, 0.013728639110922813, -0.04179370403289795, -0.03081563487648964, -0.00039534326060675085, -0.01086366456001997, -0.10168596357107162, 0.013267184607684612, -0.07564578205347061, 0.011000312864780426, -0.05202589929103851, -0.05372094362974167, -0.02462317980825901, -0.05059050768613815, -0.0003871808585245162, -0.017482955008745193, -0.06437724083662033], [-0.10402361303567886, -0.04163477197289467, 0.02164493501186371, 0.026499135419726372, -0.0016725646564736962, 0.06747674196958542, -0.04147159308195114, -0.03616596758365631, 0.05646811053156853, -0.027404293417930603, -0.03673303499817848, -0.038427192717790604, -0.006934222765266895, 0.047083426266908646, -0.04382907599210739, 0.08214635401964188, -0.11250405013561249, 0.021443035453557968, 0.06708918511867523, -0.06648183614015579, -0.01456682849675417, -0.006206273101270199, -0.02860439382493496, -0.057625334709882736, -0.08009283989667892, 0.21049544215202332, 0.05406729876995087, 0.10169313848018646, -0.04543577879667282, -0.10621726512908936, 0.14157462120056152, -0.20742453634738922, 0.06322449445724487, 0.33408889174461365, -0.0016927120741456747], [0.018752871081233025, -0.1203511506319046, 0.09415756165981293, -0.05872771516442299, -0.0835312008857727, 0.09989512711763382, 0.032900262624025345, -0.02685808390378952, -0.10608339309692383, 0.06026139110326767, -0.10825127363204956, 0.10030805319547653, 0.032596856355667114, 0.020646316930651665, -0.11589071154594421, 0.03516959398984909, 0.0003298802475910634, 0.10926445573568344, -0.03215637430548668, 0.01996014080941677, -0.07868767529726028, 0.016842002049088478, 0.1158725842833519, -0.041627440601587296, -0.08444160223007202, -0.06702668964862823, -0.10838615149259567, 0.15238158404827118, -0.08234315365552902, 0.014299130998551846, -0.1599774956703186, 0.07908082753419876, 0.05481305345892906, 0.027234509587287903, 0.019610146060585976], [0.04329187050461769, 0.055392809212207794, 0.008299174718558788, -0.017939133569598198, -0.03261147812008858, -0.09315801411867142, -0.03929273411631584, 0.020249217748641968, -0.0474051833152771, 0.04837666451931, 0.011462153866887093, -0.04061116278171539, -0.0014085739385336637, -0.04307980090379715, -0.007994997315108776, -0.028909428045153618, -0.060855139046907425, -0.0458788201212883, 0.10963558405637741, 0.05123968422412872, 0.04490654170513153, 0.014977550134062767, 0.025048166513442993, 0.010581498965620995, -0.06351418048143387, 0.02167956344783306, -0.055757470428943634, 0.053334373980760574, -0.004448314663022757, -0.002979360520839691, -0.05421215295791626, 0.03300099074840546, 0.056807741522789, 0.04965062811970711, -0.0021118219010531902], [-0.04106022045016289, -0.03321502357721329, 0.08496041595935822, -0.05125603452324867, -0.009077247232198715, -0.008394874632358551, -0.05220025032758713, -0.005303685553371906, -0.013083728030323982, 0.017299842089414597, -0.04612521082162857, -0.11624343693256378, -0.04595710709691048, -0.003101951675489545, -0.04614163935184479, 0.12273429334163666, -0.07176902890205383, 0.1039746105670929, 0.05024978145956993, -0.02356758899986744, -0.02173127420246601, -0.03576520085334778, -0.010964315384626389, -0.11829794198274612, 0.08148997277021408, 0.12011650204658508, 0.12816882133483887, 0.005832568742334843, 0.04597104713320732, -0.025529351085424423, 0.018680311739444733, -0.09400715678930283, -0.04182421788573265, 0.03390849009156227, -0.05443364381790161], [0.02573523297905922, -0.04266409948468208, 0.043828267604112625, -0.006840280257165432, -0.01803487539291382, -0.01916251890361309, 0.02541542984545231, -0.018677571788430214, 0.06617094576358795, 0.03670777752995491, 0.005209307186305523, -0.020914100110530853, -0.03566696122288704, 0.08431818336248398, 0.02981414459645748, -0.03668768331408501, 0.08881430327892303, 0.006877881474792957, 0.055289458483457565, -0.049127254635095596, -0.08058574795722961, 0.005972761660814285, -0.023163892328739166, -0.04272766038775444, 0.019711874425411224, 0.06800501048564911, -0.06451854854822159, 0.04158118739724159, 0.02539198100566864, 0.08461736142635345, -0.012891460210084915, -0.04519006237387657, 0.03498156741261482, -0.053930364549160004, -0.02590145543217659], [0.02326035313308239, -0.055120132863521576, -0.04841866344213486, 0.01921170949935913, -0.008844279684126377, -0.022330552339553833, -0.008963298983871937, 0.00837904866784811, -0.040662575513124466, 0.1220884621143341, 0.06802096217870712, 0.07840598374605179, 0.06935052573680878, 0.04859863966703415, 0.061998046934604645, 0.07839282602071762, 0.04512947425246239, -0.05641531944274902, 0.05984250456094742, 0.04319899529218674, 0.05969339609146118, -0.09277427941560745, 0.06377528607845306, -0.020093346014618874, 0.004238881170749664, 0.07151137292385101, -0.010656348429620266, -0.010234372690320015, 0.045140691101551056, 0.014291997067630291, -0.09383273124694824, 0.06138072535395622, 0.03464064002037048, -0.015124295838177204, 0.00263525964692235], [0.10046587884426117, -0.07813367247581482, -0.11762089282274246, 0.14740684628486633, -0.09903685003519058, -0.06131384149193764, 0.12167449295520782, 0.026849688962101936, 0.0513354130089283, -0.014537228271365166, 0.0370052307844162, 0.07476285845041275, 0.14644543826580048, 0.0028624762780964375, 0.02841821312904358, 0.06435591727495193, -0.08154109865427017, -0.1248573437333107, 0.044512879103422165, 0.11440650373697281, -0.02993926964700222, 0.02366691641509533, -0.12383219599723816, 0.012276846915483475, -0.05956996977329254, 0.004483097232878208, -0.04134438931941986, -0.08199304342269897, 0.0624336376786232, 0.1006138026714325, -0.1530933678150177, -0.15330810844898224, -0.055034488439559937, -0.04728549346327782, -0.09740077704191208], [0.009361558593809605, 0.03771235793828964, 0.02700476162135601, -0.0045522041618824005, -0.1294451802968979, 0.14176149666309357, -0.019531376659870148, 0.0172506682574749, -0.10122167319059372, -0.03714146092534065, -0.03572870045900345, -0.06425268203020096, 0.12764476239681244, -0.10017082840204239, 0.08441674709320068, 0.004365383181720972, -0.04823651164770126, 0.15208634734153748, 0.07443547248840332, -0.13387028872966766, 0.1578969806432724, -0.014546232298016548, -0.01660510152578354, -0.006503273732960224, 0.03662180155515671, 0.04894140362739563, 0.054598383605480194, 0.029753297567367554, -0.06866773962974548, 0.07397253811359406, -0.060355644673109055, -0.03587624058127403, -0.048239048570394516, -0.053985897451639175, -0.02527700923383236], [0.056031566113233566, -0.08537595719099045, 0.10565904527902603, 0.07476495951414108, -0.056535206735134125, -0.06987355649471283, -0.17305994033813477, -0.11255095154047012, -0.12927880883216858, -0.040999408811330795, -0.03413292393088341, -0.075603187084198, 0.09685803204774857, 0.11693208664655685, 0.07846854627132416, 0.12656743824481964, 0.019942909479141235, 0.11822353303432465, -0.01132972165942192, -0.03350371867418289, -0.005508081987500191, -0.11375390738248825, 0.12186761200428009, -0.07752169668674469, -0.05143221467733383, -0.015834441408514977, -0.11266465485095978, -0.11995289474725723, 0.05849228799343109, -0.13475903868675232, -0.10225255787372589, 0.09644065797328949, -0.0880812257528305, 0.06283863633871078, 0.09606768190860748], [0.0014011581661179662, 0.08247920870780945, -0.03539910912513733, 0.004234657622873783, -0.01902480237185955, 0.01312295626848936, -0.03586100786924362, -0.05721454322338104, -0.06910664588212967, -0.07808265089988708, -0.06160583719611168, -0.01742331124842167, -0.02961788885295391, 0.03565351292490959, -0.0017469965387135744, 0.03891318663954735, 0.016331760212779045, -0.03524567559361458, 0.011057372204959393, 0.039670076221227646, -0.03474891930818558, 0.04843604192137718, -0.005209838040173054, 0.017246028408408165, 0.03519706055521965, 0.060159776359796524, -0.009325051680207253, -0.012337199412286282, -0.024274226278066635, -0.04439786449074745, -0.06037282198667526, 0.06155148148536682, 0.0034479459282010794, -0.05984056368470192, 0.002158059738576412], [0.06516144424676895, 0.03223920986056328, 0.02633921429514885, 0.008553965948522091, -0.11925210803747177, 0.13873916864395142, 0.05404062941670418, -0.05517132580280304, -0.030265506356954575, 0.027483606711030006, -0.11532121151685715, 0.09963106364011765, -0.010307513177394867, 0.05335470661520958, -0.006425684783607721, -0.08546392619609833, -0.14415034651756287, -0.04745890945196152, 0.026047861203551292, -0.030785884708166122, 0.016375642269849777, 0.06601108610630035, -0.038955241441726685, -0.027313509956002235, 0.05579375475645065, -0.1356409639120102, 0.0031059805769473314, -0.01350977923721075, -0.057549308985471725, 0.05809270963072777, 0.057192735373973846, -0.15742430090904236, 0.052392978221178055, -0.09311190247535706, -0.05945337936282158], [-0.06120448186993599, -0.0791911706328392, 0.05475357919931412, -0.01897348277270794, -0.10617078095674515, 0.035676900297403336, 0.0007870335830375552, 0.010315511375665665, -0.08209002017974854, 0.06486647576093674, -0.01134505681693554, 0.009554537944495678, -0.0907507836818695, 0.06035780906677246, -0.005454451777040958, -0.07535803318023682, 0.12815433740615845, -0.13427473604679108, -0.11818932741880417, -0.06275299191474915, -0.07405435293912888, 0.06902538985013962, 0.09069108217954636, -0.10370416194200516, -0.09918983280658722, 0.013147877529263496, 0.10692479461431503, -0.009343797340989113, -0.13048891723155975, 0.054573338478803635, 0.023389717563986778, 0.11131490767002106, 0.024933699518442154, -0.10681427270174026, 0.0740257278084755], [0.03403019905090332, 0.02845178358256817, 0.09898807108402252, -0.0053922939114272594, 0.053445156663656235, 0.051824990659952164, -0.01790892519056797, -0.028621939942240715, 0.02602633461356163, 0.0591743066906929, -0.02563364803791046, 0.03618369996547699, -0.026109138503670692, 0.06898584961891174, -0.0012569488026201725, 0.0610162653028965, -0.06963763386011124, 0.01617073081433773, -0.029882164672017097, -0.04408620670437813, -0.08142421394586563, -0.0571674108505249, -0.021309349685907364, 0.019330913200974464, 0.03590596839785576, 0.014378923922777176, 0.09551861137151718, -0.0030461905989795923, -0.027152199298143387, 0.06488293409347534, 0.01181954238563776, 0.019208785146474838, -0.06981773674488068, 0.017976710572838783, 0.011010486632585526], [0.040808551013469696, -0.07479012757539749, 0.030096082016825676, -0.008970788680016994, -0.01924825832247734, 0.018253283575177193, 0.014347650110721588, -0.010390575975179672, -0.012949428521096706, 0.004145776852965355, -0.043699026107788086, 0.10383071005344391, 0.03301439434289932, -0.1166384145617485, -0.05651795491576195, 0.06038140878081322, 0.10127486288547516, 0.010565868578851223, 0.02631826139986515, -0.020080100744962692, -0.057400427758693695, -0.039088692516088486, 0.007917824201285839, -0.07536724209785461, -0.09626571089029312, -0.11745063960552216, 0.012729422189295292, 0.05485018342733383, -0.0941726341843605, 0.022610409185290337, -0.10815665125846863, -0.07430029660463333, 0.0010959346545860171, 0.018855158239603043, -0.004943039268255234], [-0.05417511239647865, 0.04262233525514603, 0.023889735341072083, -0.04953095689415932, 0.001521780272014439, 0.04692106321454048, -0.041347749531269073, -0.07385439425706863, -0.028182948008179665, -0.009500985033810139, 0.0008620586595498025, -0.00844610296189785, -0.03527583181858063, 0.05399113893508911, 0.04348955303430557, -0.05122506991028786, -0.0034207887947559357, 0.06947318464517593, 0.08001203089952469, 0.00856811460107565, -0.015222611837089062, 0.003027223516255617, 0.02959408052265644, 0.002476869150996208, 0.0838790014386177, -0.08434682339429855, -0.05570578947663307, -0.025114918127655983, -0.009432947263121605, 0.07710929960012436, -0.061176132410764694, -0.05627913028001785, -0.025049462914466858, -0.08433327823877335, -0.0189308300614357], [0.04515855759382248, 0.0860786885023117, 0.10179241001605988, 0.16166150569915771, 0.0006734804483130574, -0.08078831434249878, 0.0650087296962738, -0.02792816050350666, -0.07094788551330566, -0.05916699767112732, -0.056627675890922546, -0.0739426463842392, -0.09693586081266403, 0.05526398494839668, 0.047984734177589417, 0.023140927776694298, 0.00046915278653614223, -0.07005515694618225, 0.009702484123408794, -0.06636593490839005, -0.04315589740872383, -0.06314264982938766, -0.03478094935417175, 0.06712544709444046, 0.018187297508120537, 0.0009268869180232286, -0.009388711303472519, 0.07138645648956299, 0.09728115797042847, -0.05562438443303108, 0.03366606682538986, -0.021296489983797073, 0.02478906325995922, -0.01555033028125763, -0.10663732886314392], [0.023810362443327904, -0.006965960841625929, 0.025381360203027725, -0.029523732140660286, -0.039964400231838226, 0.08411043137311935, -0.01444437075406313, -0.039033442735672, 0.04000339284539223, 0.03725704923272133, -0.036425504833459854, 0.07688001543283463, 0.008010712452232838, 0.013939389027655125, -0.010937116108834743, 0.026725493371486664, -0.06451515853404999, -0.019559785723686218, -0.04712459072470665, 0.01374874822795391, 0.076822429895401, -0.0012717654462903738, -0.04503576084971428, -0.02447858266532421, -0.06495064496994019, -0.05062204971909523, -0.0557476244866848, 0.011376249603927135, -0.03350362554192543, 0.042923856526613235, -0.016359999775886536, -0.0429011806845665, -0.014369386248290539, -0.03717571869492531, 0.024371881037950516], [-0.018780866637825966, 0.048824191093444824, 0.02468830533325672, -0.003689385252073407, -0.0622415617108345, 0.000145125828566961, -0.01730336621403694, -0.05385707691311836, -0.04764533415436745, 0.0511048398911953, 0.020867208018898964, -0.02089649997651577, -0.016580527648329735, -0.046169303357601166, 0.020297087728977203, 0.07186880707740784, 0.03856642171740532, 0.00926889292895794, 0.06836558133363724, 0.031883131712675095, 0.04361940175294876, -0.030339417979121208, 0.014697343111038208, -0.0037438892759382725, 0.04702834412455559, 0.03375520184636116, 0.05253775790333748, 0.036339256912469864, 0.04698915034532547, 0.0558803491294384, 0.011520348489284515, -0.02761830948293209, 0.05616562440991402, 0.00697544077411294, -0.003804222447797656], [-0.05508222430944443, -0.05873962864279747, 0.04484078288078308, 0.007787440903484821, -0.011897014454007149, -0.008309293538331985, 0.06010351702570915, -0.0003014912363141775, 0.01645941659808159, -0.03485480323433876, -0.03951439633965492, -0.01304615754634142, -0.043276526033878326, 0.02569747529923916, 0.009538909420371056, -0.07521744817495346, -0.07758929580450058, 0.015962116420269012, 0.0588621087372303, -0.03929559886455536, -0.04815870150923729, 0.02934248186647892, -0.06743112206459045, -0.028022242709994316, -0.0710684284567833, 0.05831754952669144, 0.053005725145339966, 0.025390852242708206, 0.05350238084793091, -0.0690770074725151, 0.02652807906270027, -0.06657431274652481, 0.06675275415182114, 0.20601968467235565, 0.029378419741988182], [-0.03539436310529709, 0.04873260110616684, -0.05119309946894646, -0.04995618388056755, -0.04523420333862305, -0.00850913766771555, 0.010567538440227509, -0.07518935948610306, -0.011980623938143253, -0.11012579500675201, 0.057952817529439926, 0.016635118052363396, 0.06117551773786545, -0.031546685844659805, -0.04676791653037071, -0.023669516667723656, 0.04204615205526352, 0.049796752631664276, 0.05275679752230644, -0.017121566459536552, -0.013095266185700893, 0.02157560922205448, 0.05835855379700661, -0.034952618181705475, -0.008886091411113739, -0.048252761363983154, -0.0005651870742440224, -0.01893855631351471, 0.046015601605176926, 0.022828713059425354, -0.03911209478974342, 0.05832718312740326, -0.003267185064032674, -0.0746062844991684, -0.006518338341265917], [-0.0521649494767189, -0.08496951311826706, -0.013162032701075077, -0.08014879375696182, -0.03701884299516678, -0.07596277445554733, -0.044935520738363266, 0.014051498845219612, -0.04081524908542633, 0.0002121004945365712, -0.013797390274703503, 0.06256621330976486, 0.08621036261320114, 0.027345625683665276, -0.08330849558115005, -0.01753663457930088, -0.1054537445306778, 0.04203628748655319, 0.0782216265797615, -0.018706321716308594, -0.03635900840163231, -0.020952412858605385, -0.0050736283883452415, 0.03353840857744217, 0.007758292835205793, -0.05198242887854576, 0.011643430218100548, -0.0060133785009384155, 0.09601010382175446, -0.018934249877929688, -0.008314218372106552, -0.04863901436328888, -0.07059602439403534, -0.10155188292264938, 0.02242400497198105], [0.01886797323822975, -0.0011990634957328439, 0.006224183365702629, 0.055318113416433334, -0.04600674286484718, 0.13473065197467804, -0.014247430488467216, -0.004932984709739685, -0.012899595312774181, -0.06259952485561371, -7.745002949377522e-05, 0.03152081370353699, 0.0084845507517457, 0.09604516625404358, -0.059903584420681, 0.0064590186811983585, 0.09095137566328049, 0.06870279461145401, -0.08593728393316269, -0.07423434406518936, 0.10379524528980255, 0.018864275887608528, 0.05154566839337349, 0.09647031873464584, -0.08405162394046783, 0.047540146857500076, 0.03935175761580467, 0.038978684693574905, -0.04458699747920036, 0.025123221799731255, 0.10582553595304489, 0.007905921898782253, 0.057198163121938705, 0.23338311910629272, 0.04400969296693802], [0.05324157327413559, -0.0923556312918663, 0.01522793434560299, -0.09504193067550659, -0.11682695895433426, -0.011033683083951473, -0.013180325739085674, 0.04997224360704422, -0.021579835563898087, 0.05439773574471474, -0.04735461249947548, -0.0639912486076355, 0.02340174838900566, 0.0035089110024273396, 0.08307013660669327, 0.08591586351394653, 0.026151293888688087, -0.18868789076805115, 0.14238834381103516, 0.024535521864891052, 0.14662960171699524, 0.06287675350904465, -0.07783740013837814, -0.12165341526269913, -0.11671122908592224, 0.033922046422958374, -0.07873066514730453, -0.033926378935575485, 0.05933345481753349, -0.01388577651232481, 0.03252013772726059, 0.05082770437002182, 0.06862363964319229, 0.07385297119617462, -0.08852171897888184], [0.033383071422576904, 0.00047577539226040244, 0.0009091211250051856, -0.1372196078300476, -0.03219589963555336, 0.012932841666042805, -0.004183655139058828, 0.03654038906097412, 0.0037767954636365175, 0.09059786051511765, -0.08404813706874847, 0.10177619010210037, -0.07474949210882187, -0.044416435062885284, 0.007424863055348396, -0.05695605278015137, 0.0479014627635479, 0.01865260675549507, -0.001582276076078415, 0.028948312625288963, -0.0069329929538071156, 0.10232517868280411, -0.011507286690175533, 0.0002041876141447574, 0.03365139290690422, 0.07135166972875595, -0.03364121913909912, 0.021259907633066177, -0.08028075098991394, 0.024688461795449257, 0.052403587847948074, -0.12486769258975983, 0.044703975319862366, -0.07181969285011292, 0.011199450120329857], [0.009778660722076893, -0.028510062023997307, 0.015022747218608856, 0.12095163762569427, 0.0986819714307785, 0.045027244836091995, 0.024133071303367615, 0.04484224691987038, -0.05534706637263298, 0.0339970700442791, -0.00894321408122778, 0.05905524641275406, 0.044595878571271896, -0.007894235663115978, -0.006752493791282177, -0.000233244922128506, -0.04613656923174858, -0.05318804830312729, 0.02023668959736824, -0.04564695432782173, -0.12322182208299637, 0.013970771804451942, 0.07141034305095673, 0.06749267131090164, 0.03387390077114105, 0.04980335012078285, 0.08000577986240387, -0.018408315256237984, 0.05117209255695343, 0.0755697637796402, 0.026594195514917374, -0.00666683679446578, -0.01859385147690773, -0.03277410939335823, 0.04036672040820122]], "b1": [-0.02930418588221073, 0.023748597130179405, 0.08551575988531113, -0.034895602613687515, -0.037920769304037094, 0.0661354809999466, -0.07602635025978088, 0.06932823359966278, 0.09913096576929092, 0.017955033108592033, -0.08873400837182999, 0.11508360505104065, -0.08919811248779297, -0.05947409197688103, 0.00394953228533268, -0.02579801343381405, 0.05168094485998154, -0.08850494027137756, 0.06203985586762428, 0.004636397585272789, 0.05282069370150566, -0.03532513231039047, -0.14226524531841278, -0.10110004991292953, 0.004118208773434162, 0.04546399787068367, -0.05602111294865608, 0.09492221474647522, -0.03114047832787037, 0.03630794584751129, -0.04754706472158432, 0.017732275649905205, -0.025497401133179665, 0.061672795563936234, 0.09539096057415009, -0.030945373699069023, -0.07489992678165436, -0.12970972061157227, -0.0008941924315877259, 0.049219079315662384, 0.0814947560429573, 0.02280915528535843, 0.013293019495904446, 0.061545662581920624, 0.04026682302355766, -0.08024165034294128, -0.01087771076709032, -0.061023857444524765, -0.08253511041402817, -0.0024846037849783897, -0.18739154934883118, -0.00791558064520359, -0.23566760122776031, -0.04973025992512703, 0.014781972393393517, -0.12001856416463852, 0.060640085488557816, -0.17510253190994263, 0.004569005221128464, 0.07236494868993759, -0.07369925826787949, -0.04972059279680252, 0.016246160492300987, 0.011073912493884563, -0.04256850481033325, 0.10314233601093292, 0.13522885739803314, -0.0005719055770896375, 0.08971033245325089, 0.018488585948944092, 0.024556878954172134, -0.2123211920261383, -0.050629813224077225, 0.05183880031108856, -0.03590446710586548, -0.04106355831027031, 0.075980044901371, -0.12796716392040253, 0.003336940659210086, -0.062498465180397034, -0.03788245841860771, 0.001686955220066011, -0.04690800979733467, 0.07404366135597229, -0.06975334882736206, 0.000947243592236191, 0.06146273761987686, 0.038345739245414734, 0.05677265673875809, -0.10087046027183533, -0.0015715210465714335, -0.03142901882529259, -0.08410421013832092, -0.0540994331240654, 0.05256868153810501, 0.010274532251060009], "W2": [[0.0019458397291600704, 0.033785589039325714, -0.03538772463798523, 0.014060026034712791, -0.04659648239612579, -0.018690556287765503, 0.006466602440923452, 0.06904689222574234, -0.018434632569551468, -0.01587364636361599, 0.05250991880893707, 0.027920827269554138, 0.001338467001914978, 0.01675659418106079, 0.0038927982095628977, 0.05413106083869934, 0.033649273216724396, 0.028057323768734932, 0.000948218279518187, 0.02009451761841774, 0.020161695778369904, 0.00038579775718972087, 0.1496397852897644, 0.033706557005643845, 0.038127776235342026, 0.03718171268701553, 0.004517252091318369, 0.0653594583272934, -0.005426815245300531, 0.0505032055079937, 0.039820656180381775, 0.009632243774831295, 0.03052261285483837, 0.049798235297203064, 0.06650185585021973, 0.04611674323678017, 0.04815692454576492, 0.049614254385232925, -0.021933643147349358, 0.008440474979579449, 0.04165661707520485, -0.06672632694244385, 0.038621336221694946, 0.01228599064052105, -0.031862106174230576, 0.01867368258535862, 0.04439381882548332, -0.04264345392584801, 0.03320159763097763, 0.019308291375637054, -0.02414180524647236, 0.03963590785861015, 0.03400539606809616, 0.032959796488285065, -0.00857582688331604, 0.06456403434276581, 0.019350312650203705, 0.0021117881406098604, 0.07671504467725754, 0.011020912788808346, -0.03980100899934769, 0.05862923711538315, 0.005079885479062796, 0.042741257697343826, -0.0061563532799482346, -0.02220805361866951, -0.03438648581504822, 0.05000046268105507, -0.009359851479530334, -0.0034681325778365135, -0.028829939663410187, 0.0246102474629879, 0.06198643147945404, 0.011117320507764816, 0.007832024246454239, -0.0047469972632825375, -0.03224698826670647, 0.05523550882935524, 0.015514597296714783, 0.05647493153810501, -0.0011250120587646961, 0.0414111353456974, 0.036814481019973755, -0.017308665439486504, 0.04244821518659592, -0.016125250607728958, 0.023535678163170815, 0.0656283050775528, -0.010731995105743408, 0.018820160999894142, 0.032265398651361465, -0.015472102910280228, 0.027958419173955917, 0.08140542358160019, 0.0078039527870714664, 0.003176360158249736], [0.017483089119195938, 0.042796313762664795, 0.01804650016129017, 0.05831339955329895, 0.04273912310600281, -0.011567719280719757, -0.03577208146452904, -0.0017359618796035647, 0.085459865629673, 0.07535993307828903, -0.021186070516705513, -0.011898485012352467, -0.015078512951731682, -0.006503223441541195, 0.015525479800999165, -0.038183413445949554, 0.04016297683119774, 0.010418597608804703, -0.02232549712061882, -0.041387032717466354, 0.01573103480041027, -0.006068314891308546, -0.053983043879270554, 0.0019964391831308603, 0.02648121304810047, -0.06693284213542938, 0.004326009191572666, 0.049116525799036026, 0.0055886185728013515, -0.03436870872974396, 0.017201315611600876, 0.025346361100673676, 0.008899197913706303, -0.015394010581076145, -0.006222917698323727, -0.0580940917134285, -0.03049599565565586, -0.07663408666849136, -0.04175784811377525, -0.05599476024508476, -0.0028399783186614513, 0.01169663667678833, -0.0016630042809993029, 0.057035576552152634, -0.03576602041721344, -0.0032680181320756674, 0.03473648801445961, 0.005004297010600567, 0.048928096890449524, -0.035094987601041794, -0.04515741765499115, 0.026762500405311584, -0.002795523963868618, 0.009002706967294216, -0.02311031147837639, -0.015616555698215961, -0.06967395544052124, 0.05817759782075882, 0.00011220994201721624, -0.04608749598264694, 0.015098907984793186, -0.022606823593378067, 0.038931943476200104, -0.02125549130141735, -0.03311191126704216, 0.008083810098469257, 0.02629193291068077, -0.007081827614456415, 0.0178520530462265, -0.02101648971438408, -0.03084627538919449, 0.004325301852077246, 0.011745482683181763, 0.014067724347114563, 0.011431843042373657, -0.005481352563947439, 0.030110018327832222, 0.06204814836382866, 0.015545167028903961, -0.0777396634221077, -0.019483761861920357, 0.020257823169231415, 0.013536731712520123, -0.037368085235357285, -0.007095685228705406, 0.013947130180895329, 0.03183741122484207, 0.07463144510984421, 0.017372285947203636, -0.04006068781018257, -0.004736466333270073, 0.011455151252448559, -0.05519863963127136, 0.0016711781499907374, -0.021989142522215843, 0.033723633736371994], [0.05780390650033951, 0.0031199378427118063, 0.0642339214682579, 0.08645617216825485, -0.009557247161865234, 0.0020847574342042208, -0.076059490442276, -0.047367170453071594, -0.04811514541506767, 0.020894456654787064, 0.0418490432202816, 0.003873937763273716, 0.10838701575994492, 0.015525558963418007, 0.06917912513017654, -0.030590970069169998, -0.12970155477523804, -0.029074177145957947, -0.03556320071220398, -0.019624877721071243, -0.028887582942843437, 0.008983123116195202, -0.09330098330974579, -0.01882437989115715, 0.10279340296983719, 0.05101212114095688, 0.060833096504211426, -0.022847773507237434, 0.09867078810930252, 0.05673714354634285, -0.04965432733297348, 0.035064443945884705, -0.0423981249332428, 0.010378917679190636, 0.000516614003572613, 0.019072890281677246, -0.009449200704693794, -0.07375013083219528, -0.03686242923140526, 0.09445176273584366, -0.12555691599845886, 0.049671269953250885, -0.03971393033862114, -0.028223715722560883, -0.007254940923303366, 0.13061368465423584, 0.07486893236637115, 0.012787566520273685, -0.027690373361110687, -0.016909020021557808, 0.1447645127773285, -0.10167036950588226, 0.15363477170467377, -0.05833280459046364, 0.04809047281742096, 0.011699113063514233, -0.10593307763338089, 0.03278227150440216, -0.07276884466409683, 0.031059755012392998, -0.046023979783058167, -0.047431837767362595, -0.01670556142926216, 0.017475979402661324, 0.013642598874866962, 0.07464458793401718, 0.024008195847272873, -0.05229572206735611, 0.023743804544210434, -0.0013100218493491411, 0.05561716854572296, -0.009789182804524899, -0.021696090698242188, -0.0533614456653595, -0.027309687808156013, 0.04329254850745201, -0.10534051805734634, 0.02068360522389412, -0.066462442278862, -0.03829050436615944, 0.051902249455451965, 0.09039697796106339, 0.027667466551065445, -0.05259823054075241, 0.11560355871915817, 0.08942459523677826, -0.07460501790046692, -0.03911098092794418, 0.015131999738514423, 0.044493239372968674, 0.0513140968978405, 0.038269296288490295, -0.009618847630918026, 0.006455403286963701, -0.005889322608709335, -0.021265706047415733], [0.017948973923921585, -0.04606446251273155, -0.010629737749695778, -0.10778257250785828, 0.012393881566822529, 0.016382301226258278, 0.01195987593382597, -0.06884314119815826, 0.031914252787828445, 0.08860135823488235, -0.07421796768903732, -0.04397149756550789, 0.010516162030398846, -0.10694731026887894, 0.008521401323378086, -0.01870659366250038, 0.0059998915530741215, -0.07727387547492981, 0.02879120595753193, -0.017616894096136093, 0.030578259378671646, -0.09859756380319595, 0.07265986502170563, -0.07561472058296204, -0.06664443761110306, -0.018502801656723022, 0.02583909034729004, 0.006639939267188311, -0.06733706593513489, 0.07953354716300964, 0.0022745144087821245, 0.06217333674430847, 0.013247496448457241, -0.004118707031011581, 0.020354565232992172, -0.03885683789849281, -0.0370703749358654, 0.11438479274511337, 0.01959601789712906, -0.01722649298608303, 0.017554529011249542, -0.01710987463593483, 0.05591372027993202, -0.06031043455004692, 0.04442792385816574, -0.053182125091552734, 0.0058825332671403885, -0.04231530427932739, -0.006070626433938742, 0.04930341616272926, -0.12691465020179749, 0.020173221826553345, -0.07999575883150101, 0.06353633105754852, -0.05296787619590759, 0.06124862655997276, -0.02236182987689972, -0.044992562383413315, 0.030191613361239433, -0.006407424341887236, -0.04794151335954666, 0.03764336556196213, -0.04864742234349251, 0.02704956941306591, 0.02727569080889225, 0.046486739069223404, -0.05371410772204399, 0.019246617332100868, 0.052724260836839676, 0.03256411477923393, -0.028858426958322525, -0.15414969623088837, -0.0016770117217674851, 0.019738400354981422, -0.0122472383081913, 0.007990650832653046, 0.0747765377163887, 0.015697438269853592, -0.011464424431324005, 0.032300855964422226, -0.05529158189892769, 0.03667003661394119, 0.01737041398882866, 0.047850411385297775, 0.004216789733618498, 0.06039954721927643, -0.004843462724238634, 0.07446914166212082, -0.022659311071038246, 0.004026575013995171, -0.044187918305397034, -0.0377887599170208, -0.01121213473379612, -0.043089546263217926, -0.05842329189181328, 0.0027106215711683035], [0.006145968101918697, -0.011900123208761215, -0.014667598530650139, 0.07155655324459076, 0.12190879881381989, -0.006024462170898914, -0.054214753210544586, 0.017150448635220528, -0.02948525920510292, -0.07513856142759323, 0.05309726670384407, -0.0006970349350012839, 0.08540111035108566, 0.04727704077959061, 0.06641468405723572, -0.0429738350212574, 0.0074160052463412285, -0.016627414152026176, -0.029489820823073387, 0.04736706241965294, -0.08612353354692459, 0.04515570402145386, -0.14415548741817474, 0.11624593287706375, -0.0777372494339943, 0.038373254239559174, -0.0032697059214115143, 0.053436119109392166, -0.045755255967378616, -0.007836900651454926, -0.012953611090779305, 0.017662592232227325, 0.009012808091938496, 0.04374010115861893, -0.053817350417375565, 0.01366640254855156, 0.009928695857524872, -0.05150661617517471, 0.017537495121359825, -0.03584566339850426, 0.020682435482740402, -0.006563776638358831, 0.02256867103278637, 0.020769914612174034, 0.055499088019132614, 0.08202806115150452, -0.08178684115409851, 0.05167630687355995, 0.034270841628313065, 0.05844118446111679, 0.04247402027249336, -0.07604309171438217, 0.09374894201755524, 0.026530951261520386, 0.010222300887107849, -0.10749931633472443, -0.05872367322444916, -0.017273258417844772, -0.006933816708624363, -0.02247593365609646, -0.015529995784163475, 0.0736050009727478, -0.008633432909846306, -0.06720160692930222, 0.039226796478033066, -0.07390367239713669, 0.023458115756511688, -0.014981853775680065, 0.059708599001169205, -0.03418918699026108, 0.011209959164261818, 0.10477709025144577, -0.025435835123062134, 0.06239877641201019, 0.004876716528087854, 0.015792302787303925, -0.027661697939038277, -0.052689988166093826, 0.03169473260641098, 0.008650342933833599, 0.051998089998960495, -0.014647683128714561, -0.07269687950611115, 0.0029368503019213676, 0.018082698807120323, 0.008542783558368683, -0.072161465883255, -0.02188171073794365, -0.051029399037361145, -0.013871670700609684, -0.01555484626442194, -0.0539931021630764, 0.0889464020729065, 0.06569503992795944, 0.044458258897066116, -0.060127027332782745], [0.006183282472193241, 0.02240641787648201, -0.010923362337052822, 0.01716826856136322, -0.01995229721069336, 0.011088701896369457, 0.012610037811100483, 0.01095457375049591, 0.02508145570755005, 0.019341718405485153, 0.023468470200896263, 0.017217416316270828, -0.011746607720851898, -0.005965063814073801, -0.007948661223053932, 0.010929557494819164, 0.005967761389911175, 0.013061621226370335, 0.004331858828663826, -0.004369127098470926, 0.02627667598426342, -0.0025145274121314287, 0.027238670736551285, -0.021684395149350166, 0.008451075293123722, 0.02420641854405403, -0.0038387631066143513, 0.03493855521082878, -0.01145477220416069, 0.004079432226717472, 0.009519454091787338, -0.0006764114950783551, 0.01754583977162838, -0.0016788531793281436, 0.009114784188568592, 0.00808433722704649, 0.00998777151107788, 0.011652314104139805, -0.005757534876465797, -0.003238781588152051, 0.008021103218197823, -0.018653204664587975, 0.0017615902470424771, 0.0056062908843159676, -0.007555524352937937, -0.002705784747377038, 0.008208563551306725, -0.004779197741299868, 0.00545884994789958, 0.0006287560681812465, -0.005185059271752834, 0.002469964325428009, 0.008500659838318825, 0.0014909785240888596, -0.002445683116093278, 0.014979162253439426, -0.018485087901353836, 0.02140512503683567, 0.012448308989405632, 0.0013572119642049074, 0.009483689442276955, 0.02718248777091503, 0.009810793213546276, 0.015802782028913498, -0.009322144091129303, -0.006997628137469292, -0.007555895484983921, 0.017262091860175133, 0.00434722239151597, -0.011760164052248001, -0.00149849324952811, 0.034826986491680145, 0.010179927572607994, 0.001212187111377716, -0.01088693831115961, -0.004914761055260897, 0.017195643857121468, 0.029885536059737206, -0.01752062886953354, -0.01971936784684658, -0.006660899613052607, 0.002252854174003005, 0.009633845649659634, -0.004818526562303305, 0.007837850600481033, -0.012018779292702675, 0.0039055338129401207, -0.002669398207217455, -0.004823137540370226, -0.0010601581307128072, 0.00419295160099864, -0.002125402679666877, -0.006834154482930899, 0.011644850485026836, 0.0029366493690758944, 0.007885483093559742], [-0.08064795285463333, 0.06455163657665253, 0.00813736580312252, -0.06753011792898178, -0.1299736350774765, 0.07174847275018692, -0.07220032811164856, -0.034978002309799194, -0.02678949199616909, 0.08171539753675461, -0.08354970067739487, -0.08493615686893463, 0.038810886442661285, -0.06318340450525284, 0.00687407748773694, 0.035831697285175323, 0.060915507376194, 0.033752813935279846, 8.117199467960745e-05, 0.06630709022283554, 0.03928191587328911, -0.08544419705867767, 0.03895234316587448, -0.025806974619627, -0.01254273671656847, -0.029934659600257874, 0.013833357021212578, 0.010701343417167664, -0.06169430539011955, -0.0015819041291251779, 0.035203102976083755, -0.05617700144648552, 0.04207592457532883, 0.05579991266131401, 0.03384459391236305, 0.014054195955395699, 0.056917499750852585, 0.007712234742939472, 0.03644401580095291, 0.059027738869190216, 0.09306556731462479, -0.0125148119404912, -0.0531817302107811, -0.05657048150897026, 0.06742476671934128, -0.12739144265651703, -0.060099031776189804, 0.013397231698036194, -0.035671450197696686, 0.02983510121703148, -0.04135030508041382, 0.07595662772655487, -0.11175283044576645, -0.06728466600179672, -0.07512771338224411, -0.06006504222750664, 0.01687603071331978, -0.0576447956264019, 0.05553531274199486, -0.058255650103092194, -0.04339953884482384, -0.017005832865834236, -0.019755156710743904, 0.00044745244667865336, -0.0381263867020607, 0.06588490307331085, -0.020548639819025993, -0.032228756695985794, 0.05931330472230911, 0.09037890285253525, 0.05893878638744354, -0.08479413390159607, -0.060371801257133484, 0.07424701750278473, -0.021869070827960968, -0.066020667552948, 0.027159862220287323, -0.05512689799070358, 0.05181504786014557, 0.027803845703601837, 0.033836424350738525, 0.03173962980508804, -0.06540624797344208, 0.04430631548166275, -0.02400398999452591, 0.0022992445155978203, -0.09303243458271027, 0.01146569661796093, 0.07548651844263077, -0.06924652308225632, -0.04520503431558609, -0.006931984797120094, -0.03731143847107887, -0.029113752767443657, -0.11052177846431732, -0.013496622443199158], [0.032681435346603394, -0.07312429696321487, -0.0579487644135952, 0.0787648931145668, -0.09349073469638824, -0.023656761273741722, 0.028248107060790062, -0.0630253329873085, -0.041365958750247955, -0.06055968627333641, -0.024460753425955772, 0.014789532870054245, 0.08242999017238617, -0.07716602087020874, -0.03873547911643982, 0.03581564500927925, -0.07294487953186035, 0.11080355942249298, 0.02418910339474678, 0.04670673608779907, -0.007990664802491665, 0.025422047823667526, 0.11526789516210556, -0.013446646742522717, -0.01478126272559166, 0.04568503424525261, 0.018454572185873985, 0.019716540351510048, -0.038734108209609985, -0.008751829154789448, -0.037073902785778046, 0.01734178699553013, 0.019686980172991753, -0.013318517245352268, 0.05815184488892555, 0.058644428849220276, 0.05557640269398689, 0.15072935819625854, 0.08212333172559738, -0.03245437517762184, 0.025791490450501442, -0.04481988772749901, -0.056474972516298294, -0.027951641008257866, 0.0064420853741467, -0.005890663713216782, 0.021417705342173576, -0.07029738277196884, -0.04489915817975998, 0.004093504976481199, 0.0002753974695224315, 0.10482044517993927, 0.019788922742009163, 0.035157617181539536, 0.021630629897117615, 0.0769760012626648, 0.07510886341333389, 0.07120423763990402, 0.012192818336188793, 0.01987415738403797, -0.04329314082860947, -0.0006530364626087248, 0.014393060468137264, 0.0021256960462778807, -0.0016485846135765314, -0.07147567719221115, -0.049520574510097504, 0.0010477588512003422, 0.012956789694726467, -0.09774618595838547, -0.028901658952236176, 0.05129199102520943, -0.05276218429207802, -0.015398682095110416, 0.03126630187034607, -0.0539679229259491, 0.01349970418959856, 0.027790028601884842, -0.06336325407028198, 0.06556907296180725, -0.028652897104620934, 0.04995838552713394, 0.01436526793986559, -0.04221644997596741, -0.054883621633052826, -0.03830878436565399, 0.03956225514411926, -0.011300740763545036, -0.02708709053695202, 0.043321505188941956, -0.05347278341650963, -0.045036349445581436, 0.010294416919350624, 0.07532647997140884, -0.013340541161596775, 0.000983153935521841], [0.032108914107084274, -0.08373444527387619, -0.03846214711666107, 0.0486099012196064, 0.04606365039944649, 0.02360459603369236, -0.07265397161245346, -0.002253831597045064, -0.058470219373703, -0.07399854809045792, 0.043338995426893234, -0.09486262500286102, 0.10053764283657074, 0.06965339928865433, 0.05773630365729332, -0.06493397057056427, 0.0021276366896927357, -0.06975080817937851, -0.03441917151212692, -0.014294332824647427, -0.045560140162706375, -0.0026642256416380405, -0.11345097422599792, 0.10520535707473755, 0.07752387225627899, -0.04580995813012123, 0.05421266704797745, -0.022357754409313202, -0.02683798037469387, -0.054590240120887756, -0.022137949243187904, -0.03324948623776436, 0.07012059539556503, 0.004563485272228718, -0.033211130648851395, 0.055393703281879425, -0.0028653531335294247, -0.07266278564929962, 0.06552144885063171, 0.11194086074829102, 0.028531743213534355, -0.023433521389961243, 0.018142057582736015, -0.0330798365175724, 0.002946325112134218, 0.09114013612270355, -0.04151832312345505, 0.010687826201319695, -0.037407685071229935, 0.00509635079652071, 0.09960243850946426, -0.06350014358758926, 0.031178275123238564, -0.040961187332868576, 0.010054626502096653, 0.012717590667307377, -0.0778658539056778, 0.0814586877822876, 0.040650345385074615, -0.007945326156914234, 0.09836677461862564, -0.08884651213884354, 0.015252334997057915, 0.016367027536034584, 0.0027823159471154213, -0.02505146898329258, 0.0729520320892334, -0.07231545448303223, -0.05279741436243057, 0.07703454792499542, 0.01161042507737875, 0.017928479239344597, 0.04764755070209503, -0.032009247690439224, 0.07870238274335861, 0.027873609215021133, -0.05594763159751892, -0.009643678553402424, 0.07787790149450302, 0.02457403391599655, -0.01554157119244337, -0.05677540972828865, -0.0828992947936058, 0.008825288154184818, 0.023364659398794174, 0.02835281752049923, -0.04530477896332741, -0.02369413524866104, -0.027872402220964432, 0.023747539147734642, -0.02986866421997547, -0.049410440027713776, 0.10389985889196396, -0.05181115120649338, 0.049767546355724335, -0.0007267436012625694], [0.06904744356870651, 0.08722591400146484, 0.056039974093437195, 0.09162422269582748, -0.11358079314231873, 0.06013728305697441, 0.0426766574382782, 0.09493941813707352, 0.06642064452171326, -0.003624193836003542, 0.035517510026693344, 0.030060773715376854, 0.017936183139681816, -0.01939193718135357, 0.036265771836042404, 0.04523417726159096, 0.11552519351243973, 0.035670313984155655, 0.005209242459386587, -0.022325821220874786, 0.11496193706989288, 0.00529790623113513, 0.2117343544960022, -0.021213959902524948, -0.037772271782159805, 0.1178392618894577, -0.04720363765954971, 0.07259641587734222, -0.07848694920539856, 0.05319863557815552, 0.01693119667470455, -0.039998382329940796, 0.10573689639568329, -0.02946517989039421, 0.018065879121422768, 0.05716848745942116, 0.0013999537331983447, 0.057825181633234024, 0.04773533344268799, -0.03863111510872841, 0.003398375352844596, -0.13299046456813812, 0.05152048170566559, -0.08761679381132126, -0.0028276143129915, 0.0474480465054512, -0.013444582000374794, -0.018373386934399605, 0.010254927910864353, -0.012815620750188828, 0.018319102004170418, 0.04448309913277626, -0.0020154505036771297, -0.007111250888556242, -0.027783740311861038, 0.0844867080450058, -0.06749129295349121, 0.05304892733693123, 0.025665296241641045, 0.059902023524045944, 0.08889742940664291, -0.058175232261419296, 0.07808053493499756, 0.0780603438615799, -0.029874827712774277, -0.014681601896882057, -0.11214500665664673, 0.03946966305375099, 0.02814892679452896, -0.07921239733695984, -0.05977174639701843, 0.058069776743650436, 0.008907649666070938, 0.014124205335974693, -0.017726611346006393, -0.05483734980225563, -0.054382044821977615, 0.09180623292922974, -0.006215843837708235, 0.007068910636007786, 0.004069230053573847, 0.06482071429491043, 0.018675243481993675, 0.009894592687487602, 0.06012949347496033, -0.02651655487716198, 0.04446783661842346, 0.08525846898555756, -0.05790101736783981, -0.042097050696611404, 0.00772522808983922, 0.09840068221092224, -0.014738951809704304, 0.01591840386390686, 0.015749290585517883, 0.019387677311897278], [0.007899963296949863, -0.028070729225873947, -0.02844906412065029, 0.005987071897834539, 0.03544469177722931, -0.01550997793674469, 0.014916564337909222, -0.029943708330392838, -0.032036151736974716, -0.08292019367218018, -0.03218754753470421, -0.025531578809022903, -0.02516980655491352, 0.020241325721144676, 0.004761572927236557, -0.04760988429188728, 0.03539889305830002, -0.007570105604827404, -0.03640710189938545, -0.001376040861941874, 0.029103847220540047, 0.04333501309156418, -0.14177820086479187, 0.002104039303958416, -0.041133467108011246, -0.09880798310041428, -0.020908072590827942, -0.012095517478883266, 0.03342750668525696, -0.05962319299578667, -0.021045666188001633, 0.012504436075687408, -0.08413117378950119, -0.030997389927506447, -0.021469000726938248, -0.0450182780623436, -0.052631210535764694, -0.1680563986301422, -0.03928522765636444, -0.00564262829720974, -0.02484969235956669, 0.01231425441801548, -0.033433955162763596, 0.026236290112137794, -0.029611196368932724, 0.003688445780426264, -0.04002551734447479, 0.00527211045846343, -0.020403781905770302, -0.02049470879137516, 0.04986296594142914, 0.025036077946424484, -0.009061743505299091, 0.0314471572637558, -0.024600183591246605, -0.06291716545820236, -0.030567219480872154, 0.08706813305616379, -0.08680681139230728, -0.025073010474443436, 0.03300165385007858, 0.020177818834781647, 0.033594608306884766, -0.06353947520256042, -0.023170212283730507, 0.00956015381962061, -0.033999111503362656, 0.014894811436533928, 0.021863380447030067, -0.007939107716083527, 0.006206596735864878, -0.041555531322956085, -0.04147331416606903, -0.04150921106338501, -0.012147966772317886, 0.006742330268025398, -0.017373468726873398, -0.02798847295343876, -0.08829253911972046, -0.02622770518064499, 0.030573228374123573, -0.05888844653964043, 0.01839374378323555, 0.04171491041779518, -0.012531064450740814, -0.03348061814904213, 0.03487468883395195, -0.03569408133625984, -0.00402042968198657, -0.06766707450151443, -0.023404214531183243, -0.022166308015584946, -0.05806560441851616, -0.06620519608259201, 0.025358933955430984, 0.053115639835596085], [-0.06128790229558945, 0.0372079461812973, 0.048016395419836044, 0.04705392196774483, 0.06707026064395905, -0.021984757855534554, 0.03979441151022911, 0.03718262165784836, 0.02060008980333805, 0.056239206343889236, -0.02646970935165882, 0.0032338493037968874, 0.07837577164173126, 0.07947190850973129, 0.015744827687740326, 0.018994683399796486, -0.06524410098791122, -0.006789754144847393, -0.03252430260181427, -0.039779674261808395, 0.03526316210627556, -0.010957194492220879, -0.10528784245252609, 0.027960140258073807, -0.016490574926137924, -0.03763366490602493, 0.016582045704126358, -0.01410810649394989, 0.004253476858139038, -0.05013512074947357, -0.06269171833992004, 0.003782304236665368, 0.009033825248479843, -0.052056826651096344, 0.04419708624482155, 0.013075556606054306, -0.05106157436966896, 0.012201003730297089, 0.032761380076408386, -0.014899819158017635, -0.040587253868579865, 0.03866088017821312, 0.056967705488204956, -0.016732150688767433, 0.012799860909581184, 0.10316066443920135, -0.014258106239140034, 0.05945981666445732, -0.014239714480936527, -0.026653975248336792, 0.12444265186786652, -0.013419085182249546, 0.10380503535270691, 0.013405132107436657, 0.023923562839627266, -0.10147174447774887, 0.041940148919820786, 0.0805419310927391, 0.003976165782660246, 0.042330048978328705, -0.053299855440855026, -0.05105743557214737, -0.030949609354138374, 0.0010916281025856733, 0.003093732288107276, -0.030858593061566353, 0.027505237609148026, -0.04389618709683418, 0.030426135286688805, 0.04557220637798309, 0.0253804512321949, 0.040948912501335144, 0.016513830050826073, -0.04407021403312683, 0.03558771312236786, -0.020252760499715805, -0.028895646333694458, -0.04253021255135536, -0.029181649908423424, -0.02795678749680519, 0.018077708780765533, -0.048824939876794815, -0.04488265886902809, -0.03174275532364845, -0.005024712532758713, 0.04950839653611183, -0.03336264565587044, 0.004566670395433903, -0.0051918597891926765, 0.0849594846367836, 0.014586302451789379, 0.05742198973894119, 0.011154952459037304, -0.013935071416199207, 0.03687315806746483, -0.02866591513156891], [0.006076658144593239, 0.027092715725302696, -0.045225150883197784, -0.07608282566070557, -0.06287011504173279, -0.006416596472263336, -0.0008800390642136335, 0.043724220246076584, 0.024323157966136932, 0.005544540472328663, 0.0022933348082005978, -0.046657025814056396, 0.01809118315577507, -0.07312710583209991, -0.04023415222764015, 0.025697549805045128, -0.041922278702259064, -0.03994440659880638, 0.05854373797774315, 0.029232969507575035, -0.0174700990319252, -0.018454335629940033, 0.03600640967488289, -0.07229181379079819, 0.014549771323800087, -0.020164303481578827, 0.012795004062354565, 0.04364825040102005, -0.05357447639107704, 0.05914529412984848, -0.02487138845026493, 0.01405631098896265, -0.0510750450193882, -0.0007753246463835239, -0.034376174211502075, -0.018514562398195267, 0.007301384583115578, 0.1016116738319397, 0.015864485874772072, 0.018779031932353973, 0.07569750398397446, 0.01822805218398571, 0.005654513370245695, -0.0539083257317543, 0.012333869002759457, -0.05764741823077202, -0.007732337806373835, 0.0158048328012228, -0.03531981632113457, 0.0078061604872345924, -0.08697111904621124, 0.0026470967568457127, -0.07606428116559982, -0.009698042646050453, 0.01665136031806469, 0.022383039817214012, 0.023595299571752548, -0.10657685995101929, 0.013375185430049896, 0.06202542036771774, -0.06130053475499153, -0.024916496127843857, 0.014953119680285454, 0.021315865218639374, -0.00807136483490467, 0.04508226364850998, 0.01934863068163395, -0.023829113692045212, -0.02093716710805893, 0.0506635382771492, 0.023984378203749657, -0.05255632475018501, 0.011031845584511757, -0.002741224830970168, 0.0013758203713223338, -0.042829375714063644, -0.0034328093752264977, -0.020068537443876266, 0.05410458520054817, -0.02229112945497036, -0.016324812546372414, 0.0017554357182234526, -0.04091370850801468, 0.029380297288298607, 0.021523579955101013, -0.011667095124721527, -0.01805449277162552, 0.018046360462903976, 0.0024611458647996187, -0.04524069279432297, -0.03419778496026993, -0.029159363359212875, 0.032745473086833954, -0.0670812800526619, -0.03595006465911865, 0.031922075897455215], [-0.00040154915768653154, -0.0027151722460985184, -0.00087147974409163, -0.0017229256918653846, 0.0012529821833595634, -0.00041155918734148145, -0.0014111758209764957, 0.008460122160613537, 0.002768979873508215, 0.009412101469933987, 0.0006170407868921757, -0.0021579863969236612, -0.002724595367908478, 0.0014982061693444848, 0.0005624127225019038, -1.6015272194636054e-05, -0.0019773722160607576, -0.0006117662996985018, -0.0001659426634432748, 0.00032977687078528106, -0.0016324443276971579, -0.0009653804008848965, 0.0001546015264466405, -0.001982460729777813, -0.0019356987904757261, 0.0013944051461294293, 0.0006507128127850592, 0.006902872119098902, 0.0006209458806551993, 1.5114312191144563e-05, -0.0007993541075848043, -0.00019674004579428583, -0.00187564711086452, 0.0014152079820632935, -0.0005494178622029722, 0.0002646550419740379, -0.0002702393103390932, 0.0012253039749339223, 0.0010155296185985208, 0.0013851139228790998, 0.00046176588512025774, 0.0012821402633562684, 0.0029915182385593653, -0.0016001973999664187, 0.0006085046334192157, 0.0010739200515672565, -0.00044694944517686963, 0.00023012535530142486, -0.0015987433725968003, 0.00029369996627792716, 0.00021407162421382964, -0.0032032127492129803, -0.0001213598225149326, -0.0005470322212204337, -0.00015258140047080815, 2.8227825168869458e-05, 0.0069497753866016865, -0.0035732800606638193, -0.000469982682261616, 0.00029565434670075774, -0.00576017564162612, 0.002886710222810507, -0.0025461784098297358, -0.00038974988274276257, 0.0007661281852051616, 4.448697654879652e-05, -0.0007862732745707035, -0.0015096424613147974, 6.178572220960632e-05, 0.0003887325874529779, 0.00016807533393148333, 0.00219052005559206, -0.00037393890670500696, 0.0005764423403888941, 0.004950251895934343, 0.0001957284694071859, -0.0006237632478587329, -0.005423669703304768, -0.00196383916772902, 0.00513810757547617, 0.000490909384097904, 0.0001926505210576579, -0.00061498221475631, 0.00014547874161507934, -0.000895847100764513, 0.0006874061073176563, -0.0007834676071070135, 0.007984280586242676, 0.0003894418478012085, 0.0003026081540156156, -0.00035323071642778814, 0.004213353153318167, 0.00185819820035249, -0.001222784398123622, 0.00021878446568734944, -0.0004447305982466787], [-0.002318242797628045, -0.008560501039028168, 0.0074930512346327305, -0.014470068737864494, -0.0032747203949838877, 0.0010360322194173932, 0.0014626153279095888, -0.009066079743206501, -0.0026179172564297915, -0.028555983677506447, 0.005368758458644152, -0.010909688659012318, -0.0124606192111969, -0.008522490039467812, 0.0017110523767769337, -0.0004023411311209202, 0.008223664946854115, -0.006624925881624222, -0.00022383453324437141, 0.0013845673529431224, 0.005792554002255201, -0.0034051286056637764, 0.0032810638658702374, -0.006126803811639547, -0.0030293064191937447, -0.006219713483005762, 0.0014516591327264905, 0.03164598345756531, -0.006740659009665251, 0.0012729128357023, 0.0011148862540721893, -0.0039844634011387825, -0.00292211608029902, -0.002573127392679453, 0.012619886547327042, 0.0033022467978298664, -0.00020668980141635984, 0.006316469516605139, 0.004660573322325945, -0.004598993342369795, 0.0017708103405311704, 0.003675439627841115, -0.005105571821331978, -0.004446336533874273, 0.0005962431896477938, -0.0017436705529689789, -0.0010261163115501404, 0.0006247812998481095, -0.0016925475792959332, 0.00046068159281276166, -0.019129909574985504, -0.00419510155916214, -0.014337784610688686, -0.004721477162092924, -0.0035862107761204243, -0.003306887112557888, 0.0227656252682209, 0.001659804373048246, 0.003399111097678542, 0.005447967443615198, 0.009879602119326591, -0.01065400056540966, -0.0007829166133888066, -0.0018310581799596548, 0.002664068015292287, -0.002420901320874691, -0.0009218522463925183, 0.0015155611326918006, -0.00041092815808951855, 0.00687177712097764, -0.0034117393661290407, -0.005056141410022974, -0.0049343956634402275, 0.003887841710820794, -0.009622259065508842, -0.0013663364807143807, -0.000665100000333041, -0.005942157004028559, -0.009053140878677368, 0.007236534729599953, -0.0006721642566844821, 0.0004343820328358561, -0.005444197449833155, 0.005588696338236332, -0.005660164635628462, 0.006772186607122421, -0.0038977894000709057, 0.017169909551739693, 0.001872495748102665, 0.008723721839487553, -0.004046903923153877, -0.0017424972029402852, 0.0016597434878349304, -0.005566713400185108, -0.0046020494773983955, -0.0029978922102600336], [0.020282872021198273, 0.016314081847667694, -0.02851276844739914, 0.05096950754523277, -0.048060666769742966, -0.008567566983401775, 0.02966264635324478, 0.06699977815151215, -0.033787861466407776, -0.048270341008901596, -0.006705058738589287, 0.05226340889930725, -0.00010255032975692302, -0.02742241881787777, -0.010105444118380547, 0.045301489531993866, -7.126080163288862e-05, 0.040694862604141235, 0.012599957175552845, -0.002912407275289297, 0.035248998552560806, 0.01617189683020115, 0.10608524084091187, -0.020837867632508278, -0.0015882202424108982, 0.08234267681837082, -0.006428946275264025, -0.0021923671010881662, -0.016090774908661842, -0.002728597493842244, 0.004312917124480009, -0.012025413103401661, 0.009954179637134075, -0.019253063946962357, 0.002069383393973112, 0.01998911239206791, 0.04246629402041435, 0.09795285761356354, 0.018625296652317047, -0.01395166665315628, 0.04401267692446709, -0.03170236572623253, -0.0036276374012231827, -0.01779136061668396, -0.009366262704133987, -0.01479424349963665, -0.0025703972205519676, -0.04814932122826576, -0.011077605187892914, 0.022215941920876503, 0.04997765272855759, 0.06286398321390152, -0.012072678655385971, -0.006068502552807331, -0.011420635506510735, 0.03726658970117569, -0.00655229389667511, 0.032112967222929, 0.05048426240682602, 0.00818421971052885, 0.0027278310153633356, -0.017034683376550674, 0.007811289746314287, 0.026941273361444473, -0.012336197309195995, -0.03519737347960472, -0.0008433926850557327, 0.043268341571092606, -0.013942696154117584, -0.04702205955982208, -0.009764614515006542, 0.017189884558320045, 0.02449502982199192, 0.003648711834102869, 0.008713480085134506, -0.008104146458208561, 0.03473851457238197, 0.05021018907427788, -0.015619530342519283, 0.019139990210533142, -0.020060740411281586, 0.002017925726249814, 0.02191261015832424, 0.020366542041301727, 0.00729704275727272, -0.008534710854291916, -0.0012646927498281002, 0.004226522520184517, 0.004389048088341951, 0.02941513992846012, 0.0010495493188500404, 0.011516810394823551, 0.018341755494475365, 0.051923662424087524, -0.007882492616772652, 0.0030544372275471687], [-0.022121811285614967, -0.021309128031134605, -0.02708013355731964, 0.06190107762813568, 0.0418509840965271, 0.0003043251344934106, -0.03855830430984497, -0.016578983515501022, -0.04416351020336151, -0.009281217120587826, 0.016630688682198524, -0.013150966726243496, 0.05297613888978958, 0.006959435064345598, 0.038337357342243195, -0.013746636919677258, 0.019352847710251808, -0.016466202214360237, -0.01415191125124693, 0.00488112261518836, -0.024957064539194107, -0.0062942830845713615, -0.05866316705942154, 0.08583682030439377, -0.004705873783677816, -0.014542312361299992, 0.018731161952018738, -0.05751602724194527, 0.011366080492734909, -0.02029811404645443, -0.015458140522241592, -0.01051303930580616, -0.002886839210987091, 0.006494713015854359, -0.017046775668859482, 0.03255295753479004, -0.0036152577959001064, -0.024900132790207863, -0.0009599835611879826, 0.03803576901555061, -0.05279088765382767, 0.015969445928931236, 0.0009484124020673335, -0.01055868435651064, 0.023409709334373474, 0.04501941800117493, -0.0237628985196352, 0.015643013641238213, -0.007155411411076784, 0.01632014662027359, 0.06302271783351898, -0.03681139647960663, 0.05105877295136452, -0.008628061041235924, -0.0006200667703524232, -0.01782730408012867, -0.03796808421611786, 0.07331030815839767, -0.017955586314201355, -0.02147934027016163, 0.039771806448698044, 0.035202909260988235, -0.016420487314462662, -0.008240994065999985, 0.012681379914283752, -0.0009486088529229164, -0.011451728641986847, -0.033276788890361786, 0.014891110360622406, -0.007506410591304302, 0.006813987623900175, 0.04153925180435181, -0.010251658968627453, 0.0010306406766176224, 0.03646880015730858, 0.0203666053712368, 0.0023022261448204517, -0.022883260622620583, 0.052177440375089645, 0.010928086005151272, 0.0071951886638998985, -0.024058913812041283, -0.017074285075068474, 0.004340195097029209, -0.007104681339114904, 0.00831650197505951, -0.01441869605332613, 0.008476882241666317, 0.015567912720143795, -0.0024452742654830217, 0.006357989739626646, -0.004453003406524658, 0.05215182527899742, 0.02348240278661251, -0.010889117605984211, -0.021725665777921677], [-0.02199939265847206, 0.07979603111743927, 0.041026998311281204, -0.056543268263339996, -0.154791459441185, 0.04749085009098053, -0.025084055960178375, 0.05329490080475807, -0.035985641181468964, 0.07895269244909286, -0.031586091965436935, 0.014638574793934822, 0.07829280197620392, -0.07541673630475998, -0.0049807666800916195, 0.07344473898410797, 0.018890373408794403, -0.021466467529535294, 0.03770602121949196, 0.056336209177970886, 0.031169865280389786, 0.056738805025815964, 0.17109955847263336, -0.03310924395918846, -0.0120012778788805, 0.04488112032413483, 0.034829769283533096, 0.07028428465127945, -0.02004220522940159, 0.08359739929437637, 0.035946283489465714, -0.03819693997502327, 0.028752421960234642, 0.07978864759206772, 0.06527582556009293, 0.0976405143737793, 0.05554082989692688, 0.1522633284330368, 0.0709196999669075, -0.01363498903810978, -0.030776530504226685, -0.052596572786569595, -0.06123683974146843, -0.06981182098388672, 0.07138890773057938, 0.0437861792743206, -0.024112267419695854, -0.11218170076608658, 0.07454979419708252, 0.02372358925640583, 0.03455950692296028, -0.06053964048624039, 0.07867159694433212, 0.04042888060212135, -0.04893546551465988, 0.10447812080383301, -0.017307104542851448, -0.012571415863931179, 0.044001925736665726, 0.041279442608356476, -0.02013321779668331, -0.011853278614580631, -0.007505594287067652, 0.11802084743976593, -0.05807402729988098, -0.05307269096374512, -0.08705281466245651, 0.013282226398587227, -0.000565155001822859, -0.04087302088737488, -0.08158756047487259, 0.0710134506225586, 0.021951643750071526, 0.049067288637161255, -0.07346678525209427, -0.07037611305713654, 0.08003305643796921, 0.040341220796108246, 0.01523100771009922, -0.006717113312333822, -0.03173544630408287, 0.06213541701436043, 0.09641693532466888, -0.016019290313124657, 0.07502538710832596, 0.05147264525294304, 0.03430008143186569, -0.014259851537644863, 0.06992745399475098, -0.005177312530577183, -0.046238191425800323, -0.02939498797059059, -0.05137990787625313, 0.1155056431889534, 0.026092244312167168, -0.023063387721776962], [-0.03420410677790642, -0.06067463010549545, -0.011432959698140621, 0.04813412204384804, 0.060240667313337326, -0.04678426310420036, -0.03539997339248657, -0.05013488233089447, -0.02098044753074646, -0.00870717503130436, -0.009405409917235374, -0.05238070338964462, 0.014929424040019512, 0.009697342291474342, 0.06327593326568604, -0.03572063520550728, 0.04564400017261505, 0.018326163291931152, -0.04389836639165878, 0.029932519420981407, 0.029717618599534035, 0.036536771804094315, -0.07718624174594879, 0.04821639135479927, 0.03040693700313568, 0.027324894443154335, 0.013708905316889286, 0.07507488876581192, 0.0346897691488266, -0.030758310109376907, -0.02682689018547535, -0.010205293074250221, -0.020067693665623665, -0.008848099038004875, -0.060944486409425735, 0.014741544611752033, -0.03220337629318237, -0.009354165755212307, 0.04866698756814003, 0.05504944920539856, -0.05403167009353638, 0.0012607017997652292, -0.008806686848402023, 0.0067932698875665665, 0.027513250708580017, -0.00018461942090652883, -0.03508223220705986, 0.008386768400669098, 0.006778523791581392, 0.017602402716875076, 0.06847815215587616, -0.034126169979572296, 0.11417528986930847, -0.020342759788036346, 0.051773034036159515, -0.017023490741848946, -0.0021693077869713306, 0.0996762216091156, -0.04164295271039009, -0.03922901302576065, 0.04403276368975639, 0.019490038976073265, -0.0016950815916061401, 0.017328498885035515, 0.04451608657836914, 0.0035027398262172937, 0.0419529564678669, 0.011199194006621838, -0.029093842953443527, 0.02533336915075779, -0.014482240192592144, 0.014530546963214874, -0.018402572721242905, -0.01484301034361124, 0.049149077385663986, -0.0038656177930533886, -0.045832253992557526, -0.008349194191396236, 0.04906758666038513, 0.003485206514596939, -0.003998436965048313, 0.012888624332845211, 0.0009235962643288076, -0.034350622445344925, -0.011195692233741283, 0.04101323336362839, 0.018023928627371788, 0.054184168577194214, -0.025786245241761208, -0.005618852097541094, 0.013357884250581264, 0.02231096848845482, 0.044100530445575714, 0.007759477943181992, -0.04307461902499199, -0.03697901591658592], [-0.02782481536269188, -0.005340682342648506, 0.044108785688877106, -0.07324846088886261, -0.11953284591436386, 0.03981335088610649, -0.019338158890604973, 0.023340972140431404, -0.02757287211716175, 0.03639845922589302, 0.01823372207581997, -0.0030700755305588245, -0.06865475326776505, -0.008855309337377548, -0.08055669069290161, 0.004375108052045107, -0.09297602623701096, -0.0343015231192112, 0.08172159641981125, 0.06991269439458847, 0.024033665657043457, -0.061682842671871185, 0.1421191394329071, -0.14150069653987885, -0.03422572463750839, 0.11118590086698532, 0.10869722068309784, -0.010099584236741066, 0.0075957756489515305, 0.01634550653398037, -0.004543598275631666, -0.04219764098525047, -0.04212116450071335, -0.04691092669963837, -0.005693781189620495, 0.012726133689284325, -0.0403938852250576, 0.06923458725214005, -0.011909019201993942, -0.022729521617293358, -0.014962923713028431, -0.006214456632733345, 0.062267910689115524, -0.005600459408015013, -0.06006035581231117, 0.006772808730602264, -0.02278606966137886, -0.001048969803377986, -0.033120185136795044, 0.052235908806324005, -0.11590463668107986, 0.10031865537166595, -0.11977414041757584, -0.0004195350338704884, 0.026225894689559937, -0.001620285795070231, -0.039133790880441666, -0.18108785152435303, 0.02850559540092945, 0.11913123726844788, -0.06537850946187973, -0.014770767651498318, -0.007827701047062874, 0.0023722078185528517, 0.04570932686328888, 0.04883485659956932, 0.015108661726117134, 0.018341319635510445, -0.007079873699694872, 0.011436495929956436, 0.03393200784921646, -0.1817425787448883, 0.05895552039146423, -0.006648680195212364, -0.05849021300673485, -0.04661252349615097, 0.0557478591799736, 0.01333199068903923, 0.07149337977170944, 0.0640370175242424, -0.07051832228899002, 0.15272898972034454, 0.06130867078900337, 0.07229499518871307, 0.030336538329720497, -0.015685388818383217, -0.002551815239712596, 0.01548671629279852, 0.04089811071753502, 0.001141324988566339, -0.05012526363134384, -0.007033423986285925, 0.0491250604391098, -0.062406133860349655, 0.006365818902850151, -0.003229325171560049], [-0.03393399715423584, 0.02557040937244892, -0.03818584606051445, -0.022345339879393578, 0.03886266052722931, -0.011320963501930237, -0.0005948020261712372, -0.026055827736854553, -0.09376291930675507, 0.053289785981178284, -0.06628893315792084, -0.0984092578291893, -0.009959847666323185, 0.015595445409417152, 0.010927030816674232, -0.00850859098136425, 0.08881918340921402, -0.05944440886378288, -0.06661195307970047, 0.013056496158242226, 0.023224446922540665, 0.02983320690691471, -0.07732155174016953, 0.04559946805238724, -0.062346138060092926, 0.03412409499287605, 0.04469025880098343, 0.004720304161310196, -0.015882067382335663, -0.03100794367492199, -0.04608488827943802, -0.07673150300979614, 0.09045745432376862, 0.006851005367934704, -0.07779551297426224, 0.0780751034617424, -0.03131181374192238, -0.03338434547185898, 0.03164224699139595, 0.025430286303162575, 0.016529008746147156, 0.036273516714572906, 0.05397946760058403, -0.06168258935213089, -0.0017350816633552313, 0.060322877019643784, -0.07140855491161346, -0.0016612699255347252, -0.058385565876960754, -0.0039044858422130346, 0.022305166348814964, -0.06421738862991333, -0.033409472554922104, -0.02546725608408451, -0.038198161870241165, -0.07594901323318481, -0.06624556332826614, 0.08479475975036621, -0.01626560091972351, 0.0028823893517255783, 0.028256697580218315, 0.07319250702857971, -0.07094085216522217, -0.012894906103610992, 0.05513201653957367, -0.07229818403720856, -0.005379333160817623, 0.016722075641155243, -0.04330269247293472, 0.010976926423609257, 0.0442156121134758, 0.034541770815849304, -0.09292536973953247, 0.019698718562722206, 0.09091334789991379, 0.045199278742074966, 0.0032174596562981606, -0.031838588416576385, 0.08787377923727036, 0.08032994717359543, 0.005392131395637989, 0.002396250842139125, -0.083038829267025, 0.031751181930303574, -0.02408480830490589, 0.03292408213019371, -0.03506005182862282, -0.05449100583791733, 0.014997115358710289, 0.06609924137592316, -0.057147879153490067, 0.06979057937860489, -0.006176894064992666, -0.008025511167943478, -0.07539676129817963, -0.022779718041419983], [0.0005285170627757907, 0.013850717805325985, 0.022102419286966324, -0.11313419044017792, -0.02552737668156624, 0.0285927914083004, -0.050773993134498596, 0.049095891416072845, -0.07731243968009949, -0.00895626563578844, -0.09904137998819351, 0.06665640324354172, -0.1299913227558136, -0.08515231311321259, -0.047397930175065994, -0.018281567841768265, 0.015772920101881027, 0.035519592463970184, 0.07936182618141174, -0.009115331806242466, 0.05009056255221367, -0.16023993492126465, 0.10660077631473541, -0.0982523038983345, -0.05742953345179558, 0.0744660496711731, 0.0420939140021801, -0.04220022261142731, 0.05292285606265068, 0.12409613281488419, 0.08870740234851837, 0.08101392537355423, -0.018749801442027092, 0.03594306856393814, -0.06039143726229668, -0.014958186075091362, 0.010548318736255169, 0.10512673109769821, 0.081669382750988, -0.08427482843399048, 0.12035658210515976, -0.024602031335234642, 0.05853908136487007, -0.10047001391649246, 0.102669358253479, 0.013009559363126755, 0.058910150080919266, 0.035144224762916565, 0.05828061327338219, -0.008977464400231838, -0.2120761275291443, 0.017028674483299255, -0.13528865575790405, 0.05774233490228653, -0.01803004927933216, 0.08070816099643707, 0.029151111841201782, -0.15843252837657928, 0.021453332155942917, -0.05680987983942032, -0.04594920575618744, -0.09010228514671326, 0.020772619172930717, -0.004330527503043413, 0.03080458752810955, 0.031133096665143967, -0.0641539990901947, -0.0070512969978153706, 0.06460890173912048, -0.014028748497366905, 0.0793120265007019, -0.21302437782287598, 0.1127835065126419, 0.062008053064346313, -0.047975342720746994, 0.04031461849808693, -0.008004684932529926, 6.0705999203491956e-05, 0.024426549673080444, -0.012233513407409191, -0.053745634853839874, 0.05775059014558792, 0.0213079322129488, -0.012764065526425838, 0.07411602139472961, -0.08351434767246246, 0.10681559145450592, -0.007172882556915283, -0.00384972314350307, -0.09718681126832962, -0.0702483132481575, -0.13794314861297607, 0.004836477804929018, 0.05379264056682587, 0.008054456673562527, 0.030028272420167923], [-0.09255803376436234, 0.039060432463884354, 0.05997849628329277, -0.12021289020776749, -0.07345430552959442, -0.03557778149843216, 0.006706608925014734, -0.012319722212851048, 0.0628616064786911, -0.02752869389951229, 0.003370496444404125, -0.07008231431245804, -0.07356303930282593, -0.05480428412556648, 0.04460334777832031, 0.04267871752381325, 0.003837587544694543, 0.023864643648266792, 0.0461837500333786, 0.03730317950248718, -0.0018142815679311752, -0.04952520877122879, 0.04756183177232742, -0.0460578091442585, -0.06984177976846695, -0.013875852338969707, -0.020375091582536697, 0.015850694850087166, 0.010097895748913288, 0.03766540065407753, 0.06131047382950783, -0.09003276377916336, 0.07444223016500473, 0.007276044227182865, -0.048766788095235825, 0.009966658428311348, 0.03979313001036644, 0.045047152787446976, 0.049119431525468826, -0.013691955246031284, -0.034634627401828766, 0.039617039263248444, -0.009508426301181316, -0.05182429403066635, -0.0468689389526844, -0.029823580756783485, -0.019997263327240944, -0.043377965688705444, -0.05434989556670189, -0.023471036925911903, -0.09047102183103561, 0.06516692042350769, -0.06926023215055466, 0.008270403370261192, -0.020317375659942627, -0.1091364175081253, 0.1044657900929451, -0.09963545948266983, -0.01886080391705036, 0.052197135984897614, 0.0655490905046463, -0.07464142143726349, 0.018415117636322975, -0.04019661992788315, 0.015123357996344566, -0.032455675303936005, -0.04801136627793312, -0.04655947908759117, 0.08168986439704895, 0.026774613186717033, -0.002058696700260043, -0.04658516123890877, -0.035701602697372437, 0.017825642600655556, 0.03466648980975151, -0.01894495263695717, 0.06620388478040695, -0.03232154995203018, 0.0832098126411438, 0.054150089621543884, 0.03102364018559456, -0.027984891086816788, 0.009041743353009224, -0.04614438861608505, -0.021490229293704033, -0.0336758978664875, -0.024827728047966957, -0.04334462806582451, 0.05394837632775307, -0.06602919846773148, 0.02606651559472084, 0.011717258021235466, 0.03736548125743866, 0.0540434755384922, -0.03243398666381836, -0.033435478806495667], [0.043375302106142044, -0.003754958976060152, -0.028492484241724014, 0.06016792356967926, -0.05844216048717499, 0.01465887762606144, -0.05059657618403435, 0.018496312201023102, 0.09573013335466385, -0.04210377857089043, -0.013651737943291664, 0.01049440074712038, 0.0710316076874733, -0.02962084859609604, -0.0016587807331234217, 0.11645562201738358, 0.07952618598937988, 0.004985828883945942, -0.057904813438653946, 0.012385770678520203, 0.10000689327716827, 0.08473289757966995, 0.13520623743534088, 0.09704509377479553, 0.0003230952424928546, 0.07268496602773666, -0.013796890154480934, -0.07281611859798431, -0.0376291424036026, -0.014784853905439377, 0.04450178146362305, -0.037146009504795074, 0.10768277198076248, -0.02445819228887558, -0.07573212683200836, 0.001181139494292438, 0.08585495501756668, 0.1450643688440323, -0.042821239680051804, 0.04133087396621704, 0.05950608849525452, 0.017705244943499565, -0.021151814609766006, -0.0011471445905044675, -0.08626289665699005, -0.039196666330099106, 0.03241937607526779, 0.007892739959061146, -0.016960058361291885, 0.007622487377375364, -0.05564175173640251, -0.025026720017194748, 0.0033852963242679834, 0.0398717075586319, -0.070233553647995, -0.043363165110349655, 0.00026591913774609566, -0.014706764370203018, 0.10378554463386536, -0.0345347635447979, -0.05822187662124634, -0.05489945039153099, 0.008002004586160183, 0.05249002203345299, 0.032699912786483765, 0.003974332474172115, 0.05453847721219063, 0.02580222859978676, 0.010328404605388641, 0.023168688639998436, 0.0533934086561203, -0.007664864882826805, 0.07483681291341782, 0.07108990103006363, -0.03537123277783394, -0.03264424949884415, 0.042582228779792786, 0.10988189280033112, -0.004623524844646454, 0.03815114498138428, -0.022418903186917305, 0.06521275639533997, 0.03797631338238716, -0.03911130130290985, -0.037560660392045975, -0.06792540103197098, 0.003572317538782954, 0.008976339362561703, 0.02602350153028965, 0.03759521618485451, -0.07070430368185043, -0.028017980977892876, 0.0422583632171154, -0.014942333102226257, 0.06609520316123962, -0.02579394355416298], [-0.07658659666776657, 0.03252983093261719, -0.04891808703541756, 0.07102761417627335, 0.03419221192598343, -0.09046164155006409, -0.05248517915606499, -0.05128397047519684, 0.025581998750567436, -0.0250987745821476, 0.05740465596318245, -0.12382183969020844, 0.11579768359661102, 0.019832413643598557, -0.028291020542383194, -0.05040178447961807, -0.02342040464282036, -0.0815093144774437, -0.06942626088857651, -0.03322600573301315, -0.08132074028253555, 0.05907410383224487, -0.1713726669549942, 0.10442855209112167, 0.04133966192603111, -0.021250532940030098, 0.04870768263936043, -0.07257527858018875, 0.08648616075515747, -0.07014884054660797, -0.01572142355144024, -0.06356760859489441, 0.0036617584992200136, -0.06598822772502899, -0.040496744215488434, 0.03922928869724274, -0.0017199693247675896, -0.0661507174372673, 0.007084840442985296, 0.09551232308149338, -0.1339622586965561, 0.07234372943639755, 0.08296609669923782, -0.028290700167417526, 0.0576859787106514, 0.09572727233171463, -0.0770002156496048, 0.06265141069889069, -0.0742138996720314, 0.012698124162852764, 0.046746447682380676, -0.02277451753616333, 0.11189339309930801, -0.0730205699801445, 0.03752019256353378, -0.032126788049936295, -0.0030882402788847685, 0.12227379530668259, -0.055010661482810974, -0.10308019071817398, 0.08455470204353333, 0.05551382526755333, -0.026577521115541458, -0.020163603127002716, 0.03933035582304001, -0.018443839624524117, -0.018105395138263702, -0.07939280569553375, -0.0485721081495285, -0.014480135403573513, 0.04754708707332611, 0.12579968571662903, -0.07216136157512665, 0.004966269247233868, 0.08102696388959885, 0.0483105406165123, -0.07657843828201294, -0.0511164553463459, 0.045966267585754395, 0.05552298575639725, -0.00709906779229641, -0.0005804584943689406, -0.06636369973421097, -0.01876296103000641, -0.054471950978040695, 0.045604635030031204, -0.06681261211633682, 0.04221335053443909, -0.019013825803995132, 0.057794053107500076, 0.05694729834794998, -0.06087735667824745, 0.06286652386188507, 0.032412197440862656, 0.05926217883825302, -0.07627005875110626], [-0.053094882518053055, 0.04190786927938461, 0.06010443717241287, -0.09076740592718124, -0.011695846915245056, -0.03380274772644043, -0.001497531426139176, 0.024204405024647713, 0.028744712471961975, 0.0017787840915843844, -0.039412349462509155, -0.012653033249080181, 0.00021610794647131115, -0.08744597434997559, -0.004025893751531839, -0.006859978660941124, -0.050943389534950256, 0.002669675974175334, -0.014368143863976002, -0.0076379552483558655, -0.021735088899731636, -0.011757946573197842, 0.006981366779655218, -0.006437683943659067, 0.016452765092253685, -0.04771656170487404, 0.005356970243155956, -0.028318291530013084, -0.03381935879588127, 0.03171488642692566, -0.001408914104104042, -0.007153262849897146, 0.00794506911188364, 0.03590914607048035, -0.016241565346717834, 0.033355794847011566, 0.015606938861310482, 0.06017023324966431, -0.008297642692923546, -0.013019845820963383, 0.05814614146947861, -0.020670350641012192, 0.02586538903415203, -0.03987395763397217, 0.05017141252756119, -0.028293970972299576, -0.03939732909202576, -0.0178062804043293, 0.0004150511813350022, 0.019685212522745132, -0.07020342350006104, 0.053146541118621826, -0.04989437758922577, -0.00592321390286088, -0.03305783495306969, -0.044370878487825394, 0.07950512319803238, -0.10509014129638672, 0.0008277547894977033, 0.021920926868915558, 0.05971338227391243, -0.05141709744930267, -0.010951381176710129, -0.0252605639398098, 0.026255011558532715, -0.03808268904685974, 0.04139558970928192, -0.0015519013395532966, 0.024988722056150436, 0.07837672531604767, -0.028583459556102753, -0.027726558968424797, 0.013490386307239532, -0.005719561595469713, 0.020213384181261063, 0.003627133322879672, 0.07618299871683121, -0.07086016237735748, 0.032944876700639725, -0.01415372546762228, -0.03167359530925751, 0.023520560935139656, -0.032671138644218445, -0.02058190293610096, -0.04969004914164543, -0.013516745530068874, -0.03091687336564064, -0.05842992663383484, 0.043997686356306076, 0.0027289690915495157, 0.021569401025772095, -0.00015017918485682458, 0.01831095851957798, -0.06524556875228882, -0.04246174171566963, -0.025745095685124397], [0.02981841191649437, 0.013798278756439686, -0.06848249584436417, 0.09838692098855972, 0.00899171270430088, 0.068153016269207, 0.00965490285307169, 0.08201659470796585, 0.026675086468458176, 0.10941579937934875, 0.0824941024184227, 0.06513465940952301, -0.05838252604007721, 0.035607654601335526, -0.08344320952892303, 0.05674790218472481, 0.00026080524548888206, 0.0673021748661995, 0.04213573411107063, -0.014396676793694496, -0.03580484539270401, 0.011069903150200844, 0.10511849820613861, -0.07108817249536514, 0.003913742955774069, 0.11992501467466354, 0.018355611711740494, 0.09603150188922882, -0.0850263312458992, -0.019445938989520073, -0.01282464899122715, 0.0029917284846305847, 0.05346081778407097, 0.07728775590658188, 0.10644754767417908, 0.0018261069199070334, 0.09826034307479858, 0.1468864381313324, -0.04854502156376839, -0.06932762265205383, 0.027174251154065132, -0.10745827853679657, -0.03848749026656151, 0.032281000167131424, -0.035742659121751785, -0.0026914295740425587, -0.03624190762639046, 0.01580939069390297, 0.10565348714590073, -0.023507751524448395, -0.06252866983413696, 0.06332278251647949, -0.023312615230679512, 0.01676313392817974, 0.0679929256439209, 0.12498801946640015, 0.08270984888076782, 0.01236387062817812, 0.09725359082221985, 0.03220190852880478, 0.07491950690746307, 0.029054412618279457, 0.0038407896645367146, 0.06971589475870132, -0.04107533022761345, -0.0718185231089592, -0.04576907679438591, -0.038210440427064896, -0.05439033359289169, 0.009456239640712738, -0.03599711135029793, 0.04731767997145653, 0.11850500106811523, 0.02362924814224243, -0.06720500439405441, -0.001361135859042406, -0.0580611489713192, 0.1384592205286026, 0.022298960015177727, -0.03408564627170563, -0.07253354042768478, -0.0012778735253959894, -0.04242347180843353, -0.025128360837697983, 0.0893564373254776, -0.06386782974004745, -0.042427729815244675, -0.02416323870420456, -0.05759735777974129, 0.05124839022755623, -0.02964686043560505, 0.02071073092520237, 0.049755487591028214, 0.0747801885008812, -0.0051523903384804726, 0.05536327138543129], [0.049976009875535965, 0.0458097867667675, -0.06823179125785828, -0.042001090943813324, -0.07066874951124191, 0.0589587427675724, 0.0741034746170044, 0.015339944511651993, 0.062166977673769, 0.030418571084737778, 0.058572907000780106, 0.04681725427508354, 0.08791429549455643, 0.011710012331604958, 0.05939668044447899, -0.006432072259485722, -0.021785089746117592, 0.00494746956974268, 0.026997024193406105, 0.05972376465797424, 0.0058275507763028145, 0.03269960731267929, 0.02499152161180973, 0.04103158041834831, -0.009986218065023422, -0.044347986578941345, -0.0437430813908577, 0.026906996965408325, 0.06629964709281921, -0.050300076603889465, -0.01511994656175375, 0.004616057500243187, -0.04913914203643799, 0.03638487681746483, 0.013763860799372196, 0.013081192038953304, 0.028852704912424088, -0.06815695762634277, 0.003499919082969427, -0.06402358412742615, -0.007672402076423168, -0.0010966038098558784, -0.0494232140481472, -0.04592900350689888, 0.04292704164981842, -0.0387800894677639, 0.07636719197034836, -0.03035147674381733, 0.012266399338841438, -0.04421931877732277, -0.08126085996627808, 0.05610734969377518, 0.010096663609147072, 0.0006885041366331279, 0.07146469503641129, 0.029099296778440475, 0.07106460630893707, 0.01181455422192812, 0.010676655918359756, -0.03296734392642975, 0.06868281960487366, -0.008470511063933372, 0.03295435011386871, 0.06318102777004242, 0.025428814813494682, -0.043684475123882294, -0.0382864736020565, 0.03772298991680145, 0.02631099708378315, -0.04971567541360855, 0.05023268610239029, 0.07745672017335892, 0.04626201465725899, -0.05343484878540039, 0.07011532038450241, 0.040751438587903976, -0.036194510757923126, 0.05074514076113701, 0.08551911264657974, -0.029972998425364494, 0.003569473512470722, -0.002413299633190036, -0.01291123777627945, -0.013653379864990711, -0.021212255582213402, -0.031303178519010544, 0.003560546087101102, 0.06632620841264725, -0.04493078589439392, -0.016724057495594025, 0.014170587062835693, 0.014321014285087585, 0.02466232143342495, -0.00741210114210844, 0.021471917629241943, 0.012123825028538704], [-0.07225120812654495, -0.08793219178915024, -0.008983859792351723, 0.00537188071757555, 0.1188296526670456, 0.029705196619033813, -0.08237120509147644, -0.06456546485424042, -0.04479086026549339, -0.050199031829833984, 0.07770737260580063, -0.0642557218670845, 0.01576237566769123, 0.062093816697597504, 0.07196208089590073, -0.0700984001159668, 0.07470361143350601, -0.05879731848835945, -0.01431875117123127, 0.03487613797187805, -0.07048463821411133, 0.035448480397462845, -0.06942210346460342, 0.12155754864215851, -0.05957310274243355, 0.026104703545570374, -0.02606888860464096, 0.07282978296279907, 0.05953187495470047, 0.0034240353852510452, -0.02282208576798439, -0.01844887249171734, -0.06708834320306778, -0.021037468686699867, 0.039795536547899246, 0.0736144557595253, 0.0033529349602758884, -0.06695878505706787, 0.09416989237070084, 0.10019378364086151, -0.09207210689783096, 0.0137763237580657, -0.019876331090927124, -0.0885874330997467, 0.05886365473270416, -0.0059816124849021435, -0.076848104596138, 0.06953777372837067, 0.059185780584812164, 0.06711961328983307, 0.050875090062618256, -0.08863193541765213, 0.0006634400342591107, 0.0024626736994832754, 0.03508591651916504, -0.023477202281355858, 0.015804724767804146, 0.09301231056451797, -0.00031883595511317253, -0.018721599131822586, 0.03294255957007408, -0.015131855383515358, -0.008200595155358315, 0.02468974143266678, 0.022479478269815445, 0.06367357820272446, 0.034464750438928604, -0.02191483974456787, 0.06473258882761002, -0.047437842935323715, -0.005734081380069256, 0.0904439240694046, 0.004345142748206854, 0.016357073560357094, -0.03588611260056496, 0.04024603217840195, -0.05000346526503563, -0.01740853674709797, 0.01381245069205761, 0.07222158461809158, -0.04942097142338753, -0.060408614575862885, -0.08100540190935135, -0.0028662574477493763, -0.02279292605817318, 0.05782664567232132, 0.0043959286995232105, 0.05076715350151062, 0.05555761232972145, -0.014569805935025215, -0.03426774591207504, -0.07486626505851746, 0.022546496242284775, -0.047003064304590225, -0.0955570787191391, -0.0740598663687706], [-0.10819622874259949, -0.08222983777523041, 0.0655573308467865, 0.01583743840456009, 0.09733699262142181, -0.01109684631228447, 0.04250263422727585, -0.04843269661068916, -0.016811784356832504, -0.01577286422252655, -0.037833280861377716, -0.008262128569185734, 0.006008055992424488, 0.009413364343345165, 0.08626853674650192, 0.02944689430296421, 0.06366422772407532, -0.009470163844525814, -0.04391752555966377, -0.02015736885368824, 0.020077161490917206, -0.043654583394527435, -0.06795629858970642, 0.1031874269247055, -0.0723714753985405, 0.02490091696381569, 0.019733503460884094, -0.09046065807342529, 0.035710759460926056, 0.061547279357910156, -0.07241945713758469, 0.004105442203581333, 0.07527203112840652, -0.055349040776491165, 0.02244153432548046, 0.036759890615940094, 0.02585712820291519, -0.09580164402723312, -0.010258390568196774, 0.03423750028014183, -0.048761285841464996, 0.01421841699630022, -0.00867419969290495, -0.06464731693267822, 0.06064659729599953, 0.06240924820303917, -0.011552327312529087, 0.07562830299139023, 0.03624168783426285, 0.024164995178580284, 0.04584861546754837, -0.05750759318470955, 0.01233571395277977, 0.025050142779946327, -0.06867826730012894, -0.12479129433631897, 0.05294353514909744, 0.08825547993183136, -0.006359621416777372, -0.03346586227416992, 0.09896817058324814, 0.04212477430701256, -0.04631021246314049, -0.03160906210541725, -0.05028017982840538, 0.01006307639181614, -0.024231908842921257, -0.0417320653796196, 0.06343580037355423, -0.0475693978369236, -0.02732129953801632, 0.023540671914815903, -0.0891067162156105, -0.05147721245884895, -0.00949144084006548, -0.0029239822179079056, 0.07733844965696335, -0.019351070746779442, 0.03619901090860367, 5.658821464749053e-05, 0.027843564748764038, -0.0698414221405983, 0.0355454757809639, -0.05228807404637337, 0.05707910656929016, -0.06308604031801224, -0.015483669005334377, -0.046004246920347214, 0.018026746809482574, 0.042477548122406006, 0.02371983602643013, 0.02372553199529648, 0.08981183916330338, 0.08472275733947754, -0.018336044624447823, -0.02990279346704483], [0.016514848917722702, -0.023644020780920982, -0.07551632076501846, 0.04228845611214638, -0.044509630650281906, 0.06630546599626541, 0.03016619011759758, 0.11490928381681442, 0.037498123943805695, 0.08711303025484085, -0.033601533621549606, 0.06660860776901245, -0.014673547819256783, -0.007827786728739738, 0.0024257078766822815, 0.021094534546136856, -0.05489615350961685, -0.009861573576927185, -0.0009880163706839085, 0.04993961751461029, 0.0949159637093544, -0.034389566630125046, 0.13288068771362305, -0.0036289256531745195, 0.04450560361146927, 0.08763322234153748, 0.04541366547346115, 0.040696099400520325, -0.07344363629817963, 0.03920554369688034, 0.011037491261959076, -0.023168470710515976, 0.08010680973529816, -0.02897626906633377, -0.023087263107299805, 0.05598120391368866, 0.08212324976921082, 0.08205405622720718, 0.04571950063109398, -0.013017816469073296, 0.04811674356460571, -0.06802200525999069, 0.07274071127176285, -0.023430371657013893, -0.05272076278924942, -0.046982891857624054, -0.009075963869690895, -0.08469552546739578, 0.08475720137357712, 0.04729579761624336, -0.02963564358651638, -0.04480350762605667, 0.045366909354925156, 0.053974516689777374, 0.02022111415863037, 0.11419220268726349, 0.09092345088720322, 0.007614749483764172, 0.037930235266685486, 0.013670478947460651, 0.09034606069326401, 0.010541876778006554, 0.04339606314897537, 0.06233664229512215, -0.016376294195652008, -0.08717205375432968, -0.05844513699412346, 0.009540551342070103, 0.032108698040246964, -0.0767158567905426, 0.027404092252254486, 0.007780654821544886, 0.09167639911174774, -0.0031731079798191786, 0.033252354711294174, 0.00977590773254633, 0.010163863189518452, 0.0874827653169632, 0.08050165325403214, -0.014616943895816803, -0.04213952273130417, 0.048635270446538925, 0.008286543190479279, -0.0019857718143612146, 0.0462435819208622, 0.03585025295615196, 0.058068182319402695, -0.01625075191259384, 0.01648869179189205, 0.0026426934637129307, -0.01029407512396574, -0.020652195438742638, -0.009001366794109344, 0.10255235433578491, -0.007695219479501247, 0.016450656577944756], [-0.01792445406317711, -0.011536178179085255, -0.008923854678869247, -0.04029719904065132, -0.05467631295323372, -0.011035836301743984, -0.014683394692838192, -0.0005768784321844578, -0.012940327636897564, 0.024570103734731674, 0.01821066625416279, -0.008935258723795414, 0.007071933709084988, -0.032990455627441406, -0.007231177296489477, -0.002274181228131056, -0.008380493149161339, 0.003942446317523718, 0.006075897254049778, 0.026286322623491287, -0.004597695078700781, -0.008568051271140575, 0.041186317801475525, 0.0063789524137973785, 0.02802051045000553, -0.024710986763238907, 0.034660108387470245, -0.020773431286215782, -0.01933865435421467, 0.00175836484413594, -0.008575746789574623, -0.028180347755551338, 0.03040541708469391, -0.01141653023660183, 0.05621989071369171, -0.022194769233465195, 0.0035518109798431396, 0.032769326120615005, 0.02404695563018322, -2.3043698092806153e-05, 0.01925092563033104, -0.014893702231347561, -0.011551440693438053, -0.03366545960307121, -0.021136561408638954, -0.027069400995969772, 0.017634645104408264, 0.017702767625451088, -0.005411922466009855, 0.00595836853608489, -0.026713760569691658, -0.015422774478793144, -0.06947822123765945, -0.00813464354723692, -0.015008430927991867, -0.01835389994084835, -0.039928458631038666, -0.08500033617019653, 0.02137131057679653, 0.021893812343478203, -0.003770355135202408, -0.019319696351885796, -0.01662268489599228, 0.00698428601026535, 0.00681539811193943, -0.003997054882347584, 0.024454208090901375, -0.008063536137342453, 0.0032577980309724808, 0.01538139209151268, -0.029623640701174736, -0.08901819586753845, -0.0157255157828331, 0.0015808880561962724, -0.0011867498978972435, -0.008556361310184002, 0.04121670871973038, 0.0021774873603135347, -0.006944724358618259, 0.029308289289474487, -0.016825232654809952, 0.00705469585955143, -0.01586972177028656, 0.02045200578868389, -0.01944703236222267, 0.03217804804444313, -0.02737058699131012, -0.026835089549422264, 0.00411243038251996, -0.009302033111453056, -0.014842985197901726, -0.006784034892916679, 0.01134938932955265, -0.04849071428179741, -0.03295635059475899, 0.0011926975566893816]], "b2": [-0.03276762738823891, 0.02397214062511921, -0.08222966641187668, 0.003792881267145276, 0.05786044895648956, -0.034064922481775284, 0.06697938591241837, -0.008858158253133297, 0.03526534512639046, 0.03056829608976841, -0.023493561893701553, 0.05413851886987686, 0.034241244196891785, -0.013377474620938301, -0.02862580120563507, -0.059909891337156296, 0.011229063384234905, 0.07609321177005768, -0.033039454370737076, -0.024892345070838928, -0.09217218309640884, 0.04274969547986984, -0.028951255604624748, 0.07353905588388443, -0.05234221741557121, 0.020880330353975296, 0.0082962391898036, -0.08563879132270813, 0.0026113956701010466, 0.06983155757188797, 0.03915201872587204, -0.01376780029386282], "W3": [[-0.07512298226356506, -0.08440444618463516, 0.23932470381259918, 0.19760996103286743, 0.16367387771606445, -0.03323981910943985, 0.2371928095817566, -0.14674700796604156, 0.16305093467235565, -0.14739784598350525, -0.13196317851543427, 0.1049138680100441, 0.12458861619234085, 0.0033233901485800743, 0.014341502450406551, -0.09110210835933685, 0.06833001226186752, -0.20679831504821777, 0.09924909472465515, 0.2737361490726471, 0.18966858088970184, 0.31354424357414246, 0.17275330424308777, -0.20910079777240753, 0.25848615169525146, 0.12163728475570679, -0.1930430382490158, -0.13251762092113495, 0.19594326615333557, 0.18615464866161346, -0.13547886908054352, 0.08659658581018448]], "b3": [-0.04959803819656372]}, "2023": {"W1": [[0.013285141438245773, -0.09118317067623138, -0.0026636184193193913, -0.09990431368350983, 0.01427369937300682, -0.13392235338687897, 0.04309475049376488, -0.0409386046230793, 0.05325284227728844, 0.03190835937857628, 0.03580882027745247, 0.010220551863312721, -0.02242257632315159, 0.09587952494621277, 0.03166944906115532, -0.09274748712778091, -0.08875070512294769, -0.0932684987783432, -0.02211836166679859, 0.06618823856115341, 0.00854155607521534, 0.08573877811431885, -0.07421915233135223, -0.007628231775015593, 0.07625454664230347, -0.023661814630031586, -0.03705232962965965, 0.06460248678922653, -0.1612979620695114, 0.06137679144740105, 0.12819470465183258, -0.07881875336170197, -0.0538671500980854, -0.10535100847482681, -0.003725358983501792], [-0.06697620451450348, -0.12140586227178574, -0.0905241146683693, -0.07584837079048157, 0.00468679703772068, -0.023539312183856964, -0.04885187745094299, 0.03169319033622742, -0.12168746441602707, 0.001931778620928526, 0.054405923932790756, 0.14590011537075043, -0.06457945704460144, -0.07121235877275467, -0.00026743917260318995, 0.09277249127626419, 0.05076242983341217, 0.08641301840543747, -0.00039050163468346, 0.050375085324048996, -0.051623061299324036, -0.023552456870675087, -0.03912577033042908, -0.005346323829144239, -0.07942727208137512, -0.05947276949882507, -0.07815414667129517, 0.041962992399930954, -0.0847049430012703, 0.002256574807688594, -0.1260938197374344, 0.03612717241048813, 0.017365258187055588, -0.05314222350716591, -0.030526163056492805], [-0.0754147544503212, 0.009348095394670963, 0.04828091710805893, -0.059701643884181976, -0.08968966454267502, -0.07529360055923462, -0.08032262325286865, 0.05619402602314949, 0.04299348592758179, -0.04661908000707626, 0.08385985344648361, 0.03257453814148903, 0.02571294829249382, -0.03017168864607811, 0.049621812999248505, -0.06248708441853523, -0.014028992503881454, 0.09034525603055954, 0.062447741627693176, 0.04519273713231087, -0.09012719988822937, 0.0261845625936985, -0.049536701291799545, -0.08943770080804825, -0.0010340941371396184, 0.05853474885225296, 0.09464328736066818, 0.04335043579339981, -0.006276273634284735, 0.08182407915592194, 0.017921138554811478, 0.032310619950294495, -0.010054389014840126, -0.008484709076583385, 0.005853443872183561], [0.02198750711977482, -0.01588364876806736, 0.01283815037459135, -0.021183621138334274, 0.060921672731637955, 0.001111219055019319, 0.09821882843971252, 0.05232197791337967, -0.004111523739993572, 0.02593303844332695, 0.05122734233736992, 0.09258480370044708, 0.0634874477982521, 0.04257139936089516, -0.0003230055735912174, -0.15941138565540314, -0.005786251276731491, 0.023151597008109093, -0.0016119570937007666, -0.05712705850601196, -0.15469612181186676, -0.037542782723903656, 0.022759560495615005, -0.04872032627463341, 0.0059041716158390045, 0.03146829083561897, -0.11481382697820663, 0.07007741928100586, -0.08711495995521545, -0.09024262428283691, 0.11287304759025574, -0.20997799932956696, 0.013202134519815445, 0.20785444974899292, 0.0519145242869854], [0.016840144991874695, -0.04357842728495598, 0.07408039271831512, -0.0364697128534317, 0.025097403675317764, -0.08932358026504517, -0.020321954041719437, 0.029186511412262917, 0.04935769736766815, 0.03559206426143646, 0.09781200438737869, -0.012804269790649414, 0.0001770309027051553, 0.07345576584339142, 0.07106033712625504, -0.13632798194885254, 0.17482122778892517, -0.09457976371049881, -0.12728162109851837, 0.029994193464517593, -0.10262561589479446, 0.03981214016675949, -0.017370784655213356, -0.045934781432151794, 0.06102052703499794, 0.06730479747056961, -0.05358957499265671, 0.04968005046248436, 0.14734789729118347, 0.06490455567836761, 0.0031935961451381445, -0.17892737686634064, -0.009778691455721855, -0.041302185505628586, -0.04768422991037369], [0.03359687700867653, 0.011908457614481449, 0.020326826721429825, -0.08642849326133728, 0.08507294207811356, 0.0012826041784137487, -0.016164757311344147, -0.0029384030494838953, 0.037006694823503494, -0.00456508481875062, -0.07558716088533401, 0.012066773138940334, -0.09361811727285385, -0.09944773465394974, 0.07188324630260468, -0.023642079904675484, -0.03273262456059456, 0.049564018845558167, 0.13396887481212616, -0.05888711288571358, -0.007499419152736664, 0.029486291110515594, 0.0686740055680275, -0.019599106162786484, -0.06566088646650314, 0.037259649485349655, 0.004692388698458672, 0.03334835544228554, 0.018417540937662125, 0.0458819754421711, -0.08123229444026947, 0.09227412194013596, 0.11880394071340561, 0.02005542442202568, 0.1083555668592453], [-0.050661951303482056, 0.015950437635183334, -0.021744459867477417, -0.0879795178771019, -0.05698613449931145, -0.07590039819478989, -0.01257256604731083, 0.03671194612979889, -0.006504411343485117, -0.08581644296646118, -0.012327470816671848, 0.030204344540834427, -0.03144995868206024, 0.014207442291080952, 0.031422264873981476, 0.05773830786347389, 0.04779183864593506, -0.061785951256752014, -0.05584348365664482, -0.08013755083084106, 0.038053032010793686, 0.037245821207761765, 0.040356703102588654, 0.0009924861369654536, -0.02904525212943554, -0.04588600993156433, -0.04834198206663132, -0.05761256814002991, 0.006616361904889345, 0.04720919206738472, 0.04289670288562775, -0.07164236158132553, 0.01581456884741783, -0.015404244884848595, 0.007846707478165627], [0.05807274952530861, -0.13119985163211823, -0.09891357272863388, -0.11580904573202133, -0.027616623789072037, 0.1465625762939453, 0.05936386436223984, -0.09704847633838654, 0.10527849197387695, 0.001610680716112256, -0.09327518194913864, 0.028854969888925552, -0.08540824800729752, -0.03715532645583153, -0.06059438735246658, 0.11013515293598175, 0.010101154446601868, -0.09451626241207123, -0.05924179404973984, -0.06409544497728348, 0.07544726133346558, -0.07150150090456009, 0.03221374377608299, -0.016430186107754707, -0.06969182193279266, -0.07499674707651138, 0.11148756742477417, 0.027129556983709335, -0.0103443069383502, -0.037961557507514954, -0.06440112739801407, -0.14057840406894684, -0.07414812594652176, -0.058651864528656006, -0.07579833269119263], [0.06679652631282806, 0.007899770513176918, 0.04834159091114998, 0.06188058480620384, -0.1009225994348526, -0.11268402636051178, -0.08296407759189606, -0.10044965893030167, -0.04847357049584389, -0.07659073919057846, 0.09338561445474625, 0.11321253329515457, -0.10592131316661835, -0.053014788776636124, -0.08247194439172745, 0.009398788213729858, 0.026261471211910248, 0.023660602048039436, 0.1086212545633316, 0.051755331456661224, -0.01973608322441578, -0.046263113617897034, 0.011861925944685936, 0.05232074484229088, -0.02569311298429966, -0.07782834768295288, 0.08471006900072098, 0.11701612174510956, -0.010460970923304558, -0.003796168603003025, 0.016514014452695847, -0.07724644988775253, -0.06152611970901489, -0.08448150753974915, -0.1046341061592102], [0.05504857748746872, -0.03918764367699623, -0.10584944486618042, 0.03411583974957466, -0.08320622146129608, 0.03825044259428978, -0.0667886957526207, -0.007505371700972319, 0.10668456554412842, -0.006034940015524626, -0.004649328999221325, 0.08340994268655777, 0.039960674941539764, -0.015074026770889759, -0.09369191527366638, -0.09380645304918289, -0.0758797898888588, -0.07923863083124161, 0.08458122611045837, -0.04191414266824722, 0.16080446541309357, -0.050447676330804825, -0.0709051862359047, -0.097456194460392, -0.06959772855043411, -0.06297794729471207, -0.0010819529416039586, -0.014595475979149342, 0.010746677406132221, -0.08084447681903839, -0.06330784410238266, -0.06954096257686615, 0.015048443339765072, -0.10622671991586685, -0.11556801944971085], [-0.047473061829805374, 0.0397782102227211, -0.037632379680871964, -0.0474800206720829, -0.058400966227054596, 0.0644431859254837, 0.02354063093662262, -0.03893164172768593, 0.0048162625171244144, 0.04409484565258026, -0.09861339628696442, 0.0538068450987339, 0.05271672084927559, -0.032893285155296326, -0.011076175607740879, 0.0780843198299408, -0.07325064390897751, -0.03548215329647064, -0.043733980506658554, 0.05009527504444122, 0.041922491043806076, -0.02655109204351902, 0.035299137234687805, 0.04686982184648514, -0.03920571506023407, -0.02272336184978485, 0.041170403361320496, 0.017823904752731323, -0.09011435508728027, 0.003710730466991663, -0.017032556235790253, 0.03821651637554169, -0.0038357360754162073, -0.09260638058185577, 0.007167423609644175], [-0.05654899775981903, -0.010512184351682663, -0.03297213464975357, -0.20478010177612305, -0.06259433180093765, -0.11951935291290283, -0.0701792761683464, 0.06216927617788315, 0.06382673978805542, 0.010055530816316605, 0.06554552912712097, 0.14899896085262299, -0.09482330828905106, -0.0925729051232338, -0.0698191300034523, -0.09750371426343918, 0.07123757898807526, 0.024856029078364372, -0.02700895443558693, -0.0823182612657547, 0.03318367525935173, 0.018486713990569115, -0.03132341802120209, 0.005290375091135502, 0.05822892487049103, 0.11011145263910294, -0.12484318017959595, -0.007031474262475967, -0.11784826219081879, -0.07829829305410385, -0.0432446226477623, -0.0210777185857296, -0.041516512632369995, -0.0605560801923275, -0.05977359786629677], [-0.12717613577842712, -0.1157073825597763, 0.04439937695860863, -0.010433435440063477, 0.0025108305271714926, 0.06205014884471893, 0.03197840228676796, 0.01351269893348217, -0.032618917524814606, -0.0345388688147068, 0.02648688480257988, -0.05561821907758713, -0.009270470589399338, 0.03515035659074783, 0.12262175977230072, -0.02825714647769928, -0.0862802267074585, 0.04104683920741081, 0.007550468668341637, 0.0221165232360363, -0.046538110822439194, 0.036430228501558304, -0.005463262088596821, -0.04015388712286949, -0.04883582890033722, 0.10307864844799042, 0.03308429196476936, 0.005538115743547678, 0.06559829413890839, -0.1172521561384201, 0.03046494722366333, -0.14966651797294617, 0.05452338606119156, 0.17646430432796478, -0.01328895054757595], [0.05891676992177963, -0.08481753617525101, 0.038935206830501556, 0.05465090647339821, 0.013644302263855934, -0.009944428689777851, 0.02282724902033806, 0.10293615609407425, 0.06992942094802856, -0.0006271539023146033, 0.05546193569898605, 0.07133615761995316, 0.07101108878850937, -0.01937040314078331, 0.010406472720205784, -0.12154105305671692, 0.021852100268006325, 0.04959949478507042, 0.02967289835214615, -0.0020196251571178436, -0.11794690042734146, 0.002298137405887246, -0.047788482159376144, -0.015041991136968136, -0.028521889820694923, -0.030264757573604584, -0.09430064260959625, 0.054116230458021164, 0.14135326445102692, 0.0034895185381174088, 0.0755312442779541, -0.2081502228975296, 0.0947374626994133, 0.05671589449048042, -0.014672106131911278], [-0.04274291172623634, -0.011544343084096909, -0.00026743824128061533, 0.02975187450647354, -0.044003210961818695, -0.039360225200653076, 0.01958315819501877, 0.01875913143157959, -0.0035839301999658346, 0.004185859113931656, -0.04249660670757294, 0.008559633046388626, 0.008059791289269924, -0.008996103890240192, -0.02681138552725315, 0.020076312124729156, 0.018375277519226074, 0.024829542264342308, 0.05101123824715614, -0.045276135206222534, 0.064128078520298, 0.02929435856640339, -0.037588756531476974, -0.04328130930662155, -0.035738248378038406, 0.014031410217285156, -0.037597086280584335, 0.006105408072471619, 0.02275514043867588, -0.0050311037339270115, 0.024884432554244995, -0.04980143904685974, 0.040811553597450256, 0.016620012000203133, 0.021722964942455292], [0.0033195186406373978, 0.1269700676202774, 0.005800534039735794, 0.12588709592819214, -0.008742744103074074, -0.11365694552659988, -0.008061202242970467, 0.03173123672604561, -0.011675960384309292, 0.047845736145973206, -0.023867167532444, -0.010227011516690254, -0.10806673020124435, 0.0032065310515463352, 0.06535176187753677, -0.0760134905576706, 0.006379565689712763, 0.06504201143980026, -0.010181990452110767, -0.03455042839050293, -0.02371843531727791, 0.01659582555294037, -0.028863077983260155, -0.0362960509955883, -0.001637119916267693, -0.03951995447278023, 0.018924282863736153, -0.02989077940583229, -0.008055037818849087, -0.0540514774620533, 0.07504238933324814, 0.09075197577476501, 0.03585464507341385, 0.06130294129252434, -0.0030690492130815983], [0.006488009821623564, 0.008726251311600208, 0.02892812341451645, -0.02287045493721962, -0.025194896385073662, -0.10659684985876083, 0.009415635839104652, -0.06142718344926834, -0.05550346150994301, 0.016411151736974716, 0.005498763173818588, 0.03381893411278725, 0.007250137161463499, 0.005501973908394575, 0.013528107665479183, 0.10800738632678986, -0.035208456218242645, 0.044106949120759964, 0.05462602153420448, 0.005148420576006174, 0.09241679310798645, 0.03556697443127632, 0.029942596331238747, -0.014800692908465862, -0.03391186520457268, 0.0822547972202301, -0.008195736445486546, -0.010624219663441181, -0.07289743423461914, -0.01130159106105566, 0.018000565469264984, 0.02273111417889595, 0.0028551656287163496, 0.08678262680768967, 0.0567043237388134], [-0.03454591706395149, 0.038329824805259705, 0.07169916480779648, 0.12355925887823105, 0.05067361146211624, -0.04317634552717209, -0.0038923483807593584, 0.0010323881870135665, 0.07371493428945541, -0.032655857503414154, 0.07421480119228363, 0.049553629010915756, -0.0717889592051506, 0.06770742684602737, 0.10705232620239258, 0.06909294426441193, -0.06886015832424164, -0.023354796692728996, 0.06645312160253525, 0.06934686005115509, 0.007333890534937382, -0.02784905768930912, 0.04792280122637749, 0.029523802921175957, -0.09598498046398163, 0.027211831882596016, -0.05521592125296593, 0.007845412939786911, 0.09415257722139359, -0.0011277777375653386, -0.016743095591664314, 0.09403026849031448, -0.009152944199740887, -0.05354991927742958, -0.06773344427347183], [0.042849935591220856, 0.039938393980264664, 0.031114401295781136, -0.036366309970617294, 0.013662335462868214, -0.04502088576555252, 0.05853339657187462, 0.03918687626719475, 0.008134011179208755, 0.010663771070539951, -0.017170006409287453, -0.018511056900024414, -0.0707312673330307, -0.06439372897148132, -0.07878737896680832, -0.016618480905890465, 0.02985497936606407, 0.08455844223499298, 0.04482007026672363, -0.028046973049640656, 0.01530394982546568, -0.02545730397105217, 0.028144801035523415, 0.011203620582818985, -0.04108657315373421, 0.010169191285967827, 0.013728027231991291, 0.0053648194298148155, -0.03904268145561218, -0.07164682447910309, -0.06691355258226395, 0.06419829279184341, 0.03365499898791313, 0.00603485619649291, -0.046914249658584595], [0.023168370127677917, 0.006065670400857925, 0.013294952921569347, -0.009135621599853039, -0.0513729602098465, -0.03711719065904617, -0.014340140856802464, 0.033770669251680374, -0.024871814996004105, 0.03917194902896881, -0.03210005164146423, -0.037554774433374405, -0.06950655579566956, -0.019962409511208534, 0.04319249838590622, -0.005211901385337114, -0.023864757269620895, 0.042963892221450806, -0.01933034136891365, 0.021206026896834373, 0.0613308921456337, -0.006702433340251446, 0.031731825321912766, 0.021342972293496132, -0.011431681923568249, -0.037731677293777466, 0.11262547224760056, 0.006998176220804453, -0.07773766666650772, 0.003283821977674961, 0.08715657889842987, -0.00036116858245804906, 0.058048151433467865, 0.042694903910160065, 0.003345942823216319], [0.05129225179553032, -0.10109934210777283, -0.09773893654346466, -0.07658877223730087, -0.01017477922141552, -0.02930539660155773, -0.03768310323357582, -0.08244927227497101, -0.001165986992418766, -0.0658654198050499, -0.03361710533499718, 0.12455152720212936, -0.014115002937614918, 0.051473576575517654, -0.010444722138345242, 0.11116751283407211, 0.0939701572060585, -0.05651964247226715, -0.054376836866140366, 0.05605669692158699, -0.05280086770653725, 0.09144843369722366, 0.007636582013219595, 0.02132211998105049, -0.09196735173463821, 0.02160952053964138, -0.0007536937482655048, -0.03932742029428482, 0.03701763227581978, -0.01871339976787567, -0.003611438674852252, -0.12712423503398895, 0.03635824844241142, -0.10676132142543793, 0.06708262860774994], [0.017687179148197174, 0.11168787628412247, -0.06077850982546806, 0.015651877969503403, 0.05035438761115074, -0.04368601739406586, -0.07478948682546616, -0.08770161122083664, -0.03310765326023102, 0.0031439708545804024, -0.10144853591918945, 0.057623013854026794, -0.014403124339878559, -0.004231404047459364, -0.03651817888021469, 0.12688954174518585, -0.005799203645437956, -0.01280295755714178, -0.0027763424441218376, 0.07610512524843216, 0.10052323341369629, 0.05830689147114754, 0.0199417807161808, 0.018616842105984688, 0.04306218400597572, 0.00814646203070879, 0.15916605293750763, -0.04317701607942581, -0.11859766393899918, 0.04444630444049835, -0.04527979716658592, 0.07002747058868408, -0.11253741383552551, -0.10644503682851791, 0.0005631880485452712], [-0.09072744846343994, 0.1073828712105751, 0.039040789008140564, -0.10401369631290436, 0.008072907105088234, -0.021063728258013725, 0.04792057350277901, -0.025758637115359306, -0.009377754293382168, -0.01238558255136013, 0.06089490279555321, 0.0622839592397213, -0.025886762887239456, 0.06250885128974915, 0.06851369887590408, -0.13588090240955353, 0.08533543348312378, 0.054509080946445465, 0.054291460663080215, 0.015586632303893566, 0.08639982342720032, -0.09185807406902313, 0.027548298239707947, 0.014957360923290253, 0.02798440493643284, -0.09885178506374359, -0.17731450498104095, -0.029228869825601578, -0.005779154133051634, -0.17854811251163483, -0.15500517189502716, 0.3063456118106842, -0.07017068564891815, 0.1630786657333374, -0.0038145240396261215], [-0.02977881394326687, -0.05490124970674515, 0.1217382401227951, 0.059376005083322525, -0.026252543553709984, -0.09595612436532974, 0.04429174214601517, -0.048717088997364044, 0.01925484836101532, 0.03764756768941879, 0.049646418541669846, -0.1182226687669754, 0.002030610106885433, 0.0027239948976784945, 0.017368966713547707, 0.14100538194179535, -0.08960415422916412, -0.05695807933807373, 0.07860563695430756, -0.006625293754041195, 0.02149399369955063, 0.027007009834051132, 0.003969301003962755, 0.028695356100797653, -0.045812904834747314, 0.16378581523895264, -0.015798572450876236, 0.026500018313527107, -0.17971397936344147, -0.11269562691450119, 0.05414162576198578, -0.1402551233768463, 0.047198712825775146, 0.1711113005876541, -0.00844484381377697], [-0.07059954851865768, -0.07902422547340393, -0.0741085633635521, -0.0871153250336647, 0.06397227197885513, -0.04725711792707443, 0.07188012450933456, -0.050287798047065735, 0.05927445366978645, -0.061567679047584534, 0.06034422293305397, -0.004228349309414625, -0.008049477823078632, -0.013780295848846436, -0.06327562779188156, 0.04950032755732536, -0.04615332931280136, -0.03650223836302757, -0.00814647227525711, 0.058297522366046906, -0.05439320579171181, -0.05285017937421799, 0.06425664573907852, 0.0008296766900457442, -0.02729172445833683, -0.09194950759410858, 0.15826502442359924, 0.054179124534130096, -0.15957343578338623, -0.016952941194176674, -0.09229516237974167, -0.035196270793676376, 0.024478930979967117, -0.11876443773508072, 0.08243659138679504], [-0.006427277345210314, -0.10062454640865326, 0.08859670907258987, -0.14993244409561157, 0.051688093692064285, 0.005869885440915823, 0.050037357956171036, 0.014319179579615593, -0.03377220779657364, 0.029609553515911102, 0.032249271869659424, 0.07855527102947235, -0.019545288756489754, -0.008381346240639687, 0.002921504434198141, -0.023244429379701614, 0.09734728932380676, -0.04580676183104515, -0.04245620593428612, -0.11890053004026413, 0.0820859894156456, -0.10008518397808075, 0.020392583683133125, -0.053140487521886826, -0.010831299237906933, 0.07860005646944046, -0.1324603110551834, -0.014570660889148712, 0.05484206974506378, -0.07614556699991226, -0.1355002522468567, 0.024256426841020584, 0.026130471378564835, 0.15405495464801788, -0.03651941940188408], [0.019469911232590675, -0.033254750072956085, -0.01156205590814352, 0.021500946953892708, -0.02216678485274315, 0.11732465028762817, -0.03168262168765068, -0.0693836659193039, -0.03771304711699486, -0.005232503637671471, -0.002812906401231885, -0.012615958228707314, -0.0030809680465608835, 0.010139958932995796, 0.04776156693696976, -0.05914239212870598, -0.02100471220910549, 0.04716536030173302, -0.04404471069574356, -0.019794469699263573, -0.009336236864328384, 0.007305063772946596, 0.002781268674880266, 0.01746685802936554, 0.02350861206650734, -0.041161373257637024, 0.022750142961740494, -0.008305748924612999, 0.031085029244422913, 0.008814041502773762, -0.031217968091368675, -0.05212334915995598, 0.020913872867822647, 0.010262299329042435, 0.04187041148543358], [0.009159918874502182, -0.020585594698786736, -0.05860452353954315, 0.006096224766224623, -0.08257009088993073, -0.021347418427467346, -0.06792604923248291, -0.016207508742809296, 0.03382493928074837, -0.0008303583017550409, -0.14102207124233246, 0.12526600062847137, 0.02352236583828926, 0.11908940225839615, 0.10625476390123367, 0.09707601368427277, -0.015487534925341606, 0.044977620244026184, -0.02687074989080429, 0.10912366956472397, 0.11393435299396515, -0.08776921033859253, -0.04445229470729828, 0.06998563557863235, 0.07493966072797775, 0.02931312471628189, 0.09899020940065384, -0.06538547575473785, -0.096712626516819, -0.039385829120874405, -0.05618351325392723, -0.11299921572208405, 0.08096527308225632, -0.0567638985812664, -0.004763541743159294], [0.053739894181489944, 0.04687952995300293, 0.10177333652973175, 0.01743266172707081, -0.06337853521108627, 0.08925767987966537, 0.032682787626981735, -0.007129548117518425, -0.043426916003227234, -0.05771154910326004, 0.09810235351324081, -0.06666124612092972, 0.09506537765264511, -0.07881433516740799, -0.03822370991110802, -0.04358281195163727, 0.055402420461177826, 0.032318271696567535, -0.053329527378082275, -0.05239671841263771, 0.017040349543094635, 0.0389619842171669, -0.02885597012937069, -0.06167019531130791, -0.017505664378404617, -0.008082905784249306, 0.042667537927627563, -0.036800283938646317, -0.000648135261144489, 0.05982556939125061, 0.07781191915273666, -0.12682722508907318, -0.04193425923585892, 0.10520681738853455, -0.014874573796987534], [0.008153180591762066, 0.026904286816716194, 0.06120423227548599, -0.05682709068059921, -0.025446157902479172, -0.00619910005480051, 0.06430159509181976, 0.0380355529487133, 0.029462473466992378, 0.014684254303574562, -0.01989464834332466, -0.00142005761153996, 0.005400366615504026, -0.06840700656175613, -0.07457543909549713, 0.022047044709324837, 0.0723939910531044, 0.05944482982158661, 0.018823711201548576, 0.007983716204762459, 0.05821435898542404, -0.07017689943313599, 0.03548131883144379, -0.034683164209127426, -0.05849318206310272, 0.029011115431785583, -0.002775352681055665, -0.03622572124004364, -0.03374225273728371, -0.055901091545820236, -0.05270615220069885, 0.11953754723072052, 0.05813364312052727, 0.07805225253105164, -0.06039672717452049], [0.05742138624191284, 0.009961226023733616, 0.14817485213279724, -0.008446038700640202, 0.04376685246825218, 0.08339425176382065, 0.021499892696738243, -0.04122218117117882, -0.0019393988186493516, 0.07110189646482468, -0.04086649417877197, 0.023983946070075035, -0.06964828073978424, -0.05323735252022743, -0.01612224243581295, -0.07989201694726944, -0.06647952646017075, 0.01047767885029316, 0.015705887228250504, 0.10161194950342178, 0.11653001606464386, -0.0007593244081363082, 0.055440504103899, -0.06376922875642776, -0.05509834736585617, 0.042869601398706436, 0.08914453536272049, 0.01281656976789236, 0.012265863828361034, 0.028585469350218773, 0.00425301119685173, 0.06904776394367218, 0.047265272587537766, -0.006851533427834511, -0.05977622792124748], [-0.018963085487484932, 0.0246804878115654, 0.10554510354995728, 0.02947191148996353, 0.06546703726053238, 0.003448840929195285, -0.05823003128170967, -0.0067558279260993, 0.03217826783657074, 0.018418462947010994, 0.009668946266174316, 0.022980259731411934, -0.0027497767005115747, 0.03505722060799599, 0.03441480174660683, -0.011968987993896008, 0.056292545050382614, -0.016991226002573967, -0.06767518818378448, -0.011654221452772617, -0.028286969289183617, 0.034329600632190704, 0.05322311818599701, 0.023387331515550613, 0.05208376422524452, -0.06814683228731155, -0.03493897616863251, -0.007606239523738623, -0.004868718795478344, 0.02818243019282818, -0.012509307824075222, -0.06451266258955002, 0.01801322028040886, 0.009166047908365726, -0.010101019404828548], [-0.08620909601449966, -0.08203884959220886, 0.05464740842580795, 0.040527258068323135, -0.07738487422466278, 0.03205839917063713, -0.04405607655644417, -0.09570428729057312, -0.0026077532675117254, -0.05501283332705498, -0.05636272951960564, -0.0038852549623697996, -0.009416853077709675, -0.0352664440870285, 0.020160065963864326, -0.02831968665122986, -0.03186942636966705, 0.011213291436433792, 0.037948500365018845, -0.0336432158946991, 0.07952802628278732, 0.018597986549139023, 0.008151691406965256, -0.030023464933037758, -0.048021022230386734, 0.07940738648176193, -0.005935152992606163, 0.007392145227640867, -0.02108333632349968, -0.05085573345422745, -0.05282791703939438, 0.10624672472476959, 0.011887785978615284, 0.13895873725414276, -0.04169592261314392], [0.023665262386202812, -0.011707772500813007, 0.03917261213064194, 0.030056606978178024, -0.0494055449962616, -0.05668198689818382, -0.03349769115447998, 0.033235177397727966, 0.05022794008255005, 0.06238109618425369, 0.002006229478865862, -0.017810143530368805, -0.07388590276241302, -0.011654716916382313, -0.04885527491569519, 0.04765965789556503, -0.04106666520237923, 0.02753523178398609, 0.0926632210612297, -0.08195780962705612, 0.08947473764419556, 0.012662472203373909, -0.05848335102200508, 0.03829740732908249, 0.03959406539797783, -0.06830958276987076, 0.029791690409183502, 0.07078935205936432, 0.06658539175987244, 0.014736080542206764, -0.011557934805750847, 0.0533381886780262, -0.07419456541538239, -0.005169471260160208, -0.03522060438990593], [0.0193749088793993, -0.08750886470079422, -0.04163151606917381, -0.07490943372249603, 0.04334765300154686, 0.04044993594288826, 0.05049750581383705, -0.007394777145236731, -0.08166486769914627, -0.031535468995571136, -0.1264437735080719, 0.03374120965600014, -0.06912973523139954, 0.03751147910952568, 0.0717536136507988, -0.08122322708368301, 0.056323353201150894, 0.03666473925113678, -0.07554709911346436, -0.09007175266742706, 0.13091722130775452, -0.05844634026288986, 0.052608102560043335, 0.04206710308790207, 0.021647728979587555, -0.11800791323184967, 0.056022316217422485, -0.052616897970438004, -0.11887743324041367, 0.09923359006643295, -0.03145766630768776, -0.05017144978046417, -0.09915870428085327, -0.014923684298992157, -0.049510542303323746], [-0.08027293533086777, -0.05342292785644531, 0.034239836037158966, -0.014682568609714508, -0.03054499253630638, 0.060648366808891296, -0.014277562499046326, -0.0002967230975627899, 0.013391646556556225, 0.07475259900093079, -0.024357939139008522, -0.03470870852470398, 0.0047350553795695305, 0.022240810096263885, 0.058828484266996384, 0.03965280205011368, 0.03478356823325157, -0.02052232250571251, 0.02078273519873619, -0.037796854972839355, 0.07354540377855301, -0.029771996662020683, -0.006293303798884153, -0.00923767127096653, -0.06265700608491898, 0.13471990823745728, -0.042347539216279984, 0.020455660298466682, 0.01961803250014782, -0.08307192474603653, -0.005423903930932283, 0.05220077931880951, 0.02251252718269825, 0.2566906809806824, 0.01667569950222969], [-0.05006539821624756, -0.06600183248519897, 0.022379405796527863, 0.05619468539953232, -0.08273166418075562, 0.01129195000976324, -0.03565947711467743, 0.022078407928347588, -0.0775531604886055, 0.072176493704319, -0.049412235617637634, -0.0485064722597599, -0.07695753127336502, -0.00675611849874258, 0.0893346518278122, 0.012447386048734188, -0.05288182944059372, -0.05822517350316048, -0.05187847465276718, -0.054853685200214386, 0.05329131335020065, -0.07431117445230484, -0.06252504140138626, -0.038753509521484375, 0.047357991337776184, 0.03059619665145874, -0.09198562055826187, 0.07770061492919922, 0.07345417141914368, -0.019164301455020905, -0.00035855043097399175, 0.05503213033080101, -0.038568343967199326, 0.034209154546260834, -0.028747500851750374], [0.014345546253025532, 0.022207453846931458, 0.007644844241440296, 0.01111063826829195, 0.07723108679056168, -0.03445823863148689, -0.05136051028966904, 0.11057239025831223, -0.024814117699861526, 0.057270556688308716, -0.031802840530872345, 0.004919611383229494, -0.023428848013281822, 0.06239873170852661, 0.057388365268707275, -0.12901824712753296, -0.07875771075487137, -0.01836559548974037, 0.06138567626476288, 0.054342012852430344, 0.09840056300163269, -0.03867054730653763, -0.018111707642674446, 0.013315650634467602, 0.07910089194774628, 0.1122610867023468, -0.201279878616333, -0.06193828582763672, 0.14562416076660156, -0.11411253362894058, -0.1783929020166397, 0.3200237452983856, 0.0004295277176424861, 0.12883295118808746, -0.023605598136782646], [-0.008299977518618107, -0.02605527825653553, 0.015983546152710915, -0.01121976412832737, -0.0037802953738719225, 0.026152070611715317, -0.06770571321249008, 0.01896698772907257, 0.024498289451003075, 0.006554739084094763, 0.020319685339927673, -0.036948297172784805, 0.01796993799507618, 0.05612621828913689, 0.060075417160987854, 0.0759575143456459, -0.04289611056447029, -0.03587237745523453, 0.01898203045129776, -0.017844827845692635, 0.03286521136760712, -0.045737311244010925, 0.028636211529374123, 0.03772425651550293, -0.018444322049617767, 0.09652809798717499, -0.02868269942700863, -0.0060202633030712605, -0.023715514689683914, -0.01332841720432043, 0.0030994305852800608, 0.03329642862081528, -0.041009072214365005, 0.12233094871044159, 0.059742800891399384], [-0.017858050763607025, -0.0289450716227293, 0.05671413615345955, 0.08639843761920929, -0.006736366078257561, 0.0770006850361824, 0.08349224179983139, 0.08661604672670364, -0.007037041708827019, 0.06722232699394226, -0.05089234560728073, 0.02397751249372959, 0.008982861414551735, 0.024658383801579475, 0.044787853956222534, -0.12202060222625732, -0.04562879353761673, -0.045860424637794495, 0.04658723250031471, -0.04136408865451813, 0.0419735461473465, 0.03973853215575218, -0.03281053155660629, 0.10001640766859055, -0.11031149327754974, 0.061976708471775055, -0.13253431022167206, 0.00800421740859747, -0.07945357263088226, 0.013574887998402119, 0.03048923797905445, -0.08431586623191833, 0.07655122131109238, 0.010411948896944523, 0.08956781774759293], [0.05360760539770126, 0.09853203594684601, -0.0053888168185949326, 0.05815731734037399, 0.1063331738114357, -0.018157104030251503, 0.047449298202991486, 0.03388061001896858, 0.02079520747065544, 0.045767076313495636, 0.06663340330123901, -0.03985096514225006, -0.02513568475842476, -0.09090910851955414, -0.13120655715465546, -0.03885285556316376, 0.044230349361896515, 0.009866617619991302, -0.0141757195815444, -0.04714776948094368, -0.0780508890748024, -0.026021355763077736, -0.03277205675840378, 0.015409178100526333, 0.047603361308574677, 0.09549667686223984, -0.027848631143569946, -0.048917729407548904, 0.0476907342672348, -0.104390449821949, 0.07046786695718765, 0.11954514682292938, 0.03970061242580414, -0.020968122407794, 0.03457562252879143], [-0.012923703528940678, -0.06680478155612946, 0.025954239070415497, 0.004063262138515711, -0.011323687620460987, -0.015874700620770454, 0.03359111398458481, 0.08023151755332947, 0.055231042206287384, 0.009941508993506432, -0.022800598293542862, -0.019656553864479065, 0.003261612495407462, -0.028446253389120102, -0.0050414567813277245, -0.06025969609618187, 0.0006498069269582629, 0.04631717875599861, 0.06041807681322098, -0.0008467336883768439, -0.10810717940330505, 0.01949905976653099, -0.07144974917173386, -0.05722181499004364, -0.1350858211517334, -0.058173682540655136, -0.09131931513547897, -0.04058480262756348, 0.02538057044148445, 0.06473702192306519, 0.09641456604003906, -0.08012396842241287, 0.07211866229772568, -0.026658600196242332, -0.025499464944005013], [-0.03492490202188492, 0.02714509516954422, -0.06763704121112823, -0.045560453087091446, 0.006798733491450548, -0.07401958107948303, -0.01192737277597189, -0.004713223781436682, 0.04299076274037361, 0.057120926678180695, -0.01045900397002697, -0.10170605778694153, 0.09432705491781235, -0.023574501276016235, 0.014731703326106071, -0.005158720072358847, -0.05311231687664986, -0.014302697032690048, 0.04721054807305336, 0.029950063675642014, -0.00827028974890709, -0.029499080032110214, -0.0347534641623497, 0.0008303350768983364, 0.009604380466043949, 0.08814465999603271, 0.08872317522764206, 0.0516103133559227, -0.012049328535795212, -0.0787353664636612, -0.014486934058368206, 0.03751792758703232, -0.05881140008568764, -0.011347081512212753, -0.06739455461502075], [0.061414964497089386, -0.010977187193930149, 0.0758713036775589, 0.06039043888449669, 0.007982684299349785, -0.0075308275409042835, 0.02755916118621826, 0.086805060505867, 0.1006474569439888, 0.03152373433113098, 0.046138979494571686, 0.0398351289331913, 0.03514288365840912, -0.027675002813339233, -0.001365775358863175, -0.13936541974544525, 0.038436368107795715, -0.019105233252048492, -0.04727773368358612, -0.0025721408892422915, -0.08097927272319794, -0.021968739107251167, -0.018500186502933502, -0.052581336349248886, -0.026059024035930634, -0.05185003578662872, -0.05105051025748253, 0.021773790940642357, 0.07941366732120514, 0.03556961566209793, 0.031337838619947433, -0.17281199991703033, 0.04015679657459259, -0.02733524516224861, -0.03976737707853317], [-0.05070759356021881, 0.020168427377939224, 0.040220607072114944, 0.030218081548810005, 0.06509896367788315, 0.016746116802096367, 0.051522329449653625, -0.0593821257352829, 0.04680247977375984, 0.07025716453790665, 0.028483202680945396, -0.06637982279062271, -0.00719042681157589, 0.04228240251541138, -0.045299988240003586, 0.0772920548915863, 0.03397798910737038, -0.003772987052798271, -0.09424148499965668, 0.02042325586080551, 0.07283410429954529, 0.0050085969269275665, -0.036014359444379807, 0.10664516687393188, 0.0625208169221878, -0.018787091597914696, 0.008037323132157326, 0.02274458296597004, 0.028307294473052025, -0.08487455546855927, 0.04547322541475296, 0.1093217208981514, -0.10744202882051468, 0.03752945363521576, 0.023913973942399025], [0.000827544427011162, -0.07810694724321365, -0.041951168328523636, 0.031050371006131172, 0.0076715643517673016, -0.040808238089084625, 0.0548127107322216, -0.015098923817276955, 0.05732431635260582, -0.0071442159824073315, -0.08949242532253265, 0.009231200441718102, 0.07581157237291336, -0.01744650863111019, -0.00318456650711596, -0.04940173402428627, -0.023406242951750755, -0.042029689997434616, 0.03490316495299339, 0.027538543567061424, -0.13375000655651093, 0.023335348814725876, 0.01758064329624176, -0.07086215913295746, 0.00046336001832969487, 0.06880175322294235, -0.06179837882518768, 0.02145124040544033, 0.02606593258678913, -0.029166322201490402, -0.018366442993283272, -0.15370063483715057, 0.05571921914815903, 0.18677565455436707, -0.05088476464152336], [0.0252111554145813, 0.07169128954410553, 0.03383711352944374, 0.045200251042842865, 0.017294691875576973, 0.0251912884414196, 0.03735404461622238, -0.003210757626220584, 0.07080310583114624, -0.028107011690735817, -0.05355754494667053, -0.013214606791734695, -0.10008622705936432, -0.0772412046790123, -0.05658502131700516, -0.07888766378164291, -0.004614208824932575, 0.09404543042182922, 0.07913386076688766, 0.046681396663188934, 0.02281142771244049, -0.004098902456462383, 0.009163551032543182, -0.02020777203142643, -0.03966332599520683, -0.02582620456814766, 0.06566493213176727, 0.051604270935058594, -0.07924363017082214, 0.010882511734962463, -0.04127077758312225, -0.04620040953159332, 0.008316868916153908, 0.002373400144279003, 0.011373215354979038], [0.030752193182706833, 0.11452439427375793, -0.06941074877977371, -0.04581258073449135, -0.04463670402765274, 0.0007778856088407338, -0.059504132717847824, 0.035107918083667755, -0.042882487177848816, -0.04675455763936043, 0.06291912496089935, -0.061890821903944016, 0.04684031754732132, 0.05277688801288605, -0.01665210910141468, 0.06489768624305725, 0.038715142756700516, 0.1099376529455185, -0.13867351412773132, -0.015138527378439903, -0.042001932859420776, -0.13681359589099884, -0.046328965574502945, 0.03359847888350487, 0.047019001096487045, -0.02281329408288002, 0.002361078280955553, -0.023045461624860764, -0.01634981855750084, 0.029337385669350624, -0.15735602378845215, 0.13774240016937256, -0.07454027235507965, -0.047667428851127625, -0.06173889338970184], [0.024094432592391968, 0.07098644971847534, -0.027739541605114937, -0.027480674907565117, -0.018974285572767258, 0.0914231389760971, 0.06563256680965424, -0.016808677464723587, 0.038216784596443176, 0.0075456309132277966, -0.08173926174640656, 0.15448950231075287, -0.009995571337640285, 0.046305105090141296, -0.10068824142217636, -0.10936803370714188, -0.10390204936265945, -0.08438880741596222, 0.008584377355873585, -0.016619594767689705, 0.12820929288864136, -0.04758918285369873, 0.03198714181780815, 0.014312203973531723, -0.033085186034440994, -0.019984880462288857, 0.09793002158403397, -0.05715570226311684, -0.024090174585580826, 0.07465747743844986, 0.02480081096291542, 0.031169142574071884, 0.004619148094207048, 0.06769059598445892, 0.06504539400339127], [-0.07207342237234116, -0.01591290533542633, 0.010687134228646755, 0.016851218417286873, 0.03093896247446537, 0.004598128143697977, -0.06983271986246109, 0.015988167375326157, 0.0026655765250325203, 0.04214612767100334, -0.008267577737569809, -0.03028031624853611, 0.0782015472650528, 0.033571600914001465, -0.018283404409885406, -0.04154288396239281, 0.016912581399083138, -0.03832204267382622, 0.03414051607251167, -0.039084699004888535, 0.02692706696689129, 0.010541651397943497, 0.01494157686829567, -0.007617611438035965, -0.019778702408075333, 0.05733183026313782, -0.017681632190942764, -0.03746843338012695, -0.03870989754796028, 0.002732184948399663, -0.028203889727592468, 0.008475172333419323, 0.016583891585469246, 0.044452544301748276, 0.0009734589839354157], [-0.04304063692688942, -0.0670454353094101, 0.0766758918762207, -0.001353649073280394, -0.07966986298561096, 0.08411573618650436, 0.0627448707818985, -0.023711461573839188, -0.02103509195148945, -0.010070825926959515, 0.03206567093729973, 0.022144729271531105, 0.02759450301527977, 0.014836759306490421, 0.12496594339609146, -0.0006957872537896037, -0.01315032597631216, -0.01145794428884983, 0.052284125238657, -0.0742364451289177, -0.1210688129067421, 0.05804162845015526, 0.04335681349039078, -0.061475176364183426, -0.010176094248890877, 0.10523968189954758, 0.05663709342479706, -0.03008228726685047, 0.08595248311758041, -0.09858032315969467, 0.1270563155412674, -0.22297945618629456, 0.012827315367758274, 0.2030491679906845, 0.07273747026920319], [0.09577600657939911, -0.057431597262620926, -0.17563247680664062, 0.03879629448056221, 0.06890320032835007, -0.007016255520284176, -0.08759433776140213, 0.014271876774728298, -0.040047869086265564, -0.041201163083314896, 0.040979623794555664, 0.10863500833511353, 0.12642665207386017, 0.06604387611150742, 0.08210641145706177, -0.028167827054858208, 0.023830140009522438, -0.018668120726943016, -0.11296816170215607, -0.09059551358222961, -0.04663374647498131, 0.006897554267197847, -0.008911780081689358, -0.038077786564826965, 0.03323209285736084, 0.04784467816352844, 0.05494110286235809, -0.048988934606313705, 0.07800842821598053, 0.06977945566177368, 0.016414809972047806, -0.039471130818128586, -0.06928940117359161, -0.008096562698483467, 0.03432857617735863], [-0.05858300253748894, -0.13896526396274567, 0.08952169865369797, -0.010730345733463764, 0.024258054792881012, -0.013681499287486076, 0.052501726895570755, -0.0005922525306232274, -0.04292275384068489, 0.026212917640805244, -0.002885248977690935, 0.05636293813586235, 0.003189153503626585, -0.04674941673874855, 0.056518230587244034, -0.09013437479734421, 0.09378484636545181, 0.04713965952396393, 0.052228283137083054, 0.011444570496678352, -0.023158743977546692, 0.04707742854952812, -0.056005220860242844, -0.019018011167645454, -0.015957986935973167, 0.04708825796842575, -0.01441772561520338, 0.02924584411084652, 0.07898501306772232, -0.017812013626098633, -0.0014775667805224657, -0.19499191641807556, 0.11452934145927429, 0.17023907601833344, 0.01930021494626999], [-0.016284415498375893, 0.02865133062005043, 0.06585268676280975, 0.03641539439558983, 0.05444575846195221, -0.08239738643169403, -0.032141540199518204, 0.03378309682011604, -0.08457780629396439, 0.001393736689351499, 0.033611707389354706, -0.01543454360216856, -0.05978205054998398, -0.023965977132320404, 0.012084100395441055, 0.052897218614816666, 0.07020242512226105, -0.045344509184360504, -0.02712407521903515, 0.011563866399228573, -0.07260571420192719, -0.012948842719197273, -0.006727613974362612, -0.023580443114042282, -0.028881916776299477, -0.012403513304889202, 0.05217878893017769, 0.05943048745393753, 0.018624039366841316, 0.03157002851366997, 0.07023537904024124, 0.059797514230012894, -0.037419792264699936, 0.036925509572029114, 0.007859225384891033], [-0.004904216155409813, -0.06932099163532257, -0.008408325724303722, -0.08601570129394531, -0.04360713064670563, -0.0011634515831246972, 0.042479146271944046, -0.015242153778672218, -0.04536241665482521, 0.0010717182885855436, -0.028436049818992615, -0.045585986226797104, -0.05264178290963173, 0.04153991863131523, 0.0069769457913935184, -0.024066904559731483, 0.048384182155132294, 0.031206171959638596, -0.051960717886686325, -0.062162134796381, 0.030935116112232208, 0.015417110174894333, 0.04211488366127014, 0.006475621834397316, 0.01767895743250847, -0.016994968056678772, -0.049284521490335464, -0.003229778725653887, -0.0242161862552166, -0.055461518466472626, -0.008592084981501102, -0.06718224287033081, 0.044585660099983215, -0.010136165656149387, -0.04250744730234146], [-0.05970122292637825, 0.042600419372320175, 0.24977079033851624, 0.10854707658290863, 0.15961892902851105, -0.08285630494356155, -0.0034795172978192568, -0.08532161265611649, 0.009214297868311405, -0.012280003167688847, 0.15069204568862915, -0.02583148702979088, -0.04956749454140663, 0.1442045122385025, -0.0037313546054065228, 0.01897408626973629, -0.13588827848434448, -0.0418253168463707, 0.05427616834640503, -0.0671011433005333, 0.05340460687875748, -0.05521748214960098, 0.013201126828789711, 0.07750418782234192, -0.0440092459321022, 0.0024281442165374756, -0.0014671070966869593, -0.030217066407203674, 0.15577439963817596, 0.06282912194728851, -0.0063543603755533695, 0.18583862483501434, 0.04635554552078247, 0.03489059582352638, -0.01846345141530037], [0.0775635689496994, 0.01227770559489727, 0.06102251261472702, 0.012907687574625015, -0.07130720466375351, -0.03596388176083565, -0.02995961531996727, 0.12293238192796707, -0.04518673196434975, -0.031614892184734344, 0.03313768282532692, 0.025602126494050026, 0.020206063985824585, 0.04061359912157059, 0.03220074623823166, 0.03880871459841728, -0.05652986094355583, 0.028858009725809097, 0.044510360807180405, -0.021223988384008408, 0.11265438050031662, 0.03446625918149948, 0.059752754867076874, -0.03039523772895336, 0.016854077577590942, 0.03475910425186157, 0.053733039647340775, -0.0307173989713192, 0.07990837842226028, -0.06048395484685898, -0.07554072886705399, 0.09301116317510605, 0.06477033346891403, 0.0470733568072319, -0.0856766551733017], [-0.07740277051925659, -0.09257549047470093, 0.06645065546035767, -0.019919773563742638, 0.017415164038538933, -0.025836043059825897, 0.09152147173881531, -0.008587648160755634, -0.016466747969388962, 0.004695564974099398, -0.032330431044101715, -0.05227900668978691, 0.12262575328350067, -0.018855346366763115, 0.05953719839453697, 0.03326171264052391, 0.08077302575111389, -0.09107237309217453, 0.03372769057750702, 0.027801362797617912, -0.12833920121192932, -0.041463013738393784, 0.025707945227622986, 0.057741787284612656, -0.08935018628835678, 0.20263808965682983, 0.07491829991340637, 0.009722170419991016, -0.11188124865293503, 0.00045705234515480697, 0.05430123209953308, -0.19053949415683746, 0.04099521040916443, 0.18459755182266235, -0.006656537298113108], [-0.059072043746709824, 0.011698916554450989, 0.02197178266942501, 0.0183767881244421, -0.04952607676386833, 0.11888884752988815, -0.046237096190452576, 0.05072173476219177, 0.08884375542402267, 0.054662883281707764, 0.005243866704404354, -0.06316468119621277, -0.020686272531747818, -0.012222168035805225, -0.029137060046195984, -0.05588482692837715, -0.029757142066955566, -0.019282253459095955, 0.07120884954929352, 0.05681972950696945, 0.13322386145591736, -0.007844585925340652, 0.019197773188352585, -0.011379925534129143, -0.029277080669999123, 0.08979400247335434, -0.057655077427625656, -0.03558600693941116, -0.03760085999965668, -0.03196041285991669, -0.08226023614406586, 0.09555494785308838, 0.014365922659635544, 0.06338883191347122, -0.06988386064767838], [0.023440489545464516, -0.0034682145342230797, -0.00562930665910244, 0.0012846544850617647, 0.03730989620089531, 0.023703472688794136, -0.03134152665734291, -0.021874643862247467, -0.01918839104473591, 0.006470562890172005, -0.015989264473319054, -0.004012908786535263, 0.012745858170092106, 0.04944981634616852, 0.08070776611566544, 0.015536203049123287, -0.06559032946825027, -0.09490378201007843, 0.041755303740501404, 0.0005956144887022674, -0.03502393886446953, -0.0002479607064742595, 0.00019142148084938526, -0.01603621244430542, -0.07314226776361465, -0.08662862330675125, 0.08618510514497757, 0.03707157447934151, 0.056846458464860916, 0.05290702357888222, 0.007376557216048241, -0.03140076622366905, 0.040441274642944336, -0.03892063722014427, -0.009121621027588844], [-0.006689164321869612, -0.021904127672314644, 0.020516008138656616, -0.07859639078378677, -0.01226740051060915, 0.02291093021631241, 0.03896433860063553, -0.07978413999080658, -0.048265356570482254, -0.08091959357261658, 0.03861318528652191, 0.1019153892993927, 0.051478493958711624, -0.004567073192447424, -0.037366751581430435, -0.020298175513744354, -0.0694817453622818, -0.053687240928411484, -0.03310661390423775, -0.014082757756114006, 0.08628958463668823, 0.03176099434494972, -0.04152434691786766, 0.003227667883038521, -0.07587049156427383, 0.09361777454614639, 0.04230353981256485, 0.04902426153421402, -0.040605586022138596, -0.08039867132902145, 0.05695591866970062, -0.01240545604377985, 0.0365837886929512, 0.04231922701001167, 0.03203799948096275], [0.00024703555391170084, -0.03875698894262314, 0.05980432406067848, -0.0022813419345766306, 0.017417794093489647, -0.03675967454910278, 0.030797548592090607, -0.029713710770010948, 0.01121011096984148, -0.014826599508523941, -0.005390133708715439, 0.018828757107257843, 0.0307184886187315, 5.714026337955147e-05, 0.03396366536617279, 0.011520861648023129, -0.09119503200054169, 0.015132089145481586, 0.06825518608093262, -0.06837936490774155, 0.02856508269906044, -0.00543033704161644, 0.027002127841114998, -0.054963547736406326, -0.007442512549459934, 0.08191373944282532, 0.03153330832719803, 0.032520174980163574, 0.02823091670870781, -0.0062100873328745365, -0.0065392991527915, -0.06687922775745392, 0.028888896107673645, 0.04421113431453705, 0.0025153637398034334], [0.015306288376450539, 0.00014442761312238872, 0.0021394756622612476, 0.0013401344185695052, 0.009862343780696392, -0.009665568359196186, -0.00025822556926868856, -0.00603546341881156, -0.006206967402249575, 0.00010978212958434597, 0.00010043831571238115, 0.006013816222548485, 0.003927255980670452, 0.0030859271064400673, -0.009382917545735836, -0.005617266986519098, -0.0018346739234402776, 0.00022691582853440195, -0.004271232057362795, -0.002848456148058176, -0.019218936562538147, 0.003031156724318862, 0.0012778161326423287, 0.012118341401219368, -0.005585233680903912, -2.5462071789661422e-05, 0.029377564787864685, 0.0015937829157337546, -0.007112563122063875, 0.0020048751030117273, 0.00854858011007309, -0.00298194563947618, 0.0010342990281060338, 0.007813370786607265, 0.006971338298171759], [-0.05830071493983269, -0.00022253602219279855, 0.0008417830686084926, 0.004180803894996643, -0.11630192399024963, 0.09833980351686478, -0.01845541037619114, -0.02376546524465084, 0.0539141520857811, 0.053551509976387024, -0.03435442969202995, -0.019800815731287003, -0.040509793907403946, 0.022045491263270378, 0.030796976760029793, -0.05932191386818886, -0.014966060407459736, -0.0462401807308197, 0.009981167502701283, 0.05888764187693596, 0.045993685722351074, -0.0227368101477623, -0.03571407124400139, -0.010162211954593658, 0.034532301127910614, 0.04499964416027069, -0.047249507158994675, 0.005274889525026083, -0.055979788303375244, -0.06643786281347275, 0.05040881037712097, 0.02797507494688034, -0.0038668601773679256, 0.06245836243033409, -0.023661084473133087], [0.021501366049051285, 0.06437031179666519, -0.05264042690396309, -0.023783676326274872, -0.029824046418070793, -0.06206068769097328, -0.047666240483522415, 0.013538626953959465, 0.04002667963504791, 0.0018029945204034448, -0.07122398167848587, -0.05678027123212814, 0.01644293963909149, 0.015872739255428314, -0.05249418318271637, 0.08385645598173141, -0.02349344827234745, -0.03129265457391739, -0.08273796737194061, 0.033489566296339035, -0.005677275359630585, -0.02715030498802662, 0.01015571877360344, 0.03169741854071617, 0.0013201090041548014, -0.02046196348965168, 0.024539072066545486, 0.014076723717153072, 0.011922887526452541, -0.04108312353491783, -0.07908598333597183, -0.007066305726766586, 0.05015185475349426, 0.05511981621384621, -0.024371640756726265], [-0.01181825716048479, 0.04306996986269951, 0.12088070064783096, 0.04085319861769676, -0.044052038341760635, 0.12060780078172684, 0.010036258026957512, -0.028982892632484436, -0.024437950924038887, -0.0026637313421815634, 0.057441338896751404, 0.036500826478004456, 0.00014786921383347362, -0.001928203972056508, -0.0033438950777053833, 0.01801462471485138, 0.03434982895851135, 0.0783115029335022, -0.010892467573285103, 0.018670810386538506, -0.0586356595158577, 0.09273983538150787, 0.020261095836758614, 0.04131076857447624, 0.08793967217206955, -0.01065404899418354, -0.04417670890688896, 0.02861211635172367, 0.009500285610556602, 0.055617932230234146, 0.03247835487127304, -0.04032405465841293, -0.006298624444752932, -0.02654668316245079, -0.001204982865601778], [-0.016292208805680275, 0.05346498638391495, 0.020544124767184258, -0.0013655772199854255, 0.013097234070301056, 0.11218884587287903, 0.07357733696699142, -0.053791485726833344, 0.004091797396540642, -0.019909152761101723, 0.04280072823166847, -0.028837166726589203, -0.012097205966711044, -0.023839516565203667, -0.03059201128780842, -0.06436144560575485, 0.05263441056013107, 0.02268093451857567, 0.040783658623695374, 0.0322992242872715, -0.02426297962665558, 0.029187096282839775, -0.04283628985285759, -0.014343803748488426, -0.06750857084989548, -0.13891640305519104, -0.08423735946416855, 0.004940507933497429, -0.04168970510363579, 0.06836258620023727, -0.009716060012578964, 0.07811011373996735, 0.06057843565940857, -0.1599237024784088, 0.004359145183116198], [-0.040289171040058136, -0.09855801612138748, -0.010823378339409828, -0.07357592135667801, 0.023015938699245453, 0.021089987829327583, -0.024712426587939262, 0.02713957615196705, -0.0657489225268364, -0.07104137539863586, 0.04062856733798981, 0.03143122419714928, 0.007876258343458176, -0.033093757927417755, 0.07275897264480591, -0.0005406136624515057, 0.06663431972265244, -0.08706371486186981, -0.08206768333911896, -0.04586952179670334, 0.010890187695622444, -0.026386184617877007, -0.0019440605537965894, 0.049292515963315964, -0.08415680378675461, -0.03072597272694111, -0.034998819231987, 0.009076411835849285, -0.0063706678338348866, 0.08096850663423538, -0.04023904353380203, 0.02614223025739193, -0.06920678913593292, -0.01781252771615982, -0.01140354759991169], [0.0801435261964798, -0.0037998617626726627, 0.04361117631196976, -0.06578724086284637, 0.008774840272963047, -0.059137262403964996, 0.016579780727624893, 0.055373962968587875, -0.010546764358878136, 0.05115640163421631, 0.030191516503691673, -0.05431075394153595, -0.002862629946321249, 0.0439252071082592, 0.021325765177607536, 0.09732221812009811, 0.0813935250043869, 0.052267514169216156, 0.02147809788584709, 0.060786616057157516, 0.08091167360544205, 0.002554250881075859, 0.040991563349962234, 0.019837046042084694, -0.007054449059069157, 0.05947382003068924, 0.018979310989379883, 0.05296770855784416, 0.018140802159905434, 0.07537946105003357, 0.05090555548667908, -0.003593706525862217, -0.08166322857141495, 0.05478176847100258, -0.04959258437156677], [0.05044638365507126, -0.026107825338840485, -0.026283109560608864, 0.03012005239725113, -0.034191787242889404, 0.015043256804347038, 0.08844750374555588, -0.07391918450593948, 0.013690524734556675, -0.04529878497123718, 0.12438799440860748, -0.09563475102186203, 0.10465528815984726, -0.03343242406845093, 6.460904842242599e-05, -0.11036887019872665, -0.10699158906936646, 0.10229302942752838, -0.1484321653842926, 0.028956525027751923, -0.07504473626613617, -0.054620277136564255, 0.06548745930194855, -0.10597332566976547, -0.03926479443907738, -0.07193828374147415, 0.09777561575174332, 0.03082222305238247, 0.07747956365346909, -0.09474647045135498, -0.08764199912548065, -0.0172690749168396, -0.11342158913612366, 0.012592237442731857, -0.0017285224748775363], [-0.027553223073482513, -0.02290242165327072, 0.04680674895644188, 0.03938551992177963, 0.06780819594860077, 0.09810347110033035, 0.06494844704866409, 0.027068788185715675, 0.07203412801027298, 0.0328061506152153, -0.07719847559928894, 0.017635077238082886, 0.06927186995744705, -0.02705705724656582, -0.02302098646759987, -0.023688364773988724, 0.1136416345834732, 0.022728251293301582, 0.0012639046180993319, 0.0046982294879853725, -0.038889020681381226, -0.014945652335882187, 0.004224741831421852, 0.0008812022279016674, -0.10318469256162643, -0.012464032508432865, -0.09349928051233292, 0.013713443651795387, -0.051772646605968475, -0.0616346076130867, 0.015044896863400936, -0.04387696087360382, -0.001678760047070682, -0.021160896867513657, -0.04071009159088135], [-0.11189161241054535, -0.05618377402424812, 0.027773326262831688, 0.019753258675336838, -0.01004184503108263, 0.08406666666269302, -0.021500123664736748, -0.04199577867984772, 0.05354997143149376, -0.037009097635746, -0.07152380049228668, -0.016385497525334358, -0.0027472898364067078, 0.02714540623128414, -0.021807165816426277, 0.08653851598501205, -0.11163190752267838, -0.024642132222652435, 0.05748793110251427, -0.06372175365686417, -0.04645659402012825, 0.020863693207502365, -0.030862830579280853, -0.03630402684211731, -0.0729755386710167, 0.17242297530174255, 0.03894171491265297, 0.10302627086639404, -0.06196150928735733, -0.10310922563076019, 0.10966029763221741, -0.2246171087026596, 0.05762222036719322, 0.3644699156284332, -0.032199934124946594], [0.002091709291562438, -0.10370808839797974, 0.09011740982532501, -0.050641514360904694, -0.0898510217666626, 0.09075779467821121, 0.021413682028651237, -0.012826618738472462, -0.08647165447473526, 0.04924333840608597, -0.09028911590576172, 0.09797846525907516, 0.019833875820040703, 0.027686361223459244, -0.09544534981250763, 0.031202802434563637, -0.0052932258695364, 0.09632617980241776, -0.013858355581760406, 0.013192533515393734, -0.047313809394836426, -0.0008815015316940844, 0.08432918041944504, -0.047587960958480835, -0.0857325866818428, -0.040540553629398346, -0.09038189798593521, 0.09174542874097824, -0.06651759892702103, 0.007181300316005945, -0.16307346522808075, 0.060586199164390564, 0.04115848243236542, 0.021540172398090363, 0.013509558513760567], [0.037758611142635345, 0.036499183624982834, 0.014568409882485867, -0.01624152436852455, -0.02172192931175232, -0.08140553534030914, -0.029867345467209816, 0.0118114547803998, -0.030784960836172104, 0.036402102559804916, 0.0037783877924084663, -0.020178059116005898, -0.0031236414797604084, -0.03308659791946411, -0.011308505199849606, -0.022352369502186775, -0.05398734286427498, -0.01982729323208332, 0.11104485392570496, 0.04057837277650833, 0.03202725201845169, 0.025500359013676643, 0.02173072099685669, 0.01045298483222723, -0.03359119966626167, 0.017243608832359314, -0.025628911331295967, 0.028840413317084312, -0.007067920174449682, 0.002025698544457555, -0.03270712494850159, 0.01929003931581974, 0.04485320299863815, 0.030941130593419075, 0.012226148508489132], [-0.05012955516576767, -0.05744847282767296, 0.07456187903881073, -0.03787278011441231, -0.0032482363749295473, -0.026793183758854866, -0.04662284255027771, -0.004002493340522051, -0.013469966128468513, 0.009618899784982204, -0.02525307796895504, -0.09289706498384476, -0.015498942695558071, 0.015191301703453064, -0.0046034506522119045, 0.10459715873003006, -0.06791281700134277, 0.06900540739297867, 0.03546053543686867, -0.03989211842417717, -0.011125725694000721, -0.02940712310373783, -0.028377102687954903, -0.08186893910169601, 0.05616743862628937, 0.10804097354412079, 0.10944385826587677, 0.009705957025289536, 0.04414914920926094, -0.02803071215748787, 0.023393994197249413, -0.08652770519256592, -0.027620801702141762, 0.07158567011356354, -0.03715292736887932], [0.02673572488129139, -0.03779459372162819, 0.04810382425785065, -0.031051870435476303, -0.03748343139886856, -0.0013858447782695293, 0.025993168354034424, -0.029852522537112236, 0.04117928817868233, 0.03326338529586792, -0.016002967953681946, -0.025174951180815697, -0.030837230384349823, 0.08316558599472046, 0.03117627091705799, -0.023101486265659332, 0.079298235476017, 0.026044271886348724, 0.07775483280420303, -0.0610070638358593, -0.09276356548070908, -0.018587982282042503, -0.028626250103116035, -0.0735948383808136, 0.0009777650702744722, 0.05903131142258644, -0.03815234452486038, 0.05130686238408089, 0.013881380669772625, 0.09163857251405716, -0.014770084992051125, -0.040839117020368576, 0.03679899126291275, -0.05261073634028435, -0.03312904015183449], [0.01508871465921402, -0.03703625872731209, -0.0582730732858181, 0.005067555699497461, 0.010322296060621738, -0.011867104098200798, -0.007393002975732088, -0.014045054093003273, -0.0046541038900613785, 0.14130616188049316, 0.05601530894637108, 0.07071532309055328, 0.056258074939250946, 0.030654776841402054, 0.0159298088401556, 0.06886177510023117, 0.07114766538143158, -0.04299589991569519, 0.047365203499794006, 0.023005129769444466, 0.042366430163383484, -0.07863778620958328, 0.03909313678741455, 0.0040895892307162285, -0.015477064065635204, 0.07610096782445908, -0.035830385982990265, -0.03355636075139046, 0.020374391227960587, -0.012942093424499035, -0.12475358694791794, 0.08873705565929413, 0.03981603682041168, 0.004016348160803318, 0.0014336283784359694], [0.10493692755699158, -0.0802159234881401, -0.11130914837121964, 0.1433233618736267, -0.09881109744310379, -0.06699002534151077, 0.10463792085647583, 0.01171557605266571, 0.04098403453826904, -0.009290901012718678, 0.027772651985287666, 0.08528885990381241, 0.13451474905014038, 0.01874357834458351, 0.025273270905017853, 0.06557673960924149, -0.07904407382011414, -0.12725181877613068, 0.05243966728448868, 0.10007094591856003, -0.03724198043346405, 0.014965279959142208, -0.11169526726007462, 0.010051527991890907, -0.05588139593601227, 0.019274968653917313, -0.04065369814634323, -0.07009720057249069, 0.048294033855199814, 0.09688369184732437, -0.13752569258213043, -0.14015550911426544, -0.059925973415374756, -0.05000106990337372, -0.08757408708333969], [0.021734876558184624, 0.02260880172252655, 0.029222901910543442, 0.01219247467815876, -0.12389620393514633, 0.12496273964643478, -0.015139068476855755, -0.019581157714128494, -0.0821080431342125, -0.0220803115516901, -0.040790751576423645, -0.06030300632119179, 0.13710525631904602, -0.10459902882575989, 0.06833669543266296, -0.009009757079184055, -0.0440126396715641, 0.16107302904129028, 0.076815664768219, -0.12123311311006546, 0.15012843906879425, -0.012170534580945969, -0.008767855353653431, -0.007831950671970844, 0.036856114864349365, 0.023007230833172798, 0.048805221915245056, 0.03245731443166733, -0.08050950616598129, 0.06564872711896896, -0.06687181442975998, -0.051564522087574005, -0.04307929053902626, -0.06384279578924179, -0.005861422512680292], [0.053741902112960815, -0.08698165416717529, 0.08296529948711395, 0.08049046993255615, -0.04728298261761665, -0.03549516573548317, -0.17018143832683563, -0.126506969332695, -0.11397836357355118, -0.030471043661236763, -0.02390926703810692, -0.07823213934898376, 0.10110759735107422, 0.1102742850780487, 0.07439945638179779, 0.12155905365943909, 0.017006507143378258, 0.1198594868183136, -0.005727842915803194, -0.03151342272758484, -0.0006261580274440348, -0.10660814493894577, 0.10823854058980942, -0.058741290122270584, -0.041027721017599106, -0.0029914062470197678, -0.11407309770584106, -0.10933283716440201, 0.05715130642056465, -0.112030029296875, -0.08201393485069275, 0.08008229732513428, -0.07641413807868958, 0.05509693920612335, 0.09577566385269165], [0.016944199800491333, 0.07247231155633926, -0.026057705283164978, 0.009143323637545109, -0.021573638543486595, 0.016972113400697708, -0.024774709716439247, -0.05368661507964134, -0.05948067829012871, -0.07146088033914566, -0.05183211714029312, -0.027397843077778816, -0.025526564568281174, 0.019551817327737808, -0.01689668372273445, 0.03007977269589901, 0.012359182350337505, -0.032887257635593414, -0.009505519643425941, 0.028411628678441048, -0.030536185950040817, 0.039384983479976654, -0.0019692936912178993, 0.013090730644762516, 0.03847116231918335, 0.04372019320726395, -0.008861907757818699, -0.00698256166651845, -0.024329891428351402, -0.02939199097454548, -0.06554512679576874, 0.04939257726073265, 0.01826545223593712, -0.053543899208307266, 0.00043500063475221395], [0.06566713750362396, 0.0208901334553957, 0.0160890594124794, 0.015838857740163803, -0.10523268580436707, 0.13492779433727264, 0.04403987154364586, -0.05444956570863724, -0.003386596217751503, 0.03472470864653587, -0.10813762247562408, 0.0971200168132782, -0.023199710994958878, 0.039016108959913254, -0.022321950644254684, -0.08593809604644775, -0.13597837090492249, -0.03967203572392464, 0.04302327707409859, 0.008975219912827015, 0.017352256923913956, 0.055005915462970734, -0.0333588644862175, -0.02018776722252369, 0.042000316083431244, -0.10956661403179169, -0.003327184822410345, -0.010650111362338066, -0.05144282802939415, 0.054920222610235214, 0.05865687504410744, -0.14863376319408417, 0.042263489216566086, -0.0919720008969307, -0.0387084074318409], [-0.06234801560640335, -0.08110509812831879, 0.06679698824882507, 0.0029312269762158394, -0.09783023595809937, 0.03826456516981125, -0.012817957438528538, 0.01948891393840313, -0.1021517887711525, 0.059963636100292206, -0.028342561796307564, 0.013952442444860935, -0.09537011384963989, 0.06012755259871483, 0.009616808965802193, -0.08879005908966064, 0.1147456094622612, -0.15660491585731506, -0.1106591522693634, -0.03566231206059456, -0.022079424932599068, 0.06073817238211632, 0.08946823328733444, -0.09369269013404846, -0.10895788669586182, 0.0016187767032533884, 0.13510988652706146, 1.3981803022034e-05, -0.11931044608354568, 0.06942769140005112, 0.012975084595382214, 0.11332739889621735, 0.023697759956121445, -0.06966046988964081, 0.055853549391031265], [0.024750638753175735, 0.015392973087728024, 0.06891775876283646, -0.010177002288401127, 0.03039337880909443, 0.04232043772935867, -0.01242428831756115, -0.02020304650068283, 0.01866506226360798, 0.048786140978336334, -0.005404444877058268, 0.010617665946483612, -0.029612179845571518, 0.05282878875732422, -0.016784092411398888, 0.04848611727356911, -0.054046839475631714, 0.015642687678337097, -0.04071376100182533, -0.0343659333884716, -0.07665781676769257, -0.04820805788040161, -0.01925571635365486, 0.008138562552630901, 0.036527182906866074, 0.025049207732081413, 0.07428096234798431, 0.011641052551567554, -0.01790357008576393, 0.054741863161325455, 0.02003421075642109, 0.02914494462311268, -0.059949710965156555, 0.022195298224687576, 0.010560966096818447], [0.036311760544776917, -0.07110689580440521, 0.005816930904984474, 0.00911041535437107, -0.0508321151137352, 0.02237885445356369, 0.02140730246901512, -0.013906831853091717, -0.0046174474991858006, 0.015040759928524494, -0.04026297852396965, 0.11613680422306061, 0.0295516736805439, -0.10531312972307205, -0.06243506073951721, 0.06310005486011505, 0.08114447444677353, 0.018694313243031502, 0.043245669454336166, -0.029109233990311623, -0.04934796318411827, -0.030799057334661484, 0.013626660220324993, -0.0824243426322937, -0.10704892873764038, -0.09663725644350052, 0.01771933026611805, 0.030444983392953873, -0.09841098636388779, 0.022931821644306183, -0.10830536484718323, -0.08081232756376266, -0.004199965391308069, -0.011888394132256508, 0.009240321815013885], [-0.036589059978723526, 0.00456569017842412, -0.041784632951021194, -0.023754555732011795, -0.031998004764318466, 0.03320814296603203, -0.02417551539838314, -0.049855709075927734, -0.051920197904109955, -0.03423107787966728, 0.006545543670654297, -0.012802954763174057, -0.0067914812825620174, 0.016616616398096085, 0.019074823707342148, -0.022306840866804123, 0.009225588291883469, 0.03696054592728615, 0.04504330828785896, 0.014207699336111546, 0.052544787526130676, 0.009895289316773415, 0.032487597316503525, -0.00846732035279274, 0.10174234211444855, -0.08316341042518616, -0.016690673306584358, -0.0091009009629488, 0.010375692509114742, 0.04967658221721649, -0.0030005669686943293, -0.0030810486059635878, -0.02854396402835846, -0.13089831173419952, 0.00238508521579206], [0.028194181621074677, 0.07059706747531891, 0.0878523513674736, 0.08575288206338882, 0.006443568971008062, -0.06903230398893356, 0.03863100707530975, -0.008476285263895988, -0.030282864347100258, -0.0323205441236496, -0.037025369703769684, -0.052310552448034286, -0.0917118489742279, 0.030263742431998253, 0.026461981236934662, 0.012182190082967281, -0.0014467121800407767, -0.04694104567170143, 0.0026023914106190205, -0.03168201446533203, -0.026198476552963257, -0.04278585687279701, -0.03499206528067589, 0.041273199021816254, 0.01173652708530426, 0.0043818834237754345, -0.013748575001955032, 0.04161470755934715, 0.059137750416994095, -0.03434920683503151, 0.01763983443379402, 0.005310893524438143, 0.008248620666563511, -0.008790100924670696, -0.08843697607517242], [0.018492702394723892, -0.012312600389122963, 0.010848615318536758, -0.016572758555412292, -0.04080034792423248, 0.0699312761425972, -0.015050276182591915, -0.0345742329955101, 0.03373369947075844, 0.03209677338600159, -0.03410801663994789, 0.07114571332931519, 0.0008487331797368824, 0.015305150300264359, -0.006655729375779629, 0.02295893244445324, -0.05839655175805092, -0.012078776955604553, -0.029666008427739143, 0.01779746077954769, 0.06357116252183914, 0.00013904953084420413, -0.024302862584590912, -0.022116899490356445, -0.049673765897750854, -0.03548182547092438, -0.0414494164288044, -0.00035044446121901274, -0.03450889512896538, 0.036385394632816315, -0.005772461649030447, -0.04920707270503044, -0.012912154197692871, -0.03257210925221443, 0.014863812364637852], [0.0006060338346287608, 0.04130393639206886, 0.03442515805363655, -0.004609762225300074, -0.03256525844335556, -0.0068762158043682575, -0.002168794395402074, -0.037984829396009445, -0.025054441764950752, 0.03115382418036461, 0.0016400841996073723, -0.02933548018336296, -0.01787356287240982, -0.03563176840543747, -0.005722824949771166, 0.05628421530127525, 0.03585720434784889, 0.007163936737924814, 0.039070360362529755, 0.030147017911076546, 0.013509782962501049, -0.037616051733493805, 0.01653556525707245, 0.0006658791098743677, 0.024058392271399498, 0.029040001332759857, 0.027464916929602623, 0.03749162331223488, 0.019939584657549858, 0.041749898344278336, 0.012351537123322487, -0.005916566122323275, 0.043811965733766556, 0.021067693829536438, -0.009107597172260284], [-0.05157126113772392, -0.06459257006645203, 0.04969704523682594, -0.004408102948218584, -0.013179723173379898, -0.01376730389893055, 0.06337827444076538, -0.0027911202050745487, 0.007637155242264271, -0.05956975743174553, -0.03463584929704666, -0.01102091558277607, -0.024968676269054413, 0.017114169895648956, 0.008218207396566868, -0.0712040439248085, -0.08538179844617844, 0.01666487567126751, 0.06257999688386917, -0.04533807560801506, -0.05579795688390732, 0.03091820701956749, -0.06581941992044449, -0.03116608038544655, -0.06409784406423569, 0.04466327279806137, 0.07290679961442947, 0.01844540610909462, 0.06239205598831177, -0.04500861093401909, 0.016213295981287956, -0.0688212439417839, 0.04854496940970421, 0.16603678464889526, 0.02755347080528736], [-0.018501944839954376, 0.05060342699289322, -0.036831460893154144, -0.033140700310468674, -0.036864206194877625, 0.004823673516511917, 0.0030851447954773903, -0.03306981548666954, -0.00609766598790884, -0.0857841819524765, 0.01961596868932247, -0.007240742444992065, 0.022817337885499, -0.01712951809167862, -0.022163473069667816, -0.03296416997909546, 0.03201263025403023, 0.012077989988029003, 0.001072350307367742, 0.014676922932267189, -0.03456036373972893, 0.01857844740152359, 0.028235463425517082, -0.03880442678928375, 0.014172034338116646, -0.032720230519771576, 0.010706443339586258, -0.018751591444015503, 0.018214263021945953, 0.025897663086652756, -0.048369940370321274, 0.04197900742292404, -0.0017570038326084614, -0.06268800050020218, -0.03164008632302284], [-0.02906067483127117, -0.04527248442173004, -0.0018105548806488514, -0.05554315820336342, -0.02820540964603424, -0.04172484576702118, -0.01852707751095295, 0.021748868748545647, -0.044894903898239136, -0.018950460478663445, -0.022372450679540634, 0.04577882960438728, 0.05596967786550522, 0.01076127216219902, -0.07012790441513062, -0.021937167271971703, -0.07025633007287979, 0.012524883262813091, 0.0412430539727211, -0.020301371812820435, -0.03546472638845444, -0.015127569437026978, 0.0005048888269811869, 0.019916100427508354, 0.004534864332526922, -0.034590817987918854, 0.0031035307329148054, 0.012129598297178745, 0.054496943950653076, -0.0023963255807757378, -0.013468464836478233, -0.03525729477405548, -0.04611682891845703, -0.059193674474954605, 0.013130905106663704], [0.022275006398558617, 0.009743568487465382, 0.022334031760692596, 0.04682820662856102, -0.029175080358982086, 0.09424613416194916, -0.01609501987695694, -0.0005509459879249334, -0.001970654586330056, -0.04739871621131897, -0.015064763836562634, 0.015976708382368088, 0.0214792899787426, 0.08157245814800262, -0.03858426585793495, 0.020656920969486237, 0.0851014107465744, 0.042353998869657516, -0.07708998024463654, -0.07051679491996765, 0.08978435397148132, 0.009338364005088806, 0.037804633378982544, 0.07739348709583282, -0.0656512901186943, 0.0636485144495964, 0.024835975840687752, 0.0420062355697155, -0.06343073397874832, 0.02107611671090126, 0.06545672565698624, 0.005804846994578838, 0.04309443384408951, 0.22611206769943237, 0.03125699236989021], [0.0520758256316185, -0.10129039734601974, 0.014403901994228363, -0.06725238263607025, -0.12721870839595795, -0.005204655230045319, -0.017611006274819374, 0.0379597544670105, -0.02952527068555355, 0.0529533289372921, -0.059473659843206406, -0.035544268786907196, 0.029385056346654892, 0.013725582510232925, 0.07019716501235962, 0.07771290093660355, 0.02261049672961235, -0.1780778020620346, 0.12686195969581604, 0.02333255670964718, 0.14742179214954376, 0.05177665874361992, -0.06089549884200096, -0.11572033911943436, -0.10155601054430008, 0.03574489802122116, -0.07738444954156876, -0.021885482594370842, 0.03605394437909126, -0.015185168944299221, 0.02473987452685833, 0.04879055917263031, 0.05704197660088539, 0.04606787487864494, -0.07796884328126907], [0.024189487099647522, -0.014944884926080704, 0.015311739407479763, -0.07788533717393875, -0.0044183749705553055, 0.003253173315897584, -0.0009475808474235237, 0.04419785365462303, 0.0233214870095253, 0.08522944897413254, -0.032862208783626556, 0.05667402967810631, -0.05199182406067848, -0.028591686859726906, 0.016893938183784485, -0.07627071440219879, 0.050927672535181046, 0.00922058429569006, -0.004467437509447336, 0.0167294479906559, -0.038807883858680725, 0.061216674745082855, -0.002585219219326973, 0.008776676841080189, 0.009484989568591118, 0.07110084593296051, -0.03805902972817421, 0.022571589797735214, -0.036389850080013275, 0.00921624805778265, 0.03635586425662041, -0.12013190239667892, 0.05297088995575905, -0.015571221709251404, 0.00033067638287320733], [0.011478033848106861, -0.023181896656751633, 0.016630567610263824, 0.09259660542011261, 0.08028199523687363, 0.019202761352062225, 0.016977792605757713, 0.036291979253292084, -0.0513572134077549, 0.019988026469945908, -0.0036385487765073776, 0.04297342896461487, 0.025315500795841217, -0.008465944789350033, -0.010784182697534561, 0.0030527091585099697, -0.03306552395224571, -0.03551749885082245, 0.028952693566679955, -0.03225560858845711, -0.10534456372261047, 0.013303701765835285, 0.04915459081530571, 0.053437817841768265, 0.023426827043294907, 0.029592618346214294, 0.07257583737373352, -0.010386329144239426, 0.03659237176179886, 0.06444607675075531, 0.03149779513478279, -0.0033072619698941708, -0.01784803532063961, -0.02219311147928238, 0.036613721400499344]], "b1": [-0.02627980150282383, 0.014094225130975246, 0.06685423105955124, -0.08657632023096085, -0.04265535622835159, 0.04985051602125168, -0.05830072984099388, 0.05170276761054993, 0.09847663342952728, 0.006729957647621632, -0.0717960074543953, 0.11617657542228699, -0.11451590806245804, -0.079535111784935, -0.008282458409667015, -0.017465392127633095, 0.03719751909375191, -0.06531720608472824, 0.04889092966914177, -0.006165865808725357, 0.057154808193445206, -0.029983460903167725, -0.1346917450428009, -0.11441443860530853, 0.0023580968845635653, 0.03529823571443558, -0.0344235859811306, 0.07927893847227097, -0.015885306522250175, 0.03490084409713745, -0.05323035642504692, -0.030177129432559013, -0.032607123255729675, 0.055440329015254974, 0.0764053538441658, -0.042779143899679184, -0.06537170708179474, -0.14759483933448792, -0.006973538082093, 0.025621425360441208, 0.05798616632819176, -0.002540210261940956, 0.01011750940233469, 0.03675591200590134, 0.038569703698158264, -0.10502711683511734, 0.00020665994088631123, -0.05258958041667938, -0.07682982087135315, -0.00040355976670980453, -0.19598542153835297, 0.010211697779595852, -0.21965831518173218, -0.01179248746484518, 0.013506385497748852, -0.10856980830430984, 0.053488489240407944, -0.197158545255661, -0.0076094442047178745, 0.05086420103907585, -0.06304389238357544, -0.04095122590661049, 0.009042274206876755, -0.005236201453953981, -0.03610111027956009, 0.08297009766101837, 0.1095794215798378, 0.006116379518061876, 0.07575850188732147, 0.01964850351214409, 0.014745951630175114, -0.23049642145633698, -0.043182551860809326, 0.03303155675530434, -0.03140199929475784, -0.037613578140735626, 0.06448418647050858, -0.11602464318275452, -0.0061103953048586845, -0.0523117333650589, -0.03480140492320061, -0.005908275954425335, -0.03641897439956665, 0.058097872883081436, -0.07091701030731201, -0.012720023281872272, 0.05289138853549957, 0.023349914699792862, 0.042815376073122025, -0.08091654628515244, 0.005031582899391651, -0.024324841797351837, -0.07110480964183807, -0.04559536278247833, 0.013892964459955692, 0.013394842855632305], "W2": [[0.012911513447761536, 0.02734517492353916, -0.030411692336201668, 0.011381099931895733, -0.03878598287701607, -0.0006084131309762597, 0.004158735275268555, 0.06163111701607704, -0.007499841041862965, -0.009624611586332321, 0.042548615485429764, 0.032698825001716614, 0.001842885510995984, -0.010247817263007164, 0.014329931698739529, 0.039369285106658936, 0.026954427361488342, 0.02267848700284958, 0.006127458997070789, 0.018465645611286163, 0.021609580144286156, 0.010078947991132736, 0.10085847228765488, 0.027347974479198456, 0.038009993731975555, 0.034830596297979355, 0.0032554911449551582, 0.05967646837234497, -0.007793739438056946, 0.03441811725497246, 0.03793330490589142, -0.007785439025610685, 0.02819390408694744, 0.029971791431307793, 0.0541950985789299, 0.045795876532793045, 0.04114776477217674, 0.05742374062538147, -0.014553405344486237, 0.003347942838445306, 0.03437536582350731, -0.04755394160747528, 0.028799163177609444, -0.020668912678956985, -0.008836627006530762, 0.01694214902818203, 0.029330331832170486, -0.033862002193927765, 0.03421110659837723, 0.01655389741063118, -0.020240329205989838, 0.025825046002864838, 0.019792327657341957, 0.019784918054938316, 0.0015166359953582287, 0.04845568910241127, 0.01406204141676426, 0.0037260267417877913, 0.05905039235949516, 0.007993936538696289, -0.02814534492790699, 0.050441350787878036, -0.005061193834990263, 0.02823583595454693, -0.01228206604719162, -0.022106550633907318, -0.02529015764594078, 0.03525051474571228, -0.004507152363657951, -0.01120417844504118, -0.02449112758040428, 0.03191576153039932, 0.047381691634655, 0.006952228490263224, 0.010041480883955956, -0.014389329589903355, -0.020132021978497505, 0.04295220226049423, 0.009329219348728657, 0.04951822757720947, -0.005748222582042217, 0.03744038566946983, 0.03537730127573013, -0.015361173078417778, 0.04389556124806404, -0.005844942759722471, 0.009059067815542221, 0.052344489842653275, -0.004435043316334486, 0.008851055055856705, 0.005303350742906332, -0.012892261147499084, 0.022644005715847015, 0.06660817563533783, -0.0016980803338810802, 0.002551865763962269], [0.011330673471093178, 0.035222843289375305, 0.011841265484690666, 0.03737104311585426, 0.029239008203148842, -0.0020135357044637203, -0.02393754944205284, 0.0065198433585464954, 0.07264526188373566, 0.07802614569664001, -0.011273808777332306, -0.004066117107868195, -0.022494129836559296, -0.000892137351911515, 0.02090466581285, -0.00990983471274376, 0.03446158766746521, -0.0020084157586097717, -0.013552330434322357, -0.02395753376185894, 0.013276789337396622, 0.0060403491370379925, -0.06720004975795746, 0.005108813755214214, 0.02605755254626274, -0.07153807580471039, -0.0023006994742900133, 0.04502548649907112, 0.0020207541529089212, -0.030777914449572563, 0.028319338336586952, 0.018307285383343697, -0.0006976004806347191, -0.008161406964063644, -0.004375941585749388, -0.07913585007190704, -0.019200392067432404, -0.10590881109237671, -0.04185183346271515, -0.05139407888054848, 0.010471789166331291, 0.004988389555364847, -0.0013355175033211708, 0.04795442521572113, -0.021452059969305992, -0.008080082945525646, 0.022915109992027283, 0.003472396405413747, 0.0505194328725338, -0.029726600274443626, -0.03951548784971237, 0.009812760166823864, -0.015680689364671707, 0.005268306937068701, -0.015111781656742096, -0.01978771761059761, -0.0630536749958992, 0.038990721106529236, -0.02116973139345646, -0.03330753743648529, 0.016573423519730568, -0.01708628423511982, 0.007537560071796179, -0.00991477258503437, -0.02051488496363163, 0.008520412258803844, 0.015242593362927437, -0.0021046376787126064, 0.0060804784297943115, -0.023941535502672195, -0.026021156460046768, 0.013078439049422741, 0.013123249635100365, 0.012902002781629562, 0.013362394645810127, -0.008652846328914165, 0.010936521925032139, 0.0498259998857975, 0.015405225567519665, -0.07061871141195297, -0.015050308778882027, 0.024032874032855034, 0.014173941686749458, -0.024739084765315056, 0.006034523714333773, -0.0021638008765876293, 0.015313168987631798, 0.06238571181893349, 0.00711572403088212, -0.030628804117441177, -0.008872496895492077, 0.007551819086074829, -0.047660429030656815, 0.00235489709302783, -0.010953015647828579, 0.018228288739919662], [0.05628104507923126, 0.009453252889215946, 0.05034586787223816, 0.10034623742103577, 0.0028723033610731363, -0.0017227284843102098, -0.06446471810340881, -0.042326200753450394, -0.04337965324521065, 0.02585340104997158, 0.02910524420440197, 0.006805517245084047, 0.10133061558008194, 0.0250689759850502, 0.05208602547645569, -0.0345880426466465, -0.12137527018785477, -0.020378731191158295, -0.018828660249710083, -0.02720494195818901, -0.04169781506061554, -0.014977442100644112, -0.0767708346247673, -0.010761906392872334, 0.08575320243835449, 0.042600393295288086, 0.050362974405288696, -0.03322148695588112, 0.08218439668416977, 0.039996106177568436, -0.05704298987984657, 0.052571434527635574, -0.04210326448082924, 0.007884742692112923, -0.0070064738392829895, 0.030988376587629318, -0.018268773332238197, -0.07234484702348709, -0.03667205199599266, 0.10151370614767075, -0.1190590113401413, 0.06731073558330536, -0.036504343152046204, 0.013080242089927197, -0.010315580293536186, 0.13588136434555054, 0.07121134549379349, 0.014706345275044441, -0.0244500320404768, -0.017184117808938026, 0.15012675523757935, -0.09751152992248535, 0.15103505551815033, -0.05387766286730766, 0.03403867036104202, 0.006747345440089703, -0.1038086861371994, 0.044517043977975845, -0.05702391639351845, 0.02635670267045498, -0.04524664208292961, -0.038156718015670776, -0.011716069653630257, 0.026824478060007095, -0.0030080999713391066, 0.06264034658670425, 0.029878312721848488, -0.04810623452067375, 0.00959072820842266, 0.009674230590462685, 0.06912706047296524, -0.005577811039984226, -0.01120571419596672, -0.04093923419713974, -0.043308109045028687, 0.049660574644804, -0.09342392534017563, 0.012087012641131878, -0.052853185683488846, -0.029315557330846786, 0.046224866062402725, 0.08819406479597092, 0.006454698741436005, -0.05373720824718475, 0.12385589629411697, 0.0926404520869255, -0.054052095860242844, -0.030290236696600914, -0.00223483401350677, 0.04939216002821922, 0.054818056523799896, 0.02955353446304798, -0.007691495586186647, 0.0047467858530581, 0.02081245370209217, -0.019395072013139725], [-0.0032246061600744724, -0.03814426437020302, 0.003945476375520229, -0.12198025733232498, 0.010355878621339798, 0.015527338720858097, 0.01375367771834135, -0.06246289238333702, 0.04103703051805496, 0.09277752041816711, -0.07337961345911026, -0.03304075077176094, -0.0017244318732991815, -0.11327546089887619, 0.024590669199824333, -0.031015021726489067, 0.017454281449317932, -0.06391771137714386, 0.024500951170921326, -0.013731474056839943, 0.0360298678278923, -0.1048380583524704, 0.06905732303857803, -0.06550896912813187, -0.0711030513048172, 0.0009481677552685142, 0.026419831439852715, 0.01020621508359909, -0.030472368001937866, 0.07623807340860367, -0.0038308552466332912, 0.045904796570539474, 0.01727037876844406, -0.0018812838243320584, 0.01656348817050457, -0.011062392964959145, -0.02320566214621067, 0.12198162823915482, 0.009334641508758068, -0.018832331523299217, 0.0009468397474847734, -0.026458894833922386, 0.052325084805488586, -0.07864141464233398, 0.05902591720223427, -0.0759861171245575, 0.0036038111429661512, -0.042241171002388, -0.018222009763121605, 0.04280951991677284, -0.10903198271989822, 0.021748950704932213, -0.06914138048887253, 0.06840722262859344, -0.038625556975603104, 0.054773539304733276, -0.013353561982512474, -0.05998558923602104, 0.03440491110086441, -0.005166932940483093, -0.03941606730222702, 0.04185556247830391, -0.01918138563632965, 0.011357475072145462, 0.01591452956199646, 0.03947671502828598, -0.053699467331171036, 0.03322310745716095, 0.05314871668815613, 0.035282306373119354, -0.025538532063364983, -0.17076361179351807, -0.00946080219000578, 0.01579936034977436, -0.006219710223376751, 0.0080473767593503, 0.08335909992456436, 0.013431127183139324, 0.0035185925662517548, 0.03977359086275101, -0.04729263484477997, 0.035480666905641556, 0.0369863398373127, 0.038314979523420334, 0.00067029899219051, 0.03550795465707779, 0.0030062380246818066, 0.06849216669797897, -0.006810195278376341, 0.013767031021416187, -0.034833766520023346, -0.024241037666797638, -0.000493014114908874, -0.02594364807009697, -0.06512968242168427, 0.008226090110838413], [-0.005471787881106138, -0.022033551707863808, -0.015867015346884727, 0.07386422157287598, 0.10919099301099777, -0.01996200531721115, -0.04505002126097679, 0.014789284206926823, -0.044894538819789886, -0.07712608575820923, 0.04859919473528862, -0.0183012206107378, 0.08245499432086945, 0.05932938680052757, 0.046479012817144394, -0.052014775574207306, 0.011116805486381054, -0.020503023639321327, -0.02941899374127388, 0.033646274358034134, -0.10031147301197052, 0.048115331679582596, -0.13427774608135223, 0.11113770306110382, -0.08338256180286407, 0.03096199780702591, -0.010430960915982723, 0.04847351089119911, -0.04345456510782242, -0.009926891885697842, -0.031577590852975845, 0.030335765331983566, 0.009637407027184963, 0.04386337101459503, -0.05101034790277481, 0.01830696128308773, -0.0009812714997678995, -0.06034544110298157, 0.01867944747209549, -0.03305688872933388, 0.0017727566882967949, 0.014125070534646511, 0.022211546078324318, 0.03998679667711258, 0.03250052407383919, 0.0762433186173439, -0.08919522166252136, 0.04421114921569824, 0.03063339740037918, 0.04109073430299759, 0.04963365197181702, -0.07158435136079788, 0.09492713958024979, 0.011006860062479973, 0.001030527870170772, -0.10740475356578827, -0.052066370844841, -0.0076163215562701225, -0.015756843611598015, -0.02401641756296158, -0.016919519752264023, 0.07202490419149399, 0.0037654810585081577, -0.04537571594119072, 0.03753798454999924, -0.07573089003562927, 0.01815953105688095, -0.020504729822278023, 0.0538804717361927, -0.02899738773703575, 0.018000612035393715, 0.09937600791454315, -0.02016867883503437, 0.04864921048283577, 0.0009882630547508597, 0.022299494594335556, -0.026985177770256996, -0.057116903364658356, 0.03376668691635132, 0.011892148293554783, 0.041181571781635284, -0.02619149163365364, -0.0833725780248642, 0.0037602814845740795, 0.010490254499018192, 0.01148256380110979, -0.06436491757631302, -0.02436923235654831, -0.04359691962599754, -0.008863687515258789, -0.019691916182637215, -0.04164785519242287, 0.09091149270534515, 0.05916312709450722, 0.05223185196518898, -0.05343680456280708], [0.0038000994827598333, 0.010667410679161549, -0.006454105488955975, 0.008879076689481735, -0.010827594436705112, 0.006455438211560249, 0.004501135554164648, 0.008026553317904472, 0.014965074136853218, 0.017775781452655792, 0.011703376658260822, 0.009678156115114689, -0.006310435477644205, -0.004761938005685806, 0.0015553340781480074, 0.006767239887267351, 0.003919343929737806, 0.00604931078851223, 0.000982343452051282, -0.00010278185072820634, 0.01300849486142397, 0.0021141269244253635, 0.007959087379276752, -0.011075953021645546, 0.0048501077108085155, 0.008708476088941097, -0.0014720273902639747, 0.025777580216526985, -0.004583494272083044, 0.00010764137550722808, 0.008535388857126236, -0.0024553309194743633, 0.00814590509980917, -0.0010717140976339579, 0.005767567548900843, -0.00016516757023055106, 0.005890251137316227, 0.005525404587388039, -0.002499468857422471, -0.003564629703760147, 0.007444916293025017, -0.009060832671821117, -0.0007115729385986924, -0.000156561000039801, -0.0013505392707884312, -0.0018458524718880653, 0.0037235093768686056, -0.005476999096572399, 0.007773519027978182, 0.000731990032363683, -0.0032878899946808815, 0.0023489308077841997, 0.0008527090540155768, 0.0014657099964097142, -8.553711086278781e-05, 0.007597338408231735, -0.010794255882501602, 0.008504727855324745, 0.007102640811353922, 4.112046008231118e-05, 0.005076421424746513, 0.015619278885424137, -0.0002576053375378251, 0.005977077875286341, -0.004369039554148912, -0.003727633273229003, -0.005455869249999523, 0.007242813240736723, 0.001154850935563445, -0.00972449965775013, -0.0031191883608698845, 0.027493884786963463, 0.0056007131934165955, 0.0002997447154484689, -0.004197879694402218, -0.00484504783526063, 0.00701201893389225, 0.015334246680140495, -0.010358018800616264, -0.010861974209547043, -0.0026614167727530003, 0.0038154125213623047, 0.007423834875226021, -0.0033849957399070263, 0.006977420765906572, -0.005130239762365818, 0.0018856010865420103, 0.0013078403426334262, -0.00024894819944165647, -0.0024725592229515314, -0.0022456396836787462, -0.0012269157450646162, -0.0031071705743670464, 0.00853817816823721, -0.0010923451045528054, 0.0025367115158587694], [-0.07574304938316345, 0.05832097679376602, 0.008646953850984573, -0.08193241059780121, -0.10780446231365204, 0.059931330382823944, -0.05135892331600189, -0.028515838086605072, -0.01981334201991558, 0.07738497108221054, -0.07382173091173172, -0.07397781312465668, 0.023093508556485176, -0.0784049779176712, 0.009825736284255981, 0.018575774505734444, 0.05646217614412308, 0.040288716554641724, 0.004745637532323599, 0.050186995416879654, 0.03929577395319939, -0.08270885050296783, 0.04696674272418022, -0.039202891290187836, -0.017185553908348083, -0.010621183551847935, 0.024083979427814484, 0.01402543019503355, -0.025920847430825233, 0.013443771749734879, 0.022542081773281097, -0.042953185737133026, 0.040854405611753464, 0.052462875843048096, 0.02998988889157772, 0.028888758271932602, 0.04347149282693863, 0.031064074486494064, 0.030028382316231728, 0.05160181224346161, 0.07232992351055145, -0.02508532628417015, -0.053030334413051605, -0.0760141983628273, 0.07358914613723755, -0.1302308440208435, -0.0408649742603302, -0.00021481233125086874, -0.032104067504405975, 0.032201167196035385, -0.04582063853740692, 0.06827773153781891, -0.09475816786289215, -0.03623855859041214, -0.05921059846878052, -0.038122400641441345, 0.01377321220934391, -0.07670850306749344, 0.06372040510177612, -0.053203828632831573, -0.03837866708636284, -0.012754674069583416, -0.0002550790086388588, 0.004637936130166054, -0.034129660576581955, 0.05977702885866165, -0.018023401498794556, -0.016736870631575584, 0.05061133950948715, 0.08939255028963089, 0.050827328115701675, -0.12040822207927704, -0.04299543797969818, 0.058249764144420624, -0.02960425615310669, -0.04864965006709099, 0.031182851642370224, -0.049376796931028366, 0.057323746383190155, 0.02795945107936859, 0.025966612622141838, 0.04505664110183716, -0.04216291755437851, 0.037752240896224976, -0.01836966723203659, -0.002817163709551096, -0.06685533374547958, 0.020232023671269417, 0.05393679067492485, -0.054981499910354614, -0.03862666338682175, -0.008086816407740116, -0.015186318196356297, -0.020886393263936043, -0.10411595553159714, 0.0012526818318292499], [0.04037141427397728, -0.0693741887807846, -0.06105417758226395, 0.06676340103149414, -0.10255715250968933, -0.015109683386981487, 0.02082028239965439, -0.0652196854352951, -0.03375908359885216, -0.06764696538448334, -0.031092561781406403, 0.011590697802603245, 0.08023206144571304, -0.08512458205223083, -0.03233598172664642, 0.03264747932553291, -0.07156678289175034, 0.10106009244918823, 0.011744721792638302, 0.04070026054978371, 0.010620130226016045, 0.0055872173979878426, 0.12079505622386932, -0.014642787165939808, -0.013529499992728233, 0.05393696948885918, 0.0185688529163599, 0.014604534022510052, -0.040677059441804886, -0.006012336816638708, -0.030890952795743942, -0.02130107767879963, 0.025221381336450577, -0.02341887541115284, 0.05461639538407326, 0.07002989947795868, 0.05893605202436447, 0.16988739371299744, 0.0794118121266365, -0.010111754760146141, 0.0319000668823719, -0.04589933902025223, -0.06257825344800949, -0.06552456319332123, 0.010029999539256096, 0.002164764329791069, 0.01669379509985447, -0.06997738033533096, -0.04427612945437431, 0.018368693068623543, 0.0015505403280258179, 0.10230216383934021, 0.01835528574883938, 0.018624896183609962, 0.01772117055952549, 0.07164500653743744, 0.0640675500035286, 0.066657155752182, 0.03875535726547241, 0.02418486215174198, -0.04580061137676239, -0.006468677893280983, 0.001366852899082005, 0.005248068831861019, -0.016392692923545837, -0.061126451939344406, -0.03602420911192894, 0.014683840796351433, 0.00981241837143898, -0.09153321385383606, -0.016901109367609024, 0.05723581835627556, -0.052474167197942734, -0.02221345156431198, 0.03061048872768879, -0.04582802578806877, 0.021188216283917427, 0.02603108435869217, -0.07080597430467606, 0.06531208753585815, -0.020954320207238197, 0.03098592348396778, 0.018854940310120583, -0.03856196999549866, -0.06207996979355812, -0.014385594055056572, 0.02073070965707302, -0.01258497592061758, -0.018263401463627815, 0.027317695319652557, -0.029278719797730446, -0.04096052050590515, 0.011438371613621712, 0.06622832268476486, -0.025938810780644417, 0.0019496559398248792], [0.02015051245689392, -0.0772995799779892, -0.029030822217464447, 0.05648069456219673, 0.0451643168926239, 0.005543658509850502, -0.054290127009153366, 0.0012364957947283983, -0.061220936477184296, -0.07897650450468063, 0.044075049459934235, -0.102740578353405, 0.09891390800476074, 0.0753924772143364, 0.034779611974954605, -0.062301117926836014, 0.004423694685101509, -0.06203892081975937, -0.037701573222875595, -0.01497411634773016, -0.060692548751831055, 0.004645538982003927, -0.11620309948921204, 0.10358322411775589, 0.07090975344181061, -0.05201921612024307, 0.040279507637023926, -0.019227148965001106, -0.015658175572752953, -0.05234122648835182, -0.036373794078826904, -0.00916722510010004, 0.06591350585222244, 0.011351251974701881, -0.02591112069785595, 0.061061982065439224, -0.016433583572506905, -0.08258414268493652, 0.06261367350816727, 0.10425088554620743, 0.006333845667541027, 0.006099232472479343, 0.018713247030973434, -0.0007835054420866072, -0.015113109722733498, 0.0850401297211647, -0.048814352601766586, 0.011265315115451813, -0.03601290285587311, -0.0013067831750959158, 0.10631714761257172, -0.06332742422819138, 0.04091206565499306, -0.04854006692767143, -0.0017023558029904962, 0.00024497637059539557, -0.07143571972846985, 0.0858309268951416, 0.031232047826051712, -0.008076985366642475, 0.08709906786680222, -0.07305639237165451, 0.019143661484122276, 0.03331538662314415, 0.00752178393304348, -0.026121197268366814, 0.06363340467214584, -0.07001501321792603, -0.04322322458028793, 0.08121630549430847, 0.023039836436510086, 0.017336737364530563, 0.046834275126457214, -0.025861568748950958, 0.07613051682710648, 0.02822655253112316, -0.058213260024785995, -0.016717802733182907, 0.08376217633485794, 0.027886010706424713, -0.003433811478316784, -0.04892592132091522, -0.09406297653913498, 0.006431262008845806, 0.02078869938850403, 0.037549905478954315, -0.04149241000413895, -0.024016553536057472, -0.026890642940998077, 0.030683960765600204, -0.0214840080589056, -0.025702163577079773, 0.09520603716373444, -0.05344302952289581, 0.05443354323506355, -0.006532176863402128], [0.077646404504776, 0.08311253786087036, 0.044687669724226, 0.07935243844985962, -0.11145808547735214, 0.061131931841373444, 0.040995270013809204, 0.09453881531953812, 0.06817814707756042, 0.0018693760503083467, 0.03443446010351181, 0.044502679258584976, 0.022133512422442436, -0.041001804172992706, 0.05251656845211983, 0.053999584168195724, 0.10989636927843094, 0.04103536158800125, 0.006591520272195339, -0.006492647808045149, 0.11060043424367905, 0.010701349005103111, 0.1874999701976776, -0.012629274278879166, -0.029309004545211792, 0.11401814222335815, -0.02827737294137478, 0.06967949867248535, -0.06833270192146301, 0.046707116067409515, 0.027671054005622864, -0.05729609355330467, 0.10136687010526657, -0.025190427899360657, 0.015482398681342602, 0.06668902933597565, 0.02102641947567463, 0.08119756728410721, 0.04743221402168274, -0.03245633840560913, 0.009965317323803902, -0.11797002702951431, 0.046334970742464066, -0.11551742255687714, 0.00837045069783926, 0.05080106481909752, -0.006325879599899054, -0.028968844562768936, 0.019600844010710716, -0.002510193968191743, 0.017772508785128593, 0.03828885406255722, -0.0033620873000472784, -0.004429152701050043, -0.008754090406000614, 0.08484593033790588, -0.06328797340393066, 0.049587883055210114, 0.03791218623518944, 0.055911749601364136, 0.08185624331235886, -0.05133451521396637, 0.03244165703654289, 0.06134353578090668, -0.03273572772741318, -0.020267261192202568, -0.10308345407247543, 0.0391245037317276, 0.024123484268784523, -0.0793137177824974, -0.06271433085203171, 0.0704490914940834, 0.005147117190063, 0.019144250079989433, -0.00806066021323204, -0.05857223644852638, -0.0453052893280983, 0.08832982182502747, -0.006686634384095669, 0.00980908703058958, -0.00772287230938673, 0.07159092277288437, 0.026971276849508286, 0.001438006991520524, 0.06699616461992264, -0.008477289229631424, 0.02652859129011631, 0.08469617366790771, -0.0408182218670845, -0.04484376311302185, -0.01103270798921585, 0.08069463074207306, -0.00312878773547709, 0.023183653131127357, 0.002730883192270994, 0.015681864693760872], [-0.00585306016728282, -0.03176696598529816, -0.015277798287570477, 0.0062321508303284645, 0.0252750925719738, -0.015536236576735973, 0.0066815754398703575, -0.030229510739445686, -0.03688572719693184, -0.05233065038919449, -0.03211718052625656, -0.03231102228164673, -0.020949868485331535, 0.046399809420108795, -0.0037539575714617968, -0.022754956036806107, 0.00892017874866724, -0.009932156652212143, -0.02941225655376911, -0.01203833520412445, 0.01755736768245697, 0.020377052947878838, -0.13335955142974854, -0.01321083027869463, -0.03955593332648277, -0.09869953244924545, -0.014420193620026112, -0.018851496279239655, 0.028464917093515396, -0.055775415152311325, -0.012548021040856838, 0.006391314323991537, -0.08274419605731964, -0.008333317004144192, -0.012400183826684952, -0.1088886484503746, -0.043812211602926254, -0.17961826920509338, -0.036082666367292404, -0.00894775427877903, -0.009827718138694763, 0.021114708855748177, -0.021179627627134323, 0.054041508585214615, -0.02534450963139534, -0.004528700839728117, -0.028760315850377083, 0.009572289884090424, -0.017532210797071457, -0.02920849435031414, 0.027141023427248, 0.024228842929005623, -0.007785825524479151, 0.00920400582253933, -0.024240294471383095, -0.04413455352187157, -0.02514229528605938, 0.04221443831920624, -0.07305004447698593, -0.01037528645247221, 0.012037421576678753, 0.0003911587700713426, 0.008772144094109535, -0.045136284083127975, -0.00035867196856997907, 0.018087876960635185, -0.019379664212465286, 0.011055633425712585, 0.011349881067872047, -0.013116649352014065, 0.013116352260112762, -0.03272513300180435, -0.046900827437639236, -0.019452178850769997, -0.009044250473380089, 0.00876652542501688, -0.03184284269809723, -0.012162684462964535, -0.08252368867397308, -0.03283877298235893, 0.012364240363240242, -0.06391404569149017, 0.004364326596260071, 0.02836001105606556, -0.026232726871967316, -0.04213256016373634, 0.01726299151778221, -0.026259221136569977, -0.0009495951235294342, -0.041243113577365875, -0.011601673439145088, -0.01847764477133751, -0.05690554901957512, -0.03986658528447151, 0.01934671401977539, 0.030978845432400703], [-0.05091720074415207, 0.03025522269308567, 0.04299857094883919, 0.05051693692803383, 0.06269077211618423, -0.026844710111618042, 0.03195587545633316, 0.04135797172784805, 0.0157473087310791, 0.047571685165166855, -0.017214985564351082, -0.013150449842214584, 0.08241391181945801, 0.07625344395637512, -0.003239860525354743, -0.0036333792377263308, -0.0580851249396801, -0.007144067902117968, -0.02759210579097271, -0.03794975206255913, 0.020927786827087402, -0.00703485356643796, -0.08354047685861588, 0.02954840287566185, -0.010861988179385662, -0.03616074100136757, 0.014426566660404205, -0.01229174342006445, 0.002310248324647546, -0.04048367962241173, -0.06014303117990494, 0.025092555209994316, 0.01135479286313057, -0.04516921937465668, 0.03963024541735649, 0.020081622526049614, -0.04783535748720169, 0.0007747393101453781, 0.03341079503297806, -0.008346452377736568, -0.04995308443903923, 0.044493112713098526, 0.06198400259017944, 0.010018986649811268, 0.00035373165155760944, 0.09493321925401688, -0.01684809848666191, 0.053521186113357544, -0.005437623243778944, -0.019796978682279587, 0.12237204611301422, -0.01667989231646061, 0.09952616691589355, -0.0007917791372165084, 0.014282148331403732, -0.09394146502017975, 0.037989817559719086, 0.08045651018619537, 0.0004061494837515056, 0.04045350104570389, -0.04479626193642616, -0.04191408306360245, -0.00890104379504919, 0.005283172708004713, 0.014181082136929035, -0.025784872472286224, 0.028127213940024376, -0.03953944519162178, 0.017613695934414864, 0.05062875151634216, 0.028329482302069664, 0.04102325811982155, 0.019536921754479408, -0.03302367031574249, 0.033080097287893295, -0.0024485650938004255, -0.02889288403093815, -0.03995559364557266, -0.024071110412478447, -0.026560835540294647, 0.02041802741587162, -0.039299070835113525, -0.050169963389635086, -0.01953217200934887, -0.0036605747882276773, 0.05169994756579399, -0.024475911632180214, 0.0028258117381483316, -0.011781543493270874, 0.08587952703237534, 0.018109306693077087, 0.04768456518650055, 0.011240948922932148, -0.015351373702287674, 0.041593343019485474, -0.020553866401314735], [0.0035715331323444843, 0.02176738530397415, -0.028429443016648293, -0.07667627930641174, -0.04592159017920494, -0.00376465474255383, 0.0034324107691645622, 0.03517669811844826, 0.023526649922132492, -0.005869363900274038, -0.0026738333981484175, -0.035025935620069504, 0.003216303652152419, -0.0798514112830162, -0.030824454501271248, 0.01117860246449709, -0.03154003620147705, -0.02548581175506115, 0.033527523279190063, 0.01850529946386814, -0.007800054270774126, -0.018899250775575638, 0.04245617613196373, -0.060879185795784, 0.008525999262928963, -0.02033977024257183, 0.016865888610482216, 0.03849836066365242, -0.026804180815815926, 0.03838476538658142, -0.02866276353597641, 0.00044942094245925546, -0.034172724932432175, -0.007044600322842598, -0.030383847653865814, 0.0028352688532322645, 0.00013444920477923006, 0.09476213902235031, 0.009161459282040596, 0.01105332188308239, 0.04587298631668091, -0.0038824675139039755, -0.0005675180000253022, -0.0598607063293457, 0.016134822741150856, -0.05567765608429909, 0.0017863066168501973, 0.007516777142882347, -0.03317958861589432, 0.006987680681049824, -0.06194930151104927, 0.0007838551537133753, -0.06504867970943451, -0.0019554421305656433, 0.012440212070941925, 0.014555214904248714, 0.015350541099905968, -0.08499950170516968, 0.01719815470278263, 0.043304692953825, -0.05045728385448456, -0.023636341094970703, 0.0082789845764637, 0.010552847757935524, -0.00424005975946784, 0.029878675937652588, 0.0210769921541214, -0.010797563008964062, -0.01317000761628151, 0.04730035737156868, 0.012053130194544792, -0.05547577887773514, 0.007260752841830254, -0.002835147315636277, -0.007060565985739231, -0.024012191221117973, 0.0035535504575818777, -0.026017235592007637, 0.05232825502753258, -0.017853636294603348, -0.0040506054647266865, 0.007496714126318693, -0.027272827923297882, 0.014053859747946262, 0.009927745908498764, -0.00399664556607604, -0.01086038164794445, 0.012216493487358093, 0.0032238876447081566, -0.030801722779870033, -0.007302253041416407, -0.014641276560723782, 0.022742141038179398, -0.05657998099923134, -0.038798652589321136, 0.019690772518515587], [-7.019354961812496e-05, -0.0002592633827589452, 0.00013246633170638233, -0.00029521246324293315, 0.00043189708958379924, -2.1581461169262184e-06, -0.00014562802971340716, 0.0022387271746993065, 0.00018683761300053447, 0.002718981821089983, -1.4772051144973375e-05, -6.070220479159616e-05, -0.0002611853997223079, 9.73164351307787e-05, -5.336248068488203e-05, -8.193519170163199e-05, -0.00021656372700817883, -7.900998025434092e-05, 9.177489846479148e-05, -5.413383769337088e-05, -0.0002648333029355854, -0.0003991222765762359, 0.0003330917388666421, -0.0001679145934758708, -0.0003083528426941484, 0.00042752534500323236, 9.062512981472537e-05, 0.0027670094277709723, 0.00010101059160660952, 0.00016279052942991257, -0.00024807077716104686, 8.212190004996955e-05, 3.850925622828072e-06, 0.00014384943642653525, -8.99881633813493e-05, 0.0006073763943277299, -9.146871161647141e-05, 0.00039285316597670317, 6.851714715594426e-05, 0.00027288738056086004, -2.351994044147432e-05, 0.00021426985040307045, 0.00045483687426894903, 1.9373317627469078e-05, 0.0001407673116773367, 7.321430166484788e-05, -5.7447403378318995e-05, 0.0001953169412445277, -0.0003132008423563093, 3.78751392418053e-05, 8.568847988499328e-05, -0.00021963320614304394, 2.324212073290255e-05, -8.19588422018569e-06, 3.639450005721301e-05, -4.2317882616771385e-05, 0.0015978787560015917, -0.0003576210583560169, -5.157553096069023e-05, 4.7431742132175714e-05, -0.0006237938650883734, 0.00039035570807754993, -1.4723139429406729e-05, -8.161900041159242e-05, -3.4262819099240005e-05, 0.0001810066751204431, 0.0002979096898343414, -0.0001121285604313016, 5.4948552133282647e-05, 0.0002946847816929221, 7.868059765314683e-05, -0.00023222884919960052, -0.00010903983638854697, 4.3746349547291175e-05, 0.0006007477059029043, 0.00020119482360314578, 4.410740075400099e-05, -0.0007383860065601766, -0.00032167634344659746, 0.001091442652978003, -6.690633654216072e-06, 6.9755369622726e-05, -0.00018610322149470448, 5.589906868408434e-05, -0.00018944819748867303, 0.0002508579345885664, -3.140049739158712e-05, 0.0010340784210711718, 2.9218483177828602e-05, 9.810640040086582e-05, 0.00010287161421729252, 0.00038918634527362883, 0.00024438949185423553, -0.0002422950346954167, 0.00011738133616745472, 1.1436611202952918e-05], [-0.0009470799705013633, -0.002159228315576911, 0.0022717162501066923, -0.004051189869642258, 0.0015376703813672066, -0.00045490727643482387, 8.237304427893832e-05, -0.0038681686855852604, -0.0008631021482869983, -0.015880407765507698, 0.0020856405608356, -0.0019386649364605546, -0.003492508316412568, -0.0015854428056627512, -0.00012992827396374196, -0.001197622623294592, 0.0016101500950753689, -0.0016081011854112148, 0.00041111797327175736, -0.00014111078053247184, 0.0006722919060848653, -0.0017789423000067472, 0.002048675436526537, -0.0011718063615262508, -0.0007200995460152626, 0.0007201205007731915, 0.0007328397477976978, 0.01947241649031639, -0.0003543444909155369, 0.0011504654539749026, -0.0014420810621231794, -8.211508975364268e-05, 0.00012530741514638066, -0.0002045151632046327, 0.005355303641408682, 0.004269726108759642, -0.0007546558626927435, 0.0028627675492316484, 0.0008960725972428918, 0.00040476376307196915, -0.0009200607892125845, 0.0013897934695705771, -0.0005950498743914068, -0.0013286217581480742, 0.0004451801360119134, -5.603006138699129e-05, -0.00036854716017842293, 0.0012284866534173489, -0.0016359336441382766, 0.00022437109146267176, -0.0040109227411448956, -0.001484264270402491, -0.0015644298400729895, -0.0008366143447346985, -3.363099676789716e-05, -0.001488656154833734, 0.009779990650713444, 0.00029455829644575715, -0.000170001236256212, 0.0011180354049429297, 0.0028014748822897673, -0.0037400114815682173, 0.00022974368766881526, -0.000702159944921732, 0.00019105289538856596, 0.00028269330505281687, 0.0012387809110805392, -0.0008721037302166224, -0.00019643011910375208, 0.003303541336208582, 8.323063957504928e-05, -0.0037894444540143013, -0.0012332098558545113, 0.0003233109600841999, -0.0035239255521446466, 0.0010306808399036527, 0.0004985950654372573, -0.0027398590464144945, -0.0028378688730299473, 0.00296629942022264, 9.33971386984922e-06, 0.0008992503280751407, -0.0020755925215780735, 0.000922708713915199, -0.001528987311758101, 0.0021673873998224735, -0.0007996918866410851, 0.004763272125273943, 6.0497299273265526e-05, 0.0032282983884215355, 0.0005500466213561594, 0.00020281411707401276, 0.0012379023246467113, -0.00200793263502419, -0.0002454229979775846, -0.00046465176274068654], [0.016580801457166672, 0.004378494806587696, -0.022622348740696907, 0.037714675068855286, -0.040576718747615814, 0.004518568515777588, 0.015077711082994938, 0.046778611838817596, -0.016893941909074783, -0.03695971146225929, -0.00885082595050335, 0.03463321179151535, 0.0056418925523757935, -0.022784726694226265, 0.001993980025872588, 0.030228381976485252, 0.0005999329732730985, 0.030588170513510704, 0.005982463248074055, 0.0019317741971462965, 0.028519336134195328, 0.011041547171771526, 0.07900360226631165, -0.013075706548988819, -0.0009607516112737358, 0.06579064577817917, -0.001638689893297851, -0.0026664144825190306, -0.012654379941523075, 0.005191759672015905, 0.007898125797510147, -0.02091635763645172, 0.014042631722986698, -0.014053089544177055, 0.003489977912977338, 0.027281014248728752, 0.033787891268730164, 0.09505531191825867, 0.019705306738615036, -0.003601871430873871, 0.035349320620298386, -0.025588665157556534, -0.007483749184757471, -0.024973643943667412, -0.0003239288053009659, -0.00208282214589417, 0.0009458941058255732, -0.0346059650182724, -0.0008828248246572912, 0.020540865138173103, 0.03430379927158356, 0.04444167762994766, -0.0034831217490136623, -0.001039147493429482, -0.002820458961650729, 0.030719073489308357, -0.0031926825176924467, 0.022395361214876175, 0.048191316425800323, 0.005398680921643972, 0.0005890909233130515, -0.011073076166212559, -0.002527231117710471, 0.021758072078227997, -0.012585790827870369, -0.023831762373447418, -0.004976868163794279, 0.02552686631679535, -0.0025541516952216625, -0.03778623789548874, -0.007258564233779907, 0.025474118068814278, 0.011437220498919487, 0.001489336951635778, 0.01083336491137743, -0.01262242067605257, 0.02915900945663452, 0.034119587391614914, -0.015635181218385696, 0.016488825902342796, -0.010503449477255344, 0.0028569570276886225, 0.021054985001683235, 0.004368211142718792, 0.0008755040471442044, -0.0032001687213778496, 0.0008502996643073857, 0.0036739499773830175, 0.002391174901276827, 0.010997551493346691, -0.007181293331086636, 0.0003797499230131507, 0.01601996272802353, 0.03943207859992981, -0.01000892836600542, 0.003459377447143197], [-0.012992440722882748, -0.01479889452457428, -0.011386522091925144, 0.043489474803209305, 0.03825283795595169, -0.00809966865926981, -0.01681283861398697, -0.00996124092489481, -0.031856719404459, -0.011950857937335968, 0.011599878780543804, -0.01298157311975956, 0.03861621767282486, 0.02631782367825508, 0.011994315311312675, -0.01394916046410799, 0.011883814819157124, -0.008943499065935612, -0.008808334358036518, -0.002771129598841071, -0.02636014297604561, -0.0010474870214238763, -0.03561720997095108, 0.062365900725126266, -0.004440132528543472, 0.0007389490492641926, 0.005441013723611832, -0.04516496881842613, 0.012198872864246368, -0.0037379199638962746, -0.014109089970588684, 0.011861058883368969, 0.0018397504463791847, 0.004803025163710117, -0.010218153707683086, 0.030256234109401703, -0.008047562092542648, -0.02283194661140442, 0.0014839433133602142, 0.02841636724770069, -0.033678751438856125, 0.026543136686086655, 0.0032427555415779352, 0.01510382816195488, 0.006331663113087416, 0.03653908148407936, -0.01702597364783287, 0.009768618270754814, -0.006551459897309542, 0.005618114490061998, 0.0480818934738636, -0.028540879487991333, 0.04321484640240669, -0.0068429820239543915, 0.0010889022378250957, -0.011107203550636768, -0.02497974783182144, 0.05488061159849167, -0.012715493328869343, -0.010986316949129105, 0.022330978885293007, 0.023860063403844833, 0.0003345237346366048, -0.00027234022854827344, 0.00944080576300621, 0.0016210899921134114, -0.0038636038079857826, -0.019140684977173805, 0.005933624692261219, 0.0014145133318379521, 0.014612387865781784, 0.026987263932824135, -0.001309490529820323, 0.0014551306376233697, 0.020491592586040497, 0.017328718677163124, -0.0033271671272814274, -0.018782055005431175, 0.03894713521003723, 0.00768398167565465, 0.004585356917232275, -0.01223006471991539, -0.016467025503516197, 0.0027097156271338463, 0.0002340749924769625, 0.010383027605712414, -0.004519292153418064, 0.002211859915405512, 0.0016094460152089596, 0.004291082266718149, -0.00013527846022043377, -0.0028676106594502926, 0.03446148708462715, 0.012279774993658066, 0.011248784139752388, -0.011417093686759472], [0.007743274327367544, 0.07975176721811295, 0.029546527191996574, -0.05340031161904335, -0.1553821861743927, 0.05118230730295181, -0.023599671199917793, 0.05583324655890465, -0.024238891899585724, 0.07801826298236847, -0.024732112884521484, 0.03098435327410698, 0.07766035199165344, -0.10069063305854797, 0.007883037440478802, 0.07133296877145767, 0.017501192167401314, -0.01885082945227623, 0.03031863085925579, 0.06559349596500397, 0.04050707817077637, 0.06403463333845139, 0.16200673580169678, -0.030498523265123367, -0.003713933750987053, 0.04801711440086365, 0.03695021942257881, 0.07134491205215454, -0.025587791576981544, 0.07645468413829803, 0.052741311490535736, -0.05628477409482002, 0.03350406140089035, 0.06527723371982574, 0.06696808338165283, 0.10300374776124954, 0.06495297700166702, 0.16481435298919678, 0.06999856233596802, -0.011730492115020752, -0.02577921934425831, -0.06316544860601425, -0.06135210767388344, -0.11433761566877365, 0.06827912479639053, 0.04635259881615639, -0.01609175093472004, -0.11320532858371735, 0.08187642693519592, 0.02976979874074459, 0.029777709394693375, -0.05934101343154907, 0.06694342941045761, 0.028360169380903244, -0.025227298960089684, 0.09388852119445801, -0.020180651918053627, -0.009042319841682911, 0.050563059747219086, 0.0381280854344368, -0.014415833167731762, -0.009654615074396133, -0.023767562583088875, 0.1045679897069931, -0.0645490288734436, -0.0516548790037632, -0.08316171169281006, 0.013158734887838364, -0.003579758107662201, -0.04983140155673027, -0.0842626616358757, 0.08154363185167313, 0.022711316123604774, 0.04117535054683685, -0.06444898247718811, -0.07821005582809448, 0.0769575834274292, 0.037986453622579575, 0.011533133685588837, -0.006325724069029093, -0.029236022382974625, 0.07559260725975037, 0.10451774299144745, -0.019153490662574768, 0.08522113412618637, 0.06233373284339905, 0.015929780900478363, -0.006117355544120073, 0.050684425979852676, -0.013449609279632568, -0.047139979898929596, -0.027193481102585793, -0.040076274424791336, 0.10905832797288895, 0.00912218913435936, -0.018817242234945297], [-0.03220086544752121, -0.052479732781648636, -0.008067830465734005, 0.0480060838162899, 0.0542360357940197, -0.04094177857041359, -0.023810096085071564, -0.04317350313067436, -0.025057436898350716, -0.01504474226385355, -0.005744105204939842, -0.0511648990213871, 0.017297515645623207, 0.02872300148010254, 0.02905784733593464, -0.03792604058980942, 0.03646225854754448, 0.00808645784854889, -0.029700417071580887, 0.012630210258066654, 0.01804979145526886, 0.03386228159070015, -0.05774615705013275, 0.044340066611766815, 0.02505280077457428, 0.02574129030108452, 0.009143885225057602, 0.07280377298593521, 0.027624476701021194, -0.017309075221419334, -0.03167744353413582, 0.015302097424864769, -0.011652574874460697, -0.009431138634681702, -0.04893353208899498, 0.021778615191578865, -0.02756483107805252, -0.01125539280474186, 0.04089924693107605, 0.046654172241687775, -0.05133356153964996, 0.023642851039767265, -0.004439667798578739, 0.03509660065174103, 0.00850038044154644, 0.011291470378637314, -0.03132130205631256, 0.010582605376839638, -0.00025344989262521267, 0.012960996478796005, 0.06037471070885658, -0.027959711849689484, 0.09718361496925354, -0.019326940178871155, 0.027709411457180977, -0.02267414703965187, -0.0011752528371289372, 0.08852235227823257, -0.02333316020667553, -0.031031589955091476, 0.03691861033439636, 0.019163399934768677, 0.007502712309360504, 0.015397751703858376, 0.02997007593512535, 0.002760044764727354, 0.02916526608169079, -0.003320675576105714, -0.017068736255168915, 0.02384733036160469, 0.004428883548825979, 0.006877092644572258, -0.011839284561574459, -0.011092517524957657, 0.04110831022262573, 0.008074447512626648, -0.03816477954387665, -0.009463922120630741, 0.04832044616341591, 0.004009439144283533, 0.0023350664414465427, 0.00381217198446393, -0.013885597698390484, -0.02083035558462143, -0.014150949195027351, 0.03482973203063011, 0.011699291877448559, 0.039777614176273346, -0.012850819155573845, -0.004340329207479954, 0.006637711077928543, 0.015487737022340298, 0.04040372371673584, -0.001591829932294786, -0.010199767537415028, -0.02505333721637726], [-0.02236262336373329, 0.0033536169212311506, 0.04209262505173683, -0.09118940681219101, -0.09538213908672333, 0.04225382208824158, -0.006327259819954634, 0.021425295621156693, -0.014017808251082897, 0.0375894159078598, 0.020323213189840317, 0.013248451985418797, -0.07954791933298111, -0.04019024223089218, -0.061686936765909195, -0.010255237109959126, -0.07540655136108398, -0.021349165588617325, 0.07741853594779968, 0.05607543885707855, 0.04078344628214836, -0.07165272533893585, 0.13049989938735962, -0.1252402663230896, -0.03728775307536125, 0.11020567268133163, 0.0905207097530365, -0.005645635537803173, 0.019648808985948563, 0.023434633389115334, -0.020843643695116043, -0.0357649102807045, -0.047497205436229706, -0.04887693375349045, -0.013057416304945946, 0.04278469830751419, -0.026485545560717583, 0.08533564954996109, -0.01886533945798874, -0.011288915760815144, -0.024977663531899452, -0.01776277832686901, 0.05456683412194252, -0.02338000200688839, -0.02841944247484207, -0.02241542935371399, 0.002046824200078845, -0.006985409650951624, -0.048601485788822174, 0.05266634747385979, -0.10366012156009674, 0.09579449146986008, -0.10874859988689423, 0.015325858257710934, 0.02769174799323082, -0.003914888948202133, -0.039256975054740906, -0.17831777036190033, 0.042169664055109024, 0.09993993490934372, -0.059549134224653244, -0.012639024294912815, 0.0005721846828237176, 0.007860536687076092, 0.0344354547560215, 0.04038865864276886, 0.014278299175202847, 0.030301889404654503, 0.0005319275660440326, 0.016617007553577423, 0.04078163951635361, -0.19335514307022095, 0.05164433270692825, -0.005101548507809639, -0.06540769338607788, -0.027913004159927368, 0.07746417820453644, 0.004188045859336853, 0.07792322337627411, 0.0718327984213829, -0.05539799854159355, 0.1379653811454773, 0.06837420165538788, 0.04725716635584831, 0.028374291956424713, -0.024419425055384636, 0.01088821142911911, 0.02125958539545536, 0.03817052021622658, -0.0038410115521401167, -0.025081634521484375, 1.1528371942404192e-05, 0.03960319608449936, -0.05196450278162956, -0.005906914826482534, -0.003536070231348276], [-0.047172911465168, 0.007246822118759155, -0.03176400810480118, 0.001298840157687664, 0.05426976457238197, -0.027684466913342476, 5.80960295337718e-05, -0.02321084775030613, -0.09471158683300018, 0.04373028874397278, -0.057364560663700104, -0.10871750116348267, 0.0020434611942619085, 0.04823284596204758, -0.0016031911363825202, -0.034417226910591125, 0.08365534991025925, -0.06604265421628952, -0.05876659229397774, -0.003297567367553711, -0.0001315949484705925, 0.03716667741537094, -0.08605685830116272, 0.04889823868870735, -0.06314031779766083, 0.04341923072934151, 0.030496088787913322, 0.0025635778438299894, 0.001162810018286109, -0.017764095216989517, -0.0543958880007267, -0.032634004950523376, 0.08626922219991684, 0.012311904691159725, -0.07261655479669571, 0.09093118458986282, -0.028122812509536743, -0.04294871911406517, 0.032185956835746765, 0.030725104734301567, -5.411098391050473e-05, 0.05711112171411514, 0.053823571652173996, -0.012718151323497295, -0.009571137838065624, 0.06781424582004547, -0.08292362093925476, -0.00039902247954159975, -0.05374009534716606, 0.0024589765816926956, 0.0324401892721653, -0.06609000265598297, 0.004259674809873104, -0.03515630587935448, -0.03748353570699692, -0.08357340842485428, -0.06336056441068649, 0.09066753089427948, -0.008436935022473335, -0.006255961023271084, 0.0326717346906662, 0.07260791212320328, -0.027213197201490402, 0.004422287922352552, 0.05259576439857483, -0.06792392581701279, -0.005799046717584133, 0.007907846942543983, -0.02961520478129387, 0.015540224500000477, 0.05201934650540352, 0.032412007451057434, -0.07722904533147812, 0.011086818762123585, 0.08521794527769089, 0.05355086922645569, -0.0023331400007009506, -0.03927123174071312, 0.09002048522233963, 0.07560852915048599, 0.004540818743407726, -0.016313109546899796, -0.08653111755847931, 0.02521582506597042, -0.022211048752069473, 0.0241945032030344, -0.02918688952922821, -0.04928024113178253, 0.00611952506005764, 0.0707310140132904, -0.048573944717645645, 0.05659377947449684, 0.011342963203787804, -0.0007614592905156314, -0.03852232173085213, -0.028784846886992455], [-0.01724839396774769, 0.008942963555455208, 0.021837083622813225, -0.12862591445446014, -0.027561785653233528, 0.04018183797597885, -0.03901982679963112, 0.04824265465140343, -0.07650787383317947, 0.006823978386819363, -0.08838777989149094, 0.07347257435321808, -0.14316268265247345, -0.1053839847445488, -0.022215763106942177, -0.017702732235193253, 0.02601701207458973, 0.04886176064610481, 0.07527480274438858, 0.0019572603050619364, 0.055126260966062546, -0.162429079413414, 0.10154744237661362, -0.07798605412244797, -0.060774486511945724, 0.08245846629142761, 0.029860438778996468, -0.040872830897569656, 0.06366939097642899, 0.1210155114531517, 0.07110510021448135, 0.06290848553180695, -0.020740197971463203, 0.026972567662596703, -0.062453754246234894, 0.011034390889108181, 0.006547865457832813, 0.11002889275550842, 0.0595601461827755, -0.06861922889947891, 0.10188974440097809, -0.04089777171611786, 0.04468929022550583, -0.11598434299230576, 0.12283989042043686, -0.01945325918495655, 0.05628431960940361, 0.010396823287010193, 0.04472867771983147, 0.004951604641973972, -0.18351544439792633, 0.015818247571587563, -0.13082514703273773, 0.07648487389087677, -0.008821925148367882, 0.0837189182639122, 0.030244464054703712, -0.15355022251605988, 0.023038484156131744, -0.040775567293167114, -0.04104596748948097, -0.06743649393320084, 0.0172965656965971, -0.0075570824556052685, 0.02458750084042549, 0.031185904517769814, -0.0626884400844574, 0.01889130473136902, 0.06723557412624359, -0.009343849495053291, 0.07095655053853989, -0.21489442884922028, 0.0919615775346756, 0.04928159341216087, -0.056000079959630966, 0.037551287561655045, 0.02113989181816578, -0.000863278575707227, 0.035729579627513885, -0.006584873888641596, -0.04932819679379463, 0.05644465237855911, 0.036384887993335724, 0.0048207989893853664, 0.057196930050849915, -0.10439370572566986, 0.09930931031703949, 0.0002945736050605774, 0.01976706273853779, -0.07559620589017868, -0.0599924698472023, -0.09848187118768692, 0.03006698377430439, 0.04753737524151802, -0.0072031887248158455, 0.036280322819948196], [-0.08889107406139374, 0.031834110617637634, 0.049485113471746445, -0.12576265633106232, -0.05813165754079819, -0.027281930670142174, 0.009392386302351952, -0.00925744604319334, 0.06030307710170746, -0.0315183624625206, 0.0010162611724808812, -0.06675295531749725, -0.06967353075742722, -0.06602209806442261, 0.02968340739607811, 0.008361611515283585, 0.005374550353735685, 0.01315728947520256, 0.03856930509209633, 0.018977290019392967, 0.013059072196483612, -0.0403003916144371, 0.0540880523622036, -0.04778860881924629, -0.06670382618904114, 0.012937627732753754, -0.0058493902906775475, 0.022496819496154785, 0.020113719627261162, 0.04435333237051964, 0.03997635841369629, -0.06878674775362015, 0.06959886848926544, -0.007548848632723093, -0.04293657839298248, 0.020881693810224533, 0.028786437585949898, 0.06390661001205444, 0.04160487279295921, -0.012529692612588406, -0.03170305863022804, 0.013344032689929008, -0.0075781992636621, -0.06625466048717499, -0.0280512273311615, -0.043590810149908066, -0.013382037170231342, -0.037130363285541534, -0.05491715297102928, -0.0003796266973949969, -0.09801767021417618, 0.05653045326471329, -0.06357396394014359, 0.01815589889883995, -0.011289933696389198, -0.10184402763843536, 0.09445859491825104, -0.09810274839401245, 0.002105988096445799, 0.035494979470968246, 0.054589834064245224, -0.06776998937129974, 0.0205936711281538, -0.03242393955588341, 0.011064893566071987, -0.02791895717382431, -0.04571237042546272, -0.030650731176137924, 0.07135923206806183, 0.02021515741944313, -0.011182744055986404, -0.08066923916339874, -0.024672789499163628, 0.014640098437666893, 0.026479670777916908, -0.005061312112957239, 0.0687660351395607, -0.029253879562020302, 0.08063098043203354, 0.04969785362482071, 0.0244387686252594, -0.01326981745660305, 0.010453674010932446, -0.02491002157330513, -0.026302406564354897, -0.025482814759016037, -0.0044915745966136456, -0.029951879754662514, 0.043176308274269104, -0.06623441725969315, 0.01740187592804432, 0.0011388382408767939, 0.02883404679596424, 0.05421512946486473, -0.04139911010861397, -0.019414830952882767], [0.04793426766991615, -0.010952225886285305, -0.016336960718035698, 0.052122410386800766, -0.06486374884843826, 0.02631877362728119, -0.053334977477788925, 0.005719493143260479, 0.09690064936876297, -0.046028461307287216, -0.02301729843020439, 0.010223006829619408, 0.07121798396110535, -0.03444759175181389, 0.004623326472938061, 0.1094484031200409, 0.07124683260917664, 0.008144083432853222, -0.0490921251475811, 0.01815839298069477, 0.095906563103199, 0.0783768892288208, 0.13535459339618683, 0.09331071376800537, 0.009812148287892342, 0.07661083340644836, -0.007524773012846708, -0.06789783388376236, -0.04265117645263672, -0.013264545239508152, 0.05059337243437767, -0.0496186725795269, 0.0979996994137764, -0.02429874986410141, -0.07980150729417801, 0.009155922569334507, 0.08437228202819824, 0.1675293892621994, -0.020582495257258415, 0.039871565997600555, 0.06644539535045624, 0.0005659485468640924, -0.014585832133889198, -0.02126755565404892, -0.06747370958328247, -0.03138329088687897, 0.04077262431383133, -0.004721818026155233, -0.005661728326231241, 0.0188584066927433, -0.054966144263744354, -0.004782566335052252, -0.00516707357019186, 0.029604880139231682, -0.06536909192800522, -0.039150871336460114, -0.001377448090352118, -0.018845606595277786, 0.11256496608257294, -0.026825258508324623, -0.057559311389923096, -0.05072834715247154, -0.006186229642480612, 0.03741437941789627, 0.015228289179503918, 0.003482794389128685, 0.04654806852340698, 0.01670672558248043, 0.011650420725345612, 0.009248511865735054, 0.03871096670627594, 0.008923422545194626, 0.05409751087427139, 0.06213454529643059, -0.022633284330368042, -0.041247203946113586, 0.04095974192023277, 0.09508012980222702, -0.012514384463429451, 0.032609887421131134, -0.023120708763599396, 0.05759705603122711, 0.036092210561037064, -0.029401790350675583, -0.040152035653591156, -0.04916871339082718, 0.00024848541943356395, 0.005031061824411154, 0.02479511685669422, 0.0269536841660738, -0.06185092031955719, -0.03333057463169098, 0.03439748287200928, -0.016369204968214035, 0.04920661449432373, -0.015020661056041718], [-0.07623825967311859, 0.023535124957561493, -0.03685590997338295, 0.07334302365779877, 0.042735639959573746, -0.09100618958473206, -0.04109372943639755, -0.04400143399834633, 0.018715698271989822, -0.032645344734191895, 0.05703208968043327, -0.14984780550003052, 0.12070339173078537, 0.03761131316423416, -0.0404147207736969, -0.06716427206993103, -0.020410193130373955, -0.0839485228061676, -0.07062556594610214, -0.03599806874990463, -0.09194587171077728, 0.06753416359424591, -0.16647988557815552, 0.10051849484443665, 0.04049030318856239, -0.030915409326553345, 0.03811369463801384, -0.06594964116811752, 0.07836349308490753, -0.06902483105659485, -0.02728981524705887, -0.0276542566716671, 0.009735776111483574, -0.05522248521447182, -0.032906387001276016, 0.042919840663671494, -0.01638195849955082, -0.07461508363485336, 0.012250831350684166, 0.08658602088689804, -0.13814152777194977, 0.08252686262130737, 0.08690526336431503, 0.01072907168418169, 0.01820455677807331, 0.09774512052536011, -0.09089776128530502, 0.061452824622392654, -0.06966809183359146, 0.005715410690754652, 0.05384388938546181, -0.02499229647219181, 0.11584780365228653, -0.08105265349149704, 0.016918189823627472, -0.047129083424806595, 0.0028095399029552937, 0.12278569489717484, -0.04965725541114807, -0.08904466778039932, 0.07693861424922943, 0.06199997663497925, -0.004002596251666546, -0.004062657710164785, 0.038533009588718414, -0.02709253318607807, -0.012422019615769386, -0.08386646211147308, -0.038723137229681015, 0.0012763581471517682, 0.04467783868312836, 0.11929593235254288, -0.06480610370635986, 0.006249628029763699, 0.08302692323923111, 0.052070558071136475, -0.0786122977733612, -0.050061218440532684, 0.052822500467300415, 0.05911790579557419, 0.0005678792367689312, -0.008749691769480705, -0.08734740316867828, -0.01465881709009409, -0.05427556112408638, 0.06758014857769012, -0.06072444096207619, 0.03221483901143074, -0.021897047758102417, 0.06129666790366173, 0.04473237693309784, -0.043223898857831955, 0.055790990591049194, 0.02528526447713375, 0.06443562358617783, -0.06843879818916321], [-0.04362839460372925, 0.035635899752378464, 0.048240456730127335, -0.098495252430439, -0.013816903345286846, -0.026050155982375145, 0.00448242062702775, 0.02842467650771141, 0.022931070998311043, -0.007983127608895302, -0.03320736810564995, -0.01123056374490261, -0.0038107549771666527, -0.10048247873783112, -0.009624122641980648, -0.014771823771297932, -0.04206182062625885, 0.0013342428719624877, -0.004978804849088192, -0.007146264426410198, -0.007506407331675291, -0.008744552731513977, 0.025704864412546158, -0.020931893959641457, 0.0156606063246727, -0.029644742608070374, 0.010451859794557095, -0.02170787937939167, -0.015528866089880466, 0.031047336757183075, -0.009470412507653236, -0.009395329281687737, 0.014304585754871368, 0.02088213711977005, -0.013363019563257694, 0.038432009518146515, 0.0070608616806566715, 0.06989046931266785, -0.005251762922853231, -0.018850984051823616, 0.0377991646528244, -0.02946764975786209, 0.025454841554164886, -0.061411336064338684, 0.044980015605688095, -0.041737958788871765, -0.024588949978351593, -0.013686218298971653, -0.003719503525644541, 0.015203502029180527, -0.0717724859714508, 0.03925642743706703, -0.059404484927654266, 0.0008445304119959474, -0.01941956952214241, -0.03308296576142311, 0.06564360111951828, -0.09284645318984985, 0.009957155212759972, 0.0184476301074028, 0.04153569042682648, -0.04348652437329292, 0.00201070262119174, -0.019818127155303955, 0.01912675052881241, -0.028326377272605896, 0.0324522964656353, 0.003925720229744911, 0.0164358951151371, 0.07260118424892426, -0.024446483701467514, -0.047702912241220474, 0.007886955514550209, -0.010044955648481846, 0.018333477899432182, 0.002746215555816889, 0.06318610906600952, -0.06029561534523964, 0.029502935707569122, -0.013479864224791527, -0.018606036901474, 0.022921057417988777, -0.023138822987675667, -0.00496748648583889, -0.051962923258543015, -0.006644835229963064, -0.01535382866859436, -0.0442049466073513, 0.022729869931936264, 0.005018336698412895, 0.016878316178917885, -7.899086631368846e-05, 0.00986937154084444, -0.0548856295645237, -0.04361363500356674, -0.010405246168375015], [0.037800803780555725, 0.015351242385804653, -0.07089201360940933, 0.08769481629133224, 0.013004827313125134, 0.06492643058300018, 0.010279133915901184, 0.08225321769714355, 0.039192184805870056, 0.110683873295784, 0.07509972155094147, 0.07486503571271896, -0.05828842148184776, 0.02066253498196602, -0.05810031667351723, 0.05780238285660744, -0.0038615588564425707, 0.060320910066366196, 0.03566880524158478, -0.01137689221650362, -0.029025068506598473, 0.017010901123285294, 0.10187095403671265, -0.07087136805057526, 0.00804386381059885, 0.12417811900377274, 0.017432697117328644, 0.08924663066864014, -0.0828292965888977, -0.010114970616996288, -0.00396893871948123, -0.012988610193133354, 0.052195046097040176, 0.06627363711595535, 0.10167054831981659, 0.014132278971374035, 0.09928353875875473, 0.15343493223190308, -0.049749135971069336, -0.06328613311052322, 0.04323329031467438, -0.09392599761486053, -0.044815730303525925, 0.0009630619897507131, -0.023323627188801765, 4.011702549178153e-05, -0.03149571269750595, 0.011830558069050312, 0.10233119875192642, -0.012845241464674473, -0.06343431770801544, 0.054644450545310974, -0.024813825264573097, 0.01300516352057457, 0.07695870101451874, 0.11659138649702072, 0.07667341828346252, 0.007536860182881355, 0.09087216854095459, 0.02918982319533825, 0.07137318700551987, 0.026089921593666077, -0.016602016985416412, 0.056551408022642136, -0.0477018803358078, -0.06494756042957306, -0.04807816445827484, -0.021381180733442307, -0.04723024740815163, 0.005135869141668081, -0.042171478271484375, 0.05054345354437828, 0.11081627011299133, 0.017918838188052177, -0.06259076297283173, -0.013728245161473751, -0.04845789819955826, 0.128506600856781, 0.0209095049649477, -0.027897518128156662, -0.07217998802661896, 0.0068624066188931465, -0.02206774801015854, -0.036253247410058975, 0.099563829600811, -0.03612883761525154, -0.03655123710632324, -0.017919747158885002, -0.04314479976892471, 0.034875061362981796, -0.030387407168745995, 0.01701892353594303, 0.04722536727786064, 0.0719204917550087, -0.01791459508240223, 0.035262033343315125], [0.05327155813574791, 0.05226729065179825, -0.06012982875108719, -0.02991926483809948, -0.07648841291666031, 0.057684753090143204, 0.06586304306983948, 0.018718300387263298, 0.06898577511310577, 0.03862111270427704, 0.05643092840909958, 0.061278555542230606, 0.08334358036518097, -0.012953182682394981, 0.06622043997049332, 0.011844306252896786, -0.013106086291372776, 0.010406008921563625, 0.029267169535160065, 0.061386190354824066, 0.012119710445404053, 0.03582512587308884, 0.035304050892591476, 0.05002230405807495, -0.0034498926252126694, -0.02696775645017624, -0.03197650611400604, 0.028066275641322136, 0.05405859276652336, -0.035294175148010254, 0.006605550646781921, -0.008169928565621376, -0.03856058046221733, 0.03495914861559868, 0.01214989461004734, 0.024045605212450027, 0.04047640413045883, -0.04266876354813576, 0.007420026697218418, -0.058818064630031586, 0.0031127326656132936, -0.018217455595731735, -0.048900306224823, -0.06884901225566864, 0.04545317590236664, -0.02551097609102726, 0.0660422071814537, -0.032512206584215164, 0.020708860829472542, -0.02853267826139927, -0.06823328882455826, 0.0489061176776886, 0.012643991969525814, 0.008150485344231129, 0.06968196481466293, 0.04094139114022255, 0.07242880761623383, 0.01593131572008133, 0.015855280682444572, -0.02452382817864418, 0.06808088719844818, -0.0014548051403835416, 0.010705542750656605, 0.054087404161691666, 0.009331059642136097, -0.036227237433195114, -0.04457169026136398, 0.037006694823503494, 0.024484295397996902, -0.05174516141414642, 0.021691331639885902, 0.09104667603969574, 0.051545847207307816, -0.039320290088653564, 0.07872702181339264, 0.012095616199076176, -0.028243839740753174, 0.0487523078918457, 0.08416587859392166, -0.02024240791797638, -0.006209265906363726, 0.012262394651770592, 0.004868049640208483, -0.013359695672988892, -0.007111674174666405, -0.023290468379855156, 0.0005497218808159232, 0.06425762176513672, -0.028934301808476448, -0.012430286034941673, 0.0035119501408189535, 0.015974197536706924, 0.028695987537503242, 0.0035483750980347395, 0.014352579601109028, 0.010248817503452301], [-0.07227081060409546, -0.0886392891407013, -0.011394894681870937, 0.019810419529676437, 0.11454169452190399, 0.008592082187533379, -0.06565231084823608, -0.05932091549038887, -0.05363606661558151, -0.05791257694363594, 0.07677022367715836, -0.07116370648145676, 0.017311936244368553, 0.07658041268587112, 0.04556984454393387, -0.07825496047735214, 0.07361003011465073, -0.061732642352581024, -0.020075639709830284, 0.023308418691158295, -0.08153802156448364, 0.046381887048482895, -0.07618194818496704, 0.12248946726322174, -0.06089164316654205, 0.02196994051337242, -0.02641923539340496, 0.0717286467552185, 0.0571126863360405, 0.0025068779941648245, -0.03819333761930466, 0.007700145244598389, -0.06135738268494606, -0.02424907125532627, 0.03945352882146835, 0.07943350821733475, -0.008287500590085983, -0.08147090673446655, 0.09229382127523422, 0.0913500040769577, -0.09741068631410599, 0.03283524513244629, -0.018498536199331284, -0.04055868089199066, 0.033139489591121674, 0.0020716486033052206, -0.0801275372505188, 0.062170207500457764, 0.05181332305073738, 0.05167153477668762, 0.056263748556375504, -0.08840888738632202, 0.010248089209198952, -0.006381714716553688, 0.021785495802760124, -0.036335598677396774, 0.012987515889108181, 0.0999174639582634, -0.003929352853447199, -0.021895142272114754, 0.025447329506278038, -0.01017217431217432, 0.009227067232131958, 0.03364904224872589, 0.02617918699979782, 0.048213452100753784, 0.024554304778575897, -0.02816447615623474, 0.06433341652154922, -0.04145204648375511, 0.011299424804747105, 0.08690419048070908, 0.012894200161099434, 0.00979963131248951, -0.03464362770318985, 0.03882386535406113, -0.04875657334923744, -0.027155764400959015, 0.021842632442712784, 0.0702018290758133, -0.0365409329533577, -0.058603685349226, -0.08999522775411606, 0.0005495322402566671, -0.016303883865475655, 0.05164513364434242, 0.0016224145656451583, 0.044821228832006454, 0.041896965354681015, -0.013014028780162334, -0.032766759395599365, -0.0634097307920456, 0.03571237251162529, -0.04703575000166893, -0.05788644403219223, -0.064529649913311], [-0.10200712829828262, -0.08119324594736099, 0.05114484950900078, 0.022475849837064743, 0.09104271978139877, -0.02148890122771263, 0.03705569729208946, -0.04284782335162163, -0.026973236352205276, -0.016529954969882965, -0.03389626368880272, -0.01905710995197296, 0.009290257468819618, 0.024404775351285934, 0.06647922098636627, 0.0032398123294115067, 0.06052190810441971, -0.014972738921642303, -0.03863475099205971, -0.03521360456943512, 0.005890445783734322, -0.03257555514574051, -0.065416119992733, 0.10369503498077393, -0.07660520821809769, 0.03156839311122894, 0.010715203359723091, -0.08566346019506454, 0.03191989287734032, 0.05682460218667984, -0.07239260524511337, 0.03177617862820625, 0.07138802111148834, -0.05094006285071373, 0.01918334700167179, 0.039319392293691635, 0.013685331679880619, -0.09917549788951874, -0.004158022813498974, 0.033076297491788864, -0.05503477901220322, 0.024328062310814857, -0.004735393449664116, -0.030811583623290062, 0.04644012451171875, 0.06571321934461594, -0.027890421450138092, 0.07201466709375381, 0.027481960132718086, 0.020339271053671837, 0.04818054288625717, -0.06300845742225647, 0.02565450593829155, 0.01663806289434433, -0.06183015555143356, -0.11949369311332703, 0.04887524992227554, 0.08740081638097763, -0.010714998468756676, -0.03459572046995163, 0.08868399262428284, 0.04116027429699898, -0.021159488707780838, -0.015885552391409874, -0.031031696125864983, 0.0016731623327359557, -0.0260161105543375, -0.03801722824573517, 0.05614478141069412, -0.04741927608847618, -0.016101012006402016, 0.015654344111680984, -0.06833717226982117, -0.043065451085567474, -0.014989027753472328, 0.009846474044024944, 0.06933346390724182, -0.019540539011359215, 0.03838620334863663, 0.002545887604355812, 0.02806990034878254, -0.06109210476279259, 0.011161110363900661, -0.038608353585004807, 0.0573665127158165, -0.04213019832968712, -0.0011390051804482937, -0.04010572284460068, 0.010809306986629963, 0.039086226373910904, 0.011882537044584751, 0.013149138540029526, 0.08280650526285172, 0.08519578725099564, 0.0037062845658510923, -0.030406150966882706], [0.03092356212437153, -0.01882077381014824, -0.07422171533107758, 0.03748393803834915, -0.05035000294446945, 0.0608234740793705, 0.026553237810730934, 0.11287997663021088, 0.040130287408828735, 0.08590536564588547, -0.028652474284172058, 0.07446439564228058, -0.012321101501584053, -0.030676038935780525, 0.016036203131079674, 0.027293356135487556, -0.051720187067985535, -0.0066201286390423775, 0.0012112607946619391, 0.05025970935821533, 0.08865556865930557, -0.023501142859458923, 0.12194959819316864, -0.0007789631490595639, 0.04951300472021103, 0.08698088675737381, 0.03701818361878395, 0.038438405841588974, -0.07169825583696365, 0.034031182527542114, 0.019489312544465065, -0.03749513998627663, 0.07689297199249268, -0.029185842722654343, -0.02017340622842312, 0.062013715505599976, 0.0824073925614357, 0.09870614111423492, 0.04400477930903435, -0.009870815090835094, 0.051071036607027054, -0.06669742614030838, 0.0637195035815239, -0.059650152921676636, -0.03435502573847771, -0.03376109525561333, -0.0027328741271048784, -0.07751602679491043, 0.0854584127664566, 0.044037751853466034, -0.030004557222127914, -0.04201925918459892, 0.03385467454791069, 0.039197325706481934, 0.030925070866942406, 0.10451284050941467, 0.08047185093164444, 0.008478574454784393, 0.045544255524873734, 0.011052400805056095, 0.08153863251209259, 0.010366170667111874, 0.013697229325771332, 0.05115991085767746, -0.024854198098182678, -0.08034360408782959, -0.05344967171549797, 0.015034453012049198, 0.022077033296227455, -0.07616286724805832, 0.009831910021603107, 0.021621564403176308, 0.08386789262294769, -0.005061670206487179, 0.03648960590362549, -0.01024431362748146, 0.01166226714849472, 0.07779906690120697, 0.07536567002534866, -0.014727341942489147, -0.04104623571038246, 0.052921537309885025, 0.02255939692258835, -0.008357965387403965, 0.05133004114031792, 0.03974565863609314, 0.03436972200870514, -0.009723539464175701, 0.0053110504522919655, -0.005699810106307268, -0.01800459623336792, -0.0202541071921587, -0.0021191881969571114, 0.09377078711986542, -0.017111828550696373, 0.01126254815608263], [-0.0078426543623209, -0.004462450742721558, -0.0023414345923811197, -0.04752926528453827, -0.03305720537900925, -0.005000204313546419, -0.005297127645462751, -0.0043158214539289474, -0.003825802356004715, 0.015664706006646156, 0.007056251633912325, -0.0027332808822393417, -0.007871751673519611, -0.04501599073410034, -0.002870105905458331, -0.005264677572995424, -0.003231258597224951, 0.0015908944187685847, 0.009668456390500069, 0.012576964683830738, -0.0010496373288333416, -0.013786761090159416, 0.0401955172419548, -0.005507664289325476, 0.014332237653434277, -0.014409903436899185, 0.016844574362039566, -0.01647331938147545, -0.00873667560517788, 0.006977876648306847, -0.012628808617591858, -0.018675625324249268, 0.02019958384335041, -0.0056941877119243145, 0.040383826941251755, 0.00020962322014383972, 0.0008368479320779443, 0.03977210447192192, 0.010033652186393738, -0.005073848646134138, 0.007114233449101448, -0.017783695831894875, -0.009451176039874554, -0.041441693902015686, -0.0051345485262572765, -0.03198225051164627, 0.014165814034640789, 0.006935235112905502, -0.011973872780799866, 0.0035991619806736708, -0.025836912915110588, -0.008934825658798218, -0.05262938514351845, -0.0021144107449799776, -0.005601987242698669, -0.01350439339876175, -0.02705562859773636, -0.06296739727258682, 0.014802149496972561, 0.009639996103942394, -0.0064931027591228485, -0.015922391787171364, -0.0019479460315778852, 0.0006314184283837676, 0.00016258635150734335, -0.0040894984267652035, 0.01815631613135338, -0.0033536828123033047, 0.000866874004714191, 0.01733584888279438, -0.01538599282503128, -0.08226191252470016, -0.010101809166371822, 0.001957834931090474, -0.005846493411809206, -0.0058879186399281025, 0.03304174169898033, -0.0036363748367875814, 0.0016388208605349064, 0.025784624740481377, -0.005846570245921612, 0.01223390456289053, -0.010084347799420357, 0.003376499516889453, -0.01684327982366085, 0.01444344874471426, -0.01187087967991829, -0.015787741169333458, 0.0009372308268211782, -0.006832841318100691, -0.0015851336065679789, -0.0002709551772568375, 0.006434781942516565, -0.030938483774662018, -0.025903448462486267, 0.00030862988205626607]], "b2": [-0.0321907214820385, 0.02246721275150776, -0.08554743975400925, 0.012070711702108383, 0.04957113415002823, -0.02626313641667366, 0.0679745301604271, -0.016408629715442657, 0.027896247804164886, 0.024660658091306686, -0.020812496542930603, 0.04672425612807274, 0.028716402128338814, -0.004971394315361977, -0.017282195389270782, -0.05339998006820679, 0.008323189802467823, 0.06721363216638565, -0.0290536992251873, -0.020268376916646957, -0.08881155401468277, 0.04796598106622696, -0.01997494138777256, 0.06670060008764267, -0.05753046274185181, 0.022335028275847435, 0.004971093963831663, -0.08370304852724075, -0.001489192945882678, 0.06611498445272446, 0.031832072883844376, -0.013268818147480488], "W3": [[-0.06899852305650711, -0.0921156033873558, 0.24837449193000793, 0.22932128608226776, 0.1744193285703659, -0.023655595257878304, 0.24105902016162872, -0.16503190994262695, 0.1726333349943161, -0.15459828078746796, -0.13411788642406464, 0.11540620774030685, 0.1191185936331749, 0.00044467722182162106, 0.005738746374845505, -0.07976539433002472, 0.058653101325035095, -0.2191934883594513, 0.09553132206201553, 0.28897401690483093, 0.19602927565574646, 0.333385169506073, 0.1932268589735031, -0.21724006533622742, 0.26591038703918457, 0.13646453619003296, -0.1940801441669464, -0.13947853446006775, 0.20220592617988586, 0.187732994556427, -0.13989423215389252, 0.07816694676876068]], "b3": [-0.04920729249715805]}, "2024": {"W1": [[0.014631364494562149, -0.07960708439350128, -0.020343227311968803, -0.08457253873348236, 0.01842687651515007, -0.1193065494298935, 0.03633926436305046, -0.03998978063464165, 0.05102692171931267, 0.027067318558692932, 0.028229648247361183, 0.03323620185256004, -0.0014618155546486378, 0.08381640911102295, 0.020532220602035522, -0.08753436803817749, -0.07526501268148422, -0.060494840145111084, -0.01716584712266922, 0.05600250884890556, 0.012561165727674961, 0.0639900341629982, -0.0657527819275856, -0.005251991096884012, 0.05163555592298508, -0.022163214161992073, -0.018768394365906715, 0.06172123923897743, -0.17660588026046753, 0.05932870879769325, 0.1361197531223297, -0.060259055346250534, -0.042697638273239136, -0.09600063413381577, -0.008439205586910248], [-0.06392701715230942, -0.12991836667060852, -0.127613365650177, -0.09069719910621643, 0.008334963582456112, -0.021895024925470352, -0.05029907822608948, 0.031414322555065155, -0.10110202431678772, 0.011805410496890545, 0.07652843743562698, 0.15194036066532135, -0.06129402294754982, -0.06568702310323715, -0.008239678107202053, 0.08783703297376633, 0.056846983730793, 0.0957178846001625, 0.00833908375352621, 0.03641254082322121, -0.04256744310259819, -0.03501046076416969, -0.029712693765759468, 4.6744607971049845e-05, -0.09451314806938171, -0.05719025805592537, -0.09290541708469391, 0.032398272305727005, -0.06780946254730225, -0.004621149972081184, -0.12900012731552124, 0.03839275613427162, 0.019158655777573586, -0.05235520005226135, -0.040511783212423325], [-0.041528332978487015, 0.005217145662754774, 0.038446564227342606, -0.03381090238690376, -0.069326251745224, -0.052458103746175766, -0.04164320230484009, 0.0407470241189003, 0.02767532505095005, -0.02167475037276745, 0.044443242251873016, 0.028177523985505104, 0.014341720379889011, -0.028009802103042603, 0.027311328798532486, -0.05040821433067322, -0.010776100680232048, 0.06712014228105545, 0.057410553097724915, 0.02805505320429802, -0.06584399938583374, 0.034517884254455566, -0.03479542210698128, -0.06961201131343842, 0.011070845648646355, 0.028857523575425148, 0.0650242492556572, 0.016674013808369637, -0.005126266740262508, 0.03989924490451813, 0.025956494733691216, 0.015892593190073967, -0.0048976438120007515, -0.012633372098207474, -0.004548604134470224], [0.010908311232924461, -0.01998008042573929, 0.02317129075527191, -0.037291720509529114, 0.055396243929862976, 0.00932708103209734, 0.08505858480930328, 0.0527428463101387, -0.0051160529255867004, 0.021792948246002197, 0.038498878479003906, 0.0952216386795044, 0.07662661373615265, 0.035531144589185715, -0.0014340828638523817, -0.1405755579471588, -0.01408417709171772, 0.006172597408294678, -0.018666310235857964, -0.05483313277363777, -0.17418289184570312, -0.03462693840265274, 0.020478609949350357, -0.03997037932276726, -0.00464269146323204, 0.04262951388955116, -0.09481323510408401, 0.05210109427571297, -0.10160519182682037, -0.08953515440225601, 0.10458186268806458, -0.20884418487548828, 0.022324129939079285, 0.18757756054401398, 0.04083110764622688], [0.026642844080924988, -0.03140384703874588, 0.07280304282903671, -0.058521568775177, 0.015145781449973583, -0.08147680759429932, -0.01046404056251049, 0.02164277620613575, 0.03894752636551857, 0.05258160084486008, 0.0901549682021141, -0.027311380952596664, 0.004758395720273256, 0.056053753942251205, 0.051639728248119354, -0.12574216723442078, 0.1541224718093872, -0.09414374828338623, -0.10725338757038116, 0.020715150982141495, -0.08026272803544998, 0.04438893869519234, -0.007852450013160706, -0.06145430728793144, 0.05275657773017883, 0.05691583454608917, -0.06310237944126129, 0.04930325224995613, 0.13404370844364166, 0.06713087111711502, -0.013128825463354588, -0.14907710254192352, -0.002849992597475648, -0.037925247102975845, -0.06569796800613403], [0.02945510484278202, 0.002100475365296006, 0.01434132270514965, -0.09258447587490082, 0.07907791435718536, -0.007745418697595596, -0.024777112528681755, 0.0010177362710237503, 0.03723255544900894, -0.015381081961095333, -0.06536819040775299, 0.016049116849899292, -0.09320896863937378, -0.0814787745475769, 0.07231716811656952, -0.036346498876810074, -0.03164741024374962, 0.05724340304732323, 0.13661926984786987, -0.029280947521328926, 0.009759844280779362, 0.0279395692050457, 0.053100764751434326, -0.014404281973838806, -0.06399859488010406, 0.043584756553173065, 0.007775032892823219, 0.040680691599845886, 0.015368681401014328, 0.03391159698367119, -0.08329451829195023, 0.09594579041004181, 0.10630083084106445, 0.028676185756921768, 0.0999598503112793], [-0.043998342007398605, -0.00045063061406835914, -0.022568779066205025, -0.07969368994235992, -0.05324467271566391, -0.07393243908882141, -0.014896946959197521, 0.031268246471881866, -0.012990377843379974, -0.09032963961362839, -0.007537234574556351, 0.03425624594092369, -0.028832631185650826, 0.017195189371705055, 0.02526411972939968, 0.04715289548039436, 0.048663679510354996, -0.07848268002271652, -0.0697639137506485, -0.08312872052192688, 0.031816381961107254, 0.02282622829079628, 0.035867031663656235, 0.0018691513687372208, -0.033982060849666595, -0.05460486188530922, -0.043982718139886856, -0.05661022290587425, 0.008164456114172935, 0.05394560098648071, 0.05206972733139992, -0.06942665576934814, 0.005034117493778467, -0.03146238252520561, 0.009666355326771736], [0.04758012294769287, -0.12375947088003159, -0.05187512934207916, -0.11592143028974533, -0.03743404522538185, 0.13877014815807343, 0.03950749337673187, -0.08172669261693954, 0.11013799160718918, -0.0055529107339680195, -0.06701670587062836, 0.03863964602351189, -0.07156935334205627, -0.02396790124475956, -0.04556521028280258, 0.11022032797336578, -0.006212348118424416, -0.07768724113702774, -0.04804287850856781, -0.05503106117248535, 0.07657405734062195, -0.0669485405087471, 0.020958147943019867, -0.017835553735494614, -0.08213641494512558, -0.07583577185869217, 0.11173951625823975, 0.015654856339097023, 0.011130713857710361, -0.0318581759929657, -0.07873862981796265, -0.12608115375041962, -0.07245854288339615, -0.04229137673974037, -0.08015210926532745], [0.06488540768623352, 0.016156455501914024, 0.04238129034638405, 0.047762006521224976, -0.08682381361722946, -0.10591188818216324, -0.06517533212900162, -0.09083280712366104, -0.03801704943180084, -0.0716816708445549, 0.07570131123065948, 0.11646173894405365, -0.09494540840387344, -0.05330017954111099, -0.08434721827507019, 0.006715146824717522, 0.03156134486198425, 0.020493481308221817, 0.09466704726219177, 0.0400957353413105, -0.022628195583820343, -0.037829384207725525, 0.010226747952401638, 0.047316648066043854, -0.020875856280326843, -0.07894711196422577, 0.07289700210094452, 0.10182592272758484, -0.017745494842529297, 0.009368455968797207, 0.004975813440978527, -0.05401630327105522, -0.05758897215127945, -0.07959552109241486, -0.08772902935743332], [0.04720496013760567, -0.05768777057528496, -0.07852263003587723, 0.017124228179454803, -0.09116733819246292, 0.009843857027590275, -0.0713551864027977, -0.0017725974321365356, 0.10155566036701202, 0.009597083553671837, 0.0074869985692203045, 0.08772850036621094, 0.03945846110582352, -0.0024762554094195366, -0.06699090451002121, -0.08413871377706528, -0.08080931752920151, -0.06614209711551666, 0.09100428223609924, -0.031222747638821602, 0.15225356817245483, -0.04033694416284561, -0.05899382755160332, -0.0839194804430008, -0.06944797933101654, -0.056159671396017075, 0.003688475349918008, -0.030352363362908363, 0.007520783226937056, -0.05902129411697388, -0.051168493926525116, -0.054438743740320206, -0.0007151720928959548, -0.102381132543087, -0.11023259907960892], [-0.02907278947532177, 0.04144083335995674, -0.03935275971889496, -0.032912902534008026, -0.04085083678364754, 0.059833645820617676, 0.030138203874230385, -0.027832763269543648, 0.019067630171775818, 0.02443843148648739, -0.07285374402999878, 0.047240737825632095, 0.04684971645474434, -0.03759883716702461, -0.030612125992774963, 0.0601145438849926, -0.07318060100078583, -0.04529950022697449, -0.04270927235484123, 0.037094324827194214, 0.03967084363102913, -0.02293943613767624, 0.03377676010131836, 0.03006586804986, -0.04107773303985596, -0.024263333529233932, 0.023401770740747452, 0.02911926619708538, -0.08292167633771896, -0.0020414588507264853, -0.018186204135417938, 0.02859530970454216, -0.0025928656104952097, -0.0700710192322731, -0.008714048191905022], [-0.043268248438835144, -0.019054051488637924, -0.023284580558538437, -0.1734498143196106, -0.07385613769292831, -0.11071734875440598, -0.06974352151155472, 0.050765130668878555, 0.07120860368013382, 0.014792365953326225, 0.04468375816941261, 0.11318296194076538, -0.11034244298934937, -0.07928215712308884, -0.06570770591497421, -0.09957128763198853, 0.07893755286931992, 0.026896124705672264, -0.031500194221735, -0.06786517053842545, 0.03437749668955803, 0.0028550243005156517, -0.03308926150202751, -0.002314169192686677, 0.04643002152442932, 0.08960572630167007, -0.093153215944767, -0.003646397963166237, -0.1488831639289856, -0.05358399078249931, -0.048097509890794754, -0.02372809685766697, -0.03610611706972122, -0.04832795262336731, -0.06928961724042892], [-0.10277599841356277, -0.08785583823919296, 0.03806835785508156, -0.0007395243737846613, 0.008061804808676243, 0.06430152803659439, 0.040840256959199905, 0.011718432419002056, -0.02352256141602993, -0.004563746973872185, 0.001440756139345467, -0.06350695341825485, -0.014329014346003532, 0.027004890143871307, 0.12325089424848557, -0.024904850870370865, -0.06557146459817886, 0.015828602015972137, 0.012397187761962414, 0.022814789786934853, -0.04811247065663338, 0.0335460864007473, -0.016606049612164497, -0.0194338858127594, -0.06552757322788239, 0.11115369200706482, -0.01289701834321022, 0.004036153666675091, 0.042134612798690796, -0.12023581564426422, 0.03632523864507675, -0.1462535262107849, 0.08235238492488861, 0.19744336605072021, -0.017642011865973473], [0.04434295371174812, -0.08811403065919876, 0.04797835275530815, 0.03915103152394295, 0.0010841023176908493, 0.0211322121322155, 0.033232636749744415, 0.10145819187164307, 0.08449357748031616, 0.01945216953754425, 0.050953272730112076, 0.03663595765829086, 0.0498051643371582, -0.013675689697265625, 0.020078938454389572, -0.15367184579372406, 0.03391227498650551, 0.02261507883667946, -0.024389589205384254, 0.005603962577879429, -0.1412453055381775, 0.0076942648738622665, -0.030517499893903732, -0.009557083249092102, -0.02695302478969097, -0.03056013211607933, -0.08336542546749115, 0.021692201495170593, 0.1215846836566925, 0.020236320793628693, 0.054240632802248, -0.21329686045646667, 0.08053287118673325, 0.045556604862213135, -0.0073692165315151215], [-0.022508544847369194, -0.01669594645500183, 0.005966271739453077, 0.011543827131390572, -0.0264141708612442, -0.02774742804467678, 0.023065203800797462, 0.0123491445556283, -0.00387375894933939, 0.009316514246165752, -0.02407381683588028, 0.001956585329025984, 0.012872392311692238, -0.011375795118510723, -0.014154182747006416, 0.006312910933047533, 0.01827199012041092, 0.008907219395041466, 0.027028877288103104, -0.02704758755862713, 0.03421026095747948, 0.025009125471115112, -0.027640672400593758, -0.03184093162417412, -0.025018058717250824, 0.01782255806028843, -0.02689708210527897, 0.01177497860044241, 0.013153662905097008, -0.010001616552472115, 0.014196895994246006, -0.04779943823814392, 0.03738056495785713, 0.025052303448319435, 0.013377377763390541], [0.010280320420861244, 0.12558680772781372, -1.4505106264550705e-05, 0.11591041088104248, 0.001126229064539075, -0.12221541255712509, 0.002050766721367836, 0.01923251524567604, -0.022444816306233406, 0.04917747527360916, -0.005466395989060402, -0.006498353555798531, -0.10956501960754395, -0.011409160681068897, 0.021544504910707474, -0.15083050727844238, 0.05621302127838135, 0.1151144877076149, -0.0035001682117581367, -0.04862811416387558, -0.024150636047124863, 0.002735911402851343, -0.021804379299283028, -0.027557842433452606, 0.01517616305500269, -0.0306337121874094, -0.030796607956290245, -0.043616026639938354, -0.0036199770402163267, -0.06484763324260712, 0.0717092826962471, 0.104725681245327, 0.0449962243437767, 0.03970586508512497, 0.0036896236706525087], [-0.006437893491238356, 0.00947861559689045, 0.016741527244448662, -0.036823075264692307, -0.01421134453266859, -0.07363144308328629, 0.003931658808141947, -0.061933308839797974, -0.05068371072411537, 0.005470920819789171, -0.01976069062948227, 0.016745055094361305, 0.0009145124349743128, 0.009890453889966011, 0.01514419075101614, 0.10223688185214996, -0.0313265435397625, 0.02090415544807911, 0.031970031559467316, 0.0008227519574575126, 0.0758558064699173, 0.02111257053911686, 0.017356274649500847, -0.006715470924973488, -0.005540228448808193, 0.06620508432388306, -0.0061669196002185345, 0.003387488890439272, -0.06463620066642761, -0.00832316279411316, -0.004046651069074869, 0.030322441831231117, -0.0020397603511810303, 0.09225881844758987, 0.028057590126991272], [-0.026433445513248444, 0.039526138454675674, 0.058355946093797684, 0.1184353232383728, 0.04916231334209442, -0.04787478968501091, 0.0027373433113098145, 0.011472267098724842, 0.0778672844171524, -0.017302032560110092, 0.06605199724435806, 0.06664251536130905, -0.053889986127614975, 0.05288086086511612, 0.09450974315404892, 0.05810009688138962, -0.06806211173534393, -0.004280418623238802, 0.08130283653736115, 0.069437675178051, 0.025447804480791092, -0.024774443358182907, 0.04163752868771553, 0.0162349883466959, -0.09559707343578339, 0.025936782360076904, -0.04052484408020973, 0.01218949444591999, 0.07994028180837631, -0.008621581830084324, -0.020796746015548706, 0.09549800306558609, -0.0033476599492132664, -0.04117603972554207, -0.0551387295126915], [0.019212841987609863, 0.02114127203822136, 0.03182860091328621, -0.030942898243665695, 0.0022619494702667, -0.0669092908501625, 0.0406525619328022, 0.04123387113213539, 0.00924699753522873, 0.021085308864712715, 0.0012084456393495202, -0.006159736774861813, -0.08727005124092102, -0.05320942401885986, -0.07617012411355972, -0.024967122822999954, 0.04774875566363335, 0.10359812527894974, 0.054459117352962494, -0.05244101583957672, 0.029590066522359848, -0.042223334312438965, 0.046970125287771225, 0.034390971064567566, -0.05879851058125496, 0.028457866981625557, 0.005238568875938654, 0.017748676240444183, -0.014886961318552494, -0.06638345867395401, -0.13927830755710602, 0.08887195587158203, 0.0016054284060373902, 0.02079273760318756, -0.03394566848874092], [0.01135301310569048, -0.005825564730912447, 0.008565197698771954, 0.0037796590477228165, -0.0378962978720665, -0.02046329714357853, -0.014975296333432198, 0.023004766553640366, -0.016299830749630928, 0.027539294213056564, -0.02066902630031109, -0.01603798009455204, -0.04011889174580574, -0.00648921262472868, 0.028677938506007195, -0.0037741605192422867, -0.013530389405786991, 0.018351169303059578, -0.008449510671198368, 0.01495098602026701, 0.044411879032850266, -0.003945736214518547, 0.014421495608985424, -0.0003414130478631705, -0.008592561818659306, -0.011934864334762096, 0.08320966362953186, 0.00046536404988728464, -0.04100342467427254, 0.004304686561226845, 0.04231112450361252, 0.008932450786232948, 0.03034590370953083, 0.03773356229066849, 0.004570902790874243], [0.05423102527856827, -0.10392171889543533, -0.11176551878452301, -0.08288854360580444, -0.004615905229002237, -0.03170499950647354, -0.04684329777956009, -0.06965304166078568, -0.0013972807209938765, -0.061881810426712036, -0.008835306391119957, 0.12646228075027466, -0.006627104710787535, 0.051912564784288406, -0.0016716169193387032, 0.10698479413986206, 0.08516819775104523, -0.047248490154743195, -0.053948793560266495, 0.03585099056363106, -0.05641459301114082, 0.07840745896100998, 0.01529763638973236, 0.02292955107986927, -0.08087636530399323, 0.022708971053361893, -0.0029887312557548285, -0.04046475142240524, 0.047753430902957916, -0.011916554532945156, 0.013299803249537945, -0.11271214485168457, 0.030481457710266113, -0.09100378304719925, 0.054323915392160416], [0.010437621735036373, 0.12216413021087646, -0.06772585213184357, 0.015510346740484238, 0.04637053236365318, -0.049009304493665695, -0.07373498380184174, -0.08558797091245651, -0.043235789984464645, -0.007774885278195143, -0.08973021060228348, 0.048097968101501465, -0.031378306448459625, 0.0007118459907360375, -0.03718292713165283, 0.11920472234487534, -0.000561462074983865, 0.0013209794415161014, 0.0008501111879013479, 0.06814660131931305, 0.10107588768005371, 0.04771711677312851, 0.029442736878991127, 0.021171189844608307, 0.04650694504380226, 0.016184836626052856, 0.13586394488811493, -0.024882353842258453, -0.12430386245250702, 0.0334741547703743, -0.04667967930436134, 0.03610203415155411, -0.11024253070354462, -0.08353590965270996, -0.012205368839204311], [-0.08037832379341125, 0.09196784347295761, 0.026482535526156425, -0.10297103226184845, 0.012501378543674946, -0.016277261078357697, 0.03488651290535927, -0.024871565401554108, -0.010097554884850979, 0.011383488774299622, 0.03893692418932915, 0.05436628311872482, -0.03201841935515404, 0.05708755925297737, 0.05438733473420143, -0.11600623279809952, 0.07811227440834045, 0.050728071480989456, 0.06943970173597336, 0.0071968031115829945, 0.07730203866958618, -0.08760317414999008, 0.02472304180264473, 0.011962512508034706, 0.014444218017160892, -0.08590404689311981, -0.18283472955226898, -0.02423214167356491, 0.0228181891143322, -0.16850219666957855, -0.1612655371427536, 0.31020209193229675, -0.05361942574381828, 0.1485222578048706, -0.024993831291794777], [-0.034329961985349655, -0.0408521369099617, 0.11247479170560837, 0.03393363952636719, -0.018691521137952805, -0.08990753442049026, 0.04952146112918854, -0.04226367920637131, 0.008484956808388233, 0.03185942769050598, 0.029596703127026558, -0.12388239055871964, 0.005234919022768736, 0.0011081795673817396, 0.021888144314289093, 0.1273088902235031, -0.0800722986459732, -0.07947133481502533, 0.05299243703484535, -0.01474440935999155, 0.016498351469635963, 0.020404933020472527, -0.0013930295826867223, 0.023547455668449402, -0.03359542787075043, 0.15540868043899536, -0.02186182513833046, 0.025586042553186417, -0.17413467168807983, -0.1087115928530693, 0.0342661589384079, -0.14275458455085754, 0.04899200424551964, 0.17747318744659424, -0.003112572245299816], [-0.060879748314619064, -0.0561240017414093, -0.05807759612798691, -0.07314459979534149, 0.07148806005716324, -0.04099062457680702, 0.06145162135362625, -0.03468149155378342, 0.06976201385259628, -0.049872905015945435, 0.0658729150891304, 0.00717166205868125, -0.0048926817253232, -0.015133709646761417, -0.06262484937906265, 0.04522542282938957, -0.04850819706916809, -0.028075963258743286, -0.007060456555336714, 0.0528784915804863, -0.05255158245563507, -0.05607619136571884, 0.06318037211894989, 0.011731860227882862, -0.03475238010287285, -0.07076478004455566, 0.1404796689748764, 0.0542147196829319, -0.14739878475666046, -0.00033711627474986017, -0.09281141310930252, -0.028861327096819878, 0.020944852381944656, -0.1064726933836937, 0.07070425152778625], [-0.013541175983846188, -0.08596398681402206, 0.08481540530920029, -0.11253666132688522, 0.02037958987057209, 0.004243939649313688, 0.02613934688270092, 0.013447791337966919, -0.02288060449063778, 0.03269818052649498, 0.026090890169143677, 0.060922447592020035, -0.0265190526843071, 0.00953668262809515, 0.013694560155272484, -0.017845051363110542, 0.08005286753177643, -0.03930436447262764, -0.03639683127403259, -0.10580837726593018, 0.0605863593518734, -0.09113511443138123, 0.0025857051368802786, -0.04175879806280136, -0.016999829560518265, 0.05709134787321091, -0.11654126644134521, -0.014528664760291576, 0.06372667849063873, -0.07098591327667236, -0.11608889698982239, 0.030573982745409012, 0.019586987793445587, 0.1617959439754486, -0.04614008590579033], [-0.0020045305136591196, -0.010074129328131676, -0.006706807762384415, -0.0010508049745112658, -0.009692481718957424, 0.014500979334115982, -0.004091991111636162, -0.005624907091259956, -0.00420262711122632, 0.010825993493199348, 0.004033949226140976, -0.0021084630861878395, -0.00613827258348465, 0.001457761274650693, 0.008778602816164494, -0.01234793197363615, -0.006506566423922777, 0.0006497153080999851, 0.0033954852260649204, 0.004305546637624502, 0.007266792468726635, -0.000658875098451972, 0.0017561929998919368, -0.0016761177685111761, -0.0035243842285126448, -0.003013185691088438, 0.005183730274438858, -0.00013065985694993287, 0.005999586544930935, -0.0022302318830043077, -0.0015614301664754748, 0.001420962275005877, 0.0038742292672395706, 0.01094857882708311, 0.0025639538653194904], [0.013215713202953339, -0.012092639692127705, -0.04690470173954964, -0.008592580445110798, -0.08617763221263885, -0.022102169692516327, -0.05126580968499184, -0.0207049660384655, 0.02729371190071106, -0.0007053172448650002, -0.12945373356342316, 0.10518903285264969, 0.00910142157226801, 0.0910574421286583, 0.0810093879699707, 0.08314742147922516, -0.010183759965002537, 0.029108505696058273, -0.03467205911874771, 0.09890299290418625, 0.10170911252498627, -0.08132387697696686, -0.028495552018284798, 0.056244444102048874, 0.055564820766448975, 0.018795983865857124, 0.08824830502271652, -0.0449598953127861, -0.08741334080696106, -0.028393736109137535, -0.05852407217025757, -0.09359057247638702, 0.06549244374036789, -0.03797658532857895, -0.008184385485947132], [0.04290229454636574, 0.030169948935508728, 0.08934026956558228, -0.0012974818237125874, -0.06494970619678497, 0.09674470871686935, 0.02107059955596924, -0.016569802537560463, -0.03543386235833168, -0.05570419132709503, 0.11047950387001038, -0.06784687936306, 0.0848337784409523, -0.06117834150791168, -0.018081290647387505, -0.04998927563428879, 0.06641799211502075, 0.04660728946328163, -0.03493966534733772, -0.03788149729371071, 0.019260091707110405, 0.03268224745988846, -0.014999579638242722, -0.06862200051546097, 0.0001996305218199268, 0.0007151241879910231, 0.04131648316979408, -0.021120592951774597, 0.009572277776896954, 0.06514696031808853, 0.06558085232973099, -0.12836547195911407, -0.03708796575665474, 0.09565350413322449, -0.015578178688883781], [0.020366957411170006, -0.007305321283638477, 0.02047734335064888, -0.02963612601161003, -0.043168097734451294, 0.009897439740598202, 0.05977168679237366, 0.0011573168449103832, 0.022757036611437798, 0.005546938627958298, -0.02033066935837269, 0.027173751965165138, 0.021748704835772514, -0.06286879628896713, -0.05946214124560356, 0.028024518862366676, 0.04532608389854431, 0.05557738617062569, 0.03412683680653572, 0.0035851008724421263, 0.04611317068338394, -0.03644103929400444, 0.01026726420968771, -0.030193200334906578, -0.048431508243083954, 0.00560883479192853, 0.01331173162907362, -0.019405728206038475, -0.03638824075460434, -0.03666798025369644, -0.041542913764715195, 0.04760820046067238, 0.03894476965069771, 0.024767523631453514, -0.03115692175924778], [0.046388763934373856, 0.0423017255961895, 0.14951464533805847, 0.023571699857711792, 0.018420586362481117, 0.05715344846248627, 0.010632962919771671, -0.00943458266556263, 0.0017189182108268142, 0.045854371041059494, -0.04233408719301224, 0.008878455497324467, -0.06880456209182739, -0.03802680969238281, -0.005317742470651865, -0.06437618285417557, -0.05783017724752426, 0.014200538396835327, 0.02105686441063881, 0.08501141518354416, 0.07441186904907227, 0.015633519738912582, 0.03180850297212601, -0.04850660264492035, -0.03263533487915993, 0.017946556210517883, 0.0840713158249855, 0.012380028143525124, 0.0056489272974431515, 0.029056543484330177, 0.024729907512664795, 0.06208806857466698, 0.03788600116968155, -0.0011507339077070355, -0.04108938202261925], [0.0008916580700315535, 0.005938700400292873, 0.0972045436501503, 0.03395592048764229, 0.05373965948820114, 0.003967086784541607, -0.05602581053972244, 0.002448440296575427, 0.04773201793432236, 0.03713754191994667, 0.014725684188306332, 0.013266121037304401, 0.007591078523546457, 0.036817461252212524, 0.03733575716614723, -0.03125345706939697, 0.06486811488866806, -0.011243526823818684, -0.05430050939321518, -0.01238252967596054, -0.017965059727430344, 0.034143686294555664, 0.041081540286540985, 0.0263933427631855, 0.03462410718202591, -0.06184864416718483, -0.03973082825541496, -0.0002688938402570784, 0.00172514864243567, 0.03100966475903988, 0.0007072963053360581, -0.09310261905193329, 0.03702986612915993, 0.00028427844517864287, 0.0029876064509153366], [-0.0578998401761055, -0.059874095022678375, 0.04267789050936699, 0.020865464583039284, -0.05399711802601814, 0.02204134687781334, -0.040296461433172226, -0.05565497279167175, 0.005231685005128384, -0.015327929519116879, -0.03929207846522331, -0.009690508246421814, -0.020714396610856056, -0.01784735731780529, 0.019160741940140724, -0.03822914510965347, -0.016220945864915848, 0.010798824951052666, 0.031238717958331108, -0.019176023080945015, 0.07199415564537048, 0.01606290228664875, 0.013848796486854553, -0.017673984169960022, -0.029844297096133232, 0.06278069317340851, -0.01991509646177292, -3.4954539387399564e-06, -0.005729427561163902, -0.052432555705308914, -0.04531608521938324, 0.1089811846613884, 0.02017194963991642, 0.12985143065452576, -0.029586607590317726], [0.02849017083644867, -0.02416890114545822, 0.03379571810364723, 0.013822689652442932, -0.05066382512450218, -0.05941532924771309, -0.020736457780003548, 0.016356585547327995, 0.05272010713815689, 0.0631767213344574, 0.0032533903140574694, -0.003812862327322364, -0.06766124069690704, -0.021950403228402138, -0.060653164982795715, 0.044415831565856934, -0.03622162342071533, 0.030273398384451866, 0.09277690201997757, -0.08536656945943832, 0.09773901849985123, 0.0028174167964607477, -0.060681018978357315, 0.03258473426103592, 0.02126697078347206, -0.08539996296167374, 0.03278685733675957, 0.05214603617787361, 0.05898667871952057, 0.01598374918103218, -0.01968957856297493, 0.04136938229203224, -0.07125451415777206, -0.022472165524959564, -0.042902618646621704], [0.013193940743803978, -0.08264744281768799, -0.02575763128697872, -0.06945453584194183, 0.033147986978292465, 0.0314185656607151, 0.054909031838178635, 0.0022439975291490555, -0.09328825771808624, -0.03482735902070999, -0.11727183312177658, 0.03370949998497963, -0.07176974415779114, 0.023251617327332497, 0.041197288781404495, -0.07756968587636948, 0.047401174902915955, 0.0326291099190712, -0.06297138333320618, -0.08031056076288223, 0.12254401296377182, -0.0485515333712101, 0.05156548321247101, 0.0392669253051281, 0.020926829427480698, -0.10740790516138077, 0.05878203734755516, -0.05340886861085892, -0.11900418996810913, 0.10024171322584152, -0.027526823803782463, -0.038855962455272675, -0.09335298091173172, -0.023503197357058525, -0.04561222344636917], [-0.046083830296993256, -0.048318736255168915, 0.028207633644342422, -0.011815245263278484, -0.02780825085937977, 0.04704310745000839, -0.006442129146307707, 0.0013626914005726576, 0.004916430450975895, 0.07696197926998138, -0.01251286081969738, -0.028685903176665306, 0.005681865848600864, 0.014515230432152748, 0.038482122123241425, 0.012031156569719315, 0.02130579575896263, -0.01602780818939209, 0.014706350862979889, -0.024894606322050095, 0.05210394412279129, -0.027767477557063103, -0.0042852298356592655, -0.005762457847595215, -0.053923461586236954, 0.1173548474907875, -0.05073283240199089, 0.008346873335540295, 0.0012190554989501834, -0.08195461332798004, -0.015604562126100063, 0.07810047268867493, 0.013623874634504318, 0.23828819394111633, 0.009965513832867146], [-0.05804767087101936, -0.06949938833713531, 0.04008707031607628, 0.05109300836920738, -0.0890776738524437, 0.018835406750440598, -0.03277643397450447, 0.014880879782140255, -0.08131781220436096, 0.05838581919670105, -0.0677228793501854, -0.04557608813047409, -0.08649951964616776, -0.0046492451801896095, 0.08270791172981262, 0.008775386027991772, -0.050299253314733505, -0.08779408782720566, -0.06467198580503464, -0.06187731772661209, 0.05484568700194359, -0.07667457312345505, -0.06988521665334702, -0.056127216666936874, 0.036826521158218384, 0.022005142644047737, -0.0833728238940239, 0.05511973053216934, 0.08549787849187851, -0.022457880899310112, 0.004359729588031769, 0.05228902027010918, -0.04759105294942856, 0.0330374501645565, -0.036597203463315964], [0.008463977836072445, 0.036206260323524475, 0.0021434237714856863, 0.001883548335172236, 0.0649365484714508, -0.031665362417697906, -0.049206189811229706, 0.08994777500629425, -0.025248896330595016, 0.0603458508849144, -0.04324273765087128, 0.01180364191532135, -0.022122792899608612, 0.053965359926223755, 0.049719274044036865, -0.13831380009651184, -0.06548295170068741, 0.0036636495497077703, 0.07381435483694077, 0.04728175699710846, 0.09962751716375351, -0.031228354200720787, -0.016572723165154457, 0.014714348129928112, 0.05926020070910454, 0.10275648534297943, -0.19114823639392853, -0.06405548751354218, 0.12322947382926941, -0.12145072221755981, -0.1904592216014862, 0.33164799213409424, 0.0035246023908257484, 0.11852365732192993, -0.025080347433686256], [-0.013433489948511124, -0.022820746526122093, 0.021353065967559814, -0.019057346507906914, -0.006397077813744545, 0.019338052719831467, -0.03399057686328888, 0.005981928668916225, 0.010064051486551762, 0.025167828425765038, -0.0021239793859422207, -0.03372754529118538, 0.012004565447568893, 0.032685961574316025, 0.035674478858709335, 0.05297878384590149, -0.02795550413429737, -0.034084223210811615, 0.008297532796859741, -0.019883178174495697, 0.029811888933181763, -0.033677902072668076, 0.012464644387364388, 0.021205836907029152, -0.02540482021868229, 0.08638760447502136, -0.026909399777650833, -0.00354078970849514, -0.020097792148590088, -0.02501351200044155, -0.001760585349984467, 0.04501032829284668, -0.022932052612304688, 0.1435489058494568, 0.026423195376992226], [-0.004463807214051485, -0.04555928334593773, 0.047314051538705826, 0.061460044234991074, -0.007451078854501247, 0.07950305938720703, 0.07839227467775345, 0.06520669907331467, -0.0027100828010588884, 0.07204978913068771, -0.027358295395970345, 0.013804344460368156, 0.009681758470833302, 0.012179026380181313, 0.039039526134729385, -0.10724187642335892, -0.02118493989109993, -0.028594767674803734, 0.033412471413612366, -0.03318088874220848, 0.010349946096539497, 0.040270090103149414, -0.021964607760310173, 0.07079954445362091, -0.09424800425767899, 0.029613366350531578, -0.12410005927085876, 0.016645122319459915, -0.03991209715604782, -0.004230061545968056, 0.04699626937508583, -0.09794759750366211, 0.06140187755227089, -0.008733524940907955, 0.07165103405714035], [0.01864060014486313, 0.08471158146858215, -0.0005171758821234107, 0.05028151720762253, 0.0899849534034729, -0.0341939851641655, 0.05803930014371872, 0.021972279995679855, -0.003423144342377782, 0.049545224756002426, 0.06423042714595795, -0.020686106756329536, -0.029053932055830956, -0.09936316311359406, -0.12917588651180267, -0.023106317967176437, 0.05473681166768074, 0.052069488912820816, 0.004897356033325195, -0.04987333342432976, -0.07059793919324875, -0.03740078583359718, -0.020020250231027603, 0.03817199543118477, 0.0413668155670166, 0.08705046027898788, -0.041235171258449554, -0.022474924102425575, 0.03616029769182205, -0.1073041781783104, 0.01753648929297924, 0.1687440723180771, 0.04241284728050232, 0.004271626006811857, 0.04318695515394211], [-0.019007885828614235, -0.03610638156533241, -0.01497566606849432, -0.016950933262705803, -0.015615868382155895, -0.022311173379421234, 0.02971174754202366, 0.0724138393998146, 0.02388465218245983, -0.0012811771593987942, -0.050710227340459824, -0.04255366325378418, -0.01265337597578764, -0.02563135139644146, -0.017066504806280136, -0.03981567174196243, -0.016978401690721512, 0.03249099478125572, 0.05070478841662407, 0.0021662807557731867, -0.14321152865886688, 0.004847091622650623, -0.05977822467684746, -0.07960142195224762, -0.1391858458518982, -0.03017750009894371, -0.07925087213516235, -0.053842246532440186, -0.009658781811594963, 0.0700567215681076, 0.0681709423661232, -0.02724987082183361, 0.05018781125545502, -0.024532122537493706, -0.047453492879867554], [-0.0333283469080925, 0.016761889681220055, -0.04469607397913933, -0.029608523473143578, 0.0029330053366720676, -0.0485609695315361, -0.009709548205137253, -0.0013001718325540423, 0.029248880222439766, 0.044597405940294266, -0.0017826483817771077, -0.08876627683639526, 0.09264962375164032, -0.01989740878343582, 0.014124331064522266, -0.0021163972560316324, -0.056180864572525024, -0.0013943351805210114, 0.05278094857931137, 0.0220327265560627, 0.0015521657187491655, -0.02352769300341606, -0.020218651741743088, 0.0065521555952727795, 0.014504440128803253, 0.07094133645296097, 0.08642060309648514, 0.04496642202138901, -0.009745487943291664, -0.07243796437978745, -0.004902600776404142, 0.012219490483403206, -0.04752570018172264, -0.0128689706325531, -0.045963104814291], [0.049931250512599945, -0.03536158427596092, 0.07433804124593735, 0.04618191719055176, -0.005616500973701477, -0.005470057018101215, 0.027452848851680756, 0.08681858330965042, 0.09079831838607788, 0.022390615195035934, 0.03893211483955383, 0.050634462386369705, 0.0503162257373333, -0.02291487343609333, 0.006101572420448065, -0.1313653290271759, 0.05518750846385956, 0.013946188613772392, -0.029665391892194748, -0.005269511602818966, -0.08525851368904114, -0.011550520546734333, -0.007444388698786497, -0.03961215168237686, -0.015528202056884766, -0.022217588499188423, -0.04548723250627518, 0.010103493928909302, 0.06486494094133377, 0.03254328668117523, 0.02441491186618805, -0.1766197234392166, 0.0477912537753582, -0.028902066871523857, -0.026051510125398636], [-0.029775036498904228, 0.03085389733314514, 0.03902309760451317, 0.025626104325056076, 0.04780581220984459, 0.013365090824663639, 0.05228289216756821, -0.049474090337753296, 0.027567392215132713, 0.055445849895477295, 0.01840500719845295, -0.0740530863404274, -0.005301595199853182, 0.032893791794776917, -0.04933673143386841, 0.05740884318947792, 0.009962127543985844, -0.012149247340857983, -0.07275727391242981, 0.017447490245103836, 0.07583297044038773, 0.016576293855905533, -0.020829394459724426, 0.09772701561450958, 0.06326806545257568, -0.02786027453839779, 0.030198831111192703, 0.028962967917323112, 0.017124181613326073, -0.07414788007736206, 0.024342525750398636, 0.09396475553512573, -0.08108284324407578, 0.01719527691602707, 0.03711017966270447], [-0.018058238551020622, -0.07964690774679184, -0.03169099986553192, 0.0055829621851444244, 0.001593796070665121, -0.026501506567001343, 0.046479783952236176, -0.004318181425333023, 0.038096748292446136, 0.016274694353342056, -0.060156483203172684, -0.003671357873827219, 0.05854779854416847, -0.012220104224979877, 0.0041940719820559025, -0.04930132254958153, -0.007393457926809788, -0.017612796276807785, 0.03226953744888306, 0.013317340053617954, -0.10444480180740356, 0.01530543714761734, 0.01934477500617504, -0.06653029471635818, -0.007285488303750753, 0.054686594754457474, -0.06791196763515472, 0.013685856945812702, 0.01762359030544758, -0.029899194836616516, -0.0052871499210596085, -0.13508211076259613, 0.04001675918698311, 0.15682701766490936, -0.046507541090250015], [0.0147011186927557, 0.042742032557725906, 0.0643274262547493, 0.017876671627163887, 0.025865662842988968, -0.0108195710927248, 0.011459888890385628, 0.013223463669419289, 0.012145898304879665, -0.025606973096728325, -0.020355015993118286, -0.0014083792921155691, -0.02244403585791588, -0.01675672084093094, -0.021225063130259514, -0.003512298222631216, 0.016606153920292854, 0.007211712189018726, -0.004726889077574015, 0.01669079251587391, -0.04256373643875122, 0.01572197489440441, -0.0059884535148739815, -0.0005247247754596174, 0.006964405998587608, -0.013643868267536163, 0.01075444184243679, 0.006064021494239569, -0.023052942007780075, 0.03302720934152603, 0.014030789025127888, -0.03248412907123566, -0.006828772835433483, -0.009845947846770287, -0.01571870595216751], [0.035596802830696106, 0.10307597368955612, -0.07526041567325592, -0.043733980506658554, -0.0469217486679554, 0.014934315346181393, -0.06374374032020569, 0.02350359968841076, -0.03550100699067116, -0.0422140397131443, 0.03854254260659218, -0.059357721358537674, 0.03726441413164139, 0.05606849491596222, -0.006248075980693102, 0.05380522459745407, 0.035269107669591904, 0.08587221056222916, -0.11303509026765823, -0.0060089449398219585, -0.05854376032948494, -0.11383870989084244, -0.03971109539270401, 0.02621361054480076, 0.03247478976845741, -0.02165140025317669, 0.0009932824177667499, -0.007504065986722708, -0.015575163997709751, 0.03774099797010422, -0.13996106386184692, 0.14118127524852753, -0.06335723400115967, -0.03602546080946922, -0.06945790350437164], [0.0382426381111145, 0.05230109766125679, -0.02395421639084816, -0.011159319430589676, -0.018735740333795547, 0.07534652203321457, 0.049876973032951355, -0.013821836560964584, 0.03175782039761543, -0.00021700355864595622, -0.06291351467370987, 0.13422921299934387, -0.00854233093559742, 0.021632159128785133, -0.08860108256340027, -0.09566835314035416, -0.09017536044120789, -0.05948076397180557, 0.01229473203420639, -0.0030857070814818144, 0.11027371138334274, -0.038302429020404816, 0.02029094658792019, 0.014185700565576553, -0.03179285302758217, -0.01683855801820755, 0.07456457614898682, -0.038943495601415634, -0.03181041404604912, 0.07693660259246826, 0.020452849566936493, 0.012806142680346966, 0.020388875156641006, 0.038530681282281876, 0.034503672271966934], [-0.05488676205277443, -0.02056630328297615, 0.017413659021258354, 0.0017828003037720919, 0.016794027760624886, -0.0035844717640429735, -0.03188242390751839, 0.0033682165667414665, 0.0027419584803283215, 0.03897041082382202, -0.012308469042181969, -0.03214840963482857, 0.0703665167093277, 0.016239456832408905, -0.008694472722709179, -0.022753482684493065, 0.00976440031081438, -0.03419700637459755, 0.03136778622865677, -0.03265416994690895, 0.0217486210167408, 0.009181502275168896, 0.010203873738646507, -0.006064146291464567, -0.014859124086797237, 0.06812958419322968, -0.013893064111471176, -0.0061824386939406395, -0.03394588083028793, -0.01788168214261532, -0.010593061335384846, -0.006717754527926445, 0.019267428666353226, 0.07025758177042007, 0.0001479017228120938], [-0.0619947612285614, -0.06452906131744385, 0.07042787969112396, -0.013035065494477749, -0.06633049249649048, 0.08334636688232422, 0.056081727147102356, -0.030498843640089035, -0.024256477132439613, -0.022851254791021347, 0.03922731801867485, 0.009046211838722229, 0.036306124180555344, 0.022942133247852325, 0.12781743705272675, 0.002319741528481245, -0.02262715809047222, 0.004627550020813942, 0.058564331382513046, -0.07032263278961182, -0.09226399660110474, 0.04227333888411522, 0.03023742325603962, -0.04541903734207153, -0.0075636268593370914, 0.09970813989639282, 0.05990839749574661, -0.020917857065796852, 0.08573883026838303, -0.10003796964883804, 0.0933942124247551, -0.23366175591945648, 0.00865766778588295, 0.19328221678733826, 0.05769334360957146], [0.07934229075908661, -0.06324511766433716, -0.13294360041618347, 0.023537196218967438, 0.043123576790094376, -0.021295083686709404, -0.07807323336601257, 0.03079926408827305, -0.036195117980241776, -0.029400188475847244, 0.039871856570243835, 0.11377299576997757, 0.11666665971279144, 0.05503765493631363, 0.0694248378276825, -0.02356518618762493, 0.010318576358258724, -0.013627048581838608, -0.09777085483074188, -0.06275384873151779, -0.024409865960478783, 0.011622976511716843, 0.006429140921682119, -0.03384663909673691, 0.020965948700904846, 0.04083390533924103, 0.06840351969003677, -0.056294042617082596, 0.05530606210231781, 0.06389620900154114, 0.02860933728516102, -0.07224097847938538, -0.05631730705499649, -0.025258377194404602, 0.03337458521127701], [-0.04468131437897682, -0.13296249508857727, 0.06959865242242813, -0.008022326044738293, 0.002528541022911668, -0.007191190961748362, 0.06091366708278656, -0.0033830683678388596, -0.038108088076114655, 0.037013471126556396, 0.006177613511681557, 0.037888895720243454, 0.010308843106031418, -0.03565654531121254, 0.050713445991277695, -0.11626307666301727, 0.09686511754989624, 0.06796752661466599, 0.046012986451387405, 0.007480838801711798, -0.049191027879714966, 0.039824772626161575, -0.06950853019952774, -0.023672273382544518, -0.017890263348817825, 0.01857933960855007, -0.026559971272945404, 0.010919352062046528, 0.07243651151657104, -0.021696336567401886, 0.01165733952075243, -0.21057850122451782, 0.10650144517421722, 0.12234407663345337, 0.015360594727098942], [-0.021160637959837914, 0.01894831843674183, 0.06409571319818497, 0.039814967662096024, 0.0646049827337265, -0.07760057598352432, -0.022500494495034218, 0.03431062027812004, -0.08360479772090912, -0.004089816473424435, 0.030930692330002785, -0.0029177695978432894, -0.055654652416706085, -0.029628099873661995, 0.010632925666868687, 0.06166251376271248, 0.06308591365814209, -0.05006074532866478, -0.03317473828792572, 0.00028761732392013073, -0.07743029296398163, -0.01546610426157713, 0.002270095283165574, -0.02061547338962555, -0.023179035633802414, -0.01957077346742153, 0.053854018449783325, 0.06260690093040466, 0.022478897124528885, 0.031177356839179993, 0.06269669532775879, 0.0342659130692482, -0.03865979611873627, 0.024998746812343597, 0.01491713896393776], [-0.0130936149507761, -0.07587353140115738, 0.004655458498746157, -0.08994192630052567, -0.055957868695259094, -0.009823769330978394, 0.03577559441328049, -0.01474651787430048, -0.04525616019964218, -0.0007086416590027511, -0.028055140748620033, -0.050455667078495026, -0.06096246838569641, 0.044947728514671326, 0.0047743842005729675, -0.03345073387026787, 0.06077751889824867, 0.022194962948560715, -0.050099972635507584, -0.07368279993534088, 0.028599420562386513, 0.013187645934522152, 0.044212765991687775, 0.007046010345220566, 0.01463156659156084, -0.01539858989417553, -0.05807868763804436, -0.021902941167354584, -0.013828055001795292, -0.055616989731788635, -0.0091527309268713, -0.05087947100400925, 0.03372197970747948, -0.0012394199147820473, -0.03739131614565849], [-0.055354200303554535, 0.050907645374536514, 0.25196823477745056, 0.10590407252311707, 0.1467643678188324, -0.07945434004068375, 0.010243094526231289, -0.08776968717575073, 0.0037566651590168476, -0.002066956600174308, 0.12783952057361603, -0.014489192515611649, -0.049193188548088074, 0.12126553803682327, -0.009451759979128838, 0.01293161977082491, -0.12919370830059052, -0.03525478392839432, 0.04858969897031784, -0.06059878692030907, 0.06386728584766388, -0.055309146642684937, 0.0055881985463202, 0.05139334872364998, -0.056988589465618134, -0.025053149089217186, 0.004912613891065121, -0.02142404206097126, 0.15079867839813232, 0.05969918519258499, -0.031871434301137924, 0.186323881149292, 0.04137347638607025, 0.03826769441366196, -0.006640320178121328], [0.06044076383113861, 0.00817611999809742, 0.054612766951322556, 0.008706171996891499, -0.05452713370323181, -0.020259877666831017, -0.019567733630537987, 0.08729204535484314, -0.032334573566913605, -0.016444522887468338, 0.013713004067540169, 0.01896820217370987, 0.008350012823939323, 0.01873980462551117, 0.0163441002368927, 0.015238882042467594, -0.0373782254755497, 0.00805778056383133, 0.028734173625707626, -0.005969425663352013, 0.07990533113479614, 0.021591588854789734, 0.030716830864548683, -0.024698346853256226, 0.0017649169312790036, 0.025108933448791504, 0.044377513229846954, -0.023722337558865547, 0.047270797193050385, -0.03633131459355354, -0.0625491663813591, 0.08074180781841278, 0.04419291391968727, 0.04416706785559654, -0.05428989604115486], [-0.07887722551822662, -0.07821577787399292, 0.06027983874082565, -0.03295907378196716, 0.008787200786173344, -0.020404189825057983, 0.08291212469339371, -0.006486250087618828, -0.026759548112750053, 0.006245959084481001, -0.02705792337656021, -0.05926806107163429, 0.11663144081830978, -0.009170998819172382, 0.07232201844453812, 0.03211681544780731, 0.07740381360054016, -0.07131320238113403, 0.041858937591314316, 0.005431727040559053, -0.11167007684707642, -0.029427656903862953, 0.02586100623011589, 0.04699110984802246, -0.08023592084646225, 0.18886490166187286, 0.05764429271221161, 0.009304778650403023, -0.11786477267742157, -0.010295689105987549, 0.03736519441008568, -0.2006404846906662, 0.0480838343501091, 0.18481425940990448, -0.004078132566064596], [-0.036088310182094574, 7.154727063607424e-05, 0.0160124059766531, 0.008360069245100021, -0.03015122562646866, 0.06285658478736877, -0.029034573584794998, 0.025958558544516563, 0.0433146096765995, 0.04241069778800011, -0.004394352901726961, -0.029231417924165726, -0.01014044787734747, 0.0014298275345936418, 0.0022044889628887177, -0.03391074016690254, -0.009194661863148212, -0.016931850463151932, 0.04356159642338753, 0.025995386764407158, 0.08341342210769653, -0.008903827518224716, 0.004909772425889969, -0.0008438715594820678, -0.02159375511109829, 0.06747785210609436, -0.03978040814399719, -0.020162751898169518, -0.019354652613401413, -0.02710162289440632, -0.038841575384140015, 0.08028457313776016, 0.009089861065149307, 0.07874731719493866, -0.019761163741350174], [0.01774187944829464, -0.013039717450737953, -0.011356010101735592, 0.0030909685883671045, 0.018862606957554817, 0.014340164139866829, -0.02869637869298458, -0.01620166376233101, -0.01238708570599556, 0.007071467116475105, -0.0036116531118750572, 0.014053435996174812, 0.011842358857393265, 0.032050494104623795, 0.05660625919699669, 0.006154046393930912, -0.05140424147248268, -0.05914444103837013, 0.0388769693672657, 0.00975470244884491, -0.019820265471935272, 0.0024228859692811966, -0.0010454102884978056, -0.01523644756525755, -0.06866824626922607, -0.06290075182914734, 0.0753643587231636, 0.012580307200551033, 0.039646852761507034, 0.03762880340218544, 0.009121276438236237, -0.03281286358833313, 0.019763147458434105, -0.0358777791261673, -0.01162116602063179], [-0.0007101398659870028, -0.021012401208281517, 0.01626979373395443, -0.06641678512096405, -0.02457699179649353, 0.03537303954362869, 0.04891978204250336, -0.07200933247804642, -0.04515586048364639, -0.07570690661668777, 0.03386516496539116, 0.09243747591972351, 0.04967712238430977, -0.021695422008633614, -0.04701879993081093, -0.012462259270250797, -0.06583551317453384, -0.05304722487926483, -0.033375438302755356, -0.010376890189945698, 0.07915123552083969, 0.025876177474856377, -0.03459838777780533, 0.000434128480264917, -0.06435951590538025, 0.05184454843401909, 0.033741239458322525, 0.051078006625175476, -0.0397038459777832, -0.06888655573129654, 0.03309156000614166, -0.013612249866127968, 0.03574902564287186, 0.03786361590027809, 0.012858851812779903], [-0.013859262689948082, -0.03374967724084854, 0.05527403578162193, -0.013342218473553658, 0.006564267911016941, -0.03631482645869255, 0.03561237454414368, -0.03236200660467148, 0.0008066761074587703, -0.02300596423447132, -0.009925954975187778, 0.01810593716800213, 0.039245475083589554, -0.003752885852009058, 0.028976142406463623, 0.021204739809036255, -0.08960063755512238, 0.02237541973590851, 0.06352590769529343, -0.06126951053738594, 0.027409514412283897, -0.004902608692646027, 0.02027808129787445, -0.049267787486314774, 0.005402381531894207, 0.07998240739107132, 0.036154575645923615, 0.032059893012046814, 0.02569553069770336, -0.01461979653686285, -0.014519128017127514, -0.07614265382289886, 0.026144027709960938, 0.05164037644863129, 0.003042511874809861], [0.008851487189531326, 0.00028792759985662997, 0.0008380216313526034, 0.0019673423375934362, 0.005091285798698664, -0.0038581397384405136, 0.0006290975143201649, -0.0007959988433867693, -0.004036556929349899, -0.0011809932766482234, -0.001927070552483201, 0.0034756080713123083, 0.0010311155347153544, 0.0005199306178838015, -0.006407927256077528, -0.00682467594742775, -0.0007412696140818298, -0.0008683372871018946, -0.003060088725760579, -0.003475726582109928, -0.011439671739935875, -0.0005310173146426678, -0.0006442491430789232, 0.005027597304433584, -0.005727928597480059, -0.0007804045453667641, 0.02202172391116619, 0.0013674122747033834, -0.003374803578481078, 0.0010968249989673495, 0.00258556823246181, -0.0013952641747891903, 0.001491645583882928, 0.006075809709727764, 0.002470226027071476], [-0.04334954917430878, -0.009816974401473999, 0.003013554261997342, -0.0036720656789839268, -0.09033054858446121, 0.08638975769281387, -0.01437106728553772, -0.022233348339796066, 0.04868577793240547, 0.04407326132059097, -0.03132496401667595, -0.015511713922023773, -0.03431588038802147, 0.013968845829367638, 0.024880370125174522, -0.04314472898840904, -0.014168445020914078, -0.046420805156230927, 0.008796093985438347, 0.049131035804748535, 0.04425739496946335, -0.012050732970237732, -0.02532484568655491, -0.013790756464004517, 0.016648361459374428, 0.04749353229999542, -0.043834853917360306, 0.0047270022332668304, -0.046156350523233414, -0.04639441892504692, 0.04809878021478653, 0.02674788050353527, 0.0024530498776584864, 0.06085754558444023, -0.028794288635253906], [0.02142723836004734, 0.07041754573583603, -0.07955735176801682, -0.029067862778902054, -0.02464442327618599, -0.060764800757169724, -0.04502641782164574, 0.00927666574716568, 0.037801001220941544, -0.0019932484719902277, -0.07056744396686554, -0.06374591588973999, 0.023584458976984024, 0.0149228535592556, -0.06113305315375328, 0.09745369106531143, -0.027595723047852516, -0.04082385078072548, -0.08858104795217514, 0.032452087849378586, 0.006347971968352795, -0.03147323429584503, 0.017537076026201248, 0.036595419049263, 0.00966683216392994, -0.025356218218803406, 0.03048846870660782, 0.021891463547945023, 0.01618857868015766, -0.047277048230171204, -0.08672305196523666, 0.009219861589372158, 0.04977484419941902, 0.05352581664919853, -0.020279845222830772], [-0.001400698209181428, 0.024935690686106682, 0.10623372346162796, 0.029102757573127747, -0.04320418834686279, 0.09280535578727722, 0.0025809796061366796, -0.019725611433386803, -0.013048119843006134, -0.006473312620073557, 0.04194659739732742, 0.020956287160515785, 0.0033525917679071426, 0.0014013131149113178, 0.010371046140789986, 0.012771409936249256, 0.010871753096580505, 0.07555457949638367, 0.014516321942210197, 0.008104264736175537, -0.05364113673567772, 0.07112908363342285, 0.01673891767859459, 0.022734448313713074, 0.08014218509197235, -0.010207739658653736, -0.017807945609092712, 0.022077379748225212, 0.0066560255363583565, 0.05501893535256386, 0.03904517740011215, -0.04313715174794197, 0.004358493722975254, -0.028026187792420387, -0.004105454310774803], [0.0033603121992200613, 0.06623806804418564, -0.015678146854043007, -0.0028901351615786552, 0.005794689990580082, 0.07178758829832077, 0.04722537845373154, -0.03659103438258171, -0.009761776775121689, -0.024772051721811295, 0.015353658236563206, -0.030344735831022263, -0.019421938806772232, -0.017548004165291786, -0.02132827416062355, -0.055600930005311966, 0.0390656441450119, 0.007611365057528019, 0.009509033523499966, 0.0459366999566555, -0.043133340775966644, 0.03854586184024811, -0.010120413266122341, -0.009965053759515285, -0.02064318023622036, -0.11377830803394318, -0.0342707522213459, 0.01709768734872341, -0.03886997699737549, 0.058406926691532135, -0.031195905059576035, 0.0656374916434288, 0.0319681353867054, -0.14947324991226196, -0.0029370738193392754], [-0.011566505767405033, -0.03991198167204857, -0.01166597194969654, -0.032239850610494614, 0.03370068967342377, -0.010968917980790138, -0.012937436811625957, 0.023707957938313484, -0.030156470835208893, -0.043655067682266235, 0.022940315306186676, 0.02580159902572632, 0.0037018335424363613, -0.015668978914618492, 0.0250992551445961, 0.006057852413505316, 0.04187354817986488, -0.04094650596380234, -0.0458524227142334, -0.027885405346751213, -0.01234233845025301, -0.00987119972705841, 0.0025035105645656586, 0.026906531304121017, -0.041861239820718765, -0.012797785922884941, -0.02178391069173813, 0.0009776782244443893, 0.010371842421591282, 0.04380137845873833, -0.007427910342812538, 0.002695941599085927, -0.028041772544384003, -0.016624214127659798, 0.0018723165849223733], [0.07313621044158936, -0.014984184876084328, 0.037714600563049316, -0.05745980888605118, 0.011189356446266174, -0.05063840374350548, 0.01659313775599003, 0.05234948545694351, -0.006370013114064932, 0.052676692605018616, 0.028376266360282898, -0.04797028750181198, 0.009587745182216167, 0.03286484256386757, 0.011307398788630962, 0.07466978579759598, 0.0606125071644783, 0.03711778298020363, 0.02436050772666931, 0.04178314283490181, 0.08307391405105591, 0.0060540116392076015, 0.034376103430986404, 0.02857423946261406, 0.008571412414312363, 0.050144702196121216, 0.004098517820239067, 0.04439026862382889, 0.019760536029934883, 0.04606993496417999, 0.048078879714012146, -0.013702310621738434, -0.05605098605155945, 0.05490601435303688, -0.0356084443628788], [0.026139281690120697, -0.02914503961801529, -0.024477094411849976, 0.03331504017114639, -0.032920852303504944, 0.03481128811836243, 0.06511392444372177, -0.06477423012256622, 0.009861636906862259, -0.03301873803138733, 0.1198413074016571, -0.09134776890277863, 0.08833339810371399, -0.025838924571871758, 0.008375495672225952, -0.09692944586277008, -0.0992492213845253, 0.10357408970594406, -0.10614029318094254, 0.032790180295705795, -0.0557016097009182, -0.042503200471401215, 0.051587749272584915, -0.08023307472467422, -0.03472631052136421, -0.0599718801677227, 0.09086886793375015, 0.026040593162178993, 0.07051009684801102, -0.08668208122253418, -0.05297703295946121, -0.027227090671658516, -0.09739002585411072, 0.0028220561798661947, 0.00856836698949337], [-0.006525159347802401, -0.022669656202197075, 0.04633958637714386, 0.053260549902915955, 0.07692190259695053, 0.0644758865237236, 0.05981916934251785, 0.013047588989138603, 0.05339287221431732, 0.04953118413686752, -0.08201107382774353, 0.0094246044754982, 0.06552299857139587, -0.026813585311174393, -0.020049601793289185, -0.014265109784901142, 0.11172663420438766, 0.023265544325113297, 0.015084563754498959, -0.007538081146776676, -0.02683083340525627, -0.01978752762079239, 0.0015685289399698377, 0.020597325637936592, -0.12089365720748901, -0.01059139147400856, -0.10882741212844849, 0.006306018680334091, -0.04365062713623047, -0.07281812280416489, 0.013767169788479805, -0.036652930080890656, 0.006678472273051739, -0.04462176561355591, -0.010041051544249058], [-0.09700750559568405, -0.05877799540758133, 0.03278256952762604, 0.0011906526051461697, -0.015608694404363632, 0.09243655204772949, -0.002651094226166606, -0.04778846725821495, 0.044106073677539825, -0.014202099293470383, -0.07341097295284271, -0.019479406997561455, -0.00914507731795311, 0.007426750380545855, -0.00453520892187953, 0.08692025393247604, -0.10633411258459091, -0.07215987890958786, 0.03697887435555458, -0.042858537286520004, -0.03887539356946945, 0.021913757547736168, -0.01906682550907135, -0.02480742335319519, -0.07937289774417877, 0.16619469225406647, 0.021286485716700554, 0.08852933347225189, -0.09532234072685242, -0.09332418441772461, 0.06809057295322418, -0.26247596740722656, 0.06304074078798294, 0.3637548089027405, -0.025464391335844994], [0.011901442892849445, -0.07773958891630173, 0.07581811398267746, -0.03471340239048004, -0.07961314171552658, 0.07546642422676086, 0.013298191130161285, -0.0063468762673437595, -0.06977599859237671, 0.025058837607502937, -0.06403499096632004, 0.08496633917093277, 0.01723409630358219, 0.01419321820139885, -0.07404226809740067, 0.01643524132668972, -0.0001301257434533909, 0.07645142823457718, -0.011538418009877205, 0.010745257139205933, -0.03588728606700897, -0.013643895275890827, 0.05026867613196373, -0.040023449808359146, -0.07588185369968414, -0.02514331415295601, -0.08177267014980316, 0.05857546254992485, -0.04284273460507393, -0.0027054981328547, -0.13222099840641022, 0.06847234070301056, 0.03696337342262268, 0.028345352038741112, 0.0009002066217362881], [0.018010493367910385, 0.01811199076473713, 0.008663238026201725, -0.010150285437703133, -0.015444871038198471, -0.05722304433584213, -0.025999048724770546, 0.009673354215919971, -0.014748472720384598, 0.031562838703393936, -0.0056333099491894245, -0.008609945885837078, -0.001455176854506135, -0.013462207280099392, 0.001328392536379397, -0.028053533285856247, -0.03495432436466217, -0.01753605343401432, 0.08626312762498856, 0.024516860023140907, 0.03773891180753708, 0.0057653263211250305, 0.00997905246913433, 0.011351936496794224, -0.03059796243906021, 0.022505218163132668, -0.02124357782304287, 0.013071122579276562, -0.00559184281155467, -0.007386039011180401, -0.03127215802669525, 0.03834157437086105, 0.03052418678998947, 0.049553267657756805, 0.0065629021264612675], [-0.04198731482028961, -0.031056541949510574, 0.07364892959594727, -0.0472162626683712, -0.009986629709601402, -0.013729213736951351, -0.04539608955383301, -0.009962991811335087, -0.014473309740424156, 0.003943721763789654, -0.009598784148693085, -0.10545088350772858, -0.01946384645998478, 0.01846720650792122, -0.0034398813731968403, 0.10053946822881699, -0.0695694088935852, 0.06855442374944687, 0.02247362770140171, -0.015462251380085945, -0.015873603522777557, -0.027390209957957268, -0.006290767807513475, -0.07862808555364609, 0.07384428381919861, 0.09269659966230392, 0.11019382625818253, 0.01221095398068428, 0.052883461117744446, 0.005002271384000778, 0.03678729757666588, -0.07304088026285172, -0.03165384754538536, 0.051992133259773254, -0.041101615875959396], [0.03256244212388992, -0.019284334033727646, 0.04776938259601593, -0.013506133109331131, -0.025745946913957596, 0.013351606205105782, 0.017884857952594757, -0.03208107873797417, 0.024721479043364525, 0.030981026589870453, -0.0066970293410122395, -0.018533021211624146, -0.010875876061618328, 0.06303952634334564, 0.02691039815545082, -0.014156593941152096, 0.055870600044727325, 0.017420461401343346, 0.05956663936376572, -0.05396716296672821, -0.06610095500946045, -0.0008519934490323067, -0.01657159812748432, -0.04798772186040878, 0.0032028313726186752, 0.043230071663856506, -0.03058585710823536, 0.03456337749958038, 0.009349176660180092, 0.06879046559333801, -0.012995991855859756, -0.033759184181690216, 0.03330708295106888, -0.05082685127854347, -0.0215954277664423], [0.007529635448008776, -0.037206534296274185, -0.10297638177871704, 0.006760448683053255, 0.01577121391892433, -0.031587738543748856, -0.011716391891241074, 0.005160755477845669, 0.020937401801347733, 0.13069424033164978, 0.06790903210639954, 0.08098457008600235, 0.05739975348114967, 0.014404288493096828, 0.00493898568674922, 0.049908217042684555, 0.07607632875442505, -0.013668997213244438, 0.05376596376299858, -0.0051040444523096085, 0.029934706166386604, -0.04838338866829872, 0.029068216681480408, 0.015278575010597706, -0.019788002595305443, 0.07343447208404541, -0.05368892848491669, -0.04369927942752838, 0.019540254026651382, -0.024975690990686417, -0.11885279417037964, 0.09003247320652008, 0.026303911581635475, 0.0022630789317190647, 0.001469514681957662], [0.10621952265501022, -0.07201395183801651, -0.11419160664081573, 0.11976280063390732, -0.08070161193609238, -0.07050065696239471, 0.09633088856935501, 0.015637481585144997, 0.053182803094387054, 0.002607421251013875, 0.03280898928642273, 0.08770685642957687, 0.1313612163066864, 0.009438936598598957, 0.018935013562440872, 0.057238273322582245, -0.07495471835136414, -0.10733097791671753, 0.04648171737790108, 0.08829738944768906, -0.021298080682754517, 0.021552683785557747, -0.09228049963712692, 0.031816475093364716, -0.05086468905210495, 0.01737602986395359, -0.032977327704429626, -0.06563601642847061, 0.04065680503845215, 0.0870315432548523, -0.1356898844242096, -0.12762990593910217, -0.05409599095582962, -0.042708151042461395, -0.0682477131485939], [0.007055582944303751, 0.014524029567837715, 0.05112605169415474, 0.005339222960174084, -0.0975833535194397, 0.12024322897195816, -0.02352774515748024, -0.010845489799976349, -0.06640171259641647, -0.02132519707083702, -0.03229270502924919, -0.06156928464770317, 0.1268051713705063, -0.08193450421094894, 0.08036411553621292, -0.0008861307869665325, -0.03773615136742592, 0.13491904735565186, 0.06519490480422974, -0.11346247047185898, 0.13174325227737427, -0.015261183492839336, -0.005926511250436306, 0.009238958358764648, 0.03611794859170914, 0.04339040070772171, 0.04494406282901764, 0.04116206616163254, -0.07540196180343628, 0.060566991567611694, -0.04840145632624626, -0.04496167227625847, -0.04433257877826691, -0.0440942719578743, 0.0029080375097692013], [0.042513273656368256, -0.08133629709482193, 0.07995685935020447, 0.07609692215919495, -0.033173155039548874, -0.024645330384373665, -0.15713046491146088, -0.11557808518409729, -0.09862993657588959, -0.026848504319787025, -0.02447821944952011, -0.07187172025442123, 0.10077819228172302, 0.10426432639360428, 0.07919979095458984, 0.12031523883342743, 0.007799524813890457, 0.11652744561433792, 0.0001504819665569812, -0.02812662534415722, 0.0019967136904597282, -0.10764168202877045, 0.0882134735584259, -0.0420277938246727, -0.05134955048561096, 0.015563049353659153, -0.10304267704486847, -0.09130582213401794, 0.046615127474069595, -0.09580741077661514, -0.06439273804426193, 0.06375537067651749, -0.06765028834342957, 0.06446784734725952, 0.08137765526771545], [-0.0002834873739629984, 0.04458359628915787, -0.018785813823342323, 0.0008842957904562354, 0.007362927310168743, 0.0081627881154418, -0.00544105377048254, -0.017100250348448753, -0.030254943296313286, -0.04035371541976929, -0.029048070311546326, -0.013309448957443237, -0.008502797223627567, 0.004464061465114355, -0.004881580360233784, 0.01303073763847351, 0.010985465720295906, -0.012486212886869907, -0.011505942791700363, 0.017581628635525703, -0.014983008615672588, 0.022877568379044533, 0.0031632641330361366, 0.008989933878183365, 0.026463212445378304, -0.0012310490710660815, 0.003052483545616269, -0.0023039348889142275, -0.017637085169553757, -0.010971245355904102, -0.024566587060689926, 0.028855107724666595, -0.0013862551422789693, -0.04545283317565918, -0.004501789342612028], [0.062125299125909805, 0.008526604622602463, 0.017788253724575043, 0.016329050064086914, -0.10354603081941605, 0.10795644670724869, 0.011650565080344677, -0.04712945967912674, -0.0042227208614349365, 0.03424134477972984, -0.0973433330655098, 0.09258566051721573, -0.019507933408021927, 0.04986709728837013, 0.0017972432542592287, -0.08899875730276108, -0.12173733860254288, -0.04876791313290596, 0.03337295725941658, 0.026890572160482407, 0.02988748624920845, 0.05698201060295105, -0.0336042158305645, -0.029642870649695396, 0.031917862594127655, -0.08560042083263397, 0.0005781107465736568, -0.030782483518123627, -0.05244076997041702, 0.0659041777253151, 0.07335662841796875, -0.12007111310958862, 0.03196926414966583, -0.0655127540230751, -0.0362900048494339], [-0.06002066656947136, -0.07997890561819077, 0.06741391867399216, -0.007707281969487667, -0.10113789141178131, 0.02672322653234005, -0.013894978910684586, 0.026205135509371758, -0.10566885769367218, 0.05247482284903526, -0.01041642390191555, 0.027323149144649506, -0.08832770586013794, 0.05419513210654259, 0.0007170341559685767, -0.07081133127212524, 0.09149288386106491, -0.1112503632903099, -0.08153319358825684, -0.02827993594110012, -0.025296702980995178, 0.05415542051196098, 0.08781003206968307, -0.09969402849674225, -0.09650806337594986, -0.0014198474818840623, 0.14494775235652924, -0.02335950918495655, -0.11610499769449234, 0.06127762421965599, 0.029354866594076157, 0.06434306502342224, 0.01121456641703844, -0.0673380047082901, 0.04199175536632538], [0.0199858658015728, 0.011807224713265896, 0.06512710452079773, -0.003812379902228713, 0.01249430887401104, 0.03763395920395851, -0.009931598789989948, -0.01875968836247921, 0.0141723258420825, 0.03133557364344597, -0.01779630221426487, 0.0013917724136263132, -0.02554940991103649, 0.04050542786717415, -0.01240936852991581, 0.043688926845788956, -0.049458399415016174, 0.011708076111972332, -0.033296745270490646, -0.021123982965946198, -0.06373970955610275, -0.03300349786877632, -0.009565464220941067, 0.009208130650222301, 0.03391675651073456, 0.01728227362036705, 0.06816671788692474, 0.007516540586948395, -0.008576705120503902, 0.04830873757600784, 0.019105082377791405, 0.01726219616830349, -0.04377411678433418, 0.01725720427930355, 0.005481156520545483], [0.03557407110929489, -0.08007276058197021, -0.017593756318092346, -0.01932653598487377, -0.06960093975067139, 0.024802271276712418, 0.040878213942050934, -0.023992853239178658, -0.0072411829605698586, 0.011670244857668877, -0.023819580674171448, 0.11133542656898499, 0.031137339770793915, -0.10335525125265121, -0.07209984958171844, 0.06253547221422195, 0.07021582126617432, 0.023716764524579048, 0.05178384110331535, -0.024302134290337563, -0.03398331627249718, -0.026729697361588478, 0.00997497420758009, -0.07167495042085648, -0.10223085433244705, -0.07965315133333206, 0.02518765814602375, 0.020787041634321213, -0.08655064553022385, 0.011216532438993454, -0.10270979255437851, -0.050761133432388306, -0.0026556854136288166, -0.011981124989688396, -0.007954162545502186], [-0.022581344470381737, -0.0037806560285389423, -0.035869840532541275, -0.020703257992863655, -0.030539700761437416, 0.010134531185030937, 0.0007036170572973788, -0.051975857466459274, -0.04087957739830017, -0.03476562723517418, -0.0018832923378795385, -0.005239896476268768, 0.0026915541384369135, -0.0012272398453205824, -0.0020744253415614367, 0.016580602154135704, -0.028431642800569534, 0.01645931974053383, 0.03509622439742088, 0.0050392006523907185, 0.05456959456205368, 0.008710081689059734, 0.01364536490291357, -0.008016608655452728, 0.042735468596220016, -0.04194114729762077, 0.006995202042162418, 0.008222080767154694, 0.003721165470778942, 0.020384596660733223, -0.012116562575101852, 0.009821122512221336, -0.013770774006843567, -0.0582563616335392, -0.0005148658528923988], [0.006209711544215679, 0.07310929149389267, 0.08995519578456879, 0.03954407572746277, 0.006616256665438414, -0.07314125448465347, 0.03780115768313408, 0.0013848699163645506, -0.023576976731419563, -0.02084958925843239, -0.029786475002765656, -0.04791641607880592, -0.09219525009393692, -0.007459491025656462, -0.0214309711009264, 0.00843235943466425, 0.013212105259299278, -0.015199951827526093, 0.011860067956149578, -0.020060671493411064, -0.025430483743548393, -0.04033447057008743, -0.023345572873950005, 0.025859881192445755, 0.013816440477967262, 0.005643308628350496, -0.011361649259924889, 0.04315981641411781, 0.04163963720202446, -0.036380961537361145, 0.005530646536499262, 0.04459352046251297, 0.00446904543787241, 0.012582983821630478, -0.06145322695374489], [0.014071998186409473, -0.008047640323638916, 0.008388744667172432, -0.015713397413492203, -0.02872973307967186, 0.043841611593961716, -0.003866200800985098, -0.023170989006757736, 0.03228365629911423, 0.014185532927513123, -0.02485908381640911, 0.05620803311467171, -0.000649078341666609, 0.003110466292127967, -0.00927014835178852, 0.019502980634570122, -0.046990323811769485, -0.017889847978949547, -0.025337940081954002, 0.018078360706567764, 0.053109828382730484, -0.0009477782878093421, -0.019081177189946175, -0.012753717601299286, -0.03570769727230072, -0.02167518436908722, -0.030780723318457603, 0.00507697556167841, -0.027023376896977425, 0.0368405245244503, -0.0030424760188907385, -0.02403217926621437, -0.005819229409098625, -0.019735652953386307, 0.00470842095091939], [0.008074039593338966, 0.03544154763221741, 0.03196433186531067, -0.007321670185774565, -0.035680755972862244, 0.0020704118069261312, 0.0017664312617853284, -0.03328074887394905, -0.026982618495821953, 0.013686688616871834, 0.004880248103290796, -0.03286963701248169, -0.011093818582594395, -0.03261542320251465, -0.010129833593964577, 0.05205133557319641, 0.0247505996376276, -0.0018486694898456335, 0.01537257805466652, 0.02364342287182808, 0.0006945959758013487, -0.023530077189207077, 0.015417423099279404, 0.0018956282874569297, 0.04991721734404564, 0.019862258806824684, 0.027218712493777275, 0.03403167799115181, 0.02050730213522911, 0.04472062364220619, 0.017240174114704132, -0.004998166114091873, 0.046970710158348083, 0.021801384165883064, -0.010540329851210117], [-0.04818267375230789, -0.05063076689839363, 0.0416235588490963, -0.0019893059507012367, -0.011380450800061226, -0.021850641816854477, 0.037955816835165024, 0.001969295786693692, 0.005081911105662584, -0.03502281382679939, -0.009721322916448116, -0.01260952278971672, -0.012420943938195705, 0.019323309883475304, 0.017677534371614456, -0.056485697627067566, -0.07316729426383972, 0.015121600590646267, 0.050273746252059937, -0.039089351892471313, -0.04030568525195122, 0.023690521717071533, -0.04247438907623291, -0.022405235096812248, -0.04776022955775261, 0.05215141922235489, 0.05845499783754349, 0.009951780550181866, 0.05607231333851814, -0.045354604721069336, 0.013127570040524006, -0.05731036141514778, 0.03846822306513786, 0.13982073962688446, 0.015250458382070065], [-0.020843153819441795, 0.05006925016641617, -0.04050053283572197, -0.025888700038194656, -0.032685425132513046, 0.007258748169988394, 0.005096623674035072, -0.02538658306002617, -0.0005445822607725859, -0.0662049800157547, 0.025494888424873352, -0.009742818772792816, 0.02014465443789959, -0.01725480705499649, -0.012292963452637196, -0.03662693127989769, 0.029151786118745804, 0.005632363259792328, -0.005922719370573759, 0.01851567253470421, -0.03435388579964638, 0.019017037004232407, 0.01875670999288559, -0.04052123427391052, 0.014220735058188438, -0.016735170036554337, 0.018445085734128952, -0.012152832001447678, 0.014021750539541245, 0.01639186590909958, -0.04955684393644333, 0.03842981159687042, -0.0019406037172302604, -0.05493719503283501, -0.03414253890514374], [-0.02748488448560238, -0.05269409343600273, -0.00338655523955822, -0.04018743708729744, -0.020205305889248848, -0.06045016273856163, -0.025041228160262108, 0.017955198884010315, -0.03343670815229416, -0.010572781786322594, -0.004723638296127319, 0.06660262495279312, 0.06995929032564163, 0.017998529598116875, -0.059353359043598175, -0.004186110571026802, -0.07841599732637405, 0.022230762988328934, 0.05074666813015938, -0.0341126024723053, -0.022234812378883362, -0.016997965052723885, -0.0044632270000875, 0.010450406931340694, 0.004934301134198904, -0.03239104524254799, 0.006806958466768265, -0.0019613252952694893, 0.07992123812437057, -0.010536096058785915, 0.0007096135523170233, -0.03976672887802124, -0.04825260117650032, -0.059702903032302856, 0.001192582189105451], [0.018863942474126816, 0.0015371674671769142, 0.024027913808822632, 0.03522796183824539, -0.018995855003595352, 0.0827019214630127, -0.024461688473820686, -0.0036057026591151953, 0.00616259453818202, -0.028892505913972855, -0.0166312288492918, 0.004045500885695219, 0.025041325017809868, 0.08232986927032471, -0.02209281176328659, 0.01931767724454403, 0.08930011838674545, 0.03750458359718323, -0.07821124792098999, -0.0739007368683815, 0.09117044508457184, -0.009630759246647358, 0.02603045664727688, 0.0695456936955452, -0.06323179602622986, 0.06463022530078888, 0.026386579498648643, 0.03648439794778824, -0.061054255813360214, 0.01751849614083767, 0.03996571898460388, -0.005223930813372135, 0.029640905559062958, 0.21644428372383118, 0.0325210765004158], [0.0426018051803112, -0.08804131299257278, 0.018546808511018753, -0.07401089370250702, -0.11229152977466583, 0.0042973109520971775, -0.02250862494111061, 0.02002258412539959, -0.01749265380203724, 0.04399523511528969, -0.05451815202832222, -0.03857192397117615, 0.013632413931190968, 0.019634926691651344, 0.07115428149700165, 0.05214320868253708, 0.020765723660588264, -0.1592845618724823, 0.1185135692358017, 0.029402362182736397, 0.1412016898393631, 0.048723164945840836, -0.05321037396788597, -0.08811265975236893, -0.09509842842817307, 0.013151092454791069, -0.07039231061935425, -0.022946981713175774, 0.03250396251678467, -0.010545548982918262, 0.01812155358493328, 0.05558105558156967, 0.045844096690416336, 0.04840915650129318, -0.074656642973423], [0.02140449360013008, 0.026827318593859673, -0.02585669979453087, -0.08743634074926376, -4.918469858239405e-05, -0.0033872402273118496, -0.013331100344657898, 0.017298758029937744, 0.0030574121046811342, 0.046782881021499634, -0.04953179135918617, 0.07439929991960526, -0.07943220436573029, -0.03902260214090347, -0.01520262286067009, -0.052162013947963715, 0.025918571278452873, 0.03294115513563156, 0.00953998789191246, 0.034429557621479034, -0.00931291002780199, 0.07468906790018082, -0.002319485414773226, 0.020130105316638947, 0.007791444659233093, 0.0527370311319828, 0.005912246648222208, 0.00545890536159277, -0.09229802340269089, 0.036157287657260895, 0.0351007841527462, -0.10416662693023682, 0.019806155934929848, -0.04488428682088852, 0.006463364232331514], [0.015146470628678799, 0.003461001440882683, 0.04020531103014946, 0.050862159579992294, 0.026426857337355614, 0.011105997487902641, 0.00724073825404048, 0.02286316268146038, -0.02222592569887638, 0.007645797915756702, 0.002507183002308011, 0.008720163255929947, -0.006060927174985409, -0.009294621646404266, -0.013411890715360641, 0.00414861598983407, -0.02024141699075699, -0.00463350024074316, 0.011573730036616325, -0.004617870319634676, -0.0453391969203949, 0.012964985333383083, 0.012886390089988708, 0.014414102770388126, 0.02382161095738411, 0.012013668194413185, 0.05106112360954285, -0.0030211759731173515, 0.010698826983571053, 0.03362569212913513, 0.02684757672250271, -0.008681199513375759, 0.002195545006543398, -0.005433160346001387, 0.015852391719818115]], "b1": [-0.02657056786119938, 0.018293417990207672, 0.04702022299170494, -0.09372348338365555, -0.0346490815281868, 0.04004655405879021, -0.0514565072953701, 0.04597396403551102, 0.08440195769071579, 0.0109394621104002, -0.05587777495384216, 0.11227864772081375, -0.16473180055618286, -0.10071661323308945, -0.012582777068018913, -0.023605434224009514, 0.01630675420165062, -0.06410867720842361, 0.03507033362984657, -0.0028266787994652987, 0.05539485067129135, -0.02580646611750126, -0.11701886355876923, -0.13115477561950684, 0.0012418539263308048, 0.027225228026509285, -0.0061391438357532024, 0.069526307284832, -0.010837722569704056, 0.01493019051849842, -0.017808107659220695, -0.03531485050916672, -0.028121601790189743, 0.04680578038096428, 0.07041618227958679, -0.023985067382454872, -0.06643743813037872, -0.14453549683094025, -0.005861053708940744, -0.0012323074042797089, 0.04770682752132416, 0.009054734371602535, 0.009920467622578144, 0.010574059560894966, 0.030055372044444084, -0.09051842242479324, 0.043411705642938614, -0.05092804506421089, -0.058259692043066025, -0.01601175218820572, -0.19098630547523499, 0.029925992712378502, -0.21239961683750153, -0.013057293370366096, 0.010152309201657772, -0.10541041195392609, 0.03745489940047264, -0.21417152881622314, -0.01280887145549059, 0.03294669836759567, -0.05434618145227432, -0.037557072937488556, 0.006373331882059574, -0.0108898701146245, -0.03796735405921936, 0.060660749673843384, 0.07257255911827087, 0.013979737646877766, 0.06426524370908737, 0.015534530393779278, 0.006204897537827492, -0.26046615839004517, -0.017056414857506752, 0.014298362657427788, -0.025516191497445107, -0.025455225259065628, 0.045210737735033035, -0.10429349541664124, -0.003248270135372877, -0.04512500390410423, -0.00715846149250865, -0.005267672706395388, -0.030634697526693344, 0.047511082142591476, -0.05151626467704773, -0.022107146680355072, 0.048424892127513885, 0.01271209679543972, 0.03986179083585739, -0.06612739711999893, 0.004624833352863789, -0.016045058146119118, -0.06046745926141739, -0.041580989956855774, 0.030932515859603882, 0.03018517605960369], "W2": [[0.011937740258872509, 0.028527356684207916, -0.01393407117575407, 0.013877292163670063, -0.027039583772420883, 0.00929348450154066, 0.008392678573727608, 0.052922070026397705, -0.0037033986300230026, -0.005686014890670776, 0.028433596715331078, 0.02844364196062088, 0.011265904642641544, -0.007982935756444931, 0.006783575285226107, 0.030670031905174255, 0.019884463399648666, 0.016040433198213577, 0.019009966403245926, 0.00714532844722271, 0.015305517241358757, 0.008663696236908436, 0.055446237325668335, 0.01910504512488842, 0.029735082760453224, 0.025623850524425507, 0.001996342558413744, 0.045435842126607895, -0.009891372174024582, 0.022361353039741516, 0.020719952881336212, -0.011080059222877026, 0.019057197496294975, 0.024608192965388298, 0.043184779584407806, 0.025822822004556656, 0.0322185680270195, 0.04218103364109993, -0.0011907076695933938, -0.00499315932393074, 0.024072401225566864, -0.027925672009587288, 0.022346584126353264, -0.01873352937400341, -0.010182362049818039, 0.008332934230566025, -0.0065026720985770226, -0.019321464002132416, 0.024478929117321968, 0.011839693412184715, -0.017894543707370758, 0.015150551684200764, -0.00019771585357375443, 0.010272197425365448, 0.007625537924468517, 0.036335695534944534, 0.008836910128593445, 0.004719678312540054, 0.025706639513373375, 0.002288002520799637, -0.017803704366087914, 0.037315670400857925, -0.0017702660989016294, 0.019735857844352722, -0.016900477930903435, -0.01730984076857567, -0.01744471862912178, 0.006063132081180811, -0.00010716747783590108, -0.006868212018162012, -0.01796089857816696, 0.030375991016626358, 0.030554484575986862, 0.008414531126618385, -0.004040092695504427, -0.009087461046874523, -0.010925588198006153, 0.030472170561552048, 0.005405171774327755, 0.03705726936459541, -0.0041261715814471245, 0.026254212483763695, 0.018695440143346786, -0.010871697217226028, 0.034675806760787964, 0.0022079807240515947, 0.007353180553764105, 0.029737723991274834, -0.004610653035342693, 0.00658548204228282, -0.001826438121497631, -0.009363231249153614, 0.01172899641096592, 0.04925299063324928, 0.0007324849721044302, -0.004960196558386087], [0.007305508945137262, 0.033978015184402466, 0.009694893844425678, 0.024882376194000244, 0.012347414158284664, 0.0022091581486165524, -0.010875645093619823, 0.005747159011662006, 0.051163095980882645, 0.06478375196456909, -0.003235507756471634, 0.0016577704809606075, -0.02794002555310726, 0.008747018873691559, 0.007295831106603146, -0.012691864743828773, 0.02363336645066738, -0.0024023959413170815, -0.008428185246884823, -0.010674715042114258, 0.006849529687315226, 0.00803630892187357, -0.06312098354101181, -0.004974565468728542, 0.022485340014100075, -0.05902751162648201, -0.0006832116632722318, 0.036171332001686096, -0.0050040376372635365, 3.6167853977531195e-05, 0.015151996165513992, 0.009779393672943115, -0.02107207663357258, 0.0011503584682941437, -0.0009257793426513672, -0.06733577698469162, -0.00630998844280839, -0.09129790961742401, -0.03508446365594864, -0.026636993512511253, -0.006557281129062176, -0.001596108078956604, 0.004689551889896393, 0.026031900197267532, -0.013640133664011955, -0.014522929675877094, 0.0037184027023613453, 0.00048631109530106187, 0.0355207696557045, -0.024290332570672035, -0.039372898638248444, 0.008582542650401592, -0.010252299718558788, 0.006287454627454281, -0.005791814532130957, -0.014980088919401169, -0.0505385585129261, 0.020848512649536133, -0.023701751604676247, -0.024666352197527885, 0.01357949711382389, -0.016597695648670197, 0.0030408650636672974, -0.0059687914326786995, -0.014037524349987507, -0.0036934318486601114, 0.0006363517022691667, 0.0008219219744205475, 0.0032623393926769495, -0.020434532314538956, -0.021041713654994965, 0.008560236543416977, 0.004583208821713924, 1.7356896933051758e-05, -0.002351023955270648, -0.002266343915835023, 0.0052683730609714985, 0.039041198790073395, 0.0100094648078084, -0.0667479857802391, -0.007865042425692081, 0.01546347327530384, 0.0018977979198098183, -0.01311465073376894, 0.005651355255395174, 0.006622195243835449, 0.0002994315873365849, 0.03796153888106346, 0.0014852675376459956, -0.028747811913490295, -0.008185113780200481, 0.0074818553403019905, -0.04189460724592209, 0.0010196065995842218, -0.006849255412817001, 0.0036902360152453184], [0.050414182245731354, 0.0067604887299239635, 0.048946812748909, 0.09894455969333649, 0.0041981590911746025, -0.003775714198127389, -0.06333500146865845, -0.051117584109306335, -0.0264173224568367, 0.019552411511540413, 0.032017841935157776, 0.014716451987624168, 0.11267440021038055, 0.028577975928783417, 0.04346252605319023, -0.0448400117456913, -0.10641811043024063, -0.012019725516438484, -0.023402761667966843, -0.024529559537768364, -0.04723278433084488, -0.0302024744451046, -0.08178789168596268, -0.00032243484747596085, 0.056462325155735016, 0.05454225465655327, 0.009051471017301083, -0.04378707706928253, 0.07667244970798492, 0.042134176939725876, -0.04824084788560867, 0.06004367768764496, -0.03211528807878494, 0.017168845981359482, -0.019921468570828438, 0.026980997994542122, -0.012412085197865963, -0.0715455710887909, -0.0357867106795311, 0.10081138461828232, -0.11521653831005096, 0.055603064596652985, -0.03528619930148125, 0.014509262517094612, -0.0007778461440466344, 0.12881392240524292, 0.02478840947151184, 0.02157481387257576, -0.015846557915210724, -0.0006461285520344973, 0.1538732945919037, -0.09901638329029083, 0.1651468276977539, -0.043879203498363495, 0.033466484397649765, 0.00652520265430212, -0.08137397468090057, 0.05153454467654228, -0.03237808868288994, 0.020771197974681854, -0.031966473907232285, -0.030054155737161636, -0.006847071461379528, 0.02905445545911789, -0.0032448815181851387, 0.04758870601654053, 0.03373950719833374, -0.026229338720440865, 0.018838336691260338, 0.007007951382547617, 0.061020877212285995, -0.01612645760178566, -0.007290230598300695, -0.02165338024497032, -0.05705631524324417, 0.03167196363210678, -0.07172352820634842, -0.0017325942171737552, -0.05832016095519066, -0.022678719833493233, 0.033891934901475906, 0.056052811443805695, 0.018416618928313255, -0.05402764678001404, 0.11289913952350616, 0.05238502472639084, -0.044662538915872574, -0.02772507444024086, -0.007103997748345137, 0.047521986067295074, 0.04631734639406204, 0.0170062854886055, -0.003049002029001713, 0.009038135409355164, -0.001051914063282311, -0.015585399232804775], [-0.009377241134643555, -0.023093173280358315, 0.007024412043392658, -0.1272214651107788, 0.004052654840052128, 0.003464403562247753, 0.009301044046878815, -0.0616895817220211, 0.03201144188642502, 0.08507410436868668, -0.05783876031637192, -0.028443660587072372, -0.0339755117893219, -0.11653584241867065, 0.01621241308748722, -0.028062190860509872, 0.015246053226292133, -0.05229591205716133, 0.027854211628437042, 0.0026863962411880493, 0.035284873098134995, -0.09262272715568542, 0.07546427100896835, -0.05603257194161415, -0.06809620559215546, 0.007894473150372505, 0.0016203549457713962, 0.015523931942880154, -0.017797548323869705, 0.0386020690202713, 0.009463967755436897, 0.025985537096858025, 0.027713805437088013, -0.004698468372225761, 0.0039725033566355705, 0.0073129842057824135, -0.021717339754104614, 0.11960276961326599, 0.008624233305454254, -0.023063138127326965, 0.026179146021604538, -0.006518911570310593, 0.047207191586494446, -0.06654021888971329, 0.04233627766370773, -0.05223732441663742, -0.001063479227013886, -0.026523079723119736, -0.02127879485487938, 0.019626924768090248, -0.09109309315681458, 0.01921885274350643, -0.07342498749494553, 0.04320009797811508, -0.03617766126990318, 0.04113207384943962, -0.006679334677755833, -0.06532155722379684, 0.027344977483153343, -0.0029785691294819117, -0.035217516124248505, 0.03814053162932396, -0.012202623300254345, 0.00806780718266964, 0.012005161494016647, 0.035884130746126175, -0.04707314819097519, 0.013380717486143112, 0.04242485389113426, 0.03193508833646774, -0.011057963594794273, -0.16491778194904327, -0.0017425690311938524, 0.014708595350384712, 0.012304330244660378, 0.0024823963176459074, 0.08586450666189194, 0.02000909112393856, 0.00712190056219697, 0.03473528474569321, -0.03440289944410324, 0.03340388461947441, 0.026334421709179878, 0.02761552855372429, 0.002543703420087695, 0.030531637370586395, 0.0233699269592762, 0.046006593853235245, 0.001892798114567995, 0.012350717559456825, -0.023900892585515976, -0.02121785655617714, 0.008749337866902351, -0.01451210305094719, -0.04261046275496483, 0.01334652304649353], [-0.010732256807386875, -0.03840737044811249, -0.026648633182048798, 0.06912083923816681, 0.09512533992528915, -0.02188314124941826, -0.0475153885781765, 0.016138287261128426, -0.04414362460374832, -0.07837291806936264, 0.042584121227264404, -0.022153476253151894, 0.08096841722726822, 0.061391834169626236, 0.038015350699424744, -0.057773660868406296, 0.012586616910994053, -0.031495317816734314, -0.04463537409901619, 0.026483092457056046, -0.09480445086956024, 0.04238992929458618, -0.1309986710548401, 0.11069633811712265, -0.07671266049146652, 0.02938912995159626, -0.0037058552261441946, 0.046649251133203506, -0.02878996729850769, 0.0026998999528586864, -0.04360198229551315, 0.03234925493597984, 0.007794893812388182, 0.03493475914001465, -0.0491141639649868, 0.021608740091323853, -0.013892387971282005, -0.06654179841279984, 0.016938019543886185, -0.025754373520612717, -0.004739510826766491, 0.013876577839255333, 0.01082827989012003, 0.04232492297887802, 0.03015741892158985, 0.07125405222177505, -0.04882119223475456, 0.03635692596435547, 0.023876486346125603, 0.04630642384290695, 0.054734908044338226, -0.07157387584447861, 0.09590892493724823, -0.008038667030632496, -9.872378723230213e-05, -0.10324057191610336, -0.04431767761707306, 0.0021363156847655773, 0.0016140714287757874, -0.018257681280374527, -0.012146198190748692, 0.07006467133760452, 0.002014385536313057, -0.031560610979795456, 0.04868632182478905, -0.062204983085393906, 0.017356231808662415, -0.011514225043356419, 0.04299717769026756, -0.02921825461089611, 0.015517719089984894, 0.09684118628501892, -0.017189787700772285, 0.0391961969435215, 0.0014033106854185462, 0.01154416799545288, -0.033895161002874374, -0.056067775934934616, 0.030584892258048058, 0.01592826656997204, 0.02429993636906147, -0.03145812451839447, -0.08459474891424179, -0.004509232938289642, 0.013088986277580261, 0.006270574871450663, -0.06699413806200027, -0.020443212240934372, -0.03636538237333298, -0.005708146840333939, -0.012812400236725807, -0.04137225076556206, 0.08669771254062653, 0.048398859798908234, 0.016287699341773987, -0.033998988568782806], [0.0019000008469447494, 0.00839842576533556, -0.0010009180987253785, 0.004230953752994537, -0.005045271944254637, 0.0038843390066176653, 0.0024904452729970217, 0.004417093936353922, 0.006401169579476118, 0.011523472145199776, 0.00343010644428432, 0.005173237528651953, -0.0025661943946033716, -0.0004134552727919072, 0.00041574722854420543, 0.003197902347892523, 0.0014906056458130479, 0.0022374398540705442, 0.0019152015447616577, -0.0004129556764382869, 0.0047439211048185825, 0.001744731911458075, -0.00097727554384619, -0.007628996390849352, 0.002405493753030896, -0.0006443369784392416, -4.936755067319609e-05, 0.015215606428682804, -0.003028023988008499, 0.0007577837095595896, 0.0032100083772093058, -0.0016643176786601543, -0.0013787912903353572, 0.0008746693492867053, 0.003734549740329385, -0.006134328432381153, 0.003238935489207506, -0.0030394531786441803, -0.002313767559826374, -0.002735845046117902, 0.002065262757241726, -0.003978873137384653, 0.0012967156944796443, -0.0005278665339574218, -0.001455610035918653, -0.0023473745677620173, -0.0002977627154905349, -0.0019424166530370712, 0.0039459834806621075, -0.0005669069360010326, -0.00475382199510932, 0.001014017965644598, -0.002308139344677329, 0.001368653029203415, 0.0005575657123699784, 0.003571218578144908, -0.006985735148191452, 0.0027605118229985237, -0.0005777888000011444, -0.0012323586270213127, 0.0018969073425978422, 0.007605884224176407, 2.354179378016852e-05, 0.0013758332934230566, -0.002671909984201193, -0.0029887757264077663, -0.004428235813975334, 0.0009885766776278615, 0.0003218736674170941, -0.005817308556288481, -0.002021330874413252, 0.016177693381905556, 0.002128257416188717, -8.106720088107977e-06, -0.0052981069311499596, -0.0012244891840964556, 0.0022458056919276714, 0.006796665955334902, -0.007726030889898539, -0.009350351989269257, -0.0004345711786299944, 0.0008796944748610258, 0.0019445903599262238, -0.0008926170994527638, 0.0029117842204868793, -0.0007876619347371161, 0.0006408840417861938, 0.0004950115107931197, -0.0003378592955414206, -0.0026780935004353523, -0.0012197974137961864, -0.00044954067561775446, -0.004126688931137323, 0.004548612516373396, -0.00013461659546010196, -0.0007161354878917336], [-0.06267224997282028, 0.06135183572769165, 0.005321436561644077, -0.10195471346378326, -0.09781233221292496, 0.04182525724172592, -0.049454011023044586, -0.02670338749885559, -0.020490210503339767, 0.07075503468513489, -0.06235308200120926, -0.07473641633987427, -0.009170816279947758, -0.09169740229845047, 0.002933802315965295, 0.01626969315111637, 0.048916175961494446, 0.037131864577531815, 0.007296653930097818, 0.0417775958776474, 0.036161039024591446, -0.08148497343063354, 0.06190925091505051, -0.03808800131082535, -0.025140995159745216, -0.011215615086257458, 0.0024937952402979136, 0.016522563993930817, -0.01205239724367857, -0.0036706882528960705, 0.029444245621562004, -0.04014497622847557, 0.04581056907773018, 0.04362049698829651, 0.01918145827949047, 0.03373433277010918, 0.02510324865579605, 0.046415600925683975, 0.024009648710489273, 0.030182113870978355, 0.08759605139493942, -0.012680241838097572, -0.04827561601996422, -0.07249748706817627, 0.061019882559776306, -0.09616722911596298, -0.033523641526699066, 3.099736204603687e-05, -0.027855662629008293, 0.01341419667005539, -0.03574173524975777, 0.05948443338274956, -0.10085596144199371, -0.0334150604903698, -0.06138165667653084, -0.03149113431572914, 0.0154330525547266, -0.07911358028650284, 0.046533580869436264, -0.0405384786427021, -0.03409326821565628, -0.011485833674669266, -0.0015194992301985621, 0.0036927838809788227, -0.023347359150648117, 0.057388100773096085, -0.02117289789021015, -0.01608847826719284, 0.0395413339138031, 0.08360971510410309, 0.050160061568021774, -0.1307407170534134, -0.025035984814167023, 0.04888743534684181, -0.010231008753180504, -0.03471032530069351, 0.033252764493227005, -0.040021561086177826, 0.0541713684797287, 0.024230359122157097, 0.015595601871609688, 0.046846725046634674, -0.03209232911467552, 0.03307780995965004, -0.012745309621095657, 0.005961661692708731, -0.038231201469898224, 0.011678674258291721, 0.04585396870970726, -0.04347530007362366, -0.03174325078725815, -0.007432075217366219, -0.008466292172670364, -0.014277303591370583, -0.08181546628475189, 0.012069806456565857], [0.03869572654366493, -0.057219844311475754, -0.04241357371211052, 0.06138001009821892, -0.09800591319799423, -0.004417827352881432, 0.03390810266137123, -0.059942688792943954, -0.031254202127456665, -0.05960780382156372, -0.040122952312231064, 0.011149697005748749, 0.08826149255037308, -0.09572502225637436, -0.017740290611982346, 0.051947221159935, -0.05244424566626549, 0.08845090866088867, 0.028864292427897453, 0.023805610835552216, 0.02212856337428093, -0.0023627313785254955, 0.14263705909252167, -0.008296926505863667, -0.014763030223548412, 0.056089866906404495, 0.008993220515549183, 0.007946835830807686, -0.04253005608916283, -0.03886248171329498, -0.022729620337486267, -0.03998473286628723, 0.04099585488438606, -0.031069576740264893, 0.04974077269434929, 0.06271746009588242, 0.06198600307106972, 0.17658844590187073, 0.07318009436130524, -0.012668980285525322, 0.041338205337524414, -0.05669702589511871, -0.05369696393609047, -0.07137402147054672, -0.0002386315172770992, 0.002442333148792386, -0.006507313344627619, -0.05844256281852722, -0.03200573846697807, 0.02571866288781166, 0.0014628085773438215, 0.09432002902030945, -0.00360811036080122, 0.020335590466856956, 0.016084956005215645, 0.06615854799747467, 0.04860999062657356, 0.06181591749191284, 0.027873970568180084, 0.01988086849451065, -0.043309763073921204, -0.009444175288081169, -0.0009134687134064734, 0.011807416565716267, -0.02596898376941681, -0.04485708475112915, -0.025024278089404106, 0.011633845046162605, 0.0032975254580378532, -0.07284098863601685, -0.013346804305911064, 0.06450815498828888, -0.031192511320114136, -0.005507792811840773, 0.022609231993556023, -0.03432983532547951, 0.021949319168925285, 0.012184490449726582, -0.06121694669127464, 0.07102302461862564, -0.005046619102358818, 0.034211665391922, 0.02969709038734436, -0.03202914446592331, -0.0665552169084549, -0.014928809367120266, 0.017176462337374687, -0.009087933227419853, -0.018464861437678337, 0.030943971127271652, -0.023857276886701584, -0.03408938646316528, 0.01851283386349678, 0.06970036774873734, -0.0029106864240020514, -0.014177471399307251], [0.015694640576839447, -0.08482365310192108, -0.038400400429964066, 0.05181032046675682, 0.036548785865306854, -0.00452927453443408, -0.05523741990327835, 0.00933948252350092, -0.053395237773656845, -0.07886148989200592, 0.04648875445127487, -0.0940035954117775, 0.09604954719543457, 0.07475697994232178, 0.021229982376098633, -0.06286928802728653, 0.013087931089103222, -0.057950522750616074, -0.0538262203335762, -0.004915760830044746, -0.06449475139379501, 0.009225372225046158, -0.11426576972007751, 0.10919338464736938, 0.0624886155128479, -0.041584551334381104, 0.012032483704388142, -0.014678120613098145, 0.00010264769662171602, -0.02841358073055744, -0.03432769700884819, 0.00795226451009512, 0.05504104495048523, 0.005049708299338818, -0.019008401781320572, 0.052348777651786804, -0.02832666039466858, -0.08958754688501358, 0.05535702034831047, 0.0884641706943512, -0.0007577784708701074, 0.006842193193733692, 0.012749996967613697, 0.00844651460647583, -0.0005027101724408567, 0.07182873040437698, -0.01820632442831993, 0.00893075205385685, -0.02628837525844574, 0.011666986159980297, 0.10499120503664017, -0.06507858633995056, 0.04885707050561905, -0.046161774545907974, -0.0033209773246198893, 0.0025432955008000135, -0.061175260692834854, 0.0924774631857872, 0.033268075436353683, -0.0007873488357290626, 0.08152301609516144, -0.06361474841833115, 0.011426444165408611, 0.026281055063009262, 0.022352244704961777, -0.016681117936968803, 0.0576665997505188, -0.0364280566573143, -0.02919834479689598, 0.06966816633939743, 0.02283487468957901, 0.024467499926686287, 0.038834672421216965, -0.01708287000656128, 0.06954339891672134, 0.01726815663278103, -0.06633637845516205, -0.02788945659995079, 0.07591219246387482, 0.031000664457678795, -0.0060567003674805164, -0.05037768930196762, -0.07652465999126434, 0.007021704223006964, 0.02501458302140236, 0.018004778772592545, -0.04480541869997978, -0.01617911085486412, -0.017667435109615326, 0.02867146022617817, -0.01575505919754505, -0.03319573029875755, 0.09010010957717896, -0.0489373542368412, 0.017836475744843483, 0.0011015678755939007], [0.07502549886703491, 0.08794312179088593, 0.03712701052427292, 0.07512844353914261, -0.10451004654169083, 0.06409808248281479, 0.04466792568564415, 0.09668098390102386, 0.06389252841472626, 0.0033173460979014635, 0.03262093663215637, 0.046296849846839905, 0.03665344789624214, -0.045721422880887985, 0.03518237546086311, 0.06583496928215027, 0.09942364692687988, 0.043413229286670685, 0.03335840255022049, -0.007549562957137823, 0.10286310315132141, 0.012170197442173958, 0.16177165508270264, -0.008226312696933746, -0.020104655995965004, 0.10383676737546921, -0.006012251600623131, 0.06018610671162605, -0.06695351749658585, 0.04563462734222412, 0.022126607596874237, -0.05935816094279289, 0.09024077653884888, -0.013976184651255608, 0.01800285466015339, 0.062148917466402054, 0.03786418214440346, 0.09050865471363068, 0.049843680113554, -0.03778590261936188, 0.017030656337738037, -0.11019793897867203, 0.04735846817493439, -0.10582321882247925, -0.0056084454990923405, 0.04030001908540726, -0.042607419192790985, -0.036548711359500885, 0.018216758966445923, 0.004169669933617115, 0.008769404143095016, 0.03033096343278885, -0.02078934758901596, 0.004216326400637627, 0.0065344455651938915, 0.09098093956708908, -0.06008036062121391, 0.047954924404621124, 0.027916759252548218, 0.048991404473781586, 0.07611559331417084, -0.05191807448863983, 0.021173257380723953, 0.05639919638633728, -0.042614929378032684, -0.028989465907216072, -0.08962327241897583, 0.018645454198122025, 0.018595626577734947, -0.06560761481523514, -0.06825881451368332, 0.07870978862047195, 0.009293874725699425, 0.022521566599607468, -0.020001664757728577, -0.05104275792837143, -0.037673335522413254, 0.08021188527345657, -0.009127792902290821, 0.006211649626493454, -0.00584885710850358, 0.06659471988677979, 0.019013874232769012, -0.0009511292446404696, 0.06668355315923691, 0.007120934780687094, 0.02707994170486927, 0.072660893201828, -0.03779328987002373, -0.03707239776849747, -0.016956256702542305, 0.07732629030942917, -0.0028003472834825516, 0.030334604904055595, 0.011172388680279255, -0.007422857452183962], [-0.009272335097193718, -0.015872878953814507, -0.004222591407597065, 0.003624559147283435, 0.015239308588206768, -0.013869321905076504, 0.006910034455358982, -0.029071679338812828, -0.026262499392032623, -0.0348515510559082, -0.021139660850167274, -0.020306114107370377, -0.035746678709983826, 0.029830077663064003, -0.0010279418202117085, -0.025040030479431152, -0.005767496768385172, -0.002125293482095003, -0.028956906870007515, -0.010830105282366276, 0.00914745032787323, 0.0044445982202887535, -0.09696908295154572, -0.026535511016845703, -0.02575068548321724, -0.07630345970392227, -0.003609687089920044, -0.018517540767788887, 0.009220481850206852, -0.023016249760985374, -0.006057914812117815, 0.004417148418724537, -0.06692331284284592, -0.004340923856943846, -0.011020095087587833, -0.09775067865848541, -0.02172968164086342, -0.13550744950771332, -0.040162280201911926, -0.0014568613842129707, -0.02203073911368847, 0.0030693102162331343, -0.006688916124403477, 0.03918617218732834, -0.009858665987849236, -0.017832906916737556, 0.008402480743825436, 0.01304024737328291, -0.012821616604924202, -0.02735908143222332, 0.004800670780241489, 0.01887817680835724, 0.0016396199353039265, 0.012427165172994137, -0.013403385877609253, -0.030548833310604095, -0.02365267090499401, 0.008376600220799446, -0.051471732556819916, -0.010072187520563602, -0.00044237321708351374, -0.006419285200536251, 0.002536345273256302, -0.03028007410466671, 0.004799393005669117, 0.006284323520958424, -0.022954419255256653, 0.010280091315507889, 0.0026869995053857565, -0.01936621591448784, 0.006224658340215683, -0.029940275475382805, -0.02881847694516182, -0.01894836314022541, -0.004639386665076017, 0.008911886252462864, -0.022949067875742912, -0.007494034245610237, -0.052056726068258286, -0.02669820562005043, 0.003139829495921731, -0.041590671986341476, 0.004117882344871759, 0.01430292148143053, -0.0234203077852726, -0.018518468365073204, -0.0025874092243611813, -0.011032873764634132, 0.0011943699792027473, -0.03150833398103714, -0.0052211652509868145, -0.004351683426648378, -0.048627059906721115, -0.026370547711849213, 0.0028175332117825747, 0.007828817702829838], [-0.037895459681749344, 0.015331457369029522, 0.03354228287935257, 0.046281419694423676, 0.05228014290332794, -0.027879439294338226, 0.015353500843048096, 0.03618024289608002, 0.01818646490573883, 0.04694264009594917, -0.009321397170424461, -0.01643155701458454, 0.07399877160787582, 0.07176830619573593, 0.0019320897990837693, -0.016515403985977173, -0.04159099981188774, -0.0027494714595377445, -0.03817378729581833, -0.021582823246717453, 0.015531462617218494, -0.006552401464432478, -0.07440060377120972, 0.034214816987514496, -0.013907909393310547, -0.02510477416217327, 0.0027169687673449516, -0.005655659828335047, 0.011963230557739735, -0.026071351021528244, -0.0542791523039341, 0.0269831083714962, 0.010536820627748966, -0.03431399166584015, 0.032452818006277084, 0.018093988299369812, -0.04661191999912262, -0.008609025739133358, 0.02505975216627121, 0.0020678984001278877, -0.04394377022981644, 0.040689945220947266, 0.05326937884092331, 0.020615138113498688, 0.005061626899987459, 0.07787513732910156, -0.006975430063903332, 0.04076681658625603, -0.007430917117744684, -0.005684598349034786, 0.12063968181610107, -0.010012704879045486, 0.09606654196977615, -0.009859326295554638, 0.008477428928017616, -0.07443582266569138, 0.03768548369407654, 0.07656373083591461, 0.007752148434519768, 0.037011098116636276, -0.03447285667061806, -0.02891153283417225, -0.005613251589238644, 0.0024901102297008038, 0.020677680149674416, -0.009877732023596764, 0.03282148763537407, -0.02042989991605282, 0.014594538137316704, 0.04070921242237091, 0.023825861513614655, 0.03749336302280426, 0.011155025102198124, -0.018375961109995842, 0.035956431180238724, -0.0005343564553186297, -0.02405824139714241, -0.031490474939346313, -0.018306607380509377, -0.016098085790872574, 0.011508243158459663, -0.03786265850067139, -0.03390941396355629, -0.015746915712952614, -0.0009023242746479809, 0.03213300183415413, -0.02332395128905773, 0.0023724972270429134, -0.008269456215202808, 0.07393664866685867, 0.016189100220799446, 0.04526627063751221, 0.014526808634400368, -0.013130477629601955, 0.012920939363539219, -0.009439190849661827], [0.002672609407454729, 0.013890478760004044, -0.024947812780737877, -0.08010099083185196, -0.03772124648094177, -0.008901406079530716, 0.0021953361574560404, 0.027757801115512848, 0.014687112532556057, -0.012058642692863941, -0.005265773274004459, -0.030689826235175133, -0.02370179444551468, -0.07861470431089401, -0.020821893587708473, 0.012937248684465885, -0.022715765982866287, -0.018951212987303734, 0.024809816852211952, 0.012141847051680088, -0.004516284447163343, -0.023861484602093697, 0.051940932869911194, -0.048500921577215195, 0.0004096883931197226, -0.014892218634486198, 0.0002919964026659727, 0.03519034758210182, -0.013120227493345737, 0.010110019706189632, -0.01088655274361372, -0.007514767348766327, -0.006275890860706568, -0.015816576778888702, -0.028458165004849434, 0.015722839161753654, -0.0035349985118955374, 0.08468735963106155, 0.007965313270688057, -0.008015998639166355, 0.05033435672521591, 0.00374100380577147, -0.008143361657857895, -0.05131519213318825, 0.011690055951476097, -0.039875008165836334, -0.0008157853735610843, 0.005019221920520067, -0.021304186433553696, -0.004216486122459173, -0.058680210262537, -0.006826130207628012, -0.0732821524143219, -0.001721019041724503, -6.349245813908055e-05, 0.014588774181902409, 0.013046612031757832, -0.07469017058610916, 0.011501169763505459, 0.03381862863898277, -0.040340591222047806, -0.025754237547516823, 0.002695950213819742, 0.0028775378596037626, -0.00022153784811962396, 0.022433428093791008, 0.017271852120757103, -0.0023056205827742815, -0.0111170569434762, 0.04029696062207222, 0.006036972627043724, -0.05987255647778511, 0.004818454850465059, -7.94249790487811e-05, 0.001164207817055285, -0.012798452749848366, 0.0014693656703457236, -0.02418493665754795, 0.042339857667684555, -0.015448682941496372, -0.0006111445836722851, 0.01433197595179081, -0.013186084106564522, 0.008045127615332603, 0.00419381819665432, -0.004613327328115702, 0.0015755465719848871, 0.006842464674264193, 0.005801670253276825, -0.023609770461916924, -0.0007843886851333082, -0.02460429258644581, 0.020270006731152534, -0.03760029003024101, -0.012655102647840977, 0.014803683385252953], [-3.2873385862330906e-06, -6.729058077326044e-05, 1.4359286069520749e-05, -2.815009247569833e-05, 0.0001601398689672351, -1.7423795725335367e-05, -5.117707041790709e-05, 0.00034815771505236626, 1.7334348740405403e-05, 0.0006896527484059334, 1.900405521837456e-07, -1.0333005775464699e-05, -1.1915525419681217e-06, 4.0431674278806895e-05, -1.675727980909869e-05, -1.3446127013594378e-05, -1.22514866234269e-05, -2.0364510419312865e-05, -1.4193372408044524e-05, 1.8139298845198937e-06, 1.0911403478530701e-05, -1.709056959953159e-05, -1.4141508472675923e-05, 5.5880012951092795e-05, -2.6186760351265548e-06, 7.036065653664991e-05, -5.415633950178744e-07, 0.0011050382163375616, 4.7022796934470534e-05, -3.06351807921601e-06, -1.0351296623412054e-05, 4.815679130842909e-05, 3.779889811994508e-05, -2.7796819267678075e-05, -5.341033829608932e-05, 0.00010823983757290989, -3.467054557404481e-05, 7.461760833393782e-05, 1.762777174008079e-05, 5.104511001263745e-05, 7.641732372576371e-05, 3.2871641451492906e-05, 1.6513589798705652e-05, 6.54688774375245e-05, 2.7504220270202495e-05, 2.047094858426135e-05, 1.9804459952865727e-05, -2.311746902705636e-05, -7.109654688974842e-05, -1.0872090570046566e-05, 3.45480912073981e-05, -2.4859295081114396e-05, 3.218580104658031e-06, 9.62309968599584e-06, 3.1618353659723653e-06, -2.6866317057283595e-05, 0.0003717830986715853, -1.6342022718163207e-05, 1.8802996919475845e-06, 1.4386067960003857e-05, -5.538211189559661e-05, 4.558438740787096e-05, 1.9734311536012683e-06, -5.974410214548698e-06, 4.598293344315607e-06, 8.693473500898108e-05, 9.498812869424e-05, -1.4574944543710444e-05, 3.433581150602549e-05, 6.314178608590737e-05, 2.0558452888508327e-05, -0.00014200559235177934, 1.6332145378328278e-06, -1.0004804607888218e-05, 8.917145896703005e-05, 3.941696195397526e-05, 1.2256716217962094e-05, -7.053657463984564e-05, -2.5511208150419407e-05, 0.00022345705656334758, 1.8136371409127605e-06, 1.715879625407979e-05, -4.010049906355562e-06, 1.6042737115640193e-05, -1.7836035112850368e-05, 4.464287485461682e-06, 1.6050176782300696e-05, 5.3869553084950894e-05, 2.8895476134493947e-05, 4.431953129824251e-06, 9.564356332703028e-06, -8.640719897812232e-06, 5.281811172608286e-05, -6.916795246070251e-05, 2.6414889362058602e-05, 3.405738607398234e-05], [-0.00015404859732370824, -0.00045101321302354336, 0.0001665494783082977, -0.0005804375396110117, 0.0009334952337667346, -0.00039124215254560113, -0.00030764451366849244, -0.0008095749653875828, 7.662426651222631e-05, -0.005795102100819349, 0.0003161409986205399, -0.00037982818321324885, -0.00023076498473528773, 3.270056186011061e-05, -0.0001035251043504104, -0.00029756728326901793, 0.0002902401320170611, -9.94186193565838e-05, -0.00019761029398068786, 2.398358083155472e-05, 0.0001799981837393716, -8.975358650786802e-05, 0.0002613745746202767, 0.00047030969290062785, 0.00010210240725427866, 0.0007611779146827757, 8.501722732034978e-06, 0.008009886369109154, 0.0003925229830201715, 4.32596898463089e-05, -0.0002900584368035197, 0.000290943804429844, 0.0005180816515348852, -9.883096208795905e-05, 0.001146024907939136, 0.0011915911454707384, -0.00028007858782075346, 0.0008278318564407527, 0.00025316825485788286, 0.0003123579372186214, 0.00022649158199783415, 0.0003657685883808881, -0.0001501888909842819, 0.0002380584628554061, 0.0002237295702798292, 0.00020271995163057, 3.2801079214550555e-05, 1.000276370177744e-05, -0.0005269105895422399, -5.727744792238809e-05, 0.00011773334699682891, -0.00015246686234604567, 7.813933916622773e-05, -0.00010738493438111618, -3.5302429751027375e-05, -0.00018926507618743926, 0.002628630492836237, 0.0001292030356125906, 5.329111809260212e-05, 0.00034072299604304135, 0.0005227178335189819, -0.0006667625857517123, 1.1100399206043221e-05, -9.915076952893287e-05, 0.0001288407511310652, 0.0006150222616270185, 0.0006991946720518172, -0.00012111348769394681, 9.338676318293437e-05, 0.0009312125039286911, 0.0001430522825103253, -0.0012089540250599384, -1.0260220733471215e-05, -6.771504558855668e-05, 0.000175842855242081, 0.00023854758183006197, 0.00020723727357108146, -0.00031045201467350125, -0.00044642697321251035, 0.0009626736864447594, 7.144148639781633e-06, 0.0002507329627405852, -0.00021230663696769625, 8.739590703044087e-05, 6.472254608524963e-05, 0.00021947069035377353, -3.0487697586067952e-05, 0.0004951649461872876, 0.0001346502103842795, 0.00045717923785559833, 0.00012687130947597325, -7.411424303427339e-05, 0.0005514781223610044, -0.0004922120133414865, 4.8362584493588656e-05, 0.0001948281715158373], [0.006678249686956406, 0.006169254891574383, -0.005442265421152115, 0.020395051687955856, -0.0194553229957819, 0.007498797960579395, 0.0084106819704175, 0.02371932566165924, -0.007748805917799473, -0.02316548302769661, -0.008146309293806553, 0.016850344836711884, 0.009815745986998081, -0.009048069827258587, 0.0014097890816628933, 0.01976030319929123, 0.0014470885507762432, 0.01302496436983347, 0.010777798481285572, -0.00021787708101328462, 0.01271811407059431, 0.004225660115480423, 0.03610710799694061, -0.005412933882325888, -0.0025219526141881943, 0.029173463582992554, 0.0005656258435919881, -0.0072351316921412945, -0.008339925669133663, -0.001740123494528234, 0.0055477675050497055, -0.010407066904008389, 0.007692966144531965, -0.0053933304734528065, 0.0014277006266638637, 0.010977773927152157, 0.016855692490935326, 0.04167499020695686, 0.010231629945337772, -0.0032865593675523996, 0.022143647074699402, -0.01299147680401802, -0.002137555507943034, -0.009555420838296413, -0.0025389157235622406, -0.0007535978802479804, -0.0030061586294323206, -0.011974501423537731, 0.0009825716260820627, 0.008421752601861954, 0.013025546446442604, 0.021623849868774414, -0.005106119439005852, 0.003195390338078141, 0.000790035875979811, 0.015786651521921158, -0.005157631356269121, 0.01115099061280489, 0.012609467841684818, -0.0005147100891917944, -0.003482539439573884, -0.007436951156705618, -0.0004037531034555286, 0.009153610095381737, -0.008115662261843681, -0.009411324746906757, -0.004561196081340313, 0.003706208895891905, 0.00029496572096832097, -0.018695972859859467, -0.0018418137915432453, 0.020723147317767143, 0.007005799561738968, 0.0035803671926259995, 0.00030870898626744747, -0.004656841512769461, 0.011498469859361649, 0.01356177031993866, -0.010876137763261795, 0.007771930191665888, -0.00218715937808156, 0.001082352944649756, 0.007931719534099102, 0.0006720589590258896, -0.0029756734147667885, -0.0024793746415525675, 0.004161261022090912, 0.0005387109122239053, -0.0006484600016847253, 0.004580597393214703, -0.00465876329690218, 0.0016206688014790416, 0.005417926702648401, 0.02088342048227787, -0.0013299657730385661, -0.00263060606084764], [-0.008153161965310574, -0.01363740861415863, -0.004086886998265982, 0.029552385210990906, 0.026746904477477074, -0.009116006083786488, -0.012262395583093166, -0.005659479647874832, -0.01611500047147274, -0.010421926155686378, 0.008587503805756569, -0.009905137121677399, 0.029381658881902695, 0.02532355859875679, 0.004989758599549532, -0.013879089616239071, 0.009047946892678738, -0.005179448518902063, -0.011869270354509354, -0.00023074817727319896, -0.0171036534011364, 2.1179763280088082e-05, -0.024360746145248413, 0.04906713590025902, -0.0018250637222081423, 0.01035134494304657, 0.00042810256127268076, -0.02894791215658188, 0.014478526078164577, 0.0013614329509437084, -0.010232318192720413, 0.014150343835353851, 0.004710122011601925, 0.001387222670018673, -0.007325710728764534, 0.01836208812892437, -0.008619721978902817, -0.016927601769566536, 0.004272149875760078, 0.021629398688673973, -0.021631544455885887, 0.01635099947452545, 0.0003541756886988878, 0.016225621104240417, 0.00492973392829299, 0.026138991117477417, -0.0007227263413369656, 0.005435630679130554, -0.006441778969019651, 0.006831372156739235, 0.040648967027664185, -0.01933971792459488, 0.03793363645672798, -0.006290789227932692, 0.002548020798712969, -0.007950772531330585, -0.009683257900178432, 0.04270211607217789, -0.0008967127650976181, -0.0037486383225768805, 0.016002042219042778, 0.018626278266310692, 0.00012973102275282145, -0.00018441176507622004, 0.009531962685286999, 0.0053446064703166485, 0.0023154783993959427, -0.004639468155801296, 0.003823990235105157, 0.003725124057382345, 0.010826554149389267, 0.021378125995397568, -0.0005256036529317498, 0.0009686403791420162, 0.013378748670220375, 0.007455540355294943, -0.004093014635145664, -0.010126604698598385, 0.025802915915846825, 0.007148148026317358, 0.000613634823821485, -0.009415678679943085, -0.009684324264526367, -0.00045959255658090115, 0.004303691443055868, 0.004197622649371624, -0.003919791430234909, -3.090314567089081e-05, 0.00035378997563384473, 0.007390825543552637, 0.0008242274052463472, -0.0020130600314587355, 0.023317167535424232, 0.0033129758667200804, -0.0029639482963830233, -0.00030780266388319433], [0.02099612168967724, 0.08945483714342117, 0.03094531036913395, -0.040065914392471313, -0.14396604895591736, 0.061726219952106476, -0.005668213125318289, 0.06185373291373253, -0.011942729353904724, 0.08028095960617065, -0.01598464511334896, 0.039934996515512466, 0.08992023766040802, -0.10040660202503204, 0.005541719030588865, 0.08410651236772537, 0.025060610845685005, -0.012870380654931068, 0.057619545608758926, 0.05647556483745575, 0.0411137230694294, 0.06690926104784012, 0.15448147058486938, -0.02297593653202057, 0.0036329813301563263, 0.05081802234053612, 0.029925312846899033, 0.06636856496334076, -0.03260422125458717, 0.07702155411243439, 0.045027513056993484, -0.060603443533182144, 0.037235327064991, 0.07546568661928177, 0.07203785330057144, 0.09319309145212173, 0.077997587621212, 0.16697171330451965, 0.06998725980520248, -0.022841524332761765, -0.022311633452773094, -0.06272130459547043, -0.05190211161971092, -0.11673038452863693, 0.04765421524643898, 0.03938359394669533, -0.059896137565374374, -0.10617776960134506, 0.07568740844726562, 0.03687846288084984, 0.02209518663585186, -0.06247832626104355, 0.038209110498428345, 0.031029533594846725, -0.002263536211103201, 0.10432028770446777, -0.019067833200097084, -0.0014842796372249722, 0.041499242186546326, 0.031386975198984146, -0.013824193738400936, -0.008725276216864586, -0.01858563907444477, 0.09598562121391296, -0.07319847494363785, -0.048781994730234146, -0.06982696056365967, -0.0015592840500175953, 0.0005848585860803723, -0.03797811269760132, -0.09101194143295288, 0.09129397571086884, 0.024827955290675163, 0.04680318757891655, -0.06813549250364304, -0.06531444191932678, 0.07262072712182999, 0.03487471491098404, 0.013854426331818104, -0.004838305991142988, -0.01916670612990856, 0.07914071530103683, 0.09849979728460312, -0.017525622621178627, 0.08705566823482513, 0.07092288881540298, 0.016180912032723427, -0.00156039884313941, 0.03153235465288162, -0.009049835614860058, -0.0454493872821331, -0.022313669323921204, -0.036997437477111816, 0.1113215833902359, 0.02978578396141529, -0.03542177379131317], [-0.029664332047104836, -0.0548824816942215, -0.005482863634824753, 0.044844552874565125, 0.04431699961423874, -0.03863805532455444, -0.024526579305529594, -0.035365551710128784, -0.017437709495425224, -0.011144478805363178, -0.00231117382645607, -0.04552984982728958, 0.02916123904287815, 0.03731083869934082, 0.02098585106432438, -0.042593814432621, 0.036813437938690186, 0.002298333216458559, -0.04102344810962677, 0.00904189981520176, 0.010775521397590637, 0.031247571110725403, -0.058511510491371155, 0.050068389624357224, 0.01901811920106411, 0.02874676138162613, 0.0015236499020829797, 0.07173050194978714, 0.03168268874287605, -0.015091677196323872, -0.029987888410687447, 0.0239453986287117, -0.0024762696120887995, -0.0037233075127005577, -0.04135214909911156, 0.021491065621376038, -0.031502604484558105, -0.023284820839762688, 0.032974112778902054, 0.0426359698176384, -0.05015768110752106, 0.019750528037548065, -0.0021975813433527946, 0.036893330514431, 0.009625161066651344, 0.025975629687309265, -0.006409134715795517, 0.01039002276957035, -0.0036080130375921726, 0.0194805096834898, 0.06831657141447067, -0.021058399230241776, 0.09959768503904343, -0.023422887548804283, 0.018859775736927986, -0.021228529512882233, 0.005210731644183397, 0.08821054548025131, -0.008348926901817322, -0.020473994314670563, 0.03579296916723251, 0.02681449055671692, 0.0033450520131736994, 0.00750863179564476, 0.03208385780453682, 0.006534389220178127, 0.026822343468666077, -0.004060071427375078, -0.007848504930734634, 0.02305658906698227, 0.007487159688025713, 0.014800192788243294, -0.009333894588053226, -0.0057680802419781685, 0.04663850739598274, 0.007006858941167593, -0.03453810140490532, -0.006607995368540287, 0.04579836502671242, 0.00819340255111456, 8.038685336941853e-05, -0.006821262650191784, -0.010521628893911839, -0.01658426970243454, -0.009669397957623005, 0.02092425338923931, -2.0512703486019745e-05, 0.026430804282426834, -0.006304553709924221, 0.0032525197602808475, 0.006140226498246193, 0.021584315225481987, 0.03807055950164795, -0.0033583270851522684, -0.025501828640699387, -0.009316565468907356], [-0.026370497420430183, 0.014695686288177967, 0.0405852347612381, -0.1028425469994545, -0.07792790234088898, 0.026534242555499077, -0.006888372823596001, 0.023228464648127556, -0.01596212573349476, 0.03881754353642464, 0.017449527978897095, 0.007788484916090965, -0.09109096974134445, -0.06212354078888893, -0.04370333254337311, -0.013407365418970585, -0.0732063427567482, -0.009964645840227604, 0.07959101349115372, 0.037942059338092804, 0.04185834899544716, -0.07894633710384369, 0.13535897433757782, -0.1202196478843689, -0.0404176265001297, 0.10815148800611496, 0.025406111031770706, 0.0007561002857983112, 0.030158184468746185, -0.0029586537275463343, -0.011571920476853848, -0.031910572201013565, -0.021236713975667953, -0.04646632820367813, -0.020780714228749275, 0.04253045469522476, -0.024945594370365143, 0.09764760732650757, -0.01402251049876213, -0.013796459883451462, -0.004641884472221136, -0.002891127485781908, 0.05577481910586357, -0.0289803184568882, -0.029457753524184227, -0.011378863826394081, -0.013617745600640774, 0.0007342950557358563, -0.039620306342840195, 0.0264976117759943, -0.0967898815870285, 0.08972761780023575, -0.09629499167203903, 0.011332967318594456, 0.012605329044163227, -0.013257971964776516, -0.029246723279356956, -0.1685507744550705, 0.02542572095990181, 0.09052513539791107, -0.06214777007699013, -0.013370739296078682, -0.0004844353534281254, 0.006362526677548885, 0.028775658458471298, 0.0418042428791523, 0.004389102570712566, 0.020058900117874146, 0.00649427343159914, 0.026971276849508286, 0.040382348001003265, -0.18963049352169037, 0.04931502044200897, -0.001968136290088296, -0.034857820719480515, -0.019974512979388237, 0.08590350300073624, 0.009661994874477386, 0.07359333336353302, 0.06821662932634354, -0.044769350439310074, 0.11536718159914017, 0.06793420016765594, 0.042419273406267166, 0.024763094261288643, -0.019027885049581528, 0.03235994279384613, 0.011966731399297714, 0.03619704395532608, -0.003555466653779149, -0.014125245623290539, 0.005789255257695913, 0.04573722556233406, -0.03850694000720978, 0.002148989588022232, 0.001894680317491293], [-0.04471280798316002, 0.0023060364183038473, -0.029094165191054344, -0.024435512721538544, 0.022334575653076172, -0.02676686830818653, -0.007707095704972744, -0.015731383115053177, -0.08469944447278976, 0.04354891553521156, -0.04846641793847084, -0.1062125638127327, -0.006955583114176989, 0.029836466535925865, 0.0008440471137873828, -0.032947152853012085, 0.08721257746219635, -0.05243122950196266, -0.05812814459204674, 0.009040278382599354, 0.0027435861993581057, 0.037677884101867676, -0.04973815008997917, 0.04210695996880531, -0.06240421533584595, 0.017446447163820267, 0.01466309279203415, 0.013696029782295227, -0.0036533779930323362, -0.015999067574739456, -0.05707573518157005, -0.023567453026771545, 0.07457364350557327, 0.014101876877248287, -0.06203099712729454, 0.07133438438177109, -0.044821832329034805, -0.026803597807884216, 0.033354997634887695, 0.02009911648929119, 0.016871675848960876, 0.03833795711398125, 0.052134159952402115, -0.01181132160127163, 0.0009141354821622372, 0.038804344832897186, -0.036574508994817734, -0.004387126304209232, -0.04654780030250549, 0.009893440641462803, 0.01986822485923767, -0.045712653547525406, -0.005975428968667984, -0.041279882192611694, -0.04146505519747734, -0.07705715298652649, -0.04953927919268608, 0.07093851268291473, 0.010364122688770294, 0.00300157954916358, 0.025523260235786438, 0.07497056573629379, -0.01921476423740387, 0.00023427398991771042, 0.0559505894780159, -0.04285974055528641, -0.005174532998353243, 0.002638713689520955, -0.01697687618434429, 0.013161946088075638, 0.04213400557637215, 0.02171207405626774, -0.05879468470811844, 0.017252855002880096, 0.09020590782165527, 0.027002165094017982, -0.0013154220068827271, -0.023944787681102753, 0.0926135927438736, 0.07905534654855728, -0.010455534793436527, -0.013021336868405342, -0.07611965388059616, 0.024673711508512497, -0.02367391437292099, 0.02484685368835926, -0.030620060861110687, -0.039227161556482315, 0.0067486087791621685, 0.057439204305410385, -0.0420486256480217, 0.06573819369077682, 0.008383474312722683, -0.007560365367680788, -0.0660954937338829, -0.009630963206291199], [-0.022608162835240364, 0.026911912485957146, 0.01848553493618965, -0.13440613448619843, -0.019055165350437164, 0.03456201031804085, -0.03888159617781639, 0.04761818051338196, -0.07131494581699371, -0.0014802179066464305, -0.07753662765026093, 0.06462325900793076, -0.14736442267894745, -0.12342048436403275, -0.0166401918977499, -0.022947195917367935, 0.016027826815843582, 0.03648409992456436, 0.0809570923447609, 0.00885688979178667, 0.05201902985572815, -0.1609460860490799, 0.10884656757116318, -0.07299597561359406, -0.05560014396905899, 0.08773801475763321, 0.0032741876784712076, -0.034661032259464264, 0.056923218071460724, 0.08417802304029465, 0.07519935816526413, 0.049009423702955246, -0.000502253242302686, 0.014227394945919514, -0.06724657118320465, 0.0304703451693058, 0.007008770946413279, 0.12052374333143234, 0.054769616574048996, -0.0646178126335144, 0.1122712567448616, -0.014084557071328163, 0.030973587185144424, -0.11167436093091965, 0.10245516151189804, -0.011241215281188488, 0.03565070405602455, 0.01577465794980526, 0.0302724726498127, -0.000808308832347393, -0.1787213534116745, 0.00014184505562298, -0.13626950979232788, 0.0680057629942894, -0.0085381381213665, 0.07414979487657547, 0.03486914932727814, -0.14646023511886597, 0.021250007674098015, -0.03287513926625252, -0.03980033099651337, -0.06844420731067657, 0.013391382060945034, -0.004520380403846502, 0.025663159787654877, 0.024739360436797142, -0.07473983615636826, 0.010131270624697208, 0.06218995153903961, -0.003993152640759945, 0.06683086603879929, -0.21082158386707306, 0.08856767416000366, 0.04513179883360863, -0.034259162843227386, 0.026085838675498962, 0.03632446005940437, -0.0039006622973829508, 0.038940250873565674, -0.0073340944945812225, -0.04263739660382271, 0.049822237342596054, 0.02498672343790531, 0.012288793921470642, 0.06063808128237724, -0.07010109722614288, 0.11552891135215759, -0.007185898721218109, 0.025581160560250282, -0.05917337164282799, -0.04752258583903313, -0.10137245804071426, 0.04067790135741234, 0.042632829397916794, -0.0023237920831888914, 0.04015906900167465], [-0.06935525685548782, 0.03757088631391525, 0.04361259564757347, -0.12406378239393234, -0.04891499504446983, -0.02945602498948574, 0.00526536675170064, -0.0052798716351389885, 0.056225359439849854, -0.026159435510635376, -0.011019923724234104, -0.0453895628452301, -0.08250116556882858, -0.08112526684999466, 0.01965365745127201, 0.008926467038691044, 0.0005181391607038677, 0.011803724803030491, 0.04233424738049507, 0.020630164071917534, 0.019163616001605988, -0.047735586762428284, 0.08086135238409042, -0.05526174604892731, -0.061306245625019073, 0.022458530962467194, -0.001773608848452568, 0.026522135362029076, 0.0268442090600729, 0.019691340625286102, 0.03599219769239426, -0.05937232822179794, 0.061981841921806335, -0.012980216182768345, -0.04219374805688858, 0.02649625577032566, 0.018458761274814606, 0.08430751413106918, 0.02899722382426262, -0.01964498683810234, -0.0030265813693404198, 0.025920402258634567, -0.010578193701803684, -0.059266820549964905, -0.021140074357390404, -0.0318181999027729, -0.0070370095781981945, -0.025060489773750305, -0.03747089207172394, -0.003124837763607502, -0.0848536565899849, 0.05830977112054825, -0.07425631582736969, 0.0101960189640522, -0.015840746462345123, -0.08686386793851852, 0.08692191541194916, -0.10499349236488342, 0.010425071232020855, 0.038318902254104614, 0.045728329569101334, -0.06043381989002228, 0.012261729687452316, -0.020299049094319344, 0.01113983429968357, -0.002470510546118021, -0.0396147258579731, -0.016320204362273216, 0.0524936281144619, 0.021069364622235298, 0.0007000684272497892, -0.10604299604892731, -0.01116259302943945, 0.013198619708418846, 0.036275774240493774, -0.005595935974270105, 0.0655486136674881, -0.016555722802877426, 0.07486879080533981, 0.04133741557598114, 0.012077025137841702, 0.007250621914863586, 0.016387196257710457, -0.018324585631489754, -0.013692621141672134, -0.00736645981669426, 0.01709699258208275, -0.02240818366408348, 0.033204592764377594, -0.05112367123365402, 0.020125607028603554, 0.0027715754695236683, 0.03364986926317215, 0.0454973466694355, -0.02390912175178528, 0.0004855878942180425], [0.04447527602314949, -0.01893112063407898, -0.010278746485710144, 0.057757213711738586, -0.07542833685874939, 0.021877409890294075, -0.03502639755606651, -0.014921553432941437, 0.07296858727931976, -0.04901878908276558, -0.044377271085977554, 0.010016445070505142, 0.08128385245800018, -0.040739674121141434, 0.002546628937125206, 0.12048660963773727, 0.06068895757198334, 0.02449336089193821, -0.02510373294353485, 0.012228677980601788, 0.08707604557275772, 0.06081677973270416, 0.14962731301784515, 0.08277880400419235, -0.0032183039002120495, 0.07591560482978821, 0.0054361457005143166, -0.06895925849676132, -0.044384829699993134, -0.0400889553129673, 0.03211984783411026, -0.05538274347782135, 0.08502523601055145, -0.02640955150127411, -0.07664532214403152, 0.013589223846793175, 0.07924506068229675, 0.17909207940101624, 0.0035290850792080164, 0.035263244062662125, 0.06949182599782944, -0.010253011249005795, -0.006158496718853712, -0.034388501197099686, -0.06604049354791641, -0.02337731048464775, 0.009017306379973888, -0.01927589252591133, -0.011578382924199104, 0.031009823083877563, -0.044797398149967194, 0.020391391590237617, -0.010451795533299446, 0.022080274298787117, -0.05493776872754097, -0.022919293493032455, -0.006472726818174124, -0.018653318285942078, 0.07990019023418427, -0.017816156148910522, -0.06272204965353012, -0.04924815148115158, -0.004796321503818035, 0.0283825546503067, -0.002510298741981387, -0.0008653922122903168, 0.044851161539554596, 0.006103750318288803, 0.005763290449976921, 0.00015775054635014385, 0.03456120193004608, 0.024082133546471596, 0.03381137177348137, 0.05109424889087677, -0.02250504679977894, -0.032130979001522064, 0.0420900359749794, 0.08236361294984818, -0.01919122412800789, 0.03640943020582199, -0.015210943296551704, 0.043756335973739624, 0.02470340207219124, -0.023620884865522385, -0.06613034754991531, -0.03637026250362396, 0.009806494228541851, -0.0025706191081553698, 0.012307743541896343, 0.026978107169270515, -0.056370243430137634, -0.03022599034011364, 0.03945409506559372, -0.0021801157854497433, 0.05498228594660759, -0.020941926166415215], [-0.06613549590110779, -0.004512695595622063, -0.041724737733602524, 0.06897589564323425, 0.034404776990413666, -0.08428028225898743, -0.04886021837592125, -0.03768055513501167, 0.016729915514588356, -0.033430635929107666, 0.055911727249622345, -0.13803400099277496, 0.12183427810668945, 0.04913624748587608, -0.03065275400876999, -0.07646829634904861, -0.012759733013808727, -0.08020174503326416, -0.08872239291667938, -0.0232907235622406, -0.09465925395488739, 0.06435060501098633, -0.16032080352306366, 0.10722235590219498, 0.0344715490937233, -0.02621588669717312, 0.017007481306791306, -0.05871903896331787, 0.07638167589902878, -0.04592661187052727, -0.03320280462503433, -0.009741743095219135, 0.01121093425899744, -0.04802301898598671, -0.02767092175781727, 0.039924256503582, -0.0321366973221302, -0.08512550592422485, 0.013918856158852577, 0.0783899649977684, -0.13212986290454865, 0.06995389610528946, 0.07724300026893616, 0.02222762256860733, 0.02586221508681774, 0.0864618569612503, -0.04923383519053459, 0.05307886376976967, -0.058421846479177475, 0.02335788682103157, 0.06271261721849442, -0.024906769394874573, 0.11982416361570358, -0.08397045731544495, 0.010865269228816032, -0.0408354252576828, 0.003927253652364016, 0.1274314969778061, -0.018202362582087517, -0.0744844451546669, 0.07426963001489639, 0.06617384403944016, -0.005094474647194147, 0.0022722892463207245, 0.051818110048770905, -0.019437020644545555, 0.002489680191501975, -0.053484126925468445, -0.028298720717430115, 0.0012145137879997492, 0.04322211071848869, 0.11912986636161804, -0.05614553391933441, 0.00860766600817442, 0.08175502717494965, 0.03685007616877556, -0.08886165171861649, -0.05053175240755081, 0.05072036385536194, 0.0602254681289196, -0.00584854930639267, -0.021296195685863495, -0.07634761929512024, -0.016341211274266243, -0.04525578022003174, 0.04293343424797058, -0.0658365860581398, 0.0287144985049963, -0.018473446369171143, 0.05743005499243736, 0.03943939879536629, -0.0430050864815712, 0.05403132364153862, 0.01441405899822712, 0.014799202792346478, -0.04526262357831001], [-0.025933608412742615, 0.02969127707183361, 0.031238524243235588, -0.09575901925563812, -0.014010271988809109, -0.02487833984196186, 0.0026710019446909428, 0.022323917597532272, 0.02268601953983307, -0.006253921892493963, -0.030212657526135445, -0.0030253699515014887, -0.03647594153881073, -0.10054619610309601, -0.00843491405248642, -0.009863723069429398, -0.0356898307800293, 0.0027516859117895365, 0.005408689379692078, 0.0025119611527770758, 0.001750487950630486, -0.02998287044465542, 0.05447163060307503, -0.03351663798093796, 0.0032036281190812588, -0.010842384770512581, 0.0006110806134529412, -0.019091526046395302, -0.0012090640375390649, 0.01075027510523796, 0.00359611539170146, -0.014229079708456993, 0.01932043768465519, 0.012505993247032166, -0.017000840976834297, 0.03175577148795128, 0.0002750444982666522, 0.08215570449829102, -0.0031437634024769068, -0.023458657786250114, 0.05306626856327057, -0.007589196320623159, 0.008779886178672314, -0.054625023156404495, 0.03622015565633774, -0.03429487347602844, -0.011067142710089684, -0.011348409578204155, -0.0067536234855651855, 0.0013102840166538954, -0.0578266903758049, 0.03339260071516037, -0.06890136003494263, 0.00097771140281111, -0.01632075384259224, -0.011766730807721615, 0.0578627847135067, -0.0955028310418129, 0.012210520915687084, 0.01804620586335659, 0.034327343106269836, -0.03891417384147644, -4.383688428788446e-05, -0.010689608752727509, 0.009262455627322197, -0.0018567722290754318, 0.031938180327415466, 0.0012336464133113623, 0.00932564027607441, 0.06266991794109344, -0.013125445693731308, -0.07960424572229385, 0.00893911998718977, -0.0013172207400202751, 0.018471691757440567, -0.001162892091087997, 0.054312244057655334, -0.04258042946457863, 0.02546333335340023, -0.015200439840555191, -0.009571009315550327, 0.035789113491773605, -0.005081760231405497, -0.004947550129145384, -0.03499498590826988, 6.327045412035659e-05, 0.0041881403885781765, -0.02512587048113346, 0.017420124262571335, -0.00022822133905719966, 0.016380788758397102, -0.004630706738680601, 0.011861077509820461, -0.03659440204501152, -0.01787533238530159, 0.004652413073927164], [0.04589448869228363, 0.02703266032040119, -0.057241082191467285, 0.08364133536815643, -0.009537323378026485, 0.07094623893499374, 0.01759256236255169, 0.08336266130208969, 0.040274281054735184, 0.10660041868686676, 0.06931021809577942, 0.07112526893615723, -0.030326252803206444, 0.014872081577777863, -0.04511721804738045, 0.07149781286716461, 0.0012881242437288165, 0.05433531478047371, 0.06366860866546631, -0.011382131837308407, -0.02431233413517475, 0.01788855530321598, 0.09549262374639511, -0.05947455018758774, 0.013005080632865429, 0.11378972977399826, 0.015825921669602394, 0.08002949506044388, -0.07681957632303238, 6.484624464064837e-05, -0.005139428190886974, -0.02209213376045227, 0.05359653756022453, 0.07219387590885162, 0.09898167103528976, 0.024104878306388855, 0.10606693476438522, 0.15204904973506927, -0.03171813115477562, -0.0570940338075161, 0.04455751180648804, -0.09495867788791656, -0.03963181748986244, -0.02174125798046589, -0.03410400077700615, 0.0019527511904016137, -0.0629831999540329, -0.006076447665691376, 0.08906184881925583, -0.006021581590175629, -0.0654165968298912, 0.03974730148911476, -0.03095395490527153, 0.012906738556921482, 0.07947979122400284, 0.11682754755020142, 0.07019852101802826, 0.012719785794615746, 0.06941372156143188, 0.02226603403687477, 0.06748206913471222, 0.022347651422023773, -0.012117204256355762, 0.056361619383096695, -0.06041266396641731, -0.06150563061237335, -0.04543246701359749, -0.023396102711558342, -0.037071533501148224, 0.005820673890411854, -0.052712224423885345, 0.058511897921562195, 0.10475209355354309, 0.026472344994544983, -0.07129955291748047, -0.019958892837166786, -0.04839317873120308, 0.11814233660697937, 0.016835195943713188, -0.02871885523200035, -0.052198536694049835, 0.010477638803422451, -0.0315786637365818, -0.03525674343109131, 0.09855621308088303, -0.01593513786792755, -0.028939805924892426, -0.014072488062083721, -0.04323933646082878, 0.03446884825825691, -0.03230796754360199, 0.016835611313581467, 0.041013576090335846, 0.07398766279220581, -0.011718396097421646, 0.008735223673284054], [0.04555080458521843, 0.06253071129322052, -0.04163537546992302, -0.023359347134828568, -0.0686071515083313, 0.05696282163262367, 0.058589112013578415, 0.023069879040122032, 0.0641409233212471, 0.03906009718775749, 0.052815549075603485, 0.056574080139398575, 0.07527153939008713, -0.011349150910973549, 0.04616842791438103, 0.01463090069591999, -0.012327875941991806, 0.003599607152864337, 0.04586732015013695, 0.04403531923890114, 0.012363946996629238, 0.035908330231904984, 0.015431810170412064, 0.04210837930440903, 0.0038457356858998537, -0.02047506906092167, -0.008258243091404438, 0.02302243560552597, 0.0367092601954937, -0.0123263169080019, 0.007167126052081585, -0.013762392103672028, -0.030305566266179085, 0.04085772857069969, 0.01422109454870224, 0.016330324113368988, 0.04686867073178291, -0.03882221505045891, 0.0029535850044339895, -0.04927600175142288, -0.0026215598918497562, -0.02112117037177086, -0.04053833335638046, -0.05889279022812843, 0.020434895530343056, -0.02102852612733841, 0.007189445197582245, -0.02798747830092907, 0.02054937183856964, -0.021379753947257996, -0.06951904296875, 0.04054531082510948, -0.0014051897451281548, 0.012250331230461597, 0.06036454811692238, 0.034699343144893646, 0.06398597359657288, 0.014885668642818928, 0.008396079763770103, -0.026103299111127853, 0.0629797950387001, -0.004686749540269375, 0.008044486865401268, 0.044115982949733734, -0.010888973250985146, -0.03922911360859871, -0.041111573576927185, 0.01615801639854908, 0.019380372017621994, -0.039313316345214844, 0.0022098999470472336, 0.08663684874773026, 0.043729931116104126, -0.02469051070511341, 0.05625368654727936, 0.0024422132410109043, -0.02339424006640911, 0.04470601677894592, 0.07723992317914963, -0.02353360503911972, -0.006265835836529732, 0.014653208665549755, 0.002913168864324689, -0.013360477983951569, 0.004221662878990173, -0.00693368399515748, -0.00383382779546082, 0.054467134177684784, -0.02411031536757946, -0.011623856611549854, -0.0010490496642887592, 0.017403410747647285, 0.017580850049853325, 0.010959465987980366, 0.01267845369875431, -0.007075651083141565], [-0.06575565040111542, -0.10145361721515656, -0.03324909508228302, 0.0274753849953413, 0.09195740520954132, -0.0019770138897001743, -0.0745701715350151, -0.05017843842506409, -0.06271016597747803, -0.06854243576526642, 0.071200892329216, -0.06593454629182816, 0.03055412322282791, 0.07847429066896439, 0.031528040766716, -0.08545427024364471, 0.07052891701459885, -0.0796964168548584, -0.04029257595539093, 0.018906600773334503, -0.09070899337530136, 0.042238201946020126, -0.09315866976976395, 0.12809106707572937, -0.05615917965769768, 0.026742422953248024, -0.012115932069718838, 0.06693404167890549, 0.05694272741675377, 0.011576075106859207, -0.051726777106523514, 0.013038614764809608, -0.04542136564850807, -0.03811473771929741, 0.03238123655319214, 0.07091262936592102, -0.016559356823563576, -0.09121225774288177, 0.08413650095462799, 0.0794355571269989, -0.09964660555124283, 0.023844879120588303, -0.026183877140283585, -0.02862490527331829, 0.030594147741794586, 0.0185227207839489, -0.04398270323872566, 0.051305461674928665, 0.045514412224292755, 0.05820998176932335, 0.061732906848192215, -0.10165687650442123, 0.02625453472137451, -0.027302928268909454, 0.01688305474817753, -0.05628686770796776, 0.005378123838454485, 0.11256228387355804, 0.013140273280441761, -0.0219377800822258, 0.025950061157345772, -0.012480195611715317, 0.005298450123518705, 0.03388272225856781, 0.04085640236735344, 0.025184867903590202, 0.01460651308298111, -0.01852555200457573, 0.048140984028577805, -0.04058930650353432, 0.010867994278669357, 0.09685556590557098, 0.018505947664380074, 0.004991676192730665, -0.035662442445755005, 0.0212250929325819, -0.06279849261045456, -0.04333680123090744, 0.01652461290359497, 0.06486117094755173, -0.032795339822769165, -0.06732388585805893, -0.10206034779548645, -0.008323381654918194, -0.007237445097416639, 0.0353081077337265, -0.016928566619753838, 0.03559805080294609, 0.0261344276368618, -0.00860794261097908, -0.02371053211390972, -0.0761546865105629, 0.04287783056497574, -0.04564140364527702, -0.08094139397144318, -0.043838467448949814], [-0.08381321281194687, -0.07788889855146408, 0.04353654757142067, 0.02608131617307663, 0.07824444770812988, -0.017549196258187294, 0.015616891905665398, -0.03772396594285965, -0.02234582044184208, -0.016850950196385384, -0.02724829688668251, -0.015521914698183537, 0.013774137012660503, 0.029910163953900337, 0.055210862308740616, -0.015228306874632835, 0.062451452016830444, -0.025614658370614052, -0.04282991588115692, -0.02002185396850109, 0.0017801499925553799, -0.022784193977713585, -0.06593980640172958, 0.1046026349067688, -0.06742168217897415, 0.03780865669250488, 0.0038336196448653936, -0.07625216245651245, 0.034176211804151535, 0.059541407972574234, -0.07755591720342636, 0.032505787909030914, 0.06394931674003601, -0.04586299508810043, 0.016438456252217293, 0.03714285045862198, -0.0017394444439560175, -0.08711516857147217, -0.005244162864983082, 0.03366754204034805, -0.05578768998384476, 0.02850683033466339, -0.0049124835059046745, -0.013705660589039326, 0.03841356188058853, 0.06651102751493454, -0.012355206534266472, 0.059861671179533005, 0.021177204325795174, 0.031448233872652054, 0.057142604142427444, -0.059038784354925156, 0.043159984052181244, -0.0031414381228387356, -0.052936259657144547, -0.12625212967395782, 0.047736890614032745, 0.09047367423772812, 0.006815864238888025, -0.02748570591211319, 0.0859050452709198, 0.04495233669877052, -0.014580217190086842, -0.009948164224624634, -0.00925478246062994, -0.0023003253154456615, -0.021152354776859283, -0.02792353741824627, 0.04987939074635506, -0.047529079020023346, -0.008786172606050968, 0.01577708311378956, -0.05124268680810928, -0.02559306100010872, -0.010966592468321323, 0.0038116690702736378, 0.06064234673976898, -0.019123105332255363, 0.03723021224141121, 0.0023927162401378155, 0.014747049659490585, -0.06333190947771072, 0.005200909450650215, -0.04064466431736946, 0.06350905448198318, -0.025683866813778877, -0.008151916787028313, -0.03593512624502182, 0.004256842657923698, 0.03933881223201752, 0.017916548997163773, 0.01334468275308609, 0.08010589331388474, 0.07350689172744751, -0.024882763624191284, -0.02001175470650196], [0.035191021859645844, -0.005896733608096838, -0.05725443735718727, 0.03830982372164726, -0.054292336106300354, 0.06335426867008209, 0.02958272583782673, 0.11124646663665771, 0.04137082025408745, 0.08312594890594482, -0.019604140892624855, 0.06867261976003647, 0.00782214105129242, -0.02962207794189453, 0.009813492186367512, 0.043273262679576874, -0.041564807295799255, -0.0031555905006825924, 0.027165014296770096, 0.036030713468790054, 0.07899836450815201, -0.01505834236741066, 0.10916323214769363, 0.0026534930802881718, 0.0511389784514904, 0.07820875942707062, 0.02063649334013462, 0.033113326877355576, -0.06500398367643356, 0.03623390570282936, 0.0126605574041605, -0.03985029086470604, 0.07024756819009781, -0.01771237514913082, -0.013640359975397587, 0.05460570752620697, 0.08573796600103378, 0.10002359747886658, 0.04262395575642586, -0.014583688229322433, 0.04933397099375725, -0.06316255778074265, 0.06217619404196739, -0.06211575120687485, -0.03964376449584961, -0.022015372291207314, -0.03572447970509529, -0.06996703892946243, 0.07464557886123657, 0.03881490230560303, -0.032213494181632996, -0.04552485793828964, 0.010724872350692749, 0.03227812051773071, 0.03539757430553436, 0.09717586636543274, 0.07060498744249344, 0.011619615368545055, 0.03427917882800102, 0.008501561358571053, 0.07369948923587799, 0.008126702159643173, 0.009503372944891453, 0.04872366786003113, -0.03668152540922165, -0.06725295633077621, -0.04440481960773468, 0.002365190302953124, 0.013166739605367184, -0.06065943464636803, -0.007299122866243124, 0.032429397106170654, 0.07381676882505417, 0.0053708963096141815, 0.021309103816747665, -0.014951657503843307, 0.008831088431179523, 0.07103037089109421, 0.07033694535493851, -0.014114315621554852, -0.02558821253478527, 0.05401510000228882, 0.016135597601532936, -0.008992919698357582, 0.052526939660310745, 0.04256177693605423, 0.028271298855543137, -0.0045675961300730705, -0.006227695383131504, -0.001421856926754117, -0.019582657143473625, -0.0174343753606081, -0.0015998429153114557, 0.09101618826389313, -0.0027642417699098587, -0.007985410280525684], [-0.0010183654958382249, -0.0013030542759224772, -0.0021963308099657297, -0.04578700289130211, -0.022437922656536102, -0.005572488531470299, -0.003950139973312616, -0.004015969578176737, -0.00022660191461909562, 0.008360731415450573, 0.001029617735184729, -0.0037707402370870113, -0.0216799508780241, -0.04229014739394188, -0.005550963804125786, -0.0022053467109799385, -0.002438846044242382, -0.0014369337586686015, 0.007264688145369291, 0.004789070226252079, 0.000517750158905983, -0.014805343002080917, 0.03639429062604904, -0.012421724386513233, 0.0044162096455693245, -0.008295204490423203, 0.0004684962041210383, -0.010280598886311054, -0.004659721627831459, -0.0015679079806432128, -0.005703224800527096, -0.013517766259610653, 0.013747038319706917, -0.005791591480374336, 0.021861162036657333, 0.010064979083836079, -0.004321780521422625, 0.03744012862443924, 0.0029515568166971207, -0.012255907990038395, 0.013322757557034492, -0.0029776403680443764, -0.007530783768743277, -0.030978119000792503, -0.00017326338274870068, -0.01912512816488743, -0.0008840718655847013, 0.0023944624699652195, -0.008033129386603832, -0.003533064853399992, -0.02281789481639862, -0.00763707235455513, -0.041612789034843445, -0.0038814714644104242, -0.0071014054119586945, -0.008257577195763588, -0.012858899310231209, -0.04521345719695091, 0.006972184870392084, 0.007200766820460558, -0.005009188782423735, -0.012859349139034748, -0.0008323610527440906, -0.0017448525177314878, -1.1015607924491633e-05, 0.0030468550976365805, 0.012919728644192219, -0.0014462026301771402, -0.002026208909228444, 0.016107626259326935, -0.007326480001211166, -0.06624415516853333, -0.0043357182294130325, 0.0018770473543554544, 0.0013253671349957585, -0.004714035429060459, 0.020140307024121284, -0.004136800300329924, 0.003089889883995056, 0.01609594188630581, -0.000595375313423574, 0.013468900695443153, -0.0044676936231553555, -0.001210468471981585, -0.008201852440834045, 0.005332533270120621, -0.002215230604633689, -0.0060987165197730064, 0.0008299193577840924, -0.007150181569159031, 0.0023063533008098602, -0.004252846352756023, 0.004280726425349712, -0.017178043723106384, -0.0032669072970747948, 0.0025876828003674746]], "b2": [-0.025747040286660194, 0.019185543060302734, -0.08483436703681946, 0.009775123558938503, 0.04478057101368904, -0.019688434898853302, 0.06564979255199432, -0.020182309672236443, 0.02576589584350586, 0.02315501682460308, -0.029264133423566818, 0.04374310001730919, 0.02587398886680603, -0.0014768191613256931, -0.007221226580440998, -0.04033129662275314, 0.006177714094519615, 0.06454209238290787, -0.028684135526418686, -0.01909448765218258, -0.08326296508312225, 0.0526956282556057, -0.013767499476671219, 0.05833955481648445, -0.05982007458806038, 0.024136926978826523, 0.0034784276504069567, -0.07893473654985428, -0.0069082654081285, 0.06264404952526093, 0.02963949181139469, -0.012876423075795174], "W3": [[-0.05784167721867561, -0.08466774970293045, 0.2536984384059906, 0.2219689041376114, 0.1817742884159088, -0.015366634353995323, 0.2343708872795105, -0.1837555319070816, 0.17907238006591797, -0.16035357117652893, -0.10555694252252579, 0.11453203111886978, 0.11969994008541107, 7.010222907410935e-05, 0.0013590961461886764, -0.04273819178342819, 0.050547342747449875, -0.22763065993785858, 0.10926532745361328, 0.2931961417198181, 0.18546052277088165, 0.346273809671402, 0.2090647667646408, -0.22685058414936066, 0.27005118131637573, 0.14428795874118805, -0.19657030701637268, -0.1337195634841919, 0.2271626591682434, 0.19143730401992798, -0.1406603455543518, 0.065379798412323]], "b3": [-0.048224449157714844]}}''')


def _erf_np(x):
    """Abramowitz-Stegun 7.1.26, |err|<1.5e-7。"""
    sign = np.sign(x)
    ax = np.abs(x)
    t = 1.0 / (1.0 + 0.3275911 * ax)
    y = 1.0 - (((((1.061405429 * t - 1.453152027) * t) + 1.421413741) * t
                - 0.284496736) * t + 0.254829592) * t * np.exp(-ax * ax)
    return sign * y


def _gelu_np(x):
    return 0.5 * x * (1.0 + _erf_np(x / np.sqrt(2.0)))


def _predict_year(year, values):
    w = MLP_WEIGHTS[str(_model_year(year))]
    W1 = np.asarray(w["W1"], dtype=np.float64)
    b1 = np.asarray(w["b1"], dtype=np.float64)
    W2 = np.asarray(w["W2"], dtype=np.float64)
    b2 = np.asarray(w["b2"], dtype=np.float64)
    W3 = np.asarray(w["W3"], dtype=np.float64)
    b3 = np.asarray(w["b3"], dtype=np.float64)
    X = values.astype(np.float64)
    h = _gelu_np(X @ W1.T + b1)
    h = _gelu_np(h @ W2.T + b2)
    return (h @ W3.T + b3).ravel()


# ---------------------------------------------------------------- 主入口

def main(datasources, start_date, end_date):
    import dai  # noqa: F401  (平台环境)
    df1m = _load_bar1m(datasources, start_date, end_date, buf_days=100)
    fl = _load_factorlib(start_date, end_date, buf_days=100)
    fin = _load_fin(datasources, start_date, end_date)
    stk_grid = _stk("2019-01-01", _fin_grid_end(end_date))

    feats = _features(df1m, fl, fin, stk_grid)

    panel = None
    for name in FEATURE_COLS:
        f = feats[name].rename(columns={"factor": name})
        panel = f if panel is None else panel.merge(
            f, on=["date", "instrument"], how="inner")
    panel = panel.sort_values(["date", "instrument"]).reset_index(drop=True)
    panel = panel[(panel["date"] >= pd.to_datetime(start_date))
                  & (panel["date"] <= pd.to_datetime(end_date))]
    # 先收敛到股票池再 z: 训练面板的特征缓存本身就只含池内股票,
    # 若 z 在全市场(含池外 ~2000 只)上算, 均值/std 不同导致输出偏移
    stk = _stk(start_date, end_date)
    panel = panel.merge(stk, on=["date", "instrument"], how="inner")
    for name in FEATURE_COLS:
        g = panel.groupby("date", group_keys=False)[name]
        panel[name] = (panel[name] - g.transform("mean")) \
            / g.transform("std").replace(0, np.nan)

    panel["year"] = panel["date"].dt.year
    pred = np.full(len(panel), np.nan)
    for yr in sorted(panel["year"].unique()):
        idx = np.flatnonzero((panel["year"] == yr).to_numpy())
        values = panel.iloc[idx][FEATURE_COLS].fillna(0.0) \
            .astype("float32").to_numpy()
        pred[idx] = _predict_year(int(yr), values)
    panel["factor"] = pred * SIGN

    out = panel
    out["factor"] = out["factor"].replace([np.inf, -np.inf], np.nan)
    out = out.dropna(subset=["factor"])
    return out[["date", "instrument", "factor"]]
